# 45 — chỉ số ST không chú giải + chẩn đoán phần dư (COAD)

Chỉ nạp map đã lưu rồi tính. **CPU là đủ, không cần GPU.**

## Phần D — chẩn đoán, chạy TRƯỚC khi quyết định làm FROST 1.0.2

Bản **liên tục** của `run20` Phần D. Vòng 20 đo η²(niche) của phần dư (cấu trúc theo **lớp**) và được 0.001–0.005. Nhưng phần dư có thể có tự tương quan **không gian** mà không có cấu trúc theo lớp — hai thứ khác nhau. Ở đây fit Θ trên T1, lấy phần dư trên T2, đo Moran's I của từng chiều phần dư theo toạ độ vật lý, so với mốc hoán vị toạ độ.

- Chênh so với hoán vị **< 0.02** → phần dư không mang cấu trúc không gian liên tục; số hạng phạt Laplacian của 1.0.2 **không có gì để tác động** → đóng hướng đó bằng số liệu.
- Chênh **≥ 0.02** → còn tín hiệu không gian chưa sửa → số hạng mục tiêu có cơ sở.

## Phần M — hai chỉ số ST mới cho MỌI phương pháp trong bảng chính

Nhãn chuyển sang = bỏ phiếu 15-NN nhãn tham chiếu (đúng `run22._knn_bacc`), rồi so **tổ chức không gian** của nhãn chuyển với nhãn **thật**:

- **Moran's I trên nhãn** — từng loại tế bào, chỉ báo nhị phân, trọng số từ toạ độ.
- **PAS** — tỉ lệ tế bào có nhãn khác đa số láng giềng **không gian**.

Cả hai báo cáo dưới dạng **|Δ so với nhãn thật|**, không phải giá trị thô: gán tất cả cùng một nhãn cho PAS = 0 và Moran's I suy biến, nên "thấp hơn là tốt hơn" trên giá trị thô sẽ thưởng cho nhãn vô dụng.

## Phần A — sàng α: chặn trên của FROST 1.0.2

Làm trơn không gian **hậu kỳ** map FROST rồi chấm lại. Nghiệm của `min_Z ‖Z−Z₀‖² + α·tr(ZᵀLZ)` là `(I+αL)⁻¹Z₀ ≈ (1−α)Z₀ + α·W_sp·Z₀` — đúng **toán tử gần kề** của số hạng phạt Laplacian định thêm vào M-step.

**Đây là CHẶN TRÊN, không phải 1.0.2.** Làm trơn hậu kỳ tác động theo **từng tế bào**; M-step phải thể hiện mọi thứ qua **một ma trận W tuyến tính duy nhất**, tự do ít hơn hẳn. Nên: đường α hỏng → 1.0.2 **chắc chắn** hỏng, dừng; đường α đẹp → 1.0.2 *có thể* được, lúc đó mới đáng viết code.

Ba nhánh — α=0 của `pre` chính là **FROST 1.0**, α=0 của `nocoral` chính là **u_cmix**:

| nhánh | công thức | nghĩa |
|---|---|---|
| `nocoral` | `smooth(F)` | tác dụng thuần của làm trơn |
| `pre` | `coral(smooth(F))` | **khớp với Laplacian ở M-step** |
| `post` | `smooth(coral(F))` | chữa triệu chứng trực tiếp |

**Bẫy đọc `d_cov`:** ở nhánh `pre` nó ≈ 0 với **mọi** α một cách tầm thường, vì CORAL chạy sau cùng nên ép lại `Σ(Z[T]) = Σ_x`. Đừng đọc đó là "an toàn" — `d_cov` chỉ có nghĩa cho `nocoral` và `post`. Với `pre`, rủi ro trộn lô phải chấm bằng `R23.evaluate` (**chưa làm ở đây**): Σ khớp không bảo đảm iLISI khớp, vì iLISI phụ thuộc trộn lô **cục bộ**.

## Phần T — trần của `xfer_main`

Ở n=12, `xfer_main` **không phân biệt được** bốn can thiệp khác hẳn nhau về cấu trúc (bỏ CORAL p=0.79 · `usher_cc` p=0.85 · `usher_final` p=0.68 · `class_ridge_pl_cc` p=0.29), trong khi cùng chỉ số đó tách FROST khỏi harmony (+0.030) và scanorama (+0.099) rất rõ. Đó là dấu hiệu **bão hoà trong nội bộ họ USHER**. Phần T đo trần nằm ở đâu.

Oracle theo nhãn là vô dụng (đặt tế bào nguồn cạnh tế bào Xenium cùng nhãn thì kNN đương nhiên đúng → ~1.0 tầm thường). Nên đo hai đại lượng không suy biến:

- **`tran_dich`** — Xenium → Xenium (ref → phần giữ lại). Cùng chú giải, cùng nhúng, khoảng cách công nghệ **bằng 0**. Trần của chính thước đo.
- **`tran_nguon`** — nguồn → nguồn (T → Q) trong không gian nguồn. M-step là ánh xạ **tuyến tính** nên không tạo ra khả năng phân tách vốn không có.
- **`*_ngau`** — hoán vị nhãn dự đoán, cho biết sàn.

Dùng `lab_tgt_ev`/`lab_src_ev` + `groups_ev`, tức **đúng nhãn và đúng nhóm** mà `xfer_main` chấm cho cặp đó (thô hay mịn), nếu không hai con số không so được.

**Đọc:** `tran_dich` sát `xfer_main` của FROST (xem `out23e`) → `xfer_main` bão hoà, từ nay chỉ dùng làm ràng buộc *không được tụt*, không dùng xếp hạng biến thể.

## Phần R — FROST 1.0.3: CORAL quay theo không gian

Khuyết tật ST (nhãn chuyển **vụn** trong không gian: PAS vượt sự thật 4/4, Moran thấp hơn 4/4) do **chính CORAL** gây ra — đo được trên hai bộ map độc lập. Nên sửa ở CORAL.

CORAL là `Z = (F−μ_s)·Σ_s^(−1/2)·Σ_x^(1/2) + μ_x`. Nghiệm đó **không duy nhất**: với mọi `R` trực giao, `A_R = Σ_s^(−1/2)·R·Σ_x^(1/2)` cho `Cov(Z) = Σ_x^(1/2)·RᵀR·Σ_x^(1/2) = Σ_x`, y hệt nhau. Chọn `R = I` là **tuỳ tiện** — một bậc tự do chuẩn đang bị bỏ phí. Nhưng `Σ_s^(−1/2) R Σ_x^(1/2)` **không** trực giao nên `R` có đổi hình học kNN, tức đổi nhãn chuyển và tính liền mạch.

Dùng bậc tự do đó để tối thiểu độ gồ ghề không gian. Với `B = (F−μ_s)Σ_s^(−1/2)` và `M = BᵀLB`:

```
min_{RᵀR=I}  tr(ZᵀLZ) = tr(Rᵀ M R Σ_x)
```

**Có nghiệm đóng** — bất đẳng thức vết Ruhe: ghép trị riêng **nhỏ nhất** của `M` với trị riêng **lớn nhất** của `Σ_x`, tức `R* = U_M[:, ::-1] @ U_Xᵀ`. Đổi dấu cột không đổi mục tiêu nên ép `det = +1` được miễn phí. `R(t) = cực-phân-rã((1−t)I + t·R*)`, `t=0` cho đúng CORAL hiện hành.

Đã kiểm bằng số (n=800, d=12): `det(R*) = +1`; `rough(R*)` thấp hơn `rough(I)` 25% **và thấp hơn điểm tốt nhất trong 4000 `R` ngẫu nhiên**; nội suy giữ trực giao tới 1e-15.

**Vì sao hơn phần A:** phần A làm trơn đầu ra nên **bóp** phương sai (`d_cov` lên 0.22–0.49 ở nhánh `post`). Ở đây `Σ` bị **ghim cứng**, chỉ còn quay — ràng buộc chặt hơn hẳn, nên khả năng bào vào tín hiệu sinh học thấp hơn. Chi phí: vài giây mỗi `t`, **không dựng lại map**.

**Tiêu chí 1.0.3:** `pas_abs_dev` giảm > 0.005 và `total_v2` không tụt quá 0.01 so với `t=0`. `xfer_main` in ra để biết giá phải trả, **không** dùng làm điều kiện.

## Phần V — gộp bền vững thay cho trung bình

`pas_true` = 0.20-0.38 không phải "tín hiệu kém". Nó là phát biểu chính xác rằng lân cận là mẫu **bị nhiễm bẩn 20-38%**. Điểm sụp đổ của **trung bình là 0** — một tế bào khác loại đủ kéo lệch; của **trung vị là 0.5**. Mức nhiễm bẩn đo được nằm gọn trong vùng trung vị còn sống mà trung bình đã chết.

Mà **mọi thứ dự án đã thử đều là trung bình**: `cmix`, beta không gian, làm trơn alpha, phạt Laplacian, CORAL quay, co rút Phần K. Không cái nào dùng ước lượng bền vững.

Phần V giữ **nguyên** cơ chế co rút của Phần K (`w` tính một lần, dùng chung mọi nhánh) và chỉ đổi đúng một thứ: ước lượng vị trí `m_i`. Chênh lệch quy thẳng về nó.

**Dự đoán đặt TRƯỚC khi chạy:** nếu nguyên nhân là nhiễm bẩn thì mức hơn của trung vị phải **tăng theo `pas_true`** — cosmx6k .378 > visiumhd_ff .288 > visiumhd_ffpe .258 > stereoseq .197. Không theo thứ tự đó thì giả thuyết sai, **không được diễn giải lại sau khi nhìn số**. Báo cáo tự in phán quyết.

Mốc phải vượt là `raw`, không phải `tb`.

## Phần W — không gian làm bộ lọc CHIỀU, không phải bộ lọc tế bào

Tín hiệu không gian đúng **62-80%** (`pas_true` 0.20-0.38). Dùng ở cấp **từng tế bào** thì vô dụng: mỗi lân cận là mẫu cỡ 15 với 1/3 nhiễm bẩn. Cùng tín hiệu đó ở cấp **quần thể** trên 45.000 tế bào có sai số chuẩn ~0.005 — **yếu ở cá thể, rất chắc ở tổng thể**. Sáu ý tưởng trước đều là MỘT ý tưởng (gộp láng giềng vào tế bào) dùng ở cấp sai.

Ở đây không gian không nói *tế bào này giống hàng xóm*, nó nói **chiều này mới là sinh học**: nhiễu kỹ thuật (độ sâu, hiệu ứng lô, tràn tín hiệu giữa bin) không có tổ chức không gian; cấu trúc mô thì có.

**Cổng là `SPEARMAN_ng_đích`** — tương quan giữa Moran theo chiều của nguồn và của Xenium. Hai bên cùng cơ sở nên chiều so được. Cao → *chiều có cấu trúc không gian* là tính chất chung, độc lập công nghệ, đánh trọng số theo nó có nghĩa. ~0 → đặc thù từng công nghệ, trọng số tuỳ tiện, ý tưởng chết.

Và một phép thử **trực tiếp**, không suy luận gián tiếp: chấm với top-N chiều chọn theo Moran so với top-N chọn theo **phương sai**. Hơn → không gian chọn ra sinh học mà phương sai bỏ sót.

## Phần X — FROST 1.0.4: Spatial-CORAL

**Lập luận là một chứng minh, không phải một quan sát.** CORAL tính `Σ_s`, `Σ_x` từ ma trận tế_bào×chiều. Hoán vị thứ tự hàng không đổi hiệp phương sai. Vậy **CORAL bất biến với hoán vị tế bào**, nên nó mù thông tin không gian *về mặt cấu trúc* — không phải vì chưa tinh chỉnh đủ. Không α, không ridge, không lưới siêu tham số nào sửa được.

Phần W đo chỗ mù đó tốn gì: Moran **trung vị theo chiều** lệch 40–50% giữa nguồn và Xenium (vhd_ff 0.410 vs 0.285; stereoseq 0.438 vs 0.285; cosmx6k 0.033 vs 0.131 — ngược chiều). Căn chỉnh hiện tại khớp `Σ` xong rồi để nguyên khoảng lệch đó.

Thêm **mô-men bậc hai có trọng số không gian**, trong hệ đã tẩy trắng:

```
A = Bᵀ W̃_s B / n_s     B  = (F−μ_s)Σ_s^(−1/2)
G = Bxᵀ W̃_x Bx / n_x    Bx = (X−μ_x)Σ_x^(−1/2)
```

`W̃ = (Wn+Wnᵀ)/2` vì `Wn` chuẩn hoá theo hàng nên không đối xứng. Bậc tự do còn lại sau CORAL **đúng là nhóm trực giao** (lập luận ở Phần R), và trong hệ tẩy trắng mô-men không gian của `Z` là `RᵀAR`. Nên

```
min_{RᵀR=I} ‖RᵀAR − G‖_F   <=>   max_R tr(RᵀAR·G)
```

**Nghiệm đóng** — lại là Ruhe, nhưng ghép **thuận**: `max = Σ λ↑(A)λ↑(G)`, đạt tại `R* = U_A U_Gᵀ`. `R(t) = cực-phân-rã((1−t)I + t·R*)`; `t=0` cho **đúng FROST** khi `SPC_TAG=cmix`. Không huấn luyện, không siêu tham số ngoài `t`, cắm thẳng vào khe CORAL.

**Tính mới:** scGPT-spatial và Nicheformer nhét không gian vào *encoder*. Chưa ai khớp mô-men không gian ở bước **căn chỉnh hậu kỳ** — mà đó đúng là chỗ FROST sống.

**Ba cổng, đọc theo thứ tự; trượt cổng nào dừng tại đó:**

1. `d_sp_san` so với `d_sp_t0` — sàn là chặn dưới của `‖RᵀAR−G‖/‖G‖` trên **toàn** nhóm trực giao, tính thẳng từ hai phổ. Sàn ≈ t0 → phép quay về nguyên tắc không sửa được gì, 1.0.4 chết bất kể các số còn lại.
2. `d_cov` phải **đứng yên** ở mức `RIDGE≈0.01` với mọi `t`. Tăng theo `t` là **code sai**, không phải phát hiện (bài học Phần R: ridge phá bất biến chuẩn ở d=512).
3. `hoanvi@t1` so với `t1` — `R*` dựng từ đồ thị nguồn **đã hoán vị**. Ngang nhau → thứ mua được là bậc tự do quay nói chung, **không phải** thông tin không gian; hết tính mới. Phần K đã từng thua chính nhánh null này 4/4, nên cổng này không thừa.

Chỉ khi qua cả ba mới đọc `total_v2`/`xfer_main`, và mốc so là `t0`, **không** phải bảng `u_cmix_cc` của vòng 23e (khác đường tính, lệch vài phần nghìn).

## Phần Y — cổng CPU cho C_sp: toạ độ làm ma trận cấu trúc GW

**Dữ kiện quyết định:** vòng 17 Phần A đo **đăng ký affine thất bại 0/4** (`same_label` chỉ nhỉnh ngẫu nhiên 2–6 điểm, `top1/top2` ≈ 1, stereoseq lệch 174°/2556 µm). Nên mọi cách dùng toạ độ ở chi phí **liên-miền** — gồm `spatial_b*` của vòng 18 — là nạp nhiễu.

PASTE cố ý **không** đặt toạ độ vào chi phí liên-lát: toạ độ chỉ vào ma trận khoảng cách **trong một lát** `D`, dùng làm số hạng cấu trúc Gromov–Wasserstein, vì toạ độ tuyệt đối phụ thuộc cách cắt và đặt mô nên không so được giữa các lát. Hệ quả: số hạng đó **bất biến với xoay, tịnh tiến, lật** → miễn nhiễm với việc đăng ký thất bại. Đó là khe duy nhất còn hợp lệ cho toạ độ ở bộ dữ liệu này, và `struct_cache.py` `MODE="custom"` đã là đúng khe đó (vòng 17 dùng nó với đặc trưng niche → `C_niche`, `C_mix`; `C_mix` thành FROST). **Toạ độ chưa bao giờ vào đó.**

`spatial_utils.py` của USHER có cửa sổ không gian nhưng dùng **cùng biên cửa sổ** cho hai bên ("For affine-aligned data"), tức đòi đúng cái đăng ký bất khả → lấy bản vá **độc lập** mỗi bên, khớp **số tế bào**, vị trí tuỳ ý.

**Vì sao cổng CPU trước, không sửa `src_cuda` ngay:** đổi cách lấy batch + thêm `C_sp` là vá `src_cuda` rồi chạy lại USHER trên GPU, 4 cặp × 3 seed. Tám ý tưởng trước đã chết. Mọi thứ cần để đoán ý tưởng này sống hay chết đo được trên CPU trong vài phút.

**Ba cổng:**

1. `chênh = tk_kgian − tk_biểu_hiện`, **trên cả hai bên**. Việc duy nhất của ma trận cấu trúc là nói "hai tế bào này gần nhau về cấu trúc". Chênh âm rõ ở cả hai bên → biểu hiện đã nói chính xác hơn toạ độ, trộn toạ độ vào `C` chỉ làm bẩn. Đúng logic đã giết Phần W.
2. `chồng_lấp_sp` và `d1_tỉ_lệ` — chẩn đoán lệch mật độ. Cổng **yếu nhất** (sau khi khớp số tế bào nó phần nào thành hệ quả hình học), đọc để giải thích chứ không để kết luận.
3. `lợi_sp` so với `lợi_kgian_ngau` và `lợi_ex` — giải GW **thật** trên một cặp bản vá rồi đo tỉ lệ khối lượng ghép vào cặp **cùng nhãn**, trừ mức ngẫu nhiên. Đây là ước lượng trực tiếp thứ `C_sp` sẽ mua được bên trong USHER, không cần chạy USHER. Null là **hình học ngẫu nhiên** (toạ độ đều trong cùng hộp bao), **không** phải hoán vị: GW bất biến với hoán vị đồng thời hàng–cột nên hoán vị là null rỗng.

Hiệu chuẩn (dữ liệu tổng hợp): `tk_kgian` trên hình học ngẫu nhiên tụt đúng về mức ngẫu nhiên (0.4085 vs 0.409), và `đồng_ý` trên kế hoạch vận chuyển đều trả đúng mức ngẫu nhiên.

**Qua cả ba thì bước 2 mới đáng làm:** batch theo bản vá không gian độc lập hai bên + `C = (1−λ)C_expr + λ·C_sp` trong `struct_cache`, λ quét quanh 0.1–0.3 theo gợi ý α của PASTE (họ thấy α=1, tức toạ độ không thôi, khớp đúng **0** cặp).

## Chạy lại để bổ sung phần mới

`run45` nạp lại `summary.json` từ Input khi thư mục làm việc chưa có, và **bỏ qua mọi mục đã chấm**. Nên muốn thêm phần A vào kết quả cũ: Add output 45a cũ (`out23s`) làm Input, đặt `ONLY_PART="A"`, chạy lại. D và M không bị chấm lại; `summary.md` cuối cùng vẫn có đủ ba phần.

## Phải đọc trong log

- `đồ thị: cắt X% cạnh` — nếu **> 30%** thì đồ thị không gian đã rời rạc, chỉ số ST đọc thận trọng; cân nhắc tăng `R_MAX_UM`.
- `nhãn giữ k/n` — số loại tế bào qua được bộ lọc tự tương quan nền. Nếu `0/n` thì mô này không có loại nào đủ tổ chức không gian, phần M bị bỏ cho cặp đó.
- `tiếp nối <- ...` — xác nhận đã nạp được summary cũ từ Input. KHÔNG thấy dòng này khi chạy lại nghĩa là D/M sẽ bị chấm lại từ đầu.

**Add Input:**

**Add Input (đủ, không thiếu cái nào):**
1. `spatch-coad-lite` — dữ liệu lite (count, toạ độ, nhãn)
2. `spatch-coad` — dữ liệu đầy đủ
3. output **03a** — `emb_scgpt/` (embedding scGPT của tế bào)
4. output **03d** — quy tắc niche data
5. output **04b** — `recoverable_classes.json`
6. output **21** — `out21/`, map USHER-ST sàn 200 (tên `tgt_floor`)
7. output **22** — `out22/`, map các sàn khác
8. output **31a + 32a** — `out23j/`, map `cmix` (FROST) **và** cache niche (`R22.ctx` gọi `get_niche_label_comp`, thiếu là chạy lại rất lâu)
- output **23j** — map `cmix` → chấm **FROST**
- output **23n** — map `usher_orig` → chấm **USHER bản gốc 25 vòng** (vòng 38 nhánh B1, chỉ seed 0, chỉ 4 cặp COAD)

**Kết quả:** tải `out23s/summary.json` và `out23s/summary.md`.

In [ ]:
# Cell 1 — COAD: dùng dữ liệu/đường dẫn gốc của COAD (không phải bản lite theo mô)
import os, glob
os.environ.setdefault("SMOKE", "0")
os.environ["TISSUE"] = "coad"; os.environ["MODE26"] = "report"; os.environ["RUN_PREFIX"] = "23"
os.environ["SKIP_INPUT_CHECK"] = "1"
os.environ.setdefault("COMBOS_ALL", "cosmx6k/pca,visiumhd_ff/scgpt,visiumhd_ffpe/pca,stereoseq/scgpt")
os.environ.setdefault("FINAL_FLOOR", "200"); os.environ.setdefault("WITH_SYMPHONY", "1"); os.environ.setdefault("WITH_NICHE", "0")
os.environ.setdefault("PATCH_SEEDS", "0,1,2"); os.environ.setdefault("ALT_FLOORS", "0"); os.environ.setdefault("SCANO_RETRY", "1"); os.environ.setdefault("WITH_PROC", "1"); 
os.environ.setdefault("OUT_DIR_45", "/kaggle/working/out23s")
# Phần cần chạy: "D" = chẩn đoán phần dư (cổng quyết định FROST 1.0.2),
#               "M" = Moran's I trên nhãn + PAS,
#               "A" = sàng α (chặn trên của FROST 1.0.2, RẺ),
#               "B" = chấm ĐẦY ĐỦ R23.evaluate ở vài điểm α (ĐẮT, ~1-2 giờ),
#               "T" = TRẦN của xfer_main (RẺ),
#               "R" = CORAL QUAY theo không gian (CPU, ~1 giờ),
#               "K" = CỔNG khử nhiễu không gian trên nhúng nguồn (CPU, ~15 phút),
#               "S" = THANG ĐO: pas_true theo bán kính lân cận (CPU, ~10 phút),
#               "V" = gộp BỀN VỮNG (trung vị) thay trung bình (CPU, ~20 phút),
#               "W" = Moran theo CHIỀU: không gian làm bộ lọc chiều (CPU, ~25 phút).
# CHẠY LẠI: Add output cũ của 45a làm Input rồi đặt ONLY_PART="A" — run45 nạp lại summary.json
# từ Input và BỎ QUA mọi mục đã có, nên D/M không bị chấm lại.
os.environ.setdefault("ONLY_PART", "D,M,A")
# Cắt cạnh không gian dài hơn ngần này (µm). BẮT BUỘC: Q lấy từ tile giữ lại của bàn cờ 500µm,
# không cắt thì hai tế bào ở hai tile khác nhau thành "láng giềng" qua một khe trống.
os.environ.setdefault("R_MAX_UM", "100")
# Chỉ chấm Moran's I trên nhãn CÓ tổ chức không gian (I của nhãn thật >= ngưỡng này).
os.environ.setdefault("MORAN_MIN", "0.10")
os.environ.setdefault("K_SPATIAL", "15")
# Phần A: hệ số làm trơn không gian hậu kỳ, và map nền (cmix = FROST CHƯA CORAL).
os.environ.setdefault("ALPHAS", "0.05,0.1,0.2,0.3,0.4,0.5")
os.environ.setdefault("ALPHA_TAG", "cmix")
# Phần B: các điểm α đem đi chấm đầy đủ. pre@0 = FROST 1.0, PHẢI giữ để có mốc so.
os.environ.setdefault("PART_B_ARMS", "pre@0,pre@0.2,pre@0.3,pre@0.4,pre@0.5")
# Phần T: cỡ tập Xenium giữ lại để đo trần.
os.environ.setdefault("N_T_MAX", "20000")
# Phần R: map nền (cmix_levns = FROST 1.0.1 leverage thuần, CHƯA CORAL), lưới t, và các t
# được chấm ĐẦY ĐỦ bằng R23.evaluate (t khác chỉ có PAS/Moran, rẻ).
os.environ.setdefault("ROT_TAG", "cmix_levns")
os.environ.setdefault("ROT_TS", "0,0.25,0.5,0.75,1")
os.environ.setdefault("ROT_FULL", "0,0.5,1")
# Phần K: sigma^2 = c * trung vị(s^2). c không thứ nguyên nên khỏi ước lượng nhiễu tuyệt đối.
os.environ.setdefault("KC", "0.25,0.5,1,2,4")
# Phần S: các k để quét bán kính lân cận, trên CẢ đồ thị toàn bộ tế bào lẫn đồ thị Q.
os.environ.setdefault("SK_KS", "5,10,15,30")
# Phần V: ước lượng vị trí của lân cận. tb = trung bình (thứ mọi thứ đã dùng), trungvi = trung vị.
os.environ.setdefault("AGGS", "tb,trungvi")
# Phần W: số chiều giữ lại khi so chọn-theo-Moran với chọn-theo-phương-sai.
os.environ.setdefault("WN_TOP", "10,25,50")
# Phần X (FROST 1.0.4 Spatial-CORAL): map nền PHẢI là bản CHƯA CORAL — cmix cho t=0 = FROST.
os.environ.setdefault("SPC_TAG", "cmix")
os.environ.setdefault("SPC_TS", "0,0.25,0.5,0.75,1")
os.environ.setdefault("SPC_FULL", "0,0.5,1")
# Phần Y (cổng CPU cho C_sp): bản vá không gian độc lập mỗi bên, KHỚP SỐ TẾ BÀO chứ không
# khớp cỡ vật lý — GW chuẩn hoá C theo D.max() nên thang tuyệt đối bị chia mất, thứ phải
# so được là hình dạng phân phối khoảng cách, phụ thuộc số tế bào.
os.environ.setdefault("PATCH_N", "1500")
os.environ.setdefault("PATCH_MIN", "300")
os.environ.setdefault("N_PATCH", "8")
os.environ.setdefault("PK", "15")
os.environ.setdefault("GW_N", "800")
# cmix = FROST (out<P>j) · usher_orig = USHER BẢN GỐC 25 vòng, 1 seed (out<P>n, vòng 38 nhánh B1)
# cmix_lev/cmix_levns = FROST 1.0.1 (out<P>v, vòng 46) -> cần Add out<P>v làm Input.
os.environ.setdefault("EXTRA_MAPS", "cmix,usher_orig,cmix_lev,cmix_levns")   # map của 28; thiếu thì notebook chỉ ghi cảnh báo rồi bỏ qua   # COAD có sẵn map sàn 0 ở out21
print("TISSUE = coad | COMBOS_ALL =", os.environ["COMBOS_ALL"])
!pip install -q pot "anndata>=0.10,<0.12" scib-metrics harmonypy scanorama umap-learn
os.makedirs("/kaggle/working/usher", exist_ok=True)
for pat in ["**/xenium5k_sub.h5ad", "**/emb_scgpt/visiumhd_ff_final.npy", "**/recoverable_classes.json",
            "**/out21/*_tgt_floor_s0.npy", "**/out22/*_f200_s0.npy", "**/out23b/summary.json", "**/out23c/summary.json"]:
    h = glob.glob(f"/kaggle/input/{pat}", recursive=True); print(("OK   " if h else "THIẾU"), pat, h[:1])

# Kiểm Input RIÊNG cho notebook này — thiếu cái nào, hỏng ở đâu, nói luôn.
NEED = [
    ("**/xenium5k_sub.h5ad", "dữ liệu lite -> không chạy được gì"),
    ("**/spatch_coad/xenium5k_tx/transcriptome/adata.h5ad", "dữ liệu đầy đủ"),
    ("**/emb_scgpt/*_final.npy", "embedding scGPT của tế bào"),
    ("**/recoverable_classes.json", "tập nhóm khôi phục được -> tập lớp chấm điểm"),
    ("**/out21/*_tgt_floor_s0.npy", "map USHER-ST (sàn 200)"),
    ("**/out23j/*_cmix_s0.npy", "map FROST + cache niche")]
bad = []
for pat, why in NEED:
    h = glob.glob(f'/kaggle/input/{pat}', recursive=True)
    print(('OK   ' if h else 'THIẾU'), f'{pat:<52}', '' if h else '<- ' + why)
    if not h: bad.append(pat)
print(f'\nTHIẾU {len(bad)} Input — xem danh sách đánh số ở đầu notebook.'
      if bad else '\nĐủ Input.')

In [ ]:
# Cell kiểm phụ thuộc — `pip install -q` có thể thất bại ÂM THẦM (xung đột numpy/jax), và khi đó mọi mục cần
# scib_metrics sẽ hỏng trong khi notebook vẫn chạy tới cuối (đã xảy ra ở 33a: 8/8 mục mới ném ModuleNotFoundError).
import importlib, subprocess, sys
_BAD = []
for _mod, _pkg in (("scib_metrics", "scib-metrics"), ("harmonypy", "harmonypy"), ("scanorama", "scanorama"),
                   ("ot", "pot"), ("umap", "umap-learn"), ("anndata", "anndata>=0.10,<0.12")):
    try:
        importlib.import_module(_mod); print(f"OK    {_mod}")
    except Exception as _e:
        print(f"THIẾU {_mod} ({type(_e).__name__}) -> cài lại {_pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)
        importlib.invalidate_caches()
        try:
            importlib.import_module(_mod); print(f"      {_mod} OK sau khi cài lại")
        except Exception as _e2:
            print(f"      {_mod} VẪN THIẾU: {_e2}"); _BAD.append(_mod)
assert not _BAD, f"thiếu gói bắt buộc {_BAD} -> dừng, đừng chạy tiếp để khỏi ra bảng rỗng"

In [ ]:
%%writefile /kaggle/working/usher/run_usher.py
import concurrent.futures
import contextlib
import logging
import os
from typing import Optional, Tuple, Dict, List
import anndata as ad
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
import rapids_singlecell as rsc
import scanpy as sc
from sklearn.cluster import KMeans

logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(levelname)s - %(message)s')

# Đã bổ sung thêm FeatureTransform vào import
from model_utils import _to_dense_float32, FeatureTransform
from graph_utils import compute_knn_graph_distance, apply_cell_type_constraints
from e_step_utils import compute_transport_batch
from m_step_utils import train_global_model, apply_transfer_method, unstandardize_features, standardize_features
from plot_utils import plot_dual_umap, plot_weight_heatmap, plot_convergence, plot_transfer_debug_umap, plot_spatial_channels, plot_spatial_mapping

def align_features_fgw(
    adata_a: ad.AnnData,
    adata_b: ad.AnnData,
    e_step_method: str = 'fgw',  
    m_step_method: str = 'global',  
    sampling_strategy: str = 'celltype',  

    sketch_size: int = 1000,  
    sketch_obsm_key: str = 'X_umap',
    use_stratified_pairing: bool = True,  
    stratified_pairing_fix: bool = False,  
    cell_type_col: str = 'annotation_level_0',  

    spatial_key: str = 'X_spatial',  
    window_height: Optional[float] = None,  
    window_width: Optional[float] = None,  
    window_overlap: float = 0.1,  
    spatial_knn: int = 50,  
    n_windows_target: int = 20,  

    epsilon: float = 0.1,
    sinkhorn_iters: int = 1000,
    balanced_ot: bool = True,  
    use_linear_assignment: bool = True,

    knn_k: int = 30,  
    metric: str = 'cosine',  
    m_step_metric: str = 'euclidean',  
    use_knn_graph: bool = True,  

    celltype_probs_layer: str = 'X_celltype_probs',  
    alpha: float = 0.3,  
    gamma: float = 0.4,  
    sketch_pca_components: int = 50,
    
    n_iters: int = 30,
    steps_per_iter: int = 100,  
    lr: float = 1e-3,
    weight_decay: float = 1e-5,
    lambda_cross: float = 0.9,  
    lambda_struct: float = 0.0,            # Bổ sung lại lambda_struct cho USHER gốc
    lambda_var: float = 0.1,  
    structure_sample_size: Optional[int] = 2048, # Bổ sung lại structure_sample_size cho USHER gốc
    hidden_dim: Optional[int] = 512,  
    dropout: float = 0.0,  
    init_strategy: str = 'auto',  
    device: Optional[str] = None,
    debug_plots_path: Optional[str] = None,
    entropy_percentile: float = 90.0,
    confidence_percentile: float = 10.0,
    verbose: bool = True,
) -> Tuple[nn.Module, np.ndarray, ad.AnnData, torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor]]:
    
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device_t = torch.device(device)
    logging.info(f"Using device: {device}")

    if debug_plots_path:
        os.makedirs(debug_plots_path, exist_ok=True)
        for subdir in ["umap", "heatmap", "umap_transfer", "convergence"] + (["channels"] if sampling_strategy == 'spatial' else []):
            os.makedirs(os.path.join(debug_plots_path, subdir), exist_ok=True)

    X_a = _to_dense_float32(adata_a.X)
    X_b = _to_dense_float32(adata_b.X)

    features_a = torch.from_numpy(X_a).to(device_t)
    features_b = torch.from_numpy(X_b).to(device_t)

    n_a_orig, d_a = features_a.shape
    n_b_orig, d_b = features_b.shape

    # =====================================================================
    # DATA BATCHING
    # =====================================================================
    if sampling_strategy == 'spatial':
        from spatial_utils import prepare_spatial_batches
        batches, auxiliary_data, window_info = prepare_spatial_batches(
            adata_a, adata_b, spatial_key=spatial_key, window_height=window_height,
            window_width=window_width, window_overlap=window_overlap,
            n_windows_target=n_windows_target, spatial_knn=spatial_knn
        )
        auxiliary_features_source = auxiliary_data.get('coords_a')
        auxiliary_features_target = auxiliary_data.get('coords_b')
        knn_indices_spatial = auxiliary_data.get('knn_indices')
        n_a = n_a_orig
        sketch_to_original = None
    else:
        from celltype_utils import prepare_celltype_batches
        batches, auxiliary_data, sketch_to_original = prepare_celltype_batches(
            adata_a, adata_b, sketch_size=sketch_size, use_stratified_pairing=use_stratified_pairing,
            stratified_pairing_fix=stratified_pairing_fix, celltype_probs_layer=celltype_probs_layer,
            cell_type_col=cell_type_col, sketch_obsm_key=sketch_obsm_key,
            sketch_pca_components=sketch_pca_components, e_step_method=e_step_method, seed=2025
        )
        n_a = auxiliary_data['n_source']
        auxiliary_features_source = auxiliary_data.get('celltype_probs_a')
        auxiliary_features_target = auxiliary_data.get('celltype_probs_b')
        knn_indices_spatial = None
        
        if sketch_to_original is not None:
            features_a = features_a[sketch_to_original]

    global_feature_mean = features_b.mean(dim=0, keepdim=True)
    global_feature_std = features_b.std(dim=0, keepdim=True).clamp(min=1e-6)
    
    features_a_std = (features_a - global_feature_mean) / global_feature_std

    # =====================================================================
    # KHỞI TẠO MÔ HÌNH USHER GỐC (MLP - FeatureTransform)
    # =====================================================================
    model = FeatureTransform(input_dim=d_a, output_dim=d_b, hidden_dim=hidden_dim, dropout=dropout).to(device_t)

    if init_strategy == 'auto':
        if sampling_strategy == 'celltype':
            model.init_identity()
        else:
            model.init_random()
    elif init_strategy == 'identity':
        model.init_identity()
    elif init_strategy == 'random':
        model.init_random()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    features_a_transformed = None
    global_feature_mean = None
    global_feature_std = None
    convergence_data = [] 
    prev_iter_mappings = {} 

    def apply_model_with_scaling(features_in: torch.Tensor) -> torch.Tensor:
        if global_feature_mean is not None and global_feature_std is not None:
            features_std = standardize_features(features_in, global_feature_mean, global_feature_std)
            output_std = model(features_std)
            return unstandardize_features(output_std, global_feature_mean, global_feature_std)
        return model(features_in)

    # =====================================================================
    # E-M ITERATIONS
    # =====================================================================
    for it in tqdm(range(n_iters), desc="E-M Alignment (USHER Original)"):
        
        if sampling_strategy == 'celltype' and use_stratified_pairing and not stratified_pairing_fix:
            from celltype_utils import prepare_celltype_batches
            batches, auxiliary_data, _ = prepare_celltype_batches(
                adata_a, adata_b, sketch_size=sketch_size, use_stratified_pairing=True,
                stratified_pairing_fix=False, celltype_probs_layer=celltype_probs_layer,
                cell_type_col=cell_type_col, sketch_obsm_key=sketch_obsm_key,
                sketch_pca_components=sketch_pca_components, e_step_method=e_step_method, seed=2025 + it 
            )
            auxiliary_features_source = auxiliary_data.get('celltype_probs_a')
            auxiliary_features_target = auxiliary_data.get('celltype_probs_b')

        if m_step_method == 'transfer' and it == 0:
            features_a_transformed = features_a.clone()

        # ===== E-STEP =====
        batch_results = [None] * len(batches)
        def process_cluster(b_idx, src_idx, tgt_idx):
            ctx = torch.cuda.stream(torch.cuda.Stream(device=device_t)) if device_t.type == 'cuda' else contextlib.nullcontext()
            with ctx:
                if it == 0:
                    features_source_base = features_a
                else:
                    if m_step_method == 'transfer':
                        features_source_base = features_a_transformed
                    else:  
                        with torch.no_grad():
                            features_source_base = apply_model_with_scaling(features_a)

                if metric == 'cosine':
                    features_s_norm = F.normalize(features_source_base, p=2, dim=1)
                    features_t_norm = F.normalize(features_b, p=2, dim=1)
                else:  
                    if global_feature_mean is not None and global_feature_std is not None:
                        features_s_norm = standardize_features(features_source_base, global_feature_mean, global_feature_std)
                        features_t_norm = standardize_features(features_b, global_feature_mean, global_feature_std)
                    else:
                        features_s_norm = features_source_base
                        features_t_norm = features_b

                knn_constraint = knn_indices_spatial[src_idx] if sampling_strategy == 'spatial' and knn_indices_spatial is not None else None

                res = compute_transport_batch(
                    source_indices=src_idx if src_idx is not None else np.arange(n_a),
                    target_indices=tgt_idx,
                    features_source_all=features_s_norm, features_target_all=features_t_norm,
                    auxiliary_features_source=auxiliary_features_source,
                    auxiliary_features_target=auxiliary_features_target,  
                    gamma=gamma, epsilon=epsilon, metric=metric, balanced=balanced_ot, 
                    use_linear_assignment=use_linear_assignment, device=device_t, iteration=it, 
                    e_step_method=e_step_method, entropy_percentile=entropy_percentile, 
                    confidence_percentile=confidence_percentile, knn_k=knn_k, 
                    use_knn_graph=use_knn_graph, alpha=alpha, verbose=verbose,
                    knn_constraint_indices=knn_constraint, knn_penalty_weight=5.0,
                    sampling_strategy=sampling_strategy
                )
            return b_idx, res

        if device_t.type == 'cuda':
            with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(batches), 4)) as executor:
                futures = [executor.submit(process_cluster, i, src, tgt) for i, (src, tgt) in enumerate(batches)]
                for future in concurrent.futures.as_completed(futures):
                    b_idx, result = future.result()
                    batch_results[b_idx] = result
        else:
            for i, (src, tgt) in enumerate(batches):
                b_idx, result = process_cluster(i, src, tgt)
                batch_results[b_idx] = result

        for batch_idx, result in enumerate(batch_results):
            T = result['T']
            mapping_curr = T.argmax(dim=1).cpu().numpy()
            if it > 0 and batch_idx in prev_iter_mappings:
                mapping_prev = prev_iter_mappings[batch_idx]
                min_len = min(len(mapping_curr), len(mapping_prev))
                if min_len > 0:
                    n_changed = np.sum(mapping_curr[:min_len] != mapping_prev[:min_len])
                    convergence_data.append({
                        'iter': it, 'batch_idx': batch_idx, 'n_changed': n_changed,
                        'n_total_valid': min_len, 'pct_changed': 100.0 * n_changed / min_len
                    })
            prev_iter_mappings[batch_idx] = mapping_curr

        # =====================================================================
        # 🌟 M-STEP 🌟
        # =====================================================================
        if m_step_method == 'global':
            from m_step_utils import aggregate_training_data_from_batches
            source_agg, target_agg, agg_stats = aggregate_training_data_from_batches(
                batch_results=batch_results, features_source_all=features_a,
                features_target_all=features_b, entropy_percentile=entropy_percentile,
                confidence_percentile=confidence_percentile
            )
            
            step_losses, feature_mean, feature_std = train_global_model(
                model=model, optimizer=optimizer, source_features=source_agg,
                target_features=target_agg, steps_per_iter=steps_per_iter,
                lambda_cross=lambda_cross,
                lambda_var=lambda_var, metric=m_step_metric, 
                device=device_t,
                features_target_all=features_b  
            )
            if it == 0 or feature_mean is not None:
                global_feature_mean = feature_mean
                global_feature_std = feature_std

        elif m_step_method == 'transfer':
            features_a_transformed = apply_transfer_method(
                batch_results=batch_results, features_target_all=features_b,
                n_source_total=n_a, device=device_t
            )

        # ===== DEBUG PLOTS =====
        if debug_plots_path:
            if m_step_method == 'transfer':
                a_hat_cpu = features_a_transformed.detach().cpu().numpy()
            else:
                with torch.no_grad():
                    model.eval()  
                    a_hat_cpu = apply_model_with_scaling(features_a).detach().cpu().numpy()

            obs_sketched = adata_a.obs.iloc[sketch_to_original].copy() if sketch_to_original is not None else adata_a.obs.copy()
            obsm_dict = {spatial_key: adata_a.obsm[spatial_key][sketch_to_original]} if spatial_key and spatial_key in adata_a.obsm and sketch_to_original is not None else {}
                    
            adata_a_transformed = ad.AnnData(X=a_hat_cpu, obs=obs_sketched, obsm=obsm_dict if obsm_dict else None)
            adata_a_transformed.obs["type"] = "source_transformed"  

            adata_b_copy = adata_b.copy()
            adata_b_copy.obs["type"] = "target"

            concat_adata_iter = ad.concat([adata_a_transformed, adata_b_copy], axis=0, label="batch", keys=["source", "target"], index_unique="_")
            
            if cell_type_col in adata_a_transformed.obs.columns and cell_type_col in adata_b_copy.obs.columns:
                concat_adata_iter.obs[cell_type_col] = pd.concat([adata_a_transformed.obs[cell_type_col], adata_b_copy.obs[cell_type_col]]).values
                
            try:
                rsc.tl.pca(concat_adata_iter)
                rsc.pp.neighbors(concat_adata_iter, use_rep='X', metric=metric)
                rsc.tl.umap(concat_adata_iter)
            except Exception: pass

            plot_dual_umap(
                concat_adata_iter, cell_type_col=cell_type_col, title_prefix=f'UMAP Iter {it+1}',
                save_path=os.path.join(debug_plots_path, 'umap', f"umap_iter_{it+1:04d}.png"),
                type_palette={'source_transformed': '#1f77b4', 'target': '#ff7f0e'}
            )

        import gc; gc.collect(); torch.cuda.empty_cache()
        
    batch_mappings = [(i, r['T'].argmax(dim=1).cpu().numpy(), r['target_indices']) for i, r in enumerate(batch_results)]

    if debug_plots_path:
        plot_convergence(convergence_data=convergence_data, save_dir=os.path.join(debug_plots_path, 'convergence'))

    # Final transformation
    if m_step_method == 'transfer':
        a_hat_cpu = features_a_transformed.detach().cpu().numpy()
    else:
        with torch.no_grad():
            model.eval()
            a_hat_cpu = apply_model_with_scaling(features_a).detach().cpu().numpy()

    obs_sketched = adata_a.obs.iloc[sketch_to_original].copy() if sketch_to_original is not None else adata_a.obs.copy()
    adata_a_transformed = ad.AnnData(X=a_hat_cpu, obs=obs_sketched)
    adata_a_transformed.obs["type"] = "source_transformed"  
    
    adata_b_copy = adata_b.copy()
    adata_b_copy.obs["type"] = "target"
    concat_adata = ad.concat([adata_a_transformed, adata_b_copy], axis=0, label="batch", keys=["source", "target"], index_unique="_")
    if cell_type_col in adata_a_transformed.obs.columns and cell_type_col in adata_b_copy.obs.columns:
        concat_adata.obs[cell_type_col] = pd.concat([adata_a_transformed.obs[cell_type_col], adata_b_copy.obs[cell_type_col]]).values
    
    try:
        rsc.tl.pca(concat_adata); rsc.pp.neighbors(concat_adata, use_rep='X', metric=metric); rsc.tl.umap(concat_adata)
    except Exception: pass

    T_full = torch.zeros(n_a, n_b_orig, device=device_t, dtype=torch.float32)
    for result in batch_results:
        if use_stratified_pairing:
            T_full[torch.from_numpy(result['source_indices']).long().to(device_t)[:, None], torch.from_numpy(result['target_indices']).long().to(device_t)] = result['T']
        else:
            T_full[:, torch.from_numpy(result['target_indices']).long().to(device_t)] = result['T']

    if True:
        from model_utils import save_alignment_model
        dir_path = "../datasets/scGPT_example/"
        os.makedirs(dir_path, exist_ok=True)
        save_alignment_model(
            model=model, 
            save_path=os.path.join(dir_path, 'alignment_model.pt'), 
            feature_mean=global_feature_mean, 
            feature_std=global_feature_std, 
            gene_names=adata_a.var_names.tolist()
        )

    return model, np.full(n_a, -1, dtype=np.int64), concat_adata, T_full, global_feature_mean, global_feature_std

In [ ]:
%%writefile /kaggle/working/usher/e_step_utils.py
"""
E-step utilities for optimal transport computation (USHER Original Standard).
"""

import logging
from typing import Optional, Tuple, Dict
import numpy as np
import torch
import torch.nn.functional as F
import ot
import pandas as pd
import ot.backend as otb
ot.backend._BACKEND_IMPLEMENTATIONS = [
    b for b in ot.backend._BACKEND_IMPLEMENTATIONS 
    if b.__name__ != 'JaxBackend'
]
from graph_utils import compute_knn_graph_distance


def compute_transport_ot(
    features_source: torch.Tensor,
    features_target: torch.Tensor,
    aux_features_source: Optional[np.ndarray],
    aux_features_target: Optional[np.ndarray],
    gamma: float,
    epsilon: float,
    metric: str,
    balanced: bool,
    device: torch.device,
    iteration: int = 0,
    knn_mask: Optional[torch.Tensor] = None,
    knn_penalty: float = 5.0,
    warmstart_duals: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    verbose: bool = False
) -> torch.Tensor:
    
    n_source = features_source.shape[0]
    n_target = features_target.shape[0]
    gamma_effective = 1.0 if iteration == 0 else gamma

    if aux_features_source is not None and aux_features_target is not None:
        aux_source_torch = torch.from_numpy(aux_features_source).to(device).float()
        aux_target_torch = torch.from_numpy(aux_features_target).to(device).float()
        M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)

        if torch.isnan(M_aux).any():
            logging.warning("M_aux contains NaN values.")
            
        M_aux = M_aux / (M_aux.max().clamp(min=1e-8))
        if torch.isnan(features_source).any():
            logging.warning("features_source contains NaN values.")
            
        if metric == 'cosine':
            M_features = 1.0 - (features_source @ features_target.T)
        else: 
            M_features = torch.cdist(features_source, features_target, p=2)
            
        M_features = M_features / (M_features.max().clamp(min=1e-8))
        M = (gamma_effective) * M_aux + (1-gamma_effective) * M_features
    else:
        if metric == 'cosine':
            M = 1.0 - (features_source @ features_target.T)
        else:
            M = torch.cdist(features_source, features_target, p=2)
        M = M / (M.mean().clamp(min=1e-8))

    if knn_mask is not None:
        penalty_value = M[knn_mask].median() * knn_penalty 
        M = M.clone() 
        M[~knn_mask] += penalty_value

    p = torch.ones(n_source, device=device, dtype=torch.float32) / n_source
    q = torch.ones(n_target, device=device, dtype=torch.float32) / n_target

    backend = otb.TorchBackend()
    if balanced:
        if epsilon < 0.05:
            T = ot.bregman.sinkhorn_log(p, q, M, reg=epsilon, numItermax=1000, backend=backend)
        else:
            T = ot.bregman.sinkhorn(p, q, M, reg=epsilon, numItermax=1000, backend=backend)
        if not isinstance(T, torch.Tensor):
            T = torch.as_tensor(T, device=device).float()
    else:
        T = ot.unbalanced.sinkhorn_unbalanced(
            p.cpu().numpy(), q.cpu().numpy(), M.cpu().numpy(),
            reg=epsilon, reg_m=0.8, numItermax=1000
        )
        T = torch.from_numpy(T).to(device).float()

    row_sums = T.sum(dim=1, keepdim=True)
    row_sums[row_sums == 0] = 1.0
    T = T / row_sums
    if torch.isnan(T).any():
        T = torch.nan_to_num(T, nan=0.0)

    return T


def compute_transport_gw(
    features_source: torch.Tensor,
    features_target: torch.Tensor,
    epsilon: float,
    knn_k: int,
    metric: str,
    use_knn_graph: bool,
    device: torch.device,
    verbose: bool = False
) -> torch.Tensor:
    
    n_source = features_source.shape[0]
    n_target = features_target.shape[0]

    if use_knn_graph:
        C1 = compute_knn_graph_distance(features_source, k=knn_k, metric=metric, device=device)
        C2 = compute_knn_graph_distance(features_target, k=knn_k, metric=metric, device=device)
    else:
        C1 = torch.cdist(features_source, features_source, p=2)
        C1 = C1 / (C1.max() + 1e-8)
        C2 = torch.cdist(features_target, features_target, p=2)
        C2 = C2 / (C2.max() + 1e-8)

    p = torch.ones(n_source, device=device, dtype=torch.float32) / n_source
    q = torch.ones(n_target, device=device, dtype=torch.float32) / n_target

    T = ot.gromov.entropic_gromov_wasserstein(
        C1=C1, C2=C2, p=p, q=q, loss_fun='square_loss',
        epsilon=epsilon, max_iter=10000, tol=1e-6,
        verbose=verbose, log=False, backend='torch'
    )

    if not isinstance(T, torch.Tensor):
        T = torch.as_tensor(T, device=device).float()

    row_sums = T.sum(dim=1, keepdim=True)
    row_sums[row_sums == 0] = 1.0
    T = T / row_sums

    return T


# def compute_transport_fgw(
#     features_source: torch.Tensor,
#     features_target: torch.Tensor,
#     aux_features_source: Optional[np.ndarray],
#     aux_features_target: Optional[np.ndarray],
#     gamma: float,
#     epsilon: float,
#     alpha: float,
#     metric: str,
#     knn_k: int,
#     use_knn_graph: bool,
#     device: torch.device,
#     iteration: int = 0,
#     verbose: bool = False
# ) -> torch.Tensor:
    
#     n_source = features_source.shape[0]
#     n_target = features_target.shape[0]
#     gamma_effective = 1.0 if iteration == 0 else gamma

#     if use_knn_graph:
#         C1 = compute_knn_graph_distance(features_source, k=knn_k, metric=metric, device=device)
#         C2 = compute_knn_graph_distance(features_target, k=knn_k, metric=metric, device=device)
#     else:
#         C1 = torch.cdist(features_source, features_source, p=2)
#         C1 = C1 / (C1.mean() + 1e-8)
#         C2 = torch.cdist(features_target, features_target, p=2)
#         C2 = C2 / (C2.mean() + 1e-8)

#     if aux_features_source is not None and aux_features_target is not None:
#         aux_source_torch = torch.from_numpy(aux_features_source).float().to(device)
#         aux_target_torch = torch.from_numpy(aux_features_target).float().to(device)
#         M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)
#         M_aux = M_aux / (M_aux.max().clamp(min=1e-8))
        
#         if metric == 'cosine':
#             M_features = 1.0 - (features_source @ features_target.T)
#         else:
#             M_features = torch.cdist(features_source, features_target, p=2)
#         M_features = M_features / (M_features.max().clamp(min=1e-8))

#         M = (gamma_effective) * M_aux + (1-gamma_effective) * M_features
#     else:
#         if metric == 'cosine':
#             M = 1.0 - (features_source @ features_target.T)
#         else:
#             M = torch.cdist(features_source, features_target, p=2)
#         M = M / (M.max().clamp(min=1e-8))

#     p = torch.ones(n_source, device=device, dtype=torch.float32) / n_source
#     q = torch.ones(n_target, device=device, dtype=torch.float32) / n_target

#     T = ot.gromov.entropic_fused_gromov_wasserstein(
#         M=M, C1=C1, C2=C2, p=p, q=q, loss_fun='square_loss',
#         epsilon=epsilon, alpha=alpha, max_iter=10000,
#         tol=1e-6, verbose=verbose, log=False, backend='torch'
#     )

#     if not isinstance(T, torch.Tensor):
#         T = torch.as_tensor(T, device=device).float()

#     row_sums = T.sum(dim=1, keepdim=True)
#     row_sums[row_sums == 0] = 1.0
#     T = T / row_sums

#     return T

def compute_transport_fgw(
    features_source: torch.Tensor,
    features_target: torch.Tensor,
    aux_features_source: Optional[np.ndarray],
    aux_features_target: Optional[np.ndarray],
    gamma: float,
    epsilon: float,
    alpha: float,
    metric: str,
    knn_k: int,
    use_knn_graph: bool,
    device: torch.device,
    iteration: int = 0,
    verbose: bool = False,
    low_rank_dim: int = 64  # Kích thước Low-Rank
) -> torch.Tensor:
    
    # 🌟 KHỞI TẠO LOCK ĐỂ CHỐNG RACE CONDITION CỦA PYTORCH
    if not hasattr(compute_transport_fgw, 'lock'):
        import threading
        compute_transport_fgw.lock = threading.Lock()
        
    n_source = features_source.shape[0]
    n_target = features_target.shape[0]
    gamma_effective = 1.0 if iteration == 0 else gamma

    def fast_pca_threadsafe(X: torch.Tensor, q: int) -> torch.Tensor:
        X_centered = X - X.mean(dim=0, keepdim=True)
        cov = torch.mm(X_centered.T, X_centered) / max(1, X.shape[0] - 1)
        
        # Khóa luồng: Chỉ cho phép 1 luồng gọi linalg cùng lúc để PyTorch khởi tạo cuSOLVER an toàn
        with compute_transport_fgw.lock:
            _, eigenvectors = torch.linalg.eigh(cov)
            
        V = eigenvectors[:, -q:]
        return torch.mm(X_centered, V)

    # Ép Low-Rank để làm sạch dữ liệu trước khi dựng đồ thị KNN
    if features_source.shape[1] > low_rank_dim:
        features_source_lr = fast_pca_threadsafe(features_source, low_rank_dim)
        features_target_lr = fast_pca_threadsafe(features_target, low_rank_dim)
    else:
        features_source_lr = features_source
        features_target_lr = features_target

    # Tính toán ma trận cấu trúc C1, C2 trên không gian Low-Rank
    if use_knn_graph:
        C1 = compute_knn_graph_distance(features_source_lr, k=knn_k, metric=metric, device=device)
        C2 = compute_knn_graph_distance(features_target_lr, k=knn_k, metric=metric, device=device)
    else:
        C1 = torch.cdist(features_source_lr, features_source_lr, p=2)
        C1 = C1 / (C1.mean() + 1e-8)
        C2 = torch.cdist(features_target_lr, features_target_lr, p=2)
        C2 = C2 / (C2.mean() + 1e-8)

    # Tính toán ma trận trung gian (Feature Cost)
    if aux_features_source is not None and aux_features_target is not None:
        aux_source_torch = torch.from_numpy(aux_features_source).float().to(device)
        aux_target_torch = torch.from_numpy(aux_features_target).float().to(device)
        M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)
        M_aux = M_aux / (M_aux.max().clamp(min=1e-8))
        
        if metric == 'cosine':
            M_features = 1.0 - (features_source @ features_target.T)
        else:
            M_features = torch.cdist(features_source, features_target, p=2)
        M_features = M_features / (M_features.max().clamp(min=1e-8))

        M = (gamma_effective) * M_aux + (1-gamma_effective) * M_features
    else:
        if metric == 'cosine':
            M = 1.0 - (features_source @ features_target.T)
        else:
            M = torch.cdist(features_source, features_target, p=2)
        M = M / (M.max().clamp(min=1e-8))

    p = torch.ones(n_source, device=device, dtype=torch.float32) / n_source
    q = torch.ones(n_target, device=device, dtype=torch.float32) / n_target

    # Tối ưu FGW với C1, C2 đã được nén Low-Rank
    T = ot.gromov.entropic_fused_gromov_wasserstein(
        M=M, C1=C1, C2=C2, p=p, q=q, loss_fun='square_loss',
        epsilon=epsilon, alpha=alpha, max_iter=10000,
        tol=1e-6, verbose=verbose, log=False, backend='torch'
    )

    if not isinstance(T, torch.Tensor):
        T = torch.as_tensor(T, device=device).float()

    row_sums = T.sum(dim=1, keepdim=True)
    row_sums[row_sums == 0] = 1.0
    T = T / row_sums

    return T

def apply_linear_assignment(
    T: torch.Tensor,
    gamma_effective: float,
    e_step_method: str,
    M: Optional[torch.Tensor] = None,
    K: int = 3,                 # 🌟 Sức chứa: 1 target scRNA-seq nhận tối đa K Xenium
    tau_prob: float = 1e-4      # 🌟 Điểm neo ảo: Ngưỡng loại bỏ Outlier
) -> Tuple[torch.Tensor, np.ndarray, np.ndarray]:
    from scipy.optimize import linear_sum_assignment
    
    P = T.cpu().numpy()
    n_source, n_target = P.shape
    
    # 1. Nhân bản bến đỗ đích (Capacity-Constrained K-to-1)
    P_extended = np.repeat(P, K, axis=1)
    
    # 2. Đệm Điểm Neo Ảo (Dummy Nodes) nếu Xenium vẫn đông hơn sức chứa
    if P_extended.shape[1] < n_source:
        pad_size = n_source - P_extended.shape[1]
        dummy_cols = np.zeros((n_source, pad_size))
        P_extended = np.hstack((P_extended, dummy_cols))
        
    cost_matrix = -P_extended
    
    # 3. Chạy thuật toán Hungarian trên không gian mở rộng ảo
    row_indices, col_indices_extended = linear_sum_assignment(cost_matrix)
    
    # 4. Outlier Rejection (Kích hoạt Tế bào Ảo)
    # Từ chối ghép cặp nếu bị ép vào cột Dummy hoặc xác suất OT quá thấp
    valid_mask = (P_extended[row_indices, col_indices_extended] >= tau_prob) & \
                 (col_indices_extended < n_target * K)
    
    row_indices_valid = row_indices[valid_mask]
    col_indices_extended_valid = col_indices_extended[valid_mask]
    
    # 5. Ánh xạ ngược về ID gốc của tế bào scRNA-seq
    col_indices_valid = col_indices_extended_valid // K
    
    T_sparse = torch.zeros_like(T)
    T_sparse[row_indices_valid, col_indices_valid] = 1.0
    
    return T_sparse, row_indices_valid, col_indices_valid


def compute_transport_batch(
    source_indices: np.ndarray,
    target_indices: np.ndarray,
    features_source_all: torch.Tensor,
    features_target_all: torch.Tensor,
    auxiliary_features_source: Optional[np.ndarray] = None,
    auxiliary_features_target: Optional[np.ndarray] = None,
    gamma: float = 0.5,
    epsilon: float = 0.1,
    metric: str = 'cosine',
    balanced: bool = True,
    use_linear_assignment: bool = False,
    device: torch.device = None,
    iteration: int = 0,
    e_step_method: str = 'ot',
    entropy_percentile: float = 60.0,
    confidence_percentile: float = 40.0,
    knn_k: int = 15,
    use_knn_graph: bool = True,
    alpha: float = 0.5,
    verbose: bool = False,
    knn_constraint_indices: Optional[np.ndarray] = None,
    knn_penalty_weight: float = 5.0,
    warmstart_duals: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    sampling_strategy: str = 'celltype'
) -> Dict:
    
    if device is None:
        device = features_source_all.device

    source_idx_tensor = torch.from_numpy(source_indices).long().to(device)
    target_idx_tensor = torch.from_numpy(target_indices).long().to(device)

    features_source = features_source_all[source_idx_tensor]
    features_target = features_target_all[target_idx_tensor]

    aux_features_source = None
    aux_features_target = None
    if auxiliary_features_source is not None:
        if isinstance(auxiliary_features_source, pd.DataFrame):
            src_arr = auxiliary_features_source.to_numpy()
        else:
            src_arr = np.array(auxiliary_features_source)
        aux_features_source = src_arr[source_indices]
        
    if auxiliary_features_target is not None:
        if isinstance(auxiliary_features_target, pd.DataFrame):
            tgt_arr = auxiliary_features_target.to_numpy()
        else:
            tgt_arr = np.array(auxiliary_features_target)
        aux_features_target = tgt_arr[target_indices]

    knn_mask = None
    if knn_constraint_indices is not None and sampling_strategy == 'spatial':
        n_source_batch = features_source.shape[0]
        n_target_batch = features_target.shape[0]
        knn_mask = torch.zeros(n_source_batch, n_target_batch, device=device, dtype=torch.bool)
        for i in range(n_source_batch):
            knn_targets = knn_constraint_indices[i]
            for target_idx in knn_targets:
                if target_idx in target_indices:
                    local_idx = np.where(target_indices == target_idx)[0]
                    if len(local_idx) > 0:
                        knn_mask[i, local_idx[0]] = True

    if e_step_method == 'ot':
        T = compute_transport_ot(
            features_source, features_target, aux_features_source, aux_features_target,
            gamma, epsilon, metric, balanced, device, iteration,
            knn_mask=knn_mask, knn_penalty=knn_penalty_weight,
            warmstart_duals=warmstart_duals, verbose=verbose
        )
    elif e_step_method == 'gw':
        T = compute_transport_gw(
            features_source, features_target, epsilon=epsilon, knn_k=knn_k,
            metric=metric, use_knn_graph=use_knn_graph, device=device, verbose=verbose
        )
    elif e_step_method == 'fgw':
        T = compute_transport_fgw(
            features_source, features_target, aux_features_source, aux_features_target,
            gamma=gamma, epsilon=epsilon, alpha=alpha, metric=metric, knn_k=knn_k,
            use_knn_graph=use_knn_graph, device=device, iteration=iteration, verbose=verbose
        )
    else:
        raise ValueError(f"Unknown e_step_method: {e_step_method}.")

    n_source = T.shape[0]
    entropy = torch.zeros(n_source, device=device, dtype=torch.float32)
    max_probs = torch.zeros(n_source, device=device, dtype=torch.float32)
    
    chunk_size_ent = 5000
    for st in range(0, n_source, chunk_size_ent):
        en = min(st + chunk_size_ent, n_source)
        T_chunk = T[st:en]
        T_safe_chunk = T_chunk.clamp(min=1e-10)
        
        entropy[st:en] = -(T_chunk * torch.log(T_safe_chunk)).sum(dim=1)
        max_probs[st:en] = T_chunk.max(dim=1)[0]

    entropy_threshold = torch.quantile(entropy, entropy_percentile / 100.0)
    confidence_threshold = torch.quantile(max_probs, confidence_percentile / 100.0)

    low_entropy_mask = entropy <= entropy_threshold  
    high_confidence_mask = max_probs >= confidence_threshold  
    focused_mask = low_entropy_mask & high_confidence_mask

    row_indices = None
    col_indices = None
    if use_linear_assignment:
        gamma_effective = 1.0 if iteration == 0 else gamma
        T, row_indices, col_indices = apply_linear_assignment(
            T, gamma_effective, e_step_method, M=None,
            K=3, tau_prob=5e-5
        )

    return {
        'T': T,
        'source_indices': source_indices,
        'target_indices': target_indices,
        'row_indices': row_indices,
        'col_indices': col_indices,
        'focused_mask': focused_mask
    }

In [ ]:
%%writefile /kaggle/working/usher/m_step_utils.py
"""
M-step utilities for learning feature transformations.
"""

import logging
from typing import List, Dict, Optional, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_feature_scaler(
    features_target: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    feature_mean = features_target.mean(dim=0, keepdim=True)
    feature_std = features_target.std(dim=0, keepdim=True)
    feature_std = torch.clamp(feature_std, min=1e-6)
    logging.info(f"  [Feature Scaler] Computed from target:")
    logging.info(f"    Mean per feature: min={feature_mean.min():.6f}, max={feature_mean.max():.6f}, avg={feature_mean.mean():.6f}")
    logging.info(f"    Std per feature: min={feature_std.min():.6f}, max={feature_std.max():.6f}, avg={feature_std.mean():.6f}")
    return feature_mean, feature_std

def standardize_features(
    features: torch.Tensor,
    feature_mean: torch.Tensor,
    feature_std: torch.Tensor
) -> torch.Tensor:
    return (features - feature_mean) / feature_std

def unstandardize_features(
    features_standardized: torch.Tensor,
    feature_mean: torch.Tensor,
    feature_std: torch.Tensor
) -> torch.Tensor:
    return features_standardized * feature_std + feature_mean

def select_focused_cells(
    T: torch.Tensor,
    entropy_percentile: float = 30.0,
    confidence_percentile: float = 70.0
) -> Tuple[torch.Tensor, Dict]:
    n_source, n_target = T.shape
    T_safe = T.clamp(min=1e-10)
    entropy = -(T * torch.log(T_safe)).sum(dim=1)
    max_probs = T.max(dim=1)[0]
    entropy_threshold = torch.quantile(entropy, entropy_percentile / 100.0)
    confidence_threshold = torch.quantile(max_probs, confidence_percentile / 100.0)
    low_entropy_mask = entropy <= entropy_threshold
    high_confidence_mask = max_probs >= confidence_threshold
    focused_mask = low_entropy_mask & high_confidence_mask
    n_focused = focused_mask.sum().item()
    n_total = len(focused_mask)
    stats = {
        'n_focused': n_focused,
        'n_total': n_total,
        'pct_focused': 100 * n_focused / n_total if n_total > 0 else 0.0,
        'entropy_threshold': entropy_threshold.item(),
        'confidence_threshold': confidence_threshold.item(),
        'avg_entropy': entropy[focused_mask].mean().item() if n_focused > 0 else 0.0,
        'avg_confidence': max_probs[focused_mask].mean().item() if n_focused > 0 else 0.0
    }
    return focused_mask, stats

def aggregate_training_data_from_batches(
    batch_results: List[Dict],
    features_source_all: torch.Tensor,
    features_target_all: torch.Tensor,
    entropy_percentile: float = 30.0,
    confidence_percentile: float = 70.0
) -> Tuple[torch.Tensor, torch.Tensor, Dict]:
    all_source_features = []
    all_target_features = []
    total_focused = 0
    total_cells = 0
    batch_stats = []

    for batch_idx, result in enumerate(batch_results):
        T = result['T']
        source_indices = result['source_indices']
        target_indices = result['target_indices']

        focused_mask = result.get('focused_mask', None)
        if focused_mask is not None:
            n_focused = focused_mask.sum().item()
            n_total = len(focused_mask)
            stats = {'n_focused': n_focused, 'n_total': n_total, 'pct_focused': 100 * n_focused / n_total if n_total > 0 else 0.0, 'precomputed': True}
        else:
            focused_mask, stats = select_focused_cells(T, entropy_percentile, confidence_percentile)
            stats['precomputed'] = False
            n_focused = focused_mask.sum().item()

        if n_focused == 0:
            batch_stats.append(stats)
            continue

        focused_local_indices = torch.where(focused_mask)[0]
        focused_global_indices = source_indices[focused_local_indices.cpu().numpy()]

        device = features_source_all.device
        focused_global_idx_tensor = torch.from_numpy(focused_global_indices).long().to(device)
        source_features_batch = features_source_all[focused_global_idx_tensor]

        best_targets_local = T.argmax(dim=1)[focused_mask]
        best_targets_global = target_indices[best_targets_local.cpu().numpy()]

        best_targets_global_tensor = torch.from_numpy(best_targets_global).long().to(device)
        target_features_batch = features_target_all[best_targets_global_tensor]

        all_source_features.append(source_features_batch)
        all_target_features.append(target_features_batch)

        total_focused += n_focused
        total_cells += T.shape[0]
        batch_stats.append(stats)

    if len(all_source_features) == 0:
        raise ValueError("No focused cells found in any batch! Cannot train model.")

    source_features_aggregated = torch.cat(all_source_features, dim=0)
    target_features_aggregated = torch.cat(all_target_features, dim=0)

    aggregation_stats = {
        'total_focused': total_focused,
        'total_cells': total_cells,
        'pct_focused': 100 * total_focused / total_cells if total_cells > 0 else 0.0,
        'n_batches_used': len(all_source_features),
        'n_batches_total': len(batch_results),
        'batch_stats': batch_stats
    }
    return source_features_aggregated, target_features_aggregated, aggregation_stats

def train_global_model(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    source_features: torch.Tensor,
    target_features: torch.Tensor,
    steps_per_iter: int,
    lambda_cross: float,
    lambda_var: float,
    metric: str,
    device: torch.device,
    features_target_all: Optional[torch.Tensor] = None
) -> List[float]:
    """
    Train global transformation model (Đã gỡ bỏ hoàn toàn Entropy).
    """
    model.train()
    torch.set_grad_enabled(True)

    step_losses = []

    # ===== FEATURE STANDARDIZATION =====
    feature_mean = None
    feature_std = None
    if metric == 'euclidean':
        if features_target_all is not None:
            feature_mean, feature_std = compute_feature_scaler(features_target_all)
        else:
            feature_mean, feature_std = compute_feature_scaler(target_features)

        source_features = standardize_features(source_features, feature_mean, feature_std)
        target_features = standardize_features(target_features, feature_mean, feature_std)

    logging.info("  [M-step] Chạy Adam Optimizer...")
    for step in range(steps_per_iter):
        if hasattr(model, 'update_temperature'):
            model.update_temperature(current_step=step, max_steps=steps_per_iter)
        optimizer.zero_grad()

        # Forward pass
        source_transformed = model(source_features)
        
        # ===== Loss 1: Cross-domain alignment =====
        if metric == 'cosine':
            source_norm = F.normalize(source_transformed, p=2, dim=1)
            target_norm = F.normalize(target_features, p=2, dim=1)
            loss_cross = (1.0 - (source_norm * target_norm).sum(dim=1)).mean()
        else:  
            loss_cross = F.mse_loss(source_transformed, target_features)
        
        # ===== Loss 2: Variance preservation =====
        loss_var = (source_transformed.std(dim=0) - target_features.std(dim=0)).abs().mean()

        # ===== Combined loss (Thuần túy, không có hàm phạt) =====
        loss = lambda_cross * loss_cross + lambda_var * loss_var

        if not torch.isfinite(loss):
            logging.error(f"Non-finite loss at step {step}: loss={loss.item()}")
            break

        loss.backward()
        optimizer.step()

        step_losses.append(loss.item())

        if (step + 1) % 10 == 0 or step == 0:
            logging.info(f"  Step {step+1}/{steps_per_iter}: loss={loss.item():.6f} (cross={loss_cross.item():.4f}, var={loss_var.item():.4f})")

    if step_losses:
        logging.info(f"M-step completed: {len(step_losses)} steps, avg loss = {np.mean(step_losses):.6f}")

    return step_losses, feature_mean, feature_std

def apply_transfer_method(
    batch_results: List[Dict],
    features_target_all: torch.Tensor,
    n_source_total: int,
    device: torch.device
) -> torch.Tensor:
    n_target = features_target_all.shape[0]
    T_full = torch.zeros(n_source_total, n_target, device=device, dtype=torch.float32)

    for result in batch_results:
        T = result['T']
        source_indices = result['source_indices']
        target_indices = result['target_indices']

        source_idx_tensor = torch.from_numpy(source_indices).long().to(device)
        target_idx_tensor = torch.from_numpy(target_indices).long().to(device)

        T_full[source_idx_tensor[:, None], target_idx_tensor] = T

    features_source_transformed = T_full @ features_target_all

    nnz = (T_full > 1e-6).sum().item()
    nnz_pct = 100 * nnz / T_full.numel()
    logging.info(f"Transfer method: T_full nnz = {nnz}/{T_full.numel()} ({nnz_pct:.2f}%)")
    logging.info(f"Transfer method: Transformed {n_source_total} source cells")

    return features_source_transformed

In [ ]:
%%writefile /kaggle/working/usher/model_utils.py
"""
Model utilities for saving, loading, and applying alignment models.

This module provides utilities to save trained alignment models to disk and apply them to new data.

Usage Example
-------------
# 1. Train and save model
from spot_alignment_sinkhorn_features_only import align_features_sinkhorn
from model_utils import save_alignment_model, load_alignment_model, apply_alignment_model

model, mapping, concat_adata, T_full = align_features_sinkhorn(
    adata_source, adata_target,
    e_step_method='fgw',
    ...
)
save_alignment_model(model, 'alignment_model.pt')

# 2. Load and apply to new data
model = load_alignment_model('alignment_model.pt', device='cuda')

# IMPORTANT: New data MUST be in same PCA/feature space as training data!
adata_new_transformed = apply_alignment_model(
    model,
    adata_new_pca,  # Must have same n_features as training
    normalize_output=False,  # Set True for L2-normalized output
    device='cuda'
)

Notes
-----
- Model expects RAW features (same preprocessing as training)
- If training used PCA, new data must be projected to SAME PCA space
- L2-normalization is optional and done AFTER transformation
"""

import logging
from typing import Optional
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import anndata as ad


class FeatureTransform(nn.Module):
    """Feature transform with optional residual connection.

    - If hidden_dim is None: Simple linear transformation W(x) = Ax + b
    - If hidden_dim is set: Non-linear MLP with residual connection (when input_dim == output_dim)

    Residual connection learns the DIFFERENCE: x + ΔW(x) = target
    This is better for batch correction where source and target are similar.
    """

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: Optional[int] = None, dropout: float = 0.0):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.use_residual = False

        if hidden_dim is None:
            # Simple linear transformation
            self.net = nn.Linear(input_dim, output_dim, bias=True)
        else:
            # Non-linear MLP: input -> hidden -> output
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, output_dim),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_residual:
            # Residual connection: learn the difference/correction
            # x + ΔW(x) where ΔW is the learned correction
            return x + self.net(x)
        else:
            # Standard: learn full transformation
            return self.net(x)

    def init_identity(self):
        """
        Initialize linear layer weights to identity matrix (diagonal ones) and bias to zero.

        For linear layer:
        - If input_dim == output_dim: Initialize W as identity matrix (diagonal ones)
        - If input_dim != output_dim: Initialize W as identity for min(input_dim, output_dim), rest as zeros
        - Bias is always initialized to zero

        For MLP:
        - Initialize the LAST linear layer to identity (if possible)
        - This makes the initial transformation close to identity

        This is useful when source and target are already similar and we want to learn
        small corrections rather than a completely new transformation.
        """
        with torch.no_grad():
            if isinstance(self.net, nn.Linear):
                # Simple linear layer
                weight = self.net.weight  # (output_dim, input_dim)
                bias = self.net.bias

                # Initialize to identity (diagonal ones)
                nn.init.zeros_(weight)
                min_dim = min(self.input_dim, self.output_dim)
                for i in range(min_dim):
                    weight[i, i] = 1.0

                # Zero bias
                nn.init.zeros_(bias)

                logging.info(f"Initialized linear layer to identity (W: {weight.shape}, diagonal ones for min_dim={min_dim})")

            elif isinstance(self.net, nn.Sequential):
                # MLP: Initialize the LAST linear layer to identity
                last_linear = None
                for m in reversed(list(self.net.modules())):
                    if isinstance(m, nn.Linear):
                        last_linear = m
                        break

                if last_linear is not None:
                    weight = last_linear.weight  # (output_dim, hidden_dim)
                    bias = last_linear.bias

                    # Initialize to identity (if square) or zeros
                    nn.init.zeros_(weight)
                    min_dim = min(weight.shape[0], weight.shape[1])
                    for i in range(min_dim):
                        weight[i, i] = 1.0

                    # Zero bias
                    nn.init.zeros_(bias)

                    logging.info(f"Initialized MLP last layer to identity (W: {weight.shape}, diagonal ones for min_dim={min_dim})")
                else:
                    logging.warning("No linear layer found in MLP for identity initialization")

    def init_random(self):
        """Initialize with Xavier/Kaiming random initialization.

        Better for when source and target have different distributions,
        such as in spatial alignment tasks.
        """
        with torch.no_grad():
            if isinstance(self.net, nn.Linear):
                # Xavier uniform initialization
                nn.init.xavier_uniform_(self.net.weight)
                nn.init.zeros_(self.net.bias)
                logging.info(f"Initialized linear layer with Xavier uniform: {self.net.weight.shape}")

            elif isinstance(self.net, nn.Sequential):
                # Initialize all linear layers in MLP
                for m in self.net.modules():
                    if isinstance(m, nn.Linear):
                        nn.init.xavier_uniform_(m.weight)
                        if m.bias is not None:
                            nn.init.zeros_(m.bias)
                logging.info(f"Initialized MLP layers with Xavier uniform")


def _to_dense_float32(x) -> np.ndarray:
    """Convert sparse or dense array to dense float32."""
    if hasattr(x, "toarray"):
        return x.toarray().astype(np.float32)
    return np.asarray(x, dtype=np.float32)


def save_alignment_model(
    model: FeatureTransform,
    save_path: str,
    feature_mean: Optional[torch.Tensor] = None,
    feature_std: Optional[torch.Tensor] = None,
    gene_names: Optional[list] = None
):
    """
    Save trained alignment model to disk.

    Parameters
    ----------
    model : FeatureTransform
        Trained model from align_features_sinkhorn
    save_path : str
        Path to save the model (e.g., 'model.pt')
    feature_mean : torch.Tensor, optional
        Mean per feature column used for standardization (from align_features_sinkhorn)
        None if cosine metric was used
    feature_std : torch.Tensor, optional
        Std per feature column used for standardization (from align_features_sinkhorn)
        None if cosine metric was used
    """
    save_dict = {
        'model_state_dict': model.state_dict(),
        'input_dim': model.input_dim,
        'output_dim': model.output_dim,
        'use_residual': model.use_residual,
        'feature_mean': feature_mean.cpu() if feature_mean is not None else None,
        'feature_std': feature_std.cpu() if feature_std is not None else None,
        'gene_names': gene_names,
    }

    torch.save(save_dict, save_path)

    if feature_mean is not None and feature_std is not None:
        logging.info(f"Model saved to {save_path} (with feature scaler)")
    else:
        logging.info(f"Model saved to {save_path} (no scaler - cosine metric)")


def load_alignment_model(load_path: str, device: Optional[str] = None, hidden_dim: Optional[int] = None):
    """
    Load trained alignment model from disk.

    Parameters
    ----------
    load_path : str
        Path to saved model (e.g., 'model.pt')
    device : str, optional
        Device to load model to ('cuda' or 'mps')

    Returns
    -------
    model : FeatureTransform
        Loaded model ready for inference
    feature_mean : torch.Tensor or None
        Mean per feature column for standardization (None if not used)
    feature_std : torch.Tensor or None
        Std per feature column for standardization (None if not used)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "mps"

    checkpoint = torch.load(load_path, map_location=device)

    # Reconstruct model architecture
    model = FeatureTransform(
        input_dim=checkpoint['input_dim'],
        output_dim=checkpoint['output_dim'],
        hidden_dim=hidden_dim  # Will be inferred from state_dict
    )

    # Load weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    # Load scaler (if present)
    feature_mean = checkpoint.get('feature_mean', None)
    feature_std = checkpoint.get('feature_std', None)
    gene_names = checkpoint.get('gene_names', None)

    if feature_mean is not None and feature_std is not None:
        feature_mean = feature_mean.to(device)
        feature_std = feature_std.to(device)
        logging.info(f"Model loaded from {load_path} to {device} (with feature scaler)")
    else:
        logging.info(f"Model loaded from {load_path} to {device} (no scaler - cosine metric)")

    return model, feature_mean, feature_std, gene_names


def apply_alignment_model(
    model: FeatureTransform,
    adata_new: ad.AnnData,
    feature_mean: Optional[torch.Tensor] = None,
    feature_std: Optional[torch.Tensor] = None,
    normalize_output: bool = False,
    device: Optional[str] = None,
    gene_names: Optional[list] = None
) -> ad.AnnData:
    """
    Apply trained alignment model to new data.

    Parameters
    ----------
    model : FeatureTransform
        Trained model from align_features_sinkhorn or load_alignment_model
    adata_new : ad.AnnData
        New data to transform (MUST be in same space as training data)
        - Same PCA/feature space (same number of features)
        - Same preprocessing as training data
    feature_mean : torch.Tensor, optional
        Mean per feature column for standardization (from load_alignment_model)
        None if model was trained with cosine metric
    feature_std : torch.Tensor, optional
        Std per feature column for standardization (from load_alignment_model)
        None if model was trained with cosine metric
    normalize_output : bool
        Whether to L2-normalize the output (default: False)
        Set to True if you need normalized features for downstream distance computation
    device : str, optional
        Device to use ('cuda' or 'mps')

    Returns
    -------
    adata_transformed : ad.AnnData
        AnnData with transformed features in .X

    Notes
    -----
    - Input data (adata_new.X) should be RAW features (same as training)
    - If training data was PCA, new data should be projected to same PCs
    - If feature_mean/feature_std are provided (euclidean metric):
      1. Input is standardized
      2. Model is applied in standardized space
      3. Output is unstandardized back to original scale
    - If feature_mean/feature_std are None (cosine metric):
      Model is applied directly without standardization

    Example
    -------
    >>> # Load model with scaler
    >>> model, feature_mean, feature_std = load_alignment_model('alignment_model.pt', device='cuda')
    >>>
    >>> # Apply to new data (must be in same PCA space!)
    >>> adata_new_transformed = apply_alignment_model(
    ...     model,
    ...     adata_new_pca,
    ...     feature_mean=feature_mean,
    ...     feature_std=feature_std,
    ...     normalize_output=False,
    ...     device='cuda'
    ... )
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "mps"

    model.to(device)
    model.eval()

    # Move scaler to device if provided
    if feature_mean is not None:
        feature_mean = feature_mean.to(device)
    if feature_std is not None:
        feature_std = feature_std.to(device)

    # Load and convert data
    X_new = _to_dense_float32(adata_new.X)
    features_new = torch.from_numpy(X_new).to(device)

    # Check dimensions
    if features_new.shape[1] != model.input_dim:
        raise ValueError(
            f"Input dimension mismatch: model expects {model.input_dim} features, "
            f"but data has {features_new.shape[1]} features. "
            f"Ensure new data is in the same PCA/feature space as training data."
        )

    # Transform with optional standardization
    with torch.no_grad():
        if feature_mean is not None and feature_std is not None:
            # Euclidean metric: standardize → transform → unstandardize
            from m_step_utils import standardize_features, unstandardize_features

            features_std = standardize_features(features_new, feature_mean, feature_std)
            features_transformed_std = model(features_std)
            features_transformed = unstandardize_features(features_transformed_std, feature_mean, feature_std)

            logging.info("Applied model with feature standardization (euclidean metric)")
        else:
            # Cosine metric: direct transformation
            features_transformed = model(features_new)
            logging.info("Applied model without standardization (cosine metric)")

        # Optional L2 normalization
        if normalize_output:
            features_transformed = F.normalize(features_transformed, p=2, dim=1)

    # Create output AnnData
    adata_transformed = ad.AnnData(
        X=features_transformed.cpu().numpy(),
        obs=adata_new.obs.copy(),
        var=pd.DataFrame(index=[f'feature_{i}' for i in range(features_transformed.shape[1])])
    )

    logging.info(f"Transformed {features_new.shape[0]} cells: {features_new.shape[1]} → {features_transformed.shape[1]} features")
    if normalize_output:
        logging.info("  Output L2-normalized")

    return adata_transformed

In [ ]:
%%writefile /kaggle/working/usher/graph_utils.py
"""
Graph utilities for spatial alignment.
"""

import logging
from typing import Optional
import numpy as np
import torch
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path
from sklearn.neighbors import NearestNeighbors

def compute_knn_graph_distance(
    features: torch.Tensor,
    k: int = 30,
    metric: str = 'cosine',
    device: Optional[str] = None,
) -> torch.Tensor:
    if device is None:
        device = features.device

    features_np = features.cpu().numpy()
    n_samples = features_np.shape[0]
    k = min(k, n_samples - 1)

    features_norm = features_np / (np.linalg.norm(features_np, axis=1, keepdims=True) + 1e-8)
    if metric == 'cosine':
        nbrs = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute').fit(features_norm)
        distances, indices = nbrs.kneighbors(features_norm)
    else:
        nbrs = NearestNeighbors(n_neighbors=k+1, metric=metric, algorithm='auto').fit(features_np)
        distances, indices = nbrs.kneighbors(features_np)

    row_indices = np.repeat(np.arange(n_samples), k)
    col_indices = indices[:, 1:].flatten()
    edge_weights = np.ones(len(row_indices), dtype=np.float32)

    adjacency_matrix = csr_matrix((edge_weights, (row_indices, col_indices)), shape=(n_samples, n_samples))
    adjacency_matrix = adjacency_matrix.maximum(adjacency_matrix.T)

    try:
        from sklearn.cluster import KMeans
        from sklearn.metrics import pairwise_distances_argmin

        num_landmarks = min(512, n_samples)
        
        if n_samples > num_landmarks * 10:
            num_candidates = num_landmarks * 10
            candidate_indices = np.random.choice(n_samples, num_candidates, replace=False)
            candidate_features = features_np[candidate_indices]
            
            kmeans = KMeans(n_clusters=num_landmarks, n_init=1, random_state=42)
            kmeans.fit(candidate_features)
            
            closest_in_candidates = pairwise_distances_argmin(kmeans.cluster_centers_, candidate_features)
            landmarks = candidate_indices[closest_in_candidates]
        else:
            landmarks = np.random.choice(n_samples, num_landmarks, replace=False)

        landmark_dists = shortest_path(
            adjacency_matrix,
            directed=False,
            indices=landmarks,
            unweighted=True
        )

        Max_dist = np.nanmax(landmark_dists[landmark_dists != np.inf])
        if np.isnan(Max_dist): Max_dist = 1.0
        landmark_dists[np.isinf(landmark_dists)] = Max_dist
        
        L_dist = torch.tensor(landmark_dists.T, dtype=torch.float32, device=device)

        beta = 2.0  
        E = torch.exp(-beta * L_dist)
        M = torch.mm(E, E.t())
        M = torch.clamp(M, min=1e-30) 
        dist_matrix = - (1.0 / beta) * torch.log(M)
        
    except Exception as e:
        F = torch.tensor(features_norm, dtype=torch.float32, device=device)
        dist_matrix = 1.0 - torch.mm(F, F.t())
        dist_matrix = torch.clamp(dist_matrix, min=0.0)

    dist_matrix.fill_diagonal_(0.0)
    dist_matrix = dist_matrix / (dist_matrix.max() + 1e-8)

    return dist_matrix

def apply_cell_type_constraints(
    T: torch.Tensor,
    cell_types_source: np.ndarray,
    cell_types_target: np.ndarray,
    device: torch.device
) -> torch.Tensor:
    n_source, n_target = T.shape
    cell_types_source_expanded = np.tile(cell_types_source.reshape(-1, 1), (1, n_target))
    cell_types_target_expanded = np.tile(cell_types_target.reshape(1, -1), (n_source, 1))
    same_type_mask = (cell_types_source_expanded == cell_types_target_expanded)
    same_type_mask_tensor = torch.from_numpy(same_type_mask).to(device)

    T_constrained = T * same_type_mask_tensor
    row_sums_original = T.sum(dim=1, keepdim=True)
    row_sums_constrained = T_constrained.sum(dim=1, keepdim=True)
    row_sums_constrained = row_sums_constrained.clamp(min=1e-10)

    T_constrained = T_constrained * (row_sums_original / row_sums_constrained)
    return T_constrained

In [ ]:
%%writefile /kaggle/working/usher/celltype_utils.py
"""
Celltype-based sampling utilities for optimal transport alignment.

This module handles celltype probability loading and geosketch-based batching
strategies for the alignment process.
"""

import logging
import numpy as np
import anndata as ad
from typing import List, Tuple, Optional, Dict, Any
from gs_utils import geosketch_subsample, geosketch_target_batches, geosketch_stratified_pairing

import pandas as pd

def _load_celltype_probs(
    adata: ad.AnnData,
    celltype_probs_layer: str,
    cell_type_col: str,
    subset_indices: Optional[np.ndarray] = None
) -> np.ndarray:
    """
    Load cell type probabilities from AnnData object.

    Tries to load from adata.obsm[celltype_probs_layer] first.
    If not present, creates one-hot encoding from adata.obs[cell_type_col].

    Parameters
    ----------
    adata : ad.AnnData
        AnnData object containing cell type information
    celltype_probs_layer : str
        Key in .obsm to look for cell type probabilities
    cell_type_col : str
        Column in .obs to use for one-hot encoding if celltype_probs_layer not found
    subset_indices : np.ndarray, optional
        Indices to subset the cell type data (after loading)

    Returns
    -------
    celltype_probs : np.ndarray
        Cell type probabilities matrix (n_cells, n_celltypes)
    """
    if celltype_probs_layer in adata.obsm:
        # Load from obsm (DataFrame or array)
        celltype_data = adata.obsm[celltype_probs_layer]
        if isinstance(celltype_data, pd.DataFrame):
            celltype_probs = celltype_data.values
        else:
            celltype_probs = np.array(celltype_data)
        logging.info(f"Loaded cell type probabilities from .obsm['{celltype_probs_layer}']: shape {celltype_probs.shape}")
    else:
        # One-hot encode from categorical column
        logging.info(f"Cell type probabilities not found in .obsm['{celltype_probs_layer}'], one-hot encoding from .obs['{cell_type_col}']")
        if cell_type_col not in adata.obs:
            raise ValueError(f"Cell type column '{cell_type_col}' not found in .obs and '{celltype_probs_layer}' not found in .obsm")

        # Get cell types and create one-hot encoding
        cell_types = adata.obs[cell_type_col].values
        unique_types = np.unique(cell_types)
        n_types = len(unique_types)
        type_to_idx = {ct: i for i, ct in enumerate(unique_types)}

        celltype_probs = np.zeros((len(cell_types), n_types), dtype=np.float32)
        for i, ct in enumerate(cell_types):
            celltype_probs[i, type_to_idx[ct]] = 1.0

        logging.info(f"One-hot encoded cell types: {len(unique_types)} types, shape {celltype_probs.shape}")

    # Subset if requested
    if subset_indices is not None:
        celltype_probs = celltype_probs[subset_indices]
        logging.info(f"Subsetted cell type probabilities: shape {celltype_probs.shape}")

    return celltype_probs



def prepare_celltype_batches(
    adata_a: ad.AnnData,
    adata_b: ad.AnnData,
    sketch_size: int = 1000,
    use_stratified_pairing: bool = False,
    stratified_pairing_fix: bool = True,
    celltype_probs_layer: str = 'X_celltype_probs',
    cell_type_col: Optional[str] = None,
    sketch_obsm_key: str = 'X_umap',
    sketch_pca_components: int = 50,
    e_step_method: str = 'ot',
    seed: int = 2025,
    **kwargs
) -> Tuple[List[Tuple[Optional[np.ndarray], np.ndarray]], Dict[str, Any], Optional[np.ndarray]]:
    """
    Prepare batches for celltype-based sampling strategy.

    Parameters
    ----------
    adata_a : ad.AnnData
        Source dataset
    adata_b : ad.AnnData
        Target dataset
    sketch_size : int
        Group/batch size for sampling
    use_stratified_pairing : bool
        Whether to use stratified pairing (paired source-target groups)
    stratified_pairing_fix : bool
        Whether to use fixed groups across iterations
    celltype_probs_layer : str
        Key in .obsm for cell type probabilities
    cell_type_col : str, optional
        Column name containing cell type information
    sketch_obsm_key : str
        Key in .obsm for embeddings to use for geosketch
    sketch_pca_components : int
        Number of PCA components if sketch_obsm_key not found
    e_step_method : str
        E-step method ('ot', 'gw', or 'fgw') - determines if celltype probs are loaded
    seed : int
        Random seed for reproducibility
    **kwargs
        Additional parameters (ignored)

    Returns
    -------
    batches : List[Tuple[Optional[np.ndarray], np.ndarray]]
        List of (source_indices, target_indices) pairs
    auxiliary_data : Dict[str, Any]
        Dictionary containing:
        - celltype_probs_a: Source celltype probabilities (if applicable)
        - celltype_probs_b: Target celltype probabilities (if applicable)
        - source_groups: Source group indices (if stratified pairing)
        - target_groups: Target group indices (if stratified pairing)
        - n_source: Number of source cells after sampling
        - n_target: Number of target cells
    sketch_to_original : np.ndarray or None
        Mapping from sketched to original indices for source (if subsampled)
    """
    n_a_orig = len(adata_a)
    n_b_orig = len(adata_b)

    # Load celltype probabilities if needed for OT or FGW methods
    celltype_probs_a = None
    celltype_probs_b = None
    if e_step_method in ['ot', 'fgw']:
        logging.info(f"Loading cell type probabilities for {e_step_method} method...")
        celltype_probs_a = _load_celltype_probs(adata_a, celltype_probs_layer, cell_type_col)
        celltype_probs_b = _load_celltype_probs(adata_b, celltype_probs_layer, cell_type_col)

        if celltype_probs_a is not None:
            logging.info(f"Source cell type probs: {celltype_probs_a.shape}")
        if celltype_probs_b is not None:
            logging.info(f"Target cell type probs: {celltype_probs_b.shape}")

    batches = []
    sketch_to_original = None
    source_groups = None
    target_groups = None

    if use_stratified_pairing:
        # STRATIFIED PAIRING: Geosketch both source and target into k paired groups
        logging.info(f"=== Using STRATIFIED PAIRING sampling strategy ===")
        logging.info(f"Sketch size: {sketch_size}, Fix: {stratified_pairing_fix}")

        source_groups, target_groups = geosketch_stratified_pairing(
            adata_a, adata_b,
            sketch_size=sketch_size,
            obsm_key=sketch_obsm_key,
            pca_components=sketch_pca_components,
            seed=seed,
            fix=stratified_pairing_fix
        )

        logging.info(f"Stratified pairing: {len(source_groups)} paired groups created")

        # Create batch pairs
        batches = list(zip(source_groups, target_groups))
        n_source = n_a_orig  # Using full source dataset

    else:
        # ORIGINAL APPROACH: Sample source once, batch target
        logging.info(f"=== Using ORIGINAL sampling strategy ===")
        logging.info(f"Source sketch size: {sketch_size}, Target batch size: {sketch_size}")

        # 1. Sample source (reference) to reduce size
        if n_a_orig > sketch_size:
            logging.info(f"Geosketching source from {n_a_orig} to {sketch_size} cells...")
            sketch_to_original = geosketch_subsample(
                adata_a, sketch_size, obsm_key=sketch_obsm_key, pca_components=sketch_pca_components
            )
            n_source = len(sketch_to_original)

            # Subset celltype probabilities for source
            if celltype_probs_a is not None:
                celltype_probs_a = celltype_probs_a[sketch_to_original]
                logging.info(f"Subsetted source cell type probs to {celltype_probs_a.shape}")
        else:
            n_source = n_a_orig
            logging.info(f"No source sampling (size {n_a_orig} <= {sketch_size})")

        # 2. Sample target into batches
        target_batches = geosketch_target_batches(
            adata_b, sketch_size, obsm_key=sketch_obsm_key, pca_components=sketch_pca_components
        )

        logging.info(f"Created {len(target_batches)} target batches")

        # Create batch pairs (None for source means use all source cells)
        batches = [(None, target_batch) for target_batch in target_batches]

    # Prepare auxiliary data
    auxiliary_data = {
        'celltype_probs_a': celltype_probs_a,
        'celltype_probs_b': celltype_probs_b,
        'source_groups': source_groups,
        'target_groups': target_groups,
        'n_source': n_source,
        'n_target': n_b_orig
    }

    return batches, auxiliary_data, sketch_to_original

In [ ]:
%%writefile /kaggle/working/usher/gs_utils.py
"""
Geosketch utilities for spatial alignment.

This module provides geosketch-based sampling functions for source and target datasets.
"""

import logging
from typing import List, Tuple, Optional
import numpy as np
import torch
import anndata as ad
from sklearn.decomposition import PCA


def geosketch_subsample(
    adata: ad.AnnData,
    sketch_size: int,
    obsm_key: str = 'X_umap',
    pca_components: int = 50,
    seed: int = 42
) -> np.ndarray:
    """
    Geosketch subsample large dataset.

    Parameters
    ----------
    adata : ad.AnnData
        AnnData object containing the dataset
    sketch_size : int
        Target number of cells to sketch
    obsm_key : str
        Key in .obsm for pre-computed embeddings (default: 'X_umap')
        If not found, will compute PCA using pca_components
    pca_components : int
        Number of PCA components to compute if obsm_key not found (default: 50)
    seed : int
        Random seed for geosketch

    Returns
    -------
    sketched_indices : np.ndarray
        Indices of sketched cells in original data
    """
    # Import gs function from standalone module
    from geosketch_standalone import gs

    # Get embeddings for geosketch
    if obsm_key in adata.obsm.keys():
        features_for_sketch = adata.obsm[obsm_key]
        logging.info(f"Using pre-computed embeddings from adata.obsm['{obsm_key}'] for geosketch: {features_for_sketch.shape}")
    else:
        # Fallback to PCA
        logging.warning(f"Key '{obsm_key}' not found in adata.obsm, computing PCA with {pca_components} components")
        X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
        pca = PCA(n_components=min(pca_components, X.shape[1]))
        features_for_sketch = pca.fit_transform(X)
        logging.info(f"Computed PCA for geosketch: {X.shape} -> {features_for_sketch.shape}")

    # Geosketch on embedding space
    sketched_indices = gs(features_for_sketch, N=sketch_size, seed=seed)

    return np.array(sketched_indices)


def geosketch_target_batches(
    adata: ad.AnnData,
    batch_size: int,
    obsm_key: str = 'X_umap',
    pca_components: int = 50,
    seed: int = 42
) -> List[np.ndarray]:
    """
    Geosketch target into batches for processing.

    Parameters
    ----------
    adata : ad.AnnData
        AnnData object containing the target dataset
    batch_size : int
        Target batch size
    obsm_key : str
        Key in .obsm for pre-computed embeddings (default: 'X_umap')
        If not found, will compute PCA using pca_components
    pca_components : int
        Number of PCA components to compute if obsm_key not found (default: 50)
    seed : int
        Random seed for geosketch

    Returns
    -------
    batch_indices_list : List[np.ndarray]
        List of indices for each batch
    """
    # Import gs function from standalone module
    from geosketch_standalone import gs

    n_target = adata.n_obs

    if n_target <= batch_size:
        # Target is small enough, return single batch
        return [np.arange(n_target)]

    # Get embeddings for geosketch
    if obsm_key in adata.obsm.keys():
        features_for_sketch = adata.obsm[obsm_key]
        logging.info(f"Using pre-computed embeddings from adata.obsm['{obsm_key}'] for target batching: {features_for_sketch.shape}")
    else:
        # Fallback to PCA
        logging.warning(f"Key '{obsm_key}' not found in adata.obsm, computing PCA with {pca_components} components")
        X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
        pca = PCA(n_components=min(pca_components, X.shape[1]))
        features_for_sketch = pca.fit_transform(X)
        logging.info(f"Computed PCA for target batching: {X.shape} -> {features_for_sketch.shape}")

    # Calculate number of batches needed
    n_batches = (n_target + batch_size - 1) // batch_size

    # Geosketch to get representative indices for each batch
    batch_indices_list = []
    remaining_indices = np.arange(n_target)

    for batch_idx in range(n_batches):
        if len(remaining_indices) <= batch_size:
            # Last batch: take all remaining
            batch_indices = remaining_indices
        else:
            # Geosketch from remaining indices
            remaining_features_for_sketch = features_for_sketch[remaining_indices]
            batch_size_actual = min(batch_size, len(remaining_indices))

            sketched_local = gs(remaining_features_for_sketch, N=batch_size_actual, seed=seed + batch_idx)
            batch_indices = remaining_indices[sketched_local]

        batch_indices_list.append(batch_indices)
        remaining_indices = np.setdiff1d(remaining_indices, batch_indices)

    return batch_indices_list


def geosketch_stratified_pairing(
    adata_a: ad.AnnData,
    adata_b: ad.AnnData,
    sketch_size: int,
    obsm_key: str = 'X_umap',
    pca_components: int = 50,
    seed: int = 42,
    fix: bool = True
) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    """
    Geosketch both source and target into k groups, then pair them randomly.

    Uses stratified sampling without replacement: iteratively geosketch k groups
    from source and target (removing sampled cells each iteration), then pair them
    as (source_group[i], target_group[i]) for i=0..k-1.

    Groups with fewer than sketch_size cells are dropped for BOTH source and target
    to maintain balanced pairing.

    Parameters
    ----------
    adata_a : ad.AnnData
        Source AnnData object
    adata_b : ad.AnnData
        Target AnnData object
    sketch_size : int
        Number of cells per group (same for source and target)
    obsm_key : str
        Key in .obsm containing pre-computed embeddings (e.g., 'X_umap', 'X_pca')
        If not found, will compute PCA using pca_components
    pca_components : int
        Number of PCA components to compute if obsm_key not found (default: 50)
    seed : int
        Random seed for geosketch
        - If fix=True: Use this seed directly (same groups across calls)
        - If fix=False: Use iteration-dependent seed (e.g., seed=base_seed+iteration)
    fix : bool
        If True, use fixed seed (same groups across calls with same seed)
        If False, caller should pass iteration-dependent seed for rotating groups

    Returns
    -------
    source_groups : List[np.ndarray]
        List of k source cell indices for each group (each has sketch_size cells)
    target_groups : List[np.ndarray]
        List of k target cell indices for each group (each has sketch_size cells)
        Paired with source_groups: align source_groups[i] with target_groups[i]

    Examples
    --------
    # Fixed groups (same across iterations):
    >>> source_groups, target_groups = geosketch_stratified_pairing(
    ...     adata_a, adata_b, sketch_size=1000, obsm_key='X_umap', seed=42, fix=True
    ... )

    # Rotating groups (different each iteration):
    >>> for iteration in range(n_iters):
    ...     source_groups, target_groups = geosketch_stratified_pairing(
    ...         adata_a, adata_b, sketch_size=1000, obsm_key='X_umap',
    ...         seed=42 + iteration, fix=False
    ...     )
    """
    from geosketch_standalone import gs

    n_source = adata_a.n_obs
    n_target = adata_b.n_obs

    # Calculate max possible k (number of complete groups)
    k_source_max = n_source // sketch_size
    k_target_max = n_target // sketch_size
    k = min(k_source_max, k_target_max)

    if k == 0:
        logging.warning(f"Cannot create any complete groups: n_source={n_source}, n_target={n_target}, sketch_size={sketch_size}")
        return [], []

    logging.info(f"Stratified pairing: Creating {k} paired groups (sketch_size={sketch_size}, fix={fix})")
    logging.info(f"  Source: {n_source} cells -> {k} groups of {sketch_size} cells ({k*sketch_size}/{n_source} used, {n_source - k*sketch_size} dropped)")
    logging.info(f"  Target: {n_target} cells -> {k} groups of {sketch_size} cells ({k*sketch_size}/{n_target} used, {n_target - k*sketch_size} dropped)")

    # Get embeddings for geosketch
    # Try to use pre-computed embeddings from .obsm
    if obsm_key in adata_a.obsm.keys():
        features_a_for_sketch = adata_a.obsm[obsm_key]
        logging.info(f"Using pre-computed embeddings from adata_a.obsm['{obsm_key}'] for source: {features_a_for_sketch.shape}")
    else:
        # Fallback to PCA
        logging.warning(f"Key '{obsm_key}' not found in adata_a.obsm, computing PCA with {pca_components} components")
        from sklearn.decomposition import PCA
        X_a = adata_a.X.toarray() if hasattr(adata_a.X, 'toarray') else adata_a.X
        pca_a = PCA(n_components=min(pca_components, X_a.shape[1]))
        features_a_for_sketch = pca_a.fit_transform(X_a)
        logging.info(f"Computed PCA for source: {X_a.shape} -> {features_a_for_sketch.shape}")

    if obsm_key in adata_b.obsm.keys():
        features_b_for_sketch = adata_b.obsm[obsm_key]
        logging.info(f"Using pre-computed embeddings from adata_b.obsm['{obsm_key}'] for target: {features_b_for_sketch.shape}")
    else:
        # Fallback to PCA
        logging.warning(f"Key '{obsm_key}' not found in adata_b.obsm, computing PCA with {pca_components} components")
        from sklearn.decomposition import PCA
        X_b = adata_b.X.toarray() if hasattr(adata_b.X, 'toarray') else adata_b.X
        pca_b = PCA(n_components=min(pca_components, X_b.shape[1]))
        features_b_for_sketch = pca_b.fit_transform(X_b)
        logging.info(f"Computed PCA for target: {X_b.shape} -> {features_b_for_sketch.shape}")

    # Stratified sampling: iteratively geosketch k groups from source
    source_groups = []
    remaining_source_indices = np.arange(n_source)

    for group_idx in range(k):
        # Geosketch from remaining source cells
        remaining_source_features = features_a_for_sketch[remaining_source_indices]

        # Use different seed for each group to ensure diversity
        group_seed = seed + group_idx if not fix else seed + group_idx
        sketched_local = gs(remaining_source_features, N=sketch_size, seed=group_seed)

        # Convert local indices to global indices
        group_indices = remaining_source_indices[sketched_local]
        source_groups.append(group_indices)

        # Remove sketched cells from remaining pool
        remaining_source_indices = np.setdiff1d(remaining_source_indices, group_indices)

        logging.info(f"  Source group {group_idx + 1}/{k}: sketched {len(group_indices)} cells, {len(remaining_source_indices)} remaining")

    # Stratified sampling: iteratively geosketch k groups from target
    target_groups = []
    remaining_target_indices = np.arange(n_target)

    for group_idx in range(k):
        # Geosketch from remaining target cells
        remaining_target_features = features_b_for_sketch[remaining_target_indices]

        # Use different seed offset for target to avoid correlated sampling
        group_seed = seed + k + group_idx if not fix else seed + k + group_idx
        sketched_local = gs(remaining_target_features, N=sketch_size, seed=group_seed)

        # Convert local indices to global indices
        group_indices = remaining_target_indices[sketched_local]
        target_groups.append(group_indices)

        # Remove sketched cells from remaining pool
        remaining_target_indices = np.setdiff1d(remaining_target_indices, group_indices)

        logging.info(f"  Target group {group_idx + 1}/{k}: sketched {len(group_indices)} cells, {len(remaining_target_indices)} remaining")

    # Verify all groups have exactly sketch_size cells
    for i, (src_group, tgt_group) in enumerate(zip(source_groups, target_groups)):
        assert len(src_group) == sketch_size, f"Source group {i} has {len(src_group)} cells, expected {sketch_size}"
        assert len(tgt_group) == sketch_size, f"Target group {i} has {len(tgt_group)} cells, expected {sketch_size}"

    logging.info(f"Stratified pairing complete: {k} paired groups created")

    return source_groups, target_groups

In [ ]:
%%writefile /kaggle/working/usher/geosketch_standalone.py
"""
Standalone geosketch implementation for spot alignment.
Copied essential functions from geosketch/sketch.py to avoid import issues.
"""

import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import pairwise_distances
import sys


def log(message):
    """Simple logging function."""
    print(message)


def gs(X, N, k='auto', seed=None, replace=False,
       alpha=0.1, max_iter=200, one_indexed=False, verbose=0):
    """Sample from a data set according to a geometric plaid covering.

    Parameters
    ----------
    X : `numpy.ndarray`
        Dense vector of low dimensional embeddings with rows corresponding
        to observations and columns corresponding to feature embeddings.
    N: `int`
        Desired sketch size.
    replace: `bool`, optional (default: False)
        When `True`, draws samples with replacement from covering boxes.
    k: `int` or `'auto'` (default: `'auto'`)
        Number of covering boxes.
        When `'auto'` and replace is `True`, draws sqrt(X.shape[0])
        covering boxes.
        When `'auto'` and replace is `False`, draws N covering boxes.
    alpha: `float`
        Binary search halts when it obtains between `k * (1 - alpha)` and
        `k * (1 + alpha)` covering boxes.
    seed: `int`, optional (default: None)
        Random seed passed to numpy.
    max_iter: `int`, optional (default: 200)
        Maximum iterations at which to terminate binary seach in rare
        case of non-monotonicity of covering boxes with box side length.
    one_indexed: `bool`, optional (default: False)
        Returns a 1-indexed result (e.g., R or Matlab indexing), instead
        of a 0-indexed result (e.g., Python or C indexing).
    verbose: `bool` or `int`, optional (default: 0)
        When `True` or not equal to 0, prints logging output.

    Returns
    -------
    samp_idx
        List of indices into X that make up the sketch.
    """
    n_samples, n_features = X.shape

    # Error checking and initialization.
    if not seed is None:
        np.random.seed(seed)
    if not replace and N > n_samples:
        raise ValueError('Cannot sample {} elements from {} elements '
                         'without replacement'.format(N, n_samples))
    if not replace and N == n_samples:
        if one_indexed:
            return list(np.array(range(N)) + 1)
        else:
            return list(range(N))
    if k == 'auto':
        if replace:
            k = int(np.sqrt(n_samples))
        else:
            k = N
    if k < 1:
        raise ValueError('Cannot draw {} covering boxes.'.format(k))

    # Tranlate to make data all positive.
    # Note: `-=' operator mutates variable outside of method.
    X = X - X.min(0)

    # Scale so that maximum value equals 1.
    X /= X.max()

    # Find max value along each dimension.
    X_ptp = np.ptp(X, 0)

    # Range for binary search.
    low_unit, high_unit = 0., max(X_ptp)

    # Initialize box length.
    unit = (low_unit + high_unit) / 4.

    d_to_argsort = {}

    n_iter = 0
    while True:

        if verbose > 1:
            log('n_iter = {}'.format(n_iter))

        grid_table = np.zeros((n_samples, n_features))

        # Assign points to intervals within each dimension.
        for d in range(n_features):
            if X_ptp[d] <= unit:
                continue

            points_d = X[:, d]
            if d not in d_to_argsort:
                d_to_argsort[d] = np.argsort(points_d)
            curr_start = None
            curr_interval = -1
            for sample_idx in d_to_argsort[d]:
                if curr_start is None or \
                   curr_start + unit < points_d[sample_idx]:
                    curr_start = points_d[sample_idx]
                    curr_interval += 1
                grid_table[sample_idx, d] = curr_interval

        # Store as map from grid cells to point indices.
        grid = {}
        for sample_idx in range(n_samples):
            grid_cell = tuple(grid_table[sample_idx, :])
            if grid_cell not in grid:
                grid[grid_cell] = []
            grid[grid_cell].append(sample_idx)
        del grid_table

        if verbose:
            log('Found {} non-empty grid cells'.format(len(grid)))

        if len(grid) > k * (1 + alpha):
            # Too many grid cells, increase unit.
            low_unit = unit
            if high_unit is None:
                unit *= 2.
            else:
                unit = (unit + high_unit) / 2.

            if verbose:
                log('Grid size {}, increase unit to {}'
                    .format(len(grid), unit))

        elif len(grid) < k * (1 - alpha):
            # Too few grid cells, decrease unit.
            high_unit = unit
            if low_unit is None:
                unit /= 2.
            else:
                unit = (unit + low_unit) / 2.

            if verbose:
                log('Grid size {}, decrease unit to {}'
                    .format(len(grid), unit))
        else:
            break

        if high_unit is not None and low_unit is not None and \
           high_unit - low_unit < 1e-20:
            break

        n_iter += 1
        if n_iter >= max_iter:
            # Should rarely get here.
            sys.stderr.write('WARNING: Max iterations reached, try increasing '
                             ' alpha parameter.\n')
            break

    if verbose:
        log('Found {} grid cells'.format(len(grid)))

    # Sample grid cell, then sample point within cell.
    valid_grids = set()
    gs_idx = []
    for n in range(N):
        if len(valid_grids) == 0:
            valid_grids = set(grid.keys())
        valid_grids_list = list(valid_grids)
        grid_cell = valid_grids_list[np.random.choice(len(valid_grids))]
        valid_grids.remove(grid_cell)
        sample = np.random.choice(list(grid[grid_cell]))
        if not replace:
            grid[grid_cell].remove(sample)
            if len(grid[grid_cell]) == 0:
                del grid[grid_cell]
        gs_idx.append(sample)

    if one_indexed:
        gs_idx = [ idx + 1 for idx in gs_idx ]

    return sorted(gs_idx)

In [ ]:
%%writefile /kaggle/working/usher/spatial_utils.py
"""
Spatial windowing utilities for optimal transport alignment.

This module handles spatial coordinate-based windowing and k-NN preparation
for memory-efficient spatial alignment.
"""

import logging
import numpy as np
import torch
import anndata as ad
from typing import List, Tuple, Optional, Dict, Any
from sklearn.neighbors import NearestNeighbors


def _create_spatial_windows(
    coords: np.ndarray,
    window_height: float,
    window_width: float,
    overlap: float = 0.1
) -> Tuple[List[np.ndarray], List[Dict[str, Any]]]:
    """
    Create spatial grid windows with overlap for memory-efficient processing.

    Parameters
    ----------
    coords : np.ndarray
        Spatial coordinates (n_cells, 2)
    window_height : float
        Height of each window
    window_width : float
        Width of each window
    overlap : float
        Overlap fraction between windows (0-1)

    Returns
    -------
    windows : List[np.ndarray]
        List of indices for each window
    window_info : List[Dict]
        Metadata for each window (bounds, n_spots)
    """
    min_x, min_y = coords[:, 0].min(), coords[:, 1].min()
    max_x, max_y = coords[:, 0].max(), coords[:, 1].max()

    # Compute step size with overlap
    step_height = window_height * (1 - overlap)
    step_width = window_width * (1 - overlap)

    windows = []
    window_info = []

    y = min_y
    while y < max_y:
        x = min_x
        while x < max_x:
            # Define window bounds
            x_min, x_max = x, x + window_width
            y_min, y_max = y, y + window_height

            # Find spots in this window
            mask = (
                (coords[:, 0] >= x_min) & (coords[:, 0] <= x_max) &
                (coords[:, 1] >= y_min) & (coords[:, 1] <= y_max)
            )
            indices = np.where(mask)[0]

            if len(indices) > 0:
                windows.append(indices)
                window_info.append({
                    'x_min': x_min, 'x_max': x_max,
                    'y_min': y_min, 'y_max': y_max,
                    'n_spots': len(indices)
                })

            x += step_width
        y += step_height

    logging.info(f"Created {len(windows)} spatial windows with {overlap*100:.0f}% overlap")
    if windows:
        avg_spots = np.mean([len(w) for w in windows])
        logging.info(f"Average spots per window: {avg_spots:.0f}")

    return windows, window_info


def _compute_spatial_knn(
    coords_source: np.ndarray,
    coords_target: np.ndarray,
    k: int = 50
) -> np.ndarray:
    """
    Compute k-nearest spatial neighbors from source to target.

    Parameters
    ----------
    coords_source : np.ndarray
        Source spatial coordinates (n_source, 2)
    coords_target : np.ndarray
        Target spatial coordinates (n_target, 2)
    k : int
        Number of nearest neighbors

    Returns
    -------
    knn_indices : np.ndarray
        k-NN indices (n_source, k) - for each source cell, indices of k nearest target cells
    """
    # Ensure we don't request more neighbors than available
    k = min(k, coords_target.shape[0])

    # Fit k-NN on target coordinates
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean', n_jobs=-1)
    nn.fit(coords_target)

    # Find k-NN for each source coordinate
    _, knn_indices = nn.kneighbors(coords_source)

    return knn_indices


def _auto_compute_window_size(
    coords: np.ndarray,
    n_windows_target: int = 20
) -> Tuple[float, float]:
    """
    Automatically compute window dimensions based on spatial extent.

    Parameters
    ----------
    coords : np.ndarray
        Spatial coordinates (n_cells, 2)
    n_windows_target : int
        Target number of windows to create

    Returns
    -------
    window_height : float
        Computed window height
    window_width : float
        Computed window width
    """
    min_x, min_y = coords[:, 0].min(), coords[:, 1].min()
    max_x, max_y = coords[:, 0].max(), coords[:, 1].max()

    x_range = max_x - min_x
    y_range = max_y - min_y

    # Assuming roughly square windows
    windows_per_side = np.sqrt(n_windows_target)
    window_width = x_range / windows_per_side
    window_height = y_range / windows_per_side

    return window_height, window_width


def prepare_spatial_batches(
    adata_a: ad.AnnData,
    adata_b: ad.AnnData,
    spatial_key: str = 'X_spatial',
    window_height: Optional[float] = None,
    window_width: Optional[float] = None,
    window_overlap: float = 0.1,
    n_windows_target: int = 20,
    spatial_knn: int = 50,  # Number of spatial nearest neighbors for constraints
    **kwargs
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], Dict[str, Any], Dict[str, Any]]:
    """
    Prepare batches using spatial windowing strategy.

    Parameters
    ----------
    adata_a : ad.AnnData
        Source dataset with spatial coordinates
    adata_b : ad.AnnData
        Target dataset with spatial coordinates
    spatial_key : str
        Key in .obsm for spatial coordinates
    window_height : float, optional
        Height of spatial windows (auto-computed if None)
    window_width : float, optional
        Width of spatial windows (auto-computed if None)
    window_overlap : float
        Overlap fraction between windows
    n_windows_target : int
        Target number of windows if auto-computing dimensions
    spatial_knn : int
        Number of spatial nearest neighbors for transport constraints
    **kwargs
        Additional parameters (ignored)

    Returns
    -------
    batches : List[Tuple[np.ndarray, np.ndarray]]
        List of (source_indices, target_indices) pairs for each window
    auxiliary_data : Dict[str, Any]
        Dictionary containing:
        - coords_a: Source spatial coordinates
        - coords_b: Target spatial coordinates
        - n_source: Number of source cells
        - n_target: Number of target cells
    window_info : Dict[str, Any]
        Metadata about spatial windows:
        - windows: List of window indices
        - window_metadata: List of window bounds and statistics
        - window_height: Used window height
        - window_width: Used window width
    """
    # Extract spatial coordinates
    if spatial_key not in adata_a.obsm:
        raise ValueError(f"Spatial coordinates not found in source .obsm['{spatial_key}']")
    if spatial_key not in adata_b.obsm:
        raise ValueError(f"Spatial coordinates not found in target .obsm['{spatial_key}']")

    coords_a = adata_a.obsm[spatial_key].astype(np.float32)
    coords_b = adata_b.obsm[spatial_key].astype(np.float32)

    if coords_a.shape[1] != 2 or coords_b.shape[1] != 2:
        raise ValueError(f"Spatial coordinates must be 2D, got shapes {coords_a.shape} and {coords_b.shape}")

    logging.info(f"Loaded spatial coordinates: source {coords_a.shape}, target {coords_b.shape}")

    # Auto-compute window dimensions if not provided
    if window_height is None or window_width is None:
        window_height, window_width = _auto_compute_window_size(coords_a, n_windows_target)
        logging.info(f"Auto-computed window dimensions: height={window_height:.2f}, width={window_width:.2f}")

    # Create spatial windows for source
    windows, window_metadata = _create_spatial_windows(
        coords_a, window_height, window_width, window_overlap
    )

    # Compute spatial k-NN from source to target
    logging.info(f"Computing spatial k-NN (k={spatial_knn}) from source to target...")
    knn_indices = _compute_spatial_knn(coords_a, coords_b, k=spatial_knn)
    logging.info(f"Computed spatial k-NN indices: shape {knn_indices.shape}")

    # Create batches from spatial windows
    # For affine-aligned data, use the SAME window bounds for both source and target
    batches = []
    for window_idx, source_window_indices in enumerate(windows):
        window_meta = window_metadata[window_idx]
        x_min, x_max = window_meta['x_min'], window_meta['x_max']
        y_min, y_max = window_meta['y_min'], window_meta['y_max']

        # Find target cells within the SAME bounds (NO padding for affine-aligned data!)
        target_mask = (
            (coords_b[:, 0] >= x_min) & (coords_b[:, 0] <= x_max) &
            (coords_b[:, 1] >= y_min) & (coords_b[:, 1] <= y_max)
        )
        target_indices = np.where(target_mask)[0]

        # Skip if either window is empty
        if len(source_window_indices) == 0 or len(target_indices) == 0:
            continue

        batches.append((source_window_indices, target_indices))

    logging.info(f"Created {len(batches)} spatial window batches")
    if batches:
        avg_source = np.mean([len(s) for s, _ in batches])
        avg_target = np.mean([len(t) for _, t in batches])
        logging.info(f"Average batch size: {avg_source:.0f} source, {avg_target:.0f} target cells")

        # Debug: Check if source and target have similar spatial distributions
        if abs(avg_source - avg_target) > 0.1 * avg_source:  # More than 10% difference
            logging.warning(f"Source and target windows have different cell counts!")
            logging.info(f"  Source spatial range: X[{coords_a[:, 0].min():.1f}, {coords_a[:, 0].max():.1f}], Y[{coords_a[:, 1].min():.1f}, {coords_a[:, 1].max():.1f}]")
            logging.info(f"  Target spatial range: X[{coords_b[:, 0].min():.1f}, {coords_b[:, 0].max():.1f}], Y[{coords_b[:, 1].min():.1f}, {coords_b[:, 1].max():.1f}]")

    # Prepare auxiliary data
    auxiliary_data = {
        'coords_a': coords_a,
        'coords_b': coords_b,
        'knn_indices': knn_indices,  # Spatial k-NN indices for transport constraints
        'n_source': len(coords_a),
        'n_target': len(coords_b)
    }

    # Window information
    window_info = {
        'windows': windows,
        'window_metadata': window_metadata,
        'window_height': window_height,
        'window_width': window_width,
        'overlap': window_overlap
    }

    return batches, auxiliary_data, window_info

In [ ]:
%%writefile /kaggle/working/usher/plot_utils.py
"""
Plot utilities for spatial alignment visualization.

This module provides reusable plotting functions for alignment diagnostics.
"""

import logging
import os
from typing import Optional, List, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import rapids_singlecell as rsc
import anndata as ad


def plot_dual_umap(
    concat_adata: ad.AnnData,
    cell_type_col: str,
    title_prefix: str,
    save_path: str,
    type_palette: Optional[Dict[str, str]] = None
) -> None:
    """
    Create dual UMAP plots: one with 'type' coloring and one with cell type coloring.
    """
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Default palette for source/target types
        if type_palette is None:
            type_palette = {'source_transformed': '#1f77b4', 'target': '#ff7f0e'}

        # Plot 1: Type coloring
        sc.pl.umap(
            concat_adata,
            color='type',
            palette=type_palette,
            ax=ax1,
            show=False,
            title=f'{title_prefix} - Type'
        )

        # Plot 2: Cell type coloring (if available)
        if cell_type_col in concat_adata.obs.columns:
            sc.pl.umap(
                concat_adata,
                color=cell_type_col,
                ax=ax2,
                show=False,
                title=f'{title_prefix} - {cell_type_col}'
            )
        else:
            ax2.text(0.5, 0.5, f'{cell_type_col} column not found',
                    ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title(f'{title_prefix} - Cell Type (Not Available)')

        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close()
        logging.info(f"UMAP plot saved: {save_path}")

    except Exception as e:
        logging.error(f"Failed to create dual UMAP plot: {e}")
        plt.close('all')


def plot_weight_heatmap(
    model,
    title: str,
    save_path: str
) -> None:
    """
    Plot transformation weight matrix as heatmap.
    """
    try:
        import torch.nn as nn

        # Extract weight matrix from model
        W = None
        if hasattr(model, 'net'):
            if isinstance(model.net, nn.Linear):
                W = model.net.weight.detach().cpu().numpy()
            elif isinstance(model.net, nn.Sequential):
                # Find last linear layer
                for m in reversed(model.net):
                    if isinstance(m, nn.Linear):
                        W = m.weight.detach().cpu().numpy()
                        break

        if W is not None:
            fig, ax = plt.subplots(1, 1, figsize=(8, 8))
            sns.heatmap(W, cmap='viridis', ax=ax)
            ax.set_title(title)
            plt.savefig(save_path, bbox_inches='tight')
            plt.close('all')
            logging.info(f"Weight heatmap saved: {save_path}")
        else:
            logging.info("No linear layer found for weight heatmap")

    except Exception as e:
        logging.warning(f"Failed to create weight heatmap: {e}")
        plt.close('all')


def plot_convergence(
    convergence_data: List[Dict],
    save_dir: str
) -> None:
    """
    Plot convergence metrics over E-M iterations.
    """
    if len(convergence_data) == 0:
        logging.info("No convergence data to plot (only 1 iteration or no mappings changed)")
        return

    try:
        # Convert to DataFrame
        conv_df = pd.DataFrame(convergence_data)

        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Plot 1: Percentage of mappings that changed per iteration (per batch)
        for batch_idx in conv_df['batch_idx'].unique():
            if batch_idx == 'ALL_TARGETS':
                continue
            batch_data = conv_df[conv_df['batch_idx'] == batch_idx]
            # Special label for target cell convergence
            label = 'All Targets' if batch_idx == 'ALL_TARGETS' else f'Batch {batch_idx}'
            ax1.plot(batch_data['iteration'], batch_data['pct_changed'],
                    marker='o', label=label, linewidth=2)

        ax1.set_xlabel('E-M Iteration', fontsize=12)
        ax1.set_ylabel('Percentage of Mappings Changed (%)', fontsize=12)
        ax1.set_title('Convergence: Percentage of Changed Mappings per Iteration', fontsize=14)
        ax1.grid(True, alpha=0.3)
        ax1.legend()

        # Plot 2: Absolute number of mappings that changed per iteration (per batch)
        for batch_idx in conv_df['batch_idx'].unique():
            if batch_idx != 'ALL_TARGETS':
                continue
            batch_data = conv_df[conv_df['batch_idx'] == batch_idx]
            # Special label for target cell convergence
            label = 'All Targets' if batch_idx == 'ALL_TARGETS' else f'Batch {batch_idx}'
            ax2.plot(batch_data['iteration'], batch_data['n_changed'],
                    marker='o', label=label, linewidth=2)

        ax2.set_xlabel('E-M Iteration', fontsize=12)
        ax2.set_ylabel('Number of Mappings Changed', fontsize=12)
        ax2.set_title('Convergence: Absolute Count of Changed Mappings per Iteration', fontsize=14)
        ax2.grid(True, alpha=0.3)
        ax2.legend()

        plt.tight_layout()

        # Save plot
        os.makedirs(save_dir, exist_ok=True)
        plot_filename = os.path.join(save_dir, 'convergence_plot.png')
        plt.savefig(plot_filename, bbox_inches='tight', dpi=150)
        plt.close()

        logging.info(f"Convergence plot saved: {plot_filename}")

        # Save raw data as CSV
        csv_filename = os.path.join(save_dir, 'convergence_data.csv')
        conv_df.to_csv(csv_filename, index=False)
        logging.info(f"Convergence data saved: {csv_filename}")

    except Exception as e:
        logging.error(f"Failed to create convergence plot: {e}")
        plt.close('all')


def plot_transfer_debug_umap(
    T_full: 'torch.Tensor',
    features_b: 'torch.Tensor',
    adata_a_obs: pd.DataFrame,
    adata_b: ad.AnnData,
    cell_type_col: str,
    iteration: int,
    save_dir: str,
    metric: str = 'euclidean',
    focused_mask_full: Optional['torch.Tensor'] = None,
    use_linear_assignment: bool = False
) -> None:
    """
    Create transfer-based UMAP debug plot.
    """
    try:
        import torch

        logging.info(f"Creating transfer-based debug plot for iteration {iteration}...")

        # Diagnostics: Check if T is binary (0 or 1) as expected after linear assignment
        T_unique_values = torch.unique(T_full)
        T_nnz = (T_full > 1e-6).sum().item()
        T_sparsity = 100 * T_nnz / T_full.numel()
        logging.info(f"  T_full diagnostics: shape={T_full.shape}, unique_values={T_unique_values.tolist()[:10]}, nnz={T_nnz}/{T_full.numel()} ({T_sparsity:.2f}%)")

        # Check row sums (should be 1.0 for each source cell)
        row_sums = T_full.sum(dim=1)
        logging.info(f"  T_full row sums: min={row_sums.min().item():.4f}, max={row_sums.max().item():.4f}, mean={row_sums.mean().item():.4f}")

        # Identify valid source cells (those with non-zero row sums = have transport assignments)
        valid_source_mask = row_sums > 1e-6
        n_valid = valid_source_mask.sum().item()
        n_total = len(row_sums)
        logging.info(f"  Valid source cells (with transport): {n_valid}/{n_total} ({100*n_valid/n_total:.1f}%)")

        if n_valid == 0:
            logging.warning("No valid source cells with transport! Cannot create transfer debug plot.")
            return

        # Filter to only valid source cells
        T_full_valid = T_full[valid_source_mask]
        adata_a_obs_valid = adata_a_obs[valid_source_mask.cpu().numpy()].copy()

        # Compute transfer: T @ features_b (only for valid cells)
        with torch.no_grad():
            features_a_transfer = T_full_valid @ features_b
            a_transfer_cpu = features_a_transfer.cpu().numpy()

        # Create AnnData with transferred features (only valid cells)
        adata_a_transfer = ad.AnnData(X=a_transfer_cpu, obs=adata_a_obs_valid)
        adata_a_transfer.obs["type"] = "source_transfer_debug"

        # Add M-step training status if focused_mask provided (only for valid cells)
        if focused_mask_full is not None:
            focused_mask_valid = focused_mask_full[valid_source_mask]
            focused_mask_cpu = focused_mask_valid.cpu().numpy()
            adata_a_transfer.obs["used_for_m_step"] = pd.Categorical(
                ["Used for Training" if f else "Excluded (High Entropy)" for f in focused_mask_cpu]
            )
            logging.info(f"M-step usage (valid cells only): {focused_mask_cpu.sum()}/{len(focused_mask_cpu)} cells used for training " +
                        f"({100*focused_mask_cpu.sum()/len(focused_mask_cpu):.1f}%)")

        adata_b_copy = adata_b.copy()
        adata_b_copy.obs["type"] = "target"

        if focused_mask_full is not None:
            adata_b_copy.obs["used_for_m_step"] = pd.Categorical(
                ["Target (placeholder)"] * len(adata_b_copy),
                categories=["Used for Training", "Excluded (High Entropy)", "Target (placeholder)"]
            )

        concat_adata = ad.concat(
            [adata_a_transfer, adata_b_copy], axis=0,
            label="batch", keys=["source", "target"], index_unique="_",
        )

        # Set cell type column if available
        if cell_type_col in adata_a_transfer.obs.columns and cell_type_col in adata_b_copy.obs.columns:
            common_ct_values = pd.concat([adata_a_transfer.obs[cell_type_col], adata_b_copy.obs[cell_type_col]])
            concat_adata.obs[cell_type_col] = common_ct_values.values

        # Compute UMAP with RAPIDS
        rsc.tl.pca(concat_adata)
        rsc.pp.neighbors(concat_adata, use_rep='X', metric=metric)
        rsc.tl.umap(concat_adata)

        # Create plot with 3 columns if focused_mask provided, otherwise 2
        n_cols = 3 if focused_mask_full is not None else 2
        fig, axes = plt.subplots(1, n_cols, figsize=(8*n_cols, 6))
        if n_cols == 2:
            ax1, ax2 = axes
        else:
            ax1, ax2, ax3 = axes

        palette = {'source_transfer_debug': '#9467bd', 'target': '#ff7f0e'}
        sc.pl.umap(
            concat_adata,
            color='type',
            palette=palette,
            ax=ax1,
            show=False,
            title=f'Transfer Debug Iter {iteration} - Type'
        )

        if cell_type_col in concat_adata.obs.columns:
            sc.pl.umap(
                concat_adata,
                color=cell_type_col,
                ax=ax2,
                show=False,
                title=f'Transfer Debug Iter {iteration} - {cell_type_col}'
            )
        else:
            ax2.text(0.5, 0.5, f'{cell_type_col} column not found',
                    ha='center', va='center', transform=ax2.transAxes)

        # Plot 3: M-step training status (if focused_mask provided)
        if focused_mask_full is not None and 'used_for_m_step' in concat_adata.obs.columns and use_linear_assignment:
            import torch

            T_full_valid_t = T_full_valid.t()  
            target_to_source = T_full_valid_t.argmax(dim=1).cpu().numpy()  
            target_has_match = (T_full_valid_t.max(dim=1)[0] > 0.5).cpu().numpy()  

            focused_mask_valid = focused_mask_full[valid_source_mask]
            focused_mask_cpu = focused_mask_valid.cpu().numpy()

            target_m_step_status = []
            for t_idx in range(len(target_has_match)):
                if target_has_match[t_idx]:
                    source_idx = target_to_source[t_idx]
                    if focused_mask_cpu[source_idx]:
                        target_m_step_status.append('Matched to Training Cell')
                    else:
                        target_m_step_status.append('Matched to Excluded Cell')
                else:
                    target_m_step_status.append('No Source Match')

            target_only = concat_adata[concat_adata.obs['type'] == 'target'].copy()

            target_only.obs['m_step_status'] = pd.Categorical(
                target_m_step_status,
                categories=['Matched to Training Cell', 'Matched to Excluded Cell', 'No Source Match'],
                ordered=True
            )

            m_step_palette = {
                'Matched to Training Cell': '#2ca02c',  # Green - good matches
                'Matched to Excluded Cell': '#d62728',  # Red - excluded matches
                'No Source Match': '#CCCCCC'  # Gray - unmatched
            }

            sc.pl.umap(
                target_only,  
                color='m_step_status',
                palette=m_step_palette,
                ax=ax3,
                show=False,
                title=f'Transfer Debug Iter {iteration} - Target M-step Status'
            )

        os.makedirs(save_dir, exist_ok=True)
        plot_filename = os.path.join(save_dir, f"umap_transfer_iter_{iteration:04d}.png")
        plt.savefig(plot_filename, bbox_inches='tight', dpi=150)
        plt.close()

        logging.info(f"Transfer debug UMAP saved: {plot_filename}")

    except Exception as e:
        logging.warning(f"Failed to create transfer debug plot for iteration {iteration}: {e}")


def plot_spatial_channels(
    adata_source: ad.AnnData,
    adata_target: ad.AnnData,
    spatial_key: str,
    iteration: int,
    save_path: str,
    channel_idx: int = 0,
    figsize: tuple = (15, 5),
    cmap: str = 'jet',
    point_size: float = 1,
    alpha: float = 0.8,
    dpi: int = 150
) -> None:
    """
    Create spatial channel plots showing gene expression on spatial coordinates.
    """
    try:
        # Check if spatial coordinates are available
        if spatial_key not in adata_source.obsm or spatial_key not in adata_target.obsm:
            logging.warning(f"Spatial coordinates not found in .obsm['{spatial_key}']")
            return

        # Create figure
        fig, ax = plt.subplots(1, 2, figsize=figsize)

        # Extract spatial coordinates
        spatial_coords_source = adata_source.obsm[spatial_key]
        spatial_coords_target = adata_target.obsm[spatial_key]

        # Extract expression values for the specified channel
        if hasattr(adata_source.X, 'toarray'):  # Sparse matrix
            expression_source = adata_source.X[:, channel_idx].toarray().flatten()
        else:  # Dense array
            expression_source = adata_source.X[:, channel_idx]

        if hasattr(adata_target.X, 'toarray'):  # Sparse matrix
            expression_target = adata_target.X[:, channel_idx].toarray().flatten()
        else:  # Dense array
            expression_target = adata_target.X[:, channel_idx]

        # Source (transformed) spatial plot
        scatter_source = ax[0].scatter(
            spatial_coords_source[:, 1], spatial_coords_source[:, 0],
            c=expression_source, cmap=cmap, s=point_size, alpha=alpha
        )
        ax[0].set_xticks([])
        ax[0].set_yticks([])
        ax[0].set_title('Source Transformed')
        ax[0].set_aspect('equal', 'box')
        plt.colorbar(scatter_source, ax=ax[0], fraction=0.046, pad=0.04)

        # Target spatial plot
        scatter_target = ax[1].scatter(
            spatial_coords_target[:, 1], spatial_coords_target[:, 0],
            c=expression_target, cmap=cmap, s=point_size, alpha=alpha
        )
        ax[1].set_xticks([])
        ax[1].set_yticks([])
        ax[1].set_title('Target')
        ax[1].set_aspect('equal', 'box')
        plt.colorbar(scatter_target, ax=ax[1], fraction=0.046, pad=0.04)

        # Add title and save
        plt.suptitle(f'Channel {channel_idx} - Iteration {iteration}')

        # Create directory if needed
        channels_dir = os.path.join(save_path, 'channels')
        os.makedirs(channels_dir, exist_ok=True)

        plot_filename = os.path.join(channels_dir, f"channel{channel_idx}_iter_{iteration:04d}.png")
        plt.savefig(plot_filename, bbox_inches='tight', dpi=dpi)
        plt.close('all')

        logging.info(f"Spatial channel plot saved to {plot_filename}")

    except Exception as e:
        logging.warning(f"Failed to generate spatial channel plot: {e}")
        plt.close('all')


def plot_spatial_mapping(
    coords_source: np.ndarray,
    coords_target: np.ndarray,
    mapping: np.ndarray,
    iteration: int,
    save_path: str,
    n_samples: int = 5000,
    figsize: tuple = (10, 10),
    alpha_lines: float = 0.3,
    alpha_points: float = 0.5,
    linewidth: float = 0.5,
    point_size_source: float = 2,
    point_size_target: float = 2,
    seed: int = 42
) -> None:
    """
    Visualize spatial alignment by drawing lines from source to mapped target coordinates.
    """
    try:
        np.random.seed(seed)

        # Filter valid mappings (exclude -1 values)
        valid_mask = mapping >= 0
        valid_indices = np.where(valid_mask)[0]

        if len(valid_indices) == 0:
            logging.warning("No valid mappings to visualize")
            return

        # Sample if we have too many points
        if len(valid_indices) > n_samples:
            sampled_indices = np.random.choice(valid_indices, n_samples, replace=False)
        else:
            sampled_indices = valid_indices

        logging.info(f"Plotting {len(sampled_indices)} spatial mappings out of {len(valid_indices)} valid mappings")

        # Create figure
        fig, ax = plt.subplots(figsize=figsize)

        # Plot all source and target points as background
        ax.scatter(coords_source[:, 0], coords_source[:, 1],
                  c='lightblue', s=point_size_source, alpha=alpha_points * 0.5,
                  label='Source', zorder=1)
        ax.scatter(coords_target[:, 0], coords_target[:, 1],
                  c='lightgreen', s=point_size_target, alpha=alpha_points * 0.5,
                  label='Target', zorder=1)

        # Plot the sampled mappings
        for i in sampled_indices:
            target_idx = mapping[i]
            if 0 <= target_idx < len(coords_target):  # Double-check validity
                # Draw line from source to target
                ax.plot([coords_source[i, 0], coords_target[target_idx, 0]],
                       [coords_source[i, 1], coords_target[target_idx, 1]],
                       'gray', alpha=alpha_lines, linewidth=linewidth, zorder=2)

        # Highlight the sampled source points
        ax.scatter(coords_source[sampled_indices, 0], coords_source[sampled_indices, 1],
                  c='blue', s=point_size_source, alpha=alpha_points, zorder=3)

        # Highlight the corresponding target points
        target_indices = mapping[sampled_indices]
        valid_targets = target_indices[target_indices >= 0]
        if len(valid_targets) > 0:
            ax.scatter(coords_target[valid_targets, 0], coords_target[valid_targets, 1],
                      c='green', s=point_size_target, alpha=alpha_points, zorder=3)

        ax.set_xlabel('X coordinate')
        ax.set_ylabel('Y coordinate')
        ax.set_title(f'Spatial Alignment - Iteration {iteration}\n({len(sampled_indices)} mapped pairs shown)')
        ax.set_aspect('equal', 'box')
        ax.legend(loc='best')

        # Create directory if needed
        mapping_dir = os.path.join(save_path, 'mapping')
        os.makedirs(mapping_dir, exist_ok=True)

        plot_filename = os.path.join(mapping_dir, f"spatial_mapping_iter_{iteration:04d}.png")
        plt.savefig(plot_filename, bbox_inches='tight', dpi=150)
        plt.close()

        logging.info(f"Spatial mapping plot saved to {plot_filename}")

    except Exception as e:
        logging.warning(f"Failed to generate spatial mapping plot: {e}")
        plt.close('all')

In [ ]:
%%writefile /kaggle/working/usher/struct_cache.py
"""Bộ nhớ đệm ma trận cấu trúc cho E-step FGW của USHER (dùng với stratified_pairing_fix=True: các batch CỐ ĐỊNH qua các vòng).

Trong code (cả bản gốc lẫn bản CUDA của bạn), compute_transport_fgw gọi compute_knn_graph_distance HAI lần cho MỖI batch ở
MỖI vòng EM:
  * C2 (target): target không bao giờ bị biến đổi và batch cố định  -> C2 của một batch GIỐNG HỆT nhau ở mọi vòng
                 -> chỉ cần tính MỘT lần cho mỗi batch. Vì chỉ tính một lần, dùng geodesic CHÍNH XÁC (BFS) thay cho landmark.
  * C1 (source): tính trên source ĐÃ biến đổi f(x) -> đổi theo từng vòng -> vẫn phải tính lại (giữ nguyên landmark của bạn).
    Chế độ 'both' (ablation): tính C1 trên embedding GỐC của source (đúng ký hiệu Cs của bài báo) -> cũng chỉ tính một lần.

MODE = "off"    : hành vi y hệt code gốc (không đụng gì).
MODE = "target" : C2 đệm (chính xác, 1 lần/batch); C1 như code gốc.
MODE = "both"   : C2 và C1 đều đệm; C1 tính trên embedding gốc (F_SRC).
MODE = "custom" (17, mục 3): C1 VÀ C2 đều đệm nhưng tính trên đặc trưng TUỲ Ý (ví dụ vi môi trường/niche) thay vì
    embedding tế bào — F_SRC, F_TGT (đã cắt đúng theo T/tgt, đúng thứ tự batch mà align_features_fgw dùng). Đặc trưng
    đưa vào USHER (cross-cost, output) không đổi; chỉ cấu trúc trong-miền C_s/C_t đổi.
MODE = "target_landmark" (20, Phần K, RESEARCH_CONTEXT mục 8 rà soát): C2 vẫn đệm 1 lần/batch nhưng dùng CHÍNH hàm
    landmark của code gốc (`graph_utils.compute_knn_graph_distance`, 512 landmark, β=2, soft-min) thay vì geodesic
    chính xác — để C_t và C_s dùng CÙNG "thước" (mặc định C_s luôn qua landmark này, C_t "target" lại dùng BFS chính
    xác, hai hàm khác nhau về hình dạng phân phối khoảng cách). C1 như code gốc (fallback_fn = landmark, không đệm).
Đồ thị giống hệt bản landmark của bạn: kNN cosine k=30 trên không gian low-rank, đối xứng hoá bằng max, trọng số 1.
"""
import hashlib, threading
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path
from sklearn.neighbors import NearestNeighbors

MODE = "off"
F_SRC = None          # embedding/đặc trưng gốc của source (cùng thứ tự Fs[T]), dùng khi MODE = "both" hoặc "custom"
F_TGT = None          # đặc trưng gốc của target (cùng thứ tự Fx[tgt]), chỉ dùng khi MODE = "custom" (17)
CACHE = {}
STATS = {"target_computed": 0, "target_hits": 0, "source_computed": 0, "source_hits": 0}
_lock = threading.Lock()


def reset(mode="off", f_src=None, f_tgt=None):
    global MODE, F_SRC, F_TGT
    MODE, F_SRC, F_TGT = mode, f_src, f_tgt
    CACHE.clear()
    for k in STATS: STATS[k] = 0


def exact_geodesic(F, k=30):
    """Geodesic chính xác (số bước) trên đúng đồ thị mà bản landmark dùng; chuẩn hoá theo max như code gốc."""
    import torch
    X = F.detach().float().cpu().numpy() if hasattr(F, "detach") else np.asarray(F, np.float32)
    n = len(X); k = min(k, n - 1)
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    idx = NearestNeighbors(n_neighbors=k + 1, metric="cosine", algorithm="brute").fit(Xn).kneighbors(Xn, return_distance=False)
    A = csr_matrix((np.ones(n * k, np.float32), (np.repeat(np.arange(n), k), idx[:, 1:].ravel())), shape=(n, n))
    A = A.maximum(A.T)
    D = shortest_path(A, directed=False, unweighted=True)
    fin = np.isfinite(D)
    D[~fin] = D[fin].max() if fin.any() else 1.0
    np.fill_diagonal(D, 0.0)
    return torch.from_numpy((D / (D.max() + 1e-8)).astype(np.float32))


def _lowrank_like_code(X, q=64):
    """Như fast_pca_threadsafe trong e_step_utils (dùng khi tính C1 trên embedding gốc)."""
    import torch
    X = X - X.mean(dim=0, keepdim=True)
    if X.shape[1] <= q: return X
    cov = torch.mm(X.T, X) / max(1, X.shape[0] - 1)
    with _lock:
        _, V = torch.linalg.eigh(cov)
    return torch.mm(X, V[:, -q:])


def _key(side, idx):
    return side, hashlib.md5(np.ascontiguousarray(np.asarray(idx, dtype=np.int64)).tobytes()).hexdigest()


def target_C(tgt_idx, features_target_lr, knn_k, device):
    key = _key("tgt", tgt_idx)
    if key in CACHE:
        STATS["target_hits"] += 1; return CACHE[key]
    if MODE == "custom":
        import torch, torch.nn.functional as Fn
        X = torch.from_numpy(np.asarray(F_TGT[np.asarray(tgt_idx)], np.float32)).to(device)
        C = exact_geodesic(_lowrank_like_code(Fn.normalize(X, p=2, dim=1)), knn_k).to(device)
    elif MODE == "target_landmark":
        from graph_utils import compute_knn_graph_distance
        C = compute_knn_graph_distance(features_target_lr, k=knn_k, metric="cosine", device=device)
    else:
        C = exact_geodesic(features_target_lr, knn_k).to(device)   # "target"/"both": như code gốc (embedding tế bào)
    CACHE[key] = C; STATS["target_computed"] += 1
    return C


def source_C(src_idx, features_source_lr, knn_k, device, fallback_fn, metric):
    if MODE not in ("both", "custom"):
        return fallback_fn(features_source_lr, k=knn_k, metric=metric, device=device)   # y hệt code gốc
    key = _key("src", src_idx)
    if key in CACHE:
        STATS["source_hits"] += 1; return CACHE[key]
    import torch, torch.nn.functional as Fn
    X = torch.from_numpy(np.asarray(F_SRC[np.asarray(src_idx)], np.float32)).to(device)
    C = exact_geodesic(_lowrank_like_code(Fn.normalize(X, p=2, dim=1)), knn_k).to(device)
    CACHE[key] = C; STATS["source_computed"] += 1
    return C

In [ ]:
%%writefile /kaggle/working/usher/aux_spatial.py
"""aux_spatial.py — Aux không gian kiểu bài báo USHER (RESEARCH_CONTEXT U9: khoảng cách toạ độ sau đăng ký affine)
cho FGW (vòng 18, mục 4). Giống `struct_cache.py` về hình thức: module-level state + `reset()` để USHER-ST không
biết gì về nó khi beta = 0 (hành vi y hệt code gốc).

Trộn thêm số hạng khoảng cách không gian vào M_aux đã chuẩn hoá ([0,1]), ngay sau dòng chuẩn hoá gốc trong
`compute_transport_fgw` (và `compute_transport_ot`/Sinkhorn nếu cùng đoạn code — không dùng thực tế vì
`e_step_method` luôn là "fgw" trong mọi cấu hình của dự án; ở đó gọi `combine(M_aux, None, None)` — vô hại, trả
nguyên M_aux).

reset(beta, sigma, xy_src, xy_tgt): xy_src/xy_tgt là toạ độ (µm, ĐÃ ĐĂNG KÝ vào hệ Xenium) theo ĐÚNG thứ tự tế bào
mà align_features_fgw dùng — xy_src theo thứ tự Fs[T], xy_tgt theo thứ tự Fx[tgt] (giống F_SRC/F_TGT của
struct_cache MODE="custom"). beta = 0 -> combine() trả nguyên M_aux.
combine(M_aux, src_idx, tgt_idx): d = khoảng cách Euclid (µm) giữa toạ độ đã đăng ký của batch hiện tại;
M_sp = 1 − exp(−d²/(2σ²)) ∈ [0, 1) (không phạt vô hạn khi rất xa, khác cdist thô của M_aux); trả
(M_aux + beta · M_sp) / (1 + beta).
"""
import numpy as np

BETA = 0.0
SIGMA = 100.0
XY_SRC = None   # (n_T, 2) µm, đã đăng ký vào hệ Xenium, cùng thứ tự Fs[T]
XY_TGT = None   # (n_tgt, 2) µm, cùng thứ tự Fx[tgt]


def reset(beta=0.0, sigma=100.0, xy_src=None, xy_tgt=None):
    global BETA, SIGMA, XY_SRC, XY_TGT
    BETA, SIGMA, XY_SRC, XY_TGT = beta, sigma, xy_src, xy_tgt


def combine(M_aux, src_idx, tgt_idx):
    if BETA <= 0 or src_idx is None or tgt_idx is None or XY_SRC is None or XY_TGT is None:
        return M_aux
    import torch
    xs = torch.from_numpy(np.asarray(XY_SRC[np.asarray(src_idx)], np.float32)).to(M_aux.device)
    xt = torch.from_numpy(np.asarray(XY_TGT[np.asarray(tgt_idx)], np.float32)).to(M_aux.device)
    d = torch.cdist(xs, xt, p=2)
    M_sp = 1.0 - torch.exp(-(d ** 2) / (2 * SIGMA ** 2))
    return (M_aux + BETA * M_sp) / (1.0 + BETA)

In [ ]:
# Cell vá — y hệt vòng 20–22
P = "/kaggle/working/usher/"
def patch(f, old, new):
    s = open(P + f).read(); n = s.count(old)
    assert n == 1, f"{f}: đoạn cần vá xuất hiện {n} lần (cần đúng 1): {old[:50]!r}"
    open(P + f, "w").write(s.replace(old, new)); print(f"vá {f}: {old.splitlines()[0][:70]!r}\n   -> {new.splitlines()[0][:90]!r}")
PATCHES = [
 [
  "run_usher.py",
  "import rapids_singlecell as rsc",
  "try:\n    import rapids_singlecell as rsc\nexcept Exception:\n    rsc = None   # chỉ dùng để vẽ UMAP debug"
 ],
 [
  "plot_utils.py",
  "import rapids_singlecell as rsc",
  "try:\n    import rapids_singlecell as rsc\nexcept Exception:\n    rsc = None   # chỉ dùng để vẽ UMAP debug"
 ],
 [
  "run_usher.py",
  "import scanpy as sc",
  "try:\n    import scanpy as sc\nexcept Exception:\n    sc = None"
 ],
 [
  "plot_utils.py",
  "import scanpy as sc",
  "try:\n    import scanpy as sc\nexcept Exception:\n    sc = None"
 ],
 [
  "run_usher.py",
  "if True:\n        from model_utils import save_alignment_model\n        dir_path = \"../datasets/scGPT_example/\"",
  "if os.environ.get(\"USHER_SAVE_DIR\"):\n        from model_utils import save_alignment_model\n        dir_path = os.environ[\"USHER_SAVE_DIR\"]"
 ],
 [
  "run_usher.py",
  "e_step_method=e_step_method, seed=2025\n",
  "e_step_method=e_step_method, seed=2025 + int(os.environ.get(\"USHER_SEED\", 0))\n"
 ],
 [
  "run_usher.py",
  "seed=2025 + it",
  "seed=2025 + int(os.environ.get(\"USHER_SEED\", 0)) + it"
 ],
 [
  "run_usher.py",
  "max_workers=min(len(batches), 4)",
  "max_workers=min(len(batches), int(os.environ.get(\"USHER_EM_WORKERS\", 4)))"
 ],
 [
  "m_step_utils.py",
  "import logging\n",
  "import logging, os\n"
 ],
 [
  "m_step_utils.py",
  "        loss = lambda_cross * loss_cross + lambda_var * loss_var",
  "        _id_mode = os.environ.get(\"USHER_ID_MODE\", \"full\")\n        if _id_mode == \"linear\":\n            assert isinstance(model.net, nn.Linear), \"USHER_ID_MODE=linear chỉ hợp lệ khi model.net là nn.Linear (hidden_dim=None)\"\n            _id_term = F.mse_loss(source_transformed - model.net.bias, source_features)\n        else:\n            _id_term = F.mse_loss(source_transformed, source_features)\n        loss = lambda_cross * loss_cross + lambda_var * loss_var + float(os.environ.get(\"USHER_LAMBDA_ID\", 0)) * _id_term"
 ],
 [
  "e_step_utils.py",
  "    low_rank_dim: int = 64  # Kích thước Low-Rank\n) -> torch.Tensor:",
  "    low_rank_dim: int = 64,  # Kích thước Low-Rank\n    src_idx=None, tgt_idx=None\n) -> torch.Tensor:"
 ],
 [
  "e_step_utils.py",
  "    if use_knn_graph:\n        C1 = compute_knn_graph_distance(features_source_lr, k=knn_k, metric=metric, device=device)",
  "    import struct_cache as _sc\n    if _sc.MODE != 'off' and src_idx is not None and tgt_idx is not None:\n        C1 = _sc.source_C(src_idx, features_source_lr, knn_k, device, compute_knn_graph_distance, metric)\n        C2 = _sc.target_C(tgt_idx, features_target_lr, knn_k, device)\n    elif use_knn_graph:\n        C1 = compute_knn_graph_distance(features_source_lr, k=knn_k, metric=metric, device=device)"
 ],
 [
  "e_step_utils.py",
  "            use_knn_graph=use_knn_graph, device=device, iteration=iteration, verbose=verbose\n        )\n    else:",
  "            use_knn_graph=use_knn_graph, device=device, iteration=iteration, verbose=verbose,\n            src_idx=source_indices, tgt_idx=target_indices\n        )\n    else:"
 ],
 [
  "e_step_utils.py",
  "        aux_source_torch = torch.from_numpy(aux_features_source).float().to(device)\n        aux_target_torch = torch.from_numpy(aux_features_target).float().to(device)\n        M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)\n        M_aux = M_aux / (M_aux.max().clamp(min=1e-8))\n",
  "        aux_source_torch = torch.from_numpy(aux_features_source).float().to(device)\n        aux_target_torch = torch.from_numpy(aux_features_target).float().to(device)\n        M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)\n        M_aux = M_aux / (M_aux.max().clamp(min=1e-8))\n        import aux_spatial as _asp\n        M_aux = _asp.combine(M_aux, src_idx, tgt_idx)\n"
 ],
 [
  "e_step_utils.py",
  "        aux_source_torch = torch.from_numpy(aux_features_source).to(device).float()\n        aux_target_torch = torch.from_numpy(aux_features_target).to(device).float()\n        M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)\n\n        if torch.isnan(M_aux).any():\n            logging.warning(\"M_aux contains NaN values.\")\n            \n        M_aux = M_aux / (M_aux.max().clamp(min=1e-8))\n",
  "        aux_source_torch = torch.from_numpy(aux_features_source).to(device).float()\n        aux_target_torch = torch.from_numpy(aux_features_target).to(device).float()\n        M_aux = torch.cdist(aux_source_torch, aux_target_torch, p=2)\n\n        if torch.isnan(M_aux).any():\n            logging.warning(\"M_aux contains NaN values.\")\n            \n        M_aux = M_aux / (M_aux.max().clamp(min=1e-8))\n        import aux_spatial as _asp\n        M_aux = _asp.combine(M_aux, None, None)   # compute_transport_ot: không có src_idx/tgt_idx, không dùng thực tế (e_step_method luôn 'fgw')\n"
 ]
]
for f, old, new in PATCHES: patch(f, old, new)

In [ ]:
%%writefile /kaggle/working/run04.py
"""04 — Baseline không dùng model cho SPATCH COAD (chạy trên CPU).

Câu hỏi: kNN acc thấp (CosMx, Stereo-seq, Visium HD) là do MODEL hay do DỮ LIỆU/NHÃN?
  A. Cùng tế bào, cùng thước đo, 5 biểu diễn: chỉ độ sâu | PCA biểu hiện | scGPT-spatial | Nicheformer
     + phân rã: PCA trên gene chung của mọi assay | PCA trên gene chung + cùng độ sâu (lấy mẫu con count)
  B. Count bị mất khi ánh xạ symbol -> Ensembl cho Nicheformer: gene nào, loại gì, cứu được bao nhiêu
  C. Chia bàn cờ 500 µm (T/H) + thành phần cell type, dùng lại cho 03c / 05

Đầu vào (đường dẫn đổi được bằng biến môi trường):
  LITE_DIR   bản lite (nguồn chính: đúng các tế bào đã dùng để trích embedding)
  FULL_DIR   bản đầy đủ (dự phòng: nếu không có lite thì lấy đúng các tế bào đó từ file đầy đủ)
  EMB_SCGPT, EMB_NICHE   output của 03a, 03b
"""
import os, sys, glob, json, re, time, subprocess, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, recall_score

TISSUE = os.environ.get("TISSUE", "coad").lower()   # vòng 26: coad (mặc định, giữ nguyên hành vi cũ) | ov | hcc
def _auto_dir(name):
    h = sorted(p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(p)); return h[0] if h else ""
if TISSUE == "coad":
    LITE = os.environ.get("LITE_DIR", "/kaggle/input/datasets/hepha4438/spatch-coad-lite")
    FULL = os.environ.get("FULL_DIR", "/kaggle/input/notebooks/hepha4438/spatch-coad/spatch_coad")
else:   # output notebook 26a: spatch_<mô>_lite + spatch_<mô>
    LITE = os.environ.get("LITE_DIR") or _auto_dir(f"spatch_{TISSUE}_lite")
    FULL = os.environ.get("FULL_DIR") or _auto_dir(f"spatch_{TISSUE}")
EMB_DIRS = {"scgpt": os.environ.get("EMB_SCGPT", "/kaggle/input/notebooks/hepha4438/03a-scgpt-emb/emb_scgpt"),
            "niche": os.environ.get("EMB_NICHE", "/kaggle/input/notebooks/hepha4438/03b-niche-emb/emb_niche")}
OUT = os.environ.get("OUT_DIR", "/kaggle/working/out04")
ASSAYS = ["xenium5k", "cosmx6k", "visiumhd_ffpe", "stereoseq", "visiumhd_ff", "scrna"]
SPATIAL = ASSAYS[:5]
LABEL_COL = {"scrna": "major_annotation"}
FULL_REL = {"xenium5k": "xenium5k_tx/transcriptome/adata.h5ad", "cosmx6k": "cosmx6k_tx/transcriptome/adata.h5ad",
            "visiumhd_ffpe": "visiumhd_ffpe_tx/transcriptome/adata.h5ad", "stereoseq": "stereoseq_tx/transcriptome/adata.h5ad",
            "visiumhd_ff": "visiumhd_ff_tx/transcriptome/adata.h5ad", "scrna": f"scrna_{TISSUE}/adata.h5ad"}
SYMBOL_ASSAYS = ["cosmx6k", "stereoseq", "scrna"]          # 3 assay không có Ensembl -> Nicheformer phải ánh xạ symbol
K_NN, N_MAX, N_HVG, N_PC = 15, 20000, 2000, 50              # giống knn_label_acc của 03a/03b
TILE_UM = float(os.environ.get("TILE_UM", 500))
DEPTH_Q = float(os.environ.get("DEPTH_Q", 0.25))            # độ sâu chung = min qua các assay của phân vị 25%
SEED = 0


def log(*a): print(*a, flush=True)


# =============================================================================== đọc dữ liệu (h5py, không cần anndata)
def _find(root, rel):
    if not root or not os.path.isdir(root): return None
    hits = sorted(glob.glob(f"{root}/**/{rel}", recursive=True))
    return hits[0] if hits else None

def _h5_str(arr):
    return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in arr])

def _h5_col(g, name):
    o = g[name]
    if hasattr(o, "keys"):
        if "codes" in o:
            cats = o["categories"][()]
            cats = _h5_str(cats) if cats.dtype.kind in "OSU" else cats
            codes = o["codes"][()]
            return np.array([cats[c] if c >= 0 else None for c in codes], dtype=object)
        if "values" in o:
            v = o["values"][()]
            v = _h5_str(v) if v.dtype.kind in "OS" else v
            if "mask" in o:
                v = v.astype(object); v[o["mask"][()]] = None
            return v
        raise ValueError(f"không đọc được cột {name}")
    v = o[()]
    return _h5_str(v) if v.dtype.kind in "OS" else v

def _read_X(g):
    import h5py
    if isinstance(g, h5py.Dataset): return sp.csr_matrix(g[()])
    enc = g.attrs.get("encoding-type", "csr_matrix"); enc = enc.decode() if isinstance(enc, bytes) else str(enc)
    shape = tuple(int(s) for s in g.attrs["shape"])
    ctor = sp.csc_matrix if "csc" in enc else sp.csr_matrix
    return ctor((g["data"][()], g["indices"][()], g["indptr"][()]), shape=shape).tocsr()

def read_h5(path, obs_cols):
    import h5py
    with h5py.File(path, "r") as f:
        def idx(g): return _h5_str(g[g.attrs.get("_index", "_index")][()])
        obs = pd.DataFrame(index=idx(f["obs"]))
        for c in obs_cols:
            if c in f["obs"]: obs[c] = _h5_col(f["obs"], c)
        var = pd.DataFrame(index=idx(f["var"])); var["symbol"] = var.index.astype(str)
        for gc in ("gene_ids", "gene_id", "ensembl"):
            if gc in f["var"]: var["ensembl"] = _h5_col(f["var"], gc); break
        xy = None
        if "obsm" in f and "spatial" in f["obsm"]:
            try: xy = np.asarray(f["obsm"]["spatial"][()])[:, :2].astype(float)
            except Exception: xy = None
        X = _read_X(f["X"])
    return X, obs, var, xy

def emb_ids(assay):
    for d in EMB_DIRS.values():
        if d and os.path.exists(f"{d}/{assay}_obs.csv"):
            return pd.read_csv(f"{d}/{assay}_obs.csv", index_col=0, dtype=str).index.astype(str)
    return None

def load_assay(assay):
    """Trả về X (csr, count thô), obs (label, total_counts, n_genes, x, y), var, nguồn. Đúng thứ tự tế bào của embedding."""
    lab = LABEL_COL.get(assay, "annotation")
    cols = (lab, "high_quality", "x", "y")
    ids = emb_ids(assay)
    p = _find(LITE, f"{assay}_sub.h5ad")
    if p:
        X, obs, var, xy = read_h5(p, cols); src = "lite"
        if "high_quality" in obs:                         # đúng như read_lite của 03a/03b
            keep = obs["high_quality"].astype(int).to_numpy() == 1
            X, obs = X[keep], obs.loc[keep]; xy = xy[keep] if xy is not None else None
    else:
        p = _find(FULL, FULL_REL[assay])
        if not p: raise FileNotFoundError(f"[{assay}] không có ở cả LITE ({LITE}) lẫn FULL ({FULL})")
        log(f"  [{assay}] KHÔNG có lite -> đọc bản đầy đủ {p} (chậm hơn)")
        X, obs, var, xy = read_h5(p, cols); src = "full"
        if ids is None:
            if "high_quality" in obs:
                keep = obs["high_quality"].astype(int).to_numpy() == 1
                X, obs = X[keep], obs.loc[keep]; xy = xy[keep] if xy is not None else None
    obs.index = obs.index.astype(str)
    if ids is not None:                                   # ép đúng thứ tự tế bào của embedding
        pos = pd.Index(obs.index).get_indexer(ids)
        if (pos < 0).any():
            raise ValueError(f"[{assay}] {int((pos < 0).sum())} tế bào của embedding không có trong {src}")
        if not (pos == np.arange(len(obs))).all():
            log(f"  [{assay}] sắp lại thứ tự tế bào theo embedding ({src})")
        X, obs = X[pos], obs.iloc[pos]; xy = xy[pos] if xy is not None else None
    X = sp.csr_matrix(X, dtype=np.float32)
    o = pd.DataFrame(index=obs.index)
    o["label"] = obs[lab].astype(str).to_numpy()
    o["total_counts"] = np.asarray(X.sum(1)).ravel()
    o["n_genes"] = np.asarray((X > 0).sum(1)).ravel()
    if "x" in obs and "y" in obs:
        o["x"], o["y"] = pd.to_numeric(obs["x"]).to_numpy(), pd.to_numeric(obs["y"]).to_numpy()
    elif xy is not None:
        o["x"], o["y"] = xy[:, 0], xy[:, 1]
    return X, o, var, src

def load_emb(model, assay, ids):
    d = EMB_DIRS.get(model)
    if not d or not os.path.exists(f"{d}/{assay}_final.npy"): return None
    E = np.load(f"{d}/{assay}_final.npy").astype(np.float32)
    eid = pd.read_csv(f"{d}/{assay}_obs.csv", index_col=0, dtype=str).index.astype(str)
    assert len(E) == len(eid), f"{model}/{assay}: {len(E)} hàng embedding nhưng {len(eid)} dòng obs"
    pos = pd.Index(eid).get_indexer(ids)
    if (pos < 0).any():
        log(f"  [{assay}] {model}: {int((pos < 0).sum())} tế bào thiếu -> bỏ qua model này"); return None
    return E[pos]

def reported_acc(model, assay):
    """kNN acc mà 03a/03b đã báo cáo, để kiểm tra 04 tái lập đúng."""
    try:
        d = EMB_DIRS[model]
        if model == "scgpt": return json.load(open(f"{d}/{assay}.json"))["knn_label_acc"]
        return json.load(open(f"{d}/summary.json"))["assays"][assay]["knn_label_acc"]["masked_all_cells"]
    except Exception:
        return None


# =============================================================================== biểu diễn baseline
def _unit(E):
    E = np.asarray(E, dtype=np.float64); return E / np.linalg.norm(E, axis=1, keepdims=True).clip(1e-12)

def lognorm(X):
    lib = np.asarray(X.sum(1)).ravel(); lib[lib == 0] = 1
    Y = (sp.diags(1e4 / lib) @ X).tocsr().astype(np.float32); Y.data = np.log1p(Y.data); return Y

def hvg(Y, n):
    """HVG kiểu 'seurat' (xấp xỉ, không cần scanpy): dispersion chuẩn hoá theo 20 bin trung bình."""
    mu = np.asarray(Y.mean(0)).ravel()
    ex2 = np.asarray(Y.multiply(Y).mean(0)).ravel()
    var = np.maximum(ex2 - mu ** 2, 0)
    ok = np.where(mu > 0)[0]
    if len(ok) <= n: return ok
    disp = np.log(var[ok] / mu[ok] + 1e-12)
    bins = pd.qcut(mu[ok], 20, labels=False, duplicates="drop")
    s = pd.Series(disp)
    z = s.groupby(bins).transform(lambda v: (v - v.mean()) / (v.std() if v.std() > 0 else 1.0)).fillna(0).to_numpy()
    return np.sort(ok[np.argsort(-z)[:n]])

def pca_expr(X, n_hvg=N_HVG, n_pc=N_PC):
    Y = lognorm(X); g = hvg(Y, n_hvg)
    Z = Y[:, g].toarray()
    Z -= Z.mean(0); sd = Z.std(0); sd[sd == 0] = 1; Z /= sd; np.clip(Z, None, 10, out=Z)
    n_pc = min(n_pc, Z.shape[1] - 1, Z.shape[0] - 1)
    P = PCA(n_pc, svd_solver="randomized", random_state=SEED).fit_transform(Z).astype(np.float32)
    return P, int(len(g))

def depth_only(obs):
    F = np.c_[np.log1p(obs["total_counts"].to_numpy()), np.log1p(obs["n_genes"].to_numpy())]
    return ((F - F.mean(0)) / F.std(0).clip(1e-12)).astype(np.float32)

def thin_to(X, D, rng):
    """Lấy mẫu con KHÔNG hoàn lại đúng D count mỗi tế bào (tế bào có < D count bị loại)."""
    X = X.tocsr(); tc = np.asarray(X.sum(1)).ravel()
    keep = np.where(tc >= D)[0]
    Xk = X[keep].tocsr(); data = np.rint(Xk.data).astype(np.int64); new = np.zeros_like(data)
    for i in range(Xk.shape[0]):
        s, e = Xk.indptr[i], Xk.indptr[i + 1]
        new[s:e] = rng.multivariate_hypergeometric(data[s:e], D)
    Y = sp.csr_matrix((new.astype(np.float32), Xk.indices.copy(), Xk.indptr.copy()), shape=Xk.shape)
    Y.eliminate_zeros()
    return Y, keep


# =============================================================================== thước đo (cùng định nghĩa với 03a/03b)
def knn_eval(F, labels, depth_log, unit=True, k=K_NN, n_max=N_MAX):
    """kNN acc = balanced accuracy khi đoán nhãn bằng đa số 15 hàng xóm, trên cùng tập con 20k (seed 0) như 03a/03b.
    Thêm 2 thước đo độ sâu trên cùng đồ thị kNN:
      depth_nn_rho        : Spearman(log count của tế bào, trung bình log count của 15 hàng xóm)
      depth_nn_rho_within : như trên nhưng sau khi trừ trung bình theo cell type -> độ sâu 'ngoài sinh học'"""
    n = len(F)
    sub = np.random.RandomState(0).choice(n, min(n_max, n), replace=False)
    A = _unit(F[sub]) if unit else np.asarray(F[sub], dtype=np.float64)
    nn = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(A).kneighbors(A, return_distance=False)[:, 1:]
    classes, codes = np.unique(np.asarray(labels)[sub].astype(str), return_inverse=True)
    cnt = np.zeros((len(sub), len(classes)))
    np.add.at(cnt, (np.repeat(np.arange(len(sub)), k), codes[nn].ravel()), 1)
    pred = cnt.argmax(1)                  # hoà -> nhãn nhỏ nhất theo thứ tự chữ cái, giống pd.Series.mode().iloc[0]
    rec = recall_score(codes, pred, labels=np.arange(len(classes)), average=None, zero_division=0)
    t = np.asarray(depth_log)[sub]
    r = t - pd.Series(t).groupby(codes).transform("mean").to_numpy()
    return ({"acc": round(float(balanced_accuracy_score(codes, pred)), 4),
             "depth_nn_rho": round(float(spearmanr(t, t[nn].mean(1)).correlation), 3),
             "depth_nn_rho_within": round(float(spearmanr(r, r[nn].mean(1)).correlation), 3)},
            pd.DataFrame({"class": classes, "n": np.bincount(codes, minlength=len(classes)), "recall": rec.round(3)}))

def pca_depth(E, n_genes, n_pc=5):
    U = _unit(E); p = PCA(n_pc, random_state=SEED).fit(U); pcs = p.transform(U)
    return [round(abs(float(spearmanr(n_genes, pcs[:, i]).correlation)), 3) for i in range(n_pc)]


# =============================================================================== B. mất count khi ánh xạ symbol
LOST_CATS = [("control/probe âm", r"^(NEGPRB|NEGATIVE|NEG_|SYSTEMCONTROL|FALSECODE|BLANK|CONTROL)"),
             ("ty thể (MT-)", r"^MT-"), ("MALAT1/NEAT1", r"^(MALAT1|NEAT1)$"),
             ("lncRNA/antisense", r"^(LINC\d|LOC\d)|-AS\d|-DT$|-IT\d|^ENSG|^RP\d+-|^CT[AB]-|^AC\d{6}|^AL\d{6}|^AP\d{6}"),
             ("ribosome (RPL/RPS)", r"^RP[LS]\d")]

def lost_category(sym):
    for name, pat in LOST_CATS:
        if re.search(pat, sym): return name
    return "khác (có thể là protein-coding bị lệch tên)"

def niche_ref_ids():
    hits = sorted(glob.glob("/kaggle/input/**/model_means/model.h5ad", recursive=True)
                  + glob.glob("/kaggle/working/**/model_means/model.h5ad", recursive=True))
    if not hits:
        repo = "/kaggle/working/nicheformer_repo"
        if not os.path.isdir(repo):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/theislab/nicheformer.git", repo], check=False)
        hits = sorted(glob.glob(f"{repo}/data/model_means/model.h5ad"))
    if not hits: return None
    import h5py
    with h5py.File(hits[0], "r") as f:
        g = f["var"]; ids = _h5_str(g[g.attrs.get("_index", "_index")][()])
    return pd.Index(pd.Index(ids).str.split(".").str[0])

def mygene_lookup(symbols, ref):
    """symbol -> (Ensembl trong vocab Nicheformer, nguồn 'symbol'|'alias'), chỉ nhận khi KHÔNG mơ hồ."""
    try:
        import mygene
        mg = mygene.MyGeneInfo()
        out = mg.querymany(list(symbols), scopes="symbol,alias", fields="ensembl.gene,symbol",
                           species="human", returnall=True, verbose=False)["out"]
    except Exception as e:
        log(f"  (mygene không chạy được: {type(e).__name__}: {str(e)[:120]}) -> bỏ qua phần 'cứu được'")
        return None
    return resolve_mygene(out, ref)

def resolve_mygene(out, ref):
    refset = set(ref); exact, alias = {}, {}
    for h in out:
        if h.get("notfound"): continue
        q = str(h["query"]).upper()
        ens = h.get("ensembl", [])
        ens = ens if isinstance(ens, list) else [ens]
        gids = set()
        for e in ens:
            g = e.get("gene") if isinstance(e, dict) else None
            if isinstance(g, list): gids |= set(g)
            elif g: gids.add(g)
        gids = {g for g in gids if g in refset}
        if not gids: continue
        tgt = exact if str(h.get("symbol", "")).upper() == q else alias
        tgt.setdefault(q, set()).update(gids)
    res = {}
    for q in set(exact) | set(alias):
        if len(exact.get(q, ())) == 1: res[q] = (next(iter(exact[q])), "symbol")
        elif q not in exact and len(alias.get(q, ())) == 1: res[q] = (next(iter(alias[q])), "alias")
        else: res[q] = (None, "mơ hồ")
    return res


# =============================================================================== C. bàn cờ
def full_origin(assay):
    p = _find(LITE, f"{assay}_obs_full.parquet")
    if p:
        try:
            d = pd.read_parquet(p, columns=["x", "y"]); return float(d["x"].min()), float(d["y"].min()), "obs_full"
        except Exception as e:
            log(f"  [{assay}] không đọc được {p}: {e}")
    return None

def full_composition(assay):
    p = _find(LITE, f"{assay}_obs_full.parquet")
    if not p: return None
    try:
        d = pd.read_parquet(p)
        lab = LABEL_COL.get(assay, "annotation")
        if "high_quality" in d: d = d[d["high_quality"].astype(int) == 1]
        return d[lab].astype(str).value_counts(normalize=True)
    except Exception:
        return None


# =============================================================================== chạy
def main():
    os.makedirs(OUT, exist_ok=True)
    for m, d in list(EMB_DIRS.items()):
        if not d or not os.path.isdir(d):
            hits = sorted(glob.glob(f"/kaggle/input/**/emb_{m}", recursive=True))
            EMB_DIRS[m] = hits[0] if hits else None
    log(f"LITE={LITE} ({'có' if os.path.isdir(LITE) else 'KHÔNG có'}) | FULL={FULL} ({'có' if os.path.isdir(FULL) else 'KHÔNG có'})")
    log(f"embedding: {EMB_DIRS}")
    rng = np.random.default_rng(SEED)
    S = {"assays": {}, "matched": {}, "mapping": {}, "split": {}}
    per_class, comp_rows, cache = [], [], {}

    # ------------------------------------------------------------------ A1. từng assay trên gene của chính nó
    for assay in ASSAYS:
        t0 = time.time()
        X, obs, var, src = load_assay(assay)
        lab, dlog, ng = obs["label"].to_numpy(), np.log1p(obs["total_counts"].to_numpy()), obs["n_genes"].to_numpy()
        n_cls = len(np.unique(lab))
        log(f"\n[{assay}] {X.shape[0]} tế bào x {X.shape[1]} gene | nguồn: {src} | {n_cls} nhóm | "
            f"count trung vị {np.median(obs['total_counts']):.0f}, gene trung vị {np.median(ng):.0f}")
        R = {"n_cells": int(X.shape[0]), "n_genes_panel": int(X.shape[1]), "source": src, "n_classes": int(n_cls),
             "chance": round(1 / n_cls, 4), "median_counts": float(np.median(obs["total_counts"])),
             "median_genes": float(np.median(ng)), "reps": {}}
        P, n_h = pca_expr(X); R["pca_hvg"] = n_h
        reps = {"depth_only": (depth_only(obs), False), "pca": (P, True), "pca_euclid": (P, False)}
        for m in ["scgpt", "niche"]:
            E = load_emb(m, assay, obs.index)
            if E is not None: reps[m] = (E, True)
        for name, (F, unit) in reps.items():
            met, pc = knn_eval(F, lab, dlog, unit=unit)
            if name != "depth_only": met["rho_ngenes_pc1_5"] = pca_depth(F, ng)
            if name in ("scgpt", "niche"): met["reported_03"] = reported_acc(name, assay)
            R["reps"][name] = met
            per_class.append(pc.assign(assay=assay, rep=name))
            log(f"  {name:11s} acc={met['acc']:.3f} | depth_nn_rho={met['depth_nn_rho']:+.3f} "
                f"within={met['depth_nn_rho_within']:+.3f}"
                + (f" | |rho| n_genes PC1-5 {met['rho_ngenes_pc1_5']}" if "rho_ngenes_pc1_5" in met else "")
                + (f" | 03 báo cáo {met['reported_03']}" if met.get("reported_03") is not None else ""))
        # thành phần cell type: tập con (sau lọc) và toàn mô (chỉ high_quality)
        sub_c = pd.Series(lab).value_counts(normalize=True); full_c = full_composition(assay)
        for c in sub_c.index.union(full_c.index if full_c is not None else pd.Index([])):
            comp_rows.append({"assay": assay, "class": c, "frac_subset": round(float(sub_c.get(c, 0)), 4),
                              "frac_tissue_hq": round(float(full_c.get(c, 0)), 4) if full_c is not None else None})
        # bàn cờ (chỉ assay không gian)
        if assay in SPATIAL and "x" in obs:
            org = full_origin(assay) or (float(obs["x"].min()), float(obs["y"].min()), "tập con")
            ix = np.floor((obs["x"].to_numpy() - org[0]) / TILE_UM).astype(int)
            iy = np.floor((obs["y"].to_numpy() - org[1]) / TILE_UM).astype(int)
            split = np.where((ix + iy) % 2 == 0, "T", "H")
            pd.DataFrame({"x": obs["x"], "y": obs["y"], "tile_ix": ix, "tile_iy": iy, "split": split},
                         index=obs.index).to_csv(f"{OUT}/split_{assay}.csv")
            fT = pd.Series(lab[split == "T"]).value_counts(normalize=True)
            fH = pd.Series(lab[split == "H"]).value_counts(normalize=True)
            dmax = float((fT.reindex(fT.index.union(fH.index), fill_value=0)
                          - fH.reindex(fT.index.union(fH.index), fill_value=0)).abs().max())
            S["split"][assay] = {"origin": org[2], "n_T": int((split == "T").sum()), "n_H": int((split == "H").sum()),
                                 "n_tiles": int(len(set(zip(ix, iy)))), "max_abs_diff_class_frac": round(dmax, 4)}
            log(f"  bàn cờ {TILE_UM:.0f} µm (gốc: {org[2]}): T={S['split'][assay]['n_T']} H={S['split'][assay]['n_H']} | "
                f"{S['split'][assay]['n_tiles']} ô | lệch thành phần T/H lớn nhất {dmax:.3f}")
        S["assays"][assay] = R
        cache[assay] = {"X": X, "obs": obs, "var": var}
        log(f"  ({time.time() - t0:.0f}s)")
        json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)

    # ------------------------------------------------------------------ A2. gene chung + cùng độ sâu
    log("\n" + "=" * 90 + "\nA2. Phân rã: gene chung của mọi assay -> gene chung + cùng độ sâu")
    first = {}
    for a in ASSAYS:
        s = cache[a]["var"]["symbol"].str.upper().to_numpy()
        first[a] = pd.Series(np.arange(len(s)), index=s)[~pd.Index(s).duplicated()]
    G = sorted(set.intersection(*[set(first[a].index) for a in ASSAYS]))
    log(f"  gene chung của 6 assay: {len(G)}")
    Xs = {a: cache[a]["X"][:, first[a].loc[G].to_numpy()].tocsr() for a in ASSAYS}
    tcs = {a: np.asarray(Xs[a].sum(1)).ravel() for a in ASSAYS}
    D = int(max(10, min(np.quantile(tcs[a], DEPTH_Q) for a in SPATIAL)))
    log(f"  độ sâu chung D = {D} count trên gene chung (min qua assay không gian của phân vị {DEPTH_Q:.0%})")
    S["matched"] = {"n_shared_genes": len(G), "D": D, "assays": {}}
    for a in ASSAYS:
        obs = cache[a]["obs"]; lab = obs["label"].to_numpy()
        Ps, _ = pca_expr(Xs[a])
        m1, pc1 = knn_eval(Ps, lab, np.log1p(tcs[a]))
        Y, keep = thin_to(Xs[a], D, rng)
        Pd, _ = pca_expr(Y)
        # sau khi thinning mọi tế bào có đúng D count -> không còn biến thiên độ sâu để đo; dùng độ sâu gốc
        m2, pc2 = knn_eval(Pd, lab[keep], np.log1p(tcs[a][keep]))
        # đối chứng: PCA gene chung, không thinning, nhưng CHỈ trên các tế bào giữ lại (tách hiệu ứng chọn tế bào)
        m3, _ = knn_eval(Ps[keep], lab[keep], np.log1p(tcs[a][keep]))
        S["matched"]["assays"][a] = {"shared_median_counts": float(np.median(tcs[a])), "frac_kept": round(float(len(keep) / len(lab)), 3),
                                     "pca_shared": m1, "pca_shared_keptcells": m3, "pca_shared_depthD": m2}
        per_class.append(pc1.assign(assay=a, rep="pca_shared")); per_class.append(pc2.assign(assay=a, rep="pca_shared_depthD"))
        log(f"  [{a:13s}] count trung vị trên gene chung {np.median(tcs[a]):7.0f} | giữ {len(keep)/len(lab):5.1%} tế bào | "
            f"acc gene chung {m1['acc']:.3f} -> cùng tế bào giữ lại {m3['acc']:.3f} -> + cùng độ sâu {m2['acc']:.3f}")
    json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)

    # ------------------------------------------------------------------ B. mất count khi ánh xạ symbol (Nicheformer)
    log("\n" + "=" * 90 + "\nB. Count bị mất khi ánh xạ symbol -> Ensembl (Nicheformer)")
    ref = niche_ref_ids()
    if ref is None:
        log("  không lấy được model.h5ad của Nicheformer (cần Internet ON) -> bỏ qua phần B")
    else:
        sym2ens = {}
        for a in ["xenium5k", "visiumhd_ffpe", "visiumhd_ff"]:          # đúng bảng 03b đã dùng
            v = cache[a]["var"]
            if "ensembl" in v:
                for s, e in zip(v["symbol"].str.upper(), v["ensembl"].astype(str).str.split(".").str[0]):
                    sym2ens.setdefault(s, e)
        refset = set(ref)
        log(f"  vocab Nicheformer {len(ref)} gene | bảng symbol->Ensembl của 03b: {len(sym2ens)} gene")
        lost_all, tables = {}, {}
        for a in SYMBOL_ASSAYS:
            v = cache[a]["var"]; sym = v["symbol"].str.upper().to_numpy()
            tot = np.asarray(cache[a]["X"].sum(0)).ravel()
            ens = np.array([sym2ens.get(s, "") for s in sym]); mapped = np.array([e in refset for e in ens])
            T = pd.DataFrame({"symbol": v["symbol"].to_numpy(), "SYM": sym, "count": tot, "mapped": mapped, "ens": ens})
            T["frac_of_total"] = T["count"] / max(T["count"].sum(), 1)
            L = T[~T["mapped"] & (T["count"] > 0)].copy()
            L["category"] = [lost_category(s) for s in L["SYM"]]
            tables[a] = (T, L)
            for s, orig in zip(L["SYM"], L["symbol"]): lost_all.setdefault(s, orig)
        rescue = mygene_lookup(sorted(set(lost_all.values())), ref) if lost_all else None
        ext_rows = []
        for a in SYMBOL_ASSAYS:
            T, L = tables[a]
            used = set(T.loc[T["mapped"], "ens"])
            L["rescue_ens"], L["rescue_how"] = None, "không tìm thấy"
            if rescue is not None:
                seen = {}
                for i, s in zip(L.index, L["SYM"]):
                    e, how = rescue.get(s, (None, "không tìm thấy"))
                    if e is not None and (e in used or e in seen):
                        how, e = "trùng gene đã có", None
                    if e is not None: seen[e] = s
                    L.at[i, "rescue_ens"], L.at[i, "rescue_how"] = e, how
                for s, e, how in zip(L["SYM"], L["rescue_ens"], L["rescue_how"]):
                    if e is not None: ext_rows.append({"symbol": s, "ensembl": e, "how": how, "assay": a})
            kept = float(T.loc[T["mapped"], "frac_of_total"].sum())
            resc = float(L.loc[L["rescue_ens"].notna(), "frac_of_total"].sum()) if rescue is not None else None
            by_cat = L.groupby("category")["frac_of_total"].sum().sort_values(ascending=False)
            by_how = L.groupby("rescue_how")["frac_of_total"].sum().sort_values(ascending=False)
            S["mapping"][a] = {"count_frac_kept_03b": round(kept, 4), "count_frac_rescuable": None if resc is None else round(resc, 4),
                               "lost_by_category": by_cat.round(4).to_dict(), "lost_by_rescue_status": by_how.round(4).to_dict(),
                               "n_lost_genes": int(len(L))}
            L.sort_values("count", ascending=False).drop(columns=["SYM", "mapped", "ens"]).to_csv(f"{OUT}/lost_genes_{a}.csv", index=False)
            log(f"\n  [{a}] giữ lại {kept:.1%} count (03b) | mất {1-kept:.1%} ở {len(L)} gene"
                + (f" | cứu được bằng mygene: +{resc:.1%} -> {kept+resc:.1%}" if resc is not None else ""))
            log("    mất theo loại (% tổng count): " + " | ".join(f"{k}: {v:.1%}" for k, v in by_cat.items()))
            if rescue is not None:
                log("    mất theo khả năng cứu:        " + " | ".join(f"{k}: {v:.1%}" for k, v in by_how.items()))
            top = L.sort_values("count", ascending=False).head(15)
            log("    15 gene mất nhiều count nhất: " + ", ".join(
                f"{s} {f:.1%}{'*' if e is not None else ''}" for s, f, e in zip(top["symbol"], top["frac_of_total"], top["rescue_ens"])))
        if ext_rows:
            pd.DataFrame(ext_rows).drop_duplicates(["symbol", "ensembl"]).to_csv(f"{OUT}/sym2ens_rescue.csv", index=False)
            log(f"\n  bảng ánh xạ bổ sung: {OUT}/sym2ens_rescue.csv (dấu * ở trên = cứu được)")
    json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)
    pd.concat(per_class).to_csv(f"{OUT}/knn_per_class.csv", index=False)
    pd.DataFrame(comp_rows).to_csv(f"{OUT}/composition.csv", index=False)

    # ------------------------------------------------------------------ báo cáo gọn
    lines = ["BẢNG 1 — kNN acc (balanced, 15-NN, 20k tế bào) trên cùng tế bào; 'chance' = 1/số nhóm"]
    rows = []
    for a in ASSAYS:
        R = S["assays"][a]; M = S["matched"]["assays"].get(a, {})
        g = lambda k: R["reps"].get(k, {}).get("acc")
        rows.append({"assay": a, "chance": R["chance"], "depth_only": g("depth_only"), "PCA": g("pca"), "PCA_euclid": g("pca_euclid"),
                     "scGPT-sp": g("scgpt"), "Niche": g("niche"),
                     "PCA_shared": M.get("pca_shared", {}).get("acc"), "PCA_shared_kept": M.get("pca_shared_keptcells", {}).get("acc"),
                     "PCA_shared_D": M.get("pca_shared_depthD", {}).get("acc"), "kept%": None if not M else round(100 * M["frac_kept"], 1)})
    lines.append(pd.DataFrame(rows).to_string(index=False))
    lines.append(f"(gene chung: {S['matched']['n_shared_genes']} | D = {S['matched']['D']} count)")
    lines.append("\nBẢNG 2 — độ sâu: depth_nn_rho_within (độ sâu ngoài cell type được hàng xóm 'nhớ') | |rho| n_genes với PC1")
    rows = []
    for a in ASSAYS:
        rr = S["assays"][a]["reps"]; row = {"assay": a}
        for k, lab in [("pca", "PCA"), ("scgpt", "scGPT-sp"), ("niche", "Niche")]:
            if k in rr: row[f"{lab} within"] = rr[k]["depth_nn_rho_within"]; row[f"{lab} PC1"] = rr[k]["rho_ngenes_pc1_5"][0]
        row["03 báo cáo (scGPT/Niche)"] = f"{rr.get('scgpt', {}).get('reported_03')}/{rr.get('niche', {}).get('reported_03')}"
        rows.append(row)
    lines.append(pd.DataFrame(rows).to_string(index=False))
    pc = pd.concat(per_class)
    piv = pc[pc["rep"] == "pca"].pivot_table(index="class", columns="assay", values="recall")[ASSAYS]
    lines.append("\nBẢNG 3 — recall theo cell type, PCA biểu hiện (cột = assay)")
    lines.append(piv.round(2).to_string())
    for a in ["cosmx6k", "stereoseq"]:
        q = pc[pc["assay"] == a].pivot_table(index="class", columns="rep", values="recall")
        cols = [c for c in ["pca", "scgpt", "niche", "pca_shared", "pca_shared_depthD"] if c in q.columns]
        n = pc[(pc["assay"] == a) & (pc["rep"] == "pca")].set_index("class")["n"]
        lines.append(f"\nBẢNG 4 — {a}: recall theo cell type và biểu diễn (n = số tế bào trong 20k)")
        lines.append(q[cols].assign(n=n).round(2).to_string())
    if S["split"]:
        lines.append("\nBẢNG 5 — bàn cờ\n" + pd.DataFrame(S["split"]).T.to_string())
    if S["mapping"]:
        lines.append("\nBẢNG 6 — ánh xạ symbol cho Nicheformer\n" + pd.DataFrame(
            {a: {"kept_03b": v["count_frac_kept_03b"], "rescuable": v["count_frac_rescuable"], "n_lost_genes": v["n_lost_genes"]}
             for a, v in S["mapping"].items()}).T.to_string())
    rep = "\n".join(lines)
    open(f"{OUT}/report.txt", "w").write(rep)
    log("\n" + "=" * 90 + "\n" + rep + "\nXONG")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run04b.py
"""04b — Ba câu hỏi còn mở sau 04 (CPU).

  A. Đối chứng cùng gene + cùng độ sâu, NHIỀU mức D (04 chỉ thử D = 18, quá thấp).
     Mỗi mức D: giữ tế bào có >= D count trên gene chung, lấy mẫu con đúng D count, PCA, kNN acc trên 10k tế bào.
     Kèm bản 'native' (cùng tế bào được giữ, KHÔNG lấy mẫu con) để tách hiệu ứng chọn tế bào khỏi hiệu ứng độ sâu.
  B. Nhãn có được biểu hiện gene marker ủng hộ không?
     AUROC của điểm marker (tế bào mang nhãn c so với phần còn lại), và mức 'tràn' marker NHU MÔ (Epithelial, hoặc Hepatocyte ở gan) sang các nhóm khác
     (dấu hiệu segmentation trộn transcript của tế bào bên cạnh).
  C. Tập nhóm 'khôi phục được' cho từng assay: chọn trên ô T (PCA, recall >= TAU, n >= 50), đánh giá trên ô H.
     Cho ra thước đo dùng ở 03c: acc trên 16 nhóm, acc trên tập khôi phục được, acc trên 6 nhóm thô.

Dùng lại toàn bộ hàm đọc dữ liệu / PCA / lấy mẫu con của run04.py (đã kiểm chứng ở 04).
"""
import os, sys, json, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import run04 as R
from run04 import (np, pd, sp, NearestNeighbors, ASSAYS, SPATIAL, K_NN, N_MAX, SEED, TILE_UM, log,
                   load_assay, load_emb, pca_expr, lognorm, thin_to, full_origin, _unit)
from sklearn.metrics import roc_auc_score

OUT = os.environ.get("OUT_DIR_B", "/kaggle/working/out04b")
D_LEVELS = [int(x) for x in os.environ.get("D_LEVELS", "20,40,80,160,320").split(",")]
N_EVAL = int(os.environ.get("N_EVAL", 10000))     # cùng số tế bào cho mọi assay ở phần A (kNN acc phụ thuộc mật độ)
MIN_KEEP = int(os.environ.get("MIN_KEEP", 3000))  # ít hơn số tế bào này thì không đánh giá mức D đó
TAU = float(os.environ.get("TAU", 0.5))
MIN_N_T = 50

# 6 nhóm thô: đủ thô để mọi công nghệ có cơ hội phân biệt, đủ mịn để còn ý nghĩa sinh học
COARSE = {"Epithelial": "Epithelial", "Fibroblast": "Stromal", "SMC": "Stromal", "Endothelial": "Endothelial",
          "Plasma": "Plasma", "B": "Lymphoid", "CD4T": "Lymphoid", "CD8T": "Lymphoid", "Tprolif": "Lymphoid",
          "Macrophage": "Myeloid", "Monocyte": "Myeloid", "Neutrophil": "Myeloid", "Mast": "Myeloid",
          "cDC2": "Myeloid", "mregDC": "Myeloid", "pDC": "Myeloid"}
if os.environ.get("TISSUE", "coad").lower() != "coad":   # vòng 26: nhãn mới ở OV/HCC (COAD giữ nguyên 16 nhóm)
    COARSE.update({"NK": "Lymphoid", "DC": "Myeloid", "Hepatocyte": "Epithelial", "Kupffer": "Myeloid"})
# Marker kinh điển; chỉ gene có trong panel của assay mới được dùng (số gene dùng được in ra)
MARKERS = {
    "Epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT20", "CDH1", "CEACAM5"],
    "Fibroblast": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRA"],
    "SMC": ["ACTA2", "MYH11", "DES", "CNN1", "TAGLN"],
    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "PLVAP"],
    "Plasma": ["JCHAIN", "MZB1", "IGHA1", "IGKC", "SDC1", "XBP1"],
    "B": ["MS4A1", "CD79A", "CD79B", "CD19", "BANK1"],
    "CD4T": ["CD3D", "CD3E", "CD4", "IL7R", "CD40LG"],
    "CD8T": ["CD3D", "CD3E", "CD8A", "CD8B", "GZMK"],
    "Tprolif": ["MKI67", "TOP2A", "CD3E", "CD3D"],
    "Macrophage": ["CD68", "CD163", "C1QA", "C1QB", "APOE", "MRC1"],
    "Monocyte": ["CD14", "FCN1", "LYZ", "VCAN", "S100A8"],
    "Neutrophil": ["FCGR3B", "CSF3R", "CXCR2", "S100A8", "S100A9"],
    "Mast": ["TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2"],
    "cDC2": ["CD1C", "FCER1A", "CLEC10A"],
    "mregDC": ["LAMP3", "CCR7", "FSCN1", "CCL22"],
    "pDC": ["LILRA4", "CLEC4C", "IL3RA"],
}
if os.environ.get("TISSUE", "coad").lower() != "coad":
    MARKERS.update({"NK": ["NKG7", "GNLY", "KLRD1", "NCAM1", "PRF1"], "DC": ["CD1C", "FCER1A", "CLEC10A", "LAMP3", "CCR7"],
                    "Hepatocyte": ["ALB", "APOA1", "TTR", "HP", "CYP3A4"], "Kupffer": ["CD5L", "VSIG4", "MARCO", "CD163", "TIMD4"]})


def to_coarse(lab):
    return np.array([COARSE.get(x, "Other") for x in lab])

def neighbors(F, idx, unit=True, k=K_NN):
    A = _unit(F[idx]) if unit else np.asarray(F[idx], dtype=np.float64)
    return NearestNeighbors(n_neighbors=k + 1, n_jobs=int(os.environ.get("NN_JOBS", 4))).fit(A).kneighbors(A, return_distance=False)[:, 1:]

def vote_recall(nn, lab):
    """Đa số 15 hàng xóm (hoà -> nhãn nhỏ nhất theo chữ cái, như 03/04). Trả về recall và số tế bào theo nhóm."""
    classes, codes = np.unique(np.asarray(lab).astype(str), return_inverse=True)
    cnt = np.zeros((len(lab), len(classes)))
    np.add.at(cnt, (np.repeat(np.arange(len(lab)), nn.shape[1]), codes[nn].ravel()), 1)
    pred = cnt.argmax(1)
    n = np.bincount(codes, minlength=len(classes))
    rec = np.array([(pred[codes == i] == i).mean() if n[i] else np.nan for i in range(len(classes))])
    return pd.Series(rec, index=classes), pd.Series(n, index=classes)

def pick(idx, n_max, seed=0):
    return idx if len(idx) <= n_max else np.sort(idx[np.random.RandomState(seed).choice(len(idx), n_max, replace=False)])

def make_split(obs, assay):
    if assay in SPATIAL and "x" in obs:
        org = full_origin(assay) or (float(obs["x"].min()), float(obs["y"].min()), "tập con")
        ix = np.floor((obs["x"].to_numpy() - org[0]) / TILE_UM).astype(int)
        iy = np.floor((obs["y"].to_numpy() - org[1]) / TILE_UM).astype(int)
        return np.where((ix + iy) % 2 == 0, "T", "H"), "bàn cờ"
    rs = np.random.RandomState(SEED)                        # scRNA-seq: không có toạ độ -> chia ngẫu nhiên 50/50
    return np.where(rs.rand(len(obs)) < 0.5, "T", "H"), "ngẫu nhiên"

def marker_table(X, var, lab):
    """AUROC(điểm marker của nhóm c; nhãn == c) + điểm marker biểu mô trung bình theo nhóm (để đo 'tràn')."""
    Y = lognorm(X)
    sym = var["symbol"].str.upper().to_numpy()
    pos = pd.Series(np.arange(len(sym)), index=sym)[~pd.Index(sym).duplicated()]
    scores, used = {}, {}
    for c, genes in MARKERS.items():
        g = [pos[x] for x in genes if x in pos.index]
        used[c] = [x for x in genes if x in pos.index]
        scores[c] = np.asarray(Y[:, g].mean(1)).ravel() if g else None
    auc = {}
    for c in np.unique(lab):
        y = lab == c
        if c in scores and scores[c] is not None and 0 < y.sum() < len(y):
            auc[c] = round(float(roc_auc_score(y, scores[c])), 3)
    # Lớp nhu mô làm mốc đo "tràn marker": COAD/OV = Epithelial, HCC = Hepatocyte (bản trước đặt cứng "Epithelial"
    # nên BẢNG B3 của HCC rỗng). Chọn: nhãn CÓ điểm marker, CÓ mặt trong lát, và thuộc nhóm thô "Epithelial";
    # nhiều tế bào nhất thì thắng. PARENCHYMA_CLASS=... để ép tay. Với COAD/OV kết quả không đổi.
    present = set(np.unique(lab))
    forced = os.environ.get("PARENCHYMA_CLASS")
    cand = [c for c, v in scores.items() if v is not None and c in present
            and (c == forced if forced else COARSE.get(c) == "Epithelial")]
    par = max(cand, key=lambda c: int((lab == c).sum())) if cand else None
    spill = {}
    if par is not None:
        base = scores[par]; ref = base[lab == par].mean()
        for c in np.unique(lab):
            spill[c] = round(float(base[lab == c].mean() / max(ref, 1e-12)), 3)
    return auc, spill, used, par


def main():
    os.makedirs(OUT, exist_ok=True)
    for m, d in list(R.EMB_DIRS.items()):
        if not d or not os.path.isdir(d):
            hits = sorted(R.glob.glob(f"/kaggle/input/**/emb_{m}", recursive=True))
            R.EMB_DIRS[m] = hits[0] if hits else None
    log(f"LITE={R.LITE} | embedding: {R.EMB_DIRS} | D = {D_LEVELS} | TAU = {TAU}")
    rng = np.random.default_rng(SEED)
    S = {"coarse_map": COARSE, "tau": TAU, "B_markers": {}, "C_recoverable": {}, "C_eval_H": {}, "A_depth": []}
    cache = {}

    # ------------------------------------------------------------------ B + C (từng assay)
    for a in ASSAYS:
        t0 = time.time()
        X, obs, var, src = load_assay(a)
        lab = obs["label"].to_numpy().astype(str); lab_c = to_coarse(lab)
        unk = sorted(set(lab[lab_c == "Other"]))
        if unk: log(f"  [{a}] CẢNH BÁO: nhãn không có trong bảng nhóm thô: {unk}")
        cache[a] = {"X": X, "var": var, "lab": lab, "lab_c": lab_c}
        log(f"\n[{a}] {X.shape[0]} tế bào | nguồn {src}")

        # B. marker
        auc, spill, used, par = marker_table(X, var, lab)
        S["B_markers"][a] = {"auroc": auc, "epi_spillover": spill, "parenchyma_class": par,
                             "markers_used": {c: len(v) for c, v in used.items()}}
        log("  AUROC marker: " + " ".join(f"{c}={v:.2f}({len(used.get(c, []))})" for c, v in sorted(auc.items())))

        # C. tập khôi phục được: chọn trên T bằng PCA, đánh giá trên H
        split, how = make_split(obs, a)
        P, _ = pca_expr(X)
        T, H = pick(np.where(split == "T")[0], N_MAX), pick(np.where(split == "H")[0], N_MAX)
        recT, nT = vote_recall(neighbors(P, T), lab[T])
        Sset = sorted(c for c in recT.index if recT[c] >= TAU and nT[c] >= MIN_N_T)
        S["C_recoverable"][a] = {"split": how, "classes": Sset, "recall_T_pca": recT.round(3).to_dict(),
                                 "n_T": nT.to_dict()}
        log(f"  khôi phục được (PCA trên T, recall >= {TAU}): {len(Sset)}/{len(recT)} nhóm: {Sset}")
        reps = {"pca": P}
        for m in ["scgpt", "niche"]:
            E = load_emb(m, a, obs.index)
            if E is not None: reps[m] = E
        S["C_eval_H"][a] = {}
        for name, F in reps.items():
            nn = neighbors(F, H)
            recH, _ = vote_recall(nn, lab[H]); recHc, _ = vote_recall(nn, lab_c[H])
            res = {"acc16": round(float(recH.mean()), 4),
                   "accS": round(float(recH.reindex(Sset).mean()), 4) if Sset else None,
                   "acc_coarse": round(float(recHc.mean()), 4),
                   "recall_H": recH.round(3).to_dict(), "recall_coarse_H": recHc.round(3).to_dict()}
            if name == "pca":
                res["T_vs_H_abs_diff_recall_S"] = round(float((recT.reindex(Sset) - recH.reindex(Sset)).abs().mean()), 3) if Sset else None
            S["C_eval_H"][a][name] = res
            log(f"  H | {name:6s} acc16={res['acc16']:.3f} accS={res['accS'] if res['accS'] is None else round(res['accS'], 3)} "
                f"acc_coarse={res['acc_coarse']:.3f}")
        log(f"  ({time.time() - t0:.0f}s)")
        json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)

    json.dump({a: v["classes"] for a, v in S["C_recoverable"].items()} | {"_coarse_map": COARSE, "_tau": TAU},
              open(f"{OUT}/recoverable_classes.json", "w"), indent=2, ensure_ascii=False)

    # ------------------------------------------------------------------ A. cùng gene + nhiều mức độ sâu
    log("\n" + "=" * 90 + "\nA. Gene chung + cùng độ sâu, nhiều mức D")
    first = {}
    for a in ASSAYS:
        s = cache[a]["var"]["symbol"].str.upper().to_numpy()
        first[a] = pd.Series(np.arange(len(s)), index=s)[~pd.Index(s).duplicated()]
    G = sorted(set.intersection(*[set(first[a].index) for a in ASSAYS]))
    log(f"  gene chung: {len(G)} | đánh giá trên {N_EVAL} tế bào mỗi ô (cùng mật độ cho mọi assay)")
    for a in ASSAYS:
        t0 = time.time()
        Xs = cache[a]["X"][:, first[a].loc[G].to_numpy()].tocsr()
        tc = np.asarray(Xs.sum(1)).ravel(); lab, lab_c = cache[a]["lab"], cache[a]["lab_c"]
        for D in D_LEVELS:
            kept = int((tc >= D).sum())
            row = {"assay": a, "D": D, "kept_frac": round(kept / len(tc), 3)}
            if kept < MIN_KEEP:
                S["A_depth"].append(row | {"note": f"< {MIN_KEEP} tế bào"}); continue
            Y, keep = thin_to(Xs, D, rng)
            sub = pick(np.arange(len(keep)), N_EVAL)
            for kind, M in [("thinned", Y), ("native", Xs[keep])]:
                Pk, _ = pca_expr(M)
                nn = neighbors(Pk, sub)
                r16, _ = vote_recall(nn, lab[keep][sub]); rc, _ = vote_recall(nn, lab_c[keep][sub])
                row[f"acc16_{kind}"] = round(float(r16.mean()), 4); row[f"acc_coarse_{kind}"] = round(float(rc.mean()), 4)
            S["A_depth"].append(row)
            log(f"  [{a:13s}] D={D:4d} giữ {row['kept_frac']:6.1%} | acc16 {row['acc16_native']:.3f} -> {row['acc16_thinned']:.3f} | "
                f"thô {row['acc_coarse_native']:.3f} -> {row['acc_coarse_thinned']:.3f}  (native -> cùng độ sâu)")
        log(f"  ({time.time() - t0:.0f}s)")
        json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)

    # ------------------------------------------------------------------ báo cáo
    L = []
    A = pd.DataFrame(S["A_depth"])
    def grid(col):
        g = A.pivot_table(index="assay", columns="D", values=col, aggfunc="first").reindex(ASSAYS)
        k = A.pivot_table(index="assay", columns="D", values="kept_frac", aggfunc="first").reindex(ASSAYS)
        return g.round(3).astype(str) + " (" + (100 * k).round(0).astype("Int64").astype(str) + "%)"
    for col, title in [("acc16_thinned", "acc 16 nhóm, CÙNG ĐỘ SÂU D"), ("acc_coarse_thinned", "acc 6 nhóm thô, CÙNG ĐỘ SÂU D"),
                       ("acc16_native", "acc 16 nhóm, cùng tế bào được giữ nhưng KHÔNG lấy mẫu con (tách hiệu ứng chọn tế bào)")]:
        if col in A:
            L.append(f"BẢNG A — {title}; ngoặc = % tế bào có >= D count trên {len(G)} gene chung")
            L.append(grid(col).to_string()); L.append("")
    auc = pd.DataFrame({a: S["B_markers"][a]["auroc"] for a in ASSAYS})
    L.append("BẢNG B1 — AUROC của điểm marker (nhãn c so với phần còn lại); 0.5 = marker không ủng hộ nhãn")
    L.append(auc.round(2).to_string()); L.append("")
    nm = pd.DataFrame({a: S["B_markers"][a]["markers_used"] for a in ASSAYS})
    L.append("BẢNG B2 — số gene marker có trong panel"); L.append(nm.to_string()); L.append("")
    sp_ = pd.DataFrame({a: S["B_markers"][a]["epi_spillover"] for a in ASSAYS})
    par_of = {a: (S["B_markers"][a].get("parenchyma_class") or "—") for a in ASSAYS}
    L.append("BẢNG B3 — điểm marker NHU MÔ của từng nhóm / của chính nhóm nhu mô (tràn marker; 1.0 = ngang nhóm nhu mô)")
    L.append("  lớp nhu mô dùng làm mốc theo từng assay: " + ", ".join(f"{a}={par_of[a]}" for a in ASSAYS)
             + "  ('—' = panel không có gene marker của lớp đó, ví dụ Hepatocyte không có trong panel Xenium 5K)")
    L.append(sp_.round(2).to_string()); L.append("")
    L.append(f"BẢNG C1 — nhóm khôi phục được (PCA trên T, recall >= {TAU}, n >= {MIN_N_T})")
    xs = set(S["C_recoverable"].get("xenium5k", {}).get("classes", []))
    for a in ASSAYS:
        c = S["C_recoverable"][a]["classes"]
        L.append(f"  {a:13s} {len(c):2d} nhóm | chung với Xenium: {len(set(c) & xs):2d} | {c}")
    rows = []
    for a in ASSAYS:
        for m, v in S["C_eval_H"][a].items():
            rows.append({"assay": a, "rep": m, "acc16": v["acc16"], "accS": v["accS"], "acc_coarse": v["acc_coarse"],
                         "|T-H| recall": v.get("T_vs_H_abs_diff_recall_S")})
    L.append("\nBẢNG C2 — đánh giá trên ô H (kNN chỉ trong H)")
    L.append(pd.DataFrame(rows).pivot_table(index="assay", columns="rep", values=["acc16", "accS", "acc_coarse"], aggfunc="first")
             .reindex(ASSAYS).round(3).to_string())
    rc = pd.DataFrame({a: S["C_eval_H"][a]["pca"]["recall_coarse_H"] for a in ASSAYS})
    L.append("\nBẢNG C3 — recall 6 nhóm thô trên H, PCA"); L.append(rc.round(2).to_string())
    rep = "\n".join(L)
    open(f"{OUT}/report.txt", "w").write(rep)
    log("\n" + "=" * 90 + "\n" + rep + "\nXONG")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run05.py
"""05 — USHER (code src_cuda của bạn) trên cặp chính, so với các baseline, đánh giá trên ô H (GPU T4).

Cặp: CosMx -> Xenium (chính), VHD FFPE -> Xenium (đối chứng âm: dư địa ≈ 0, USHER không được làm tệ đi).
Embedding: scGPT-spatial (03a) và Nicheformer quy tắc `data` (03d).

Giao thức chống vòng tròn:
  * USHER và mọi baseline dùng nhãn CHỈ thấy tế bào source ở ô T (nhãn + embedding). Target = Xenium (nhãn được dùng,
    như khi dùng thật: target là tham chiếu đã chú giải).
  * Đánh giá trên tế bào source ở ô H, nhãn H không bao giờ được dùng khi fit.
  * Thước đo giống hệt P3 của 03d (cùng cách lấy mẫu, cùng tham chiếu Xenium) -> hàng `raw` phải tái lập đúng số 03d.

Biểu diễn so sánh (mỗi cái là một map source -> không gian Xenium):
  raw            : không căn chỉnh
  center         : dịch trung bình (source T -> Xenium), không dùng nhãn
  class_shift    : dịch trung bình ĐÃ CÂN THÀNH PHẦN (trung bình các chênh lệch tâm từng cell type, trọng số theo T) — dùng nhãn
  class_ridge    : map tuyến tính 'gần identity' (x + ridge(x)), fit trên cặp (tế bào T, tế bào Xenium ngẫu nhiên cùng nhãn) — dùng nhãn
  usher          : USHER cấu hình mặc định của bạn (FGW, gamma=0.4, alpha=0.3, MLP hidden 512)
  (ablation chỉ ở CosMx/scGPT) usher_linear (M-step tuyến tính), usher_labels_only (gamma=1), usher_no_label (nhãn đồng nhất)
Thước đo: assay_sep (0.5 = trộn), xfer_S (chuyển nhãn Xenium -> source H, nhóm khôi phục được ở cả hai), xfer_thô,
          keep_struct = kNN-Jaccard(15, Euclid) của source H trước/sau map (phép dịch = 1.0; thấp = map bóp méo cấu trúc).
"""
import os, sys, glob, json, time, gc, logging
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B
from run04 import np, pd, log, load_assay, ASSAYS, _unit
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import RidgeCV

OUT = os.environ.get("OUT_DIR_5", "/kaggle/working/out05")
TARGET = "xenium5k"
N_REF = int(os.environ.get("N_REF", 20000))
N_MIX = int(os.environ.get("N_MIX", 6000))
N_TGT = int(os.environ.get("N_TGT", 20000))          # số tế bào Xenium làm target cho USHER (bộ nhớ T_full = n_source x n_target)
N_ITERS = int(os.environ.get("N_ITERS", 20))
SKETCH = int(os.environ.get("SKETCH", 2000))
RESUME = os.environ.get("RESUME", "1") == "1"
MAIN = {"gamma": 0.4, "alpha": 0.3, "hidden_dim": 512, "labels": "onehot"}
RUNS = [("cosmx6k", "scgpt", "usher", MAIN), ("cosmx6k", "niche_data", "usher", MAIN),
        ("visiumhd_ffpe", "scgpt", "usher", MAIN), ("visiumhd_ffpe", "niche_data", "usher", MAIN),
        ("cosmx6k", "scgpt", "usher_linear", {**MAIN, "hidden_dim": None}),
        ("cosmx6k", "scgpt", "usher_labels_only", {**MAIN, "gamma": 1.0}),
        ("cosmx6k", "scgpt", "usher_no_label", {**MAIN, "labels": "uniform"})]
ONLY = os.environ.get("ONLY_RUNS", "")                # ví dụ "0,1" để chỉ chạy vài lần
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def find_dir(name):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)); return h[0] if h else None

def load_emb(d, assay, ids):
    E = np.load(f"{d}/{assay}_final.npy").astype(np.float32)
    eid = pd.read_csv(f"{d}/{assay}_obs.csv", index_col=0, dtype=str).index.astype(str)
    pos = pd.Index(eid).get_indexer(ids); assert (pos >= 0).all(), f"{d}/{assay}: thiếu tế bào"
    return E[pos]

def maj_vote(nb, L):
    classes, codes = np.unique(L.astype(str), return_inverse=True)
    cnt = np.zeros((len(nb), len(classes)))
    np.add.at(cnt, (np.repeat(np.arange(len(nb)), nb.shape[1]), codes[nb].ravel()), 1)
    return classes[cnt.argmax(1)]

def bacc_on(y, p, keep_classes):
    m = np.isin(y, list(keep_classes))
    return round(float(balanced_accuracy_score(y[m], p[m])), 4) if m.sum() else None

def cross_assay(src, D, reps, S_shared):
    """Chép nguyên từ 03d (cùng seed, cùng lấy mẫu) để số liệu so sánh được trực tiếp."""
    rng = np.random.RandomState(0)
    ds, dx = D[src], D[TARGET]
    cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
    shared_c = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]
    per = max(1, N_MIX // max(len(shared_c), 1))
    mix_s, mix_x = [], []
    for c in shared_c:
        k = min(per, cs[c], cx[c])
        mix_s.append(rng.choice(ds["H"][ds["lab_c"][ds["H"]] == c], k, replace=False))
        mix_x.append(rng.choice(np.where(dx["lab_c"] == c)[0], k, replace=False))
    mix_s, mix_x = np.concatenate(mix_s), np.concatenate(mix_x)
    ref = np.sort(rng.choice(len(dx["lab"]), min(N_REF, len(dx["lab"])), replace=False))
    qS = ds["H"][np.isin(ds["lab"][ds["H"]], S_shared)]
    qC = ds["H"][np.isin(ds["lab_c"][ds["H"]], shared_c)]
    out = {}
    for name, R_ in reps.items():
        for centered in (False, True):
            Fs, Fx = _unit(R_[src]), _unit(R_[TARGET])
            if centered:
                Fs, Fx = _unit(Fs - Fs.mean(0)), _unit(Fx - Fx.mean(0))
            J = np.vstack([Fx[mix_x], Fs[mix_s]]); g = np.r_[np.zeros(len(mix_x), int), np.ones(len(mix_s), int)]
            nnJ = NearestNeighbors(n_neighbors=16, n_jobs=4).fit(J).kneighbors(J, return_distance=False)[:, 1:]
            sep = round(float(balanced_accuracy_score(g, (g[nnJ].mean(1) > 0.5).astype(int))), 3)
            nnR = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(Fx[ref])
            pS = maj_vote(nnR.kneighbors(Fs[qS], 15, return_distance=False), dx["lab"][ref]) if len(qS) else np.array([])
            pC = maj_vote(nnR.kneighbors(Fs[qC], 15, return_distance=False), dx["lab_c"][ref])
            out[f"{name}{'_centered' if centered else ''}"] = {
                "assay_sep": sep, "xfer_S": bacc_on(ds["lab"][qS], pS, S_shared) if len(qS) else None,
                "xfer_coarse": bacc_on(ds["lab_c"][qC], pC, shared_c)}
    return out

def keep_struct(before, after, idx, k=15):
    """Euclid trên vector gốc: phép dịch và phép quay giữ nguyên 1.0; chỉ biến dạng thật mới làm giảm."""
    A, Bm = np.asarray(before[idx], np.float64), np.asarray(after[idx], np.float64)
    ia = NearestNeighbors(n_neighbors=k + 1, n_jobs=4).fit(A).kneighbors(A, return_distance=False)[:, 1:]
    ib = NearestNeighbors(n_neighbors=k + 1, n_jobs=4).fit(Bm).kneighbors(Bm, return_distance=False)[:, 1:]
    return round(float(np.mean([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(ia, ib)])), 3)

def own_ceiling(F, H, lab, S):
    nn = B.neighbors(F, B.pick(H, R.N_MAX))
    rec, _ = B.vote_recall(nn, lab[B.pick(H, R.N_MAX)])
    return round(float(rec.reindex(S).mean()), 4)


# =============================================================================== baseline
def base_center(Fs, Fx, T):
    return Fs - Fs[T].mean(0) + Fx.mean(0)

def base_class_shift(Fs, Fx, T, lab_s, lab_x):
    cls = [c for c in np.unique(lab_s[T]) if (lab_s[T] == c).sum() >= 20 and (lab_x == c).sum() >= 20]
    w = np.array([(lab_s[T] == c).sum() for c in cls], float); w /= w.sum()
    d = sum(wi * (Fx[lab_x == c].mean(0) - Fs[T][lab_s[T] == c].mean(0)) for wi, c in zip(w, cls))
    return Fs + d

def base_class_ridge(Fs, Fx, T, lab_s, lab_x, n_pair=20000):
    rng = np.random.default_rng(0)
    src = rng.choice(T, min(n_pair, len(T)), replace=False)
    by = {c: np.where(lab_x == c)[0] for c in np.unique(lab_x)}
    src = np.array([i for i in src if lab_s[i] in by and len(by[lab_s[i]])])
    tgt = np.array([rng.choice(by[lab_s[i]]) for i in src])
    rg = RidgeCV(alphas=[1, 10, 100, 1000, 10000]).fit(Fs[src], Fx[tgt] - Fs[src])   # x + f(x): gần identity khi alpha lớn
    return Fs + rg.predict(Fs), float(rg.alpha_)


# =============================================================================== USHER
def run_usher(Fs, T, lab_s, Fx, lab_x, cfg):
    import torch, anndata as ad, run_usher as U
    logging.getLogger().setLevel(logging.WARNING)
    rng = np.random.RandomState(1)
    tgt = np.sort(rng.choice(len(Fx), min(N_TGT, len(Fx)), replace=False))
    classes = sorted(set(lab_s[T]) | set(lab_x[tgt]))
    def probs(lab):
        if cfg["labels"] == "uniform": return np.full((len(lab), len(classes)), 1.0 / len(classes), np.float32)
        P = np.zeros((len(lab), len(classes)), np.float32); P[np.arange(len(lab)), [classes.index(c) for c in lab]] = 1; return P
    a = ad.AnnData(X=Fs[T].astype(np.float32), obs=pd.DataFrame({"ct": lab_s[T]}, index=[f"s{i}" for i in T]))
    b = ad.AnnData(X=Fx[tgt].astype(np.float32), obs=pd.DataFrame({"ct": lab_x[tgt]}, index=[f"t{i}" for i in tgt]))
    a.obsm["X_celltype_probs"] = probs(lab_s[T]); b.obsm["X_celltype_probs"] = probs(lab_x[tgt])
    t0 = time.time()
    model, _, _, T_full, mean, std = U.align_features_fgw(
        a, b, e_step_method="fgw", m_step_method="global", sampling_strategy="celltype",
        sketch_size=SKETCH, sketch_obsm_key="X_none", use_stratified_pairing=True, cell_type_col="ct",
        celltype_probs_layer="X_celltype_probs", alpha=cfg["alpha"], gamma=cfg["gamma"],
        n_iters=N_ITERS, hidden_dim=cfg["hidden_dim"], device="cuda", verbose=False)
    del T_full; torch.cuda.empty_cache()
    with torch.no_grad():
        model.eval(); x = torch.from_numpy(Fs.astype(np.float32)).cuda()
        y = model((x - mean) / std) * std + mean if mean is not None else model(x)
        out = y.cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return out, time.time() - t0


# =============================================================================== chạy
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    EMB = {"scgpt": find_dir("emb_scgpt"), "niche_data": find_dir("emb_niche_data")}
    log(f"embedding: {EMB} | N_ITERS={N_ITERS} SKETCH={SKETCH} N_TGT={N_TGT}")
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    runs = [r for i, r in enumerate(RUNS) if not ONLY or str(i) in ONLY.split(",")]
    D, F = {}, {}
    for a in sorted({r[0] for r in runs} | {TARGET}):
        X, obs, var, _ = load_assay(a); del X
        lab = obs["label"].to_numpy().astype(str)
        split, _ = B.make_split(obs, a)
        D[a] = {"lab": lab, "lab_c": B.to_coarse(lab), "H": np.where(split == "H")[0], "T": np.where(split == "T")[0]}
        for m, d in EMB.items(): F[(a, m)] = load_emb(d, a, obs.index)
        log(f"[{a}] {len(lab)} tế bào | T={len(D[a]['T'])} H={len(D[a]['H'])}")
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    pairs = sorted({(r[0], r[1]) for r in runs})
    for src, m in pairs:
        key = f"{src}|{m}"; t0 = time.time()
        Fs, Fx = F[(src, m)], F[(TARGET, m)]
        ds, dx = D[src], D[TARGET]
        S_shared = sorted(set(REC[src]) & set(REC[TARGET]))
        log(f"\n=== {src} -> xenium | {m} | S chung = {S_shared}")
        maps = {"raw": Fs}
        step("baseline: center, class_shift, class_ridge")
        maps["center"] = base_center(Fs, Fx, ds["T"])
        maps["class_shift"] = base_class_shift(Fs, Fx, ds["T"], ds["lab"], dx["lab"])
        maps["class_ridge"], alpha_r = base_class_ridge(Fs, Fx, ds["T"], ds["lab"], dx["lab"])
        times = {}
        for (s_, m_, name, cfg) in runs:
            if (s_, m_) != (src, m): continue
            f = f"{OUT}/{src}_{m}_{name}.npy"
            if RESUME and os.path.exists(f):
                maps[name] = np.load(f).astype(np.float32); step(f"{name}: nạp lại từ lần chạy trước"); continue
            step(f"USHER {name} {cfg}")
            maps[name], times[name] = run_usher(Fs, ds["T"], ds["lab"], Fx, dx["lab"], cfg)
            np.save(f, maps[name].astype(np.float16))
            step(f"USHER {name} xong trong {times[name]:.0f}s")
        step("đánh giá")
        ev = cross_assay(src, D, {k: {src: v, TARGET: Fx} for k, v in maps.items()}, S_shared)
        Hs = B.pick(ds["H"], 10000)
        for k in maps: ev[k]["keep_struct"] = keep_struct(Fs, maps[k], Hs)
        S[key] = {"S_shared": S_shared, "ceiling_accS_source": own_ceiling(Fs, ds["H"], ds["lab"], S_shared),
                  "class_ridge_alpha": alpha_r, "usher_seconds": times, "eval": ev}
        log(f"  trần (kNN trong chính source, H) accS = {S[key]['ceiling_accS_source']}")
        for k, v in ev.items():
            if not k.endswith("_centered"):
                log(f"    {k:18s} assay_sep={v['assay_sep']} xfer_S={v['xfer_S']} xfer_thô={v['xfer_coarse']} keep_struct={v.get('keep_struct')}")
        json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)
        log(f"  ({time.time() - t0:.0f}s)")

    L = []
    for key, r in S.items():
        L.append(f"\nBẢNG — {key.replace('|', ' -> xenium | ')} (S chung {len(r['S_shared'])} nhóm; trần source accS = {r['ceiling_accS_source']})")
        rows = [{"map": k, **{c: v.get(c) for c in ["assay_sep", "xfer_S", "xfer_coarse", "keep_struct"]}}
                for k, v in r["eval"].items() if not k.endswith("_centered")]
        L.append(pd.DataFrame(rows).to_string(index=False))
        if r["usher_seconds"]: L.append("  thời gian USHER (s): " + json.dumps({k: round(v) for k, v in r["usher_seconds"].items()}))
    rep = "\n".join(L); open(f"{OUT}/report.txt", "w").write(rep)
    log("\n" + "=" * 90 + rep + "\nXONG")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run05c.py
"""05c — Chạy lại USHER đúng như bài báo (Pratapa et al., bioRxiv 2025) + bộ nhớ đệm cấu trúc.

Khác 05 ở đâu:
  * M_aux: xác suất cell type DỰ ĐOÁN cho source bằng hồi quy logistic huấn luyện trên biểu hiện của TARGET (Xenium, chỉ
    gene chung) — như mục A.1/A.5 của bài báo. KHÔNG dùng nhãn thật của source (05 đã dùng nhãn thật ô T: giám sát mạnh hơn
    bài báo). Target dùng one-hot nhãn chú giải của Xenium (tham chiếu).
  * Target được lấy mẫu phân tầng theo thành phần cell type DỰ ĐOÁN của source (như bài báo làm với scRNA-seq).
  * Baseline dùng nhãn giả (pseudo-label = argmax của hồi quy logistic), cùng lượng thông tin với USHER:
      lr_direct   : hồi quy logistic dự đoán thẳng nhãn source H từ biểu hiện, KHÔNG cần embedding hay căn chỉnh.
                    Đây là đối thủ trực tiếp: USHER nhận chính thông tin này làm đầu vào, nên phải vượt nó mới có giá trị
                    về chuyển nhãn.
      class_shift_pl, class_ridge_pl : như 05 nhưng dùng nhãn giả.
  * Cấu hình USHER như pipeline của bạn (scGPT_example.ipynb): sketch 4300, stratified_pairing_fix=True, 10 vòng EM.
    usher_paper (MLP 512), usher_paper_linear, usher_paper_cacheT (đệm C2 target, chính xác, 1 lần/batch),
    usher_paper_cacheST (đệm cả C1 trên embedding GỐC của source — ablation, đổi ngữ nghĩa so với code).
  * Thêm keep_struct_cos (cosine) và khoảng tin cậy bootstrap 95% cho xfer_S.
"""
import os, sys, glob, json, time, gc, logging
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M
from run04 import np, pd, sp, log, load_assay, _unit, lognorm
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

OUT = os.environ.get("OUT_DIR_5C", "/kaggle/working/out05c")
TARGET = M.TARGET
N_BOOT = int(os.environ.get("N_BOOT", 200))
# Cấu hình pipeline của bạn (example_notebooks_cuda/scGPT_example.ipynb): sketch 4300, nhóm CỐ ĐỊNH, 10 vòng EM, MLP 512
SKETCH = int(os.environ.get("SKETCH", 4300))
N_ITERS = int(os.environ.get("N_ITERS", 10))
PAPER = {"gamma": 0.4, "alpha": 0.3, "hidden_dim": 512, "cache": "off"}
RUNS = [("cosmx6k", "scgpt", "usher_paper", PAPER),
        ("cosmx6k", "scgpt", "usher_paper_linear", {**PAPER, "hidden_dim": None}),
        ("cosmx6k", "scgpt", "usher_paper_cacheT", {**PAPER, "cache": "target"}),
        ("cosmx6k", "scgpt", "usher_paper_cacheST", {**PAPER, "cache": "both"}),
        ("cosmx6k", "niche_data", "usher_paper", PAPER),
        ("cosmx6k", "niche_data", "usher_paper_cacheT", {**PAPER, "cache": "target"}),
        ("visiumhd_ffpe", "scgpt", "usher_paper", PAPER),
        ("visiumhd_ffpe", "niche_data", "usher_paper", PAPER)]
ONLY = os.environ.get("ONLY_RUNS", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


# =============================================================================== nhãn giả từ biểu hiện (như bài báo)
def shared_lognorm(Xs, vs, Xx, vx):
    ss, sx = vs["symbol"].str.upper().to_numpy(), vx["symbol"].str.upper().to_numpy()
    fs = pd.Series(np.arange(len(ss)), index=ss)[~pd.Index(ss).duplicated()]
    fx = pd.Series(np.arange(len(sx)), index=sx)[~pd.Index(sx).duplicated()]
    G = sorted(set(fs.index) & set(fx.index))
    return lognorm(Xs[:, fs.loc[G].to_numpy()]), lognorm(Xx[:, fx.loc[G].to_numpy()]), len(G)

def fit_label_model(Ys, Yx, lab_x):
    sc = StandardScaler(with_mean=False).fit(Yx)
    clf = LogisticRegression(max_iter=500, C=0.5, n_jobs=4).fit(sc.transform(Yx), lab_x)
    return clf.predict_proba(sc.transform(Ys)), clf.classes_


def strat_target(lab_src_T, lab_x, n_tgt=None, seed=1):
    """Target phân tầng theo thành phần nhãn (dự đoán) của source ô T."""
    n_tgt = M.N_TGT if n_tgt is None else n_tgt
    rng = np.random.RandomState(seed)
    comp = pd.Series(lab_src_T).value_counts(normalize=True)
    tgt = []
    for c, f in comp.items():
        pool = np.where(lab_x == c)[0]
        if len(pool): tgt.append(rng.choice(pool, min(len(pool), max(1, int(round(f * n_tgt)))), replace=False))
    return np.sort(np.concatenate(tgt))


# =============================================================================== đánh giá (như 05 + CI + cosine)
def mix_ref(src, D):
    """Tập trộn (assay_sep) và 20k tế bào tham chiếu Xenium của cross_assay_ci (cùng chuỗi RandomState(0))."""
    rng = np.random.RandomState(0)
    ds, dx = D[src], D[TARGET]
    cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
    shared_c = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]
    per = max(1, M.N_MIX // max(len(shared_c), 1))
    mix_s, mix_x = [], []
    for c in shared_c:
        k = min(per, cs[c], cx[c])
        mix_s.append(rng.choice(ds["H"][ds["lab_c"][ds["H"]] == c], k, replace=False))
        mix_x.append(rng.choice(np.where(dx["lab_c"] == c)[0], k, replace=False))
    mix_s, mix_x = np.concatenate(mix_s), np.concatenate(mix_x)
    ref = np.sort(rng.choice(len(dx["lab"]), min(M.N_REF, len(dx["lab"])), replace=False))
    return shared_c, mix_s, mix_x, ref

def cross_assay_ci(src, D, reps, S_shared, extra_pred=None):
    ds, dx = D[src], D[TARGET]
    shared_c, mix_s, mix_x, ref = mix_ref(src, D)
    qS = ds["H"][np.isin(ds["lab"][ds["H"]], S_shared)]
    qC = ds["H"][np.isin(ds["lab_c"][ds["H"]], shared_c)]
    boot = np.random.RandomState(1)
    B_idx = [boot.randint(0, len(qS), len(qS)) for _ in range(N_BOOT)]
    def ci(y, p):
        v = [balanced_accuracy_score(y[b], p[b]) for b in B_idx]
        return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]
    out = {}
    if extra_pred is not None:                                    # lr_direct: dự đoán thẳng, không embedding
        y, p = ds["lab"][qS], extra_pred[qS]
        out["lr_direct"] = {"assay_sep": None, "xfer_S": M.bacc_on(y, p, S_shared), "xfer_S_ci": ci(y, p),
                            "xfer_coarse": M.bacc_on(ds["lab_c"][qC], B.to_coarse(extra_pred[qC]), shared_c)}
    for name, R_ in reps.items():
        Fs, Fx = _unit(R_[src]), _unit(R_[TARGET])
        J = np.vstack([Fx[mix_x], Fs[mix_s]]); g = np.r_[np.zeros(len(mix_x), int), np.ones(len(mix_s), int)]
        nnJ = NearestNeighbors(n_neighbors=16, n_jobs=4).fit(J).kneighbors(J, return_distance=False)[:, 1:]
        sep = round(float(balanced_accuracy_score(g, (g[nnJ].mean(1) > 0.5).astype(int))), 3)
        nnR = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(Fx[ref])
        pS = M.maj_vote(nnR.kneighbors(Fs[qS], 15, return_distance=False), dx["lab"][ref])
        pC = M.maj_vote(nnR.kneighbors(Fs[qC], 15, return_distance=False), dx["lab_c"][ref])
        y = ds["lab"][qS]
        out[name] = {"assay_sep": sep, "xfer_S": M.bacc_on(y, pS, S_shared), "xfer_S_ci": ci(y, pS),
                     "xfer_coarse": M.bacc_on(ds["lab_c"][qC], pC, shared_c)}
    return out

def keep_struct_cos(before, after, idx, k=15):
    return M.keep_struct(_unit(before), _unit(after), idx, k)


# =============================================================================== USHER đúng bài báo
def run_usher_paper(Fs, T, P_src, Fx, tgt, lab_tgt, classes, cfg):
    import torch, anndata as ad, run_usher as U, struct_cache as SC
    logging.getLogger().setLevel(logging.WARNING)
    Pt = np.zeros((len(tgt), len(classes)), np.float32)
    Pt[np.arange(len(tgt)), [list(classes).index(c) for c in lab_tgt[tgt]]] = 1
    a = ad.AnnData(X=Fs[T].astype(np.float32), obs=pd.DataFrame(index=[f"s{i}" for i in T]))
    b = ad.AnnData(X=Fx[tgt].astype(np.float32), obs=pd.DataFrame({"ct": lab_tgt[tgt]}, index=[f"t{i}" for i in tgt]))
    a.obs["ct"] = "unknown"                                        # không dùng nhãn source
    a.obsm["X_celltype_probs"] = P_src[T].astype(np.float32); b.obsm["X_celltype_probs"] = Pt
    info = {}
    SC.reset(cfg["cache"], Fs[T].astype(np.float32) if cfg["cache"] == "both" else None)
    t0 = time.time()
    model, _, _, T_full, mean, std = U.align_features_fgw(
        a, b, e_step_method="fgw", m_step_method="global", sampling_strategy="celltype",
        sketch_size=SKETCH, sketch_obsm_key="X_none", use_stratified_pairing=True, stratified_pairing_fix=True,
        cell_type_col="ct", celltype_probs_layer="X_celltype_probs", alpha=cfg["alpha"], gamma=cfg["gamma"],
        epsilon=0.1, knn_k=30, metric="cosine", m_step_metric="euclidean", lambda_cross=0.9, lambda_var=0.1,
        entropy_percentile=90.0, confidence_percentile=10.0, use_linear_assignment=True, steps_per_iter=100, lr=1e-3,
        n_iters=N_ITERS, hidden_dim=cfg["hidden_dim"], device="cuda", verbose=False)
    info["em_seconds"] = round(time.time() - t0, 1); info["cache_stats"] = dict(SC.STATS)
    SC.reset("off")
    del T_full; torch.cuda.empty_cache()
    with torch.no_grad():
        model.eval(); x = torch.from_numpy(Fs.astype(np.float32)).cuda()
        y = model((x - mean) / std) * std + mean if mean is not None else model(x)
        out = y.cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return out, info


# =============================================================================== chạy
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    EMB = {"scgpt": M.find_dir("emb_scgpt"), "niche_data": M.find_dir("emb_niche_data")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    runs = [r for i, r in enumerate(RUNS) if not ONLY or str(i) in ONLY.split(",")]
    D, F, EXPR = {}, {}, {}
    for a in sorted({r[0] for r in runs} | {TARGET}):
        X, obs, var, _ = load_assay(a)
        lab = obs["label"].to_numpy().astype(str)
        split, _ = B.make_split(obs, a)
        D[a] = {"lab": lab, "lab_c": B.to_coarse(lab), "H": np.where(split == "H")[0], "T": np.where(split == "T")[0]}
        EXPR[a] = (X, var)
        for m, d in EMB.items(): F[(a, m)] = M.load_emb(d, a, obs.index)
        log(f"SKETCH={SKETCH} N_ITERS={N_ITERS} fix=True") if a == sorted({r[0] for r in runs} | {TARGET})[0] else None
        log(f"[{a}] {len(lab)} tế bào | T={len(D[a]['T'])} H={len(D[a]['H'])}")
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    # nhãn giả cho mỗi source (một lần, dùng chung cho cả hai model)
    PL = {}
    for src in sorted({r[0] for r in runs}):
        step(f"hồi quy logistic trên Xenium -> dự đoán {src}")
        Ys, Yx, nG = shared_lognorm(EXPR[src][0], EXPR[src][1], EXPR[TARGET][0], EXPR[TARGET][1])
        P, classes = fit_label_model(Ys, Yx, D[TARGET]["lab"])
        PL[src] = (P, classes, classes[P.argmax(1)], nG)
        acc = M.bacc_on(D[src]["lab"][D[src]["H"]], PL[src][2][D[src]["H"]], sorted(set(REC[src]) & set(REC[TARGET])))
        log(f"  {nG} gene chung | độ chính xác nhãn giả trên H (nhóm chung) = {acc}")
    for src, m in sorted({(r[0], r[1]) for r in runs}):
        key = f"{src}|{m}"; t0 = time.time()
        Fs, Fx = F[(src, m)], F[(TARGET, m)]
        ds, dx = D[src], D[TARGET]
        P, classes, pl, nG = PL[src]
        S_shared = sorted(set(REC[src]) & set(REC[TARGET]))
        log(f"\n=== {src} -> xenium | {m} | S chung = {S_shared}")
        # target phân tầng theo thành phần DỰ ĐOÁN của source (ô T)
        tgt = strat_target(pl[ds["T"]], dx["lab"])
        maps = {"raw": Fs, "center": M.base_center(Fs, Fx, ds["T"]),
                "class_shift_pl": M.base_class_shift(Fs, Fx, ds["T"], pl, dx["lab"])}
        maps["class_ridge_pl"], _ = M.base_class_ridge(Fs, Fx, ds["T"], pl, dx["lab"])
        infos = {}
        for (s_, m_, name, cfg) in runs:
            if (s_, m_) != (src, m): continue
            f = f"{OUT}/{src}_{m}_{name}.npy"
            if os.path.exists(f):
                maps[name] = np.load(f).astype(np.float32); step(f"{name}: nạp lại"); continue
            step(f"USHER {name} {cfg} | target {len(tgt)} tế bào phân tầng theo nhãn giả")
            maps[name], infos[name] = run_usher_paper(Fs, ds["T"], P, Fx, tgt, dx["lab"], classes, cfg)
            np.save(f, maps[name].astype(np.float16))
            step(f"USHER {name} xong: {infos[name]}")
        step("đánh giá")
        ev = cross_assay_ci(src, D, {k: {src: v, TARGET: Fx} for k, v in maps.items()}, S_shared, extra_pred=pl)
        Hs = B.pick(ds["H"], 10000)
        for k in maps:
            ev[k]["keep_struct"] = M.keep_struct(Fs, maps[k], Hs)
            ev[k]["keep_struct_cos"] = keep_struct_cos(Fs, maps[k], Hs)
        prev = S.get(key, {}).get("usher_info", {})
        S[key] = {"S_shared": S_shared, "n_shared_genes_lr": nG, "ceiling_accS_source": M.own_ceiling(Fs, ds["H"], ds["lab"], S_shared),
                  "usher_info": {**prev, **infos}, "eval": ev}
        log(f"  trần (kNN trong chính source, H) accS = {S[key]['ceiling_accS_source']}")
        for k, v in ev.items():
            log(f"    {k:20s} assay_sep={v['assay_sep']} xfer_S={v['xfer_S']} {v['xfer_S_ci']} xfer_thô={v['xfer_coarse']} "
                f"keep={v.get('keep_struct')} keep_cos={v.get('keep_struct_cos')}")
        json.dump(S, open(f"{OUT}/summary.json", "w"), indent=2, default=str)
        log(f"  ({time.time() - t0:.0f}s)")
    L = []
    for key, r in S.items():
        L.append(f"\nBẢNG — {key.replace('|', ' -> xenium | ')} (S chung {len(r['S_shared'])} nhóm; trần source accS = {r['ceiling_accS_source']})")
        rows = [{"map": k, "assay_sep": v["assay_sep"], "xfer_S": v["xfer_S"], "CI95": v["xfer_S_ci"], "xfer_thô": v["xfer_coarse"],
                 "keep": v.get("keep_struct"), "keep_cos": v.get("keep_struct_cos")} for k, v in r["eval"].items()]
        L.append(pd.DataFrame(rows).to_string(index=False))
        if r["usher_info"]: L.append("  USHER: " + json.dumps(r["usher_info"], default=str))
    rep = "\n".join(L); open(f"{OUT}/report.txt", "w").write(rep)
    log("\n" + "=" * 90 + rep + "\nXONG")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run06.py
"""06 — Chất lượng M_aux (A) + thước đo nội suy gene (B) + đo thời gian từng thành phần EM (C).

A. Cặp CosMx -> Xenium, scGPT-spatial. M_aux có kiểm soát (seed nhiễu = 0, dùng chung cho mọi seed USHER):
     true, struct_f (thay tỷ lệ f nhãn thật bằng argmax LR), unif_f (thay tỷ lệ f' bằng lớp khác chọn đều, f' chọn để
     balanced accuracy trên T khớp struct_f ±0.01), soft_lr (xác suất LR, như 05c).
   USHER U6 (chỉ hidden_dim đổi): linear 8 điều kiện × 3 seed, mlp {true, soft_lr} × 3 seed, + 1 lần chạy tuần tự
   (true, linear, seed 0, 1 luồng E-step) chỉ cho bảng C. Seed USHER đổi cả sketch geosketch qua USHER_SEED (U6b).
B. Nội suy gene không nhãn: 15 hàng xóm cosine trong 20k tham chiếu Xenium (đúng tập ref của cross_assay_ci) -> trung bình
   log-normalize -> Pearson/Spearman theo gene trên 10k tế bào truy vấn H của source. Tham chiếu: shuffle, jpca_c,
   jpca_c_heldout (5 fold gene), own_nn. Map 05c (nạp từ out05c) + baseline 05c tính lại + mọi map của phần A.
C. Monkeypatch (không sửa src_cuda) đo: C1/C2 (compute_knn_graph_distance), lời giải FGW
   (ot.gromov.entropic_fused_gromov_wasserstein), apply_linear_assignment, train_global_model, aggregate; thời gian thực
   E-step. Benchmark landmark (src_cuda) so với hàm gốc (USHER/usher/src) ở n = 4,300.
"""
import os, sys, glob, json, time, gc, random, shutil, threading
SMOKE = os.environ.get("SMOKE", "0") == "1"
if SMOKE:   # vài nghìn tế bào, 1 vòng EM; phải đặt TRƯỚC khi import run05/run05c (chúng đọc biến môi trường lúc import)
    for k, v in {"SKETCH": "300", "N_ITERS": "1", "N_BOOT": "20", "N_TGT": "1500", "N_REF": "1500", "N_MIX": "600"}.items():
        os.environ.setdefault(k, v)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M, run05c as C
from run04 import np, pd, sp, log, load_assay, _unit
from scipy.stats import rankdata, spearmanr
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

OUT = os.environ.get("OUT_DIR_6", "/kaggle/working/out06" + ("_smoke" if SMOKE else ""))   # smoke không lẫn vào RESUME
TARGET, SRC, EMB_A = M.TARGET, "cosmx6k", "scgpt"
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
SEEDS = [0] if SMOKE else [0, 1, 2]
FRACS = [0.33, 0.67, 1.0]
CONDS = ["true"] + [f"struct_{f}" for f in FRACS] + [f"unif_{f}" for f in FRACS] + ["soft_lr"]
MSTEP = {"linear": None, "mlp": 512}
U6 = {"gamma": 0.4, "alpha": 0.3, "cache": "off"}          # + hidden_dim theo M-step; phần còn lại tường minh trong run_usher_paper
N_Q, K_IMP, MIN_DET, N_GBOOT, N_FOLD, N_PC_J = 10000, 15, 0.05, (100 if SMOKE else 1000), 5, 50
N_SMOKE = 3000
EXPECT_05C = {   # xfer_S của RESEARCH_CONTEXT 5.2 (research/results/05c_summary.md) — kiểm tra tái lập ±0.005
    "cosmx6k|scgpt": {"lr_direct": .3732, "raw": .3371, "center": .6206, "usher_paper": .4969, "usher_paper_linear": .5274,
                      "usher_paper_cacheT": .4938, "usher_paper_cacheST": .4981},
    "cosmx6k|niche_data": {"lr_direct": .3732, "raw": .4425, "center": .4882, "usher_paper": .4471, "usher_paper_cacheT": .4357},
    "visiumhd_ffpe|scgpt": {"lr_direct": .7863, "raw": .6357, "center": .7128, "usher_paper": .8090},
    "visiumhd_ffpe|niche_data": {"lr_direct": .7863, "raw": .7159, "center": .7212, "usher_paper": .7173}}
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def run_list():
    """Seed 0 của mọi điều kiện trước (kèm lần chạy tuần tự cho bảng C), rồi seed 1, rồi seed 2."""
    L = []
    for s in SEEDS:
        L += [(c, "linear", s, False) for c in CONDS] + [(c, "mlp", s, False) for c in ("true", "soft_lr")]
        if s == 0: L.append(("true", "linear", 0, True))
    return L

def run_name(cond, mstep, seed, seq):
    return f"A_{cond}_{mstep}_s{seed}" + ("_seq" if seq else "")


# =============================================================================== A: M_aux có kiểm soát
def onehot(lab, classes):
    idx = pd.Index(classes).get_indexer(lab); assert (idx >= 0).all(), "nhãn source không có trong classes của LR"
    P = np.zeros((len(lab), len(classes)), np.float32); P[np.arange(len(lab)), idx] = 1
    return P

def noisy(lab, repl, order, f):
    """Thay nhãn của f·n tế bào đầu tiên theo `order` bằng repl (tập con lồng nhau -> độ chính xác đơn điệu theo f)."""
    out = lab.copy(); k = int(round(f * len(lab))); out[order[:k]] = repl[order[:k]]
    return out

def match_frac(lab, repl, order, target, acc_fn, n_iter=30):
    """Tìm nhị phân f' để acc_fn(noisy(...)) gần target nhất."""
    lo, hi, best = 0.0, 1.0, (np.inf, 0.0, None)
    for _ in range(n_iter):
        f = (lo + hi) / 2; a = acc_fn(noisy(lab, repl, order, f))
        if abs(a - target) < best[0]: best = (abs(a - target), f, a)
        lo, hi = (f, hi) if a > target else (lo, f)
    return best[1], best[2]

def build_aux(lab, P_lr, classes, T, acc_fn):
    """-> {cond: (M_aux cho mọi tế bào source, argmax nhãn)}, {cond: f thực dùng}. Seed nhiễu cố định = 0."""
    rng = np.random.RandomState(0)
    n, K = len(lab), len(classes)
    order_s, order_u = rng.permutation(n), rng.permutation(n)
    ci = pd.Index(classes).get_indexer(lab)
    alt = np.asarray(classes)[(ci + rng.randint(1, K, n)) % K]             # lớp KHÁC nhãn thật, chọn đều
    pl = np.asarray(classes)[P_lr.argmax(1)]
    labs, frac = {"true": lab}, {}
    for f in FRACS:
        labs[f"struct_{f}"] = noisy(lab, pl, order_s, f); frac[f"struct_{f}"] = f
        target = acc_fn(labs[f"struct_{f}"])
        frac[f"unif_{f}"], _ = match_frac(lab, alt, order_u, target, acc_fn)
        labs[f"unif_{f}"] = noisy(lab, alt, order_u, frac[f"unif_{f}"])
    aux = {c: (onehot(l, classes), l) for c, l in labs.items()}
    aux["soft_lr"] = (P_lr.astype(np.float32), pl)
    return {c: aux[c] for c in CONDS}, frac


# =============================================================================== B: nội suy gene
def col_corr(A, Bm):
    """Tương quan Pearson theo cột (NaN nếu một cột hằng)."""
    A = A - A.mean(0); Bm = Bm - Bm.mean(0)
    den = np.sqrt((A * A).sum(0) * (Bm * Bm).sum(0))
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (A * Bm).sum(0) / den
    r[den == 0] = np.nan
    return r

def rank_cols(A):
    return rankdata(A, axis=0).astype(np.float32)

def avg_rows(nb, Y):
    """Trung bình các hàng Y[nb[i]] (Y thưa hoặc đặc)."""
    n, k = nb.shape
    W = sp.csr_matrix((np.full(n * k, 1.0 / k, np.float32), nb.ravel(), np.arange(0, n * k + 1, k)), shape=(n, Y.shape[0]))
    P = W @ Y
    return np.asarray(P.toarray() if sp.issparse(P) else P, np.float32)

def knn_idx(Fq, Fr, k=K_IMP):
    return NearestNeighbors(n_neighbors=k, n_jobs=4).fit(_unit(Fr)).kneighbors(_unit(Fq), return_distance=False)

def imp_setup(Ys, Yx, genes, q, ref, H):
    """Ys, Yx: log-normalize riêng từng assay trên gene chung. Giữ gene phát hiện ở >= 5% truy vấn VÀ >= 5% tham chiếu."""
    Yq, Yr = Ys[q], Yx[ref]
    keep = (np.asarray((Yq > 0).mean(0)).ravel() >= MIN_DET) & (np.asarray((Yr > 0).mean(0)).ravel() >= MIN_DET)
    g = np.where(keep)[0]
    gn = np.asarray(genes)[g]
    mk = {x for v in B.MARKERS.values() for x in v}
    truth = Yq[:, g].toarray().astype(np.float32)
    return {"q": q, "H": H, "genes": gn, "marker": np.isin(gn, list(mk)), "truth": truth, "rank_truth": rank_cols(truth),
            "Yr": Yr[:, g].tocsr(), "YH": Ys[H][:, g].tocsr()}

def imp_score(pred, S):
    pear = col_corr(pred, S["truth"]); spear = col_corr(rank_cols(pred), S["rank_truth"])
    G = len(spear); boot = np.random.RandomState(0).randint(0, G, (N_GBOOT, G))
    med = np.nanmedian(spear[boot], axis=1)
    mk = S["marker"]
    return {"pearson_med": round(float(np.nanmedian(pear)), 4), "spearman_med": round(float(np.nanmedian(spear)), 4),
            "spearman_ci": [round(float(np.percentile(med, 2.5)), 4), round(float(np.percentile(med, 97.5)), 4)],
            "spearman_marker_med": round(float(np.nanmedian(spear[mk])), 4) if mk.any() else None,
            "n_genes": int(G), "n_marker": int(mk.sum()),
            # Chia panel gene thành 2 nửa cố định (seed 12345) -> dùng nửa A để CHỌN tham số và nửa B để
            # BÁO CÁO, tránh vòng tròn "chọn bằng nội suy rồi khoe nội suy" (vòng 35). Thêm khoá, không đổi
            # khoá cũ -> mọi script đang đọc summary vẫn chạy y nguyên.
            **(lambda h: {"spearman_med_A": round(float(np.nanmedian(spear[h])), 4),
                          "spearman_med_B": round(float(np.nanmedian(spear[~h])), 4)}
               )(np.random.RandomState(12345).rand(G) < 0.5)}

def imp_map(Fmap, Fx_ref, S):
    """Map của source (mọi tế bào) + embedding gốc của 20k tham chiếu Xenium."""
    return imp_score(avg_rows(knn_idx(Fmap[S["q"]], Fx_ref), S["Yr"]), S)

def jpca_nb(S, gidx):
    """PCA 50 trên [truy vấn; tham chiếu] (gene gidx), trừ trung bình theo từng assay; hàng xóm cosine."""
    Zq = S["truth"][:, gidx]; Zr = S["Yr"][:, gidx].toarray()
    Zq, Zr = Zq - Zq.mean(0), Zr - Zr.mean(0)
    pca = PCA(min(N_PC_J, len(gidx) - 1), svd_solver="randomized", random_state=0).fit(np.vstack([Zq, Zr]))
    return knn_idx(pca.transform(Zq), pca.transform(Zr))

def imp_refs(Fs, S):
    """shuffle, jpca_c, jpca_c_heldout, own_nn (cùng quy trình dự đoán, chỉ khác không gian tìm hàng xóm)."""
    nq, G, nr = len(S["q"]), S["truth"].shape[1], S["Yr"].shape[0]
    out = {"shuffle": imp_score(avg_rows(np.random.RandomState(0).randint(0, nr, (nq, K_IMP)), S["Yr"]), S),
           "jpca_c": imp_score(avg_rows(jpca_nb(S, np.arange(G)), S["Yr"]), S)}
    pred = np.zeros((nq, G), np.float32)
    for fold in np.array_split(np.random.RandomState(0).permutation(G), N_FOLD):
        nb = jpca_nb(S, np.setdiff1d(np.arange(G), fold))
        pred[:, fold] = avg_rows(nb, S["Yr"][:, fold])
    out["jpca_c_heldout"] = imp_score(pred, S)
    H, q = S["H"], S["q"]
    nb = NearestNeighbors(n_neighbors=K_IMP + 1, n_jobs=4).fit(_unit(Fs[H])).kneighbors(_unit(Fs[q]), return_distance=False)
    keep = nb != np.searchsorted(H, q)[:, None]                            # loại chính nó
    keep[keep.sum(1) > K_IMP, -1] = False
    out["own_nn"] = imp_score(avg_rows(nb[keep].reshape(nq, K_IMP), S["YH"]), S)
    return out


# =============================================================================== C: đo thời gian (monkeypatch)
class Prof:
    """Bộ đếm thời gian theo thành phần; mỗi luồng đồng bộ đúng stream CUDA của nó."""
    def __init__(self):
        self.sec, self.n, self.spans, self.undo, self.lock = {}, {}, [], [], threading.Lock()

    @staticmethod
    def _sync():
        import torch
        if torch.cuda.is_available(): torch.cuda.current_stream().synchronize()

    def wrap(self, mod, name, key, span=False):
        f = getattr(mod, name)
        def w(*a, **k):
            self._sync(); t0 = time.perf_counter()
            try:
                return f(*a, **k)
            finally:
                self._sync(); t1 = time.perf_counter()
                with self.lock:
                    self.sec[key] = self.sec.get(key, 0.0) + t1 - t0; self.n[key] = self.n.get(key, 0) + 1
                    if span: self.spans.append((k.get("iteration"), t0, t1))
        setattr(mod, name, w); self.undo.append((mod, name, f))

    def install(self):
        import ot, run_usher as U, e_step_utils as E, m_step_utils as MS
        self.wrap(E, "compute_knn_graph_distance", "struct_C1C2")
        self.wrap(ot.gromov, "entropic_fused_gromov_wasserstein", "fgw_solver")
        self.wrap(E, "apply_linear_assignment", "hungarian")
        self.wrap(U, "compute_transport_batch", "estep_batches_sum", span=True)
        self.wrap(U, "train_global_model", "mstep_train")
        self.wrap(MS, "aggregate_training_data_from_batches", "mstep_aggregate")
        return self

    def remove(self):
        for mod, name, f in reversed(self.undo): setattr(mod, name, f)
        self.undo = []

    def result(self, em_seconds):
        it = pd.DataFrame(self.spans, columns=["it", "t0", "t1"]).groupby("it").agg({"t0": "min", "t1": "max"})
        r = {k: round(v, 2) for k, v in self.sec.items()}
        r["estep_wall"] = round(float((it["t1"] - it["t0"]).sum()), 2)
        r["em_seconds"] = em_seconds; r["calls"] = dict(self.n)
        return r


def bench_graph(F, n=4300, reps=3):
    """Landmark (src_cuda) so với hàm gốc (USHER/usher/src) trên cùng đặc trưng low-rank 64 (như compute_transport_fgw)."""
    import torch, torch.nn.functional as Fn, graph_utils as GL, struct_cache as SC
    idx = B.pick(np.arange(len(F)), n)
    X = SC._lowrank_like_code(Fn.normalize(torch.from_numpy(F[idx].astype(np.float32)).cuda(), p=2, dim=1), 64)
    out = {"n": int(len(idx)), "low_rank": int(X.shape[1])}
    fns = {"landmark_src_cuda": GL.compute_knn_graph_distance}
    try:
        import usher_orig_graph as GO; fns["original_usher"] = GO.compute_knn_graph_distance
    except Exception as e:
        out["original_usher_error"] = repr(e)
    for name, fn in fns.items():
        try:
            fn(X, k=30, metric="cosine", device="cuda"); torch.cuda.synchronize()          # khởi động
            ts = []
            for _ in range(reps):
                t0 = time.perf_counter(); fn(X, k=30, metric="cosine", device="cuda"); torch.cuda.synchronize()
                ts.append(time.perf_counter() - t0)
            out[name] = {"sec_per_call": round(float(np.mean(ts)), 3), "runs": [round(t, 3) for t in ts]}
        except Exception as e:
            out[f"{name}_error"] = repr(e)
    return out


# =============================================================================== USHER
def usher(Fs, T, P_src, Fx, tgt, lab_x, classes, mstep, seed, seq, timing):
    import torch
    os.environ["USHER_SEED"] = str(seed)                  # vá run_usher.py: seed geosketch = 2025 + USHER_SEED (U6b)
    os.environ["USHER_EM_WORKERS"] = "1" if seq else "4"  # vá run_usher.py: số luồng E-step
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    prof = Prof().install() if timing else None
    try:
        out, info = C.run_usher_paper(Fs, T, P_src, Fx, tgt, lab_x, classes, {**U6, "hidden_dim": MSTEP[mstep]})
    finally:
        if prof: prof.remove()
    info.pop("cache_stats", None)
    if prof: info["timing"] = prof.result(info["em_seconds"])
    return out, info


# =============================================================================== chạy
def load_all(assays, emb):
    D, F, EXPR = {}, {}, {}
    for a in assays:
        X, obs, var, _ = load_assay(a)
        if SMOKE:
            keep = B.pick(np.arange(X.shape[0]), N_SMOKE); X, obs = X[keep], obs.iloc[keep]
        lab = obs["label"].to_numpy().astype(str)
        split, _ = B.make_split(obs, a)
        D[a] = {"lab": lab, "lab_c": B.to_coarse(lab), "H": np.where(split == "H")[0], "T": np.where(split == "T")[0]}
        EXPR[a] = (X, var)
        for m, d in emb.items():
            if d: F[(a, m)] = M.load_emb(d, a, obs.index)
        log(f"[{a}] {len(lab)} tế bào | T={len(D[a]['T'])} H={len(D[a]['H'])}")
    return D, F, EXPR

def eval_one(src, D, Fs, Fx, name, Fmap, S_shared, IMP, ref):
    ev = C.cross_assay_ci(src, D, {name: {src: Fmap, TARGET: Fx}}, S_shared)[name]
    Hs = B.pick(D[src]["H"], 10000)
    ev["keep_struct"] = M.keep_struct(Fs, Fmap, Hs); ev["keep_struct_cos"] = C.keep_struct_cos(Fs, Fmap, Hs)
    ev["imp"] = imp_map(Fmap, Fx[ref], IMP)
    return ev

def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    prev = sorted(glob.glob("/kaggle/input/**/out06/summary.json", recursive=True))
    if RESUME and prev and not os.path.exists(f"{OUT}/summary.json"):     # phiên Kaggle thứ hai: nạp output phiên trước
        for f in glob.glob(os.path.join(os.path.dirname(prev[0]), "*")):
            if os.path.isfile(f): shutil.copy(f, OUT)
        log(f"nạp output phiên trước từ {os.path.dirname(prev[0])}")
    if not SMOKE:   # bài học 05: không để cấu hình lệch khỏi U6 / 05c
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER U6 sketch={C.SKETCH} fix=True stratified=True n_iters={C.N_ITERS} eps=0.1 alpha={U6['alpha']} "
           f"gamma={U6['gamma']} knn_k=30 metric=cosine m_step_metric=euclidean l_cross=0.9 l_var=0.1 entropy=90 conf=10 "
           f"linear_assign=True steps=100 lr=1e-3 celltype/fgw/global sketch_obsm_key=X_none cache=off | hidden_dim: "
           f"linear=None mlp=512 | seeds={SEEDS} (USHER_SEED -> geosketch 2025+seed) | N_TGT={M.N_TGT} N_REF={M.N_REF} "
           f"N_MIX={M.N_MIX} N_BOOT={C.N_BOOT} | B: N_Q={N_Q} k={K_IMP} det>={MIN_DET} gene_boot={N_GBOOT} folds={N_FOLD} "
           f"jpca_pc={N_PC_J} | sklearn {__import__('sklearn').__version__}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs = run_list()
    for i, r in enumerate(runs): log(f"  run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt"), "niche_data": M.find_dir("emb_niche_data")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = load_all(sorted({SRC, "visiumhd_ffpe", TARGET}), EMB)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg
    for k in ("A", "B", "C"): S.setdefault(k, {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    # ---- LR (như 05c) + chuẩn bị nội suy cho mỗi source
    PL, IMP = {}, {}
    dx = D[TARGET]
    for src in (SRC, "visiumhd_ffpe"):
        step(f"hồi quy logistic trên Xenium -> {src}")
        X, var = EXPR[src]; Xx, vx = EXPR[TARGET]
        Ys, Yx, nG = C.shared_lognorm(X, var, Xx, vx)
        P, classes = C.fit_label_model(Ys, Yx, dx["lab"])
        PL[src] = (P, classes, classes[P.argmax(1)], nG)
        genes = sorted(set(var["symbol"].str.upper()) & set(vx["symbol"].str.upper()))   # cùng thứ tự với shared_lognorm
        assert len(genes) == nG == Ys.shape[1], (len(genes), nG)
        _, _, _, ref = C.mix_ref(src, D)
        IMP[src] = (imp_setup(Ys, Yx, genes, B.pick(D[src]["H"], N_Q), ref, D[src]["H"]), ref)
        del Ys, Yx; gc.collect()
        step(f"  {nG} gene chung (LR) | nội suy: {len(IMP[src][0]['genes'])} gene giữ, {int(IMP[src][0]['marker'].sum())} marker")

    # ---- C: benchmark landmark so với hàm gốc (một lần, GPU)
    if "bench" not in S["C"]:
        step("C: benchmark compute_knn_graph_distance (landmark so với gốc), n=4300")
        try:
            S["C"]["bench"] = bench_graph(F[(SRC, EMB_A)])
        except Exception as e:
            S["C"]["bench"] = {"error": repr(e)}
        save(); step(f"  {S['C']['bench']}")

    # ---- A: quét chất lượng M_aux
    src, ds = SRC, D[SRC]
    Fs, Fx = F[(src, EMB_A)], F[(TARGET, EMB_A)]
    P_lr, classes, pl, _ = PL[src]
    S_shared = sorted(set(REC[src]) & set(REC[TARGET]))
    acc = lambda l, idx=ds["T"]: M.bacc_on(ds["lab"][idx], l[idx], S_shared)
    AUX, frac = build_aux(ds["lab"], P_lr, classes, ds["T"], acc)
    A = S["A"]; A.setdefault("aux", {}); A.setdefault("base", {}); A.setdefault("runs", {})
    for c, (_, l) in AUX.items():
        A["aux"][c] = {"acc_T": acc(l), "acc_H": acc(l, ds["H"]), "frac": frac.get(c)}
    A["S_shared"] = S_shared
    log(f"\n=== A: {src} -> xenium | {EMB_A} | aux: " + json.dumps(A["aux"]))
    imp, ref = IMP[src]
    if "shared" not in A:
        step("A: raw, center")
        A["shared"] = {k: eval_one(src, D, Fs, Fx, k, v, S_shared, imp, ref)
                       for k, v in {"raw": Fs, "center": M.base_center(Fs, Fx, ds["T"])}.items()}
        save()
    for c, (_, l) in AUX.items():
        if c in A["base"]: continue
        step(f"A: baseline cho {c}")
        ev = {"aux_direct": C.cross_assay_ci(src, D, {}, S_shared, extra_pred=l)["lr_direct"]}
        ev["class_shift_aux"] = eval_one(src, D, Fs, Fx, "class_shift_aux", M.base_class_shift(Fs, Fx, ds["T"], l, dx["lab"]), S_shared, imp, ref)
        ridge, alpha_r = M.base_class_ridge(Fs, Fx, ds["T"], l, dx["lab"])
        ev["class_ridge_aux"] = eval_one(src, D, Fs, Fx, "class_ridge_aux", ridge, S_shared, imp, ref); ev["class_ridge_aux"]["alpha"] = alpha_r
        A["base"][c] = ev; save()
    for (c, mstep, seed, seq) in runs:
        name = run_name(c, mstep, seed, seq); f = f"{OUT}/{name}.npy"
        if "eval" in A["runs"].get(name, {}) and (seq or os.path.exists(f)):
            step(f"{name}: đã có, bỏ qua"); continue
        Paux, l = AUX[c]
        tgt = C.strat_target(l[ds["T"]], dx["lab"])
        if RESUME and os.path.exists(f) and "info" in A["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), A["runs"][name]["info"]
            step(f"{name}: nạp lại map")
        else:
            step(f"USHER {name} | target {len(tgt)} tế bào phân tầng theo argmax aux")
            Fmap, info = usher(Fs, ds["T"], Paux, Fx, tgt, dx["lab"], classes, mstep, seed, seq, timing=(seed == 0))
            if not seq: np.save(f, Fmap.astype(np.float16))
            A["runs"][name] = {"cond": c, "mstep": mstep, "seed": seed, "seq": seq, "info": info}; save()   # lưu info ngay
        ev = eval_one(src, D, Fs, Fx, name, Fmap, S_shared, imp, ref)
        A["runs"][name] = {"cond": c, "mstep": mstep, "seed": seed, "seq": seq, "info": info, "eval": ev}
        save()
        log(f"  >> {c:12s} {mstep:6s} seed={seed}{' tuần tự' if seq else ''} EM={info.get('em_seconds')}s "
            f"xfer_S={ev['xfer_S']} {ev['xfer_S_ci']} nội_suy_sp={ev['imp']['spearman_med']}")
        del Fmap; gc.collect()

    # ---- B: map 05c + baseline 05c + tham chiếu, cho 4 cặp
    files05 = {}
    for f in sorted(glob.glob("/kaggle/input/**/out05c/*.npy", recursive=True)):
        files05.setdefault(os.path.basename(f)[:-4], f)          # nhiều bản out05c trong input -> lấy bản đầu
    S["B"]["missing_05c"] = not files05 and not any(isinstance(v, dict) and v.get("n_05c", 0) for v in S["B"].values())
    if not files05: log("CẢNH BÁO: không thấy /kaggle/input/**/out05c/*.npy -> bỏ qua map USHER của 05c")
    for src in (SRC, "visiumhd_ffpe"):
        for m in ("scgpt", "niche_data"):
            key = f"{src}|{m}"
            if (src, m) not in F:
                S["B"][key] = {"skipped": f"thiếu embedding {m}"}; continue
            if key in S["B"] and "maps" in S["B"][key] and (S["B"][key].get("n_05c", 0) > 0 or not files05):
                step(f"B {key}: đã có, bỏ qua"); continue
            step(f"B {key}")
            ds, Fs, Fx = D[src], F[(src, m)], F[(TARGET, m)]
            P, classes, pl, _ = PL[src]
            S_sh = sorted(set(REC[src]) & set(REC[TARGET]))
            maps = {"raw": Fs, "center": M.base_center(Fs, Fx, ds["T"]),
                    "class_shift_pl": M.base_class_shift(Fs, Fx, ds["T"], pl, dx["lab"])}
            maps["class_ridge_pl"], _ = M.base_class_ridge(Fs, Fx, ds["T"], pl, dx["lab"])
            n05 = 0
            for b, f in files05.items():
                if b.startswith(f"{src}_{m}_"):
                    arr = np.load(f).astype(np.float32)
                    if len(arr) != len(Fs):
                        log(f"CẢNH BÁO: {b} có {len(arr)} tế bào, embedding có {len(Fs)} -> bỏ qua"); continue
                    maps[b[len(src) + len(m) + 2:]] = arr; n05 += 1
            ev = C.cross_assay_ci(src, D, {k: {src: v, TARGET: Fx} for k, v in maps.items()}, S_sh, extra_pred=pl)
            imp, ref = IMP[src]
            for k, v in maps.items(): ev[k]["imp"] = imp_map(v, Fx[ref], imp)
            refs = imp_refs(Fs, imp)
            names = [k for k in maps]
            rho = spearmanr([ev[k]["xfer_S"] for k in names], [ev[k]["imp"]["spearman_med"] for k in names])[0] if len(names) > 2 else None
            S["B"][key] = {"S_shared": S_sh, "n_05c": n05, "maps": ev, "refs": refs,
                           "leak": round(refs["jpca_c"]["spearman_med"] - refs["jpca_c_heldout"]["spearman_med"], 4),
                           "b3_spearman": None if rho is None or np.isnan(rho) else round(float(rho), 3)}
            save()
            for k in names + list(refs):
                v = ev.get(k, {}).get("imp") or refs[k]
                log(f"    {k:22s} xfer_S={ev.get(k, {}).get('xfer_S')} sp={v['spearman_med']} {v['spearman_ci']} pe={v['pearson_med']}")
    # B3 cho phần A (mọi map của A trên cặp cosmx/scgpt; chi tiết ở summary.json)
    a_maps = {k: r["eval"] for k, r in A["runs"].items() if not r["seq"] and "eval" in r}
    a_maps.update(A.get("shared", {}))
    for c, ev in A["base"].items():
        for k in ("class_shift_aux", "class_ridge_aux"): a_maps[f"{k}|{c}"] = ev[k]
    if len(a_maps) > 2:
        A["b3_spearman"] = round(float(spearmanr([v["xfer_S"] for v in a_maps.values()],
                                                 [v["imp"]["spearman_med"] for v in a_maps.values()])[0]), 3)
    save()
    md = summary_md(S)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG")


# =============================================================================== summary.md
def _ms(v):
    v = [x for x in v if x is not None]
    if not v: return "—"
    return f"{np.mean(v):.3f}±{np.std(v, ddof=1):.3f}" if len(v) > 1 else f"{v[0]:.3f}"

def summary_md(S):
    A, Bs, Cs = S["A"], S["B"], S["C"]
    L, W = ["# 06 — tóm tắt", S["config"], ""], []
    # ---- bảng A
    runs = [r for r in A.get("runs", {}).values() if not r["seq"] and "eval" in r]
    center = A.get("shared", {}).get("center", {}).get("xfer_S")
    L.append(f"## A. cosmx6k -> xenium5k | scgpt | S chung {len(A.get('S_shared', []))} nhóm | raw={A.get('shared', {}).get('raw', {}).get('xfer_S')} "
             f"center={center} | f' unif: " + ", ".join(f"{c}->{A['aux'][c]['frac']:.3f}" for c in CONDS if c.startswith("unif") and c in A.get("aux", {})))
    rows = []
    for mstep in MSTEP:
        for c in CONDS:
            rr = [r for r in runs if r["cond"] == c and r["mstep"] == mstep]
            if not rr: continue
            x = [r["eval"]["xfer_S"] for r in rr]; base = A["base"].get(c, {})
            d_, rg = base.get("aux_direct", {}).get("xfer_S"), base.get("class_ridge_aux", {}).get("xfer_S")
            mx = float(np.mean(x))
            rows.append({"aux": c, "M": mstep, "n": len(rr), "aux_T": A["aux"][c]["acc_T"], "aux_H": A["aux"][c]["acc_H"],
                         "xfer_S": _ms(x), "aux_dir": d_, "ridge_aux": rg,
                         "Δdir": round(mx - d_, 3) if d_ is not None else None, "Δcen": round(mx - center, 3) if center else None,
                         "Δridge": round(mx - rg, 3) if rg is not None else None,
                         "keep_cos": round(float(np.mean([r["eval"]["keep_struct_cos"] for r in rr])), 3),
                         "nội_suy_sp": _ms([r["eval"]["imp"]["spearman_med"] for r in rr]),
                         "EM_s": round(float(np.mean([r["info"].get("em_seconds") or 0 for r in rr])))})
            if mstep == "linear" and len(x) >= 3 and np.std(x, ddof=1) < 0.002:
                W.append(f"SD giữa seed của linear/{c} = {np.std(x, ddof=1):.4f} < 0.002 (USHER tuyến tính gần như tất định với batch)")
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    L.append("xfer_S = TB±SD qua seed; aux_dir = argmax aux trên H; Δ = xfer_S TB − cột tương ứng; nội_suy_sp = trung vị Spearman theo gene"
             + (f"; B3 (mọi map A) ρ={A['b3_spearman']}" if A.get("b3_spearman") is not None else ""))
    for c in CONDS:
        if c.startswith("unif") and c in A.get("aux", {}):
            s = A["aux"].get(c.replace("unif", "struct"), {})
            if s and abs(A["aux"][c]["acc_T"] - s["acc_T"]) > 0.01:
                W.append(f"{c}: acc_T {A['aux'][c]['acc_T']} không khớp {c.replace('unif', 'struct')} {s['acc_T']} trong ±0.01")
    ref05 = [r for r in runs if r["cond"] == "soft_lr" and r["seed"] == 0]
    if ref05:
        L.append("Đối chiếu 05c (cùng sketch seed 0): " + ", ".join(
            f"soft_lr/{r['mstep']}/s0={r['eval']['xfer_S']}" for r in ref05) + " (05c: linear 0.5274, mlp 0.4969)")
    # ---- bảng B
    RP = []
    L += ["", "## B. Nội suy gene (map 05c; trung vị theo gene trên 10k truy vấn H; CI 95% bootstrap theo gene)", "```"]
    for key, r in Bs.items():
        if not isinstance(r, dict) or "maps" not in r:
            if isinstance(r, dict) and "skipped" in r: W.append(f"B {key}: bỏ qua ({r['skipped']})")
            continue
        g = next(iter(r["refs"].values())); lr = r["maps"].get("lr_direct", {}).get("xfer_S")
        L.append(f"[{key}] {g['n_genes']} gene ({g['n_marker']} marker) | rò rỉ jpca_c−heldout = {r['leak']} | B3 ρ = {r['b3_spearman']} | lr_direct xfer_S = {lr}")
        rows = []
        for k, v in list(r["maps"].items()) + [(k, {"imp": v}) for k, v in r["refs"].items()]:
            if "imp" not in v: continue
            i = v["imp"]
            rows.append({"map": k, "xfer_S": v.get("xfer_S"), "pearson": i["pearson_med"], "spearman": i["spearman_med"],
                         "CI_gene": i["spearman_ci"], "sp_marker": i["spearman_marker_med"]})
            exp = EXPECT_05C.get(key, {}).get(k)
            if exp is not None and v.get("xfer_S") is not None and abs(v["xfer_S"] - exp) > 0.005:
                RP.append(f"{key} {k} {v['xfer_S']}/{exp}")
        L.append(pd.DataFrame(rows).to_string(index=False))
        exp = EXPECT_05C.get(key, {}).get("lr_direct")
        if exp is not None and lr is not None and abs(lr - exp) > 0.005: RP.append(f"{key} lr_direct {lr}/{exp}")
    L.append("```")
    if RP: W.append("tái lập 05c lệch > 0.005 (xfer_S chạy lại/05c): " + "; ".join(RP))
    if Bs.get("missing_05c"): W.append("không thấy map 05c (/kaggle/input/**/out05c/*.npy): bảng B chỉ có baseline + tham chiếu")
    L.append("Lưu ý: embedding SFM của source tính từ mọi gene (kể cả gene được dự đoán) -> rò rỉ như nhau cho mọi map; chỉ so sánh tương đối.")
    # ---- bảng C
    L += ["", "## C. Thời gian EM (giây; % của EM). Thành phần trong E-step là tổng cộng theo luồng; estep_wall = thời gian thực"]
    keys = ["struct_C1C2", "fgw_solver", "hungarian", "estep_batches_sum", "estep_wall", "mstep_train", "mstep_aggregate", "em_seconds"]
    groups = {"linear s0 (TB)": [r for r in runs if r["seed"] == 0 and r["mstep"] == "linear" and "timing" in r["info"]],
              "mlp s0 (TB)": [r for r in runs if r["seed"] == 0 and r["mstep"] == "mlp" and "timing" in r["info"]],
              "tuần tự 1 luồng": [r for r in A.get("runs", {}).values() if r["seq"] and "timing" in r["info"]]}
    rows = []
    for k in keys:
        row = {"thành phần": k}
        for g, rr in groups.items():
            if not rr: continue
            v = float(np.mean([r["info"]["timing"].get(k, 0.0) for r in rr]))
            em = float(np.mean([r["info"]["timing"]["em_seconds"] for r in rr]))
            row[f"{g} n={len(rr)}"] = f"{v:.1f} ({100 * v / em:.0f}%)"
        rows.append(row)
    if any(groups.values()): L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    else: W.append("chưa có lần chạy nào có đo thời gian (seed 0)")
    b = Cs.get("bench", {})
    L.append("Benchmark compute_knn_graph_distance n={} low-rank {}: landmark {} s/lần | gốc {} s/lần{}".format(
        b.get("n"), b.get("low_rank"), b.get("landmark_src_cuda", {}).get("sec_per_call"), b.get("original_usher", {}).get("sec_per_call"),
        "".join(f" | {k}: {v}" for k, v in b.items() if k.endswith("error"))))
    # ---- cảnh báo
    n_exp = len(run_list()) - 1
    n_done = len(runs)
    if n_done < n_exp: W.append(f"mới xong {n_done}/{n_exp} lần chạy USHER của lưới (chia phiên bằng ONLY_RUNS)")
    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in W] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run07.py
"""07 — M_aux không cần nhãn (H7): bỏ phiếu 15 hàng xóm cosine trên embedding đã trừ trung bình (`base_center`), không
dùng nhãn source, không dùng hồi quy logistic. `knn_center`: hàng xóm trong toàn bộ Xenium đã nạp, không gian = center.
`knn_iter`: lặp một lần — hàng xóm trong Xenium GỐC, không gian = map USHER của `knn_center` cùng seed. `lr` (argmax hồi
quy logistic, như 05c/06) chỉ để chấm điểm tham chiếu (acc, err_conc — H7c), KHÔNG chạy USHER với nó.
Cặp cosmx6k -> xenium5k, hai embedding scgpt và niche_data. USHER: cấu hình U6, hidden_dim=None (linear, chốt ở H2/06),
3 seed. Tái sử dụng run06.usher/eval_one/imp_setup/load_all, run05c.shared_lognorm/fit_label_model/strat_target/mix_ref/
cross_assay_ci, run05.base_center/base_class_ridge/bacc_on. Chế độ gọn (plan.md 07): không test riêng, không benchmark
thời gian, không refactor run04/04b/05/05c/06.
"""
import os, sys, glob, json, time, gc
SMOKE = os.environ.get("SMOKE", "0") == "1"
if SMOKE:   # phải đặt TRƯỚC khi import run05/run05c/run06 (đọc biến môi trường lúc import) — như run06.py
    for k, v in {"SKETCH": "300", "N_ITERS": "1", "N_BOOT": "20", "N_TGT": "1500", "N_REF": "1500", "N_MIX": "600"}.items():
        os.environ.setdefault(k, v)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6
from run04 import np, pd, sp, log, load_assay, _unit
from sklearn.neighbors import NearestNeighbors

OUT = os.environ.get("OUT_DIR_7", "/kaggle/working/out07" + ("_smoke" if SMOKE else ""))
TARGET, SRC = M.TARGET, "cosmx6k"
EMBS = ["scgpt", "niche_data"]
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX = 15
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def run_list():
    """Với mỗi seed: scgpt knn_center -> scgpt knn_iter -> niche knn_center -> niche knn_iter (thứ tự plan mục 4)."""
    return [(emb, cond, seed) for seed in SEEDS for emb in EMBS for cond in ("knn_center", "knn_iter")]

def run_name(emb, cond, seed):
    return f"{emb}_{cond}_s{seed}"


def vote_frac(nb, lab, classes):
    """Tỷ lệ phiếu của các hàng xóm theo `classes` (xác suất mềm, tổng = 1 mỗi hàng)."""
    ci = pd.Index(classes).get_indexer(lab); assert (ci >= 0).all()
    n, k = nb.shape
    P = np.zeros((n, len(classes)), np.float32)
    np.add.at(P, (np.repeat(np.arange(n), k), ci[nb].ravel()), 1)
    return P / k

def knn_aux(Fs_space, Fx_all, lab_x, classes, k=K_AUX):
    """M_aux không nhãn cho MỌI tế bào source: bỏ phiếu k hàng xóm cosine trong TOÀN BỘ Xenium đã nạp."""
    nb = NearestNeighbors(n_neighbors=k, n_jobs=4).fit(_unit(Fx_all)).kneighbors(_unit(Fs_space), return_distance=False)
    return vote_frac(nb, lab_x, classes)

def err_conc(lab_true, argmax_pred, classes_shared, min_wrong=20):
    """H7c: trung bình qua nhóm chung (tính trên T) của tỷ lệ lỗi tập trung vào MỘT lớp sai phổ biến nhất; bỏ nhóm
    có < 20 tế bào bị dự đoán sai. Cao = lỗi có hệ thống (dồn vào 1 lớp); thấp = lỗi rải đều."""
    out = []
    for c in classes_shared:
        m = lab_true == c
        wrong = argmax_pred[m] != c
        n_wrong = int(wrong.sum())
        if n_wrong < min_wrong: continue
        top = int(pd.Series(argmax_pred[m][wrong]).value_counts().iloc[0])
        out.append(top / n_wrong)
    return round(float(np.mean(out)), 4) if out else None


# =============================================================================== chạy
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    if not SMOKE:   # bài học 05: không để cấu hình lệch khỏi U6
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER U6 sketch={C.SKETCH} fix=True stratified=True n_iters={C.N_ITERS} eps=0.1 "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine hidden_dim=None(linear, chốt H2/06) "
           f"sketch_obsm_key=X_none cache=off | seeds={SEEDS} embeddings={EMBS} | aux kNN k={K_AUX} trên toàn bộ Xenium "
           f"đã nạp | N_TGT={M.N_TGT} N_REF={M.N_REF} N_MIX={M.N_MIX} N_BOOT={C.N_BOOT}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt"), "niche_data": M.find_dir("emb_niche_data")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({SRC, TARGET}), EMB)
    ds, dx = D[SRC], D[TARGET]
    S_shared = sorted(set(REC[SRC]) & set(REC[TARGET]))
    Fs = {emb: F[(SRC, emb)] for emb in EMBS}
    Fx = {emb: F[(TARGET, emb)] for emb in EMBS}
    acc = lambda lab_arr, idx: M.bacc_on(ds["lab"][idx], lab_arr[idx], S_shared)

    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("aux", {}); S.setdefault("base", {}); S.setdefault("runs", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    # ---- lr: chỉ để chấm điểm tham chiếu (H7c), KHÔNG chạy USHER
    step("hồi quy logistic (tham chiếu H7c, như 05c/06)")
    X, var = EXPR[SRC]; Xx, vx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(X, var, Xx, vx)
    P_lr, classes = C.fit_label_model(Ys, Yx, dx["lab"])
    pl = classes[P_lr.argmax(1)]
    genes = sorted(set(var["symbol"].str.upper()) & set(vx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1], (len(genes), nG)
    _, _, _, ref = C.mix_ref(SRC, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"])
    del Ys, Yx; gc.collect()
    lr_stat = {"acc_T": acc(pl, ds["T"]), "acc_H": acc(pl, ds["H"]), "err_conc": err_conc(ds["lab"][ds["T"]], pl[ds["T"]], S_shared)}
    lr_direct = C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=pl)["lr_direct"]
    for emb in EMBS:
        S["aux"].setdefault(emb, {})["lr"] = lr_stat            # lr không phụ thuộc embedding; in cho cả hai để khớp bảng
        S["base"].setdefault(emb, {})["lr_aux_direct"] = lr_direct
    save()
    log(f"  lr: {nG} gene | {lr_stat}")

    # ---- raw, center, knn_center (aux) mỗi embedding
    KC_AUX = {}
    for emb in EMBS:
        step(f"{emb}: raw, center, knn_center")
        center_map = M.base_center(Fs[emb], Fx[emb], ds["T"])
        b = S["base"][emb]
        if "raw" not in b:
            b["raw"] = R6.eval_one(SRC, D, Fs[emb], Fx[emb], "raw", Fs[emb], S_shared, IMP, ref)
            b["center"] = R6.eval_one(SRC, D, Fs[emb], Fx[emb], "center", center_map, S_shared, IMP, ref)
            save()
        P_kc = knn_aux(center_map, Fx[emb], dx["lab"], classes)
        kc_lab = classes[P_kc.argmax(1)]
        KC_AUX[emb] = (P_kc, kc_lab)
        S["aux"][emb]["knn_center"] = {"acc_T": acc(kc_lab, ds["T"]), "acc_H": acc(kc_lab, ds["H"]),
                                        "err_conc": err_conc(ds["lab"][ds["T"]], kc_lab[ds["T"]], S_shared)}
        if "knn_center_aux_direct" not in b:
            b["knn_center_aux_direct"] = C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=kc_lab)["lr_direct"]
            ridge_map, alpha_r = M.base_class_ridge(Fs[emb], Fx[emb], ds["T"], kc_lab, dx["lab"])
            b["knn_center_class_ridge_aux"] = R6.eval_one(SRC, D, Fs[emb], Fx[emb], "class_ridge_aux", ridge_map, S_shared, IMP, ref)
            b["knn_center_class_ridge_aux"]["alpha"] = alpha_r
            save()
        log(f"  [{emb}] knn_center: {S['aux'][emb]['knn_center']}")

    # ---- USHER: knn_center (aux tất định) và knn_iter (aux từ map knn_center CÙNG SEED)
    def run_and_eval(emb, cond, seed, P_aux, lab_aux):
        name = run_name(emb, cond, seed); f = f"{OUT}/{name}.npy"
        if RESUME and os.path.exists(f) and "eval" in S["runs"].get(name, {}):
            step(f"{name}: đã có, bỏ qua"); return np.load(f).astype(np.float32)
        if RESUME and os.path.exists(f) and "info" in S["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), S["runs"][name]["info"]; step(f"{name}: nạp lại map")
        else:
            tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            step(f"USHER {name} | target {len(tgt)} tế bào phân tầng theo argmax aux")
            Fmap, info = R6.usher(Fs[emb], ds["T"], P_aux, Fx[emb], tgt, dx["lab"], classes, "linear", seed, False, False)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            S["runs"][name] = {"emb": emb, "cond": cond, "seed": seed, "info": info}; save()
        ev = R6.eval_one(SRC, D, Fs[emb], Fx[emb], name, Fmap, S_shared, IMP, ref)
        S["runs"][name]["eval"] = ev; save()
        log(f"  >> {name}: EM={S['runs'][name]['info'].get('em_seconds')}s xfer_S={ev['xfer_S']} {ev['xfer_S_ci']} "
            f"nội_suy_sp={ev['imp']['spearman_med']}")
        return Fmap

    KC_MAP = {}
    def get_kc_map(emb, seed):
        if (emb, seed) not in KC_MAP:
            P_aux, lab_aux = KC_AUX[emb]
            KC_MAP[(emb, seed)] = run_and_eval(emb, "knn_center", seed, P_aux, lab_aux)
        return KC_MAP[(emb, seed)]

    for emb, cond, seed in runs:
        if cond == "knn_center":
            get_kc_map(emb, seed)
        else:                                                   # knn_iter: cần map knn_center CÙNG SEED trước (plan mục 4)
            kc_map = get_kc_map(emb, seed)
            P_aux = knn_aux(kc_map, Fx[emb], dx["lab"], classes)   # nb: source đã map, Xenium GỐC (giữ nguyên)
            lab_aux = classes[P_aux.argmax(1)]
            S["aux"][emb].setdefault("knn_iter_seeds", {})[str(seed)] = {
                "acc_T": acc(lab_aux, ds["T"]), "acc_H": acc(lab_aux, ds["H"]),
                "err_conc": err_conc(ds["lab"][ds["T"]], lab_aux[ds["T"]], S_shared)}
            S["base"][emb].setdefault("knn_iter_aux_direct_seeds", {})[str(seed)] = \
                C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=lab_aux)["lr_direct"]
            save()
            run_and_eval(emb, "knn_iter", seed, P_aux, lab_aux)
        gc.collect()

    md = summary_md(S)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG")


# =============================================================================== summary.md
def summary_md(S):
    L, W = ["# 07 — tóm tắt", S["config"], ""], []
    L.append("## Aux (acc trên nhóm chung; err_conc = H7c, cao = lỗi dồn vào 1 lớp)")
    rows = []
    for emb in EMBS:
        a = S["aux"].get(emb, {})
        if "lr" in a: rows.append({"aux": "lr", "emb": emb, **a["lr"]})
        if "knn_center" in a: rows.append({"aux": "knn_center", "emb": emb, **a["knn_center"]})
        ki = list(a.get("knn_iter_seeds", {}).values())
        if ki:
            rows.append({"aux": "knn_iter", "emb": emb, "acc_T": R6._ms([v["acc_T"] for v in ki]),
                         "acc_H": R6._ms([v["acc_H"] for v in ki]), "err_conc": R6._ms([v["err_conc"] for v in ki])})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## Chuyển nhãn (USHER: TB±SD qua 3 seed; Δcen = xfer_S − center)")
    for emb in EMBS:
        b = S["base"].get(emb, {})
        runs_e = [v for v in S["runs"].values() if v.get("emb") == emb and "eval" in v]
        cen = b.get("center", {}).get("xfer_S")
        def row(name, ev):
            x = ev.get("xfer_S")
            return {"map": name, "xfer_S": x, "xfer_thô": ev.get("xfer_coarse"), "keep_cos": ev.get("keep_struct_cos"),
                    "nội_suy_sp": (ev.get("imp") or {}).get("spearman_med"),
                    "Δcen": round(x - cen, 3) if x is not None and cen is not None else None}
        rr = []
        for name, key in [("raw", "raw"), ("center", "center"), ("aux_direct_lr", "lr_aux_direct"),
                          ("aux_direct_knn_center", "knn_center_aux_direct"), ("class_ridge_aux_knn_center", "knn_center_class_ridge_aux")]:
            if key in b: rr.append(row(name, b[key]))
        kiad = list(b.get("knn_iter_aux_direct_seeds", {}).values())
        if kiad: rr.append({"map": "aux_direct_knn_iter", "xfer_S": R6._ms([v["xfer_S"] for v in kiad]),
                            "xfer_thô": None, "keep_cos": None, "nội_suy_sp": None, "Δcen": None})
        for label, cond in [("usher_knn_center", "knn_center"), ("usher_knn_iter", "knn_iter")]:
            rl = [v["eval"] for v in runs_e if v.get("cond") == cond]
            if not rl: continue
            x = float(np.mean([e["xfer_S"] for e in rl]))
            rr.append({"map": f"{label}(n={len(rl)})", "xfer_S": R6._ms([e["xfer_S"] for e in rl]),
                      "xfer_thô": R6._ms([e["xfer_coarse"] for e in rl]), "keep_cos": R6._ms([e["keep_struct_cos"] for e in rl]),
                      "nội_suy_sp": R6._ms([e["imp"]["spearman_med"] for e in rl]),
                      "Δcen": round(x - cen, 3) if cen is not None else None})
            if label == "usher_knn_center" and len(rl) >= 3 and np.std([e["xfer_S"] for e in rl], ddof=1) < 0.002:
                W.append(f"{emb}: SD giữa seed của usher_knn_center ≈ 0 (< 0.002)")
        L.append(f"**{emb}**")
        if rr: L += ["```", pd.DataFrame(rr).to_string(index=False), "```"]

    ems = [v["info"].get("em_seconds") for v in S["runs"].values() if v.get("info", {}).get("em_seconds") is not None]
    L.append(f"Thời gian EM: {R6._ms(ems)} giây (n={len(ems)} lần chạy)")
    n_done = sum(1 for v in S["runs"].values() if "eval" in v)
    if n_done < len(run_list()): W.append(f"mới xong {n_done}/{len(run_list())} lần chạy USHER (chia phiên bằng ONLY_RUNS)")
    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in W] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run08.py
"""08 — Đặc trưng vi môi trường (niche) trong USHER không nhãn (H8). Cặp cosmx6k -> xenium5k.
Biểu diễn tế bào: `scgpt` (embedding 03a) và `pca` (joint_pca của 03d, chép tại chỗ — KHÔNG import run03d/run03c vì
kéo theo nicheformer nặng khi import). Đặc trưng `niche` (dùng chung cho cả hai biểu diễn): cộng count của 15 hàng xóm
không gian gần nhất (bản đầy đủ, lọc high_quality) trên gene chung, PCA 50 thành phần. Biểu diễn gộp `+niche w`:
[cell, w·s·niche], s = tỉ lệ chuẩn L2 trung bình (cell/niche) trên Xenium. Aux `knn_center` (run07.knn_aux) tính riêng
trong mỗi biểu diễn; USHER: run06.usher, cấu hình U6, hidden_dim=None, 3 seed. Chế độ gọn (plan.md 08): không test
riêng, không refactor run03d/run04/04b/05/05c/06/07.
"""
import os, sys, glob, json, time, gc, shutil
SMOKE = os.environ.get("SMOKE", "0") == "1"
if SMOKE:   # phải đặt TRƯỚC khi import run05/run05c/run06/run07 — như run06.py/run07.py
    for k, v in {"SKETCH": "300", "N_ITERS": "1", "N_BOOT": "20", "N_TGT": "1500", "N_REF": "1500", "N_MIX": "600"}.items():
        os.environ.setdefault(k, v)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6, run07 as R7
from run04 import np, pd, sp, log, load_assay, _unit
from sklearn.neighbors import NearestNeighbors

OUT = os.environ.get("OUT_DIR_8", "/kaggle/working/out08" + ("_smoke" if SMOKE else ""))
TARGET, SRC = M.TARGET, "cosmx6k"
REPS = ["scgpt", "pca"]
WS = [1.0, 0.5]
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX   = int(os.environ.get("K_AUX") or 15)
K_NICHE = int(os.environ.get("K_NICHE") or 15)
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def variant_name(w):
    return f"niche_w{w:g}"

ORDER5 = [("scgpt", variant_name(1.0)), ("scgpt", variant_name(0.5)), ("pca", "cell"),
          ("pca", variant_name(1.0)), ("pca", variant_name(0.5))]

def run_list():
    """Thứ tự theo seed (plan mục 4): mỗi seed chạy 5 biến thể theo ORDER5 -> 15 lần. `scgpt cell` không ở đây (nạp từ 07)."""
    return [(rep, variant, seed) for seed in SEEDS for (rep, variant) in ORDER5]

def run_name(rep, variant, seed):
    return f"{rep}_{variant}_s{seed}"


# =============================================================================== gene chung + biểu diễn `pca`, `niche`
def gene_cols(var_a, var_b):
    """Cột gene chung (symbol viết hoa, bỏ trùng) của var_a, var_b. -> (idx_a, idx_b, danh sách symbol)."""
    sa, sb = var_a["symbol"].str.upper(), var_b["symbol"].str.upper()
    fa = pd.Series(np.arange(len(sa)), index=sa.values)[~sa.duplicated().values]
    fb = pd.Series(np.arange(len(sb)), index=sb.values)[~sb.duplicated().values]
    G = sorted(set(fa.index) & set(fb.index))
    return fa.loc[G].to_numpy(), fb.loc[G].to_numpy(), G

def pca_cell(Xx, varx, Xs, vars_):
    """Chép từ run03d.joint_pca (log CP10k, HVG, scale, 50 PC), KHÔNG trừ trung bình theo assay."""
    ix, is_, G = gene_cols(varx, vars_)
    Xj = sp.vstack([Xx[:, ix], Xs[:, is_]]).tocsr()
    P, _ = R.pca_expr(Xj)
    nx = Xx.shape[0]
    return P[:nx], P[nx:], len(G)

def build_niche(assay, lite_ids, genes_upper, k=K_NICHE):
    """Bản đầy đủ (lọc high_quality) -> cộng count của k hàng xóm không gian gần nhất (µm, trong cùng assay — dữ liệu
    không có cột lát/slice riêng, mỗi assay chỉ có toạ độ của chính nó) trên gene chung, cho MỖI tế bào lite.
    -> (ma trận thưa n_lite x len(genes_upper), trung vị khoảng cách hàng xóm thứ k, số tế bào lite không khớp)."""
    p = R._find(R.FULL, R.FULL_REL[assay])
    Xf, obsf, varf, xy = R.read_h5(p, ("high_quality",))
    hq = obsf["high_quality"].astype(int).to_numpy() == 1
    Xf, full_ids, xy = Xf[hq], obsf.index[hq].astype(str), xy[hq]
    sf = varf["symbol"].str.upper()
    fmap = pd.Series(np.arange(len(sf)), index=sf.values)[~sf.duplicated().values]
    gcols = fmap.reindex(genes_upper)
    assert gcols.notna().all(), f"{assay}: {int(gcols.isna().sum())} gene chung không có ở bản đầy đủ"
    Xf_g = Xf[:, gcols.to_numpy(dtype=int)].tocsr()
    pos = pd.Index(full_ids).get_indexer(lite_ids)
    missing = int((pos < 0).sum()); frac = missing / len(lite_ids)
    if missing: log(f"  [{assay}] {missing}/{len(lite_ids)} tế bào lite ({frac:.2%}) KHÔNG khớp bản đầy đủ -> niche = 0")
    assert frac <= 0.01, f"{assay}: thiếu quá 1% ({missing}/{len(lite_ids)})"
    valid = np.where(pos >= 0)[0]; q = pos[valid]
    dist, idx = NearestNeighbors(n_neighbors=k + 1, n_jobs=4).fit(xy).kneighbors(xy[q])
    keep = idx != q[:, None]; keep[keep.sum(1) > k, -1] = False   # phòng nhiều điểm trùng toạ độ với chính nó
    nb = idx[keep].reshape(len(q), k); d15 = float(np.median(dist[keep].reshape(len(q), k)[:, -1]))
    rows = np.repeat(valid, k)
    W = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, nb.ravel())), shape=(len(lite_ids), Xf_g.shape[0]))
    return (W @ Xf_g).tocsr(), d15, missing

def niche_pca(niche_x, niche_s):
    Xj = sp.vstack([niche_x, niche_s]).tocsr()
    P, _ = R.pca_expr(Xj)
    return P[:niche_x.shape[0]], P[niche_x.shape[0]:]


# =============================================================================== chạy
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER U6 sketch={C.SKETCH} fix=True n_iters={C.N_ITERS} hidden_dim=None(linear) "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine sketch_obsm_key=X_none cache=off | "
           f"seeds={SEEDS} reps={REPS} w={WS} | k_aux={K_AUX} k_niche={K_NICHE} niche_pc=50 | "
           f"N_TGT={M.N_TGT} N_REF={M.N_REF} N_BOOT={C.N_BOOT}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({SRC, TARGET}), EMB)
    ds, dx = D[SRC], D[TARGET]
    S_shared = sorted(set(REC[SRC]) & set(REC[TARGET]))
    classes = np.array(sorted(set(dx["lab"])))                    # = classes_ của hồi quy logistic trên dx["lab"] (05c/06/07)
    acc = lambda lab_arr, idx: M.bacc_on(ds["lab"][idx], lab_arr[idx], S_shared)

    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("aux", {}); S.setdefault("base", {}); S.setdefault("runs", {}); S.setdefault("meta", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    # ---- imp_setup (không phụ thuộc embedding, như 06/07)
    step("shared_lognorm + imp_setup")
    Xs_lite, vars_ = EXPR[SRC]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1], (len(genes), nG)
    _, _, _, ref = C.mix_ref(SRC, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"])
    del Ys, Yx; gc.collect()

    # ---- biểu diễn `pca` (gene chung DÙNG CHUNG cho cả `pca cell` và `niche`, xem Ghi chú triển khai)
    Fx, Fs = {}, {}
    if F.get((TARGET, "scgpt")) is not None: Fx["scgpt"], Fs["scgpt"] = F[(TARGET, "scgpt")], F[(SRC, "scgpt")]
    step("pca cell (joint_pca chép từ 03d)")
    Fx["pca"], Fs["pca"], n_gene_pca = pca_cell(Xx_lite, varx, Xs_lite, vars_)
    _, _, G_pca = gene_cols(varx, vars_)
    log(f"  pca cell: {n_gene_pca} gene chung")

    # ---- đặc trưng niche (dùng chung cho scgpt và pca), gene chung = G_pca
    if "niche" not in S["meta"]:
        step("niche: đọc bản đầy đủ + gộp hàng xóm không gian (cosmx6k, xenium5k)")
        _, obs_x, _, _ = load_assay(TARGET); _, obs_s, _, _ = load_assay(SRC)
        if SMOKE:   # khớp đúng tập con mà run06.load_all đã lấy (cùng seed mặc định của B.pick)
            obs_x = obs_x.iloc[B.pick(np.arange(len(obs_x)), R6.N_SMOKE)]
            obs_s = obs_s.iloc[B.pick(np.arange(len(obs_s)), R6.N_SMOKE)]
        niche_x_cnt, d15_x, miss_x = build_niche(TARGET, obs_x.index.astype(str), G_pca)
        niche_s_cnt, d15_s, miss_s = build_niche(SRC, obs_s.index.astype(str), G_pca)
        niche_x, niche_s = niche_pca(niche_x_cnt, niche_s_cnt)
        np.save(f"{OUT}/niche_xenium5k.npy", niche_x.astype(np.float16)); np.save(f"{OUT}/niche_cosmx6k.npy", niche_s.astype(np.float16))
        S["meta"]["niche"] = {"d15_um": {TARGET: round(d15_x, 2), SRC: round(d15_s, 2)}, "missing": {TARGET: miss_x, SRC: miss_s},
                              "n_lite": {TARGET: len(obs_x), SRC: len(obs_s)}, "n_gene": len(G_pca)}
        save()
    else:
        niche_x = np.load(f"{OUT}/niche_xenium5k.npy").astype(np.float32); niche_s = np.load(f"{OUT}/niche_cosmx6k.npy").astype(np.float32)
    log(f"  niche: {S['meta']['niche']}")

    # ---- biểu diễn gộp +niche w = [cell, w*s*niche]; s = tỉ lệ chuẩn L2 trung bình (cell/niche) trên Xenium
    REP = {("scgpt", "cell"): (Fs["scgpt"], Fx["scgpt"]), ("pca", "cell"): (Fs["pca"], Fx["pca"])}
    for rep in REPS:
        s = float(np.mean(np.linalg.norm(Fx[rep], axis=1))) / float(np.mean(np.linalg.norm(niche_x, axis=1)))
        for w in WS:
            REP[(rep, variant_name(w))] = (np.hstack([Fs[rep], (w * s) * niche_s]), np.hstack([Fx[rep], (w * s) * niche_x]))
        S["meta"][f"scale_s_{rep}"] = round(s, 4)
    save()

    # ---- aux knn_center + baseline (raw, center, aux_direct) cho MỖI (rep, biến thể)
    AUX, CENTER = {}, {}
    for key, (Fsv, Fxv) in REP.items():
        name = f"{key[0]}_{key[1]}"
        center_v = M.base_center(Fsv, Fxv, ds["T"]); CENTER[key] = center_v
        P_aux = R7.knn_aux(center_v, Fxv, dx["lab"], classes, k=K_AUX)
        lab_aux = classes[P_aux.argmax(1)]; AUX[key] = (P_aux, lab_aux)
        S["aux"][name] = {"acc_T": acc(lab_aux, ds["T"]), "acc_H": acc(lab_aux, ds["H"]),
                          "err_conc": R7.err_conc(ds["lab"][ds["T"]], lab_aux[ds["T"]], S_shared)}
        b = S["base"].setdefault(name, {})
        if "raw" not in b:
            b["raw"] = R6.eval_one(SRC, D, Fsv, Fxv, "raw", Fsv, S_shared, IMP, ref)
            b["center"] = R6.eval_one(SRC, D, Fsv, Fxv, "center", center_v, S_shared, IMP, ref)
            b["aux_direct"] = C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=lab_aux)["lr_direct"]
            save()
        log(f"  [{name}] aux: {S['aux'][name]} | center xfer_S={b['center']['xfer_S']}")

    # ---- USHER
    def run_and_eval(rep, variant, seed):
        key = (rep, variant); name = run_name(rep, variant, seed); f = f"{OUT}/{name}.npy"
        Fsv, Fxv = REP[key]; P_aux, lab_aux = AUX[key]
        if RESUME and os.path.exists(f) and "eval" in S["runs"].get(name, {}):
            step(f"{name}: đã có, bỏ qua"); return
        if RESUME and os.path.exists(f) and "info" in S["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), S["runs"][name]["info"]; step(f"{name}: nạp lại map")
        else:
            tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            step(f"USHER {name} | target {len(tgt)} tế bào")
            Fmap, info = R6.usher(Fsv, ds["T"], P_aux, Fxv, tgt, dx["lab"], classes, "linear", seed, False, False)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            S["runs"][name] = {"rep": rep, "variant": variant, "seed": seed, "info": info}; save()
        ev = R6.eval_one(SRC, D, Fsv, Fxv, name, Fmap, S_shared, IMP, ref)
        S["runs"][name]["eval"] = ev; save()
        log(f"  >> {name}: EM={S['runs'][name]['info'].get('em_seconds')}s xfer_S={ev['xfer_S']} {ev['xfer_S_ci']} "
            f"nội_suy_sp={ev['imp']['spearman_med']}")
        del Fmap; gc.collect()

    # scgpt cell: nạp map knn_center của 07 (không chạy lại USHER); thiếu thì tự chạy (plan mục 4)
    for seed in SEEDS:
        name = run_name("scgpt", "cell", seed)
        if "eval" in S["runs"].get(name, {}): step(f"{name}: đã có, bỏ qua"); continue
        src07 = sorted(glob.glob(f"/kaggle/input/**/out07/scgpt_knn_center_s{seed}.npy", recursive=True))
        if not src07:
            log(f"CẢNH BÁO: thiếu out07/scgpt_knn_center_s{seed}.npy -> tự chạy USHER scgpt cell seed {seed}")
            run_and_eval("scgpt", "cell", seed); continue
        shutil.copy(src07[0], f"{OUT}/{name}.npy")
        Fmap = np.load(f"{OUT}/{name}.npy").astype(np.float32)
        ev = R6.eval_one(SRC, D, Fs["scgpt"], Fx["scgpt"], name, Fmap, S_shared, IMP, ref)
        S["runs"][name] = {"rep": "scgpt", "variant": "cell", "seed": seed, "info": {}, "eval": ev, "source": "07"}; save()
        log(f"  >> {name} (nạp từ 07): xfer_S={ev['xfer_S']} nội_suy_sp={ev['imp']['spearman_med']}")

    for rep, variant, seed in runs:
        run_and_eval(rep, variant, seed)

    md = summary_md(S)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG")


# =============================================================================== summary.md
def _row_usher(S, rep, variant):
    rl = [v["eval"] for v in S["runs"].values() if v.get("rep") == rep and v.get("variant") == variant and "eval" in v]
    if not rl: return None
    return {"xfer_S": [e["xfer_S"] for e in rl], "xfer_thô": [e["xfer_coarse"] for e in rl],
            "keep_cos": [e["keep_struct_cos"] for e in rl], "sp": [e["imp"]["spearman_med"] for e in rl]}

def summary_md(S):
    L, W = ["# 08 — tóm tắt", S["config"], ""], []
    nm = S["meta"].get("niche", {})
    L.append(f"Khoảng cách hàng xóm thứ {K_NICHE} (µm, trung vị): {nm.get('d15_um')} | tế bào lite khớp bản đầy đủ: "
             f"{ {k: nm.get('n_lite', {}).get(k, 0) - nm.get('missing', {}).get(k, 0) for k in nm.get('n_lite', {})} } "
             f"/ {nm.get('n_lite')} | {nm.get('n_gene')} gene chung | scale s: " +
             ", ".join(f"{k[8:]}={v}" for k, v in S["meta"].items() if k.startswith("scale_s_")))

    L.append("## Aux + xfer_S (USHER: TB±SD qua 3 seed)")
    rows = []
    for rep in REPS:
        for variant in ["cell", variant_name(1.0), variant_name(0.5)]:
            name = f"{rep}_{variant}"
            if name not in S["aux"]: continue
            a, b = S["aux"][name], S["base"].get(name, {})
            u = _row_usher(S, rep, variant)
            rows.append({"rep": rep, "biến thể": variant, "acc_H": a["acc_H"], "err_conc": a["err_conc"],
                        "center": b.get("center", {}).get("xfer_S"), "aux_direct": b.get("aux_direct", {}).get("xfer_S"),
                        "usher_xfer_S": R6._ms(u["xfer_S"]) if u else "—", "usher_thô": R6._ms(u["xfer_thô"]) if u else "—",
                        "usher_keep_cos": R6._ms(u["keep_cos"]) if u else "—", "usher_sp": R6._ms(u["sp"]) if u else "—",
                        "center_sp": b.get("center", {}).get("imp", {}).get("spearman_med")})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    def mean_sd(rep, variant):
        u = _row_usher(S, rep, variant)
        return (None, None) if not u else (float(np.mean(u["xfer_S"])), float(np.std(u["xfer_S"], ddof=1)) if len(u["xfer_S"]) > 1 else 0.0)
    def beats(a, b, extra=0.0):
        (ma, sa), (mb, sb) = a, b
        if ma is None or mb is None: return None
        return round(ma - mb, 3), (ma - mb) >= max(0.02, 2 * max(sa, sb)) - extra

    L.append("## N1–N3 (thô)")
    for rep in REPS:
        d = beats(mean_sd(rep, variant_name(1.0)), mean_sd(rep, "cell"))
        if d: L.append(f"- N1 {rep}: usher(+niche w1) − usher(cell) = {d[0]} -> {'ĐẠT' if d[1] else 'không đạt'}")
    for rep in REPS:
        cb = S["base"].get(f"{rep}_{variant_name(1.0)}", {}).get("center", {}).get("xfer_S")
        cc = S["base"].get(f"{rep}_cell", {}).get("center", {}).get("xfer_S")
        if cb is not None and cc is not None:
            L.append(f"- N2 {rep}: center(+niche w1) − center(cell) = {round(cb - cc, 3)} -> {'ĐẠT' if cb - cc >= 0.02 else 'không đạt'}")
    d3 = beats(mean_sd("pca", variant_name(1.0)), mean_sd("scgpt", variant_name(1.0)))
    if d3:
        n3 = d3[1] or abs(d3[0]) < 0.02
        L.append(f"- N3: usher(pca+niche w1) − usher(scgpt+niche w1) = {d3[0]} -> {'không cần SFM' if n3 else 'SFM vẫn cần'}")
    pca_cell_center = S["base"].get("pca_cell", {}).get("center", {}).get("xfer_S")
    if pca_cell_center is not None:
        ok = abs(pca_cell_center - 0.76) <= 0.03
        L.append(f"- Tái lập 03d: center(pca cell) = {pca_cell_center} (kỳ vọng 0.76±0.03) -> {'khớp' if ok else 'LỆCH'}")
        if not ok: W.append(f"center(pca cell) = {pca_cell_center} lệch khỏi 0.76±0.03 của 03d")

    ems = [v["info"].get("em_seconds") for v in S["runs"].values() if v.get("info", {}).get("em_seconds") is not None]
    L.append(f"Thời gian EM: {R6._ms(ems)} giây (n={len(ems)} lần chạy)")
    n_done = sum(1 for v in S["runs"].values() if "eval" in v)
    n_exp = len(run_list()) + len(SEEDS)
    if n_done < n_exp: W.append(f"mới xong {n_done}/{n_exp} lần chạy (kể cả scgpt cell nạp từ 07); chia phiên bằng ONLY_RUNS")
    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in W] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run09.py
"""09 — Đo lại mọi map (06, 07, 08) với giao thức sửa gốc cosine (chỉ CPU, không chạy USHER, không nhúng src_cuda).

Bug đã phát hiện (RESEARCH_CONTEXT mục 3, "[ĐANG SỬA] Gốc toạ độ của cosine"): `base_center` dời source về trung bình
Xenium rồi đo cosine quanh gốc 0 (Xenium KHÔNG trừ trung bình); 03d trừ trung bình CẢ HAI assay trước khi đo cosine.
Cosine phụ thuộc gốc toạ độ -> `center` của 05–08 thấp hơn 03d khoảng 0.04.

Giao thức (plan.md mục 3):
  old : `run06.eval_one` như cũ (không đổi) — phải tái lập số đã báo cáo ở 06/07/08 trong ±0.002.
  new : gọi μx = trung bình biểu diễn Xenium (mọi tế bào đã nạp, đúng biểu diễn đang xét). Với mỗi map, gọi
        `run06.eval_one` trên (Fmap − μx, Fx − μx), Fs truyền cho keep = Fs − Fs.mean(0). Aux kNN cũng trừ μx hai phía.
Không chạy USHER: chỉ nạp lại map .npy đã có (out06/out07/out08) và dựng lại biểu diễn gốc (embedding 03a, `pca`/`niche`
của run08) để tính lại xfer_S/xfer_coarse/keep_cos/nội suy/aux dưới cả hai giao thức, rồi so tiêu chí cũ của 06/07/08.
"""
import os, sys, glob, json, gc
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6, run07 as R7, run08 as R8
from run04 import np, pd, sp, log, load_assay, _unit

OUT = os.environ.get("OUT_DIR_9", "/kaggle/working/out09")
TARGET, SRC = M.TARGET, "cosmx6k"


def find_input(sub, name):
    h = sorted(glob.glob(f"/kaggle/input/**/{sub}/{name}", recursive=True))
    return h[0] if h else None

def ms(vals):
    v = [x for x in vals if x is not None]
    if not v: return None, None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)

def beats(ma, sa, mb, sb):
    if ma is None or mb is None: return None
    thr = round(max(0.02, 2 * max(sa or 0, sb or 0)), 4)
    d = round(ma - mb, 4)
    return {"delta": d, "thr": thr, "pass": d >= thr}

def crit_line(label, old, new):
    if old is None or new is None: return f"- {label}: thiếu dữ liệu"
    chg = " -> ĐỔI KẾT LUẬN" if old["pass"] != new["pass"] else ""
    return (f"- {label}: cũ Δ={old['delta']} (ngưỡng {old['thr']}) {'ĐẠT' if old['pass'] else 'không'} | "
            f"mới Δ={new['delta']} (ngưỡng {new['thr']}) {'ĐẠT' if new['pass'] else 'không'}{chg}")


def eval_both(SRC_, D, Fs, Fx, name, Fmap, S_shared, IMP, ref):
    """`run06.eval_one` dưới giao thức cũ và mới (mục 3): mới trừ μx = Fx.mean(0) ở Fmap/Fx; Fs cho keep = Fs-Fs.mean(0)."""
    old = R6.eval_one(SRC_, D, Fs, Fx, name, Fmap, S_shared, IMP, ref)
    mu = Fx.mean(0)
    new = R6.eval_one(SRC_, D, Fs - Fs.mean(0), Fx - mu, name, Fmap - mu, S_shared, IMP, ref)
    return old, new

def aux_both(space, Fx, classes, lab_x, k=15):
    """`run07.knn_aux` dưới giao thức cũ và mới (trừ μx = Fx.mean(0) ở cả hai phía)."""
    P_old = R7.knn_aux(space, Fx, lab_x, classes, k=k)
    mu = Fx.mean(0)
    P_new = R7.knn_aux(space - mu, Fx - mu, lab_x, classes, k=k)
    return classes[P_old.argmax(1)], classes[P_new.argmax(1)]

def acc_row(ds, dx, S_shared, lab_arr):
    return {"acc_T": M.bacc_on(ds["lab"][ds["T"]], lab_arr[ds["T"]], S_shared),
            "acc_H": M.bacc_on(ds["lab"][ds["H"]], lab_arr[ds["H"]], S_shared),
            "err_conc": R7.err_conc(ds["lab"][ds["T"]], lab_arr[ds["T"]], S_shared)}


# Số đã báo cáo (RESEARCH_CONTEXT 5.3/5.4/5.5), dưới giao thức CŨ — kiểm tra tái lập ±0.002
EXPECT_06 = {"center": 0.621, "true_linear": 0.813, "struct_0.33_linear": 0.807, "struct_0.67_linear": 0.688,
             "struct_1.0_linear": 0.516, "unif_0.33_linear": 0.803, "unif_0.67_linear": 0.794, "unif_1.0_linear": 0.690,
             "soft_lr_linear": 0.517, "true_mlp": 0.808, "soft_lr_mlp": 0.486}
EXPECT_07 = {"scgpt_center": 0.621, "scgpt_knn_center": 0.696, "scgpt_knn_iter": 0.700,
             "niche_data_center": 0.488, "niche_data_knn_center": 0.571, "niche_data_knn_iter": 0.607}
EXPECT_08 = {"scgpt_cell_center": 0.621, "scgpt_cell_usher": 0.696, "scgpt_niche_w1_center": 0.656,
             "scgpt_niche_w1_usher": 0.735, "pca_cell_center": 0.719, "pca_cell_usher": 0.798,
             "pca_niche_w1_center": 0.756, "pca_niche_w1_usher": 0.798}


def main():
    os.makedirs(OUT, exist_ok=True)
    W = []
    EMB = {"scgpt": M.find_dir("emb_scgpt"), "niche_data": M.find_dir("emb_niche_data")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({SRC, TARGET}), EMB)
    ds, dx = D[SRC], D[TARGET]
    S_shared = sorted(set(REC[SRC]) & set(REC[TARGET]))
    acc = lambda l, idx=ds["T"]: M.bacc_on(ds["lab"][idx], l[idx], S_shared)
    log(f"S chung = {S_shared} | scgpt={'OK' if F.get((SRC,'scgpt')) is not None else 'THIẾU'} "
        f"niche_data={'OK' if F.get((SRC,'niche_data')) is not None else 'THIẾU'}")

    Xs_lite, vars_ = EXPR[SRC]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1]
    _, _, _, ref = C.mix_ref(SRC, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
    P_lr, classes = C.fit_label_model(*C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)[:2], dx["lab"])

    S = {"06": {}, "07": {}, "08": {}, "aux": {}, "criteria": [], "repro": []}

    # =========================================================================== 06: aux tất định (build_aux) + map
    step_maps06 = {}
    if F.get((SRC, "scgpt")) is not None:
        Fs06, Fx06 = F[(SRC, "scgpt")], F[(TARGET, "scgpt")]
        AUX06, frac06 = R6.build_aux(ds["lab"], P_lr, classes, ds["T"], acc)
        aux_dir = {c: C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=l)["lr_direct"]["xfer_S"] for c, (_, l) in AUX06.items()}
        for name06, Fmap0 in [("raw", Fs06), ("center", M.base_center(Fs06, Fx06, ds["T"]))]:
            old, new = eval_both(SRC, D, Fs06, Fx06, name06, Fmap0, S_shared, IMP, ref)
            S["06"][name06] = {"old": old, "new": new}
        for cond, mstep, seed, seq in R6.run_list():
            if seq: continue
            fp = find_input("out06", f"A_{cond}_{mstep}_s{seed}.npy")
            if not fp: W.append(f"06: thiếu A_{cond}_{mstep}_s{seed}.npy"); continue
            Fmap = np.load(fp).astype(np.float32)
            old, new = eval_both(SRC, D, Fs06, Fx06, f"{cond}_{mstep}_s{seed}", Fmap, S_shared, IMP, ref)
            step_maps06.setdefault((cond, mstep), []).append((old, new))
            del Fmap
        for (cond, mstep), pairs in step_maps06.items():
            S["06"][f"{cond}_{mstep}"] = {"old": [p[0] for p in pairs], "new": [p[1] for p in pairs],
                                          "aux_dir": aux_dir.get(cond)}
    else:
        W.append("06: thiếu embedding scgpt -> bỏ qua cả nhóm 06")

    # =========================================================================== 07: knn_center/knn_iter, 2 embedding
    for emb in R7.EMBS:
        if F.get((SRC, emb)) is None: W.append(f"07/{emb}: thiếu embedding -> bỏ qua"); continue
        Fs, Fx = F[(SRC, emb)], F[(TARGET, emb)]
        old, new = eval_both(SRC, D, Fs, Fx, "raw", Fs, S_shared, IMP, ref); S["07"][f"{emb}_raw"] = {"old": old, "new": new}
        center_map = M.base_center(Fs, Fx, ds["T"])
        old, new = eval_both(SRC, D, Fs, Fx, "center", center_map, S_shared, IMP, ref); S["07"][f"{emb}_center"] = {"old": old, "new": new}
        lab_old, lab_new = aux_both(center_map, Fx, classes, dx["lab"])
        S["aux"][f"{emb}_knn_center"] = {"old": acc_row(ds, dx, S_shared, lab_old), "new": acc_row(ds, dx, S_shared, lab_new)}
        kc_maps = {}
        for cond in ("knn_center", "knn_iter"):
            olds, news = [], []
            for seed in R7.SEEDS:
                fp = find_input("out07", f"{emb}_{cond}_s{seed}.npy")
                if not fp: W.append(f"07/{emb}/{cond}: thiếu s{seed}"); continue
                Fmap = np.load(fp).astype(np.float32)
                o, n = eval_both(SRC, D, Fs, Fx, f"{emb}_{cond}_s{seed}", Fmap, S_shared, IMP, ref)
                olds.append(o); news.append(n)
                if cond == "knn_center": kc_maps[seed] = Fmap
                else: del Fmap
            S["07"][f"{emb}_{cond}"] = {"old": olds, "new": news}
        # aux knn_iter theo giao thức mới/cũ: kNN trên map knn_center CÙNG SEED (trung bình qua seed cho bảng aux)
        rows_o, rows_n = [], []
        for seed, kcm in kc_maps.items():
            lo, ln = aux_both(kcm, Fx, classes, dx["lab"])
            rows_o.append(acc_row(ds, dx, S_shared, lo)); rows_n.append(acc_row(ds, dx, S_shared, ln))
        if rows_o:
            S["aux"][f"{emb}_knn_iter"] = {"old": {k: ms([r[k] for r in rows_o])[0] for k in rows_o[0]},
                                           "new": {k: ms([r[k] for r in rows_n])[0] for k in rows_n[0]}}
        del kc_maps; gc.collect()

    # =========================================================================== 08: dựng lại pca + niche, 6 biến thể
    _, obs_x, _, _ = load_assay(TARGET); _, obs_s, _, _ = load_assay(SRC)
    lite_x, lite_s = obs_x.index.astype(str), obs_s.index.astype(str)
    if F.get((SRC, "scgpt")) is None:
        W.append("08: thiếu embedding scgpt -> bỏ qua cả nhóm 08")
    else:
        Fx_pca, Fs_pca, n_gene_pca = R8.pca_cell(Xx_lite, varx, Xs_lite, vars_)
        _, _, G_pca = R8.gene_cols(varx, vars_)
        step_niche = find_input("out08", "niche_xenium5k.npy")
        if step_niche and find_input("out08", "niche_cosmx6k.npy"):
            niche_x = np.load(step_niche).astype(np.float32); niche_s = np.load(find_input("out08", "niche_cosmx6k.npy")).astype(np.float32)
        else:
            niche_x_cnt, d15_x, miss_x = R8.build_niche(TARGET, lite_x, G_pca)
            niche_s_cnt, d15_s, miss_s = R8.build_niche(SRC, lite_s, G_pca)
            niche_x, niche_s = R8.niche_pca(niche_x_cnt, niche_s_cnt)
        Fx_08, Fs_08 = {"scgpt": F[(TARGET, "scgpt")], "pca": Fx_pca}, {"scgpt": F[(SRC, "scgpt")], "pca": Fs_pca}
        REP08 = {("scgpt", "cell"): (Fs_08["scgpt"], Fx_08["scgpt"]), ("pca", "cell"): (Fs_08["pca"], Fx_08["pca"])}
        for rep in R8.REPS:
            s = float(np.mean(np.linalg.norm(Fx_08[rep], axis=1))) / float(np.mean(np.linalg.norm(niche_x, axis=1)))
            for w in R8.WS:
                REP08[(rep, R8.variant_name(w))] = (np.hstack([Fs_08[rep], (w * s) * niche_s]), np.hstack([Fx_08[rep], (w * s) * niche_x]))
        files08 = [(r, v, s, R8.run_name(r, v, s)) for (r, v, s) in R8.run_list()]
        files08 += [("scgpt", "cell", s, R8.run_name("scgpt", "cell", s)) for s in R8.SEEDS]
        by_variant = {}
        for rep, variant, seed, fname in files08:
            fp = find_input("out08", f"{fname}.npy")
            if not fp: W.append(f"08/{rep}_{variant}: thiếu s{seed}"); continue
            Fmap = np.load(fp).astype(np.float32)
            Fsv, Fxv = REP08[(rep, variant)]
            assert len(Fmap) == Fsv.shape[0], f"08/{fname}: {len(Fmap)} dòng map != {Fsv.shape[0]} tế bào biểu diễn"
            o, n = eval_both(SRC, D, Fsv, Fxv, fname, Fmap, S_shared, IMP, ref)
            by_variant.setdefault((rep, variant), []).append((o, n)); del Fmap
        for (rep, variant), pairs in by_variant.items():
            name = f"{rep}_{variant}"; Fsv, Fxv = REP08[(rep, variant)]
            center_map = M.base_center(Fsv, Fxv, ds["T"])
            o, n = eval_both(SRC, D, Fsv, Fxv, "center", center_map, S_shared, IMP, ref)
            lab_o, lab_n = aux_both(center_map, Fxv, classes, dx["lab"])
            S["aux"][name] = {"old": acc_row(ds, dx, S_shared, lab_o), "new": acc_row(ds, dx, S_shared, lab_n)}
            ad_o = C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=lab_o)["lr_direct"]
            ad_n = C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=lab_n)["lr_direct"]
            S["08"][name] = {"usher_old": [p[0] for p in pairs], "usher_new": [p[1] for p in pairs],
                             "center_old": o, "center_new": n, "aux_direct_old": ad_o, "aux_direct_new": ad_n}

    md = summary_md(S, EXPECT_06, EXPECT_07, EXPECT_08, W)
    json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG")


# =============================================================================== summary.md
def _sp(x): return None if x is None else x.get("imp", {}).get("spearman_med")

def summary_md(S, EXPECT_06, EXPECT_07, EXPECT_08, W):
    L = ["# 09 — tóm tắt (giao thức cosine: cũ so với sửa gốc = trừ μx Xenium)", ""]
    repro = []

    L.append("## 06 (scgpt, hàng = cond_mstep; USHER gộp TB±SD qua 3 seed)")
    rows = []
    for name, v in S["06"].items():
        if isinstance(v["old"], list):
            mo, so = ms([e["xfer_S"] for e in v["old"]]); mn, sn = ms([e["xfer_S"] for e in v["new"]])
            rows.append({"map": name, "xfer_S_old": f"{mo:.3f}±{so:.3f}" if mo is not None else "—",
                        "xfer_S_new": f"{mn:.3f}±{sn:.3f}" if mn is not None else "—",
                        "sp_old": R6._ms([_sp(e) for e in v["old"]]), "sp_new": R6._ms([_sp(e) for e in v["new"]]),
                        "keep_cos_new": R6._ms([e["keep_struct_cos"] for e in v["new"]])})
            exp = EXPECT_06.get(name)
            if exp is not None and mo is not None and abs(mo - exp) > 0.002: repro.append(f"06 {name}: cũ={mo:.4f} so với {exp} (báo cáo)")
        else:
            rows.append({"map": name, "xfer_S_old": v["old"]["xfer_S"], "xfer_S_new": v["new"]["xfer_S"],
                        "sp_old": _sp(v["old"]), "sp_new": _sp(v["new"]), "keep_cos_new": v["new"]["keep_struct_cos"]})
            exp = EXPECT_06.get(name)
            if exp is not None and abs(v["old"]["xfer_S"] - exp) > 0.002: repro.append(f"06 {name}: cũ={v['old']['xfer_S']} so với {exp}")
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## 07 (hàng = emb_map; USHER gộp TB±SD qua 3 seed)")
    rows = []
    for name, v in S["07"].items():
        if isinstance(v["old"], list):
            mo, so = ms([e["xfer_S"] for e in v["old"]]); mn, sn = ms([e["xfer_S"] for e in v["new"]])
            rows.append({"map": name, "xfer_S_old": f"{mo:.3f}±{so:.3f}" if mo is not None else "—",
                        "xfer_S_new": f"{mn:.3f}±{sn:.3f}" if mn is not None else "—",
                        "sp_old": R6._ms([_sp(e) for e in v["old"]]), "sp_new": R6._ms([_sp(e) for e in v["new"]]),
                        "keep_cos_new": R6._ms([e["keep_struct_cos"] for e in v["new"]])})
            exp = EXPECT_07.get(name)
            if exp is not None and mo is not None and abs(mo - exp) > 0.002: repro.append(f"07 {name}: cũ={mo:.4f} so với {exp}")
        else:
            rows.append({"map": name, "xfer_S_old": v["old"]["xfer_S"], "xfer_S_new": v["new"]["xfer_S"],
                        "sp_old": _sp(v["old"]), "sp_new": _sp(v["new"]), "keep_cos_new": v["new"]["keep_struct_cos"]})
            exp = EXPECT_07.get(name)
            if exp is not None and abs(v["old"]["xfer_S"] - exp) > 0.002: repro.append(f"07 {name}: cũ={v['old']['xfer_S']} so với {exp}")
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## 08 (hàng = rep_biến thể; USHER/center gộp TB±SD qua 3 seed)")
    rows = []
    for name, v in S["08"].items():
        mo, so = ms([e["xfer_S"] for e in v["usher_old"]]); mn, sn = ms([e["xfer_S"] for e in v["usher_new"]])
        rows.append({"map": name, "center_old": v["center_old"]["xfer_S"], "center_new": v["center_new"]["xfer_S"],
                    "aux_dir_old": v["aux_direct_old"]["xfer_S"], "aux_dir_new": v["aux_direct_new"]["xfer_S"],
                    "usher_old": f"{mo:.3f}±{so:.3f}" if mo is not None else "—", "usher_new": f"{mn:.3f}±{sn:.3f}" if mn is not None else "—",
                    "sp_new": R6._ms([_sp(e) for e in v["usher_new"]])})
        for key, exp_key in [("center_old", f"{name}_center"), ("usher_old", f"{name}_usher")]:
            exp = EXPECT_08.get(exp_key)
            val = v["center_old"]["xfer_S"] if key == "center_old" else mo
            if exp is not None and val is not None and abs(val - exp) > 0.002: repro.append(f"08 {exp_key}: cũ={val:.4f} so với {exp}")
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## Aux kNN (07 + 08; acc_H, err_conc)")
    rows = [{"aux": k, "acc_H_old": v["old"]["acc_H"], "acc_H_new": v["new"]["acc_H"],
            "err_conc_old": v["old"]["err_conc"], "err_conc_new": v["new"]["err_conc"]} for k, v in S["aux"].items()]
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    def xfer06(name, proto):
        v = S["06"].get(name)
        if v is None: return None, None
        if isinstance(v[proto], list): return ms([e["xfer_S"] for e in v[proto]])
        return v[proto]["xfer_S"], 0.0
    L.append("## Tiêu chí (không có giả thuyết mới; chỉ chấm lại)")
    for cond in ["struct_0.67", "struct_1.0", "unif_0.67", "unif_1.0"]:
        for mstep in ["linear"]:
            m, s = xfer06(f"{cond}_{mstep}", "old"); mn, sn = xfer06(f"{cond}_{mstep}", "new")
            ad = S["06"].get(f"{cond}_{mstep}", {}).get("aux_dir")
            L.append(crit_line(f"06 A1 {cond}", beats(m, s, ad, 0), beats(mn, sn, ad, 0)))
    cen_o, _ = xfer06("center", "old"); cen_n, _ = xfer06("center", "new")
    for cond in R6.CONDS:
        m, s = xfer06(f"{cond}_linear", "old"); mn, sn = xfer06(f"{cond}_linear", "new")
        L.append(crit_line(f"06 A2 {cond}", beats(m, s, cen_o, 0), beats(mn, sn, cen_n, 0)))
    for f in [0.33, 0.67, 1.0]:
        mu, su = xfer06(f"unif_{f}_linear", "old"); mun, sun = xfer06(f"unif_{f}_linear", "new")
        ms_, ss = xfer06(f"struct_{f}_linear", "old"); msn, ssn = xfer06(f"struct_{f}_linear", "new")
        L.append(crit_line(f"06 A3 f={f} (unif vượt struct)", beats(mu, su, ms_, ss), beats(mun, sun, msn, ssn)))
    for cond in ["true", "soft_lr"]:
        ml, sl = xfer06(f"{cond}_linear", "old"); mln, sln = xfer06(f"{cond}_linear", "new")
        mm, sm = xfer06(f"{cond}_mlp", "old"); mmn, smn = xfer06(f"{cond}_mlp", "new")
        L.append(crit_line(f"06 A4 {cond} (linear vượt mlp)", beats(ml, sl, mm, sm), beats(mln, sln, mmn, smn)))
    for emb in R7.EMBS:
        kc_o, sc_o = ms([e["xfer_S"] for e in S["07"].get(f"{emb}_knn_center", {}).get("old", [])])
        kc_n, sc_n = ms([e["xfer_S"] for e in S["07"].get(f"{emb}_knn_center", {}).get("new", [])])
        cen_o7 = S["07"].get(f"{emb}_center", {}).get("old", {}).get("xfer_S")
        cen_n7 = S["07"].get(f"{emb}_center", {}).get("new", {}).get("xfer_S")
        L.append(crit_line(f"07 H7a {emb}", beats(kc_o, sc_o, cen_o7, 0), beats(kc_n, sc_n, cen_n7, 0)))
        ki_o, si_o = ms([e["xfer_S"] for e in S["07"].get(f"{emb}_knn_iter", {}).get("old", [])])
        ki_n, si_n = ms([e["xfer_S"] for e in S["07"].get(f"{emb}_knn_iter", {}).get("new", [])])
        L.append(crit_line(f"07 H7b {emb}", beats(ki_o, si_o, kc_o, sc_o), beats(ki_n, si_n, kc_n, sc_n)))
    for rep in R8.REPS:
        cell, w1 = f"{rep}_cell", f"{rep}_{R8.variant_name(1.0)}"
        uo1, so1 = ms([e["xfer_S"] for e in S["08"].get(w1, {}).get("usher_old", [])]); un1, sn1 = ms([e["xfer_S"] for e in S["08"].get(w1, {}).get("usher_new", [])])
        uoc, soc = ms([e["xfer_S"] for e in S["08"].get(cell, {}).get("usher_old", [])]); unc, snc = ms([e["xfer_S"] for e in S["08"].get(cell, {}).get("usher_new", [])])
        L.append(crit_line(f"08 N1 {rep}", beats(uo1, so1, uoc, soc), beats(un1, sn1, unc, snc)))
        c1o = S["08"].get(w1, {}).get("center_old", {}).get("xfer_S"); c1n = S["08"].get(w1, {}).get("center_new", {}).get("xfer_S")
        cco = S["08"].get(cell, {}).get("center_old", {}).get("xfer_S"); ccn = S["08"].get(cell, {}).get("center_new", {}).get("xfer_S")
        L.append(crit_line(f"08 N2 {rep}", beats(c1o, 0, cco, 0), beats(c1n, 0, ccn, 0)))
    pw1, sw1 = f"pca_{R8.variant_name(1.0)}", f"scgpt_{R8.variant_name(1.0)}"
    po, spo = ms([e["xfer_S"] for e in S["08"].get(pw1, {}).get("usher_old", [])]); pn, spn = ms([e["xfer_S"] for e in S["08"].get(pw1, {}).get("usher_new", [])])
    so_, sso = ms([e["xfer_S"] for e in S["08"].get(sw1, {}).get("usher_old", [])]); sn_, ssn = ms([e["xfer_S"] for e in S["08"].get(sw1, {}).get("usher_new", [])])
    r3o, r3n = beats(po, spo, so_, sso), beats(pn, spn, sn_, ssn)
    for r in (r3o, r3n):
        if r: r["pass"] = r["pass"] or abs(r["delta"]) < 0.02
    L.append(crit_line("08 N3 (pca vs scgpt, +niche w1)", r3o, r3n))
    pca_cell_new = S["08"].get("pca_cell", {}).get("center_new", {}).get("xfer_S")
    if pca_cell_new is not None:
        ok = abs(pca_cell_new - 0.76) <= 0.02
        L.append(f"- Tái lập 03d: center(pca cell, mới) = {pca_cell_new} (kỳ vọng 0.76±0.02) -> {'khớp' if ok else 'LỆCH'}")
        if not ok: W.append(f"center(pca cell, mới) = {pca_cell_new} lệch khỏi 0.76±0.02")

    if repro: W = [f"tái lập giao thức cũ lệch > 0.002: {r}" for r in repro] + W
    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in W] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run10.py
"""10 — Bảng tổng kết CosMx -> Xenium: USHER không nhãn với aux VÀ đánh giá đều theo giao thức cosine mới (RESEARCH_CONTEXT
mục 3, [CHỐT, 09]: mọi cosine trừ trung bình Xenium μx ở cả hai phía). Khác 09: có huấn luyện USHER thật (aux mới, không
phải map cũ đo lại). 5 biểu diễn: scgpt, scgpt+niche w1, niche_data (Nicheformer, không ghép niche không gian), pca,
pca+niche w1 (dựng như 08). Aux: `run07.knn_aux` (xác suất mềm) trên (center − μx, Fx − μx). USHER: `run06.usher`, đầu
vào GỐC (không trừ μx) — chỉ aux và đánh giá dùng giao thức mới. Đánh giá: `run09.eval_both` (chỉ lấy nhánh `new`).
"""
import os, sys, glob, json, gc, time
SMOKE = os.environ.get("SMOKE", "0") == "1"
if SMOKE:   # phải đặt TRƯỚC khi import run05/run05c/run06/run07/run08 — như các vòng trước
    for k, v in {"SKETCH": "300", "N_ITERS": "1", "N_BOOT": "20", "N_TGT": "1500", "N_REF": "1500", "N_MIX": "600"}.items():
        os.environ.setdefault(k, v)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6, run07 as R7, run08 as R8, run09 as R9
from run04 import np, pd, sp, log, load_assay, _unit

OUT = os.environ.get("OUT_DIR_10", "/kaggle/working/out10" + ("_smoke" if SMOKE else ""))
TARGET, SRC = M.TARGET, "cosmx6k"
REPS5 = ["scgpt", "scgpt_niche_w1", "niche_data", "pca", "pca_niche_w1"]
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX = 15
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def run_list():
    """Thứ tự theo seed (plan mục 4): mỗi seed chạy đủ 5 biểu diễn -> 15 lần."""
    return [(rep, seed) for seed in SEEDS for rep in REPS5]

def run_name(rep, seed):
    return f"{rep}_s{seed}"

def eval_new(SRC_, D, Fs, Fx, name, Fmap, S_shared, IMP, ref):
    """Giao thức mới (09): `run09.eval_both`, chỉ lấy nhánh `new`."""
    _, new = R9.eval_both(SRC_, D, Fs, Fx, name, Fmap, S_shared, IMP, ref)
    return new

# Số đã chốt của 09 (research/results/09_summary.md, cột "trừ μx cả hai phía"), dùng để so sánh F2 và kiểm tra tái lập center
USHER_09 = {"scgpt": 0.736, "scgpt_niche_w1": 0.762, "niche_data": 0.588, "pca": 0.802, "pca_niche_w1": 0.805}
CENTER_09 = {"scgpt": 0.6607, "niche_data": 0.6061, "pca": 0.7661, "scgpt_niche_w1": 0.7033, "pca_niche_w1": 0.7903}


def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER U6 sketch={C.SKETCH} fix=True n_iters={C.N_ITERS} hidden_dim=None(linear) "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine sketch_obsm_key=X_none cache=off | "
           f"GIAO THỨC COSINE MỚI (trừ μx Xenium 2 phía) | seeds={SEEDS} reps={REPS5} k_aux={K_AUX} | "
           f"N_TGT={M.N_TGT} N_REF={M.N_REF} N_BOOT={C.N_BOOT}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt"), "niche_data": M.find_dir("emb_niche_data")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({SRC, TARGET}), EMB)
    ds, dx = D[SRC], D[TARGET]
    S_shared = sorted(set(REC[SRC]) & set(REC[TARGET]))
    classes = np.array(sorted(set(dx["lab"])))
    acc = lambda lab_arr, idx: M.bacc_on(ds["lab"][idx], lab_arr[idx], S_shared)

    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("aux", {}); S.setdefault("base", {}); S.setdefault("runs", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    # ---- imp_setup (như 06-09)
    step("shared_lognorm + imp_setup")
    Xs_lite, vars_ = EXPR[SRC]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1]
    _, _, _, ref = C.mix_ref(SRC, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()

    # ---- 5 biểu diễn: scgpt, niche_data (embedding), pca (joint_pca chép ở 08), + niche w1 cho scgpt/pca (như 08)
    step("dựng biểu diễn pca + niche (như 08)")
    Fx_pca, Fs_pca, n_gene_pca = R8.pca_cell(Xx_lite, varx, Xs_lite, vars_)
    _, _, G_pca = R8.gene_cols(varx, vars_)
    niche_f = f"{OUT}/niche_xenium5k.npy"
    if os.path.exists(niche_f) and os.path.exists(f"{OUT}/niche_cosmx6k.npy"):
        niche_x = np.load(niche_f).astype(np.float32); niche_s = np.load(f"{OUT}/niche_cosmx6k.npy").astype(np.float32)
    else:
        _, obs_x, _, _ = load_assay(TARGET); _, obs_s, _, _ = load_assay(SRC)
        if SMOKE:
            obs_x = obs_x.iloc[B.pick(np.arange(len(obs_x)), R6.N_SMOKE)]; obs_s = obs_s.iloc[B.pick(np.arange(len(obs_s)), R6.N_SMOKE)]
        niche_x_cnt, d15_x, miss_x = R8.build_niche(TARGET, obs_x.index.astype(str), G_pca)
        niche_s_cnt, d15_s, miss_s = R8.build_niche(SRC, obs_s.index.astype(str), G_pca)
        niche_x, niche_s = R8.niche_pca(niche_x_cnt, niche_s_cnt)
        np.save(niche_f, niche_x.astype(np.float16)); np.save(f"{OUT}/niche_cosmx6k.npy", niche_s.astype(np.float16))
        S["niche_d15_um"] = {TARGET: round(d15_x, 2), SRC: round(d15_s, 2)}; save()

    REP = {}
    if F.get((SRC, "scgpt")) is not None: REP["scgpt"] = (F[(SRC, "scgpt")], F[(TARGET, "scgpt")])
    if F.get((SRC, "niche_data")) is not None: REP["niche_data"] = (F[(SRC, "niche_data")], F[(TARGET, "niche_data")])
    REP["pca"] = (Fs_pca, Fx_pca)
    for rep in ("scgpt", "pca"):
        if rep not in REP: continue
        Fs_r, Fx_r = REP[rep]
        s = float(np.mean(np.linalg.norm(Fx_r, axis=1))) / float(np.mean(np.linalg.norm(niche_x, axis=1)))
        REP[f"{rep}_niche_w1"] = (np.hstack([Fs_r, s * niche_s]), np.hstack([Fx_r, s * niche_x]))
        S.setdefault("scale_s", {})[rep] = round(s, 4)
    save()

    # ---- aux (xác suất mềm, giao thức mới) + baseline mỗi biểu diễn
    AUX = {}
    for rep in REPS5:
        if rep not in REP: log(f"CẢNH BÁO: thiếu biểu diễn {rep} (embedding không có) -> bỏ qua"); continue
        Fs, Fx = REP[rep]; mu = Fx.mean(0)
        center_map = M.base_center(Fs, Fx, ds["T"])
        P_aux = R7.knn_aux(center_map - mu, Fx - mu, dx["lab"], classes, k=K_AUX)   # xác suất mềm, giữ nguyên cho USHER
        lab_aux = classes[P_aux.argmax(1)]
        AUX[rep] = (P_aux, lab_aux)
        S["aux"][rep] = {"acc_T": acc(lab_aux, ds["T"]), "acc_H": acc(lab_aux, ds["H"]),
                         "err_conc": R7.err_conc(ds["lab"][ds["T"]], lab_aux[ds["T"]], S_shared)}
        b = S["base"].setdefault(rep, {})
        if "raw" not in b:
            b["raw"] = eval_new(SRC, D, Fs, Fx, "raw", Fs, S_shared, IMP, ref)
            b["center"] = eval_new(SRC, D, Fs, Fx, "center", center_map, S_shared, IMP, ref)
            b["aux_direct"] = C.cross_assay_ci(SRC, D, {}, S_shared, extra_pred=lab_aux)["lr_direct"]
            save()
        log(f"  [{rep}] aux: {S['aux'][rep]} | center xfer_S={b['center']['xfer_S']}")

    # ---- USHER
    def run_and_eval(rep, seed):
        name = run_name(rep, seed); f = f"{OUT}/{name}.npy"
        Fs, Fx = REP[rep]; P_aux, lab_aux = AUX[rep]
        if RESUME and os.path.exists(f) and "eval" in S["runs"].get(name, {}):
            step(f"{name}: đã có, bỏ qua"); return
        if RESUME and os.path.exists(f) and "info" in S["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), S["runs"][name]["info"]; step(f"{name}: nạp lại map")
        else:
            tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            step(f"USHER {name} | target {len(tgt)} tế bào")
            Fmap, info = R6.usher(Fs, ds["T"], P_aux, Fx, tgt, dx["lab"], classes, "linear", seed, False, False)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            S["runs"][name] = {"rep": rep, "seed": seed, "info": info}; save()
        ev = eval_new(SRC, D, Fs, Fx, name, Fmap, S_shared, IMP, ref)
        S["runs"][name]["eval"] = ev; save()
        log(f"  >> {name}: EM={S['runs'][name]['info'].get('em_seconds')}s xfer_S={ev['xfer_S']} {ev['xfer_S_ci']} "
            f"nội_suy_sp={ev['imp']['spearman_med']}")
        del Fmap; gc.collect()

    for rep, seed in runs:
        if rep not in REP: continue
        run_and_eval(rep, seed)

    md = summary_md(S)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG")


# =============================================================================== summary.md
def _ms(v):
    v = [x for x in v if x is not None]
    if not v: return None, None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)

def beats(ma, sa, mb, sb):
    if ma is None or mb is None: return None
    thr = round(max(0.02, 2 * max(sa or 0, sb or 0)), 4)
    d = round(ma - mb, 4)
    return {"delta": d, "thr": thr, "pass": d >= thr}

def summary_md(S):
    L, W = ["# 10 — tóm tắt (giao thức cosine MỚI: trừ μx Xenium hai phía, cho cả aux lẫn đánh giá)", S["config"], ""], []
    rows, usher_stat = [], {}
    for rep in REPS5:
        if rep not in S["aux"]: W.append(f"thiếu biểu diễn {rep}"); continue
        a, b = S["aux"][rep], S["base"][rep]
        rl = [v["eval"] for v in S["runs"].values() if v.get("rep") == rep and "eval" in v]
        mo, so = _ms([e["xfer_S"] for e in rl]); usher_stat[rep] = (mo, so)
        rows.append({"rep": rep, "acc_H": a["acc_H"], "err_conc": a["err_conc"], "center": b["center"]["xfer_S"],
                    "aux_direct": b["aux_direct"]["xfer_S"], "usher": f"{mo:.3f}±{so:.3f}" if mo is not None else "—",
                    "usher_09": USHER_09.get(rep), "Δ_center": round(mo - b["center"]["xfer_S"], 3) if mo is not None else None,
                    "xfer_thô": R6._ms([e["xfer_coarse"] for e in rl]) if rl else "—",
                    "keep_cos": R6._ms([e["keep_struct_cos"] for e in rl]) if rl else "—",
                    "sp_usher": R6._ms([e["imp"]["spearman_med"] for e in rl]) if rl else "—",
                    "sp_center": b["center"]["imp"]["spearman_med"]})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## F1–F4")
    for rep in REPS5:
        if rep not in usher_stat: continue
        mo, so = usher_stat[rep]; cen = S["base"][rep]["center"]["xfer_S"]
        r = beats(mo, so, cen, 0)
        tag = " (= F4, Nicheformer)" if rep == "niche_data" else ""
        L.append(f"- F1 {rep}{tag}: usher − center Δ={r['delta']} (ngưỡng {r['thr']}) {'ĐẠT' if r['pass'] else 'không'}" if r else f"- F1 {rep}: thiếu")
        exp = USHER_09.get(rep)
        if mo is not None and exp is not None:
            d2 = round(mo - exp, 4)
            L.append(f"- F2 {rep}: usher(mới) − usher_09(aux cũ) = {d2}" + (" [≥0.02]" if abs(d2) >= 0.02 else ""))
    if "pca_niche_w1" in usher_stat and "scgpt_niche_w1" in usher_stat:
        mp, sp_ = usher_stat["pca_niche_w1"]; ms_, ss = usher_stat["scgpt_niche_w1"]
        r3 = beats(mp, sp_, ms_, ss)
        if r3:
            r3["pass"] = r3["pass"] or abs(r3["delta"]) < 0.02
            L.append(f"- F3 (pca+niche w1 vs scgpt+niche w1): Δ={r3['delta']} (ngưỡng {r3['thr']}) {'ĐẠT' if r3['pass'] else 'không'}")
    for rep, exp in CENTER_09.items():
        v = S.get("base", {}).get(rep, {}).get("center", {}).get("xfer_S")
        if v is not None and abs(v - exp) > 0.002: W.append(f"center({rep}) mới = {v} lệch khỏi số 09 ({exp}) > 0.002")

    ems = [v["info"].get("em_seconds") for v in S["runs"].values() if v.get("info", {}).get("em_seconds") is not None]
    L.append(f"Thời gian EM: {R6._ms(ems)} giây (n={len(ems)} lần chạy)")
    n_done = sum(1 for v in S["runs"].values() if "eval" in v)
    n_exp = len(run_list())
    if n_done < n_exp: W.append(f"mới xong {n_done}/{n_exp} lần chạy USHER; chia phiên bằng ONLY_RUNS")
    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in W] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run11.py
"""11 — Gói chỉ số chuẩn của bài báo USHER (RESEARCH_CONTEXT U8): trộn lô (iLISI, kBET, BRAS) + bảo toàn sinh học
(NMI, ARI, cLISI, ASW), dùng hàm có sẵn của `scib-metrics` (không tự viết lại). Chấm trên 5 biểu diễn của vòng 10
(scgpt, scgpt+niche w1, niche_data, pca, pca+niche w1) × {raw, center, USHER 3 seed}. Chỉ CPU, không chạy USHER —
chỉ nạp lại map `.npy` của `out10/` (và `summary.json` nếu có, để khỏi tính lại xfer_S/nội suy).
"""
import os, sys, glob, json, gc, warnings, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6, run07 as R7, run08 as R8, run09 as R9, run10 as R10
from run04 import np, pd, sp, log, load_assay, _unit
from scipy.stats import spearmanr

OUT = os.environ.get("OUT_DIR_11", "/kaggle/working/out11")
TARGET, SRC = M.TARGET, "cosmx6k"
REPS5 = R10.REPS5
SEEDS = R10.SEEDS
K_KNN, N_MAX_CLASS, EVAL_SEED = 90, 1000, 0
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def find_input(sub, name):
    h = sorted(glob.glob(f"/kaggle/input/**/{sub}/{name}", recursive=True))
    return h[0] if h else None


# =============================================================================== tập đánh giá (khớp thành phần, cố định)
def build_eval_idx(ds, dx, S_shared, n_max=N_MAX_CLASS, seed=EVAL_SEED):
    """Với mỗi nhóm trong S chung: min(n_max, n_H_cosmx, n_xenium) tế bào TỪ MỖI ASSAY (cùng tập cho mọi map)."""
    rng = np.random.RandomState(seed)
    idx_s, idx_t = [], []
    for c in S_shared:
        pool_s = ds["H"][ds["lab"][ds["H"]] == c]
        pool_t = np.where(dx["lab"] == c)[0]
        k = min(n_max, len(pool_s), len(pool_t))
        if k == 0: continue
        idx_s.append(rng.choice(pool_s, k, replace=False)); idx_t.append(rng.choice(pool_t, k, replace=False))
    if not idx_s or not idx_t:
        raise ValueError(f"build_eval_idx: không nhóm nào trong {list(S_shared)[:8]} có mặt ở cả hai phía "
                         f"(nhãn source: {sorted(set(np.asarray(ds['lab'])))[:8]}...) -> sai tập nhóm/nhãn")
    return np.concatenate(idx_s), np.concatenate(idx_t)


# =============================================================================== metrics_pack (dùng chung, các vòng sau import)
def metrics_pack(F_src, F_tgt, lab_src, lab_tgt, idx_src, idx_tgt, k=K_KNN, seed=EVAL_SEED):
    """Trộn lô + bảo toàn sinh học kiểu scIB, trên (F_src[idx_src] − μx, F_tgt[idx_tgt] − μx) đã chuẩn hoá L2 (~cosine),
    μx = F_tgt.mean(0) (giao thức mới, RESEARCH_CONTEXT mục 3). Dùng nguyên hàm scib_metrics; hàm nào thiếu/lỗi thì bỏ
    và ghi cảnh báo trong out["_warnings"], không tự viết lại chỉ số."""
    import scib_metrics as sm
    from scib_metrics.nearest_neighbors import pynndescent
    mu = F_tgt.mean(0)
    X = _unit(np.vstack([F_src[idx_src] - mu, F_tgt[idx_tgt] - mu])).astype(np.float32)
    labels = np.concatenate([np.asarray(lab_src)[idx_src], np.asarray(lab_tgt)[idx_tgt]])
    batch = np.array(["src"] * len(idx_src) + ["tgt"] * len(idx_tgt))   # nhãn lô chung chung -> dùng lại được cho cặp khác
    out, warns = {}, []

    def safe(name, fn):
        if not hasattr(sm, name):
            warns.append(f"scib_metrics thiếu hàm {name} (phiên bản cũ?)"); return None
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return fn()
        except Exception as e:
            warns.append(f"{name} lỗi: {e!r}"); return None

    nn = pynndescent(X, n_neighbors=min(k, len(X) - 1), random_state=seed)
    out["ilisi"] = safe("ilisi_knn", lambda: round(float(sm.ilisi_knn(nn, batch)), 4))
    out["clisi"] = safe("clisi_knn", lambda: round(float(sm.clisi_knn(nn, labels)), 4))
    out["kbet"] = safe("kbet_per_label", lambda: round(float(sm.kbet_per_label(nn, batch, labels)), 4))
    na = safe("nmi_ari_cluster_labels_leiden", lambda: sm.nmi_ari_cluster_labels_leiden(nn, labels, n_jobs=1))
    out["nmi"], out["ari"] = (round(float(na["nmi"]), 4), round(float(na["ari"]), 4)) if na else (None, None)
    out["asw"] = safe("silhouette_label", lambda: round(float(sm.silhouette_label(X, labels)), 4))
    out["bras"] = safe("bras", lambda: round(float(sm.bras(X, labels, batch)), 4))
    # bổ sung (2026-09-30): các chỉ số bài báo USHER có nêu nhưng trước đây thiếu (graph connectivity, batch ASW),
    # và isolated labels (mặc định của scib-metrics Benchmarker)
    out["graph_conn"] = safe("graph_connectivity", lambda: round(float(sm.graph_connectivity(nn, labels)), 4))
    out["batch_asw"] = safe("silhouette_batch", lambda: round(float(sm.silhouette_batch(X, labels, batch)), 4))
    out["iso_labels"] = safe("isolated_labels", lambda: round(float(sm.isolated_labels(X, labels, batch)), 4))
    bm = [out[k_] for k_ in ("ilisi", "kbet", "bras") if out[k_] is not None]
    om = [out[k_] for k_ in ("nmi", "ari", "clisi", "asw") if out[k_] is not None]
    out["batch"] = round(float(np.mean(bm)), 4) if bm else None
    out["bio"] = round(float(np.mean(om)), 4) if om else None
    out["total"] = round(0.4 * out["batch"] + 0.6 * out["bio"], 4) if out["batch"] is not None and out["bio"] is not None else None
    # tổng hợp kiểu scib-metrics Benchmarker (không min-max scale): batch = TB(BRAS, iLISI, kBET, graph conn);
    # bio = TB(isolated labels, NMI, ARI, ASW nhãn, cLISI); tổng = 0.4 batch + 0.6 bio. Giữ "total" cũ để so với vòng trước.
    b2 = [out[k_] for k_ in ("bras", "ilisi", "kbet", "graph_conn") if out.get(k_) is not None]
    o2 = [out[k_] for k_ in ("iso_labels", "nmi", "ari", "asw", "clisi") if out.get(k_) is not None]
    out["batch_v2"] = round(float(np.mean(b2)), 4) if b2 else None
    out["bio_v2"] = round(float(np.mean(o2)), 4) if o2 else None
    out["total_v2"] = round(0.4 * out["batch_v2"] + 0.6 * out["bio_v2"], 4) if out["batch_v2"] is not None and out["bio_v2"] is not None else None
    out["n_eval"] = {"src": int(len(idx_src)), "tgt": int(len(idx_tgt))}
    out["_warnings"] = warns
    return out


def _ms(vals):
    v = [x for x in vals if x is not None]
    if not v: return None, None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)

def beats(mo, so, mc, sc, thr_min=0.02):
    if mo is None or mc is None: return None
    thr = max(thr_min, 2 * max(so or 0, sc or 0))
    d = mo - mc
    return "vượt" if d >= thr else ("kém" if -d >= thr else "ngang")


def main():
    os.makedirs(OUT, exist_ok=True)
    import scib_metrics
    W = []
    cfg = (f"CẤU HÌNH: scib_metrics={scib_metrics.__version__} | k_knn={K_KNN} | tập đánh giá: min({N_MAX_CLASS}, "
           f"n_H_cosmx, n_xenium) mỗi nhóm chung, seed={EVAL_SEED} | giao thức cosine mới (trừ μx Xenium)")
    log(cfg)

    EMB = {"scgpt": M.find_dir("emb_scgpt"), "niche_data": M.find_dir("emb_niche_data")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({SRC, TARGET}), EMB)
    ds, dx = D[SRC], D[TARGET]
    S_shared = sorted(set(REC[SRC]) & set(REC[TARGET]))
    idx_src, idx_tgt = build_eval_idx(ds, dx, S_shared)
    cfg += f" | n_eval src={len(idx_src)} tgt={len(idx_tgt)}"
    log(f"S chung = {S_shared} | tập đánh giá: {len(idx_src)} CosMx (H) + {len(idx_tgt)} Xenium")

    # ---- IMP (chỉ dùng nếu phải tính lại xfer_S/nội suy, không đọc được out10/summary.json)
    Xs_lite, vars_ = EXPR[SRC]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1]
    _, _, _, ref = C.mix_ref(SRC, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()

    # ---- dựng lại 5 biểu diễn (đúng code 08/10): scgpt, niche_data (embedding), pca, + niche w1 cho scgpt/pca
    step("dựng biểu diễn pca + niche (đọc lại cache của 10 nếu có)")
    Fx_pca, Fs_pca, _ = R8.pca_cell(Xx_lite, varx, Xs_lite, vars_)
    _, _, G_pca = R8.gene_cols(varx, vars_)
    niche_x_f, niche_s_f = find_input("out10", "niche_xenium5k.npy"), find_input("out10", "niche_cosmx6k.npy")
    if niche_x_f and niche_s_f:
        niche_x, niche_s = np.load(niche_x_f).astype(np.float32), np.load(niche_s_f).astype(np.float32)
    else:
        W.append("không thấy out10/niche_*.npy -> tự dựng lại từ dữ liệu đầy đủ")
        _, obs_x, _, _ = load_assay(TARGET); _, obs_s, _, _ = load_assay(SRC)
        nx_cnt, _, _ = R8.build_niche(TARGET, obs_x.index.astype(str), G_pca)
        ns_cnt, _, _ = R8.build_niche(SRC, obs_s.index.astype(str), G_pca)
        niche_x, niche_s = R8.niche_pca(nx_cnt, ns_cnt)

    REP = {}
    if F.get((SRC, "scgpt")) is not None: REP["scgpt"] = (F[(SRC, "scgpt")], F[(TARGET, "scgpt")])
    if F.get((SRC, "niche_data")) is not None: REP["niche_data"] = (F[(SRC, "niche_data")], F[(TARGET, "niche_data")])
    REP["pca"] = (Fs_pca, Fx_pca)
    for rep in ("scgpt", "pca"):
        if rep not in REP: continue
        Fs_r, Fx_r = REP[rep]
        s = float(np.mean(np.linalg.norm(Fx_r, axis=1))) / float(np.mean(np.linalg.norm(niche_x, axis=1)))
        REP[f"{rep}_niche_w1"] = (np.hstack([Fs_r, s * niche_s]), np.hstack([Fx_r, s * niche_x]))

    # ---- đọc out10/summary.json nếu có (đỡ phải tính lại xfer_S/nội suy)
    j10 = find_input("out10", "summary.json")
    S10 = json.load(open(j10)) if j10 else None
    if not j10: W.append("không thấy out10/summary.json -> tự tính lại xfer_S/nội suy (chậm hơn)")

    def eval_map(rep, name, Fmap, fallback_key=None, seed=None):
        """(ev, warn) — ev = {xfer_S, xfer_coarse, assay_sep, imp}. Ưu tiên đọc out10/summary.json; thiếu thì tính lại."""
        Fs, Fx = REP[rep]
        if S10 is not None:
            src = S10.get("runs", {}).get(name, {}).get("eval") if seed is not None else S10.get("base", {}).get(rep, {}).get(fallback_key)
            if src: return src, None
        return R10.eval_new(SRC, D, Fs, Fx, name, Fmap, S_shared, IMP, ref), f"{name}: tính lại xfer_S/nội suy (thiếu trong out10)"

    # ---- chấm từng biểu diễn: raw, center, USHER (3 seed)
    S = {"config": cfg, "reps": {}}
    for rep in REPS5:
        if rep not in REP: W.append(f"thiếu biểu diễn {rep}"); continue
        Fs, Fx = REP[rep]
        step(f"{rep}: raw, center")
        center_map = M.base_center(Fs, Fx, ds["T"])
        row = {}
        for variant, Fmap in [("raw", Fs), ("center", center_map)]:
            ev, w = eval_map(rep, variant, Fmap, fallback_key=variant)
            if w: W.append(w)
            mp = metrics_pack(Fmap, Fx, ds["lab"], dx["lab"], idx_src, idx_tgt)
            W += [f"{rep}/{variant}: {x}" for x in mp.pop("_warnings")]
            row[variant] = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "assay_sep": ev["assay_sep"],
                            "sp": ev["imp"]["spearman_med"]}
        usher_maps = []
        for seed in SEEDS:
            name = R10.run_name(rep, seed)
            fp = find_input("out10", f"{name}.npy")
            if not fp: W.append(f"{rep}: thiếu USHER seed {seed} (out10/{name}.npy)"); continue
            step(f"{name}: metrics_pack")
            Fmap = np.load(fp).astype(np.float32)
            ev, w = eval_map(rep, name, Fmap, seed=seed)
            if w: W.append(w)
            mp = metrics_pack(Fmap, Fx, ds["lab"], dx["lab"], idx_src, idx_tgt)
            W += [f"{name}: {x}" for x in mp.pop("_warnings")]
            usher_maps.append({**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "assay_sep": ev["assay_sep"],
                               "sp": ev["imp"]["spearman_med"]})
            del Fmap; gc.collect()
        row["usher"] = usher_maps
        S["reps"][rep] = row
        json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    md = summary_md(S, REPS5, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG")


# =============================================================================== summary.md
METRIC_COLS = ["ilisi", "kbet", "bras", "assay_sep", "nmi", "ari", "clisi", "asw", "batch", "bio", "total", "xfer_S", "sp"]

def _agg(maps, key):
    return _ms([m[key] for m in maps])

def summary_md(S, REPS5, W):
    L = ["# 11 — tóm tắt (chỉ số scib-metrics, giao thức cosine mới)", S["config"], ""]
    rows = []   # bảng gọn: raw + center (mỗi cột một chỉ số) + usher gộp TB±SD, 3 dòng/biểu diễn
    for rep, row in S["reps"].items():
        for variant in ("raw", "center"):
            v = row[variant]; rows.append({"rep": rep, "variant": variant, **{c: v.get(c) for c in METRIC_COLS}})
        u = {"rep": rep, "variant": f"usher(n={len(row['usher'])})"}
        for c in METRIC_COLS:
            m, sd = _agg(row["usher"], c); u[c] = f"{m:.3f}±{sd:.3f}" if m is not None else "—"
        rows.append(u)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## M1 (USHER so với center, mỗi chỉ số: vượt/kém/ngang)")
    m1rows = []
    for rep, row in S["reps"].items():
        r = {"rep": rep}
        for c in ["ilisi", "kbet", "bras", "nmi", "ari", "clisi", "asw"]:
            mo, so = _agg(row["usher"], c); mc = row["center"].get(c)
            r[c] = beats(mo, so, mc, 0) or "—"
        m1rows.append(r)
    if m1rows: L += ["```", pd.DataFrame(m1rows).to_string(index=False), "```"]

    L.append("## M2 (Spearman xếp hạng 10 map: xfer_S so với NMI/ASW/tổng)")
    xs, nmis, asws, tots = [], [], [], []
    for rep, row in S["reps"].items():
        for variant in ("center",):
            v = row[variant]; xs.append(v["xfer_S"]); nmis.append(v["nmi"]); asws.append(v["asw"]); tots.append(v["total"])
        mo, _ = _agg(row["usher"], "xfer_S"); mn, _ = _agg(row["usher"], "nmi"); ma, _ = _agg(row["usher"], "asw"); mt, _ = _agg(row["usher"], "total")
        xs.append(mo); nmis.append(mn); asws.append(ma); tots.append(mt)
    def rho(a, b):
        pairs = [(x, y) for x, y in zip(a, b) if x is not None and y is not None]
        return round(float(spearmanr(*zip(*pairs))[0]), 3) if len(pairs) > 2 else None
    L.append(f"ρ(xfer_S, NMI) = {rho(xs, nmis)} | ρ(xfer_S, ASW) = {rho(xs, asws)} | ρ(xfer_S, tổng) = {rho(xs, tots)}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run12.py
"""12 — PCA so với scGPT-spatial, và "niche chỉ cho aux", trên ba cặp giải trình tự -> Xenium (visiumhd_ffpe,
visiumhd_ff, stereoseq), đủ chỉ số (xfer + scib-metrics). Mỗi cặp: 2 biểu diễn (`pca`, `scgpt`) × 2 cách lấy M_aux
(`aux_cell`: kNN trên center(rep); `aux_niche`: kNN trên center([rep, niche]), NHƯNG đặc trưng USHER vẫn chỉ là `rep`)
× 3 seed = 12 lần USHER/cặp. Biến môi trường PAIRS chọn cặp chạy mỗi phiên (mặc định cả ba).
"""
import os, sys, glob, json, gc, time
SMOKE = os.environ.get("SMOKE", "0") == "1"
if SMOKE:   # phải đặt TRƯỚC khi import run05/run05c/run06/run07/run08 — như các vòng trước
    for k, v in {"SKETCH": "300", "N_ITERS": "1", "N_BOOT": "20", "N_TGT": "1500", "N_REF": "1500", "N_MIX": "600"}.items():
        os.environ.setdefault(k, v)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6, run07 as R7, run08 as R8, run09 as R9, run10 as R10, run11 as R11
from run04 import np, pd, sp, log, load_assay, _unit

OUT = os.environ.get("OUT_DIR_12", "/kaggle/working/out12" + ("_smoke" if SMOKE else ""))
TARGET = M.TARGET
PAIRS_ALL = ["visiumhd_ffpe", "visiumhd_ff", "stereoseq"]
PAIRS = [p.strip() for p in os.environ.get("PAIRS", ",".join(PAIRS_ALL)).split(",") if p.strip()]
REPS = ["pca", "scgpt"]
AUXES = ["aux_cell", "aux_niche"]
ORDER4 = [(rep, aux) for rep in REPS for aux in AUXES]
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX = 15
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def run_list():
    """Thứ tự theo seed (plan mục 4): mỗi seed chạy đủ 4 cấu hình (pca/scgpt × cell/niche) -> 12 lần/cặp."""
    return [(rep, aux, seed) for seed in SEEDS for (rep, aux) in ORDER4]

def run_name(rep, aux, seed):
    return f"{rep}_{aux}_s{seed}"


def process_pair(pair):
    OUT_p = f"{OUT}/{pair}"; os.makedirs(OUT_p, exist_ok=True)
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH [{pair}]: USHER U6 sketch={C.SKETCH} fix=True n_iters={C.N_ITERS} hidden_dim=None(linear) "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine sketch_obsm_key=X_none cache=off | "
           f"giao thức cosine mới | seeds={SEEDS} reps={REPS} aux={AUXES} k_aux={K_AUX} scib_k={R11.K_KNN} | "
           f"N_TGT={M.N_TGT} N_REF={M.N_REF} N_BOOT={C.N_BOOT}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  [{pair}] run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({pair, TARGET}), EMB)
    ds, dx = D[pair], D[TARGET]
    S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
    cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
    coarse_groups = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]   # như cross_assay_ci
    use_coarse = len(S_shared) < 3     # Stereo: 2 nhóm chung -> dùng nhãn/thước đo thô (plan mục 3)
    lab_src_ev = ds["lab_c"] if use_coarse else ds["lab"]; lab_tgt_ev = dx["lab_c"] if use_coarse else dx["lab"]
    groups_ev = coarse_groups if use_coarse else S_shared
    cfg += f" | S_chung={S_shared} (thô={coarse_groups})" + (" [dùng nhãn/thước đo THÔ cho cặp này]" if use_coarse else "")
    log(f"[{pair}] {cfg}")
    idx_src, idx_tgt = R11.build_eval_idx({**ds, "lab": lab_src_ev}, {**dx, "lab": lab_tgt_ev}, groups_ev)

    S = json.load(open(f"{OUT_p}/summary.json")) if RESUME and os.path.exists(f"{OUT_p}/summary.json") else {}
    S["config"] = cfg; S.setdefault("aux", {}); S.setdefault("base", {}); S.setdefault("runs", {})
    def save(): json.dump(S, open(f"{OUT_p}/summary.json", "w"), indent=1, default=str)

    Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1]
    _, _, _, ref = C.mix_ref(pair, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()

    step(f"[{pair}] pca + niche (đúng code 08)")
    Fx_pca, Fs_pca, _ = R8.pca_cell(Xx_lite, varx, Xs_lite, vars_)
    _, _, G_pca = R8.gene_cols(varx, vars_)
    nx_f, ns_f = f"{OUT_p}/niche_xenium5k.npy", f"{OUT_p}/niche_{pair}.npy"
    if os.path.exists(nx_f) and os.path.exists(ns_f):
        niche_x, niche_s = np.load(nx_f).astype(np.float32), np.load(ns_f).astype(np.float32)
    else:
        _, obs_x, _, _ = load_assay(TARGET); _, obs_s, _, _ = load_assay(pair)
        if SMOKE:
            obs_x = obs_x.iloc[B.pick(np.arange(len(obs_x)), R6.N_SMOKE)]; obs_s = obs_s.iloc[B.pick(np.arange(len(obs_s)), R6.N_SMOKE)]
        nx_cnt, d15_x, miss_x = R8.build_niche(TARGET, obs_x.index.astype(str), G_pca)
        ns_cnt, d15_s, miss_s = R8.build_niche(pair, obs_s.index.astype(str), G_pca)
        niche_x, niche_s = R8.niche_pca(nx_cnt, ns_cnt)
        np.save(nx_f, niche_x.astype(np.float16)); np.save(ns_f, niche_s.astype(np.float16))
        S["niche_d15_um"] = {TARGET: round(d15_x, 2), pair: round(d15_s, 2)}; save()
    log(f"[{pair}] niche: {S.get('niche_d15_um', 'đã có cache')}")

    classes = np.array(sorted(set(dx["lab"])))
    Fs = {"scgpt": F.get((pair, "scgpt")), "pca": Fs_pca}
    Fx = {"scgpt": F.get((TARGET, "scgpt")), "pca": Fx_pca}

    def bacc(y_true, y_pred, keep): return M.bacc_on(y_true, y_pred, keep) if len(keep) else None

    # ---- aux_cell + aux_niche mỗi rep (xác suất mềm, giao thức mới); đặc trưng USHER vẫn là rep (KHÔNG ghép niche)
    AUX, NICHE_SCALE = {}, {}
    for rep in REPS:
        if Fs[rep] is None: W.append(f"thiếu biểu diễn {rep}"); continue
        Fsr, Fxr = Fs[rep], Fx[rep]
        center_map = M.base_center(Fsr, Fxr, ds["T"]); mu = Fxr.mean(0)
        P_cell = R7.knn_aux(center_map - mu, Fxr - mu, dx["lab"], classes, k=K_AUX)
        lab_cell = classes[P_cell.argmax(1)]
        s = float(np.mean(np.linalg.norm(Fxr, axis=1))) / float(np.mean(np.linalg.norm(niche_x, axis=1))); NICHE_SCALE[rep] = s
        Fs_c, Fx_c = np.hstack([Fsr, s * niche_s]), np.hstack([Fxr, s * niche_x])
        center_c = M.base_center(Fs_c, Fx_c, ds["T"]); mu_c = Fx_c.mean(0)
        P_niche = R7.knn_aux(center_c - mu_c, Fx_c - mu_c, dx["lab"], classes, k=K_AUX)
        lab_niche = classes[P_niche.argmax(1)]
        AUX[rep] = {"aux_cell": (P_cell, lab_cell), "aux_niche": (P_niche, lab_niche)}
        for aux, lab_a in [("aux_cell", lab_cell), ("aux_niche", lab_niche)]:
            S["aux"][f"{rep}_{aux}"] = {
                "acc_T": bacc(ds["lab"][ds["T"]], lab_a[ds["T"]], S_shared), "acc_H": bacc(ds["lab"][ds["H"]], lab_a[ds["H"]], S_shared),
                "acc_T_thô": bacc(ds["lab_c"][ds["T"]], B.to_coarse(lab_a)[ds["T"]], coarse_groups),
                "acc_H_thô": bacc(ds["lab_c"][ds["H"]], B.to_coarse(lab_a)[ds["H"]], coarse_groups),
                "err_conc": R7.err_conc(ds["lab"][ds["T"]], lab_a[ds["T"]], S_shared)}
        save()
        log(f"[{pair}/{rep}] aux_cell: {S['aux'][f'{rep}_aux_cell']} | aux_niche: {S['aux'][f'{rep}_aux_niche']}")

    def eval_and_metrics(name, Fsr, Fxr, Fmap):
        ev = R10.eval_new(pair, D, Fsr, Fxr, name, Fmap, S_shared, IMP, ref)
        mp = R11.metrics_pack(Fmap, Fxr, lab_src_ev, lab_tgt_ev, idx_src, idx_tgt)
        W_ = mp.pop("_warnings")
        return ev, mp, [f"{name}: {x}" for x in W_]

    # ---- baseline mỗi rep: raw, center, aux_direct (cell, niche)
    for rep in REPS:
        if rep not in AUX: continue
        b = S["base"].setdefault(rep, {})
        if "raw" not in b:
            Fsr, Fxr = Fs[rep], Fx[rep]; center_map = M.base_center(Fsr, Fxr, ds["T"])
            for variant, Fmap in [("raw", Fsr), ("center", center_map)]:
                ev, mp, w = eval_and_metrics(variant, Fsr, Fxr, Fmap); W += w
                b[variant] = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "assay_sep": ev["assay_sep"],
                             "sp": ev["imp"]["spearman_med"]}
            for aux in AUXES:
                b[f"aux_direct_{aux}"] = C.cross_assay_ci(pair, D, {}, S_shared, extra_pred=AUX[rep][aux][1])["lr_direct"]
            save()

    # ---- USHER (12 lần/cặp)
    def run_and_eval(rep, aux, seed):
        name = run_name(rep, aux, seed); f = f"{OUT_p}/{name}.npy"
        Fsr, Fxr = Fs[rep], Fx[rep]; P_aux, lab_aux = AUX[rep][aux]
        if RESUME and os.path.exists(f) and "eval" in S["runs"].get(name, {}):
            step(f"[{pair}] {name}: đã có, bỏ qua"); return
        if RESUME and os.path.exists(f) and "info" in S["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), S["runs"][name]["info"]; step(f"[{pair}] {name}: nạp lại map")
        else:
            tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            step(f"USHER [{pair}] {name} | target {len(tgt)} tế bào")
            Fmap, info = R6.usher(Fsr, ds["T"], P_aux, Fxr, tgt, dx["lab"], classes, "linear", seed, False, False)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            S["runs"][name] = {"rep": rep, "aux": aux, "seed": seed, "info": info}; save()
        ev, mp, w = eval_and_metrics(name, Fsr, Fxr, Fmap); W.extend(w)
        S["runs"][name]["eval"] = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "assay_sep": ev["assay_sep"],
                                   "sp": ev["imp"]["spearman_med"]}
        save()
        log(f"  >> [{pair}] {name}: EM={S['runs'][name]['info'].get('em_seconds')}s xfer_S={ev['xfer_S']} xfer_thô={ev['xfer_coarse']}")
        del Fmap; gc.collect()

    for rep, aux, seed in runs:
        if rep not in AUX: continue
        run_and_eval(rep, aux, seed)

    md = summary_md(pair, S, use_coarse, W)
    open(f"{OUT_p}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + f"\n[{pair}]\n" + md + "\nXONG " + pair)


# =============================================================================== summary.md (mỗi cặp)
def _ms(vals):
    v = [x for x in vals if x is not None]
    if not v: return None, None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)

def beats(mo, so, mc, sc, thr_min=0.02):
    if mo is None or mc is None: return None
    thr = max(thr_min, 2 * max(so or 0, sc or 0)); d = mo - mc
    return {"delta": round(d, 4), "thr": round(thr, 4), "pass": d >= thr}

TABLE_COLS = ["acc_H", "err_conc", "xfer_S", "xfer_coarse", "ilisi", "kbet", "bras", "nmi", "ari", "asw",
              "batch", "bio", "total", "sp", "keep_struct_cos"]

def summary_md(pair, S, use_coarse, W):
    xfer_key = "xfer_coarse" if use_coarse else "xfer_S"
    L = [f"# 12/{pair} — tóm tắt" + (" (dùng thước đo/nhãn THÔ)" if use_coarse else ""), S["config"], ""]
    rows = []
    for rep in REPS:
        b = S["base"].get(rep)
        if not b: continue
        for variant in ("raw", "center"):
            v = b[variant]; a = S["aux"].get(f"{rep}_aux_cell", {})
            rows.append({"rep": rep, "map": variant, "acc_H": a.get("acc_H"), "err_conc": a.get("err_conc"),
                        **{c: v.get(c) for c in TABLE_COLS if c not in ("acc_H", "err_conc")}})
        for aux in AUXES:
            a = S["aux"].get(f"{rep}_{aux}", {})
            rows.append({"rep": rep, "map": f"aux_direct_{aux[4:]}", "acc_H": a.get("acc_H"), "err_conc": a.get("err_conc"),
                        xfer_key: b.get(f"aux_direct_{aux}", {}).get("xfer_S")})
        for aux in AUXES:
            rl = [v["eval"] for v in S["runs"].values() if v.get("rep") == rep and v.get("aux") == aux and "eval" in v]
            r = {"rep": rep, "map": f"usher_{aux}(n={len(rl)})", "acc_H": S["aux"].get(f"{rep}_{aux}", {}).get("acc_H"),
                "err_conc": S["aux"].get(f"{rep}_{aux}", {}).get("err_conc")}
            for c in TABLE_COLS:
                if c in ("acc_H", "err_conc"): continue
                m, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m:.3f}±{sd:.3f}" if m is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## G1–G3")
    def usher_ms(rep, aux, key): return _ms([v["eval"].get(key) for v in S["runs"].values() if v.get("rep") == rep and v.get("aux") == aux and "eval" in v])
    mo1, so1 = usher_ms("scgpt", "aux_cell", xfer_key); mp1, sp1 = usher_ms("pca", "aux_cell", xfer_key)
    to1, sto1 = usher_ms("scgpt", "aux_cell", "total"); tp1, stp1 = usher_ms("pca", "aux_cell", "total")
    r_xfer, r_tot = beats(mp1, sp1, mo1, so1), beats(tp1, stp1, to1, sto1)
    g1 = "PCA lợi thế" if (r_xfer and r_xfer["pass"] and r_tot and r_tot["pass"]) else \
         ("scGPT lợi thế" if (r_xfer and not r_xfer["pass"] and r_tot and not r_tot["pass"]) else "hoà/lẫn lộn")
    L.append(f"- G1: PCA-scGPT Δxfer={r_xfer['delta'] if r_xfer else None} Δtổng={r_tot['delta'] if r_tot else None} -> {g1}")
    for rep in REPS:
        mo, so = usher_ms(rep, "aux_cell", xfer_key); mc = S["base"].get(rep, {}).get("center", {}).get(f"xfer_{xfer_key[5:]}")
        to, sto = usher_ms(rep, "aux_cell", "total"); tc = S["base"].get(rep, {}).get("center", {}).get("total")
        r1, r2 = beats(mo, so, mc, 0), beats(to, sto, tc, 0)
        L.append(f"- G2 {rep}: xfer Δ={r1['delta'] if r1 else None} ({'ĐẠT' if r1 and r1['pass'] else 'không'}) | "
                f"tổng Δ={r2['delta'] if r2 else None} ({'ĐẠT' if r2 and r2['pass'] else 'không'})")
    for rep in REPS:
        moc, soc = usher_ms(rep, "aux_cell", xfer_key); mon, son = usher_ms(rep, "aux_niche", xfer_key)
        nc, snc = usher_ms(rep, "aux_cell", "nmi"); nn, snn = usher_ms(rep, "aux_niche", "nmi")
        ra = beats(mon, son, moc, soc)
        rb = (round(nn - nc, 4) >= -0.02) if (nn is not None and nc is not None) else None
        ok = bool(ra and ra["pass"] and rb)
        L.append(f"- G3 {rep}: niche vs cell xfer Δ={ra['delta'] if ra else None} | ΔNMI={round(nn - nc, 4) if (nn is not None and nc is not None) else None} -> {'niche-cho-aux tốt hơn' if ok else 'không rõ lợi'}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    for pair in PAIRS:
        assert pair in PAIRS_ALL, f"PAIRS không hợp lệ: {pair}"
        process_pair(pair)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run13.py
"""13 — Regularization về identity trong M-step (U3b): vá `src_cuda/m_step_utils.py` (KHÔNG sửa file gốc) để thêm số
hạng `LAMBDA_ID * ||f(x) - x||^2` vào loss, quét λ_id trên hai cặp tốt nhất mỗi loại assay:
  cosmx6k / pca -> xenium5k       (ảnh, PCA tốt nhất — 10)
  visiumhd_ff / scgpt -> xenium5k (giải trình tự thưa, scGPT tốt nhất — 12)
λ_id = 0 phải cho hành vi y hệt cũ (số hạng thêm nhân 0). Aux `aux_cell` (giao thức mới, như 12), U6, hidden_dim=None.
"""
import os, sys, glob, json, gc, time
SMOKE = os.environ.get("SMOKE", "0") == "1"
if SMOKE:   # phải đặt TRƯỚC khi import run05/run05c/run06/run07/run08 — như các vòng trước
    for k, v in {"SKETCH": "300", "N_ITERS": "1", "N_BOOT": "20", "N_TGT": "1500", "N_REF": "1500", "N_MIX": "600"}.items():
        os.environ.setdefault(k, v)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run04 as R, run04b as B, run05 as M, run05c as C, run06 as R6, run07 as R7, run08 as R8, run09 as R9, run11 as R11, run12 as R12
from run04 import np, pd, sp, log, load_assay, _unit

OUT = os.environ.get("OUT_DIR_13", "/kaggle/working/out13" + ("_smoke" if SMOKE else ""))
TARGET = M.TARGET
PAIR_REP = {"cosmx6k": "pca", "visiumhd_ff": "scgpt"}     # cặp cố định + biểu diễn tốt nhất của loại assay đó (10, 12)
PAIRS = [p.strip() for p in os.environ.get("PAIRS", ",".join(PAIR_REP)).split(",") if p.strip()]
LAMBDAS = [0, 0.1, 0.3, 1, 3]
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX = 15
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")

# Số 10/12 để kiểm tra tái lập λ_id=0 (plan mục 4)
EXPECT_LAM0 = {"cosmx6k": {"xfer_S": 0.809, "total": 0.671}, "visiumhd_ff": {"xfer_S": 0.678, "total": 0.702}}


def run_list():
    """Thứ tự theo seed (plan mục 4): mỗi seed chạy đủ 5 λ_id -> 15 lần/cặp."""
    return [(lam, seed) for seed in SEEDS for lam in LAMBDAS]

def run_name(lam, seed):
    return f"lam{lam:g}_s{seed}"


def process_pair(pair):
    rep = PAIR_REP[pair]
    OUT_p = f"{OUT}/{pair}"; os.makedirs(OUT_p, exist_ok=True)
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH [{pair}/{rep}]: USHER U6 sketch={C.SKETCH} fix=True n_iters={C.N_ITERS} hidden_dim=None(linear) "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine sketch_obsm_key=X_none cache=off | "
           f"giao thức cosine mới | aux=aux_cell k_aux={K_AUX} scib_k={R11.K_KNN} | λ_id∈{LAMBDAS} seeds={SEEDS} | "
           f"N_TGT={M.N_TGT} N_REF={M.N_REF} N_BOOT={C.N_BOOT}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  [{pair}] run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({pair, TARGET}), EMB)
    ds, dx = D[pair], D[TARGET]
    S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
    log(f"[{pair}] S chung = {S_shared}")

    S = json.load(open(f"{OUT_p}/summary.json")) if RESUME and os.path.exists(f"{OUT_p}/summary.json") else {}
    S["config"] = cfg; S.setdefault("base", {}); S.setdefault("runs", {})
    def save(): json.dump(S, open(f"{OUT_p}/summary.json", "w"), indent=1, default=str)

    Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1]
    _, _, _, ref = C.mix_ref(pair, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
    idx_src, idx_tgt = R11.build_eval_idx(ds, dx, S_shared)

    if rep == "pca":
        Fx_rep, Fs_rep, _ = R8.pca_cell(Xx_lite, varx, Xs_lite, vars_)
    else:
        Fs_rep, Fx_rep = F[(pair, "scgpt")], F[(TARGET, "scgpt")]

    classes = np.array(sorted(set(dx["lab"])))
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"]); mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], classes, k=K_AUX)   # aux_cell, giao thức mới (như 12)
    lab_aux = classes[P_aux.argmax(1)]
    log(f"[{pair}/{rep}] aux_cell acc_H={M.bacc_on(ds['lab'][ds['H']], lab_aux[ds['H']], S_shared)}")

    def drift(Fmap, Fs):
        return float(np.median(np.linalg.norm(Fmap - Fs, axis=1)) / np.median(np.linalg.norm(Fs, axis=1)))

    def eval_all(name, Fmap):
        ev = R9.eval_both(pair, D, Fs_rep, Fx_rep, name, Fmap, S_shared, IMP, ref)[1]   # nhánh mới
        mp = R11.metrics_pack(Fmap, Fx_rep, ds["lab"], dx["lab"], idx_src, idx_tgt)
        w = [f"{name}: {x}" for x in mp.pop("_warnings")]
        return {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "keep_struct_cos": ev["keep_struct_cos"],
                "sp": ev["imp"]["spearman_med"], "drift": drift(Fmap, Fs_rep)}, w

    # ---- baseline: raw, center
    b = S["base"]
    if "raw" not in b:
        for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
            ev, w = eval_all(variant, Fmap); W += w; b[variant] = ev
        save()

    # ---- USHER, quét λ_id
    for lam, seed in runs:
        name = run_name(lam, seed); f = f"{OUT_p}/{name}.npy"
        if RESUME and os.path.exists(f) and "eval" in S["runs"].get(name, {}):
            step(f"[{pair}] {name}: đã có, bỏ qua"); continue
        if RESUME and os.path.exists(f) and "info" in S["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), S["runs"][name]["info"]; step(f"[{pair}] {name}: nạp lại map")
        else:
            tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            step(f"USHER [{pair}] {name} | target {len(tgt)} tế bào")
            os.environ["USHER_LAMBDA_ID"] = str(lam)   # vá m_step_utils.py: đọc biến này ngay trước loss (U3b)
            Fmap, info = R6.usher(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], classes, "linear", seed, False, False)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            S["runs"][name] = {"lam": lam, "seed": seed, "info": info}; save()
        ev, w = eval_all(name, Fmap); W += w
        S["runs"][name]["eval"] = ev; save()
        log(f"  >> [{pair}] {name}: EM={S['runs'][name]['info'].get('em_seconds')}s xfer_S={ev['xfer_S']} sp={ev['sp']} drift={ev['drift']}")
        del Fmap; gc.collect()

    exp = EXPECT_LAM0.get(pair)
    if exp:
        rl0 = [v["eval"] for v in S["runs"].values() if v.get("lam") == 0 and "eval" in v]
        if rl0:
            mx = float(np.mean([e["xfer_S"] for e in rl0])); mt = float(np.mean([e["total"] for e in rl0]))
            if abs(mx - exp["xfer_S"]) > 0.01: W.append(f"λ=0 xfer_S={mx:.4f} lệch khỏi số 10/12 ({exp['xfer_S']}) > 0.01")
            if abs(mt - exp["total"]) > 0.01: W.append(f"λ=0 tổng scIB={mt:.4f} lệch khỏi số 10/12 ({exp['total']}) > 0.01")

    md = summary_md(pair, S, W)
    open(f"{OUT_p}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + f"\n[{pair}]\n" + md + "\nXONG " + pair)


# =============================================================================== summary.md
def _ms(vals):
    v = [x for x in vals if x is not None]
    if not v: return None, None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)

def beats(ma, sa, mb, sb, thr_min=0.02):
    if ma is None or mb is None: return None
    thr = max(thr_min, 2 * max(sa or 0, sb or 0)); d = ma - mb
    return {"delta": round(d, 4), "thr": round(thr, 4), "pass": d >= thr}

def not_much_worse(m_lam, m_0, margin=0.02):
    """(m_lam − m_0) ≥ −margin: λ không kém λ=0 quá `margin` (ngưỡng cố định, không theo SD — đúng chữ I1)."""
    return None if m_lam is None or m_0 is None else (m_lam - m_0) >= -margin

TABLE_COLS = ["xfer_S", "xfer_coarse", "ilisi", "kbet", "bras", "nmi", "ari", "asw", "batch", "bio", "total", "sp", "keep_struct_cos", "drift"]

def summary_md(pair, S, W):
    L = [f"# 13/{pair} ({PAIR_REP[pair]}) — tóm tắt (quét λ_id)", S["config"], ""]
    rows = []
    for variant in ("raw", "center"):
        v = S["base"].get(variant)
        if v: rows.append({"map": variant, **{c: v.get(c) for c in TABLE_COLS}})
    for lam in LAMBDAS:
        rl = [v["eval"] for v in S["runs"].values() if v.get("lam") == lam and "eval" in v]
        r = {"map": f"λ_id={lam}(n={len(rl)})"}
        for c in TABLE_COLS:
            m, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m:.3f}±{sd:.3f}" if m is not None else "—"
        rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    def lam_ms(lam, key):
        rl = [v["eval"] for v in S["runs"].values() if v.get("lam") == lam and "eval" in v]
        return _ms([e.get(key) for e in rl])
    sp0, ssp0 = lam_ms(0, "sp"); t0, st0 = lam_ms(0, "total"); x0, sx0 = lam_ms(0, "xfer_S")
    best = None
    for lam in LAMBDAS[1:]:
        spl, sspl = lam_ms(lam, "sp"); tl, _ = lam_ms(lam, "total"); xl, _ = lam_ms(lam, "xfer_S")
        r_sp = beats(spl, sspl, sp0, ssp0)
        if r_sp and r_sp["pass"] and not_much_worse(tl, t0) and not_much_worse(xl, x0):
            best = lam; break
    L.append(f"- I1: λ_id nhỏ nhất đạt = {best if best is not None else 'không có'} (nội suy vượt λ=0 mà tổng/chuyển nhãn không kém quá 0.02)")
    if best is not None:
        spl, _ = lam_ms(best, "sp"); tl, _ = lam_ms(best, "total")
        cen_sp = S["base"].get("center", {}).get("sp"); cen_tot = S["base"].get("center", {}).get("total")
        L.append(f"- I2 (λ_id={best}): nội suy cách center {round(spl - cen_sp, 4) if cen_sp is not None else None} | "
                f"tổng hơn center {round(tl - cen_tot, 4) if cen_tot is not None else None}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    for pair in PAIRS:
        assert pair in PAIR_REP, f"PAIRS không hợp lệ: {pair}"
        process_pair(pair)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run14.py
"""14 — Regularization identity CHỈ phạt phần tuyến tính (U3b, tiếp theo 13). Vòng 13 dùng λ·MSE(f(x), x): phạt cả
bias, nên CosMx (cần dịch chuyển lớn) bị kéo về vị trí thô, trộn lô sụp (RESEARCH_CONTEXT 5.10). Vòng này thêm chế độ
`linear`: λ·MSE(Wx, x) = λ·E‖(W − I)x‖² (không phạt bias) qua biến môi trường USHER_ID_MODE, giữ nguyên bản vá `full`
của 13. Tái sử dụng gần như nguyên vẹn `run13.py` (luồng, đánh giá, drift, bảng) — chỉ đổi biến môi trường và thêm
cột `bias_norm`. Chỉ import `run13` (mọi module khác lấy qua thuộc tính của nó, như plan yêu cầu).
"""
import os, sys, glob, json, gc, shutil
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run13 as R13
R, B, M, C, R6, R7, R8, R9, R11 = R13.R, R13.B, R13.M, R13.C, R13.R6, R13.R7, R13.R8, R13.R9, R13.R11
np, pd, sp, log, load_assay, _unit = R13.np, R13.pd, R13.sp, R13.log, R13.load_assay, R13._unit
SMOKE = R13.SMOKE   # đã xử lý biến môi trường SMOKE khi import run13 (ở trên, trước khi run13 import run05/06/...)

OUT = os.environ.get("OUT_DIR_14", "/kaggle/working/out14" + ("_smoke" if SMOKE else ""))
TARGET = R13.TARGET
PAIR_REP = R13.PAIR_REP
PAIRS = [p.strip() for p in os.environ.get("PAIRS", ",".join(PAIR_REP)).split(",") if p.strip()]
LAMBDAS, SEEDS, K_AUX = R13.LAMBDAS, R13.SEEDS, R13.K_AUX
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = R13.time.time()
def step(msg): log(f"    [{R13.time.time() - T0:6.0f}s] {msg}")

run_list, run_name = R13.run_list, R13.run_name   # y hệt 13: λ_id ∈ {0,.1,.3,1,3} × seed {0,1,2}, seed 0 mọi λ trước

# Số 13 (RESEARCH_CONTEXT 5.10, dạng full) để kiểm tra tái lập λ=0 (±0.01) và làm mốc so sánh I4 ở λ=0.3
EXPECT_LAM0_R13 = {"cosmx6k": {"xfer_S": 0.810, "total": 0.675}, "visiumhd_ff": {"xfer_S": 0.678, "total": 0.696}}
FULL_LAM03_R13 = {"cosmx6k": {"total": 0.604, "sp": 0.168}, "visiumhd_ff": {"total": 0.691, "sp": 0.169}}


def read_bias_norm(save_dir, Fs):
    """Đọc ‖bias‖ của model vừa lưu (cơ chế USHER_SAVE_DIR có sẵn trong run_usher.py, patch từ 05c) / trung vị ‖x‖."""
    import torch
    ck = torch.load(f"{save_dir}/alignment_model.pt", map_location="cpu")
    b = ck["model_state_dict"]["net.bias"].numpy()
    return float(np.linalg.norm(b)) / float(np.median(np.linalg.norm(Fs, axis=1)))


def process_pair(pair):
    rep = PAIR_REP[pair]
    OUT_p = f"{OUT}/{pair}"; os.makedirs(OUT_p, exist_ok=True)
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH [{pair}/{rep}]: USHER U6 sketch={C.SKETCH} fix=True n_iters={C.N_ITERS} hidden_dim=None(linear) "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine sketch_obsm_key=X_none cache=off | "
           f"USHER_ID_MODE=linear (E||(W-I)x||^2, không phạt bias) | giao thức cosine mới | aux=aux_cell k_aux={K_AUX} "
           f"scib_k={R11.K_KNN} | λ_id∈{LAMBDAS} seeds={SEEDS} | N_TGT={M.N_TGT} N_REF={M.N_REF} N_BOOT={C.N_BOOT}"
           + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  [{pair}] run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted({pair, TARGET}), EMB)
    ds, dx = D[pair], D[TARGET]
    S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
    log(f"[{pair}] S chung = {S_shared}")

    S = json.load(open(f"{OUT_p}/summary.json")) if RESUME and os.path.exists(f"{OUT_p}/summary.json") else {}
    S["config"] = cfg; S.setdefault("base", {}); S.setdefault("runs", {})
    def save(): json.dump(S, open(f"{OUT_p}/summary.json", "w"), indent=1, default=str)

    Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
    Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
    genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1]
    _, _, _, ref = C.mix_ref(pair, D)
    IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
    idx_src, idx_tgt = R11.build_eval_idx(ds, dx, S_shared)

    if rep == "pca":
        Fx_rep, Fs_rep, _ = R8.pca_cell(Xx_lite, varx, Xs_lite, vars_)
    else:
        Fs_rep, Fx_rep = F[(pair, "scgpt")], F[(TARGET, "scgpt")]

    classes = np.array(sorted(set(dx["lab"])))
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"]); mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], classes, k=K_AUX)   # aux_cell, giao thức mới (như 12/13)
    lab_aux = classes[P_aux.argmax(1)]
    log(f"[{pair}/{rep}] aux_cell acc_H={M.bacc_on(ds['lab'][ds['H']], lab_aux[ds['H']], S_shared)}")

    def drift(Fmap, Fs):
        return float(np.median(np.linalg.norm(Fmap - Fs, axis=1)) / np.median(np.linalg.norm(Fs, axis=1)))

    def eval_all(name, Fmap):
        ev = R9.eval_both(pair, D, Fs_rep, Fx_rep, name, Fmap, S_shared, IMP, ref)[1]   # nhánh mới
        mp = R11.metrics_pack(Fmap, Fx_rep, ds["lab"], dx["lab"], idx_src, idx_tgt)
        w = [f"{name}: {x}" for x in mp.pop("_warnings")]
        return {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "keep_struct_cos": ev["keep_struct_cos"],
                "sp": ev["imp"]["spearman_med"], "drift": drift(Fmap, Fs_rep), "bias_norm": None}, w

    # ---- baseline: raw, center (không có model -> bias_norm = None)
    b = S["base"]
    if "raw" not in b:
        for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
            ev, w = eval_all(variant, Fmap); W += w; b[variant] = ev
        save()

    # ---- USHER, chế độ linear (E||(W-I)x||^2), quét λ_id
    for lam, seed in runs:
        name = run_name(lam, seed); f = f"{OUT_p}/{name}.npy"
        if RESUME and os.path.exists(f) and "eval" in S["runs"].get(name, {}):
            step(f"[{pair}] {name}: đã có, bỏ qua"); continue
        if RESUME and os.path.exists(f) and "info" in S["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), S["runs"][name]["info"]
            bias_norm = S["runs"][name].get("bias_norm"); step(f"[{pair}] {name}: nạp lại map")
        else:
            tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            step(f"USHER [{pair}] {name} | target {len(tgt)} tế bào")
            os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = str(lam)   # vá m_step_utils.py (mục 3)
            save_dir = f"{OUT_p}/_model_tmp_{name}"; os.environ["USHER_SAVE_DIR"] = save_dir
            Fmap, info = R6.usher(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], classes, "linear", seed, False, False)
            bias_norm = read_bias_norm(save_dir, Fs_rep); shutil.rmtree(save_dir, ignore_errors=True)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            S["runs"][name] = {"lam": lam, "seed": seed, "info": info, "bias_norm": bias_norm}; save()
        ev, w = eval_all(name, Fmap); W += w; ev["bias_norm"] = bias_norm
        S["runs"][name]["eval"] = ev; save()
        log(f"  >> [{pair}] {name}: EM={S['runs'][name]['info'].get('em_seconds')}s xfer_S={ev['xfer_S']} sp={ev['sp']} "
            f"drift={ev['drift']} bias_norm={bias_norm}")
        del Fmap; gc.collect()

    exp = EXPECT_LAM0_R13.get(pair)
    if exp:
        rl0 = [v["eval"] for v in S["runs"].values() if v.get("lam") == 0 and "eval" in v]
        if rl0:
            mx = float(np.mean([e["xfer_S"] for e in rl0])); mt = float(np.mean([e["total"] for e in rl0]))
            if abs(mx - exp["xfer_S"]) > 0.01: W.append(f"λ=0 xfer_S={mx:.4f} lệch khỏi số 13 ({exp['xfer_S']}) > 0.01")
            if abs(mt - exp["total"]) > 0.01: W.append(f"λ=0 tổng scIB={mt:.4f} lệch khỏi số 13 ({exp['total']}) > 0.01")

    md, best = summary_md(pair, S, W)
    S["i1_best"] = best; save()
    open(f"{OUT_p}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + f"\n[{pair}]\n" + md + "\nXONG " + pair)


# =============================================================================== summary.md
_ms, beats, not_much_worse = R13._ms, R13.beats, R13.not_much_worse
TABLE_COLS = R13.TABLE_COLS + ["bias_norm"]

def summary_md(pair, S, W):
    L = [f"# 14/{pair} ({PAIR_REP[pair]}) — tóm tắt (quét λ_id, chỉ phạt phần tuyến tính)", S["config"], ""]
    rows = []
    for variant in ("raw", "center"):
        v = S["base"].get(variant)
        if v: rows.append({"map": variant, **{c: v.get(c) for c in TABLE_COLS}})
    for lam in LAMBDAS:
        rl = [v["eval"] for v in S["runs"].values() if v.get("lam") == lam and "eval" in v]
        r = {"map": f"λ_id={lam}(n={len(rl)})"}
        for c in TABLE_COLS:
            m, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m:.3f}±{sd:.3f}" if m is not None else "—"
        rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    def lam_ms(lam, key):
        rl = [v["eval"] for v in S["runs"].values() if v.get("lam") == lam and "eval" in v]
        return _ms([e.get(key) for e in rl])
    sp0, ssp0 = lam_ms(0, "sp"); t0, st0 = lam_ms(0, "total"); x0, sx0 = lam_ms(0, "xfer_S")
    best = None
    for lam in LAMBDAS[1:]:
        spl, sspl = lam_ms(lam, "sp"); tl, _ = lam_ms(lam, "total"); xl, _ = lam_ms(lam, "xfer_S")
        r_sp = beats(spl, sspl, sp0, ssp0)
        if r_sp and r_sp["pass"] and not_much_worse(tl, t0) and not_much_worse(xl, x0):
            best = lam; break
    L.append(f"- I1: λ_id nhỏ nhất đạt = {best if best is not None else 'không có'}")

    if best is not None:
        spl, _ = lam_ms(best, "sp"); tl, _ = lam_ms(best, "total")
        cen_sp = S["base"].get("center", {}).get("sp"); cen_tot = S["base"].get("center", {}).get("total")
        L.append(f"- I2 (λ_id={best}): nội suy cách center {round(spl - cen_sp, 4) if cen_sp is not None else None} | "
                f"tổng hơn center {round(tl - cen_tot, 4) if cen_tot is not None else None}")

    if pair == "cosmx6k":
        t03, st03 = lam_ms(0.3, "total")
        r4 = beats(t03, st03, FULL_LAM03_R13["cosmx6k"]["total"], 0)
        L.append(f"- I4 (λ=0.3): tổng scIB linear={t03} so với full={FULL_LAM03_R13['cosmx6k']['total']} (13) -> "
                f"Δ={r4['delta'] if r4 else None} {'ĐẠT (xác nhận giả thuyết bias)' if r4 and r4['pass'] else 'không đạt'}")
    L.append(f"- Đối chiếu dạng `full` (13, λ=0.3): {pair} tổng={FULL_LAM03_R13.get(pair, {}).get('total')} "
            f"nội suy={FULL_LAM03_R13.get(pair, {}).get('sp')}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n", best


def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    for pair in PAIRS:
        assert pair in PAIR_REP, f"PAIRS không hợp lệ: {pair}"
        process_pair(pair)

    # I1-chung: có một λ_id đạt I1 ở CẢ HAI cặp (chỉ tính được khi cả hai cặp đã chạy trong phiên này hoặc có sẵn output)
    i1 = {}
    for pair in PAIR_REP:
        f = f"{OUT}/{pair}/summary.json"
        if os.path.exists(f):
            s = json.load(open(f)); i1[pair] = s.get("i1_best")
    if len(i1) == len(PAIR_REP) and all(v is not None for v in i1.values()) and len(set(i1.values())) == 1:
        line = f"I1-chung: λ_id = {next(iter(i1.values()))} đạt ở cả hai cặp -> đề xuất làm mặc định"
    else:
        line = f"I1-chung: không có λ_id chung, hoặc chưa đủ dữ liệu cả hai cặp ({i1})"
    log(line)
    open(f"{OUT}/i1_chung.txt", "w").write(line + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run15.py
"""15 — Kiểm chứng identity reg λ_id = 0.3 (dạng `linear`, U3b/13/14) ở 6 tổ hợp (cặp, biểu diễn) KHÁC với hai tổ hợp
đã quét ở 13/14 (cosmx6k/pca, visiumhd_ff/scgpt — không lặp lại): cosmx6k/scgpt, visiumhd_ff/pca, visiumhd_ffpe/pca,
visiumhd_ffpe/scgpt, stereoseq/pca, stereoseq/scgpt. λ_id ∈ {0, 0.3} × seed {0,1,2}/tổ hợp, USHER_ID_MODE=linear.
Tái sử dụng run14 (vá λ_id+USHER_ID_MODE, đánh giá xfer/scib/nội suy/drift, bias_norm qua USHER_SAVE_DIR) và run12
(nạp nhiều cặp cùng lúc, dựng pca/scgpt, aux_cell giao thức mới, fallback nhãn thô cho Stereo). Chỉ import run14, run12.
"""
import os, sys, glob, json, gc, shutil, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run14 as R14, run12 as R12
R, B, M, C, R6, R7, R8, R9, R11 = R14.R, R14.B, R14.M, R14.C, R14.R6, R14.R7, R14.R8, R14.R9, R14.R11
np, pd, sp, log, load_assay, _unit = R14.np, R14.pd, R14.sp, R14.log, R14.load_assay, R14._unit
_ms, beats, not_much_worse = R14._ms, R14.beats, R14.not_much_worse
SMOKE = R14.SMOKE   # xử lý biến môi trường SMOKE đã xảy ra khi import run14 -> run13 (trước khi import run06...)

OUT = os.environ.get("OUT_DIR_15", "/kaggle/working/out15" + ("_smoke" if SMOKE else ""))
TARGET = R14.TARGET
COMBOS_ALL = ["cosmx6k/scgpt", "visiumhd_ff/pca", "visiumhd_ffpe/pca", "visiumhd_ffpe/scgpt", "stereoseq/pca", "stereoseq/scgpt"]
COMBOS = [c.strip() for c in os.environ.get("COMBOS", ",".join(COMBOS_ALL)).split(",") if c.strip()]
LAMBDAS = [0, 0.3]
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX = R12.K_AUX
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")

# Mốc λ_id=0 để kiểm tra tái lập (plan mục 4, ±0.01): visiumhd_*/stereoseq từ bảng 5.9 (12, aux_cell);
# cosmx6k/scgpt chưa có ở 12 (12 chỉ chạy 3 cặp giải trình tự) -> lấy từ 5.7/5.8 (10/11, USHER không nhãn scGPT không niche).
EXPECT_LAM0 = {
    "cosmx6k/scgpt": {"xfer": 0.688, "total": 0.621},
    "visiumhd_ff/pca": {"xfer": 0.674, "total": 0.667},
    "visiumhd_ffpe/pca": {"xfer": 0.780, "total": 0.659},
    "visiumhd_ffpe/scgpt": {"xfer": 0.764, "total": 0.656},
    "stereoseq/pca": {"xfer": 0.545, "total": 0.493},
    "stereoseq/scgpt": {"xfer": 0.609, "total": 0.535},
}
TABLE_COLS = ["xfer_main", "xfer_coarse", "ilisi", "kbet", "bras", "nmi", "ari", "asw", "batch", "bio", "total", "sp",
              "keep_struct_cos", "drift"]


def run_list():
    """Thứ tự (plan mục 3): seed 0 mọi tổ hợp × λ trước, rồi seed 1, seed 2 — để một phiên Kaggle bị ngắt giữa chừng
    vẫn có ít nhất 1 seed cho MỌI tổ hợp thay vì xong hẳn vài tổ hợp và bỏ trắng phần còn lại."""
    return [(combo, lam, seed) for seed in SEEDS for combo in COMBOS for lam in LAMBDAS]

def run_name(combo, lam, seed):
    return f"{combo.replace('/', '_')}_lam{lam:g}_s{seed}"


def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    W = []
    for combo in COMBOS: assert combo in COMBOS_ALL, f"COMBOS không hợp lệ: {combo}"
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER U6 sketch={C.SKETCH} fix=True n_iters={C.N_ITERS} hidden_dim=None(linear) "
           f"alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} knn_k=30 metric=cosine sketch_obsm_key=X_none cache=off | "
           f"USHER_ID_MODE=linear (E||(W-I)x||^2) | giao thức cosine mới | aux=aux_cell k_aux={K_AUX} scib_k={R11.K_KNN} | "
           f"λ_id∈{LAMBDAS} seeds={SEEDS} combos={COMBOS} | N_TGT={M.N_TGT} N_REF={M.N_REF} N_BOOT={C.N_BOOT}"
           + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  run {i:3d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]

    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    needed_pairs = sorted({c.split("/")[0] for c in COMBOS})
    D, F, EXPR = R6.load_all(sorted(set(needed_pairs) | {TARGET}), EMB)

    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("combos", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    # ---- ngữ cảnh mỗi CẶP (dùng chung cho các rep của cùng cặp): S_chung/thô, idx đánh giá scib, IMP nội suy gene
    PCTX = {}
    for pair in needed_pairs:
        ds, dx = D[pair], D[TARGET]
        S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
        cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
        coarse_groups = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]   # như cross_assay_ci
        use_coarse = len(S_shared) < 3   # Stereo: 2 nhóm chung -> dùng nhãn/thước đo thô (plan mục 3, như 12)
        lab_src_ev = ds["lab_c"] if use_coarse else ds["lab"]; lab_tgt_ev = dx["lab_c"] if use_coarse else dx["lab"]
        groups_ev = coarse_groups if use_coarse else S_shared
        idx_src, idx_tgt = R11.build_eval_idx({**ds, "lab": lab_src_ev}, {**dx, "lab": lab_tgt_ev}, groups_ev)
        Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
        Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
        genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
        assert len(genes) == nG == Ys.shape[1]
        _, _, _, ref = C.mix_ref(pair, D)
        IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
        classes = np.array(sorted(set(dx["lab"])))
        PCTX[pair] = dict(ds=ds, dx=dx, S_shared=S_shared, use_coarse=use_coarse,
                           xfer_key=("xfer_coarse" if use_coarse else "xfer_S"),
                           idx_src=idx_src, idx_tgt=idx_tgt, lab_src_ev=lab_src_ev, lab_tgt_ev=lab_tgt_ev,
                           IMP=IMP, ref=ref, classes=classes, Xs_lite=Xs_lite, vars_=vars_, Xx_lite=Xx_lite, varx=varx)
        log(f"[{pair}] S_chung={S_shared} (thô={coarse_groups})" + (" [DÙNG NHÃN/THƯỚC ĐO THÔ]" if use_coarse else ""))

    pca_cache = {}
    def rep_features(pair, rep):
        if rep == "pca":
            if pair not in pca_cache:
                pctx = PCTX[pair]
                Fx_pca, Fs_pca, _ = R8.pca_cell(pctx["Xx_lite"], pctx["varx"], pctx["Xs_lite"], pctx["vars_"])
                pca_cache[pair] = (Fs_pca, Fx_pca)
            return pca_cache[pair]
        return F[(pair, "scgpt")], F[(TARGET, "scgpt")]

    def drift(Fmap, Fs):
        return float(np.median(np.linalg.norm(Fmap - Fs, axis=1)) / np.median(np.linalg.norm(Fs, axis=1)))

    def eval_all(pair, name, Fmap, Fs_rep, Fx_rep):
        pctx = PCTX[pair]
        ev = R9.eval_both(pair, D, Fs_rep, Fx_rep, name, Fmap, pctx["S_shared"], pctx["IMP"], pctx["ref"])[1]
        mp = R11.metrics_pack(Fmap, Fx_rep, pctx["lab_src_ev"], pctx["lab_tgt_ev"], pctx["idx_src"], pctx["idx_tgt"])
        w = [f"{name}: {x}" for x in mp.pop("_warnings")]
        out = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "keep_struct_cos": ev["keep_struct_cos"],
               "sp": ev["imp"]["spearman_med"], "drift": drift(Fmap, Fs_rep)}
        out["xfer_main"] = out[pctx["xfer_key"]]
        return out, w

    # ---- ngữ cảnh mỗi TỔ HỢP (pair, rep): rep_features, aux_cell (giao thức mới, như 12/13/14), baseline raw/center
    COMBO = {}
    for combo in COMBOS:
        pair, rep = combo.split("/")
        Sc = S["combos"].setdefault(combo, {"pair": pair, "rep": rep, "base": {}, "runs": {}})
        Fs_rep, Fx_rep = rep_features(pair, rep)
        pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
        center_map = M.base_center(Fs_rep, Fx_rep, ds["T"]); mu = Fx_rep.mean(0)
        P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=K_AUX)
        lab_aux = pctx["classes"][P_aux.argmax(1)]
        log(f"[{combo}] aux_cell acc_H={M.bacc_on(ds['lab'][ds['H']], lab_aux[ds['H']], pctx['S_shared'])}")
        if "raw" not in Sc["base"]:
            for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
                ev, w = eval_all(pair, f"{combo}/{variant}", Fmap, Fs_rep, Fx_rep); W += w; Sc["base"][variant] = ev
            save()
        COMBO[combo] = dict(pair=pair, rep=rep, Fs_rep=Fs_rep, Fx_rep=Fx_rep, P_aux=P_aux, lab_aux=lab_aux)

    # ---- lưới USHER: λ_id ∈ {0, 0.3} × seed {0,1,2}/tổ hợp, USHER_ID_MODE=linear, thứ tự seed-trước (run_list)
    for combo, lam, seed in runs:
        name = run_name(combo, lam, seed); f = f"{OUT}/{name}.npy"
        cc = COMBO[combo]; pair = cc["pair"]; Fs_rep, Fx_rep = cc["Fs_rep"], cc["Fx_rep"]
        pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
        Sc = S["combos"][combo]
        if RESUME and os.path.exists(f) and "eval" in Sc["runs"].get(name, {}):
            step(f"{name}: đã có, bỏ qua"); continue
        if RESUME and os.path.exists(f) and "info" in Sc["runs"].get(name, {}):
            Fmap, info = np.load(f).astype(np.float32), Sc["runs"][name]["info"]
            bias_norm = Sc["runs"][name].get("bias_norm"); step(f"{name}: nạp lại map")
        else:
            tgt = C.strat_target(cc["lab_aux"][ds["T"]], dx["lab"])
            step(f"USHER {name} | target {len(tgt)} tế bào")
            os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = str(lam)   # vá m_step_utils.py (U3b)
            save_dir = f"{OUT}/_model_tmp_{name}"; os.environ["USHER_SAVE_DIR"] = save_dir
            Fmap, info = R6.usher(Fs_rep, ds["T"], cc["P_aux"], Fx_rep, tgt, dx["lab"], pctx["classes"], "linear", seed, False, False)
            bias_norm = R14.read_bias_norm(save_dir, Fs_rep); shutil.rmtree(save_dir, ignore_errors=True)
            info.pop("cache_stats", None); np.save(f, Fmap.astype(np.float16))
            Sc["runs"][name] = {"lam": lam, "seed": seed, "info": info, "bias_norm": bias_norm}; save()
        ev, w = eval_all(pair, name, Fmap, Fs_rep, Fx_rep); W += w; ev["bias_norm"] = bias_norm
        Sc["runs"][name] = {**Sc["runs"][name], "eval": ev}; save()
        log(f"  >> {name}: EM={Sc['runs'][name]['info'].get('em_seconds')}s xfer_main={ev['xfer_main']} sp={ev['sp']} "
            f"drift={ev['drift']} bias_norm={bias_norm}")
        del Fmap; gc.collect()

    # ---- kiểm tra tái lập λ_id=0 so với số 10/12 (±0.01, plan mục 4)
    for combo in COMBOS:
        exp = EXPECT_LAM0.get(combo)
        rl0 = [v["eval"] for v in S["combos"].get(combo, {}).get("runs", {}).values() if v.get("lam") == 0 and "eval" in v]
        if not exp or not rl0: continue
        mx = float(np.mean([e["xfer_main"] for e in rl0])); mt = float(np.mean([e["total"] for e in rl0]))
        if abs(mx - exp["xfer"]) > 0.01: W.append(f"{combo} λ=0 xfer={mx:.4f} lệch khỏi số 10/12 ({exp['xfer']}) > 0.01")
        if abs(mt - exp["total"]) > 0.01: W.append(f"{combo} λ=0 tổng scIB={mt:.4f} lệch khỏi số 10/12 ({exp['total']}) > 0.01")

    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 15")


# =============================================================================== summary.md (một file, mọi tổ hợp)
def summary_md(S, W):
    L = ["# 15 — kiểm chứng identity reg λ_id=0.3 (linear) ở 6 tổ hợp mới", S["config"], ""]
    rows = []
    for combo in COMBOS:
        Sc = S["combos"].get(combo)
        if not Sc: continue
        for variant in ("raw", "center"):
            v = Sc["base"].get(variant)
            if v: rows.append({"tổ_hợp": combo, "map": variant, **{c: v.get(c) for c in TABLE_COLS}})
        for lam in LAMBDAS:
            rl = [v["eval"] for v in Sc["runs"].values() if v.get("lam") == lam and "eval" in v]
            r = {"tổ_hợp": combo, "map": f"λ_id={lam}(n={len(rl)})"}
            for c in TABLE_COLS:
                m, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m:.3f}±{sd:.3f}" if m is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    def lam_ms(combo, lam, key):
        rl = [v["eval"] for v in S["combos"].get(combo, {}).get("runs", {}).values() if v.get("lam") == lam and "eval" in v]
        return _ms([e.get(key) for e in rl])

    L.append("## J1 mỗi tổ hợp (gap = nội suy(center) − nội suy(λ=0))")
    graded, achieved = 0, 0
    for combo in COMBOS:
        Sc = S["combos"].get(combo)
        if not Sc: continue
        cen_sp = Sc["base"].get("center", {}).get("sp")
        sp0, ssp0 = lam_ms(combo, 0, "sp")
        if cen_sp is None or sp0 is None:
            L.append(f"- {combo}: thiếu dữ liệu"); continue
        gap = round(cen_sp - sp0, 4)
        if gap < 0.02:
            L.append(f"- {combo}: gap={gap} < 0.02 -> chỉ mô tả (không có gì để sửa)"); continue
        sp03, ssp03 = lam_ms(combo, 0.3, "sp"); t0, _ = lam_ms(combo, 0, "total"); t03, _ = lam_ms(combo, 0.3, "total")
        x0, _ = lam_ms(combo, 0, "xfer_main"); x03, _ = lam_ms(combo, 0.3, "xfer_main")
        r_sp = beats(sp03, ssp03, sp0, ssp0)
        a_ok = bool(r_sp and r_sp["pass"] and (r_sp["delta"] >= 0.5 * gap))
        b_ok = not_much_worse(t03, t0); c_ok = not_much_worse(x03, x0)
        ok = bool(a_ok and b_ok and c_ok)
        graded += 1; achieved += int(ok)
        d_t = round(t03 - t0, 4) if (t03 is not None and t0 is not None) else None
        d_x = round(x03 - x0, 4) if (x03 is not None and x0 is not None) else None
        L.append(f"- {combo}: gap={gap} Δnội_suy={r_sp['delta'] if r_sp else None} Δtổng={d_t} Δchuyển_nhãn={d_x} "
                f"-> {'ĐẠT' if ok else 'KHÔNG ĐẠT'}")

    graded_total, achieved_total = graded + 1, achieved + 1   # +1: FF/scGPT đã ĐẠT I1 ở 13/14 (plan mục 4)
    j2 = (achieved_total / graded_total) >= 0.8
    L.append(f"- J2: {achieved_total}/{graded_total} tổ hợp được chấm ĐẠT J1 (tính cả FF/scGPT đã đạt ở 13,14) -> "
            f"{'ĐỀ XUẤT λ_id=0.3 (linear) làm mặc định USHER-ST' if j2 else 'chưa đủ 80%, KHÔNG đề xuất mặc định'}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run16.py
"""16 — Khép giai đoạn 1: USHER-ST (CẤU HÌNH MẶC ĐỊNH, RESEARCH_CONTEXT mục 6 [CHỐT, 15]: U6 hidden_dim=None, aux
aux_cell giao thức mới, USHER_ID_MODE=linear + USHER_LAMBDA_ID=0.3, struct_cache MODE="target") so với 3 baseline
tích hợp không đổi FM: Harmony, Scanorama (chỉnh CẢ HAI phía), và MNN-ref (tham chiếu Xenium CỐ ĐỊNH, kiểu
mnnCorrect/Seurat anchors, tự viết vì không có gói MNN embedding sẵn phù hợp). 4 cặp/biểu diễn mặc định:
cosmx6k/pca, visiumhd_ff/scgpt, visiumhd_ffpe/pca, stereoseq/scgpt. Tái sử dụng run15/run14 (vá λ_id+USHER_ID_MODE,
nạp cặp, dựng pca/scgpt, aux_cell, đánh giá eval_both nhánh mới + metrics_pack, nhãn thô Stereo), struct_cache.py.
"""
import os, sys, glob, json, gc, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run15 as R15, run14 as R14
R, B, M, C, R6, R7, R8, R9, R11, R12 = R15.R, R15.B, R15.M, R15.C, R15.R6, R15.R7, R15.R8, R15.R9, R15.R11, R15.R12
np, pd, sp, log, load_assay, _unit = R15.np, R15.pd, R15.sp, R15.log, R15.load_assay, R15._unit
_ms, beats, not_much_worse = R15._ms, R15.beats, R15.not_much_worse
SMOKE = R15.SMOKE   # xử lý biến môi trường SMOKE đã xảy ra khi import run15 -> run14 -> ... -> run06 (như 15)
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

OUT = os.environ.get("OUT_DIR_16", "/kaggle/working/out16" + ("_smoke" if SMOKE else ""))
TARGET = R15.TARGET
ONLY_METHODS = {m.strip() for m in os.environ.get("ONLY_METHODS", "").split(",") if m.strip()}   # 16c: ví dụ ONLY_METHODS=harmony
def _want(m): return not ONLY_METHODS or m in ONLY_METHODS
# cặp/biểu diễn mặc định (plan mục 3). Vòng 26: COMBOS_ALL=... để đổi cho mô khác (OV chọn rep theo accS của 26c).
COMBOS_ALL = [c.strip() for c in os.environ.get("COMBOS_ALL", "cosmx6k/pca,visiumhd_ff/scgpt,visiumhd_ffpe/pca,stereoseq/scgpt").split(",") if c.strip()]
# 16b: mặc định chỉ chạy 2 tổ hợp còn thiếu của lần chạy 16 bị hết RAM; COMBOS=... để đổi
COMBOS = [c.strip() for c in os.environ.get("COMBOS", "visiumhd_ffpe/pca,stereoseq/scgpt").split(",") if c.strip()]
SEEDS = [0] if SMOKE else [0, 1, 2]     # chỉ USHER-ST; Harmony/Scanorama/MNN-ref tất định (seed 0, plan mục 3)
K_AUX = R12.K_AUX
K_MNN = 20
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")

# Số 14/15 (λ_id=0.3, linear) để kiểm tra tái lập khi bật struct_cache MODE="target" (plan mục 4, ±0.01)
EXPECT_LAM03 = {
    "cosmx6k/pca": {"xfer": 0.810, "total": 0.634},
    "visiumhd_ff/scgpt": {"xfer": 0.682, "total": 0.703},
    "visiumhd_ffpe/pca": {"xfer": 0.784, "total": 0.677},
    "stereoseq/scgpt": {"xfer": 0.604, "total": 0.510},
}
TABLE_COLS = ["xfer_main", "ilisi", "kbet", "bras", "graph_conn", "batch_asw", "nmi", "ari", "asw", "clisi", "iso_labels",
              "batch", "bio", "total", "total_v2", "sp", "keep_struct_cos", "ref_drift"]
METHOD_ORDER = ["raw", "center", "mnn_ref", "harmony", "scanorama", "usher"]
METHOD_LABEL = {"raw": "raw", "center": "center", "mnn_ref": "MNN-ref", "harmony": "Harmony",
                "scanorama": "Scanorama", "usher": "USHER-ST"}


def run_list():
    """Chỉ đánh số phần USHER-ST (12 lần GPU, tốn thời gian nhất); baseline CPU luôn chạy hết mỗi lần gọi combo."""
    return [(combo, seed) for seed in SEEDS for combo in COMBOS]

def run_name(combo, seed):
    return f"{combo.replace('/', '_')}_usher_s{seed}"


def usher_cached(Fs, T, P_src, Fx, tgt, lab_x, classes, seed):
    """Như run06.usher nhưng cache='target' (CẤU HÌNH MẶC ĐỊNH giai đoạn 1: C_t đệm geodesic chính xác 1 lần/batch,
    C_s vẫn landmark) và GIỮ NGUYÊN cache_stats (run06.usher bỏ đi qua info.pop, không dùng lại được ở đây)."""
    import torch, random
    os.environ["USHER_SEED"] = str(seed); os.environ["USHER_EM_WORKERS"] = "4"
    os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = "0.3"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cfg = {**R6.U6, "hidden_dim": None, "cache": "target"}
    return C.run_usher_paper(Fs, T, P_src, Fx, tgt, lab_x, classes, cfg)


def mnn_ref_correction(Fs_rep, Fx_rep, T, k=K_MNN):
    """MNN-ref (tham chiếu Xenium CỐ ĐỊNH, plan mục 3.5): MNN k=20 giữa source-T và Xenium trong không gian đã trừ μx
    + chuẩn hoá L2; vector sửa mỗi tế bào bắt cặp = TB (Xenium bạn − chính nó) trên embedding GỐC; vector sửa cho
    MỌI tế bào source = TB trọng số Gaussian (băng thông σ = trung vị khoảng cách tới láng giềng thứ 20 GIỮA các tế
    bào đã bắt cặp — cách đọc của Claude Code, plan không cho công thức σ tường minh, ghi trong Ghi chú triển khai)."""
    mu = Fx_rep.mean(0)
    Fs_n, Fx_n = _unit(Fs_rep - mu), _unit(Fx_rep - mu)
    FsT_n = Fs_n[T]
    kx = min(k, len(Fx_n)); ks = min(k, len(FsT_n))
    s2x = NearestNeighbors(n_neighbors=kx, n_jobs=4).fit(Fx_n).kneighbors(FsT_n, return_distance=False)
    x2s = NearestNeighbors(n_neighbors=ks, n_jobs=4).fit(FsT_n).kneighbors(Fx_n, return_distance=False)
    x2s_set = [set(row) for row in x2s]
    corr_by_local = {}
    for i in range(len(T)):
        for j in s2x[i]:
            if i in x2s_set[j]:
                corr_by_local.setdefault(i, []).append(Fx_rep[j] - Fs_rep[T[i]])
    info = {"n_source_T": int(len(T)), "n_paired": len(corr_by_local)}
    if not corr_by_local:
        return Fs_rep.copy().astype(np.float32), info
    paired_local = np.array(sorted(corr_by_local))
    paired_corr = np.stack([np.mean(corr_by_local[i], axis=0) for i in paired_local])
    paired_global = T[paired_local]
    kk = min(k + 1, len(paired_global))
    d_p, _ = NearestNeighbors(n_neighbors=kk, n_jobs=4).fit(Fs_n[paired_global]).kneighbors(Fs_n[paired_global])
    sigma = float(np.median(d_p[:, -1])) + 1e-8
    # tính theo khối (tránh ma trận n_source x n_paired float64 ~ nhiều GB -> hết RAM)
    P = Fs_n[paired_global].astype(np.float32); C_ = paired_corr.astype(np.float32)
    out = Fs_rep.astype(np.float32).copy()
    for a in range(0, len(Fs_n), 4096):
        d2 = pairwise_distances(Fs_n[a:a + 4096].astype(np.float32), P, squared=True)
        w = np.exp(-d2 / (2 * sigma ** 2)); w /= (w.sum(1, keepdims=True) + 1e-12)
        out[a:a + 4096] += w @ C_
    info["sigma"] = round(sigma, 4)
    return out, info


def run_harmony(Fx_rep, Fs_rep):
    import harmonypy
    X = np.vstack([Fx_rep, Fs_rep]).astype(np.float64)
    meta = pd.DataFrame({"batch": ["xenium"] * len(Fx_rep) + ["source"] * len(Fs_rep)})
    ho = harmonypy.run_harmony(X, meta, ["batch"])
    Z = np.asarray(ho.Z_corr)
    if Z.shape[0] != X.shape[0]: Z = Z.T   # harmonypy < 2 trả (d, n); 2.x trả (n, d) — lỗi IndexError ở 16/16b
    return Z[:len(Fx_rep)].astype(np.float32), Z[len(Fx_rep):].astype(np.float32)


def run_scanorama(Fx_rep, Fs_rep):
    import scanorama
    # batch_size: tính vector sửa theo khối (mặc định None = một lượt -> hết RAM ở FFPE, vòng 16 lần 1)
    integrated = scanorama.assemble([Fx_rep.astype(np.float32), Fs_rep.astype(np.float32)], batch_size=5000)
    return np.asarray(integrated[0], np.float32), np.asarray(integrated[1], np.float32)


def cmp3(m, sd, base):
    """'vượt'/'ngang'/'kém' của USHER-ST (TB±SD qua 3 seed) so với baseline tất định (SD=0), quy tắc mục 5."""
    if m is None or base is None: return None
    thr = max(0.02, 2 * (sd or 0)); d = m - base
    return "vượt" if d >= thr else ("kém" if d <= -thr else "ngang")


def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    try:
        import harmonypy; hpy_ver = harmonypy.__version__
    except Exception: hpy_ver = "?"
    try:
        import scanorama; scan_ver = getattr(scanorama, "__version__", "?")
    except Exception: scan_ver = "?"
    cfg = (f"CẤU HÌNH: USHER-ST MẶC ĐỊNH giai đoạn 1 (U6 hidden_dim=None alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} "
           f"aux=aux_cell k_aux={K_AUX} USHER_ID_MODE=linear USHER_LAMBDA_ID=0.3 struct_cache=target) sketch={C.SKETCH} "
           f"fix=True n_iters={C.N_ITERS} scib_k={R11.K_KNN} seeds(USHER)={SEEDS} | harmonypy={hpy_ver} scanorama={scan_ver} | "
           f"Harmony/Scanorama chỉnh CẢ HAI phía, fit 1 lần trên toàn bộ Xenium + toàn bộ source T∪H (không dùng nhãn) | "
           f"MNN-ref: tham chiếu Xenium CỐ ĐỊNH, k={K_MNN}, fit trên source-T, áp cho mọi tế bào | combos={COMBOS}"
           + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    runs_all = run_list()
    for i, r in enumerate(runs_all): log(f"  run {i:2d}: {run_name(*r)}")
    runs = [r for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")]
    if not _want("usher"): runs = []

    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    needed_pairs = sorted({c.split("/")[0] for c in COMBOS})
    D, F, EXPR = R6.load_all(sorted(set(needed_pairs) | {TARGET}), EMB)

    def _is16(f):   # summary.json của vòng 16 (bất kể thư mục tên gì khi upload lại)
        try: return "USHER-ST MẶC ĐỊNH" in json.load(open(f)).get("config", "")
        except Exception: return False
    prev = [f for f in sorted(glob.glob("/kaggle/input/**/summary.json", recursive=True)) if _is16(f)]
    if RESUME and prev and not os.path.exists(f"{OUT}/summary.json"):      # phiên sau: nạp output phiên trước (Add Input)
        import shutil
        os.makedirs(OUT, exist_ok=True)
        for f_ in glob.glob(os.path.join(os.path.dirname(prev[0]), "*")):
            if os.path.isfile(f_): shutil.copy(f_, OUT)
        log(f"nạp output phiên trước từ {os.path.dirname(prev[0])}")
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("pairs", {}); S.setdefault("cache_stats", {"target_computed": 0, "target_hits": 0})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    PCTX = {}
    for pair in needed_pairs:
        ds, dx = D[pair], D[TARGET]
        S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
        cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
        coarse_groups = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]
        use_coarse = len(S_shared) < 3   # Stereo (như 12/15)
        lab_src_ev = ds["lab_c"] if use_coarse else ds["lab"]; lab_tgt_ev = dx["lab_c"] if use_coarse else dx["lab"]
        groups_ev = coarse_groups if use_coarse else S_shared
        idx_src, idx_tgt = R11.build_eval_idx({**ds, "lab": lab_src_ev}, {**dx, "lab": lab_tgt_ev}, groups_ev)
        Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
        Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
        genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
        assert len(genes) == nG == Ys.shape[1]
        _, _, _, ref = C.mix_ref(pair, D)
        IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
        classes = np.array(sorted(set(dx["lab"])))
        PCTX[pair] = dict(ds=ds, dx=dx, S_shared=S_shared, xfer_key=("xfer_coarse" if use_coarse else "xfer_S"),
                           idx_src=idx_src, idx_tgt=idx_tgt, lab_src_ev=lab_src_ev, lab_tgt_ev=lab_tgt_ev,
                           IMP=IMP, ref=ref, classes=classes, Xs_lite=Xs_lite, vars_=vars_, Xx_lite=Xx_lite, varx=varx)
        log(f"[{pair}] S_chung={S_shared} (thô={coarse_groups})" + (" [DÙNG NHÃN/THƯỚC ĐO THÔ]" if use_coarse else ""))

    pca_cache = {}
    def rep_features(pair, rep):
        if rep == "pca":
            if pair not in pca_cache:
                pctx = PCTX[pair]
                Fx_pca, Fs_pca, _ = R8.pca_cell(pctx["Xx_lite"], pctx["varx"], pctx["Xs_lite"], pctx["vars_"])
                pca_cache[pair] = (Fs_pca, Fx_pca)
            return pca_cache[pair]
        return F[(pair, "scgpt")], F[(TARGET, "scgpt")]

    def ref_drift(Fx_used, Fx_orig):
        if Fx_used is Fx_orig: return 0.0
        return float(np.median(np.linalg.norm(Fx_used - Fx_orig, axis=1)) / np.median(np.linalg.norm(Fx_orig, axis=1)))

    def eval_all(pair, name, Fmap, Fs_rep, Fx_used, Fx_orig):
        pctx = PCTX[pair]
        ev = R9.eval_both(pair, D, Fs_rep, Fx_used, name, Fmap, pctx["S_shared"], pctx["IMP"], pctx["ref"])[1]
        mp = R11.metrics_pack(Fmap, Fx_used, pctx["lab_src_ev"], pctx["lab_tgt_ev"], pctx["idx_src"], pctx["idx_tgt"])
        w = [f"{name}: {x}" for x in mp.pop("_warnings")]
        out = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "keep_struct_cos": ev["keep_struct_cos"],
               "sp": ev["imp"]["spearman_med"], "ref_drift": ref_drift(Fx_used, Fx_orig)}
        out["xfer_main"] = out[pctx["xfer_key"]]
        return out, w

    for combo in COMBOS:
        pair, rep = combo.split("/")
        pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
        Sc = S["pairs"].setdefault(combo, {"pair": pair, "rep": rep, "results": {}, "notes": {}, "usher": {}})
        RES = Sc["results"]
        Fs_rep, Fx_rep = rep_features(pair, rep)
        center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])

        if "raw" not in RES and _want("raw"):
            for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
                ev, w = eval_all(pair, f"{combo}/{variant}", Fmap, Fs_rep, Fx_rep, Fx_rep); W += w; RES[variant] = ev
            save()

        if "mnn_ref" not in RES and _want("mnn_ref"):
            step(f"MNN-ref [{combo}]")
            Fs_mnn, info_mnn = mnn_ref_correction(Fs_rep, Fx_rep, ds["T"])
            ev, w = eval_all(pair, f"{combo}/mnn_ref", Fs_mnn, Fs_rep, Fx_rep, Fx_rep); W += w
            RES["mnn_ref"] = ev; Sc["notes"]["mnn_ref"] = info_mnn; save()

        if RES.get("harmony") is None and _want("harmony"):
            step(f"Harmony [{combo}]")
            try:
                Fx_h, Fs_h = run_harmony(Fx_rep, Fs_rep)
                ev, w = eval_all(pair, f"{combo}/harmony", Fs_h, Fs_rep, Fx_h, Fx_rep); W += w; RES["harmony"] = ev
            except Exception as e:
                W.append(f"{combo} harmony lỗi: {e!r}"); RES["harmony"] = None
            save()

        if "scanorama" not in RES and _want("scanorama"):
            step(f"Scanorama [{combo}]")
            try:
                Fx_s, Fs_s = run_scanorama(Fx_rep, Fs_rep)
                ev, w = eval_all(pair, f"{combo}/scanorama", Fs_s, Fs_rep, Fx_s, Fx_rep); W += w; RES["scanorama"] = ev
            except Exception as e:
                W.append(f"{combo} scanorama lỗi: {e!r}"); RES["scanorama"] = None
            save()

        mu = Fx_rep.mean(0)
        P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=K_AUX)
        lab_aux = pctx["classes"][P_aux.argmax(1)]
        tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
        for seed in SEEDS:
            if (combo, seed) not in runs: continue
            name = run_name(combo, seed); f = f"{OUT}/{name}.npy"
            if RESUME and os.path.exists(f) and "eval" in Sc["usher"].get(name, {}):
                step(f"{name}: đã có, bỏ qua"); continue
            if RESUME and os.path.exists(f) and "info" in Sc["usher"].get(name, {}):
                Fmap = np.load(f).astype(np.float32); step(f"{name}: nạp lại map")
            else:
                step(f"USHER-ST [{combo}] seed={seed}")
                Fmap, info = usher_cached(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed)
                cs_ = info.pop("cache_stats", {})
                S["cache_stats"]["target_computed"] += cs_.get("target_computed", 0)
                S["cache_stats"]["target_hits"] += cs_.get("target_hits", 0)
                np.save(f, Fmap.astype(np.float16))
                Sc["usher"][name] = {"seed": seed, "info": info}; save()
            ev, w = eval_all(pair, name, Fmap, Fs_rep, Fx_rep, Fx_rep); W += w
            Sc["usher"][name] = {**Sc["usher"][name], "eval": ev}; save()
            log(f"  >> {name}: EM={Sc['usher'][name]['info'].get('em_seconds')}s xfer_main={ev['xfer_main']} "
                f"tổng={ev['total']} bộ_đệm={S['cache_stats']}")
            del Fmap; gc.collect()

        exp = EXPECT_LAM03.get(combo)
        rl = [v["eval"] for v in Sc["usher"].values() if "eval" in v]
        if exp and rl:
            mx = float(np.mean([e["xfer_main"] for e in rl])); mt = float(np.mean([e["total"] for e in rl]))
            if abs(mx - exp["xfer"]) > 0.01: W.append(f"{combo} USHER-ST xfer={mx:.4f} lệch khỏi 14/15 ({exp['xfer']}) > 0.01")
            if abs(mt - exp["total"]) > 0.01: W.append(f"{combo} USHER-ST tổng={mt:.4f} lệch khỏi 14/15 ({exp['total']}) > 0.01")

    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 16")


# =============================================================================== summary.md
def summary_md(S, W):
    L = ["# 16 — USHER-ST (mặc định giai đoạn 1) so với Harmony/Scanorama/MNN-ref", S["config"], ""]
    rows = []
    for combo in [c for c in COMBOS_ALL if c in S.get("pairs", {})]:
        Sc = S["pairs"].get(combo)
        if not Sc: continue
        for m in ("raw", "center", "mnn_ref", "harmony", "scanorama"):
            v = Sc["results"].get(m)
            rows.append({"cặp": combo, "map": METHOD_LABEL[m], **({c: v.get(c) for c in TABLE_COLS} if v else {})})
        rl = [x["eval"] for x in Sc["usher"].values() if "eval" in x]
        r = {"cặp": combo, "map": f"USHER-ST(n={len(rl)})"}
        for c in TABLE_COLS:
            m_, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m_:.3f}±{sd:.3f}" if m_ is not None else "—"
        rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append(f"Bộ đệm C_t (struct_cache MODE=target): tính mới {S['cache_stats']['target_computed']} lần, "
             f"dùng lại {S['cache_stats']['target_hits']} lần (12 lần USHER × {C.N_ITERS} vòng EM).")

    L.append("## K1 (USHER-ST so với từng baseline: vượt/ngang/kém trên tổng scIB, chuyển nhãn chính, nội suy)")
    n_best = 0
    for combo in [c for c in COMBOS_ALL if c in S.get("pairs", {})]:
        Sc = S["pairs"].get(combo)
        if not Sc: continue
        rl = [x["eval"] for x in Sc["usher"].values() if "eval" in x]
        ut, sdt = _ms([e.get("total") for e in rl]); ux, sdx = _ms([e.get("xfer_main") for e in rl]); usp, sdsp = _ms([e.get("sp") for e in rl])
        any_beat, any_lose = False, False
        parts = []
        for base in ("harmony", "scanorama", "mnn_ref"):
            v = Sc["results"].get(base)
            if not v:
                parts.append(f"{METHOD_LABEL[base]}=lỗi/thiếu"); continue
            c_t, c_x, c_s = cmp3(ut, sdt, v.get("total")), cmp3(ux, sdx, v.get("xfer_main")), cmp3(usp, sdsp, v.get("sp"))
            parts.append(f"{METHOD_LABEL[base]}[tổng:{c_t} xfer:{c_x} nội_suy:{c_s}]")
            for c in (c_t, c_x, c_s):
                if c == "vượt": any_beat = True
                if c == "kém": any_lose = True
        L.append(f"- {combo}: " + " | ".join(parts))
        if any_beat and not any_lose: n_best += 1
    L.append(f"- K2: USHER-ST tốt nhất (không kém ai, vượt ≥1) ở {n_best}/{len([c for c in COMBOS_ALL if c in S.get('pairs', {})])} cặp")

    L.append("## K1-v2 (như K1 nhưng tổng kiểu scib-metrics Benchmarker: total_v2)")
    for combo in [c for c in COMBOS_ALL if c in S.get("pairs", {})]:
        Sc = S["pairs"].get(combo)
        rl = [x["eval"] for x in Sc["usher"].values() if "eval" in x]
        ut, sdt = _ms([e.get("total_v2") for e in rl])
        parts = [f"{METHOD_LABEL[b_]}:{cmp3(ut, sdt, (Sc['results'].get(b_) or {}).get('total_v2'))}"
                 for b_ in ("center", "mnn_ref", "harmony", "scanorama")]
        L.append(f"- {combo}: " + " | ".join(parts))
    L.append("## K3 (mô tả: so với nhóm cùng ràng buộc không đổi Xenium — center, MNN-ref)")
    k3_all = True
    for combo in [c for c in COMBOS_ALL if c in S.get("pairs", {})]:
        Sc = S["pairs"].get(combo)
        if not Sc: continue
        rl = [x["eval"] for x in Sc["usher"].values() if "eval" in x]
        ut, sdt = _ms([e.get("total") for e in rl])
        tc = Sc["results"].get("center", {}).get("total"); tm = (Sc["results"].get("mnn_ref") or {}).get("total")
        c_c, c_m = cmp3(ut, sdt, tc), cmp3(ut, sdt, tm)
        ok = (c_c == "vượt") and (c_m == "vượt")
        k3_all = k3_all and ok
        L.append(f"- {combo}: vs center={c_c} vs MNN-ref={c_m}")
    L.append(f"- K3: USHER-ST vượt cả center và MNN-ref về tổng scIB ở MỌI cặp -> {'ĐÚNG' if k3_all else 'KHÔNG'}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run17.py
"""17 — Giai đoạn 2, bước 1: (b) cấu trúc FGW theo vi môi trường (struct_cache MODE="custom") trên 4 tổ hợp mặc định
+ (a, chuẩn bị) chẩn đoán đăng ký affine lát SPATCH so với Xenium (CPU, không chạy USHER; dùng cho aux không gian ở
vòng 18). Tái sử dụng run16 (USHER-ST mặc định, eval_all, struct_cache=target) và run08 (dựng niche: 15 hàng xóm
không gian trên dữ liệu đầy đủ, PCA 50 gene chung). Mở rộng struct_cache.py (đã sửa) thêm MODE="custom": C_s, C_t
đều tính trên đặc trưng TUỲ Ý (ở đây: niche hoặc [cell, niche]) thay vì embedding tế bào, đệm 1 lần/batch như "target".
"""
import os, sys, glob, json, gc, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run16 as R16, run08 as R08
R, B, M, C, R6, R7, R9, R11, R12 = R16.R, R16.B, R16.M, R16.C, R16.R6, R16.R7, R16.R9, R16.R11, R16.R12
R8 = R08
np, pd, sp, log, load_assay, _unit = R16.np, R16.pd, R16.sp, R16.log, R16.load_assay, R16._unit
_ms, beats, not_much_worse = R16._ms, R16.beats, R16.not_much_worse
SMOKE = R16.SMOKE   # xử lý biến môi trường SMOKE đã xảy ra khi import run16 -> ... -> run06 (như 16)
from sklearn.neighbors import NearestNeighbors
from scipy.signal import fftconvolve

OUT = os.environ.get("OUT_DIR_17", "/kaggle/working/out17" + ("_smoke" if SMOKE else ""))
TARGET = R16.TARGET
COMBOS = R16.COMBOS_ALL   # 4 tổ hợp mặc định giai đoạn 1 (cosmx6k/pca, visiumhd_ff/scgpt, visiumhd_ffpe/pca, stereoseq/scgpt)
PAIRS_A = ["cosmx6k", "visiumhd_ff", "visiumhd_ffpe", "stereoseq"]   # Phần A: so với xenium5k
VARIANTS = ["default", "C_niche", "C_mix"]
CACHE_MODE = {"default": "target", "C_niche": "custom", "C_mix": "custom"}
SEEDS = [0] if SMOKE else [0, 1, 2]
K_AUX = R12.K_AUX
K_NICHE = R08.K_NICHE
COARSE_GROUPS = sorted(set(B.COARSE.values()))   # 6 nhóm thô (plan mục 3/4)
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
ONLY_PART = os.environ.get("ONLY_PART", "")   # "A" hoặc "B" để chạy riêng một phần; mặc định cả hai
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")

EPI_MARKERS = ["EPCAM", "KRT8", "KRT18", "KRT19", "CDH1"]
IMM_MARKERS = ["PTPRC", "CD3E", "CD68", "MS4A1"]
GRID_UM = 50.0
ANGLES = list(range(0, 360, 20 if SMOKE else 2))
FLIPS = [False, True]
SCALES = [1.0] if SMOKE else [round(0.8 + 0.05 * i, 2) for i in range(10)]   # 0.8..1.25 bước 0.05

# Số 16 (λ_id=0.3, linear, struct_cache=target) để kiểm tra tái lập biến thể "default" (±0.01). RESEARCH_CONTEXT
# 5.13's bảng là total_v2 (kiểu Benchmarker) — so với "total" (kiểu cũ) là lỗi so nhầm của vòng 17, sửa ở vòng 18.
EXPECT_DEFAULT = {
    "cosmx6k/pca": {"xfer": 0.810, "total_v2": 0.666}, "visiumhd_ff/scgpt": {"xfer": 0.683, "total_v2": 0.727},
    "visiumhd_ffpe/pca": {"xfer": 0.784, "total_v2": 0.699}, "stereoseq/scgpt": {"xfer": 0.607, "total_v2": 0.566},
}


def run_list():
    return [(combo, variant, seed) for seed in SEEDS for combo in COMBOS for variant in VARIANTS]

def run_name(combo, variant, seed):
    return f"{combo.replace('/', '_')}_{variant}_s{seed}"


# =============================================================================== Phần B — USHER với C_s/C_t tuỳ ý
def usher_variant(Fs_rep, T, P_src, Fx_rep, tgt, lab_x, classes, seed, cache_mode, F_struct_src=None, F_struct_tgt=None):
    """Chép từ run05c.run_usher_paper (KHÔNG sửa run05c.py): bản gốc chỉ gọi SC.reset(mode, f_src) — 2 đối số, không
    truyền f_tgt nên không đủ cho MODE='custom' (mục 3, cần cả F_SRC và F_TGT). Giữ nguyên mọi tham số cấu hình mặc
    định giai đoạn 1 (U6, hidden_dim=None, USHER_ID_MODE=linear, λ_id=0.3)."""
    import torch, anndata as ad, run_usher as U, struct_cache as SC, random
    os.environ["USHER_SEED"] = str(seed); os.environ["USHER_EM_WORKERS"] = "4"
    os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = "0.3"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    C.logging.getLogger().setLevel(C.logging.WARNING)
    Pt = np.zeros((len(tgt), len(classes)), np.float32)
    Pt[np.arange(len(tgt)), [list(classes).index(c) for c in lab_x[tgt]]] = 1
    a = ad.AnnData(X=Fs_rep[T].astype(np.float32), obs=pd.DataFrame(index=[f"s{i}" for i in T]))
    b = ad.AnnData(X=Fx_rep[tgt].astype(np.float32), obs=pd.DataFrame({"ct": lab_x[tgt]}, index=[f"t{i}" for i in tgt]))
    a.obs["ct"] = "unknown"
    a.obsm["X_celltype_probs"] = P_src[T].astype(np.float32); b.obsm["X_celltype_probs"] = Pt
    info = {}
    SC.reset(cache_mode, F_struct_src, F_struct_tgt)
    t0 = time.time()
    model, _, _, T_full, mean, std = U.align_features_fgw(
        a, b, e_step_method="fgw", m_step_method="global", sampling_strategy="celltype",
        sketch_size=C.SKETCH, sketch_obsm_key="X_none", use_stratified_pairing=True, stratified_pairing_fix=True,
        cell_type_col="ct", celltype_probs_layer="X_celltype_probs", alpha=R6.U6["alpha"], gamma=R6.U6["gamma"],
        epsilon=0.1, knn_k=30, metric="cosine", m_step_metric="euclidean", lambda_cross=0.9, lambda_var=0.1,
        entropy_percentile=90.0, confidence_percentile=10.0, use_linear_assignment=True, steps_per_iter=100, lr=1e-3,
        n_iters=C.N_ITERS, hidden_dim=None, device="cuda", verbose=False)
    info["em_seconds"] = round(time.time() - t0, 1); info["cache_stats"] = dict(SC.STATS)
    SC.reset("off")
    del T_full; torch.cuda.empty_cache()
    with torch.no_grad():
        model.eval(); x = torch.from_numpy(Fs_rep.astype(np.float32)).cuda()
        y = model((x - mean) / std) * std + mean if mean is not None else model(x)
        out = y.cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return out, info


def get_pair_niche(pair, pctx, S, save):
    """PCA 50 niche (15 hàng xóm không gian, dữ liệu đầy đủ, gene chung riêng của pair) cho pair và TARGET — như 08,
    lưu cache theo pair (nhiều tổ hợp mặc định không dùng chung pair nên trong thực tế mỗi pair chỉ tính 1 lần)."""
    # BẮT BUỘC: K vào tên cache. Nếu không, đổi K_NICHE sẽ đọc lại đặc trưng K=15 cũ và
    # sweep hoàn toàn vô nghĩa. Hậu tố rỗng ở K=15 -> mọi cache/kết quả cũ vẫn hợp lệ.
    kt = "" if R08.K_NICHE == 15 else f"_k{R08.K_NICHE}"
    nx_f, ns_f = f"{OUT}/niche_{TARGET}_{pair}{kt}.npy", f"{OUT}/niche_{pair}{kt}.npy"
    if os.path.exists(nx_f) and os.path.exists(ns_f):
        return np.load(nx_f).astype(np.float32), np.load(ns_f).astype(np.float32), pctx.get("G_pca")
    _, _, G_pca = R8.gene_cols(pctx["varx"], pctx["vars_"])
    _, obs_x, _, _ = load_assay(TARGET); _, obs_s, _, _ = load_assay(pair)
    if SMOKE:
        obs_x = obs_x.iloc[B.pick(np.arange(len(obs_x)), R6.N_SMOKE)]; obs_s = obs_s.iloc[B.pick(np.arange(len(obs_s)), R6.N_SMOKE)]
    niche_x_cnt, d15_x, miss_x = R8.build_niche(TARGET, obs_x.index.astype(str), G_pca)
    niche_s_cnt, d15_s, miss_s = R8.build_niche(pair, obs_s.index.astype(str), G_pca)
    niche_x, niche_s = R8.niche_pca(niche_x_cnt, niche_s_cnt)
    niche_x, niche_s = niche_x.astype(np.float32), niche_s.astype(np.float32)
    np.save(nx_f, niche_x.astype(np.float16)); np.save(ns_f, niche_s.astype(np.float16))
    S.setdefault("niche_meta", {})[pair] = {"k": R08.K_NICHE,
        "d15_um": {TARGET: round(d15_x, 2), pair: round(d15_s, 2)}, "n_gene": len(G_pca)}
    save(); log(f"[{pair}] niche: {S['niche_meta'][pair]}")
    return niche_x, niche_s, G_pca


def get_niche_label_comp(pair, S, save):
    """Thành phần nhãn thô (6 nhóm) của 15 hàng xóm không gian, tính trên dữ liệu đầy đủ, ánh xạ về thứ tự lite (dùng
    cho niche_agree, mục 3). Không phụ thuộc biểu diễn/biến thể -> cache theo pair."""
    cx_f, cs_f = f"{OUT}/labcomp_{TARGET}.npy", f"{OUT}/labcomp_{pair}.npy"
    if os.path.exists(cx_f) and os.path.exists(cs_f):
        return np.load(cs_f).astype(np.float32), np.load(cx_f).astype(np.float32)

    def comp_for(assay):
        p = R._find(R.FULL, R.FULL_REL[assay])
        lc = R.LABEL_COL.get(assay, "annotation")   # bản đầy đủ: cột nhãn gốc tên "annotation" (không phải "label" như load_assay)
        Xf, obsf, varf, xy = R.read_h5(p, ("high_quality", lc))
        hq = obsf["high_quality"].astype(int).to_numpy() == 1
        obsf, xy = obsf[hq], xy[hq]
        full_ids = obsf.index.astype(str)
        lab_c_full = B.to_coarse(obsf[lc].to_numpy().astype(str))
        onehot = np.stack([(lab_c_full == c).astype(np.float32) for c in COARSE_GROUPS], axis=1)
        _, obs_lite, _, _ = load_assay(assay)
        lite_ids = obs_lite.index.astype(str)
        if SMOKE: lite_ids = lite_ids[B.pick(np.arange(len(lite_ids)), R6.N_SMOKE)]
        pos = pd.Index(full_ids).get_indexer(lite_ids)
        valid = np.where(pos >= 0)[0]; q = pos[valid]
        k = min(K_NICHE, len(xy) - 1)
        dist, idx = NearestNeighbors(n_neighbors=k + 1, n_jobs=4).fit(xy).kneighbors(xy[q])
        keep = idx != q[:, None]; keep[keep.sum(1) > k, -1] = False
        nb = idx[keep].reshape(len(q), k)
        comp = np.zeros((len(lite_ids), len(COARSE_GROUPS)), np.float32)
        comp[valid] = onehot[nb].mean(1)
        return comp

    comp_x, comp_s = comp_for(TARGET), comp_for(pair)
    np.save(cx_f, comp_x.astype(np.float16)); np.save(cs_f, comp_s.astype(np.float16))
    return comp_s, comp_x


def niche_agree(Fmap, Fx_rep, ref, comp_s_lite, comp_x_lite, H_idx, k=15, seed=0):
    """cosine(p_i, q̄_i) trung bình trên 10k tế bào truy vấn H (cùng tập nội suy, B.pick(H, N_Q)); p_i = thành phần
    hàng xóm thật; q̄_i = TB thành phần hàng xóm của k Xenium gần nhất trong không gian đã căn chỉnh (giao thức mới).
    Kèm sàn ngẫu nhiên (k Xenium ngẫu nhiên từ `ref`)."""
    Q = B.pick(H_idx, R6.N_Q)
    mu = Fx_rep.mean(0)
    Xr, Qs = _unit(Fx_rep[ref] - mu), _unit(Fmap[Q] - mu)
    nn = NearestNeighbors(n_neighbors=min(k, len(ref)), n_jobs=4).fit(Xr).kneighbors(Qs, return_distance=False)
    comp_ref = comp_x_lite[ref]
    qbar = comp_ref[nn].mean(1)
    p = comp_s_lite[Q]
    def cos_rows(a, b):
        an = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12); bn = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
        return (an * bn).sum(1)
    na = float(np.mean(cos_rows(p, qbar)))
    rng = np.random.RandomState(seed)
    rand_idx = rng.choice(len(ref), size=(len(Q), min(k, len(ref))), replace=True)
    na_r = float(np.mean(cos_rows(p, comp_ref[rand_idx].mean(1))))
    return round(na, 4), round(na_r, 4)


def process_combo_b(combo, D, F, PCTX, pca_cache, S, save, W):
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Sc = S["combos"].setdefault(combo, {"pair": pair, "rep": rep, "results": {}, "runs": {}})
    RES = Sc["results"]
    Fs_rep, Fx_rep = R16_rep_features(pair, rep, F, PCTX, pca_cache)
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    niche_x, niche_s, _ = get_pair_niche(pair, pctx, S, save)
    s_scale = float(np.mean(np.linalg.norm(Fx_rep, axis=1))) / float(np.mean(np.linalg.norm(niche_x, axis=1)))
    mix_x, mix_s = np.hstack([Fx_rep, s_scale * niche_x]), np.hstack([Fs_rep, s_scale * niche_s])
    STRUCT = {"default": (None, None), "C_niche": (niche_s, niche_x), "C_mix": (mix_s, mix_x)}
    comp_s, comp_x = get_niche_label_comp(pair, S, save)

    mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=K_AUX)
    lab_aux = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])

    def eval_all(name, Fmap):
        ev = R9.eval_both(pair, D, Fs_rep, Fx_rep, name, Fmap, pctx["S_shared"], pctx["IMP"], pctx["ref"])[1]
        mp = R11.metrics_pack(Fmap, Fx_rep, pctx["lab_src_ev"], pctx["lab_tgt_ev"], pctx["idx_src"], pctx["idx_tgt"])
        w = [f"{name}: {x}" for x in mp.pop("_warnings")]
        na, na_r = niche_agree(Fmap, Fx_rep, pctx["ref"], comp_s, comp_x, ds["H"])
        out = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "keep_struct_cos": ev["keep_struct_cos"],
               "sp": ev["imp"]["spearman_med"], "niche_agree": na, "niche_agree_rand": na_r}
        out["xfer_main"] = out[pctx["xfer_key"]]
        return out, w

    if "raw" not in RES:
        for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
            ev, w = eval_all(f"{combo}/{variant}", Fmap); W.extend(w); RES[variant] = ev
        save()

    run_idx_all = run_list()
    for variant in VARIANTS:
        runs = Sc["runs"].setdefault(variant, {})
        F_ss, F_tt = STRUCT[variant]
        for seed in SEEDS:
            i = run_idx_all.index((combo, variant, seed))
            if ONLY and str(i) not in ONLY.split(","): continue
            name = run_name(combo, variant, seed); f = f"{OUT}/{name}.npy"
            if RESUME and os.path.exists(f) and "eval" in runs.get(name, {}):
                step(f"{name}: đã có, bỏ qua"); continue
            if RESUME and os.path.exists(f) and "info" in runs.get(name, {}):
                Fmap = np.load(f).astype(np.float32); step(f"{name}: nạp lại map")
            else:
                step(f"USHER [{combo}] {variant} seed={seed}")
                F_struct_src = F_ss[ds["T"]] if F_ss is not None else None
                F_struct_tgt = F_tt[tgt] if F_tt is not None else None
                Fmap, info = usher_variant(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed,
                                           CACHE_MODE[variant], F_struct_src, F_struct_tgt)
                cs_ = info.pop("cache_stats", {})
                cc = S.setdefault("cache_stats", {}).setdefault(variant, {"target_computed": 0, "target_hits": 0,
                                                                          "source_computed": 0, "source_hits": 0})
                for k_ in cc: cc[k_] += cs_.get(k_, 0)
                np.save(f, Fmap.astype(np.float16)); runs[name] = {"seed": seed, "info": info}; save()
            ev, w = eval_all(name, Fmap); W.extend(w)
            runs[name] = {**runs[name], "eval": ev}; save()
            log(f"  >> {name}: EM={runs[name]['info'].get('em_seconds')}s xfer_main={ev['xfer_main']} tổng={ev['total']} "
                f"niche_agree={ev['niche_agree']} (sàn {ev['niche_agree_rand']})")
            del Fmap; gc.collect()

    exp = EXPECT_DEFAULT.get(combo)
    rl = [v["eval"] for v in Sc["runs"].get("default", {}).values() if "eval" in v]
    if exp and rl:
        mx = float(np.mean([e["xfer_main"] for e in rl])); mt = float(np.mean([e["total_v2"] for e in rl]))
        if abs(mx - exp["xfer"]) > 0.01: W.append(f"{combo} default xfer={mx:.4f} lệch khỏi 16 ({exp['xfer']}) > 0.01")
        if abs(mt - exp["total_v2"]) > 0.01: W.append(f"{combo} default tổng_v2={mt:.4f} lệch khỏi 16 ({exp['total_v2']}) > 0.01")


def R16_rep_features(pair, rep, F, PCTX, pca_cache):
    if rep == "pca":
        if pair not in pca_cache:
            pctx = PCTX[pair]
            Fx_pca, Fs_pca, _ = R8.pca_cell(pctx["Xx_lite"], pctx["varx"], pctx["Xs_lite"], pctx["vars_"])
            pca_cache[pair] = (Fs_pca, Fx_pca)
        return pca_cache[pair]
    return F[(pair, "scgpt")], F[(TARGET, "scgpt")]


# =============================================================================== Phần A — chẩn đoán đăng ký affine
def marker_scores(X, var, genes):
    sym = var["symbol"].str.upper()
    cols = np.where(sym.isin(genes).to_numpy())[0]
    if len(cols) == 0: return np.zeros(X.shape[0], np.float32)
    return np.asarray(X[:, cols].astype(np.float32).log1p().mean(axis=1)).ravel()

def raster_sum(xy, vals, origin, shape, cell_um=GRID_UM):
    ij = np.clip(np.floor((xy - origin) / cell_um).astype(int), 0, np.array(shape) - 1)
    flat = ij[:, 1] * shape[0] + ij[:, 0]; n = shape[0] * shape[1]
    return np.stack([np.bincount(flat, weights=vals[:, c], minlength=n).reshape(shape[1], shape[0])
                     for c in range(vals.shape[1])]).astype(np.float32)

def load_full_A(assay, epi_common, imm_common):
    p = R._find(R.FULL, R.FULL_REL[assay])
    lc = R.LABEL_COL.get(assay, "annotation")   # bản đầy đủ: cột nhãn gốc tên "annotation"
    X, obs, var, xy = R.read_h5(p, ("high_quality", lc))
    hq = obs["high_quality"].astype(int).to_numpy() == 1
    X, lab, xy = X[hq], obs[lc].to_numpy().astype(str)[hq], xy[hq]
    lab_c = B.to_coarse(lab)
    epi, imm = marker_scores(X, var, epi_common), marker_scores(X, var, imm_common)
    return xy.astype(np.float64), lab_c, epi, imm

def build_channels(xy, lab_c, epi, imm, origin, shape):
    onehot = np.stack([(lab_c == c).astype(np.float32) for c in COARSE_GROUPS], axis=1)
    label_chans = raster_sum(xy, onehot, origin, shape)
    dm = raster_sum(xy, np.stack([np.ones(len(xy), np.float32), epi, imm], axis=1), origin, shape)
    density = dm[0]
    epi_mean = np.divide(dm[1], density, out=np.zeros_like(density), where=density > 0)
    imm_mean = np.divide(dm[2], density, out=np.zeros_like(density), where=density > 0)
    marker_chans = np.stack([density, epi_mean, imm_mean])
    return {"label": label_chans, "marker": marker_chans}, density > 0

def zscore_tissue(chans, tissue):
    out = np.zeros_like(chans)
    for i in range(len(chans)):
        v = chans[i][tissue]
        if v.size and v.std() > 1e-8: out[i][tissue] = (chans[i][tissue] - v.mean()) / v.std()
    return out

def transform_xy(xy, angle_deg, flip, scale, centroid):
    p = xy - centroid
    if flip: p = p * np.array([-1.0, 1.0])
    th = np.radians(angle_deg)
    Rm = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    return (p @ Rm.T) * scale

def register_one(src_xy, src_lab_c, src_epi, src_imm, tgt_grids, tgt_tissue, tgt_origin, cell_um=GRID_UM):
    """Quét (góc, lật, tỉ lệ); với mỗi cấu hình: dựng lại lưới NHỎ NHẤT của source đã biến đổi rồi tương quan chéo FFT
    với lưới target (cố định) -> đỉnh = tịnh tiến tốt nhất + điểm. Trả kết quả riêng cho 'label' và 'marker'."""
    centroid = src_xy.mean(0)
    onehot = np.stack([(src_lab_c == c).astype(np.float32) for c in COARSE_GROUPS], axis=1)
    dmv = np.stack([np.ones(len(src_xy), np.float32), src_epi, src_imm], axis=1)
    tgt_z = {k: zscore_tissue(v, tgt_tissue) for k, v in tgt_grids.items()}
    out = {}
    for ctype in ("label", "marker"):
        cands = []
        for angle in ANGLES:
            for flip in FLIPS:
                for scale in SCALES:
                    txy = transform_xy(src_xy, angle, flip, scale, centroid)
                    s_origin = txy.min(0); s_shape = tuple((np.ceil((txy.max(0) - s_origin) / cell_um) + 1).astype(int))
                    if ctype == "label":
                        chans = raster_sum(txy, onehot, s_origin, s_shape)
                    else:
                        dm = raster_sum(txy, dmv, s_origin, s_shape); dens = dm[0]
                        em = np.divide(dm[1], dens, out=np.zeros_like(dens), where=dens > 0)
                        im = np.divide(dm[2], dens, out=np.zeros_like(dens), where=dens > 0)
                        chans = np.stack([dens, em, im])
                    src_z = zscore_tissue(chans, chans[0] > 0)
                    corr = None
                    for c in range(len(src_z)):
                        r = fftconvolve(tgt_z[ctype][c], src_z[c][::-1, ::-1], mode="full")
                        corr = r if corr is None else corr + r
                    py, px = np.unravel_index(np.argmax(corr), corr.shape)
                    dy, dx_ = py - (s_shape[1] - 1), px - (s_shape[0] - 1)   # dịch lưới source (hàng,cột) so với target
                    tx_um = tgt_origin[0] + dx_ * cell_um - s_origin[0]; ty_um = tgt_origin[1] + dy * cell_um - s_origin[1]
                    cands.append({"angle": angle, "flip": flip, "scale": scale, "score": float(corr[py, px]),
                                  "tx_um": round(float(tx_um), 1), "ty_um": round(float(ty_um), 1)})
        cands.sort(key=lambda c: -c["score"])
        top1 = cands[0]
        top2 = next((c for c in cands[1:] if abs(c["angle"] - top1["angle"]) >= 20 or c["flip"] != top1["flip"]), cands[-1])
        out[ctype] = {**top1, "score_top2": top2["score"], "ratio": round(top1["score"] / (top2["score"] + 1e-8), 3)}
    return out

def apply_transform(xy, res, centroid):
    txy = transform_xy(xy, res["angle"], res["flip"], res["scale"], centroid)
    return txy + np.array([res["tx_um"], res["ty_um"]])

def verify_registration(src_xy, src_lab_c, res, tgt_xy, tgt_lab_c, radius_um=30.0):
    centroid = src_xy.mean(0)
    txy = apply_transform(src_xy, res, centroid)
    nn = NearestNeighbors(n_neighbors=1, n_jobs=4).fit(tgt_xy)
    dist, idx = nn.kneighbors(txy)
    ok = dist[:, 0] <= radius_um
    overlap = float(ok.mean())
    same_label = float((src_lab_c[ok] == tgt_lab_c[idx[ok, 0]]).mean()) if ok.any() else None
    ps = pd.Series(src_lab_c).value_counts(normalize=True); px = pd.Series(tgt_lab_c).value_counts(normalize=True)
    chance = float(sum(ps.get(c, 0) * px.get(c, 0) for c in COARSE_GROUPS))
    return {"overlap": round(overlap, 4), "same_label": (round(same_label, 4) if same_label is not None else None),
            "chance": round(chance, 4)}

def process_pair_a(pair, S, save, W):
    step(f"Phần A [{pair}] đọc dữ liệu đầy đủ")
    _, _, varx_full, _ = R.read_h5(R._find(R.FULL, R.FULL_REL[TARGET]), ())
    _, _, vars_full, _ = R.read_h5(R._find(R.FULL, R.FULL_REL[pair]), ())
    common = set(varx_full["symbol"].str.upper()) & set(vars_full["symbol"].str.upper())
    epi_g = [g for g in EPI_MARKERS if g in common]; imm_g = [g for g in IMM_MARKERS if g in common]
    xy_x, labc_x, epi_x, imm_x = load_full_A(TARGET, epi_g, imm_g)
    xy_s, labc_s, epi_s, imm_s = load_full_A(pair, epi_g, imm_g)
    origin = xy_x.min(0); shape = tuple((np.ceil((xy_x.max(0) - origin) / GRID_UM) + 1).astype(int))
    tgt_grids, tgt_tissue = build_channels(xy_x, labc_x, epi_x, imm_x, origin, shape)
    res = register_one(xy_s, labc_s, epi_s, imm_s, tgt_grids, tgt_tissue, origin)
    dev_deg = min(abs(res["marker"]["angle"] - res["label"]["angle"]), 360 - abs(res["marker"]["angle"] - res["label"]["angle"]))
    dev_um = float(np.hypot(res["marker"]["tx_um"] - res["label"]["tx_um"], res["marker"]["ty_um"] - res["label"]["ty_um"]))
    ver = {t: verify_registration(xy_s, labc_s, res[t], xy_x, labc_x) for t in ("label", "marker")}
    row = {"pair": pair, "n_gene_marker": f"epi={len(epi_g)} imm={len(imm_g)}"}
    for t in ("label", "marker"):
        row.update({f"{t}_góc": res[t]["angle"], f"{t}_lật": res[t]["flip"], f"{t}_tỉ_lệ": res[t]["scale"],
                    f"{t}_tx_um": res[t]["tx_um"], f"{t}_ty_um": res[t]["ty_um"], f"{t}_top1/top2": res[t]["ratio"],
                    f"{t}_same_label": ver[t]["same_label"], f"{t}_chance": ver[t]["chance"], f"{t}_overlap": ver[t]["overlap"]})
    row["lệch_marker_vs_label_độ"] = round(dev_deg, 1); row["lệch_marker_vs_label_um"] = round(dev_um, 1)
    dang_ky = (ver["label"]["same_label"] is not None and ver["label"]["same_label"] >= 2 * ver["label"]["chance"]
               and ver["label"]["overlap"] >= 0.3 and res["label"]["ratio"] >= 1.1)
    khong_can_nhan = dang_ky and dev_deg <= 10 and dev_um <= 100
    row["đăng_ký_được"] = dang_ky; row["không_cần_nhãn"] = khong_can_nhan
    S.setdefault("registration", {})[pair] = row
    S.setdefault("registration_transform", {})[pair] = {"centroid_src": xy_s.mean(0).tolist(),
        "label": res["label"], "marker": res["marker"], "đăng_ký_được": dang_ky, "không_cần_nhãn": khong_can_nhan}
    save()
    log(f"[{pair}] Phần A: {row}")
    return row


# =============================================================================== main + summary.md
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER-ST mặc định giai đoạn 1 (U6 hidden_dim=None alpha={R6.U6['alpha']} gamma={R6.U6['gamma']} "
           f"aux=aux_cell k_aux={K_AUX} USHER_ID_MODE=linear USHER_LAMBDA_ID=0.3) sketch={C.SKETCH} n_iters={C.N_ITERS} "
           f"scib_k={R11.K_KNN} k_niche={K_NICHE} seeds={SEEDS} biến_thể={VARIANTS} combos={COMBOS} | "
           f"Phần A: lưới {GRID_UM}µm, góc bước {ANGLES[1] - ANGLES[0]}°, tỉ_lệ={SCALES}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("combos", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    if ONLY_PART != "B":
        for pair in PAIRS_A:
            if pair in S.get("registration", {}): step(f"Phần A [{pair}]: đã có, bỏ qua"); continue
            process_pair_a(pair, S, save, W)

    if ONLY_PART != "A":
        EMB = {"scgpt": M.find_dir("emb_scgpt")}
        rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
        REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
        needed_pairs = sorted({c.split("/")[0] for c in COMBOS})
        D, F, EXPR = R6.load_all(sorted(set(needed_pairs) | {TARGET}), EMB)
        PCTX = {}
        for pair in needed_pairs:
            ds, dx = D[pair], D[TARGET]
            S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
            use_coarse = len(S_shared) < 3
            cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
            coarse_groups = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]
            lab_src_ev = ds["lab_c"] if use_coarse else ds["lab"]; lab_tgt_ev = dx["lab_c"] if use_coarse else dx["lab"]
            groups_ev = coarse_groups if use_coarse else S_shared
            idx_src, idx_tgt = R11.build_eval_idx({**ds, "lab": lab_src_ev}, {**dx, "lab": lab_tgt_ev}, groups_ev)
            Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
            Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
            genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
            assert len(genes) == nG == Ys.shape[1]
            _, _, _, ref = C.mix_ref(pair, D)
            IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
            classes = np.array(sorted(set(dx["lab"])))
            PCTX[pair] = dict(ds=ds, dx=dx, S_shared=S_shared, xfer_key=("xfer_coarse" if use_coarse else "xfer_S"),
                              idx_src=idx_src, idx_tgt=idx_tgt, lab_src_ev=lab_src_ev, lab_tgt_ev=lab_tgt_ev,
                              IMP=IMP, ref=ref, classes=classes, Xs_lite=Xs_lite, vars_=vars_, Xx_lite=Xx_lite, varx=varx)
            log(f"[{pair}] S_chung={S_shared} (thô={coarse_groups})" + (" [DÙNG NHÃN/THƯỚC ĐO THÔ]" if use_coarse else ""))
        pca_cache = {}
        for combo in COMBOS:
            process_combo_b(combo, D, F, PCTX, pca_cache, S, save, W)

    if S.get("registration_transform"):   # dùng ở vòng 18 (aux không gian, plan mục 6)
        json.dump(S["registration_transform"], open(f"{OUT}/registration.json", "w"), indent=1)

    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 17")


TABLE_COLS_B = ["xfer_main", "ilisi", "kbet", "bras", "nmi", "ari", "asw", "total", "total_v2", "sp", "keep_struct_cos",
                "niche_agree", "niche_agree_rand"]
METHOD_LABEL_B = {"raw": "raw", "center": "center", "default": "default", "C_niche": "C_niche", "C_mix": "C_mix"}

def summary_md(S, W):
    L = ["# 17 — cấu trúc FGW theo vi môi trường (Phần B) + chẩn đoán đăng ký affine (Phần A)", S["config"], ""]
    L.append("## Phần B — bảng (TB±SD qua 3 seed cho default/C_niche/C_mix)")
    rows = []
    for combo in COMBOS:
        Sc = S["combos"].get(combo)
        if not Sc: continue
        for m in ("raw", "center"):
            v = Sc["results"].get(m)
            rows.append({"tổ_hợp": combo, "map": m, **({c: v.get(c) for c in TABLE_COLS_B} if v else {})})
        for variant in VARIANTS:
            rl = [x["eval"] for x in Sc["runs"].get(variant, {}).values() if "eval" in x]
            r = {"tổ_hợp": combo, "map": f"{variant}(n={len(rl)})"}
            for c in TABLE_COLS_B:
                m_, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m_:.3f}±{sd:.3f}" if m_ is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## S1 (C_niche/C_mix so với default: niche_agree vượt, tổng_v2 và chuyển nhãn không kém quá 0.02)")
    win_count = {"C_niche": 0, "C_mix": 0}
    for combo in COMBOS:
        Sc = S["combos"].get(combo)
        if not Sc: continue
        def ms_of(variant, key):
            rl = [x["eval"] for x in Sc["runs"].get(variant, {}).values() if "eval" in x]
            return _ms([e.get(key) for e in rl])
        na0, sna0 = ms_of("default", "niche_agree"); t0, _ = ms_of("default", "total_v2"); x0, _ = ms_of("default", "xfer_main")
        parts = []
        for variant in ("C_niche", "C_mix"):
            na, sna = ms_of(variant, "niche_agree"); t, _ = ms_of(variant, "total_v2"); x, _ = ms_of(variant, "xfer_main")
            r_na = beats(na, sna, na0, sna0) if na is not None and na0 is not None else None
            ok = bool(r_na == "vượt" and not_much_worse(t, t0) and not_much_worse(x, x0))
            if ok: win_count[variant] += 1
            parts.append(f"{variant}[Δniche={round(na - na0, 4) if (na is not None and na0 is not None) else None} -> {'ĐẠT' if ok else 'không'}]")
        L.append(f"- {combo}: " + " | ".join(parts))
    best = max(win_count, key=win_count.get) if any(win_count.values()) else None
    L.append(f"- S1 tổng: C_niche đạt {win_count['C_niche']}/{len(COMBOS)}, C_mix đạt {win_count['C_mix']}/{len(COMBOS)}"
            f" -> {'biến thể tốt nhất: ' + best if best else 'không biến thể nào đạt'}")

    L.append("## Phần A — chẩn đoán đăng ký affine (dữ liệu đầy đủ)")
    if S.get("registration"): L += ["```", pd.DataFrame(list(S["registration"].values())).to_string(index=False), "```"]
    dat = [r for r in S.get("registration", {}).values() if r.get("đăng_ký_được")]
    L.append(f"- S2: {len(dat)}/{len(PAIRS_A)} cặp đăng ký được ({[r['pair'] for r in dat]}); "
            f"không cần nhãn: {[r['pair'] for r in dat if r.get('không_cần_nhãn')]}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run18.py
"""18 — Giai đoạn 2 bước 2: thước đo vi môi trường mới (niche_bacc, có mốc điều kiện theo loại tế bào) + aux không
gian từ đăng ký lát cắt (CosMx, FFPE) + chấm lại map của vòng 17 bằng thước đo mới. Tái sử dụng run17.py (comp_for/
get_niche_label_comp, get_pair_niche, R16_rep_features, usher_variant, apply_transform, đọc registration.json) và
run16.py (USHER-ST mặc định, eval_all). `aux_spatial.py` (mới, giống struct_cache.py): trộn khoảng cách toạ độ đã
đăng ký vào M_aux của FGW.
"""
import os, sys, glob, json, gc, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run17 as R17, run16 as R16
R, B, M, C, R6, R7, R9, R11, R12, R8 = R17.R, R17.B, R17.M, R17.C, R17.R6, R17.R7, R17.R9, R17.R11, R17.R12, R17.R8
np, pd, sp, log, load_assay, _unit = R17.np, R17.pd, R17.sp, R17.log, R17.load_assay, R17._unit
_ms, beats, not_much_worse = R17._ms, R17.beats, R17.not_much_worse
SMOKE = R17.SMOKE   # xử lý biến môi trường SMOKE đã xảy ra khi import run17 -> run16 -> ... -> run06 (như 17)
NearestNeighbors = R17.NearestNeighbors
from sklearn.cluster import KMeans
from scipy.stats import mode as _mode

OUT = os.environ.get("OUT_DIR_18", "/kaggle/working/out18" + ("_smoke" if SMOKE else ""))
TARGET = R17.TARGET
COMBOS_R = R16.COMBOS_ALL   # 4 tổ hợp mặc định (Phần R: chấm lại map vòng 17)
PAIRS_S = ["cosmx6k/pca", "visiumhd_ffpe/pca"]   # Phần S: 2 cặp đăng ký được (RESEARCH_CONTEXT 5.14)
BETAS = {"default": 0.0, "spatial_b0.5": 0.5, "spatial_b1": 1.0}
SIGMA_SPATIAL = 100.0
K_NICHE_TYPES = 8
RESUME = os.environ.get("RESUME", "1") == "1"
ONLY = os.environ.get("ONLY_RUNS", "").replace(" ", "")
ONLY_PART = os.environ.get("ONLY_PART", "")   # "R" hoặc "S" để chạy riêng một phần; mặc định cả hai
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


def run_list():
    """Phần S (18 lần GPU, tốn thời gian nhất); Phần R chỉ CPU, luôn chạy hết mỗi lần gọi combo."""
    return [(combo, variant, seed) for seed in R17.SEEDS for combo in PAIRS_S for variant in BETAS]

def run_name(combo, variant, seed):
    return f"{combo.replace('/', '_')}_{variant}_s{seed}"


def mode_rows(a):
    # FIX 18c: scipy.stats.mode không nhận nhãn chuỗi (SciPy>=1.11) -> đếm phiếu bằng np.unique; hoà phiếu -> lớp nhỏ nhất (như scipy)
    a = np.asarray(a)
    classes, codes = np.unique(a.ravel(), return_inverse=True)
    codes = codes.reshape(a.shape)
    cnt = np.zeros((a.shape[0], len(classes)), np.int32)
    np.add.at(cnt, (np.repeat(np.arange(a.shape[0]), a.shape[1]), codes.ravel()), 1)
    return classes[cnt.argmax(1)]


# =============================================================================== thước đo vi môi trường mới (mục 3)
def niche_metrics(Fmap, Fx_rep, ds, dx, ref, comp_s, comp_x, kmeans, Q, k=15, seed=0):
    """niche_bacc (đa số phiếu 15 Xenium gần nhất trong không gian căn chỉnh) so với niche_bacc_type (mốc có điều
    kiện: 15 Xenium NGẪU NHIÊN cùng nhóm thô với nhãn thật) -> niche_gain = phần vượt ngoài loại tế bào. Giữ
    niche_agree (17) + niche_agree_mean (mốc "không biết vị trí": so với thành phần TB toàn Xenium)."""
    mu = Fx_rep.mean(0)
    Xr, Qs = _unit(Fx_rep[ref] - mu), _unit(Fmap[Q] - mu)
    nn = NearestNeighbors(n_neighbors=min(k, len(ref)), n_jobs=4).fit(Xr).kneighbors(Qs, return_distance=False)
    comp_ref = comp_x[ref]
    type_ref = kmeans.predict(comp_ref)
    type_true_q = kmeans.predict(comp_s[Q])
    pred = mode_rows(type_ref[nn])
    counts = pd.Series(type_true_q).value_counts()
    keep = counts[counts >= 50].index.to_numpy()
    niche_bacc = M.bacc_on(type_true_q, pred, keep)

    lab_c_true_q, lab_c_ref = ds["lab_c"][Q], dx["lab_c"][ref]
    rng = np.random.RandomState(seed)
    pred_cond = np.full(len(Q), -1, dtype=type_ref.dtype)
    for c in np.unique(lab_c_true_q):
        pool = np.where(lab_c_ref == c)[0]; m = lab_c_true_q == c
        if len(pool) == 0: continue
        picks = rng.choice(len(pool), size=(int(m.sum()), min(k, len(pool))), replace=True)
        pred_cond[m] = mode_rows(type_ref[pool][picks])
    niche_bacc_type = M.bacc_on(type_true_q, pred_cond, keep)
    niche_bacc_chance = round(1.0 / len(keep), 4) if len(keep) else None
    niche_gain = round(niche_bacc - niche_bacc_type, 4) if (niche_bacc is not None and niche_bacc_type is not None) else None

    def cos_rows(a, b):
        an = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12); bn = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
        return (an * bn).sum(1)
    qbar = comp_ref[nn].mean(1); p = comp_s[Q]
    niche_agree = round(float(np.mean(cos_rows(p, qbar))), 4)
    g_x = np.repeat(comp_x.mean(0, keepdims=True), len(Q), axis=0)
    niche_agree_mean = round(float(np.mean(cos_rows(p, g_x))), 4)
    return {"niche_bacc": niche_bacc, "niche_bacc_type": niche_bacc_type, "niche_bacc_chance": niche_bacc_chance,
            "niche_gain": niche_gain, "niche_agree": niche_agree, "niche_agree_mean": niche_agree_mean}


def setup_pairs(needed_pairs):
    EMB = {"scgpt": M.find_dir("emb_scgpt")}
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    REC = {k: v for k, v in json.load(open(rec[0])).items() if not k.startswith("_")}
    D, F, EXPR = R6.load_all(sorted(set(needed_pairs) | {TARGET}), EMB)
    PCTX = {}
    for pair in needed_pairs:
        ds, dx = D[pair], D[TARGET]
        S_shared = sorted(set(REC[pair]) & set(REC[TARGET]))
        use_coarse = len(S_shared) < 3
        cs = pd.Series(ds["lab_c"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab_c"]).value_counts()
        coarse_groups = [c for c in cs.index if c in cx.index and cs[c] >= 50 and cx[c] >= 50]
        lab_src_ev = ds["lab_c"] if use_coarse else ds["lab"]; lab_tgt_ev = dx["lab_c"] if use_coarse else dx["lab"]
        groups_ev = coarse_groups if use_coarse else S_shared
        idx_src, idx_tgt = R11.build_eval_idx({**ds, "lab": lab_src_ev}, {**dx, "lab": lab_tgt_ev}, groups_ev)
        Xs_lite, vars_ = EXPR[pair]; Xx_lite, varx = EXPR[TARGET]
        Ys, Yx, nG = C.shared_lognorm(Xs_lite, vars_, Xx_lite, varx)
        genes = sorted(set(vars_["symbol"].str.upper()) & set(varx["symbol"].str.upper()))
        assert len(genes) == nG == Ys.shape[1]
        _, _, _, ref = C.mix_ref(pair, D)
        IMP = R6.imp_setup(Ys, Yx, genes, B.pick(ds["H"], R6.N_Q), ref, ds["H"]); del Ys, Yx; gc.collect()
        classes = np.array(sorted(set(dx["lab"])))
        PCTX[pair] = dict(ds=ds, dx=dx, S_shared=S_shared, groups_ev=groups_ev, use_coarse=use_coarse,
                          xfer_key=("xfer_coarse" if use_coarse else "xfer_S"),
                          idx_src=idx_src, idx_tgt=idx_tgt, lab_src_ev=lab_src_ev, lab_tgt_ev=lab_tgt_ev,
                          IMP=IMP, ref=ref, classes=classes, Xs_lite=Xs_lite, vars_=vars_, Xx_lite=Xx_lite, varx=varx)
        log(f"[{pair}] S_chung={S_shared} (thô={coarse_groups})" + (" [DÙNG NHÃN/THƯỚC ĐO THÔ]" if use_coarse else ""))
    return D, F, PCTX


# =============================================================================== Phần R — chấm lại map vòng 17 (CPU)
def process_combo_r(combo, D, F, PCTX, pca_cache, kmeans, S, save, W, npy_files):
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    Sc = S["R"].setdefault(combo, {})
    for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
        if variant not in Sc:
            Sc[variant] = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q); save()
    for variant in R17.VARIANTS:
        for seed in R17.SEEDS:
            name = R17.run_name(combo, variant, seed); key = f"{variant}_s{seed}"
            if key in Sc: continue
            f = npy_files.get(name)
            if not f: W.append(f"Phần R {combo} {key}: thiếu out17/{name}.npy, bỏ qua"); continue
            Fmap = np.load(f).astype(np.float32)
            Sc[key] = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q)
            save(); log(f"  >> Phần R {combo} {key}: {Sc[key]}"); del Fmap; gc.collect()


# =============================================================================== Phần S — aux không gian (GPU)
def lite_xy_full(assay):
    """Toạ độ (µm) theo THỨ TỰ LITE (khớp D[assay]), lấy từ dữ liệu đầy đủ qua khớp id (như build_niche/comp_for)."""
    p = R._find(R.FULL, R.FULL_REL[assay])
    _, obsf, _, xy = R.read_h5(p, ("high_quality",))
    hq = obsf["high_quality"].astype(int).to_numpy() == 1
    obsf, xy = obsf[hq], xy[hq]
    full_ids = obsf.index.astype(str)
    _, obs_lite, _, _ = load_assay(assay)
    lite_ids = obs_lite.index.astype(str)
    if SMOKE: lite_ids = lite_ids[B.pick(np.arange(len(lite_ids)), R6.N_SMOKE)]
    pos = pd.Index(full_ids).get_indexer(lite_ids)
    valid = pos >= 0
    out = np.tile(xy.mean(0), (len(lite_ids), 1)).astype(np.float64)   # thiếu khớp (hiếm, <1%) -> toạ độ TB (không NaN)
    out[valid] = xy[pos[valid]]
    missing = int((~valid).sum())
    if missing / len(lite_ids) > 0.01: raise AssertionError(f"{assay}: thiếu khớp toạ độ quá 1% ({missing}/{len(lite_ids)})")
    if missing: log(f"CẢNH BÁO [{assay}] {missing}/{len(lite_ids)} tế bào lite không khớp toạ độ đầy đủ -> dùng toạ độ TB")
    return out

def spatial_knn_eval(xy_src_reg, xy_tgt, ds, dx, S_shared, ref, comp_s, comp_x, kmeans, Q, k=15):
    nn = NearestNeighbors(n_neighbors=k, n_jobs=4).fit(xy_tgt[ref]).kneighbors(xy_src_reg[Q], return_distance=False)
    pred_lab = mode_rows(dx["lab"][ref][nn])
    qm = np.isin(ds["lab"][Q], S_shared)
    xfer_S = M.bacc_on(ds["lab"][Q][qm], pred_lab[qm], S_shared) if qm.any() else None
    type_ref = kmeans.predict(comp_x[ref]); type_true_q = kmeans.predict(comp_s[Q])
    pred_type = mode_rows(type_ref[nn])
    counts = pd.Series(type_true_q).value_counts(); keep = counts[counts >= 50].index.to_numpy()
    niche_bacc = M.bacc_on(type_true_q, pred_type, keep)
    return xfer_S, niche_bacc

def process_combo_s(combo, D, F, PCTX, pca_cache, kmeans, REG, S, save, W, runs_wanted):
    import aux_spatial as ASP
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)

    reg = REG[pair]
    xy_src_reg = R17.apply_transform(lite_xy_full(pair), reg["marker"], np.array(reg["centroid_src"]))
    xy_tgt = lite_xy_full(TARGET)
    log(f"[{combo}] biến đổi marker đã dùng: {reg['marker']}")

    mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
    lab_aux = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])

    def eval_all(name, Fmap):
        ev = R9.eval_both(pair, D, Fs_rep, Fx_rep, name, Fmap, pctx["S_shared"], pctx["IMP"], pctx["ref"])[1]
        mp = R11.metrics_pack(Fmap, Fx_rep, pctx["lab_src_ev"], pctx["lab_tgt_ev"], pctx["idx_src"], pctx["idx_tgt"])
        w = [f"{name}: {x}" for x in mp.pop("_warnings")]
        nm = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q)
        out = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "sp": ev["imp"]["spearman_med"], **nm}
        out["xfer_main"] = out[pctx["xfer_key"]]
        return out, w

    Sc = S["S"].setdefault(combo, {"results": {}, "runs": {}})
    RES = Sc["results"]
    if "raw" not in RES:
        for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
            ev, w = eval_all(f"{combo}/{variant}", Fmap); W.extend(w); RES[variant] = ev
        save()
    if "spatial_knn" not in RES:
        xfer_knn, niche_knn = spatial_knn_eval(xy_src_reg, xy_tgt, ds, dx, pctx["S_shared"], pctx["ref"], comp_s, comp_x, kmeans, Q)
        RES["spatial_knn"] = {"xfer_S": xfer_knn, "xfer_main": xfer_knn, "niche_bacc": niche_knn}
        save()

    for variant, beta in BETAS.items():
        runs = Sc["runs"].setdefault(variant, {})
        for seed in R17.SEEDS:
            name = run_name(combo, variant, seed)
            if name not in runs_wanted: continue
            f = f"{OUT}/{name}.npy"
            if RESUME and os.path.exists(f) and "eval" in runs.get(name, {}):
                step(f"{name}: đã có, bỏ qua"); continue
            if RESUME and os.path.exists(f) and "info" in runs.get(name, {}):
                Fmap = np.load(f).astype(np.float32); step(f"{name}: nạp lại map")
            else:
                step(f"USHER [{combo}] {variant} (β={beta}) seed={seed}")
                ASP.reset(beta, SIGMA_SPATIAL, xy_src_reg[ds["T"]], xy_tgt[tgt])
                Fmap, info = R17.usher_variant(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed,
                                               "target", None, None)
                ASP.reset(0.0)
                info.pop("cache_stats", None)
                np.save(f, Fmap.astype(np.float16)); runs[name] = {"seed": seed, "info": info}; save()
            ev, w = eval_all(name, Fmap); W.extend(w)
            runs[name] = {**runs[name], "eval": ev}; save()
            log(f"  >> {name}: EM={runs[name]['info'].get('em_seconds')}s xfer_main={ev['xfer_main']} tổng_v2={ev['total_v2']} "
                f"niche_bacc={ev['niche_bacc']}")
            del Fmap; gc.collect()

    exp = R17.EXPECT_DEFAULT.get(combo)
    rl = [v["eval"] for v in Sc["runs"].get("default", {}).values() if "eval" in v]
    if exp and rl:
        mx = float(np.mean([e["xfer_main"] for e in rl])); mt = float(np.mean([e["total_v2"] for e in rl]))
        if abs(mx - exp["xfer"]) > 0.01: W.append(f"{combo} default(18) xfer={mx:.4f} lệch khỏi 16/17 ({exp['xfer']}) > 0.01")
        if abs(mt - exp["total_v2"]) > 0.01: W.append(f"{combo} default(18) tổng_v2={mt:.4f} lệch khỏi 16/17 ({exp['total_v2']}) > 0.01")


# =============================================================================== main
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    # FIX 18b: run17 ghi cache labcomp vào R17.OUT (= /kaggle/working/out17, không tồn tại trong phiên 18) -> chuyển về out18
    R17.OUT = OUT
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER-ST mặc định giai đoạn 1 (U6 hidden_dim=None aux=aux_cell USHER_ID_MODE=linear "
           f"USHER_LAMBDA_ID=0.3 struct_cache=target) K_niche_types={K_NICHE_TYPES} sigma_spatial={SIGMA_SPATIAL}µm "
           f"beta={list(BETAS.values())} combos_R={COMBOS_R} pairs_S={PAIRS_S} seeds={R17.SEEDS}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg; S.setdefault("R", {}); S.setdefault("S", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    needed_pairs = sorted({c.split("/")[0] for c in COMBOS_R} | {c.split("/")[0] for c in PAIRS_S})
    D, F, PCTX = setup_pairs(needed_pairs)
    pca_cache = {}

    comp_x0 = R17.get_niche_label_comp(needed_pairs[0], S, save)[1]
    kmeans = KMeans(n_clusters=K_NICHE_TYPES, random_state=0, n_init=10).fit(comp_x0)
    log(f"KMeans niche: inertia={kmeans.inertia_:.1f}, cỡ cụm={np.bincount(kmeans.labels_)}")

    if ONLY_PART != "S":
        npy_files = {os.path.splitext(os.path.basename(f))[0]: f
                    for f in glob.glob("/kaggle/input/**/out17/*.npy", recursive=True)}
        log(f"Phần R: tìm thấy {len(npy_files)} map .npy của vòng 17")
        for combo in COMBOS_R:
            process_combo_r(combo, D, F, PCTX, pca_cache, kmeans, S, save, W, npy_files)

    if ONLY_PART != "R":
        reg_files = sorted(glob.glob("/kaggle/input/**/out17/registration.json", recursive=True))
        assert reg_files, "thiếu out17/registration.json (output vòng 17) — cần Add Input notebook 17"
        REG = json.load(open(reg_files[0]))
        runs_all = run_list()
        for i, r in enumerate(runs_all): log(f"  run {i:2d}: {run_name(*r)}")
        runs_wanted = {run_name(*r) for i, r in enumerate(runs_all) if not ONLY or str(i) in ONLY.split(",")}
        for combo in PAIRS_S:
            process_combo_s(combo, D, F, PCTX, pca_cache, kmeans, REG, S, save, W, runs_wanted)

    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 18")


# =============================================================================== summary.md
TABLE_R_COLS = ["niche_bacc", "niche_bacc_type", "niche_bacc_chance", "niche_gain", "niche_agree", "niche_agree_mean"]
TABLE_S_COLS = ["xfer_main", "ilisi", "kbet", "bras", "nmi", "ari", "total", "total_v2", "sp"] + TABLE_R_COLS

def summary_md(S, W):
    L = ["# 18 — thước đo vi môi trường mới, chấm lại vòng 17 (Phần R) + aux không gian (Phần S)", S["config"], ""]

    L.append("## Phần R — chấm lại map vòng 17 (TB±SD qua 3 seed cho default/C_niche/C_mix)")
    rows = []
    for combo in COMBOS_R:
        Sc = S["R"].get(combo, {})
        for variant in ("raw", "center"):
            if variant in Sc: rows.append({"tổ_hợp": combo, "map": variant, **{c: Sc[variant].get(c) for c in TABLE_R_COLS}})
        for variant in R17.VARIANTS:
            rl = [Sc[f"{variant}_s{seed}"] for seed in R17.SEEDS if f"{variant}_s{seed}" in Sc]
            r = {"tổ_hợp": combo, "map": f"{variant}(n={len(rl)})"}
            for c in TABLE_R_COLS:
                m_, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m_:.3f}±{sd:.3f}" if m_ is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## T0 (mô tả) + T1 (vi môi trường vượt loại tế bào) + T2 (C theo niche, chấm lại 17)")
    t2_win = {"C_niche": 0, "C_mix": 0}
    for combo in COMBOS_R:
        Sc = S["R"].get(combo, {})
        rl_def = [Sc[f"default_s{s}"] for s in R17.SEEDS if f"default_s{s}" in Sc]
        b0, sb0 = _ms([e.get("niche_bacc") for e in rl_def])
        typ = Sc.get("raw", {}).get("niche_bacc_type"); ch = Sc.get("raw", {}).get("niche_bacc_chance")
        L.append(f"- T0 {combo}: niche_bacc raw={Sc.get('raw', {}).get('niche_bacc')} center={Sc.get('center', {}).get('niche_bacc')} "
                f"default={round(b0, 4) if b0 is not None else None} | niche_bacc_type={typ} niche_bacc_chance={ch}")
        r1 = beats(b0, sb0, typ, 0) if (b0 is not None and typ is not None) else None
        L.append(f"- T1 {combo}: niche_gain(default)={round(b0 - typ, 4) if (b0 is not None and typ is not None) else None} -> "
                f"{'ĐẠT' if (r1 or {}).get('pass') else 'không'}")
        for variant in ("C_niche", "C_mix"):
            rl_v = [Sc[f"{variant}_s{s}"] for s in R17.SEEDS if f"{variant}_s{s}" in Sc]
            bv, sbv = _ms([e.get("niche_bacc") for e in rl_v])
            ok = bool(bv is not None and b0 is not None and (beats(bv, sbv, b0, sb0) or {}).get("pass"))
            if ok: t2_win[variant] += 1
            L.append(f"  - T2 {combo}/{variant}: niche_bacc={round(bv, 4) if bv is not None else None} vs default={round(b0, 4) if b0 is not None else None} -> {'ĐẠT' if ok else 'không'}")
    L.append(f"- T2 tổng: C_niche đạt {t2_win['C_niche']}/{len(COMBOS_R)}, C_mix đạt {t2_win['C_mix']}/{len(COMBOS_R)}")

    L.append("## Phần S — aux không gian (TB±SD qua 3 seed cho spatial_b*/default)")
    rows = []
    for combo in PAIRS_S:
        Sc = S["S"].get(combo, {"results": {}, "runs": {}})
        for m in ("raw", "center", "spatial_knn"):
            v = Sc["results"].get(m)
            rows.append({"cặp": combo, "map": m, **({c: v.get(c) for c in TABLE_S_COLS if c in v} if v else {})})
        for variant in BETAS:
            rl = [x["eval"] for x in Sc["runs"].get(variant, {}).values() if "eval" in x]
            r = {"cặp": combo, "map": f"{variant}(n={len(rl)})"}
            for c in TABLE_S_COLS:
                m_, sd = _ms([e.get(c) for e in rl]); r[c] = f"{m_:.3f}±{sd:.3f}" if m_ is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    L.append("## T3 (aux không gian vượt default về niche_bacc, tổng_v2/chuyển nhãn không kém quá 0.02)")
    for combo in PAIRS_S:
        Sc = S["S"].get(combo, {"results": {}, "runs": {}})
        def ms_of(variant, key):
            rl = [x["eval"] for x in Sc["runs"].get(variant, {}).values() if "eval" in x]
            return _ms([e.get(key) for e in rl])
        b0, sb0 = ms_of("default", "niche_bacc"); t0v, _ = ms_of("default", "total_v2"); x0, _ = ms_of("default", "xfer_main")
        for variant in ("spatial_b0.5", "spatial_b1"):
            bv, sbv = ms_of(variant, "niche_bacc"); tv, _ = ms_of(variant, "total_v2"); xv, _ = ms_of(variant, "xfer_main")
            ok = bool(bv is not None and b0 is not None and (beats(bv, sbv, b0, sb0) or {}).get("pass")
                     and not_much_worse(tv, t0v) and not_much_worse(xv, x0))
            L.append(f"- {combo}/{variant}: niche_bacc={round(bv, 4) if bv is not None else None} vs default={round(b0, 4) if b0 is not None else None} -> {'ĐẠT' if ok else 'không'}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run19.py
"""19 — (A) mốc loại MỊN + trần giải mã niche (khép H12); (N) kênh niche gộp SAU căn chỉnh; (C1) nội suy gene đúng
nghĩa (gene bị che khỏi biểu diễn TRƯỚC khi dựng embedding, Xenium ← source). Tái sử dụng run18.py (niche_metrics,
KMeans niche, mode_rows, setup_pairs), run17.py (get_niche_label_comp, usher_variant, R16_rep_features), run16.py,
run08.pca_cell (mẫu cho PCA có che gene / chiếu dữ liệu đầy đủ), run06 (col_corr, rank_cols, avg_rows, knn_idx).

CHÚ Ý (bài học 17–18, CLAUDE.md): các hàm cache theo biến module-level `OUT` của run17/run18 (`get_niche_label_comp`,
`get_pair_niche`) phải được gán lại `R17.OUT = OUT` (OUT của CHÍNH vòng này) trước khi gọi, nếu không sẽ đọc/ghi
nhầm thư mục `out17`/`out18` cũ. `beats(...)` (định nghĩa gốc ở run13) trả **dict** `{"delta","thr","pass"}` — luôn
đọc `["pass"]`, KHÔNG so sánh với chuỗi "vượt"/"kém"/"ngang" (lỗi đã mắc ở vòng 17–18).
"""
import os, sys, glob, json, gc, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run18 as R18, run17 as R17
R, B, M, C, R6, R7, R9, R11, R12, R8 = R18.R, R18.B, R18.M, R18.C, R18.R6, R18.R7, R18.R9, R18.R11, R18.R12, R18.R8
np, pd, sp, log, load_assay, _unit = R18.np, R18.pd, R18.sp, R18.log, R18.load_assay, R18._unit
_ms, beats, not_much_worse = R18._ms, R18.beats, R18.not_much_worse
SMOKE = R18.SMOKE   # xử lý biến môi trường SMOKE đã xảy ra khi import run18 -> run17 -> ... -> run06
NearestNeighbors = R18.NearestNeighbors
KMeans = R18.KMeans
mode_rows = R18.mode_rows
niche_metrics = R18.niche_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon

OUT = os.environ.get("OUT_DIR_19", "/kaggle/working/out19" + ("_smoke" if SMOKE else ""))
TARGET = R18.TARGET
R17.OUT, R18.OUT = OUT, OUT   # bài học 17-18: hàm cache theo OUT của các module đó phải trỏ về OUT của vòng 19

COMBOS_A = R18.COMBOS_R   # 4 tổ hợp mặc định (như 17/18)
PAIRS_C_ALL = ["cosmx6k", "visiumhd_ffpe", "stereoseq"]
PAIRS_C = [p.strip() for p in os.environ.get("PAIRS_C", ",".join(PAIRS_C_ALL)).split(",") if p.strip()]
PAIRS_N = ["cosmx6k/pca", "visiumhd_ffpe/pca"]
PARTS = {p.strip() for p in os.environ.get("ONLY_PART", "A,N,C").split(",") if p.strip()}
if "all" in PARTS: PARTS = {"A", "N", "C"}
K_NICHE_TYPES = R18.K_NICHE_TYPES
RADII = [25, 50, 100]
N_FOLDS = 5
SEEDS_MAIN = [0] if SMOKE else [0, 1, 2]
SEED0_ONLY = [0]
RESUME = os.environ.get("RESUME", "1") == "1"
T0 = time.time()
def step(msg): log(f"    [{time.time() - T0:6.0f}s] {msg}")


# =============================================================================== hạ tầng dùng chung
def rep_features(pair, rep, F, PCTX, pca_cache):
    return R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)

def lite_obs(assay):
    """obs của bản lite theo ĐÚNG thứ tự tế bào mà setup_pairs dùng (SMOKE: cùng tập con B.pick(.., N_SMOKE) như run17/18)."""
    _, obs, _, _ = load_assay(assay)
    return obs.iloc[B.pick(np.arange(len(obs)), R6.N_SMOKE)] if SMOKE else obs

def load_map17(f, n_expected):
    """FIX 19b: map out17 luôn có đủ tế bào lite; ở SMOKE phải cắt về cùng tập con như dữ liệu SMOKE."""
    Fm = np.load(f).astype(np.float32)
    if Fm.shape[0] == n_expected: return Fm
    if SMOKE:
        Fm = Fm[B.pick(np.arange(len(Fm)), R6.N_SMOKE)]
        if Fm.shape[0] == n_expected: return Fm
    raise ValueError(f"{f}: {Fm.shape[0]} hàng, cần {n_expected} (map out17 không khớp số tế bào lite)")

def kept_classes(type_true_q, n_min=50):
    counts = pd.Series(type_true_q).value_counts()
    return counts[counts >= n_min].index.to_numpy()


# =============================================================================== Phần A — mốc loại mịn + trần niche
def niche_bacc_fine_pred(Fmap, Fx_rep, ds, dx, ref, comp_s, comp_x, kmeans, Q, keep, k=15, seed=0):
    mu = Fx_rep.mean(0)
    Xr, Qs = _unit(Fx_rep[ref] - mu), _unit(Fmap[Q] - mu)
    nn = NearestNeighbors(n_neighbors=k, n_jobs=4).fit(Xr).kneighbors(Qs, return_distance=False)
    type_ref = kmeans.predict(comp_x[ref])
    type_true_q = kmeans.predict(comp_s[Q])
    rng = np.random.RandomState(seed)

    lab_fine_true_q, lab_fine_ref, lab_c_ref = ds["lab"][Q], dx["lab"][ref], dx["lab_c"][ref]
    pred_fine = np.full(len(Q), -1, dtype=type_ref.dtype); n_fb = 0
    for c in np.unique(lab_fine_true_q):
        pool = np.where(lab_fine_ref == c)[0]; m = lab_fine_true_q == c
        if len(pool) == 0:
            pool = np.where(lab_c_ref == B.to_coarse([c])[0])[0]; n_fb += int(m.sum())
        if len(pool) == 0: continue
        picks = rng.choice(len(pool), size=(int(m.sum()), min(k, len(pool))), replace=True)
        pred_fine[m] = mode_rows(type_ref[pool][picks])
    niche_bacc_fine = M.bacc_on(type_true_q, pred_fine, keep)

    lab_fine_pred_q = mode_rows(lab_fine_ref[nn])   # nhãn mịn DỰ ĐOÁN = đa số phiếu nhãn của chính 15 hàng xóm dùng cho niche_bacc
    pred_pred = np.full(len(Q), -1, dtype=type_ref.dtype)
    for c in np.unique(lab_fine_pred_q):
        pool = np.where(lab_fine_ref == c)[0]; m = lab_fine_pred_q == c
        picks = rng.choice(len(pool), size=(int(m.sum()), min(k, len(pool))), replace=True)
        pred_pred[m] = mode_rows(type_ref[pool][picks])
    niche_bacc_pred = M.bacc_on(type_true_q, pred_pred, keep)
    return {"niche_bacc_fine": niche_bacc_fine, "niche_bacc_pred": niche_bacc_pred, "frac_fallback_fine": round(n_fb / len(Q), 4)}


def ceilings_for_combo(pctx, Fs_rep, Fx_rep, comp_s, comp_x, kmeans, Q, keep):
    ds, dx = pctx["ds"], pctx["dx"]
    mu_s, mu_x = Fs_rep.mean(0), Fx_rep.mean(0)
    type_true_q = kmeans.predict(comp_s[Q])

    ref_src = B.pick(ds["T"], 20000)
    type_ref_src = kmeans.predict(comp_s[ref_src])
    nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fs_rep[ref_src] - mu_s)).kneighbors(_unit(Fs_rep[Q] - mu_s), return_distance=False)
    ceil_src_knn = M.bacc_on(type_true_q, mode_rows(type_ref_src[nn]), keep)

    sc = StandardScaler().fit(Fs_rep[ds["T"]])
    clf = LogisticRegression(max_iter=2000, C=1.0, n_jobs=4).fit(sc.transform(Fs_rep[ds["T"]]), kmeans.predict(comp_s[ds["T"]]))
    ceil_src_lr = M.bacc_on(type_true_q, clf.predict(sc.transform(Fs_rep[Q])), keep)

    Qx, Rx = B.pick(dx["H"], R6.N_Q), B.pick(dx["T"], 20000)
    type_x_R, type_x_Q = kmeans.predict(comp_x[Rx]), kmeans.predict(comp_x[Qx])
    nn2 = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx_rep[Rx] - mu_x)).kneighbors(_unit(Fx_rep[Qx] - mu_x), return_distance=False)
    ceil_xen_knn = M.bacc_on(type_x_Q, mode_rows(type_x_R[nn2]), keep)
    return {"ceil_src_knn": ceil_src_knn, "ceil_src_lr": ceil_src_lr, "ceil_xen_knn": ceil_xen_knn}


def process_combo_a(combo, F, PCTX, pca_cache, kmeans, S, save, W, npy_files):
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = rep_features(pair, rep, F, PCTX, pca_cache)
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    keep = kept_classes(kmeans.predict(comp_s[Q]))
    Sc = S["A"].setdefault(combo, {})

    for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
        if variant in Sc: continue
        base = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q)
        extra = niche_bacc_fine_pred(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q, keep)
        Sc[variant] = {**base, **extra}; save()

    for seed in R17.SEEDS:
        key = f"default_s{seed}"
        if key in Sc: continue
        name = R17.run_name(combo, "default", seed); f = npy_files.get(name)
        if not f: W.append(f"Phần A {combo} {key}: thiếu out17/{name}.npy, bỏ qua"); continue
        Fmap = load_map17(f, len(Fs_rep))
        base = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q)
        extra = niche_bacc_fine_pred(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q, keep)
        Sc[key] = {**base, **extra}; save(); log(f"  >> Phần A {combo} {key}: {Sc[key]}"); del Fmap; gc.collect()

    if "ceil" not in Sc:
        Sc["ceil"] = ceilings_for_combo(pctx, Fs_rep, Fx_rep, comp_s, comp_x, kmeans, Q, keep)
        save(); log(f"[{combo}] trần: {Sc['ceil']}")


# =============================================================================== Phần C1 — nội suy gene đúng nghĩa
def scored_genes(pair, PCTX):
    pctx = PCTX[pair]
    ix, is_, G = R8.gene_cols(pctx["varx"], pctx["vars_"])
    Ys, Yx, nG = C.shared_lognorm(pctx["Xs_lite"], pctx["vars_"], pctx["Xx_lite"], pctx["varx"])
    assert nG == len(G)
    det_s = np.asarray((Ys > 0).mean(0)).ravel(); det_x = np.asarray((Yx > 0).mean(0)).ravel()
    sel = np.where((det_s >= 0.05) & (det_x >= 0.05))[0]   # vị trí gene chấm trong danh sách gene chung G
    # FIX (Claude chat): trả cả cột của MỌI gene chung (ix: Xenium, is_: source) để che fold khỏi toàn bộ gene chung
    return np.array(G)[sel], sel, ix, is_, Ys[:, sel].tocsr(), Yx[:, sel].tocsr()

def pca_cell_masked(Xx_lite, Xs_lite, ix, is_, keep_mask):
    Xj = sp.vstack([Xx_lite[:, ix[keep_mask]], Xs_lite[:, is_[keep_mask]]]).tocsr()
    P, _ = R.pca_expr(Xj)
    nx = Xx_lite.shape[0]
    return P[:nx].astype(np.float32), P[nx:].astype(np.float32)

def bin200_pearson(pred_allH, truth_allH, xy_H, min_cell=5):
    ij = np.floor(xy_H / 200.0).astype(np.int64)
    key = ij[:, 0].astype(np.int64) * 1_000_000 + ij[:, 1]
    uniq, inv = np.unique(key, return_inverse=True)
    cnt = np.bincount(inv, minlength=len(uniq))
    keep_bins = np.where(cnt >= min_cell)[0]
    if len(keep_bins) < 3: return np.full(pred_allH.shape[1], np.nan, np.float32)
    def bin_mean(A):
        out = np.zeros((len(uniq), A.shape[1]), np.float64)
        np.add.at(out, inv, A)
        return out / cnt[:, None]
    Pb, Tb = bin_mean(pred_allH)[keep_bins], bin_mean(truth_allH)[keep_bins]
    return R6.col_corr(Pb, Tb)

def run_tangram(Xs_lite, Xx_lite, train_cols_s, train_cols_x, genes, H_sorted, W):
    """tangram-sc mode='cells' (plan mục 4): 20k nguồn -> 10k truy vấn Xenium, gene huấn luyện = gene chấm trừ fold,
    giao với 2000 HVG của fold. Trả ma trận dự đoán (len(H_sorted) x len(genes)) hoặc None nếu cài/chạy lỗi."""
    try:
        import tangram as tg, anndata as ad
    except Exception as e:
        W.append(f"tangram không cài/import được, bỏ qua: {e!r}"); return None
    try:
        src_pool = B.pick(np.arange(Xs_lite.shape[0]), 20000)
        a_s = ad.AnnData(X=Xs_lite[src_pool][:, train_cols_s].toarray())
        a_x = ad.AnnData(X=Xx_lite[H_sorted][:, train_cols_x].toarray())
        a_s.var_names = a_x.var_names = [f"g{i}" for i in range(len(train_cols_s))]
        tg.pp_adatas(a_s, a_x, genes=list(a_s.var_names))
        ad_map = tg.map_cells_to_space(a_s, a_x, mode="cells", device="cuda:0", num_epochs=500, verbose=False)
        a_s_full = ad.AnnData(X=Xs_lite[src_pool][:, genes].toarray()); a_s_full.var_names = [f"t{i}" for i in range(len(genes))]
        pred_ad = tg.project_genes(ad_map, a_s_full)
        return np.asarray(pred_ad.X)
    except Exception as e:
        W.append(f"tangram lỗi khi chạy, bỏ qua: {e!r}"); return None

def process_pair_c(pair, D, F, PCTX, S, save, W):
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Xs_lite, Xx_lite = pctx["Xs_lite"], pctx["Xx_lite"]
    Gs, sel, ix_all, is_all, Ys, Yx = scored_genes(pair, PCTX)   # ix_all = cột XENIUM, is_all = cột SOURCE
    n = len(Gs)
    log(f"[{pair}] Phần C1: {n} gene chấm")
    rng0 = np.random.RandomState(0)
    folds = np.array_split(rng0.permutation(n), N_FOLDS)

    Q = B.pick(dx["H"], R6.N_Q); H = dx["H"]
    Q_in_H = np.searchsorted(np.sort(H), Q)   # vị trí của Q trong H đã sort — dùng H đã sort làm truy vấn chung
    H_sorted = np.sort(H)
    obs_x = lite_obs(TARGET)
    xy_H = obs_x[["x", "y"]].to_numpy()[H_sorted]

    METHODS = ("shuffle", "raw", "center", "default", "tangram", "xen_own")
    ALL_SPEAR, ALL_PEAR, ALL_BIN = ({m: np.full((n,), np.nan) for m in METHODS} for _ in range(3))
    fold0_seed_spear = {}

    Sc = S["C"].setdefault(pair, {"n_genes": int(n), "results": {}})
    for fi, fold_idx in enumerate(folds):
        keep_all = np.ones(len(ix_all), bool); keep_all[sel[fold_idx]] = False   # mọi gene chung TRỪ gene của fold
        Fx_rep, Fs_rep = pca_cell_masked(Xx_lite, Xs_lite, ix_all, is_all, keep_all)
        center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
        mu = Fx_rep.mean(0)
        P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
        lab_aux = pctx["classes"][P_aux.argmax(1)]
        tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])

        truth_allH = Yx[H_sorted][:, fold_idx].toarray()

        def impute(space_src):
            nb = R6.knn_idx(Fx_rep[H_sorted] - mu, space_src - mu, k=15)
            return R6.avg_rows(nb, Ys[:, fold_idx])

        pred = {"shuffle": R6.avg_rows(rng0.randint(0, Ys.shape[0], (len(H_sorted), 15)), Ys[:, fold_idx]),
                "raw": impute(Fs_rep), "center": impute(center_map)}

        seeds_here = SEEDS_MAIN if fi == 0 else SEED0_ONLY
        pred_default_seed0 = None
        for seed in seeds_here:
            name = f"{pair}_f{fi}_s{seed}.npy"; fpath = f"{OUT}/{name}"
            if RESUME and os.path.exists(fpath):
                Fmap = np.load(fpath).astype(np.float32); step(f"C1 {name}: nạp lại map")
            else:
                step(f"USHER C1 [{pair}] fold={fi} seed={seed}")
                Fmap, info = R17.usher_variant(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed, "target", None, None)
                info.pop("cache_stats", None); np.save(fpath, Fmap.astype(np.float16))
            pred_d = impute(Fmap)
            if seed == 0: pred_default_seed0 = pred_d
            if fi == 0:
                sp_g = R6.col_corr(R6.rank_cols(pred_d[Q_in_H]), R6.rank_cols(truth_allH[Q_in_H]))
                fold0_seed_spear.setdefault(seed, float(np.nanmedian(sp_g)))
            del Fmap; gc.collect()
        pred["default"] = pred_default_seed0

        # xen_own (trần): Xenium làm cả truy vấn lẫn tham chiếu, PCA chỉ trên Xenium, gene chung trừ fold f
        Fx_own, _ = R.pca_expr(Xx_lite[:, ix_all[keep_all]])
        nb_own = R6.knn_idx(Fx_own[H_sorted] - Fx_own.mean(0), Fx_own[dx["T"]] - Fx_own.mean(0), k=15)
        pred["xen_own"] = R6.avg_rows(nb_own, Yx[dx["T"]][:, fold_idx])

        # tangram (best-effort; bỏ nếu lỗi, plan mục 4): gene huấn luyện = gene chấm trừ fold, giao 2000 HVG của fold
        # FIX (Claude chat): bản trước tráo cột Xenium/source (ix_s dùng cho ma trận source) -> tangram học trên gene sai
        hvg_local = R.hvg(R.lognorm(Xs_lite[:, is_all[keep_all]]), min(2000, int(keep_all.sum())))
        train_cols_s, train_cols_x = is_all[keep_all][hvg_local], ix_all[keep_all][hvg_local]
        pred["tangram"] = run_tangram(Xs_lite, Xx_lite, train_cols_s, train_cols_x, is_all[sel[fold_idx]], H_sorted, W)

        for m, p_ in pred.items():
            if p_ is None: continue
            sp_ = R6.col_corr(R6.rank_cols(p_[Q_in_H]), R6.rank_cols(truth_allH[Q_in_H]))
            pe_ = R6.col_corr(p_[Q_in_H], truth_allH[Q_in_H])
            bn_ = bin200_pearson(p_, truth_allH, xy_H)
            ALL_SPEAR[m][fold_idx] = sp_; ALL_PEAR[m][fold_idx] = pe_; ALL_BIN[m][fold_idx] = bn_
        Sc["results"][f"fold{fi}"] = {"n_gene": int(len(fold_idx))}
        save(); log(f"[{pair}] fold {fi} xong ({len(fold_idx)} gene)")

    def summ(m):
        return {"spearman_med": round(float(np.nanmedian(ALL_SPEAR[m])), 4), "pearson_med": round(float(np.nanmedian(ALL_PEAR[m])), 4),
                "pearson_bin200_med": round(float(np.nanmedian(ALL_BIN[m])), 4), "n_gene": int(np.sum(~np.isnan(ALL_SPEAR[m])))}
    Sc["agg"] = {m: summ(m) for m in METHODS if np.any(~np.isnan(ALL_SPEAR[m]))}
    Sc["fold0_seed_spearman"] = fold0_seed_spear
    m_def, sd_def = _ms(list(fold0_seed_spear.values()))
    Sc["fold0_seed_sd"] = sd_def
    dv, dc = ALL_SPEAR["default"], ALL_SPEAR["center"]
    both = ~np.isnan(dv) & ~np.isnan(dc)
    if both.sum() >= 5:
        try: wp = float(wilcoxon(dv[both], dc[both]).pvalue)
        except Exception: wp = None
        Sc["default_vs_center"] = {"n_better": int((dv[both] > dc[both]).sum()), "n": int(both.sum()), "wilcoxon_p": wp}
    dv2, dt2 = ALL_SPEAR["default"], ALL_SPEAR.get("tangram")
    if dt2 is not None:
        both2 = ~np.isnan(dv2) & ~np.isnan(dt2)
        if both2.sum() >= 5:
            try: wp2 = float(wilcoxon(dv2[both2], dt2[both2]).pvalue)
            except Exception: wp2 = None
            Sc["default_vs_tangram"] = {"n_better": int((dv2[both2] > dt2[both2]).sum()), "n": int(both2.sum()), "wilcoxon_p": wp2}
    save()


# =============================================================================== Phần N — kênh niche gộp sau căn chỉnh
def recover_wb(Fs_rep, Fmap):
    X_aug = np.hstack([Fs_rep, np.ones((len(Fs_rep), 1), np.float32)])
    coef, *_ = np.linalg.lstsq(X_aug, Fmap, rcond=None)
    W_, b_ = coef[:-1], coef[-1]
    pred = X_aug @ coef
    ss_res = float(((Fmap - pred) ** 2).sum()); ss_tot = float(((Fmap - Fmap.mean(0)) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else None
    return W_, b_, r2

def pca_fit(Xj):
    Y = R.lognorm(Xj); g = R.hvg(Y, R.N_HVG)
    Z = Y[:, g].toarray()
    mean = Z.mean(0); sd = Z.std(0); sd[sd == 0] = 1
    Zs = np.clip((Z - mean) / sd, None, 10)
    n_pc = min(R.N_PC, Zs.shape[1] - 1, Zs.shape[0] - 1)
    pca = PCA(n_pc, svd_solver="randomized", random_state=R.SEED).fit(Zs)
    return pca.transform(Zs).astype(np.float32), {"g": g, "mean": mean, "sd": sd, "pca": pca}

def pca_apply(Xj, tr, chunk=100000):
    out = []
    for st in range(0, Xj.shape[0], chunk):   # FIX: theo khối, tránh OOM (lognorm theo hàng nên chia khối không đổi kết quả)
        Y = R.lognorm(Xj[st:st + chunk])
        Z = Y[:, tr["g"]].toarray()
        Zs = np.clip((Z - tr["mean"]) / tr["sd"], None, 10)
        out.append(tr["pca"].transform(Zs).astype(np.float32))
    return np.vstack(out)

def full_expr_matched(assay, G):
    p = R._find(R.FULL, R.FULL_REL[assay])
    Xf, obsf, varf, xy = R.read_h5(p, ("high_quality",))
    hq = obsf["high_quality"].astype(int).to_numpy() == 1
    Xf, obsf, xy = Xf[hq], obsf[hq], xy[hq]
    sym = varf["symbol"].str.upper()
    fmap = pd.Series(np.arange(len(sym)), index=sym.values)[~sym.duplicated().values]
    cols = fmap.reindex(G)
    assert cols.notna().all(), f"{assay}: thiếu {int(cols.isna().sum())} gene khi chiếu dữ liệu đầy đủ"
    return Xf[:, cols.to_numpy(dtype=int)].tocsr(), obsf.index.astype(str), xy.astype(np.float64)

def radius_adj(xy_full, rows, r):
    """FIX (Claude chat): ma trận kề chuẩn hoá hàng (len(rows) x n_full) — TB các tế bào high_quality trong bán kính r µm
    quanh tế bào rows[i] (chỉ số trong bản đầy đủ), KHÔNG tính chính nó; không có hàng xóm -> 1 hàng xóm gần nhất.
    Chỉ tính cho các hàng cần chấm (tế bào lite) và dựng MỘT lần, dùng lại cho mọi biến thể: z = A @ emb.
    (Bản trước tính cho mọi tế bào đầy đủ bằng vòng Python và lặp lại cho từng biến thể: ~30 lần x 650k tế bào.)"""
    nn = NearestNeighbors(radius=r, n_jobs=4).fit(xy_full)
    idx_list = nn.radius_neighbors(xy_full[rows], return_distance=False)
    knn2 = nn.kneighbors(xy_full[rows], n_neighbors=2, return_distance=False)
    indptr = [0]; indices = []; n_fb = 0
    for i, idx in enumerate(idx_list):
        idx = idx[idx != rows[i]]
        if len(idx) == 0:
            idx = knn2[i][knn2[i] != rows[i]][:1]; n_fb += 1
        indices.append(idx); indptr.append(indptr[-1] + len(idx))
    indptr = np.asarray(indptr, np.int64); deg = np.diff(indptr)
    A = sp.csr_matrix((np.repeat(1.0 / deg, deg).astype(np.float32), np.concatenate(indices).astype(np.int64), indptr),
                      shape=(len(rows), len(xy_full)))
    return A, round(n_fb / len(rows), 4), float(np.median(deg))

def common_clusters(sclu_s, sclu_x):
    """Tên spatial_cluster chung giữa source và Xenium; None nếu < 3 tên chung HOẶC chỉ là số hiệu cụm (số hiệu cụm của
    hai assay được đánh độc lập -> trùng số không có nghĩa là cùng vùng mô)."""
    if sclu_s is None or sclu_x is None: return None, "thiếu cột spatial_cluster"
    cc = sorted(set(map(str, sclu_s[sclu_s != None])) & set(map(str, sclu_x[sclu_x != None])))   # noqa: E711
    if len(cc) < 3: return None, f"chỉ {len(cc)} tên chung"
    if all(c.strip().lstrip("-").replace(".", "", 1).isdigit() for c in cc): return None, "chỉ là số hiệu cụm, không có tên"
    return cc, "ok"

def read_spatial_cluster(assay):
    """spatial_cluster (obs của bản lite) theo thứ tự LITE, hoặc None nếu cột không tồn tại (mục 4b, domain_bacc)."""
    p = R._find(R.LITE, f"{assay}_sub.h5ad")
    if not p: return None
    _, obsf, _, _ = R.read_h5(p, ("spatial_cluster",))
    if "spatial_cluster" not in obsf.columns: return None
    obsf.index = obsf.index.astype(str)
    obs_lite = lite_obs(assay)
    pos = pd.Index(obsf.index).get_indexer(obs_lite.index.astype(str))
    out = np.full(len(obs_lite), None, dtype=object)
    valid = pos >= 0; out[valid] = obsf["spatial_cluster"].to_numpy()[pos[valid]]
    return out

def process_pair_n(combo, D, F, PCTX, pca_cache, kmeans, S, save, W):
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = rep_features(pair, rep, F, PCTX, pca_cache)
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)

    P_lite, tr = pca_fit(sp.vstack([pctx["Xx_lite"][:, R8.gene_cols(pctx["varx"], pctx["vars_"])[0]],
                                    pctx["Xs_lite"][:, R8.gene_cols(pctx["varx"], pctx["vars_"])[1]]]).tocsr())
    nx = pctx["Xx_lite"].shape[0]
    Fx_check, Fs_check = P_lite[:nx], P_lite[nx:]
    corr_x = np.nanmean(R6.col_corr(Fx_check, Fx_rep)); corr_s = np.nanmean(R6.col_corr(Fs_check, Fs_rep))
    log(f"[{combo}] Phần N: khớp tái tạo PCA lite: Xenium r={corr_x:.4f} source r={corr_s:.4f}")
    ok_refit = bool(corr_x >= 0.999 and corr_s >= 0.999)
    if not ok_refit: W.append(f"{combo}: tái tạo PCA lite lệch (r={corr_x:.4f}/{corr_s:.4f} < 0.999)")

    G = R8.gene_cols(pctx["varx"], pctx["vars_"])[2]
    Xx_full, ids_x_full, xy_x_full = full_expr_matched(TARGET, G)
    Xs_full, ids_s_full, xy_s_full = full_expr_matched(pair, G)
    Fx_full, Fs_full_id = pca_apply(Xx_full, tr), pca_apply(Xs_full, tr)
    del Xx_full, Xs_full; gc.collect()
    # FIX (Claude chat): đưa phép chiếu về ĐÚNG hệ toạ độ của Fx_rep/Fs_rep (hệ mà map vòng 17 dùng) bằng lstsq trên lite —
    # an toàn cả khi tái tạo PCA lệch dấu/xoay; nếu tái tạo khớp thì đây là phép đồng nhất.
    def _align(A_chk, A_ref, A_full):
        X_ = np.hstack([A_chk, np.ones((len(A_chk), 1), np.float32)])
        coef, *_ = np.linalg.lstsq(X_, A_ref, rcond=None)
        r2 = 1 - float(((A_ref - X_ @ coef) ** 2).sum()) / float(((A_ref - A_ref.mean(0)) ** 2).sum())
        return (np.hstack([A_full, np.ones((len(A_full), 1), np.float32)]) @ coef).astype(np.float32), round(r2, 5)
    Fx_full, r2_ax = _align(Fx_check, Fx_rep, Fx_full); Fs_full_id, r2_as = _align(Fs_check, Fs_rep, Fs_full_id)
    log(f"[{combo}] căn hệ toạ độ PCA đầy đủ -> gốc: R² Xenium={r2_ax} source={r2_as}")
    if min(r2_ax, r2_as) < 0.99: W.append(f"{combo}: căn hệ toạ độ PCA R² thấp ({r2_ax}/{r2_as}) — Phần N không tin được")

    obs_x_lite, obs_s_lite = lite_obs(TARGET), lite_obs(pair)
    pos_x = pd.Index(ids_x_full).get_indexer(obs_x_lite.index.astype(str))
    pos_s = pd.Index(ids_s_full).get_indexer(obs_s_lite.index.astype(str))
    n_un = int((pos_s < 0).sum()), int((pos_x < 0).sum())
    if sum(n_un): W.append(f"{combo}: {n_un} tế bào lite (source, Xenium) không khớp bản đầy đủ -> dùng tạm hàng 0")
    pos_s_c, pos_x_c = np.clip(pos_s, 0, None), np.clip(pos_x, 0, None)
    ADJ = {}
    for r in RADII:
        As, fb_s, md_s = radius_adj(xy_s_full, pos_s_c, r); Ax, fb_x, md_x = radius_adj(xy_x_full, pos_x_c, r)
        ADJ[r] = (As, Ax)
        log(f"[{combo}] r={r}µm: trung vị số hàng xóm source={md_s} Xenium={md_x}; tỉ lệ lùi 1-NN source={fb_s} Xenium={fb_x}")
        S.setdefault("N_meta", {}).setdefault(combo, {})[f"r{r}"] = {"median_nb_src": md_s, "median_nb_xen": md_x, "fallback_src": fb_s, "fallback_xen": fb_x}

    sclu_s, sclu_x = read_spatial_cluster(pair), read_spatial_cluster(TARGET)
    if sclu_s is not None and sclu_x is not None:
        log(f"[{combo}] spatial_cluster: source={sorted(set(map(str, sclu_s[sclu_s != None])))} xenium={sorted(set(map(str, sclu_x[sclu_x != None])))}")   # noqa: E711
    common_clu, why = common_clusters(sclu_s, sclu_x)
    if common_clu is None: W.append(f"{combo}: bỏ domain_bacc ({why})")

    def build_and_score(label, Fs_full_variant):
        """id/niche_r/concat_r (plan mục 4b) cho MỘT biến thể nguồn (raw = Fs_full_id, center = +hằng số dịch chuyển
        base_center, default_s{seed} = W·x+b) — Xenium (Fx_full) không đổi qua các biến thể."""
        Sc = S["N"].setdefault(combo, {})
        entry = Sc.setdefault(label, {})
        reps_N = {"id": (Fs_full_variant[pos_s_c], Fx_full[pos_x_c])}
        for r in RADII:
            As, Ax = ADJ[r]
            z_s, z_x = np.asarray(As @ Fs_full_variant, np.float32), np.asarray(Ax @ Fx_full, np.float32)   # đã theo thứ tự lite
            reps_N[f"niche_{r}"] = (z_s, z_x)
            mu_id_x, mu_z_x = Fx_full[pos_x_c].mean(0), z_x.mean(0)
            id_s, id_x = _unit(Fs_full_variant[pos_s_c] - mu_id_x), _unit(Fx_full[pos_x_c] - mu_id_x)
            nz_s, nz_x = _unit(z_s - mu_z_x), _unit(z_x - mu_z_x)
            reps_N[f"concat_{r}"] = (np.hstack([id_s, nz_s]), np.hstack([id_x, nz_x]))
        for name, (Fs_v, Fx_v) in reps_N.items():
            if name in entry: continue
            nm = niche_metrics(Fs_v, Fx_v, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q)
            nn_x = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx_v[pctx["ref"]] - Fx_v.mean(0)))
            nn_idx = nn_x.kneighbors(_unit(Fs_v[Q] - Fx_v.mean(0)), return_distance=False)
            xfer_S = M.bacc_on(ds["lab"][Q], mode_rows(dx["lab"][pctx["ref"]][nn_idx]), pctx["S_shared"])
            domain_bacc = (M.bacc_on(sclu_s[Q].astype(str), mode_rows(sclu_x[pctx["ref"]][nn_idx].astype(str)), common_clu)
                          if common_clu else None)
            entry[name] = {**nm, "xfer_S": xfer_S, "domain_bacc": domain_bacc}
        save()

    build_and_score("raw", Fs_full_id)
    c_shift = Fx_rep.mean(0) - Fs_rep[ds["T"]].mean(0)   # hằng số dịch chuyển của base_center — f tuyến tính -> khỏi chạy lại
    build_and_score("center", Fs_full_id + c_shift)

    r2s = []
    for seed in R17.SEEDS:
        f_ = f"/kaggle/input/**/out17/{combo.replace('/', '_')}_default_s{seed}.npy"
        hit = sorted(glob.glob(f_, recursive=True))
        if not hit: W.append(f"{combo}: thiếu out17 map cho seed {seed} (Phần N)"); continue
        Fmap = load_map17(hit[0], len(Fs_rep))
        Wm, bm, r2 = recover_wb(Fs_rep, Fmap); r2s.append(r2)
        if r2 is None or r2 < 0.999: W.append(f"{combo} seed{seed}: R² khôi phục W = {r2} < 0.999")
        build_and_score(f"default_s{seed}", Fs_full_id @ Wm + bm)
        log(f"[{combo}] seed{seed} Phần N xong (R²={r2})")

    S.setdefault("N_meta", {}).setdefault(combo, {}).update({"r2": r2s, "khớp_tái_tạo_pca": ok_refit, "r2_căn_hệ": [r2_ax, r2_as]})
    save()
    return Fs_full_id, Fx_full, ADJ[50]


def process_combo_niche_usher(combo, Fs_full_id, Fx_full, adj50, PCTX, kmeans, S, save, W):
    """niche_usher (mục 4b): USHER-ST mặc định huấn luyện TRÊN đặc trưng đã gộp x̄^50 (bán kính 50µm, PCA gốc CHƯA
    căn chỉnh) thay vì x. Tái sử dụng Fs_full_id/Fx_full đã chiếu ở Phần N (cùng phép biến đổi PCA) — không tính lại."""
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    As50, Ax50 = adj50   # cùng ma trận kề 50 µm như Phần N (hàng = tế bào lite)
    Fs_bar, Fx_bar = np.asarray(As50 @ Fs_full_id, np.float32), np.asarray(Ax50 @ Fx_full, np.float32)

    center_map = M.base_center(Fs_bar, Fx_bar, ds["T"])
    mu = Fx_bar.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_bar - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
    lab_aux = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    sclu_s, sclu_x = read_spatial_cluster(pair), read_spatial_cluster(TARGET)
    common_clu, _ = common_clusters(sclu_s, sclu_x)
    Sc = S["N"].setdefault(combo, {})
    for seed in R17.SEEDS:
        key = f"niche_usher_s{seed}"
        if key in Sc and "niche_50" in Sc[key]: continue
        name = f"{combo.replace('/', '_')}_niche_usher_s{seed}.npy"; fpath = f"{OUT}/{name}"
        if RESUME and os.path.exists(fpath):
            Fmap = np.load(fpath).astype(np.float32); step(f"{name}: nạp lại map")
        else:
            step(f"USHER niche_usher [{combo}] seed={seed}")
            Fmap, info = R17.usher_variant(Fs_bar, ds["T"], P_aux, Fx_bar, tgt, dx["lab"], pctx["classes"], seed, "target", None, None)
            info.pop("cache_stats", None); np.save(fpath, Fmap.astype(np.float16))
        nm = niche_metrics(Fmap, Fx_bar, ds, dx, pctx["ref"], comp_s, comp_x, kmeans, Q)
        nn_x = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx_bar[pctx["ref"]] - mu))
        nn_idx = nn_x.kneighbors(_unit(Fmap[Q] - mu), return_distance=False)
        xfer_S = M.bacc_on(ds["lab"][Q], mode_rows(dx["lab"][pctx["ref"]][nn_idx]), pctx["S_shared"])
        domain_bacc = (M.bacc_on(sclu_s[Q].astype(str), mode_rows(sclu_x[pctx["ref"]][nn_idx].astype(str)), common_clu)
                      if common_clu else None)
        Sc[key] = {"niche_50": {**nm, "xfer_S": xfer_S, "domain_bacc": domain_bacc}}
        save(); log(f"[{combo}] {key}: {Sc[key]}"); del Fmap; gc.collect()


# =============================================================================== tiếp nối xuyên tài khoản
def _deep_merge(dst, src):
    for k, v in src.items():
        if k not in dst: dst[k] = v
        elif isinstance(dst[k], dict) and isinstance(v, dict): _deep_merge(dst[k], v)
    return dst

def import_prior_outputs():
    """Dùng được ở CẢ HAI tài khoản, không cần sửa gì: chép mọi file của thư mục out19 từ các lần chạy trước (ở BẤT KỲ
    tài khoản nào, Add làm Input — kể cả lần chạy lỗi giữa chừng) vào OUT; gộp các summary.json. RESUME sẽ bỏ qua phần
    đã xong (map USHER .npy, mục A/N đã có trong summary.json) và chỉ làm phần còn thiếu."""
    import shutil
    name = os.path.basename(OUT); n_copy = 0; merged = {}
    dirs = sorted(d for d in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(d))
    for d in dirs:
        for f in glob.glob(f"{d}/*"):
            b = os.path.basename(f)
            if b == "summary.json":
                try: _deep_merge(merged, json.load(open(f)))
                except Exception as e: log(f"bỏ qua {f}: {e!r}")
            elif os.path.isfile(f) and not os.path.exists(f"{OUT}/{b}"):
                shutil.copy(f, f"{OUT}/{b}"); n_copy += 1
    if merged:
        cur = f"{OUT}/summary.json"
        if os.path.exists(cur): merged = _deep_merge(json.load(open(cur)), merged)
        json.dump(merged, open(cur, "w"), indent=1, default=str)
    log(f"Tiếp nối: {len(dirs)} thư mục {name} cũ trong Input {dirs} -> chép {n_copy} file, gộp summary.json={bool(merged)}")


# =============================================================================== main
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    import_prior_outputs()
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER-ST mặc định giai đoạn 1 (struct_cache=target, aux_spatial β=0) K_niche_types={K_NICHE_TYPES} "
           f"combos_A={COMBOS_A} pairs_C={PAIRS_C} pairs_N={PAIRS_N} radii={RADII} n_folds={N_FOLDS} seeds={SEEDS_MAIN} "
           f"parts={sorted(PARTS)} KAGGLE_ACC={os.environ.get('KAGGLE_ACC', '?')}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = cfg
    for k in ("A", "C", "N"): S.setdefault(k, {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    needed_pairs = sorted({c.split("/")[0] for c in COMBOS_A} | set(PAIRS_C) | {c.split("/")[0] for c in PAIRS_N})
    D, F, PCTX = R18.setup_pairs(needed_pairs)
    pca_cache = {}
    comp_x0 = R17.get_niche_label_comp(needed_pairs[0], S, save)[1]
    kmeans = KMeans(n_clusters=K_NICHE_TYPES, random_state=0, n_init=10).fit(comp_x0)
    log(f"KMeans niche: cỡ cụm={np.bincount(kmeans.labels_)}")

    if "A" in PARTS:
        npy_files = {os.path.splitext(os.path.basename(f))[0]: f for f in glob.glob("/kaggle/input/**/out17/*.npy", recursive=True)}
        log(f"Phần A: tìm thấy {len(npy_files)} map .npy của vòng 17")
        for combo in COMBOS_A: process_combo_a(combo, F, PCTX, pca_cache, kmeans, S, save, W, npy_files)

    if "N" in PARTS:
        for combo in PAIRS_N:
            Fs_full_id, Fx_full, adj50 = process_pair_n(combo, D, F, PCTX, pca_cache, kmeans, S, save, W)
            process_combo_niche_usher(combo, Fs_full_id, Fx_full, adj50, PCTX, kmeans, S, save, W)
            del Fs_full_id, Fx_full, adj50; gc.collect()

    if "C" in PARTS:
        for pair in PAIRS_C: process_pair_c(pair, D, F, PCTX, S, save, W)

    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 19")


# =============================================================================== summary.md
def summary_md(S, W):
    L = ["# 19 — mốc loại mịn + trần niche (A), kênh niche gộp (N), nội suy gene đúng nghĩa (C1)", S["config"], ""]

    L.append("## Phần A — mốc loại mịn + trần niche")
    rows = []
    for combo, Sc in S.get("A", {}).items():
        for variant in ("raw", "center"):
            if variant in Sc:
                v = Sc[variant]
                rows.append({"tổ_hợp": combo, "map": variant, "niche_bacc": v.get("niche_bacc"), "type": v.get("niche_bacc_type"),
                            "fine": v.get("niche_bacc_fine"), "pred": v.get("niche_bacc_pred"), "lùi_fine": v.get("frac_fallback_fine")})
        rl = [Sc[f"default_s{s}"] for s in R17.SEEDS if f"default_s{s}" in Sc]
        if rl:
            m_b, sd_b = _ms([e.get("niche_bacc") for e in rl])
            rows.append({"tổ_hợp": combo, "map": f"default(n={len(rl)})", "niche_bacc": round(m_b, 4) if m_b is not None else None,
                        "type": rl[0].get("niche_bacc_type"), "fine": rl[0].get("niche_bacc_fine"), "pred": rl[0].get("niche_bacc_pred"),
                        "lùi_fine": rl[0].get("frac_fallback_fine")})
        if "ceil" in Sc: rows.append({"tổ_hợp": combo, "map": "trần", **Sc["ceil"]})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    n_a1 = n_a2 = n_a3 = 0
    for combo, Sc in S.get("A", {}).items():
        rl = [Sc[f"default_s{s}"] for s in R17.SEEDS if f"default_s{s}" in Sc]
        if not rl or "ceil" not in Sc: continue
        m_b, sd_b = _ms([e.get("niche_bacc") for e in rl])
        r1 = beats(m_b, sd_b, rl[0].get("niche_bacc_fine"), 0)
        r2 = beats(m_b, sd_b, rl[0].get("niche_bacc_pred"), 0)
        c_knn = Sc["ceil"].get("ceil_src_knn")
        a3 = bool(m_b is not None and c_knn is not None and not_much_worse(m_b, c_knn, 0.02))   # FIX: A3 = niche_bacc ≥ trần − 0.02
        a1, a2 = bool(r1 and r1["pass"]), bool(r2 and r2["pass"])
        n_a1 += a1; n_a2 += a2; n_a3 += a3
        L.append(f"- {combo}: A1={'ĐẠT' if a1 else 'không'} A2={'ĐẠT' if a2 else 'không'} A3={'ĐẠT' if a3 else 'không'}")
    L.append(f"- A1: {n_a1}/{len(S.get('A', {}))} | A2: {n_a2}/{len(S.get('A', {}))} | A3: {n_a3}/{len(S.get('A', {}))}")

    L.append("## Phần C1 — nội suy gene đúng nghĩa")
    for pair, Sc in S.get("C", {}).items():
        agg = Sc.get("agg", {})
        rows = [{"phương_pháp": m, **v} for m, v in agg.items()]
        L.append(f"### {pair} ({Sc.get('n_genes')} gene chấm)")
        if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
        if "default_vs_center" in Sc: L.append(f"- default vs center: {Sc['default_vs_center']}")
        if "default_vs_tangram" in Sc: L.append(f"- default vs tangram: {Sc['default_vs_tangram']}")
        sd0 = Sc.get("fold0_seed_sd")
        L.append(f"- SD seed (fold 0, Spearman) = {sd0}")
        sp_med = {m: agg[m]["spearman_med"] for m in agg}
        order_ok = all(sp_med.get("shuffle", -9) < sp_med.get(m, -9) <= sp_med.get("xen_own", 9) for m in ("center", "default", "tangram") if m in sp_med)
        L.append(f"- I0 (mô tả): shuffle < {{center,default,tangram}} ≤ xen_own -> {'khớp' if order_ok else 'KHÔNG khớp'}")
        if "default" in sp_med and "center" in sp_med:
            L.append(f"- I1: default ≥ center − 0.01 -> {'ĐẠT' if not_much_worse(sp_med['default'], sp_med['center'], 0.01) else 'không'}")
            r2 = beats(sp_med["default"], sd0 or 0, sp_med["center"], 0)
            L.append(f"- I2: default vượt center (SD fold0×seed) -> {'ĐẠT' if r2 and r2['pass'] else 'không'}")
        if "default" in sp_med and "tangram" in sp_med:
            r3 = beats(sp_med["default"], sd0 or 0, sp_med["tangram"], 0)
            i3 = "vượt" if (r3 and r3["pass"]) else ("kém" if (r3 and beats(sp_med["tangram"], 0, sp_med["default"], sd0 or 0)["pass"]) else "ngang")
            L.append(f"- I3: default so tangram -> {i3}")

    L.append("## Phần N — kênh niche gộp sau căn chỉnh")
    rows = []
    for combo, Sc in S.get("N", {}).items():
        for key, entry in Sc.items():
            for name, v in entry.items():
                rows.append({"tổ_hợp": combo, "map": key, "biểu_diễn": name, **v})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    for combo, meta in S.get("N_meta", {}).items():
        L.append(f"- {combo}: R² khôi phục W = {meta.get('r2')} | khớp tái tạo PCA = {meta.get('khớp_tái_tạo_pca')}")

    n1 = n2 = n3 = 0
    for combo, Sc in S.get("N", {}).items():
        def ms_of(label, name, key):
            vals = [Sc[k][name].get(key) for k in Sc if k.startswith(label) and name in Sc.get(k, {})]
            return _ms(vals)
        nb_def, sd_def = ms_of("default_s", "concat_50", "niche_bacc")
        nb_cen = (Sc.get("center", {}).get("concat_50") or {}).get("niche_bacc")
        r1 = beats(nb_def, sd_def, nb_cen, 0) if (nb_def is not None and nb_cen is not None) else None
        ok1 = bool(r1 and r1["pass"])
        n1 += ok1
        L.append(f"- N1 {combo}: concat_50(default) niche_bacc={round(nb_def, 4) if nb_def is not None else None} vs center={nb_cen} -> {'ĐẠT' if ok1 else 'không'}")

        xfer_concat, sd_xc = ms_of("default_s", "concat_50", "xfer_S")
        xfer_id, sd_xi = ms_of("default_s", "id", "xfer_S")
        ok2 = bool(xfer_concat is not None and xfer_id is not None and not_much_worse(xfer_concat, xfer_id, 0.02))
        n2 += ok2
        L.append(f"- N2 {combo}: xfer_S(concat_50)={round(xfer_concat, 4) if xfer_concat is not None else None} vs xfer_S(id)={round(xfer_id, 4) if xfer_id is not None else None} -> {'ĐẠT' if ok2 else 'không'}")

        nb_nu, sd_nu = ms_of("niche_usher_s", "niche_50", "niche_bacc")
        nb_d50, sd_d50 = ms_of("default_s", "niche_50", "niche_bacc")
        r3 = beats(nb_nu, sd_nu, nb_d50, sd_d50) if (nb_nu is not None and nb_d50 is not None) else None
        ok3 = bool(r3 and r3["pass"])
        n3 += ok3
        L.append(f"- N3 {combo}: niche_usher niche_bacc={round(nb_nu, 4) if nb_nu is not None else None} vs default(niche_50)={round(nb_d50, 4) if nb_d50 is not None else None} -> {'ĐẠT' if ok3 else 'không'}")
    L.append(f"- N1: {n1}/{len(S.get('N', {}))} | N2: {n2}/{len(S.get('N', {}))} | N3: {n3}/{len(S.get('N', {}))}")
    L.append("- Cảnh báo vòng tròn (mục 4b): loại niche định nghĩa từ thành phần nhãn hàng xóm, kênh niche là TB embedding hàng xóm -> niche_bacc của niche_r/concat_r tăng gần như hiển nhiên; chỉ đọc so sánh TƯƠNG ĐỐI (default vs center, N3) và domain_bacc.")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run20.py
"""20 — thoát giới hạn H12 bằng map phụ thuộc vi môi trường. (D) chẩn đoán phần dư map toàn cục theo niche; (M) map
MoE có cổng theo niche (không nhãn); (T) căn chỉnh có ngữ cảnh không gian [x, x̄^25]; (B) baseline còn thiếu
(Symphony, scArches trong venv riêng; Tangram tiến trình con/fold, 10k truy vấn); (K) C_t cùng "thước" landmark với
C_s; (E) đánh giá trên MỌI nhóm tế bào (không chỉ nhóm "khôi phục được"), áp cho vòng này VÀ chấm lại bảng 16d.

Tái dùng run19.py (radius_adj, pca_fit, pca_apply, full_expr_matched, lite_obs, load_map17, import_prior_outputs,
scored_genes, pca_cell_masked, common_clusters, read_spatial_cluster), run18.niche_metrics, run17 (usher_variant,
get_niche_label_comp), run16 (eval kiểu 16d: R9.eval_both + R11.metrics_pack, mnn_ref_correction, run_harmony,
run_scanorama). CHÚ Ý (nhắc lại từ 19): gán lại R17.OUT/R18.OUT = OUT (vòng này); beats(...) trả dict, đọc ["pass"].
"""
import os, sys, glob, json, gc, time, subprocess
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run19 as R19
R17, R18 = R19.R17, R19.R18
R, B, M, C, R6, R7, R9, R11, R12, R8 = R19.R, R19.B, R19.M, R19.C, R19.R6, R19.R7, R19.R9, R19.R11, R19.R12, R19.R8
R16 = R17.R16
np, pd, sp, log, load_assay, _unit = R19.np, R19.pd, R19.sp, R19.log, R19.load_assay, R19._unit
_ms, beats, not_much_worse = R19._ms, R19.beats, R19.not_much_worse
SMOKE = R19.SMOKE   # xử lý biến môi trường SMOKE đã xảy ra khi import run19 -> ... -> run06
NearestNeighbors, KMeans, mode_rows, niche_metrics = R19.NearestNeighbors, R19.KMeans, R19.mode_rows, R19.niche_metrics
lite_obs, load_map17, scored_genes, pca_cell_masked = R19.lite_obs, R19.load_map17, R19.scored_genes, R19.pca_cell_masked
radius_adj, pca_fit, pca_apply, full_expr_matched = R19.radius_adj, R19.pca_fit, R19.pca_apply, R19.full_expr_matched
common_clusters, read_spatial_cluster = R19.common_clusters, R19.read_spatial_cluster
from sklearn.linear_model import LogisticRegression
from scipy.stats import wilcoxon
HERE = os.path.dirname(os.path.abspath(__file__))

OUT = os.environ.get("OUT_DIR_20", "/kaggle/working/out20" + ("_smoke" if SMOKE else ""))
TARGET = R19.TARGET
R17.OUT, R18.OUT, R19.OUT = OUT, OUT, OUT   # FIX 20b: import_prior_outputs & cache của run19 dùng R19.OUT   # bài học 17-19: hàm cache theo OUT của các module đó phải trỏ về OUT của vòng 20
COMBOS = R16.COMBOS_ALL   # 4 tổ hợp mặc định
PAIRS_C_ALL = ["cosmx6k", "visiumhd_ffpe", "stereoseq"]   # Phần B/Tangram (như 19)
K_NICHE_TYPES = R19.K_NICHE_TYPES   # 8 (mốc thô, như 18/19)
K_GATE = 6         # KMeans cổng niche (mục 3)
K_FINE16 = 10      # KMeans niche mịn 16 nhóm (Phần E)
R_NICHE = 25       # bán kính đặc trưng niche cho toàn vòng (mục 2: x̄^25, thang tốt nhất ở 19)
SEEDS = [0] if SMOKE else [0, 1, 2]
LAMBDA_GRID = [0.1, 1, 10, 100]
PARTS = {p.strip() for p in os.environ.get("ONLY_PART", "E,M,T,B,K").split(",") if p.strip()}
if "all" in PARTS: PARTS = {"E", "M", "T", "B", "K"}
RESUME = os.environ.get("RESUME", "1") == "1"
T0 = time.time()
TIME_BUDGET_S = float(os.environ.get("TIME_BUDGET_S", 11 * 3600))   # FIX 20d: dừng SẠCH trước giới hạn 12 giờ của Kaggle
class OutOfTime(Exception): pass
def step(msg):
    log(f"    [{time.time() - T0:6.0f}s] {msg}")
    if time.time() - T0 > TIME_BUDGET_S: raise OutOfTime(msg)

FINE16 = sorted(B.COARSE.keys())   # 16 nhóm mịn SPATCH (mục 6b)


def run_list():
    return [(combo, part, seed) for seed in SEEDS for combo in COMBOS for part in ("M", "T", "K")]

def run_name(combo, part, seed):
    return f"{combo.replace('/', '_')}_{part}_s{seed}"


# =============================================================================== hạ tầng dùng chung
def usher_variant_T(Fs_rep, T, P_src, Fx_rep, tgt, lab_x, classes, seed, cache_mode, F_struct_src=None, F_struct_tgt=None):
    """Chép từ run17.usher_variant (KHÔNG sửa run17.py): giữ lại T_full để tính đích barycentric ȳ_i = Σ_j T_ij·Fx[tgt_j]
    / Σ_j T_ij (mục 3, Phần D+M) TRƯỚC KHI xoá. Trả (Fmap, info, ybar, has_target); has_target[i] = Σ_j T_ij > 0."""
    import torch, anndata as ad, run_usher as U, struct_cache as SC, random
    os.environ["USHER_SEED"] = str(seed); os.environ["USHER_EM_WORKERS"] = "4"
    os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = "0.3"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    C.logging.getLogger().setLevel(C.logging.WARNING)
    Pt = np.zeros((len(tgt), len(classes)), np.float32)
    Pt[np.arange(len(tgt)), [list(classes).index(c) for c in lab_x[tgt]]] = 1
    a = ad.AnnData(X=Fs_rep[T].astype(np.float32), obs=pd.DataFrame(index=[f"s{i}" for i in T]))
    b = ad.AnnData(X=Fx_rep[tgt].astype(np.float32), obs=pd.DataFrame({"ct": lab_x[tgt]}, index=[f"t{i}" for i in tgt]))
    a.obs["ct"] = "unknown"
    a.obsm["X_celltype_probs"] = P_src[T].astype(np.float32); b.obsm["X_celltype_probs"] = Pt
    info = {}
    SC.reset(cache_mode, F_struct_src, F_struct_tgt)
    t0 = time.time()
    model, _, _, T_full, mean, std = U.align_features_fgw(
        a, b, e_step_method="fgw", m_step_method="global", sampling_strategy="celltype",
        sketch_size=C.SKETCH, sketch_obsm_key="X_none", use_stratified_pairing=True, stratified_pairing_fix=True,
        cell_type_col="ct", celltype_probs_layer="X_celltype_probs", alpha=R6.U6["alpha"], gamma=R6.U6["gamma"],
        epsilon=0.1, knn_k=30, metric="cosine", m_step_metric="euclidean", lambda_cross=0.9, lambda_var=0.1,
        entropy_percentile=90.0, confidence_percentile=10.0, use_linear_assignment=True, steps_per_iter=100, lr=1e-3,
        n_iters=C.N_ITERS, hidden_dim=None, device="cuda", verbose=False)
    info["em_seconds"] = round(time.time() - t0, 1); info["cache_stats"] = dict(SC.STATS)
    Tm = T_full.detach().float().cpu().numpy() if hasattr(T_full, "detach") else np.asarray(T_full, np.float32)
    row_sum = Tm.sum(1); has_target = row_sum > 0
    ybar = np.zeros((Tm.shape[0], Fx_rep.shape[1]), np.float32)
    ybar[has_target] = (Tm[has_target] @ Fx_rep[tgt].astype(np.float32)) / row_sum[has_target, None]
    SC.reset("off")
    del T_full, Tm; torch.cuda.empty_cache()
    with torch.no_grad():
        model.eval(); x = torch.from_numpy(Fs_rep.astype(np.float32)).cuda()
        y = model((x - mean) / std) * std + mean if mean is not None else model(x)
        out = y.cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return out, info, ybar, has_target


_NF_CACHE = {}
KM16 = None   # KMeans niche mịn 16 nhóm, đặt trong main() (dùng cho track ALL ở mọi phần)
_E_CTX = {}

def niche_features(combo, PCTX, F, pca_cache, S, save):
    if combo not in _NF_CACHE: _NF_CACHE[combo] = _niche_features(combo, PCTX, F, pca_cache, S, save)
    return _NF_CACHE[combo]

def _niche_features(combo, PCTX, F, pca_cache, S, save):
    """x̄^25 = TB PCA CHƯA căn chỉnh của tế bào high_quality trong bán kính 25µm (mục 2) — LUÔN PCA chung của cặp,
    kể cả khi biểu diễn danh tính là scGPT. Trả (Fs_id, Fx_id, Fs_bar, Fx_bar) theo thứ tự lite."""
    pair, rep = combo.split("/")
    pctx = PCTX[pair]
    Fs_id, Fx_id = R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)
    Fx_pca_lite, Fs_pca_lite, _ = R8.pca_cell(pctx["Xx_lite"], pctx["varx"], pctx["Xs_lite"], pctx["vars_"])
    G = R8.gene_cols(pctx["varx"], pctx["vars_"])[2]
    P_lite, tr = pca_fit(sp.vstack([pctx["Xx_lite"][:, R8.gene_cols(pctx["varx"], pctx["vars_"])[0]],
                                    pctx["Xs_lite"][:, R8.gene_cols(pctx["varx"], pctx["vars_"])[1]]]).tocsr())
    nx = pctx["Xx_lite"].shape[0]
    Fx_check, Fs_check = P_lite[:nx], P_lite[nx:]
    Xx_full, ids_x_full, xy_x_full = full_expr_matched(TARGET, G)
    Xs_full, ids_s_full, xy_s_full = full_expr_matched(pair, G)
    Fx_full, Fs_full = pca_apply(Xx_full, tr), pca_apply(Xs_full, tr)
    del Xx_full, Xs_full; gc.collect()

    def _align(A_chk, A_ref, A_full):
        """Chép từ run19.process_pair_n._align (không sửa run19.py): căn phép chiếu PCA đầy đủ về ĐÚNG hệ toạ độ
        PCA-chung lite (bằng lstsq trên tập lite) — an toàn cả khi tái tạo PCA lệch dấu/xoay."""
        X_ = np.hstack([A_chk, np.ones((len(A_chk), 1), np.float32)])
        coef, *_ = np.linalg.lstsq(X_, A_ref, rcond=None)
        return (np.hstack([A_full, np.ones((len(A_full), 1), np.float32)]) @ coef).astype(np.float32)
    Fx_full, Fs_full = _align(Fx_check, Fx_pca_lite, Fx_full), _align(Fs_check, Fs_pca_lite, Fs_full)

    obs_x_lite, obs_s_lite = lite_obs(TARGET), lite_obs(pair)
    pos_x = np.clip(pd.Index(ids_x_full).get_indexer(obs_x_lite.index.astype(str)), 0, None)
    pos_s = np.clip(pd.Index(ids_s_full).get_indexer(obs_s_lite.index.astype(str)), 0, None)
    As, fb_s, md_s = radius_adj(xy_s_full, pos_s, R_NICHE)
    Ax, fb_x, md_x = radius_adj(xy_x_full, pos_x, R_NICHE)
    Fs_bar, Fx_bar = np.asarray(As @ Fs_full, np.float32), np.asarray(Ax @ Fx_full, np.float32)
    S.setdefault("niche_meta", {})[combo] = {"fallback_src": fb_s, "fallback_xen": fb_x, "median_nb_src": md_s, "median_nb_xen": md_x}
    save()
    return Fs_id, Fx_id, Fs_bar, Fx_bar


def eval_all16d(pair, D, Fs_rep, Fx_used, name, Fmap, pctx):
    ev = R9.eval_both(pair, D, Fs_rep, Fx_used, name, Fmap, pctx["S_shared"], pctx["IMP"], pctx["ref"])[1]
    mp = R11.metrics_pack(Fmap, Fx_used, pctx["lab_src_ev"], pctx["lab_tgt_ev"], pctx["idx_src"], pctx["idx_tgt"])
    w = [f"{name}: {x}" for x in mp.pop("_warnings")]
    out = {**mp, "xfer_S": ev["xfer_S"], "xfer_coarse": ev["xfer_coarse"], "sp": ev["imp"]["spearman_med"]}
    out["xfer_main"] = out[pctx["xfer_key"]]
    return out, w


# =============================================================================== Phần D + M — MoE có cổng theo niche
def ridge_fit(X, Y, lam, theta0=None):
    d = X.shape[1]
    A = X.T @ X + lam * np.eye(d, dtype=np.float64)
    rhs = X.T @ Y + (lam * theta0 if theta0 is not None else 0.0)
    return np.linalg.solve(A, rhs)

def eta_squared(e, groups):
    grand = e.mean(0); ss_tot = float(((e - grand) ** 2).sum())
    if ss_tot <= 0: return 0.0
    ss_b = sum(int((groups == g).sum()) * float(((e[groups == g].mean(0) - grand) ** 2).sum()) for g in np.unique(groups))
    return ss_b / ss_tot

def r2_onehot(e, *group_arrays):
    grand = e.mean(0); ss_tot = float(((e - grand) ** 2).sum())
    if ss_tot <= 0: return 0.0
    cols = [np.ones(len(e), np.float64)]
    for g in group_arrays:
        for v in np.unique(g)[:-1]: cols.append((g == v).astype(np.float64))
    X = np.column_stack(cols)
    coef, *_ = np.linalg.lstsq(X, e, rcond=None)
    return 1 - float(((e - X @ coef) ** 2).sum()) / ss_tot

def sub_split_250(xy_T):
    """Bàn cờ con 250µm trên tế bào T (mục 3): chia thêm bên trong ô 500µm hiện có, xen kẽ."""
    ij = np.floor((xy_T - xy_T.min(0)) / 250.0).astype(np.int64)
    parity = (ij[:, 0] + ij[:, 1]) % 2
    return parity == 0, parity == 1

def process_combo_dm(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W, npy_files16):
    done = S.get("M", {}).get(combo, {})
    if all(all(m in done.get(f"s{sd}", {}) for m in ("default", "refit_global", "moe_niche", "moe_random", "moe_type")) and "D" in done.get(f"s{sd}", {}) for sd in SEEDS):
        log(f"[{combo}] Phần M: đã xong trong summary.json -> bỏ qua"); return
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep, Fs_bar, Fx_bar = niche_features(combo, PCTX, F, pca_cache, S, save)
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)

    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
    lab_aux_full = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux_full[ds["T"]], dx["lab"])

    gate_km = KMeans(n_clusters=K_GATE, random_state=0, n_init=10).fit(Fs_bar[ds["T"]])
    g_all = gate_km.predict(Fs_bar)   # cổng cho MỌI tế bào source (T và H)
    g_all_random = np.random.RandomState(0).permutation(g_all)   # hoán vị TOÀN CỤC, giữ cỡ cụm (mục 3, moe_random)
    S.setdefault("D", {})[combo] = {"cỡ_cụm_gate": np.bincount(g_all[ds["T"]], minlength=K_GATE).tolist()}
    save(); log(f"[{combo}] cổng niche (T): {S['D'][combo]['cỡ_cụm_gate']}")

    obs_s = lite_obs(pair); xy_T = obs_s[["x", "y"]].to_numpy()[ds["T"]]
    m1, m2 = sub_split_250(xy_T)
    T1, T2 = ds["T"][m1], ds["T"][m2]

    Sc = S["M"].setdefault(combo, {})
    for seed in SEEDS:
        key = f"s{seed}"
        f_ybar = f"{OUT}/{combo.replace('/', '_')}_ybar_s{seed}.npz"
        f_map = f"{OUT}/{combo.replace('/', '_')}_default_s{seed}.npy"
        if RESUME and os.path.exists(f_ybar) and os.path.exists(f_map):
            step(f"{combo} s{seed}: nạp lại ybar + map mặc định")
            d_ = np.load(f_ybar); ybar_T, has_tgt = d_["ybar"], d_["has_tgt"]
            Fmap_default = load_map17(f_map, len(Fs_rep))
        else:
            step(f"USHER (giữ T_full) [{combo}] s{seed}")
            Fmap_default, info, ybar_full, has_full = usher_variant_T(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"],
                                                                      pctx["classes"], seed, "target", None, None)
            ybar_T, has_tgt = ybar_full, has_full   # hàng theo thứ tự ds["T"] (0..len(T)-1)
            np.savez(f_ybar, ybar=ybar_T, has_tgt=has_tgt)
            np.save(f_map, Fmap_default.astype(np.float16))
            info.pop("cache_stats", None); Sc.setdefault(key, {})["info"] = info; save()
        nrun = sorted(glob.glob(f"/kaggle/input/**/out17/{combo.replace('/', '_')}_default_s{seed}.npy", recursive=True))
        if nrun:
            m17 = load_map17(nrun[0], len(Fs_rep))
            d_ = float(np.nanmean(np.abs(Fmap_default - m17)))
            if d_ > 0.01: W.append(f"{combo} s{seed}: map mặc định lệch out17 (chênh TB tuyệt đối {d_:.4f} > 0.01)")

        Xhas = ds["T"][has_tgt]   # tế bào T CÓ đích (chỉ số toàn cục)
        m1h, m2h = np.isin(Xhas, T1), np.isin(Xhas, T2)
        Xt = np.hstack([Fs_rep[Xhas], np.ones((len(Xhas), 1), np.float32)]).astype(np.float64)
        Yt = ybar_T[has_tgt].astype(np.float64)
        g_t, g_t_rand = g_all[Xhas], g_all_random[Xhas]

        # (D) chẩn đoán phần dư: fit Θ_glob trên T1, phần dư trên T2
        Theta_glob_T1 = ridge_fit(Xt[m1h], Yt[m1h], 1e-3)
        e2 = Yt[m2h] - Xt[m2h] @ Theta_glob_T1
        g2, g2_rand = g_t[m2h], g_t_rand[m2h]   # g_t_rand = hoán vị TOÀN CỤC (g_all_random), không phải hoán vị cục bộ T2
        cnt = pd.Series(lab_aux_full[ds["T"]]).value_counts()
        rare = set(cnt[cnt < 200].index)
        type_t = np.where(np.isin(lab_aux_full[Xhas[m2h]], list(rare)), "other", lab_aux_full[Xhas[m2h]])
        eta_niche, eta_rand = eta_squared(e2, g2), eta_squared(e2, g2_rand)
        eta_niche_given_type = r2_onehot(e2, type_t, g2) - r2_onehot(e2, type_t)
        Sc.setdefault(key, {})["D"] = {"eta_niche": round(eta_niche, 5), "eta_random": round(eta_rand, 5),
                                       "eta_niche_given_type": round(eta_niche_given_type, 5), "n_T2": int(m2h.sum())}
        save(); log(f"[{combo}] s{seed} (D): {Sc[key]['D']}")

        # (M) fit trên TOÀN T có đích -> áp cho MỌI tế bào source. gate_apply = cổng dùng khi ÁP (mọi tế bào source);
        # gate_fit (dùng để fit expert, chỉ trên Xhas) là gate_apply[Xhas].
        Theta_glob = ridge_fit(Xt, Yt, 1e-3)
        type_all = np.where(np.isin(lab_aux_full, list(rare)), "other", lab_aux_full)
        gates = {"moe_niche": (g_t, g_all), "moe_random": (g_t_rand, g_all_random), "moe_type": (type_all[Xhas], type_all)}
        X_all = np.hstack([Fs_rep, np.ones((len(Fs_rep), 1), np.float32)]).astype(np.float64)

        def eval_map(name, Fmap):
            ev, w = eval_all16d(pair, D, Fs_rep, Fx_rep, name, Fmap, pctx); W.extend(w)
            nm = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
            return {**ev, **nm, **etrack(pair, D, PCTX, S, save, Fs_rep, Fx_rep, name, Fmap, W)}

        Sc[key]["refit_global"] = eval_map(f"{combo}/refit_global/s{seed}", (X_all @ Theta_glob).astype(np.float32))
        Sc[key]["default"] = eval_map(f"{combo}/default/s{seed}", Fmap_default)
        for gname, (gate_t, gate_all) in gates.items():
            lam_best, best_err = None, np.inf
            n_bar = len(Xhas) / max(1, len(np.unique(gate_t)))
            for lam_mult in LAMBDA_GRID:
                lam = lam_mult * n_bar
                Theta_glob_1 = ridge_fit(Xt[m1h], Yt[m1h], 1e-3)
                err = 0.0
                gate_t1 = gate_t[m1h]
                for k in np.unique(gate_t1):
                    mk = gate_t1 == k
                    Th_k = ridge_fit(Xt[m1h][mk], Yt[m1h][mk], lam, Theta_glob_1)
                    mk2 = gate_t[m2h] == k
                    if mk2.any(): err += float(((Yt[m2h][mk2] - Xt[m2h][mk2] @ Th_k) ** 2).sum())
                if err < best_err: best_err, lam_best = err, lam
            experts = {}
            for k in np.unique(gate_t):
                mk = gate_t == k
                experts[k] = ridge_fit(Xt[mk], Yt[mk], lam_best, Theta_glob)
            out_g = np.zeros_like(X_all[:, :Fx_rep.shape[1]], dtype=np.float32)
            for k, Th_k in experts.items():
                mk = gate_all == k
                out_g[mk] = (X_all[mk] @ Th_k).astype(np.float32)
            miss = ~np.isin(gate_all, list(experts))
            if miss.any(): out_g[miss] = (X_all[miss] @ Theta_glob).astype(np.float32)
            Sc[key][gname] = eval_map(f"{combo}/{gname}/s{seed}", out_g)
            Sc[key][gname]["lambda"] = round(float(lam_best), 2)
            save(); log(f"[{combo}] s{seed} {gname}: λ={lam_best:.1f} niche_bacc={Sc[key][gname].get('niche_bacc')} xfer_main={Sc[key][gname].get('xfer_main')}")
        save()


# =============================================================================== Phần T — căn chỉnh có ngữ cảnh
def get_concat25_niche_bacc(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W):
    """Đọc niche_bacc của concat_25 (vòng 19, mục 4) từ out19/summary.json có trong Input; nếu không có thì tính lại
    bằng run19.process_pair_n (chỉ 2 cặp PCA có, plan mục 4)."""
    for f in sorted(glob.glob("/kaggle/input/**/out19/summary.json", recursive=True)):
        try:
            entry = json.load(open(f)).get("N", {}).get(combo, {})
            vals = [entry[k]["concat_25"]["niche_bacc"] for k in entry if k.startswith("default_s") and "concat_25" in entry.get(k, {})]
            if vals: return _ms(vals)
        except Exception as e: log(f"bỏ qua {f}: {e!r}")
    entry = S.get("N", {}).get(combo, {})
    if not any(k.startswith("default_s") and "concat_25" in entry.get(k, {}) for k in entry):
        step(f"{combo}: không có concat_25 trong Input -> tính lại theo run19.process_pair_n")
        R19.process_pair_n(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W)
        entry = S.get("N", {}).get(combo, {})
    vals = [entry[k]["concat_25"]["niche_bacc"] for k in entry if k.startswith("default_s") and "concat_25" in entry.get(k, {})]
    return _ms(vals) if vals else (None, None)

def process_combo_t(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W):
    if all(f"s{sd}" in S.get("T", {}).get(combo, {}) for sd in SEEDS) and combo in S.get("T_meta", {}):
        log(f"[{combo}] Phần T: đã xong trong summary.json -> bỏ qua"); return
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_id, Fx_id, Fs_bar, Fx_bar = niche_features(combo, PCTX, F, pca_cache, S, save)
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)

    s_x = float(np.sqrt(np.mean(np.linalg.norm(Fx_id - Fx_id.mean(0), axis=1) ** 2)))
    s_n = float(np.sqrt(np.mean(np.linalg.norm(Fx_bar - Fx_bar.mean(0), axis=1) ** 2)))
    Xctx_s = np.hstack([Fs_id / s_x, Fs_bar / s_n]).astype(np.float32)
    Xctx_x = np.hstack([Fx_id / s_x, Fx_bar / s_n]).astype(np.float32)
    d_id = Fs_id.shape[1]

    center_map = M.base_center(Xctx_s, Xctx_x, ds["T"])
    mu = Xctx_x.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Xctx_x - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
    lab_aux = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])

    Sc = S["T"].setdefault(combo, {})
    for seed in SEEDS:
        key = f"s{seed}"
        if key in Sc: continue
        f_map = f"{OUT}/{combo.replace('/', '_')}_ctx_s{seed}.npy"
        if RESUME and os.path.exists(f_map):
            Fmap_ctx = np.load(f_map).astype(np.float32); step(f"{combo} ctx s{seed}: nạp lại map")
        else:
            step(f"USHER ctx [{combo}] s{seed}")
            Fmap_ctx, info = R17.usher_variant(Xctx_s, ds["T"], P_aux, Xctx_x, tgt, dx["lab"], pctx["classes"], seed, "target", None, None)
            info.pop("cache_stats", None); np.save(f_map, Fmap_ctx.astype(np.float16))
        id_block = (Fmap_ctx[:, :d_id] * s_x).astype(np.float32)
        ev_id, w = eval_all16d(pair, D, Fs_id, Fx_id, f"{combo}/ctx_id/s{seed}", id_block, pctx); W.extend(w)
        nm_id = niche_metrics(id_block, Fx_id, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
        nm_full = niche_metrics(Fmap_ctx, Xctx_x, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
        nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Xctx_x[pctx["ref"]] - mu)).kneighbors(_unit(Fmap_ctx[Q] - mu), return_distance=False)
        xfer_full = M.bacc_on(ds["lab"][Q], mode_rows(dx["lab"][pctx["ref"]][nn]), pctx["S_shared"])
        et_id = etrack(pair, D, PCTX, S, save, Fs_id, Fx_id, f"{combo}/ctx_id/s{seed}", id_block, W)
        Sc[key] = {"ctx_id": {**ev_id, **nm_id, **et_id}, "ctx_full": {"niche_bacc": nm_full["niche_bacc"], "xfer_S": xfer_full}}
        save(); log(f"[{combo}] s{seed} ctx_id: xfer_main={ev_id['xfer_main']} niche_bacc={nm_id['niche_bacc']} | ctx_full niche_bacc={nm_full['niche_bacc']}")

    m_c25, sd_c25 = (get_concat25_niche_bacc(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W) if rep == "pca" else (None, None))   # FIX: T2 chỉ cho cặp PCA
    S.setdefault("T_meta", {})[combo] = {"concat_25_niche_bacc": m_c25}
    save()


# =============================================================================== Phần K — C_t cùng "thước" landmark
def process_combo_k(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W):
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
    lab_aux = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
    Sc = S["K"].setdefault(combo, {})
    for seed in SEEDS:
        key = f"s{seed}"
        if key in Sc: continue
        f_map = f"{OUT}/{combo.replace('/', '_')}_ctlandmark_s{seed}.npy"
        if RESUME and os.path.exists(f_map):
            Fmap = np.load(f_map).astype(np.float32); step(f"{combo} ct_landmark s{seed}: nạp lại map")
        else:
            step(f"USHER ct_landmark [{combo}] s{seed}")
            Fmap, info = R17.usher_variant(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed, "target_landmark", None, None)
            info.pop("cache_stats", None); np.save(f_map, Fmap.astype(np.float16))
        ev, w = eval_all16d(pair, D, Fs_rep, Fx_rep, f"{combo}/ct_landmark/s{seed}", Fmap, pctx); W.extend(w)
        nm = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
        Sc[key] = {**ev, **nm, **etrack(pair, D, PCTX, S, save, Fs_rep, Fx_rep, f"{combo}/ct_landmark/s{seed}", Fmap, W)}
        save(); log(f"[{combo}] s{seed} ct_landmark: xfer_main={ev['xfer_main']} tổng_v2={ev.get('total_v2')}")


# =============================================================================== Phần B — baseline còn thiếu
def setup_venv_b(W):
    vpath = "/kaggle/working/venv_b"; py = f"{vpath}/bin/python"
    if os.path.exists(py): return py
    try:
        r0 = subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", vpath], capture_output=True, text=True, timeout=600)
        if r0.returncode != 0 or not os.path.exists(f"{vpath}/bin/pip"):
            # FIX 20c: python -m venv lỗi trên Kaggle (thiếu ensurepip) -> dùng virtualenv (tự mang pip), vẫn tách khỏi kernel chính
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True, capture_output=True, text=True, timeout=600)
            subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", vpath], check=True, capture_output=True, text=True, timeout=600)
        r = subprocess.run([f"{vpath}/bin/pip", "install", "-q", "symphonypy", "scvi-tools"], capture_output=True, text=True, timeout=1800)
        if r.returncode != 0: W.append(f"cài venv_b lỗi (bỏ Symphony/scArches): {r.stderr[-800:]}")
        return py if os.path.exists(py) else None
    except Exception as e:
        W.append(f"tạo venv_b lỗi (bỏ Symphony/scArches): {e!r}"); return None

def run_baseline_subprocess(py, method, Xx, Xs, W):
    """Trao đổi dữ liệu qua file .npz trong out20/b_io/ (mục 5): input = count thô gene chung (Xx Xenium, Xs source);
    output = embedding hai phía trong cùng một không gian reference. None nếu venv/tiến trình lỗi."""
    if py is None: return None
    os.makedirs(f"{OUT}/b_io", exist_ok=True)
    inp, outp = f"{OUT}/b_io/{method}_in.npz", f"{OUT}/b_io/{method}_out.npz"
    np.savez(inp, Xx=Xx.astype(np.float32), Xs=Xs.astype(np.float32))
    r = subprocess.run([py, "baselines20_env.py", method, inp, outp], cwd=HERE, capture_output=True, text=True, timeout=3600)
    if r.returncode != 0 or not os.path.exists(outp):
        W.append(f"{method} lỗi trong venv riêng, bỏ qua: {(r.stderr or '')[-800:]}"); return None
    d = np.load(outp)
    return d["Fx"].astype(np.float32), d["Fs"].astype(np.float32), str(d["version"]) if "version" in d else "?"

def process_pair_b_embed(pair, D, PCTX, py, kmeans8, S, save, W):
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    ix, is_, G = R8.gene_cols(pctx["varx"], pctx["vars_"])
    Xx, Xs = pctx["Xx_lite"][:, ix].toarray(), pctx["Xs_lite"][:, is_].toarray()
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    Sb = S["B"].setdefault(pair, {})
    for method in ("symphony", "scarches"):
        if Sb.get(method) is not None: continue   # FIX 20e: None = lần trước lỗi -> chạy lại
        step(f"Phần B [{pair}] {method} (venv riêng)")
        res = run_baseline_subprocess(py, method, Xx, Xs, W)
        if res is None: Sb[method] = None; save(); continue
        Fx_b, Fs_b, ver = res
        ev, w = eval_all16d(pair, D, Fs_b, Fx_b, f"{pair}/{method}", Fs_b, pctx); W.extend(w)
        nm = niche_metrics(Fs_b, Fx_b, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
        et = etrack(pair, D, PCTX, S, save, Fs_b, Fx_b, f"{pair}/{method}", Fs_b, W)
        Sb[method] = {**ev, **nm, **et, "version": ver}
        save(); log(f"[{pair}] {method}: xfer_main={ev['xfer_main']} tổng_v2={ev.get('total_v2')} (v{ver})")

def tangram_fold_subprocess(Xs_train, Xx_train, Xs_target, W, fold_label):
    """1 fold = 1 tiến trình con (GPU sạch, mục 5 — vòng 19 OOM khi chạy Tangram mọi fold trong cùng tiến trình).
    Trao đổi qua .npz trong out20/b_io/: input = count gene huấn luyện (source, Xenium 10k Q) + count gene fold cần
    dự đoán (source); output = ma trận dự đoán hoặc ok=False kèm lỗi."""
    os.makedirs(f"{OUT}/b_io", exist_ok=True)
    inp, outp = f"{OUT}/b_io/tangram_{fold_label}_in.npz", f"{OUT}/b_io/tangram_{fold_label}_out.npz"
    np.savez(inp, Xs_train=Xs_train, Xx_train=Xx_train, Xs_target=Xs_target)
    r = subprocess.run([sys.executable, "tangram_worker.py", inp, outp], cwd=HERE, capture_output=True, text=True, timeout=2400)
    if r.returncode != 0 or not os.path.exists(outp):
        W.append(f"tangram {fold_label} lỗi tiến trình con, bỏ qua: {(r.stderr or '')[-800:]}"); return None
    d = np.load(outp, allow_pickle=True)
    if "ok" in d and not bool(d["ok"]):
        W.append(f"tangram {fold_label} lỗi trong tiến trình, bỏ qua: {d.get('error')}"); return None
    return d["pred"].astype(np.float32)

def process_pair_tangram(pair, PCTX, S, save, W):
    if "tangram" in S.get("Tangram", {}).get(pair, {}).get("agg", {}):
        log(f"[{pair}] Tangram: đã xong trong summary.json -> bỏ qua"); return
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Xs_lite, Xx_lite = pctx["Xs_lite"], pctx["Xx_lite"]
    Gs, sel, ix_all, is_all, Ys, Yx = scored_genes(pair, PCTX)
    n = len(Gs)
    rng0 = np.random.RandomState(0); folds = np.array_split(rng0.permutation(n), 5)
    Q = B.pick(dx["H"], R6.N_Q)

    METHODS = ("shuffle", "raw", "center", "default", "tangram")
    Sc = S["Tangram"].setdefault(pair, {"n_genes": int(n), "per_fold": {}})
    ALL_SPEAR, ALL_PEAR = ({m: np.full((n,), np.nan) for m in METHODS} for _ in range(2))
    for fi, fold_idx in enumerate(folds):
        keep_all = np.ones(len(ix_all), bool); keep_all[sel[fold_idx]] = False
        Fx_rep, Fs_rep = pca_cell_masked(Xx_lite, Xs_lite, ix_all, is_all, keep_all)
        center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
        mu = Fx_rep.mean(0)
        P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
        lab_aux = pctx["classes"][P_aux.argmax(1)]
        tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
        truth_Q = Yx[Q][:, fold_idx].toarray()

        def impute(space_src):
            nb = R6.knn_idx(Fx_rep[Q] - mu, space_src - mu, k=15)
            return R6.avg_rows(nb, Ys[:, fold_idx])
        pred = {"shuffle": R6.avg_rows(rng0.randint(0, Ys.shape[0], (len(Q), 15)), Ys[:, fold_idx]),
                "raw": impute(Fs_rep), "center": impute(center_map)}

        f_map = f"{OUT}/{pair}_tg_f{fi}_s0.npy"
        nrun = sorted(glob.glob(f"/kaggle/input/**/out19/{pair}_f{fi}_s0.npy", recursive=True))
        if RESUME and os.path.exists(f_map):
            Fmap = np.load(f_map).astype(np.float32); step(f"{pair} tangram fold{fi}: nạp lại map")
        elif nrun:
            Fmap = load_map17(nrun[0], len(Fs_rep)); np.save(f_map, Fmap.astype(np.float16))
        else:
            step(f"USHER Tangram-fold [{pair}] fold={fi} (thiếu map ở out19, chạy lại)")
            Fmap, info = R17.usher_variant(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], 0, "target", None, None)
            info.pop("cache_stats", None); np.save(f_map, Fmap.astype(np.float16))
        pred["default"] = impute(Fmap)

        hvg_local = R.hvg(R.lognorm(Xs_lite[:, is_all[keep_all]]), min(2000, int(keep_all.sum())))
        train_cols_s, train_cols_x = is_all[keep_all][hvg_local], ix_all[keep_all][hvg_local]
        src_pool = B.pick(np.arange(Xs_lite.shape[0]), 20000)
        # FIX (Claude chat): log CP10k (trên gene chung trừ fold) cho gene huấn luyện; gene fold = Ys (cùng thang với các phương pháp kNN)
        Ls = R.lognorm(Xs_lite[src_pool][:, is_all[keep_all]])[:, hvg_local].toarray()
        Lx = R.lognorm(Xx_lite[Q][:, ix_all[keep_all]])[:, hvg_local].toarray()
        pred["tangram"] = tangram_fold_subprocess(Ls.astype(np.float32), Lx.astype(np.float32),
                                                  Ys[src_pool][:, fold_idx].toarray().astype(np.float32), W, f"{pair}_f{fi}")

        for m, p_ in pred.items():
            if p_ is None: continue
            ALL_SPEAR[m][fold_idx] = R6.col_corr(R6.rank_cols(p_), R6.rank_cols(truth_Q))
            ALL_PEAR[m][fold_idx] = R6.col_corr(p_, truth_Q)
        Sc["per_fold"][f"fold{fi}"] = {"n_gene": int(len(fold_idx))}
        save(); log(f"[{pair}] Tangram fold {fi} xong"); del Fmap; gc.collect()

    def summ(m):
        return {"spearman_med": round(float(np.nanmedian(ALL_SPEAR[m])), 4), "pearson_med": round(float(np.nanmedian(ALL_PEAR[m])), 4),
                "n_gene": int(np.sum(~np.isnan(ALL_SPEAR[m])))}
    Sc["agg"] = {m: summ(m) for m in METHODS if np.any(~np.isnan(ALL_SPEAR[m]))}
    for a, b in (("default", "tangram"), ("raw", "tangram")):
        va, vb = ALL_SPEAR[a], ALL_SPEAR[b]
        both = ~np.isnan(va) & ~np.isnan(vb)
        if both.sum() >= 5:
            try: wp = float(wilcoxon(va[both], vb[both]).pvalue)
            except Exception: wp = None
            Sc[f"{a}_vs_{b}"] = {"n_better": int((va[both] > vb[both]).sum()), "n": int(both.sum()), "wilcoxon_p": wp}
    save()


# =============================================================================== Phần E — đánh giá trên MỌI nhóm
def all_groups(pair, PCTX):
    """Mọi nhãn mịn (16 nhóm SPATCH) có >= 20 tế bào ở CẢ H của source lẫn ref Xenium (mục 6b)."""
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    cs = pd.Series(ds["lab"][ds["H"]]).value_counts(); cx = pd.Series(dx["lab"][pctx["ref"]]).value_counts()
    return sorted(c for c in cs.index if c in cx.index and cs[c] >= 20 and cx[c] >= 20)

def fine16_label_comp(pair, S, save):
    """Như run17.get_niche_label_comp nhưng thành phần hàng xóm theo 16 NHÃN MỊN (không quy về thô) — dùng cho
    niche_bacc_fine16 (mục 6b). Cache riêng (labcomp16_*), không đụng cache của get_niche_label_comp."""
    cx_f, cs_f = f"{OUT}/labcomp16_{TARGET}.npy", f"{OUT}/labcomp16_{pair}.npy"
    if os.path.exists(cx_f) and os.path.exists(cs_f):
        return np.load(cs_f).astype(np.float32), np.load(cx_f).astype(np.float32)

    def comp_for(assay):
        p = R._find(R.FULL, R.FULL_REL[assay])
        lc = R.LABEL_COL.get(assay, "annotation")
        Xf, obsf, varf, xy = R.read_h5(p, ("high_quality", lc))
        hq = obsf["high_quality"].astype(int).to_numpy() == 1
        obsf, xy = obsf[hq], xy[hq]
        full_ids = obsf.index.astype(str)
        lab_full = obsf[lc].to_numpy().astype(str)
        onehot = np.stack([(lab_full == c).astype(np.float32) for c in FINE16], axis=1)
        obs_lite = lite_obs(assay); lite_ids = obs_lite.index.astype(str)
        pos = pd.Index(full_ids).get_indexer(lite_ids)
        valid = np.where(pos >= 0)[0]; q = pos[valid]
        k = min(R17.K_NICHE, len(xy) - 1)
        dist, idx = NearestNeighbors(n_neighbors=k + 1, n_jobs=4).fit(xy).kneighbors(xy[q])
        keep = idx != q[:, None]; keep[keep.sum(1) > k, -1] = False
        nb = idx[keep].reshape(len(q), k)
        comp = np.zeros((len(lite_ids), len(FINE16)), np.float32); comp[valid] = onehot[nb].mean(1)
        return comp
    comp_x, comp_s = comp_for(TARGET), comp_for(pair)
    np.save(cx_f, comp_x.astype(np.float16)); np.save(cs_f, comp_s.astype(np.float16))
    return comp_s, comp_x

def eval_xfer_all(pair, D, Fs_rep, Fx_used, name, Fmap, ALL, IMP, ref):
    return R9.eval_both(pair, D, Fs_rep, Fx_used, name, Fmap, ALL, IMP, ref)[1]["xfer_S"]

def recall_table(pair, D, Fx_used, Fmap, ALL, ref):
    ds, dx = D[pair], D[TARGET]
    mu = Fx_used.mean(0)
    qS = ds["H"][np.isin(ds["lab"][ds["H"]], ALL)]
    nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx_used[ref] - mu)).kneighbors(_unit(Fmap[qS] - mu), return_distance=False)
    pred = mode_rows(dx["lab"][ref][nn]); y = ds["lab"][qS]
    return {c: (round(float((pred[y == c] == c).mean()), 4) if (y == c).any() else None) for c in ALL}

def ceil_own_table(pair, D, Fs_rep, ALL):
    ds = D[pair]; mu = Fs_rep.mean(0)
    qS = ds["H"][np.isin(ds["lab"][ds["H"]], ALL)]
    nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fs_rep[ds["T"]] - mu)).kneighbors(_unit(Fs_rep[qS] - mu), return_distance=False)
    pred = mode_rows(ds["lab"][ds["T"]][nn]); y = ds["lab"][qS]
    return {c: (round(float((pred[y == c] == c).mean()), 4) if (y == c).any() else None) for c in ALL}

def rare_purity(pair, D, Fx_used, Fmap, ALL, ref):
    ds, dx = D[pair], D[TARGET]
    frac = pd.Series(ds["lab"][ds["H"]]).value_counts(normalize=True)
    rare = [c for c in ALL if frac.get(c, 0) < 0.02]
    if not rare: return None, rare
    mu = Fx_used.mean(0)
    qS = ds["H"][np.isin(ds["lab"][ds["H"]], rare)]
    nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx_used[ref] - mu)).kneighbors(_unit(Fmap[qS] - mu), return_distance=False)
    y = ds["lab"][qS]
    purity = (dx["lab"][ref][nn] == y[:, None]).mean(1)
    return round(float(pd.Series(purity, index=y).groupby(level=0).mean().mean()), 4), rare

def etrack(pair, D, PCTX, S, save, Fs_rep, Fx_used, name, Fmap, W):
    """FIX (Claude chat): plan mục 6b yêu cầu MỌI bảng M/T/B/K báo cả track ALL — bản trước chỉ tính ở chấm lại 16d."""
    if pair not in _E_CTX:
        pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
        ALL = all_groups(pair, PCTX)
        idx_s, idx_t = R11.build_eval_idx(ds, dx, ALL)
        c_s16, c_x16 = fine16_label_comp(pair, S, save)
        _E_CTX[pair] = (ALL, idx_s, idx_t, c_s16, c_x16, B.pick(ds["H"], R6.N_Q))
    ALL, idx_s, idx_t, c_s16, c_x16, Q = _E_CTX[pair]
    return eval_e_track(pair, D, Fs_rep, Fx_used, name, Fmap, ALL, idx_s, idx_t, PCTX[pair], c_s16, c_x16, KM16, Q, W)

def eval_e_track(pair, D, Fs_rep, Fx_used, name, Fmap, ALL, idx_src_all, idx_tgt_all, pctx, comp_s16, comp_x16, kmeans16, Q, W):
    xfer_all = eval_xfer_all(pair, D, Fs_rep, Fx_used, name, Fmap, ALL, pctx["IMP"], pctx["ref"])
    mp = R11.metrics_pack(Fmap, Fx_used, pctx["ds"]["lab"], pctx["dx"]["lab"], idx_src_all, idx_tgt_all)
    W.extend(f"{name}: {x}" for x in mp.pop("_warnings"))
    rp, rare = rare_purity(pair, D, Fx_used, Fmap, ALL, pctx["ref"])
    nm16 = niche_metrics(Fmap, Fx_used, pctx["ds"], pctx["dx"], pctx["ref"], comp_s16, comp_x16, kmeans16, Q)
    return {"xfer_all": xfer_all, "total_v2_all": mp.get("total_v2"), "iso_labels_all": mp.get("iso_labels"),
            "rare_purity": rp, "n_rare": len(rare), "niche_bacc_fine16": nm16["niche_bacc"]}

def process_pair_e16d(pair, D, F, PCTX, pca_cache, kmeans16, S, save, W, npy16_files):
    combo_ = next(c for c in COMBOS if c.split("/")[0] == pair); Se_ = S.get("E16d", {}).get(combo_, {})
    if all(k in Se_ for k in ("raw", "center", "mnn_ref", "harmony", "scanorama")) and all(f"s{sd}" in Se_.get("usher", {}) for sd in SEEDS):
        log(f"[{combo_}] Phần E: đã xong trong summary.json -> bỏ qua"); return
    """Chấm lại bảng 16d (raw, center, MNN-ref, Harmony, Scanorama, USHER-ST×3seed) trên track ALL (mục 6b). run16
    chỉ lưu map USHER -> MNN-ref/Harmony/Scanorama phải tính lại (R17.R16 = run16)."""
    combo = next(c for c in COMBOS if c.split("/")[0] == pair)
    pair_, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)
    ALL = all_groups(pair, PCTX)
    comp_s16, comp_x16 = fine16_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    idx_src_all, idx_tgt_all = R11.build_eval_idx(ds, dx, ALL)
    S.setdefault("E_groups", {})[pair] = ALL
    S.setdefault("E_ceil_own", {})[pair] = ceil_own_table(pair, D, Fs_rep, ALL)
    save(); log(f"[{pair}] ALL ({len(ALL)} nhóm): {ALL}")

    def ev(name, Fmap, Fx_used=Fx_rep):
        e0, w = eval_all16d(pair, D, Fs_rep, Fx_used, name, Fmap, pctx); W.extend(w)
        et = eval_e_track(pair, D, Fs_rep, Fx_used, name, Fmap, ALL, idx_src_all, idx_tgt_all, pctx, comp_s16, comp_x16, kmeans16, Q, W)
        rec = recall_table(pair, D, Fx_used, Fmap, ALL, pctx["ref"])
        return {**e0, **et, "recall": rec}

    Se = S["E16d"].setdefault(combo, {})
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    if "raw" not in Se: Se["raw"] = ev(f"{combo}/E/raw", Fs_rep); save()
    if "center" not in Se: Se["center"] = ev(f"{combo}/E/center", center_map); save()
    if "mnn_ref" not in Se:
        Fs_mnn, _ = R16.mnn_ref_correction(Fs_rep, Fx_rep, ds["T"]); Se["mnn_ref"] = ev(f"{combo}/E/mnn_ref", Fs_mnn); save()
    if "harmony" not in Se:
        try:
            Fx_h, Fs_h = R16.run_harmony(Fx_rep, Fs_rep); Se["harmony"] = ev(f"{combo}/E/harmony", Fs_h, Fx_h)
        except Exception as e: W.append(f"{combo} E harmony lỗi: {e!r}"); Se["harmony"] = None
        save()
    if "scanorama" not in Se:
        try:
            Fx_s, Fs_s = R16.run_scanorama(Fx_rep, Fs_rep); Se["scanorama"] = ev(f"{combo}/E/scanorama", Fs_s, Fx_s)
        except Exception as e: W.append(f"{combo} E scanorama lỗi: {e!r}"); Se["scanorama"] = None
        save()
    rl = Se.setdefault("usher", {})
    mu = Fx_rep.mean(0)
    P_aux = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], pctx["classes"], k=R17.K_AUX)
    lab_aux = pctx["classes"][P_aux.argmax(1)]
    tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
    for seed in SEEDS:
        key = f"s{seed}"
        if key in rl: continue
        name16 = f"{combo.replace('/', '_')}_usher_s{seed}"; f16 = npy16_files.get(name16)
        f20 = f"{OUT}/{combo.replace('/', '_')}_E16d_usher_s{seed}.npy"
        if f16: Fmap = load_map17(f16, len(Fs_rep))
        elif RESUME and os.path.exists(f20): Fmap = np.load(f20).astype(np.float32)
        else:
            step(f"USHER E-16d [{combo}] s{seed} (thiếu map ở out16*, chạy lại)")
            Fmap, info = R17.usher_variant(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed, "target", None, None)
            info.pop("cache_stats", None); np.save(f20, Fmap.astype(np.float16))
        rl[key] = ev(f"{combo}/E/usher/s{seed}", Fmap); save(); del Fmap; gc.collect()
    log(f"[{combo}] E-16d xong: raw xfer_all={Se['raw']['xfer_all']} center={Se['center']['xfer_all']}")


# =============================================================================== main
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    R19.import_prior_outputs()
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH: USHER-ST mặc định giai đoạn 1 (struct_cache=target, λ_id=0.3 linear) combos={COMBOS} "
           f"K_gate={K_GATE} K_niche8={K_NICHE_TYPES} K_fine16={K_FINE16} r_niche={R_NICHE}µm seeds={SEEDS} "
           f"λ_grid={LAMBDA_GRID} parts={sorted(PARTS)}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    # FIX 20e: tiếp nối từ summary.json của vòng 20 ở bất kỳ Input nào (nhận diện bằng "K_gate=" trong config; SMOKE tách riêng)
    cand = sorted(set(glob.glob("/kaggle/input/**/20_summary*.json", recursive=True)) | set(glob.glob("/kaggle/input/**/summary.json", recursive=True)))
    for fj in cand:   # summary.json của out16/17/19 cũng nằm trong Input nhưng bị loại nhờ kiểm tra "K_gate=" (chỉ vòng 20 có)
        try:
            Sj = json.load(open(fj)); cj = str(Sj.get("config", ""))
            if "K_gate=" in cj and (("| SMOKE" in cj) == SMOKE):
                R19._deep_merge(S, Sj); log(f"Tiếp nối từ {fj}")
        except Exception as e: log(f"bỏ qua {fj}: {e!r}")
    log(f"Tìm summary vòng 20 trong Input: {len(cand)} file ứng viên; các phần đã có: "
        f"{ {k: len(v) for k, v in S.items() if k in ('M', 'T', 'K', 'B', 'Tangram', 'E16d') and isinstance(v, dict)} }")
    S["config"] = cfg
    for k in ("D", "M", "T", "K", "B", "Tangram", "E16d"): S.setdefault(k, {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    needed_pairs = sorted({c.split("/")[0] for c in COMBOS} | set(PAIRS_C_ALL))
    D, F, PCTX = R18.setup_pairs(needed_pairs)
    pca_cache = {}
    comp_x0 = R17.get_niche_label_comp(needed_pairs[0], S, save)[1]
    kmeans8 = KMeans(n_clusters=K_NICHE_TYPES, random_state=0, n_init=10).fit(comp_x0)
    comp_x0_16 = fine16_label_comp(needed_pairs[0], S, save)[1]
    kmeans16 = KMeans(n_clusters=K_FINE16, random_state=0, n_init=10).fit(comp_x0_16)
    global KM16; KM16 = kmeans16
    log(f"KMeans niche8 cỡ cụm={np.bincount(kmeans8.labels_)} | niche16 cỡ cụm={np.bincount(kmeans16.labels_)}")

    try:
        run_parts(D, F, PCTX, pca_cache, kmeans8, kmeans16, S, save, W)
    except OutOfTime as e:
        W.append(f"HẾT NGÂN SÁCH THỜI GIAN ({TIME_BUDGET_S / 3600:.1f} giờ) trước bước: {e}. Kết quả đã lưu tới đây — "
                 f"Add output out20 của lượt này làm Input rồi chạy lại để làm tiếp phần còn thiếu.")
    S["warnings"] = list(dict.fromkeys(W)); save()   # FIX 20e: cảnh báo cũng nằm trong summary.json
    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 20")


def run_parts(D, F, PCTX, pca_cache, kmeans8, kmeans16, S, save, W):
    if "E" in PARTS:
        npy16_files = {os.path.splitext(os.path.basename(f))[0]: f for f in glob.glob("/kaggle/input/**/out16*/*.npy", recursive=True)}
        log(f"Phần E: tìm thấy {len(npy16_files)} map .npy của vòng 16")
        for pair in {c.split("/")[0] for c in COMBOS}:
            process_pair_e16d(pair, D, F, PCTX, pca_cache, kmeans16, S, save, W, npy16_files)

    if "M" in PARTS:
        npy_files17 = {os.path.splitext(os.path.basename(f))[0]: f for f in glob.glob("/kaggle/input/**/out17/*.npy", recursive=True)}
        for combo in COMBOS: process_combo_dm(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W, npy_files17)

    if "T" in PARTS:
        for combo in COMBOS: process_combo_t(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W)

    if "B" in PARTS:
        py = setup_venv_b(W)
        for pair in {c.split("/")[0] for c in COMBOS}: process_pair_b_embed(pair, D, PCTX, py, kmeans8, S, save, W)
        for pair in PAIRS_C_ALL: process_pair_tangram(pair, PCTX, S, save, W)

    if "K" in PARTS:
        for combo in COMBOS: process_combo_k(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W)


# =============================================================================== summary.md
def summary_md(S, W):
    L = ["# 20 — MoE theo niche (D+M), căn chỉnh ngữ cảnh (T), baseline còn thiếu (B), C_t landmark (K), mọi nhóm (E)", S["config"], ""]

    L.append("## Phần D — chẩn đoán phần dư (η², TB qua seed)")
    rows = []
    for combo, Sc in S.get("M", {}).items():
        ds_ = [Sc[f"s{s}"]["D"] for s in range(3) if f"s{s}" in Sc and "D" in Sc[f"s{s}"]]
        if not ds_: continue
        for k in ("eta_niche", "eta_random", "eta_niche_given_type"):
            m_, sd_ = _ms([d[k] for d in ds_])
            rows.append({"tổ_hợp": combo, "chỉ_số": k, "TB": round(m_, 5) if m_ is not None else None, "SD": round(sd_, 5) if sd_ is not None else None})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    n_d1 = 0
    for combo, Sc in S.get("M", {}).items():
        ds_ = [Sc[f"s{s}"]["D"] for s in range(3) if f"s{s}" in Sc and "D" in Sc[f"s{s}"]]
        if not ds_: continue
        m_n, _ = _ms([d["eta_niche"] for d in ds_]); m_r, _ = _ms([d["eta_random"] for d in ds_]); m_g, _ = _ms([d["eta_niche_given_type"] for d in ds_])
        ok = bool(m_n is not None and m_r is not None and (m_n >= m_r + 0.01) and (m_g is not None and m_g >= 0.005))
        n_d1 += ok
        L.append(f"- D1 {combo}: η²(niche)={round(m_n, 5) if m_n is not None else None} η²(random)={round(m_r, 5) if m_r is not None else None} "
                f"η²(niche|type)={round(m_g, 5) if m_g is not None else None} -> {'ĐẠT' if ok else 'không'}")
    L.append(f"- D1: {n_d1}/{len(S.get('M', {}))}")

    L.append("## Phần M — map MoE có cổng theo niche (TB±SD qua seed)")
    rows = []
    for combo, Sc in S.get("M", {}).items():
        for mname in ("default", "refit_global", "moe_niche", "moe_random", "moe_type"):
            vals = [Sc[f"s{s}"][mname] for s in range(3) if f"s{s}" in Sc and mname in Sc[f"s{s}"]]
            if not vals: continue
            r = {"tổ_hợp": combo, "map": mname}
            for c in ("xfer_main", "total_v2", "niche_bacc", "xfer_all", "total_v2_all"):
                m_, sd_ = _ms([v.get(c) for v in vals]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
            if "lambda" in vals[0]: r["λ"] = vals[0]["lambda"]
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]

    def ms_of(S_M, combo, mname, key):
        Sc = S_M.get(combo, {})
        vals = [Sc[f"s{s}"][mname].get(key) for s in range(3) if f"s{s}" in Sc and mname in Sc[f"s{s}"]]
        return _ms(vals)
    n_m0 = n_m1 = n_m2 = 0
    for combo in S.get("M", {}):
        xd, _ = ms_of(S["M"], combo, "default", "xfer_main"); td, _ = ms_of(S["M"], combo, "default", "total_v2")
        xr, _ = ms_of(S["M"], combo, "refit_global", "xfer_main"); tr_, _ = ms_of(S["M"], combo, "refit_global", "total_v2")
        m0 = bool(xd is not None and xr is not None and abs(xd - xr) <= 0.02 and td is not None and tr_ is not None and abs(td - tr_) <= 0.02)
        n_m0 += m0
        xn, sxn = ms_of(S["M"], combo, "moe_niche", "xfer_main"); tn, stn = ms_of(S["M"], combo, "moe_niche", "total_v2")
        r_x, r_t = beats(xn, sxn, xr, 0), beats(tn, stn, tr_, 0)
        m1 = bool((r_x and r_x["pass"] and not_much_worse(tn, tr_)) or (r_t and r_t["pass"] and not_much_worse(xn, xr)))
        n_m1 += m1
        xrand, sxrand = ms_of(S["M"], combo, "moe_random", "xfer_main"); trand, strand = ms_of(S["M"], combo, "moe_random", "total_v2")
        r_x2, r_t2 = beats(xn, sxn, xrand, sxrand), beats(tn, stn, trand, strand)
        m2 = bool((r_x2 and r_x2["pass"]) or (r_t2 and r_t2["pass"]))
        n_m2 += m2
        L.append(f"- {combo}: M0={'ĐẠT' if m0 else 'không'} M1={'ĐẠT' if m1 else 'không'} M2={'ĐẠT' if m2 else 'không'}")
    L.append(f"- M0: {n_m0}/{len(S.get('M', {}))} | M1: {n_m1}/{len(S.get('M', {}))} | M2: {n_m2}/{len(S.get('M', {}))}")

    L.append("## Phần T — căn chỉnh ngữ cảnh không gian")
    rows = []
    for combo, Sc in S.get("T", {}).items():
        vals_id = [Sc[k]["ctx_id"] for k in Sc if k.startswith("s") and "ctx_id" in Sc[k]]
        vals_full = [Sc[k]["ctx_full"] for k in Sc if k.startswith("s") and "ctx_full" in Sc[k]]
        if vals_id:
            r = {"tổ_hợp": combo, "biến_thể": "ctx_id"}
            for c in ("xfer_main", "total_v2", "niche_bacc", "xfer_all", "total_v2_all"):
                m_, sd_ = _ms([v.get(c) for v in vals_id]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
            rows.append(r)
        if vals_full:
            m_nb, sd_nb = _ms([v.get("niche_bacc") for v in vals_full]); m_xf, sd_xf = _ms([v.get("xfer_S") for v in vals_full])
            rows.append({"tổ_hợp": combo, "biến_thể": "ctx_full", "niche_bacc": f"{m_nb:.3f}±{sd_nb:.3f}" if m_nb is not None else "—",
                        "xfer_S": f"{m_xf:.3f}±{sd_xf:.3f}" if m_xf is not None else "—"})
        c25 = S.get("T_meta", {}).get(combo, {}).get("concat_25_niche_bacc")
        rows.append({"tổ_hợp": combo, "biến_thể": "concat_25(19)", "niche_bacc": c25})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    n_t1 = n_t2 = 0
    for combo in S.get("T", {}):
        Sc = S["T"][combo]
        vals_id = [Sc[k]["ctx_id"] for k in Sc if k.startswith("s") and "ctx_id" in Sc[k]]
        vals_full = [Sc[k]["ctx_full"] for k in Sc if k.startswith("s") and "ctx_full" in Sc[k]]
        m_d, sd_d = ms_of(S["M"], combo, "default", "xfer_main"); t_d, st_d = ms_of(S["M"], combo, "default", "total_v2")
        if vals_id:
            m_x, sd_x = _ms([v.get("xfer_main") for v in vals_id]); m_t, sd_t = _ms([v.get("total_v2") for v in vals_id])
            r_x, r_t = beats(m_x, sd_x, m_d, sd_d), beats(m_t, sd_t, t_d, st_d)
            t1 = bool((r_x and r_x["pass"] and not_much_worse(m_t, t_d)) or (r_t and r_t["pass"] and not_much_worse(m_x, m_d)))
            n_t1 += t1
        else: t1 = False
        c25 = S.get("T_meta", {}).get(combo, {}).get("concat_25_niche_bacc")
        if vals_full and c25 is not None:
            m_nb, sd_nb = _ms([v.get("niche_bacc") for v in vals_full])
            r2 = beats(m_nb, sd_nb, c25, 0); t2 = bool(r2 and r2["pass"])
            n_t2 += t2
        else: t2 = False
        L.append(f"- {combo}: T1={'ĐẠT' if t1 else 'không'} T2={'ĐẠT' if t2 else 'không'}")
    L.append(f"- T1: {n_t1}/{len(S.get('T', {}))} | T2: {n_t2}/{len(S.get('T', {}))}")

    L.append("## Phần B — Symphony, scArches, Tangram")
    rows = []
    for pair, Sb in S.get("B", {}).items():
        for m, v in Sb.items():
            if v: rows.append({"cặp": pair, "phương_pháp": m, "xfer_main": v.get("xfer_main"), "total_v2": v.get("total_v2"),
                              "xfer_all": v.get("xfer_all"), "total_v2_all": v.get("total_v2_all"), "niche_bacc": v.get("niche_bacc"), "version": v.get("version")})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    for pair, Sb in S.get("B", {}).items():
        combo = next((c for c in COMBOS if c.split("/")[0] == pair), None)
        d_, _ = ms_of(S["M"], combo, "default", "xfer_main") if combo else (None, None)
        dt, _ = ms_of(S["M"], combo, "default", "total_v2") if combo else (None, None)
        sdx = ms_of(S["M"], combo, "default", "xfer_main")[1] if combo else None
        sdt = ms_of(S["M"], combo, "default", "total_v2")[1] if combo else None
        for m in ("symphony", "scarches"):
            v = Sb.get(m)
            if not v or d_ is None: continue
            # FIX: dùng cmp3 (quy tắc mục 5 của 16) thay cho so sánh thô
            L.append(f"- B({m}) {pair}: default so {m}: total_v2 {R16.cmp3(dt, sdt, v.get('total_v2'))}, xfer_main {R16.cmp3(d_, sdx, v.get('xfer_main'))} "
                    f"| ALL: total_v2_all {v.get('total_v2_all')} xfer_all {v.get('xfer_all')}")
    rows = []
    for pair, Sc in S.get("Tangram", {}).items():
        agg = Sc.get("agg", {})
        for m, v in agg.items(): rows.append({"cặp": pair, "phương_pháp": m, **v})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    for pair, Sc in S.get("Tangram", {}).items():
        if "default_vs_tangram" in Sc: L.append(f"- I3 {pair}: default vs tangram = {Sc['default_vs_tangram']}")

    L.append("## Phần K — C_t cùng thước landmark")
    rows = []
    for combo, Sc in S.get("K", {}).items():
        vals = [Sc[k] for k in Sc if k.startswith("s")]
        if not vals: continue
        r = {"tổ_hợp": combo}
        for c in ("xfer_main", "total_v2"):
            m_, sd_ = _ms([v.get(c) for v in vals]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
        rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    n_k1 = 0
    for combo, Sc in S.get("K", {}).items():
        vals = [Sc[k] for k in Sc if k.startswith("s")]
        m_d, _ = ms_of(S["M"], combo, "default", "xfer_main"); t_d, _ = ms_of(S["M"], combo, "default", "total_v2")
        m_k, _ = _ms([v.get("xfer_main") for v in vals]); t_k, _ = _ms([v.get("total_v2") for v in vals])
        ok = bool(m_d is not None and m_k is not None and abs(m_d - m_k) < 0.02 and t_d is not None and t_k is not None and abs(t_d - t_k) < 0.02)
        n_k1 += ok
        L.append(f"- {combo}: |ct_landmark − default| xfer={round(abs(m_d - m_k), 4) if (m_d is not None and m_k is not None) else None} "
                f"tổng_v2={round(abs(t_d - t_k), 4) if (t_d is not None and t_k is not None) else None} -> {'không nhạy' if ok else 'LỆCH'}")
    L.append(f"- K1: {n_k1}/{len(S.get('K', {}))} tổ hợp không nhạy với thước C_t")

    L.append("## Phần E — đánh giá trên MỌI nhóm (bảng 16d chấm lại)")
    rows = []
    for combo, Se in S.get("E16d", {}).items():
        for m in ("raw", "center", "mnn_ref", "harmony", "scanorama"):
            v = Se.get(m)
            if v: rows.append({"cặp": combo, "map": m, "xfer_all": v.get("xfer_all"), "total_v2_all": v.get("total_v2_all"),
                              "iso_labels_all": v.get("iso_labels_all"), "rare_purity": v.get("rare_purity"), "niche_bacc_fine16": v.get("niche_bacc_fine16")})
        usher_vals = list(Se.get("usher", {}).values())
        if usher_vals:
            r = {"cặp": combo, "map": f"usher(n={len(usher_vals)})"}
            for c in ("xfer_all", "total_v2_all", "iso_labels_all", "rare_purity", "niche_bacc_fine16"):
                m_, sd_ = _ms([v.get(c) for v in usher_vals]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    n_e1 = 0
    for combo, Se in S.get("E16d", {}).items():
        usher_vals = list(Se.get("usher", {}).values())
        m_u, sd_u = _ms([v.get("total_v2_all") for v in usher_vals]); x_u, sx_u = _ms([v.get("xfer_all") for v in usher_vals])
        ok_all = True; detail = []
        pair_ = combo.split("/")[0]
        bases = {"center": Se.get("center"), "mnn_ref": Se.get("mnn_ref"),
                 "symphony": (S.get("B", {}).get(pair_) or {}).get("symphony"), "scarches": (S.get("B", {}).get(pair_) or {}).get("scarches")}
        for base, v in bases.items():   # FIX: chỉ baseline GIỮ NGUYÊN reference (Harmony đổi reference -> báo riêng), quy tắc cmp3
            if not v: continue
            c_t, c_x = R16.cmp3(m_u, sd_u, v.get("total_v2_all")), R16.cmp3(x_u, sx_u, v.get("xfer_all"))
            detail.append(f"{base}: tổng_v2_all {c_t}, xfer_all {c_x}")
            if c_t == "kém" or c_x == "kém": ok_all = False
        n_e1 += ok_all
        L.append(f"- E1 {combo}: {'; '.join(detail)} -> {'ĐẠT' if ok_all else 'không'}")
    L.append(f"- E1: {n_e1}/{len(S.get('E16d', {}))} cặp (đạt nếu ≥ 3/4)")
    L.append("- E2: xem cột rare_purity/niche_bacc_fine16 ở bảng trên; đọc mô tả, không có ngưỡng đạt/không cứng.")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run21.py
"""21 — ST theo định vị mới: (P) thông tin niche của embedding có còn SAU căn chỉnh hay chỉ mất khi đọc qua Xenium;
(R) niche làm bằng chứng danh tính cho NHÓM HIẾM (H17); (S) Symphony đúng cách (có bước sửa Harmony thật).

Tái dùng run20.py (niche_features, usher_variant_T, ridge_fit, eval_all16d, etrack, recall_table, rare_purity,
all_groups, run_baseline_subprocess, setup_venv_b, sub_split_250), run17.usher_variant, run18.niche_metrics. Nhãn
source CHỈ dùng để CHẤM (không dùng khi huấn luyện USHER/chọn M_aux/tgt — những bước đó dùng nhãn Xenium + aux dự đoán).

QUY TẮC BẮT BUỘC (CLAUDE.md + bài học 17-20): gán lại R17.OUT/R18.OUT/R19.OUT/R20.OUT = OUT (vòng này); beats(...)
trả dict (đọc ["pass"]), so baseline tất định dùng R16.cmp3 (trả chuỗi "vượt"/"ngang"/"kém"); tiếp nối từ
21_summary*.json/summary.json có dấu hiệu vòng 21 trong config, ở BẤT KỲ Input nào + kiểm tra "đã xong" ở ĐẦU mỗi
hàm; ngân sách thời gian TIME_BUDGET_S (mặc định 11 giờ, dừng SẠCH qua OutOfTime); SMOKE cắt đúng tập con qua
run19.lite_obs/load_map17; cảnh báo lưu vào summary.json; công tắc SMOKE đọc ở Cell 1.
"""
import os, sys, glob, json, gc, time, subprocess
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run20 as R20
R19, R18, R17 = R20.R19, R20.R18, R20.R17
R, B, M, C, R6, R7, R9, R11, R12, R8 = R20.R, R20.B, R20.M, R20.C, R20.R6, R20.R7, R20.R9, R20.R11, R20.R12, R20.R8
R16 = R20.R16
np, pd, sp, log, load_assay, _unit = R20.np, R20.pd, R20.sp, R20.log, R20.load_assay, R20._unit
_ms, beats, not_much_worse = R20._ms, R20.beats, R20.not_much_worse
cmp3 = R16.cmp3
SMOKE = R20.SMOKE
NearestNeighbors, KMeans, mode_rows, niche_metrics = R20.NearestNeighbors, R20.KMeans, R20.mode_rows, R20.niche_metrics
lite_obs, load_map17 = R20.lite_obs, R20.load_map17
niche_features, usher_variant_T, ridge_fit, eval_all16d = R20.niche_features, R20.usher_variant_T, R20.ridge_fit, R20.eval_all16d
etrack, recall_table, rare_purity, all_groups, sub_split_250 = R20.etrack, R20.recall_table, R20.rare_purity, R20.all_groups, R20.sub_split_250
setup_venv_b, run_baseline_subprocess = R20.setup_venv_b, R20.run_baseline_subprocess
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
HERE = os.path.dirname(os.path.abspath(__file__))

OUT = os.environ.get("OUT_DIR_21", "/kaggle/working/out21" + ("_smoke" if SMOKE else ""))
TARGET = R20.TARGET
R17.OUT, R18.OUT, R19.OUT, R20.OUT = OUT, OUT, OUT, OUT   # bài học 17-20: hàm cache theo OUT của các module đó
COMBOS = R20.COMBOS   # 4 tổ hợp mặc định
K_NICHE_TYPES = R20.K_NICHE_TYPES   # 8, loại niche thô (nhãn thật, chỉ để CHẤM)
SEEDS = [0] if SMOKE else [0, 1, 2]
RARE_FLOOR = 200
PARTS = {p.strip() for p in os.environ.get("ONLY_PART", "P,R,S").split(",") if p.strip()}
if "all" in PARTS: PARTS = {"P", "R", "S"}
RESUME = os.environ.get("RESUME", "1") == "1"
T0 = time.time()
TIME_BUDGET_S = float(os.environ.get("TIME_BUDGET_S", 11 * 3600))
class OutOfTime(Exception): pass
def step(msg):
    log(f"    [{time.time() - T0:6.0f}s] {msg}")
    if time.time() - T0 > TIME_BUDGET_S: raise OutOfTime(msg)


# =============================================================================== Phần P — bảo toàn thông tin niche
def lr_niche_bacc(X_train, y_train, X_test, y_test, keep):
    sc = StandardScaler().fit(X_train)
    clf = LogisticRegression(max_iter=2000, C=1.0, n_jobs=4).fit(sc.transform(X_train), y_train)
    return M.bacc_on(y_test, clf.predict(sc.transform(X_test)), keep)

def process_combo_p(combo, F, PCTX, pca_cache, kmeans8, S, save, W, npy_files17):
    Sc = S["P"].setdefault(combo, {})
    if "raw" in Sc and "center" in Sc and all(f"default_s{s}" in Sc for s in SEEDS):
        log(f"[{combo}] Phần P: đã xong -> bỏ qua"); return
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep = R17.R16_rep_features(pair, rep, F, PCTX, pca_cache)
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    type_true_q = kmeans8.predict(comp_s[Q])
    keep = pd.Series(type_true_q).value_counts(); keep = keep[keep >= 50].index.to_numpy()
    y_T = kmeans8.predict(comp_s[ds["T"]])
    lr_src = lr_niche_bacc(Fs_rep[ds["T"]], y_T, Fs_rep[Q], type_true_q, keep)

    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    for variant, Fmap in [("raw", Fs_rep), ("center", center_map)]:
        if variant in Sc: continue
        knn_xen = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)["niche_bacc"]
        lr_map = lr_niche_bacc(Fmap[ds["T"]], y_T, Fmap[Q], type_true_q, keep)
        Sc[variant] = {"lr_src": lr_src, "lr_map": lr_map, "knn_xen": knn_xen}; save()
    for seed in SEEDS:
        key = f"default_s{seed}"
        if key in Sc: continue
        name = f"{combo.replace('/', '_')}_default_s{seed}"; f17 = npy_files17.get(name)
        if not f17: W.append(f"Phần P {combo} {key}: thiếu out17/{name}.npy, bỏ qua"); continue
        Fmap = load_map17(f17, len(Fs_rep))
        knn_xen = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)["niche_bacc"]
        lr_map = lr_niche_bacc(Fmap[ds["T"]], y_T, Fmap[Q], type_true_q, keep)
        Wm, bm, r2 = R19.recover_wb(Fs_rep, Fmap)
        Sc[key] = {"lr_src": lr_src, "lr_map": lr_map, "knn_xen": knn_xen, "cond_W": round(float(np.linalg.cond(Wm)), 2), "r2_W": r2}
        save(); log(f"[{combo}] {key}: lr_src={lr_src} lr_map={lr_map} knn_xen={knn_xen} cond(W)={Sc[key]['cond_W']}"); del Fmap; gc.collect()


# =============================================================================== Phần R — niche làm bằng chứng nhóm hiếm
def aux_center(Fs_rep, Fx_rep, ds, dx, classes, k):
    center_map = M.base_center(Fs_rep, Fx_rep, ds["T"])
    mu = Fx_rep.mean(0)
    P = R7.knn_aux(center_map - mu, Fx_rep - mu, dx["lab"], classes, k=k)
    return classes[P.argmax(1)], P, center_map

def aux_niche_vote(center_map, Fs_bar, Fx_rep, Fx_bar, dx, classes, k):
    """M_aux = bỏ phiếu 15-NN trong [unit(center − μx), unit(x̄^25 − μ̄x)] (mục 4, hai khối trọng số bằng nhau)."""
    mu_x, mu_n = Fx_rep.mean(0), Fx_bar.mean(0)
    Xs = np.hstack([_unit(center_map - mu_x), _unit(Fs_bar - mu_n)]).astype(np.float32)
    Xx = np.hstack([_unit(Fx_rep - mu_x), _unit(Fx_bar - mu_n)]).astype(np.float32)
    P = R7.knn_aux(Xs, Xx, dx["lab"], classes, k=k)
    return classes[P.argmax(1)], P

def strat_target_floor(lab_src_T, lab_x, floor=RARE_FLOOR, n_tgt=None, seed=1):
    """Như C.strat_target nhưng đảm bảo SÀN min(floor, số có sẵn) cho MỖI nhãn, tổng vẫn ~N_TGT (mục 4, tgt_floor) —
    cách đọc của Claude Code: co phần VƯỢT sàn theo tỉ lệ nếu tổng vượt n_tgt, không co phần dưới sàn."""
    n_tgt = n_tgt or M.N_TGT
    rng = np.random.RandomState(seed)
    comp = pd.Series(lab_src_T).value_counts(normalize=True)
    alloc = {}
    for c in np.unique(lab_x):   # FIX 21b: sàn cho MỌI nhãn Xenium, kể cả nhãn aux source không đoán ra
        f = float(comp.get(c, 0.0))
        pool = np.where(lab_x == c)[0]
        if len(pool) == 0: continue
        alloc[c] = min(len(pool), max(min(floor, len(pool)), int(round(f * n_tgt))))
    total = sum(alloc.values())
    if total > n_tgt:
        scale = n_tgt / total
        for c in alloc:
            pool_n = int((lab_x == c).sum())
            alloc[c] = max(min(floor, pool_n), int(round(alloc[c] * scale)))
    tgt = [rng.choice(np.where(lab_x == c)[0], n_c, replace=False) for c, n_c in alloc.items() if n_c > 0]
    return np.sort(np.concatenate(tgt))

def aux_quality(ds, lab_aux, ALL, rare):
    y, p = ds["lab"][ds["H"]], lab_aux[ds["H"]]
    m = np.isin(y, ALL)
    aux_bacc = M.bacc_on(y[m], p[m], ALL)
    recs = [(p[y == c] == c).mean() for c in rare if (y == c).any()]
    return aux_bacc, (round(float(np.mean(recs)), 4) if recs else None)

def fit_moe_gate(Xt, Yt, m1h, m2h, gate_t, gate_all, Theta_glob, X_all, out_dim, lam_grid=(0.1, 1, 10, 100)):
    """Chép đúng vòng chọn λ + fit chuyên gia của run20.process_combo_dm (không refactor run20.py) — dùng cho
    moe_type/moe_type_niche (mục 4, CPU, không chạy USHER thêm)."""
    n_bar = len(Xt) / max(1, len(np.unique(gate_t)))
    lam_best, best_err = None, np.inf
    for lam_mult in lam_grid:
        lam = lam_mult * n_bar
        Theta_1 = ridge_fit(Xt[m1h], Yt[m1h], 1e-3)
        err = 0.0
        g1 = gate_t[m1h]
        for k in np.unique(g1):
            mk = g1 == k
            Th_k = ridge_fit(Xt[m1h][mk], Yt[m1h][mk], lam, Theta_1)
            mk2 = gate_t[m2h] == k
            if mk2.any(): err += float(((Yt[m2h][mk2] - Xt[m2h][mk2] @ Th_k) ** 2).sum())
        if err < best_err: best_err, lam_best = err, lam
    experts = {k: ridge_fit(Xt[gate_t == k], Yt[gate_t == k], lam_best, Theta_glob) for k in np.unique(gate_t)}
    out = np.zeros((X_all.shape[0], out_dim), np.float32)
    for k, Th_k in experts.items():
        mk = gate_all == k; out[mk] = (X_all[mk] @ Th_k).astype(np.float32)
    miss = ~np.isin(gate_all, list(experts))
    if miss.any(): out[miss] = (X_all[miss] @ Theta_glob).astype(np.float32)
    return out, float(lam_best)

def process_combo_r(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W, npy_files17):
    Sc = S["R"].setdefault(combo, {})
    need = ("default", "aux_niche", "aux_niche_shuf", "tgt_floor", "moe_type", "moe_type_niche")
    if all(all(v in Sc.get(f"s{s}", {}) for v in need) for s in SEEDS):
        log(f"[{combo}] Phần R: đã xong -> bỏ qua"); return
    pair, rep = combo.split("/")
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs_rep, Fx_rep, Fs_bar, Fx_bar = niche_features(combo, PCTX, F, pca_cache, S, save)
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    ALL = all_groups(pair, PCTX)
    frac = pd.Series(ds["lab"][ds["H"]]).value_counts(normalize=True)
    rare = [c for c in ALL if frac.get(c, 0) < 0.02]
    S.setdefault("R_rare", {})[combo] = rare; save()

    lab_def, P_def, center_map = aux_center(Fs_rep, Fx_rep, ds, dx, pctx["classes"], R17.K_AUX)
    lab_niche, P_niche = aux_niche_vote(center_map, Fs_bar, Fx_rep, Fx_bar, dx, pctx["classes"], R17.K_AUX)
    rng0 = np.random.RandomState(0); perm = rng0.permutation(len(Fs_bar))
    lab_niche_shuf, P_shuf = aux_niche_vote(center_map, Fs_bar[perm], Fx_rep, Fx_bar, dx, pctx["classes"], R17.K_AUX)
    aux_bacc_def, aux_rare_def = aux_quality(ds, lab_def, ALL, rare)
    aux_bacc_niche, aux_rare_niche = aux_quality(ds, lab_niche, ALL, rare)
    S.setdefault("R_aux", {})[combo] = {"default": {"bacc": aux_bacc_def, "rare_recall": aux_rare_def},
                                        "niche": {"bacc": aux_bacc_niche, "rare_recall": aux_rare_niche}}
    save(); log(f"[{combo}] aux: default bacc={aux_bacc_def} rare={aux_rare_def} | niche bacc={aux_bacc_niche} rare={aux_rare_niche}")

    obs_s = lite_obs(pair); xy_T = obs_s[["x", "y"]].to_numpy()[ds["T"]]
    m1, m2 = sub_split_250(xy_T); T1, T2 = ds["T"][m1], ds["T"][m2]
    cnt = pd.Series(lab_def[ds["T"]]).value_counts(); rare_lab_def = set(cnt[cnt < 200].index)
    cnt_n = pd.Series(lab_niche[ds["T"]]).value_counts(); rare_lab_niche = set(cnt_n[cnt_n < 200].index)
    type_def_all = np.where(np.isin(lab_def, list(rare_lab_def)), "other", lab_def)
    type_niche_all = np.where(np.isin(lab_niche, list(rare_lab_niche)), "other", lab_niche)

    VARIANTS = {
        "default": (lab_def, P_def),
        "aux_niche": (lab_niche, P_niche),
        "aux_niche_shuf": (lab_niche_shuf, P_shuf),
        "tgt_floor": (lab_def, P_def),
    }
    for seed in SEEDS:
        key = f"s{seed}"
        Sk = Sc.setdefault(key, {})
        ybar_def = has_def = None
        for vname, (lab_aux, P_aux) in VARIANTS.items():
            f_map = f"{OUT}/{combo.replace('/', '_')}_{vname}_s{seed}.npy"
            f_ybar = f"{OUT}/{combo.replace('/', '_')}_{vname}_ybar_s{seed}.npz"
            if vname in Sk:
                if vname == "default" and os.path.exists(f_ybar):   # cần ybar/has_tgt cho moe_type dù default đã chấm xong
                    d_ = np.load(f_ybar); ybar_def, has_def = d_["ybar"], d_["has_tgt"]
                continue
            if vname == "tgt_floor":
                tgt = strat_target_floor(lab_aux[ds["T"]], dx["lab"])
            else:
                tgt = C.strat_target(lab_aux[ds["T"]], dx["lab"])
            if RESUME and os.path.exists(f_map) and os.path.exists(f_ybar):
                step(f"{combo} {vname} s{seed}: nạp lại map+ybar")
                Fmap = load_map17(f_map, len(Fs_rep)); d_ = np.load(f_ybar); ybar, has_tgt = d_["ybar"], d_["has_tgt"]
            else:
                step(f"USHER [{combo}] {vname} s{seed}")
                Fmap, info, ybar, has_tgt = usher_variant_T(Fs_rep, ds["T"], P_aux, Fx_rep, tgt, dx["lab"], pctx["classes"], seed, "target", None, None)
                info.pop("cache_stats", None); np.save(f_map, Fmap.astype(np.float16)); np.savez(f_ybar, ybar=ybar, has_tgt=has_tgt)
            ev, w = eval_all16d(pair, D, Fs_rep, Fx_rep, f"{combo}/{vname}/s{seed}", Fmap, pctx); W.extend(w)
            nm = niche_metrics(Fmap, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
            et = etrack(pair, D, PCTX, S, save, Fs_rep, Fx_rep, f"{combo}/{vname}/s{seed}", Fmap, W)
            rec = recall_table(pair, D, Fx_rep, Fmap, rare, pctx["ref"]) if rare else {}
            Sk[vname] = {**ev, **nm, **et, "recall_rare": rec}
            save(); log(f"[{combo}] {key} {vname}: xfer_main={ev['xfer_main']} rare_purity={et.get('rare_purity')}"); del Fmap; gc.collect()
            if vname == "default": ybar_def, has_def = ybar, has_tgt

        if ybar_def is None:
            W.append(f"{combo} {key}: thiếu ybar của default (file bị xoá?) -> bỏ moe_type/moe_type_niche"); continue
        # moe_type / moe_type_niche (CPU, trên đích barycentric của default — không chạy USHER thêm)
        for mname, type_all in (("moe_type", type_def_all), ("moe_type_niche", type_niche_all)):
            if mname in Sk: continue
            Xhas = ds["T"][has_def]
            m1h, m2h = np.isin(Xhas, T1), np.isin(Xhas, T2)
            Xt = np.hstack([Fs_rep[Xhas], np.ones((len(Xhas), 1), np.float32)]).astype(np.float64)
            Yt = ybar_def[has_def].astype(np.float64)
            Theta_glob = ridge_fit(Xt, Yt, 1e-3)
            X_all = np.hstack([Fs_rep, np.ones((len(Fs_rep), 1), np.float32)]).astype(np.float64)
            gate_t = type_all[Xhas]
            out_g, lam_best = fit_moe_gate(Xt, Yt, m1h, m2h, gate_t, type_all, Theta_glob, X_all, Fx_rep.shape[1])
            ev, w = eval_all16d(pair, D, Fs_rep, Fx_rep, f"{combo}/{mname}/s{seed}", out_g, pctx); W.extend(w)
            nm = niche_metrics(out_g, Fx_rep, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
            et = etrack(pair, D, PCTX, S, save, Fs_rep, Fx_rep, f"{combo}/{mname}/s{seed}", out_g, W)
            rec = recall_table(pair, D, Fx_rep, out_g, rare, pctx["ref"]) if rare else {}
            Sk[mname] = {**ev, **nm, **et, "recall_rare": rec, "lambda": round(lam_best, 2)}
            save(); log(f"[{combo}] {key} {mname}: λ={lam_best:.1f} rare_purity={Sk[mname].get('rare_purity')}")


# =============================================================================== Phần S — Symphony đúng cách
def process_pair_s(pair, D, PCTX, py, kmeans8, S, save, W):
    Sb = S["S"].setdefault(pair, {})
    if Sb.get("symphony_h") is not None and Sb.get("symphony") is not None:
        log(f"[{pair}] Phần S: đã xong -> bỏ qua"); return
    pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    ix, is_, G = R8.gene_cols(pctx["varx"], pctx["vars_"])
    Xx, Xs = pctx["Xx_lite"][:, ix].toarray(), pctx["Xs_lite"][:, is_].toarray()
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    for method in ("symphony", "symphony_h"):
        if Sb.get(method) is not None: continue
        step(f"Phần S [{pair}] {method} (venv riêng)")
        res = run_baseline_subprocess(py, method, Xx, Xs, W)
        if res is None: Sb[method] = None; save(); continue
        Fx_b, Fs_b, ver = res
        ev, w = eval_all16d(pair, D, Fs_b, Fx_b, f"{pair}/{method}", Fs_b, pctx); W.extend(w)
        nm = niche_metrics(Fs_b, Fx_b, ds, dx, pctx["ref"], comp_s, comp_x, kmeans8, Q)
        et = etrack(pair, D, PCTX, S, save, Fs_b, Fx_b, f"{pair}/{method}", Fs_b, W)
        Sb[method] = {**ev, **nm, **et, "version": ver}
        save(); log(f"[{pair}] {method}: xfer_main={ev['xfer_main']} tổng_v2={ev.get('total_v2')} (v{ver})")
    if Sb.get("symphony") and Sb.get("symphony_h"):
        d_xfer = abs(Sb["symphony_h"]["xfer_main"] - Sb["symphony"]["xfer_main"])
        S.setdefault("S_check", {})[pair] = round(d_xfer, 4)
        if d_xfer < 0.005: W.append(f"{pair}: |symphony_h - symphony| xfer_main = {d_xfer:.4f} < 0.005 -> bước sửa Symphony có thể không chạy")
        save()


# =============================================================================== main
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    R19.import_prior_outputs()
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH vòng 21: USHER-ST mặc định giai đoạn 1 combos={COMBOS} K_niche8={K_NICHE_TYPES} "
           f"rare_floor={RARE_FLOOR} seeds={SEEDS} parts={sorted(PARTS)}" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    cand = sorted(set(glob.glob("/kaggle/input/**/21_summary*.json", recursive=True)) | set(glob.glob("/kaggle/input/**/summary.json", recursive=True)))
    for fj in cand:
        try:
            Sj = json.load(open(fj)); cj = str(Sj.get("config", ""))
            if "rare_floor=" in cj and (("| SMOKE" in cj) == SMOKE):
                R19._deep_merge(S, Sj); log(f"Tiếp nối từ {fj}")
        except Exception as e: log(f"bỏ qua {fj}: {e!r}")
    log(f"Tìm summary vòng 21 trong Input: {len(cand)} file ứng viên; phần đã có: {{k: len(v) for k, v in S.items() if k in ('P','R','S') and isinstance(v, dict)}}")
    S["config"] = cfg
    for k in ("P", "R", "S"): S.setdefault(k, {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    needed_pairs = sorted({c.split("/")[0] for c in COMBOS})
    D, F, PCTX = R18.setup_pairs(needed_pairs)
    pca_cache = {}
    comp_x0 = R17.get_niche_label_comp(needed_pairs[0], S, save)[1]
    kmeans8 = KMeans(n_clusters=K_NICHE_TYPES, random_state=0, n_init=10).fit(comp_x0)
    log(f"KMeans niche8 cỡ cụm={np.bincount(kmeans8.labels_)}")
    # FIX 21a (Claude chat): R20.etrack dùng R20.KM16 (chỉ run20.main đặt) -> đặt tại đây, như run20
    comp_x0_16 = R20.fine16_label_comp(needed_pairs[0], S, save)[1]
    R20.KM16 = KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(comp_x0_16)

    try:
        run_parts(D, F, PCTX, pca_cache, kmeans8, S, save, W)
    except OutOfTime as e:
        W.append(f"HẾT NGÂN SÁCH THỜI GIAN ({TIME_BUDGET_S / 3600:.1f} giờ) trước bước: {e}. Kết quả đã lưu tới đây — "
                 f"Add output out21 của lượt này làm Input rồi chạy lại để làm tiếp phần còn thiếu.")
    S["warnings"] = list(dict.fromkeys(W)); save()
    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 21")


def run_parts(D, F, PCTX, pca_cache, kmeans8, S, save, W):
    if "P" in PARTS:
        npy_files17 = {os.path.splitext(os.path.basename(f))[0]: f for f in glob.glob("/kaggle/input/**/out17/*.npy", recursive=True)}
        for combo in COMBOS: process_combo_p(combo, F, PCTX, pca_cache, kmeans8, S, save, W, npy_files17)
    if "R" in PARTS:
        npy_files17 = {os.path.splitext(os.path.basename(f))[0]: f for f in glob.glob("/kaggle/input/**/out17/*.npy", recursive=True)}
        for combo in COMBOS: process_combo_r(combo, D, F, PCTX, pca_cache, kmeans8, S, save, W, npy_files17)
    if "S" in PARTS:
        py = setup_venv_b(W)
        for pair in {c.split("/")[0] for c in COMBOS}: process_pair_s(pair, D, PCTX, py, kmeans8, S, save, W)


# =============================================================================== summary.md
def summary_md(S, W):
    L = ["# 21 — bảo toàn niche (P), niche cho nhóm hiếm (R), Symphony đúng cách (S)", S["config"], ""]

    L.append("## Phần P — thông tin niche trong embedding (TB±SD qua seed cho default)")
    rows = []
    for combo, Sc in S.get("P", {}).items():
        for variant in ("raw", "center"):
            if variant in Sc: rows.append({"tổ_hợp": combo, "map": variant, **Sc[variant]})
        vals = [Sc[f"default_s{s}"] for s in SEEDS if f"default_s{s}" in Sc]
        if vals:
            r = {"tổ_hợp": combo, "map": f"default(n={len(vals)})"}
            for c in ("lr_src", "lr_map", "knn_xen", "cond_W"):
                m_, sd_ = _ms([v.get(c) for v in vals]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    n_p1 = 0
    for combo, Sc in S.get("P", {}).items():
        vals = [Sc[f"default_s{s}"] for s in SEEDS if f"default_s{s}" in Sc]
        if not vals: continue
        m_map, _ = _ms([v["lr_map"] for v in vals]); src = vals[0]["lr_src"]
        ok = bool(m_map is not None and src is not None and abs(m_map - src) <= 0.01)
        n_p1 += ok
        L.append(f"- P1 {combo}: |lr_map − lr_src| = {round(abs(m_map - src), 4) if (m_map is not None and src is not None) else None} -> {'ĐẠT' if ok else 'không'}")
    L.append(f"- P1: {n_p1}/{len(S.get('P', {}))} tổ hợp (ĐẠT -> thông tin niche được bảo toàn trong embedding; mất chỉ khi đọc qua tham chiếu Xenium)")

    L.append("## Phần R — niche làm bằng chứng danh tính cho nhóm hiếm (TB±SD qua seed)")
    rows = []
    VNAMES = ("default", "aux_niche", "aux_niche_shuf", "tgt_floor", "moe_type", "moe_type_niche")
    for combo, Sc in S.get("R", {}).items():
        for vname in VNAMES:
            vals = [Sc[f"s{s}"][vname] for s in SEEDS if f"s{s}" in Sc and vname in Sc[f"s{s}"]]
            if not vals: continue
            r = {"tổ_hợp": combo, "biến_thể": vname}
            for c in ("xfer_main", "total_v2", "xfer_all", "total_v2_all", "rare_purity"):
                m_, sd_ = _ms([v.get(c) for v in vals]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
            if "lambda" in vals[0]: r["λ"] = vals[0]["lambda"]
            rows.append(r)
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    L.append("Chất lượng nhãn aux (độ chính xác trên H, nhóm ALL; recall nhóm hiếm):")
    for combo, aux in S.get("R_aux", {}).items():
        L.append(f"- {combo}: default bacc={aux['default']['bacc']} rare_recall={aux['default']['rare_recall']} | "
                f"niche bacc={aux['niche']['bacc']} rare_recall={aux['niche']['rare_recall']}")

    def ms_of_r(S_R, combo, vname, key):
        Sc = S_R.get(combo, {})
        vals = [Sc[f"s{s}"][vname].get(key) for s in SEEDS if f"s{s}" in Sc and vname in Sc[f"s{s}"]]
        return _ms(vals)
    n_r1 = {"aux_niche": 0, "tgt_floor": 0, "moe_type_niche": 0}; n_r2 = n_r4 = 0
    for combo in S.get("R", {}):
        rp_d, sd_d = ms_of_r(S["R"], combo, "default", "rare_purity")
        t_d, st_d = ms_of_r(S["R"], combo, "default", "total_v2"); x_d, sx_d = ms_of_r(S["R"], combo, "default", "xfer_main")
        for v in ("aux_niche", "tgt_floor", "moe_type_niche"):
            rp_v, sd_v = ms_of_r(S["R"], combo, v, "rare_purity"); t_v, _ = ms_of_r(S["R"], combo, v, "total_v2"); x_v, _ = ms_of_r(S["R"], combo, v, "xfer_main")
            r1 = beats(rp_v, sd_v, rp_d, sd_d)
            ok = bool(r1 and r1["pass"] and not_much_worse(t_v, t_d) and not_much_worse(x_v, x_d))
            n_r1[v] += ok
            L.append(f"- R1 {combo}/{v}: rare_purity={round(rp_v, 4) if rp_v is not None else None} vs default={round(rp_d, 4) if rp_d is not None else None} -> {'ĐẠT' if ok else 'không'}")
        rp_n, sd_n = ms_of_r(S["R"], combo, "aux_niche", "rare_purity"); rp_s, sd_s = ms_of_r(S["R"], combo, "aux_niche_shuf", "rare_purity")
        x_n, sx_n = ms_of_r(S["R"], combo, "aux_niche", "xfer_all"); x_s, sx_s = ms_of_r(S["R"], combo, "aux_niche_shuf", "xfer_all")
        r2a, r2b = beats(rp_n, sd_n, rp_s, sd_s), beats(x_n, sx_n, x_s, sx_s)
        ok2 = bool((r2a and r2a["pass"]) or (r2b and r2b["pass"])); n_r2 += ok2
        L.append(f"- R2 {combo}: aux_niche vs aux_niche_shuf rare_purity Δ={round(rp_n - rp_s, 4) if (rp_n is not None and rp_s is not None) else None} -> {'ĐẠT' if ok2 else 'không'}")
        rp_f, sd_f = ms_of_r(S["R"], combo, "tgt_floor", "rare_purity"); x_f, sx_f = ms_of_r(S["R"], combo, "tgt_floor", "xfer_all")
        c_rp, c_xf = cmp3(rp_n, sd_n, rp_f), cmp3(x_n, sx_n, x_f)
        L.append(f"- R3 {combo}: aux_niche so tgt_floor -> rare_purity:{c_rp} xfer_all:{c_xf}")
        rp_mt, sd_mt = ms_of_r(S["R"], combo, "moe_type", "rare_purity"); x_mt, sx_mt = ms_of_r(S["R"], combo, "moe_type", "xfer_all")
        rp_mtn, sd_mtn = ms_of_r(S["R"], combo, "moe_type_niche", "rare_purity"); x_mtn, sx_mtn = ms_of_r(S["R"], combo, "moe_type_niche", "xfer_all")
        c_rp2, c_xf2 = cmp3(rp_mtn, sd_mtn, rp_mt), cmp3(x_mtn, sx_mtn, x_mt)
        n_r4 += (c_rp2 == "vượt" or c_xf2 == "vượt")
        L.append(f"- R4 {combo}: moe_type_niche so moe_type -> rare_purity:{c_rp2} xfer_all:{c_xf2}")
    L.append(f"- R1: aux_niche {n_r1['aux_niche']}/{len(S.get('R', {}))} | tgt_floor {n_r1['tgt_floor']}/{len(S.get('R', {}))} | moe_type_niche {n_r1['moe_type_niche']}/{len(S.get('R', {}))}")
    L.append(f"- R2: {n_r2}/{len(S.get('R', {}))} | R4 (moe_type_niche vượt moe_type ở ≥1 chỉ số): {n_r4}/{len(S.get('R', {}))}")

    L.append("## Phần S — Symphony đúng cách")
    rows = []
    for pair, Sb in S.get("S", {}).items():
        for m, v in Sb.items():
            if isinstance(v, dict): rows.append({"cặp": pair, "phương_pháp": m, "xfer_main": v.get("xfer_main"),
                                                  "total_v2": v.get("total_v2"), "xfer_all": v.get("xfer_all"), "version": v.get("version")})
    if rows: L += ["```", pd.DataFrame(rows).to_string(index=False), "```"]
    for pair, d_xfer in S.get("S_check", {}).items():
        L.append(f"- Kiểm tra hợp lệ {pair}: |symphony_h − symphony| xfer_main = {d_xfer}" + (" (CẢNH BÁO: có thể không chạy bước sửa)" if d_xfer < 0.005 else ""))
    for combo in COMBOS:
        pair = combo.split("/")[0]
        Sb = S.get("S", {}).get(pair, {}); sh = Sb.get("symphony_h")
        if not sh: continue
        dv, sdv = ms_of_r(S["R"], combo, "default", "total_v2"); xv, sxv = ms_of_r(S["R"], combo, "default", "xfer_main")
        tva, stva = ms_of_r(S["R"], combo, "default", "total_v2_all"); xa, sxa = ms_of_r(S["R"], combo, "default", "xfer_all")
        if dv is None: continue
        L.append(f"- S1 {combo}: default so symphony_h -> total_v2:{cmp3(dv, sdv, sh.get('total_v2'))} xfer_main:{cmp3(xv, sxv, sh.get('xfer_main'))} "
                 f"total_v2_all:{cmp3(tva, stva, sh.get('total_v2_all'))} xfer_all:{cmp3(xa, sxa, sh.get('xfer_all'))}")

    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run22.py
"""22 — CHỐT CẤU HÌNH USHER-ST (plan.md vòng 22): (F) quét sàn target; (C) sàn tốt nhất + moe_type; (L) λ_id cho cặp
scGPT (cond(W), đọc niche qua Xenium); (M) map MLP — kiểm tra nhân quả "tuyến tính → bảo toàn niche".
CHỌN trên T2 (vùng con của T, không chấm H), BÁO CÁO trên H. Nhãn source chỉ dùng để chấm/chọn, không dùng khi huấn luyện.
Tái dùng run21 (lr_niche_bacc, aux_center, strat_target_floor, fit_moe_gate) + run20 (usher_variant_T, eval_all16d,
etrack, recall_table, rare_purity, all_groups, sub_split_250, ridge_fit). Map default/tgt_floor của out21 (nếu có trong
Input) được dùng lại làm sàn 0/200, không chạy lại USHER.
Quy tắc: gán lại R17..R20.OUT = OUT; beats trả dict (["pass"]); cmp3 cho so tất định; tiếp nối từ 22_summary*.json /
summary.json có "floors=" trong config + map đã lưu; kiểm tra "đã xong" ở đầu mỗi phần; TIME_BUDGET_S 11 giờ; SMOKE ở Cell 1.
"""
import os, sys, glob, json, gc, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run21 as R21
R20 = R21.R20
R19, R18, R17, R16, R11 = R20.R19, R20.R18, R20.R17, R20.R16, R20.R11
B, M, C, R6 = R20.B, R20.M, R20.C, R20.R6
np, pd, log, _unit, _ms = R20.np, R20.pd, R20.log, R20._unit, R20._ms
beats, not_much_worse, cmp3 = R20.beats, R20.not_much_worse, R16.cmp3
SMOKE = R20.SMOKE
NearestNeighbors, KMeans, mode_rows, niche_metrics = R20.NearestNeighbors, R20.KMeans, R20.mode_rows, R20.niche_metrics
lite_obs, load_map17 = R20.lite_obs, R20.load_map17
usher_variant_T, ridge_fit, eval_all16d = R20.usher_variant_T, R20.ridge_fit, R20.eval_all16d
etrack, recall_table, rare_purity, all_groups, sub_split_250 = R20.etrack, R20.recall_table, R20.rare_purity, R20.all_groups, R20.sub_split_250

OUT = os.environ.get("OUT_DIR_22", "/kaggle/working/out22" + ("_smoke" if SMOKE else ""))
R17.OUT, R18.OUT, R19.OUT, R20.OUT, R21.OUT = OUT, OUT, OUT, OUT, OUT
COMBOS = R20.COMBOS
SEEDS = [0] if SMOKE else [0, 1, 2]
FLOORS = [0, 200] if SMOKE else [0, 200, 500, 1000]
LAMS = [0.3, 3.0] if SMOKE else [0.3, 1.0, 3.0]
L_COMBOS = [c for c in COMBOS if c.endswith("/scgpt")]
M_COMBOS = ["visiumhd_ff/scgpt"] if SMOKE else ["cosmx6k/pca", "visiumhd_ff/scgpt"]
MLP_HIDDEN = 512
ALIAS21 = {0: "default", 200: "tgt_floor"}   # tên map vòng 21 tương ứng sàn 0 / 200
PARTS = {p.strip() for p in os.environ.get("ONLY_PART", "F,C,L,M").split(",") if p.strip()}
if "all" in PARTS: PARTS = {"F", "C", "L", "M"}
# ------------------------------------------------------------------ MAP_DIRS (sửa gốc, 2026-10-07)
# MỘT nơi duy nhất khai báo các thư mục chứa map USHER. Đã có 3 lần sự cố vì danh sách này bị viết cứng
# ở nhiều script và quên cập nhật khi thêm thư mục mới (thiếu out<P>j trong 26g; thiếu /kaggle/working;
# thiếu out<P>k ở vòng 34 -> mất 2 phiên GPU). Script nào quét map thì gọi map_dirs(P), KHÔNG tự viết lại.
# Thêm thư mục mới: thêm ĐÚNG MỘT dòng vào _SUFFIX dưới đây.
_SUFFIX = ("p", "n", "m", "k", "j", "d", "h", "v")   # out<P>p (39, ba nhánh C_s ở 25 vòng),
                                             # n (38, USHER gốc + FROST 25 vòng), m (35, trọng số
                                             # niche), k (34, β), j (31, C_niche), d, h,
                                             # v (46, FROST 1.0.1: cmix_lev/cmix_levns)
_FIXED = ("out23a", "out22", "out21")        # COAD: map sàn 200 ở out21/out22 với alias "tgt_floor"


def map_dirs(prefix):
    return tuple(f"out{prefix}{s}" for s in _SUFFIX) + _FIXED


def scan_maps(prefix, prior, log=None):
    """Nạp mọi map vào `prior` (dict tên_file -> đường dẫn). QUÉT /kaggle/working TRƯỚC vì khi nhiều bước
    chạy chung một phiên, map vừa tạo còn nằm ở working, chưa phải Input."""
    import glob as _g
    for pat in map_dirs(prefix):
        for f in (_g.glob(f"/kaggle/working/{pat}/*.np[yz]")
                  + _g.glob(f"/kaggle/input/**/{pat}/*.np[yz]", recursive=True)):
            prior.setdefault(os.path.basename(f), f)
    if log: log(f"map sẵn có: {len(prior)} file (quét {len(map_dirs(prefix))} thư mục)")
    return prior


RESUME = os.environ.get("RESUME", "1") == "1"
T0 = time.time()
TIME_BUDGET_S = float(os.environ.get("TIME_BUDGET_S", 11 * 3600))
class OutOfTime(Exception): pass
def step(msg):
    log(f"    [{time.time() - T0:6.0f}s] {msg}")
    if time.time() - T0 > TIME_BUDGET_S: raise OutOfTime(msg)

G, CTX, PRIOR = {}, {}, {}   # G: D, F, PCTX, S, save, kmeans8, pca_cache


# =============================================================================== USHER + map đã lưu
def usher_run(cx, tgt, seed, lam=0.3, hidden=None):
    """usher_variant_T với λ_id / hidden_dim đổi được: bọc run_usher.align_features_fgw để đặt env NGAY trước lúc gọi
    (usher_variant_T đặt cứng λ_id=0.3). MLP: USHER_ID_MODE=full, λ_id=0 (như MLP vòng 06; mode linear chỉ hợp lệ với nn.Linear)."""
    import run_usher as U
    orig = U.align_features_fgw
    def wrap(*a, **k):
        if hidden is None:
            os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = str(lam)
        else:
            os.environ["USHER_ID_MODE"] = "full"; os.environ["USHER_LAMBDA_ID"] = "0"; k["hidden_dim"] = hidden
        return orig(*a, **k)
    U.align_features_fgw = wrap
    try:
        Fmap, info, ybar, has = usher_variant_T(cx["Fs"], cx["ds"]["T"], cx["P_def"], cx["Fx"], tgt, cx["dx"]["lab"],
                                                cx["pctx"]["classes"], seed, "target", None, None)
    finally:
        U.align_features_fgw = orig
        os.environ["USHER_ID_MODE"] = "linear"; os.environ["USHER_LAMBDA_ID"] = "0.3"
    return Fmap, ybar, has

def get_map(combo, tag, seed, make, alias=None):
    """Nạp map (+ ȳ) đã lưu: OUT -> Input out22 -> Input out21 (tên vòng 21 `alias`); không có thì gọi make() (USHER)."""
    c_ = combo.replace("/", "_"); base = f"{c_}_{tag}_s{seed}"
    fm, fy = f"{OUT}/{base}.npy", f"{OUT}/{base}_ybar.npz"
    cands = [(fm, fy), (PRIOR.get(base + ".npy"), PRIOR.get(base + "_ybar.npz"))]
    if alias: cands.append((PRIOR.get(f"{c_}_{alias}_s{seed}.npy"), PRIOR.get(f"{c_}_{alias}_ybar_s{seed}.npz")))
    n = len(CTX[combo]["Fs"])
    for m_, y_ in cands:
        if RESUME and m_ and os.path.exists(m_):
            Fmap = load_map17(m_, n); ybar = has = None
            if y_ and os.path.exists(y_):
                d_ = np.load(y_); ybar, has = d_["ybar"], d_["has_tgt"]
            step(f"nạp lại {base} <- {m_}" + ("" if ybar is not None else " (không có ȳ)"))
            return Fmap, ybar, has
    if make is None: return None, None, None
    step(f"USHER {base}")
    Fmap, ybar, has = make()
    np.save(fm, Fmap.astype(np.float16))
    if ybar is not None: np.savez(fy, ybar=ybar, has_tgt=has)
    return Fmap, ybar, has


# =============================================================================== ngữ cảnh + đánh giá (H và T2)
def ctx(combo):
    if combo in CTX: return CTX[combo]
    D, F, PCTX, S, save, km8 = G["D"], G["F"], G["PCTX"], G["S"], G["save"], G["kmeans8"]
    pair, rep = combo.split("/"); pctx = PCTX[pair]; ds, dx = pctx["ds"], pctx["dx"]
    Fs, Fx = R17.R16_rep_features(pair, rep, F, PCTX, G["pca_cache"])
    comp_s, comp_x = R17.get_niche_label_comp(pair, S, save)
    Q = B.pick(ds["H"], R6.N_Q)
    tq = km8.predict(comp_s[Q]); vc = pd.Series(tq).value_counts(); keep = vc[vc >= 50].index.to_numpy()
    y_T = km8.predict(comp_s[ds["T"]])
    if S.setdefault("lr_src", {}).get(combo) is None:
        S["lr_src"][combo] = R21.lr_niche_bacc(Fs[ds["T"]], y_T, Fs[Q], tq, keep); save()
    ALL = all_groups(pair, PCTX)
    frac = pd.Series(ds["lab"][ds["H"]]).value_counts(normalize=True)
    rare = [c for c in ALL if frac.get(c, 0) < 0.02]
    lab_def, P_def, _ = R21.aux_center(Fs, Fx, ds, dx, pctx["classes"], R17.K_AUX)
    xy_T = lite_obs(pair)[["x", "y"]].to_numpy()[ds["T"]]
    m1, m2 = sub_split_250(xy_T); T1, T2 = ds["T"][m1], ds["T"][m2]
    ds2 = {**ds, "H": T2}
    # LỖI đã sửa (2026-10-05): trước đây truyền S_shared (nhãn MỊN) cùng lab_*_ev (có thể là nhãn THÔ) -> không khớp
    # nhóm nào và np.concatenate([]) ném ValueError. HCC lộ ra vì S_shared = ["Hepatocyte"] không có trong nhãn thô.
    g_ev = pctx.get("groups_ev") or pctx["S_shared"]
    idx_sh2 = R11.build_eval_idx({**ds2, "lab": pctx["lab_src_ev"]}, {**dx, "lab": pctx["lab_tgt_ev"]}, g_ev)
    idx_all2 = R11.build_eval_idx(ds2, dx, ALL)
    CTX[combo] = dict(pair=pair, pctx=pctx, ds=ds, dx=dx, Fs=Fs, Fx=Fx, comp_s=comp_s, comp_x=comp_x, Q=Q, tq=tq,
                      keep=keep, y_T=y_T, ALL=ALL, rare=rare, lab_def=lab_def, P_def=P_def, T1=T1, T2=T2, ds2=ds2,
                      idx_sh2=idx_sh2, idx_all2=idx_all2, Q2=B.pick(T2, R6.N_Q))
    S.setdefault("rare", {})[combo] = rare; save()
    log(f"[{combo}] ALL={len(ALL)} nhóm, hiếm={rare}, |T1|={len(T1)} |T2|={len(T2)} lr_src={S['lr_src'][combo]}")
    return CTX[combo]

def _knn_bacc(cx, Fmap, q, lab_q, lab_ref, groups):
    ref, Fx = cx["pctx"]["ref"], cx["Fx"]; mu = Fx.mean(0)
    nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx[ref] - mu)).kneighbors(_unit(Fmap[q] - mu), return_distance=False)
    return M.bacc_on(lab_q[q], mode_rows(lab_ref[ref][nn]), groups)

def t2_metrics(cx, Fmap, W, name):
    pctx, ds, dx, T2, ALL, Fx = cx["pctx"], cx["ds"], cx["dx"], cx["T2"], cx["ALL"], cx["Fx"]
    ls, lt, Ssh = pctx["lab_src_ev"], pctx["lab_tgt_ev"], pctx["S_shared"]
    o = {"xfer_main_T2": _knn_bacc(cx, Fmap, T2[np.isin(ls[T2], Ssh)], ls, lt, Ssh),
         "xfer_all_T2": _knn_bacc(cx, Fmap, T2[np.isin(ds["lab"][T2], ALL)], ds["lab"], dx["lab"], ALL)}
    for key, a, b, idx in (("total_v2_T2", ls, lt, cx["idx_sh2"]), ("total_v2_all_T2", ds["lab"], dx["lab"], cx["idx_all2"])):
        mp = R11.metrics_pack(Fmap, Fx, a, b, *idx); W.extend(f"{name} {key}: {x}" for x in mp.pop("_warnings", []))
        o[key] = mp.get("total_v2")
    o["rare_purity_T2"] = rare_purity(cx["pair"], {**G["D"], cx["pair"]: cx["ds2"]}, Fx, Fmap, ALL, pctx["ref"])[0]
    o["knn_xen_T2"] = niche_metrics(Fmap, Fx, ds, dx, pctx["ref"], cx["comp_s"], cx["comp_x"], G["kmeans8"], cx["Q2"])["niche_bacc"]
    return o

def full_eval(cx, name, Fmap, W, with_p=False):
    pctx, ds, dx, Fs, Fx = cx["pctx"], cx["ds"], cx["dx"], cx["Fs"], cx["Fx"]
    ev, w = eval_all16d(cx["pair"], G["D"], Fs, Fx, name, Fmap, pctx); W.extend(w)
    nm = niche_metrics(Fmap, Fx, ds, dx, pctx["ref"], cx["comp_s"], cx["comp_x"], G["kmeans8"], cx["Q"])
    et = etrack(cx["pair"], G["D"], G["PCTX"], G["S"], G["save"], Fs, Fx, name, Fmap, W)
    rec = recall_table(cx["pair"], G["D"], Fx, Fmap, cx["rare"], pctx["ref"]) if cx["rare"] else {}
    out = {**ev, "knn_xen": nm["niche_bacc"], **et, "recall_rare": rec, **t2_metrics(cx, Fmap, W, name)}
    if with_p:
        Wm, _, r2 = R19.recover_wb(Fs, Fmap)
        out.update(lr_map=R21.lr_niche_bacc(Fmap[ds["T"]], cx["y_T"], Fmap[cx["Q"]], cx["tq"], cx["keep"]),
                   cond_W=round(float(np.linalg.cond(Wm)), 2), r2_W=r2)
    return out

def seeds_of(d): return [d[f"s{s}"] for s in SEEDS if isinstance(d, dict) and isinstance(d.get(f"s{s}"), dict) and "error" not in d[f"s{s}"]]
def done(d): return isinstance(d, dict) and all(f"s{s}" in d for s in SEEDS)
def msk(d, k): return _ms([v.get(k) for v in seeds_of(d)])


# =============================================================================== Phần F / C / L / M
def part_f(combo, W):
    S = G["S"]; Sc = S["F"].setdefault(combo, {})
    if all(done(Sc.get(f"f{f}")) for f in FLOORS): log(f"[{combo}] Phần F: đã xong -> bỏ qua"); return
    cx = ctx(combo); T = cx["ds"]["T"]
    for f in FLOORS:
        Sf = Sc.setdefault(f"f{f}", {})
        for seed in SEEDS:
            if f"s{seed}" in Sf: continue
            def make(f=f, seed=seed):
                tgt = C.strat_target(cx["lab_def"][T], cx["dx"]["lab"]) if f == 0 else R21.strat_target_floor(cx["lab_def"][T], cx["dx"]["lab"], floor=f)
                return usher_run(cx, tgt, seed)
            Fmap, _, _ = get_map(combo, f"f{f}", seed, make, ALIAS21.get(f))
            Sf[f"s{seed}"] = full_eval(cx, f"{combo}/f{f}/s{seed}", Fmap, W, with_p=f in (0, 200))
            G["save"](); log(f"[{combo}] f{f} s{seed}: xfer_main={Sf[f's{seed}']['xfer_main']} rare_purity={Sf[f's{seed}'].get('rare_purity')} rare_T2={Sf[f's{seed}'].get('rare_purity_T2')}")
            del Fmap; gc.collect()

def choose_floor():
    S = G["S"]
    if not all(done(S["F"].get(c, {}).get(f"f{f}")) for c in COMBOS for f in FLOORS): return None, {}
    m = lambda c, f, k: msk(S["F"][c][f"f{f}"], k)[0]
    info = {}
    for f in FLOORS:
        ok = all(not_much_worse(m(c, f, "xfer_main_T2"), m(c, 0, "xfer_main_T2")) is not False and
                 not_much_worse(m(c, f, "total_v2_all_T2"), m(c, 0, "total_v2_all_T2")) is not False for c in COMBOS)
        rp = [m(c, f, "rare_purity_T2") for c in COMBOS]; rp = [x for x in rp if x is not None]
        info[f] = {"rare_T2_mean": round(float(np.mean(rp)), 4) if rp else None, "valid": bool(ok)}
    valid = [f for f in FLOORS if info[f]["valid"] and info[f]["rare_T2_mean"] is not None]
    if not valid: return 0, info
    best = max(info[f]["rare_T2_mean"] for f in valid)
    return min(f for f in valid if info[f]["rare_T2_mean"] >= best - 0.005), info

def part_c(combo, fstar, W):
    S = G["S"]; Sc = S["C"].setdefault(combo, {})
    if Sc.get("_fstar") == fstar and all(all(k in Sc.get(f"s{s}", {}) for k in ("moe_def", "moe_floor")) for s in SEEDS):
        log(f"[{combo}] Phần C: đã xong -> bỏ qua"); return
    if Sc.get("_fstar") != fstar: Sc.clear(); Sc["_fstar"] = fstar
    cx = ctx(combo); ds, Fs, Fx = cx["ds"], cx["Fs"], cx["Fx"]; T = ds["T"]
    cnt = pd.Series(cx["lab_def"][T]).value_counts()
    type_all = np.where(np.isin(cx["lab_def"], list(cnt[cnt < 200].index)), "other", cx["lab_def"])
    X_all = np.hstack([Fs, np.ones((len(Fs), 1), np.float32)]).astype(np.float64)
    for seed in SEEDS:
        Sk = Sc.setdefault(f"s{seed}", {})
        for mname, f in (("moe_def", 0), ("moe_floor", fstar)):
            if mname in Sk: continue
            _, ybar, has = get_map(combo, f"f{f}", seed, None, ALIAS21.get(f))
            if ybar is None:
                W.append(f"Phần C {combo} {mname} s{seed}: không có ȳ của sàn {f} -> bỏ"); Sk[mname] = {"error": "no ybar"}; G["save"](); continue
            step(f"Phần C {combo} {mname} s{seed}")
            Xhas = T[has]; m1h, m2h = np.isin(Xhas, cx["T1"]), np.isin(Xhas, cx["T2"])
            Xt = np.hstack([Fs[Xhas], np.ones((len(Xhas), 1), np.float32)]).astype(np.float64)
            Yt = ybar[has].astype(np.float64)
            Theta_glob = ridge_fit(Xt, Yt, 1e-3)
            out_g, lam = R21.fit_moe_gate(Xt, Yt, m1h, m2h, type_all[Xhas], type_all, Theta_glob, X_all, Fx.shape[1])
            Sk[mname] = {**full_eval(cx, f"{combo}/{mname}/s{seed}", out_g, W), "lambda": round(lam, 2)}
            G["save"](); log(f"[{combo}] {mname} s{seed}: rare_purity={Sk[mname].get('rare_purity')} rare_T2={Sk[mname].get('rare_purity_T2')}")

def part_l(combo, W):
    S = G["S"]; Sc = S["L"].setdefault(combo, {})
    new = [l for l in LAMS if l != 0.3]
    if all(done(Sc.get(f"lam{l:g}")) for l in new): log(f"[{combo}] Phần L: đã xong -> bỏ qua"); return
    cx = ctx(combo); T = cx["ds"]["T"]
    for lam in new:
        Sl = Sc.setdefault(f"lam{lam:g}", {})
        for seed in SEEDS:
            if f"s{seed}" in Sl: continue
            def make(lam=lam, seed=seed):
                return usher_run(cx, R21.strat_target_floor(cx["lab_def"][T], cx["dx"]["lab"], floor=200), seed, lam=lam)
            Fmap, _, _ = get_map(combo, f"lam{lam:g}", seed, make)
            Sl[f"s{seed}"] = full_eval(cx, f"{combo}/lam{lam:g}/s{seed}", Fmap, W, with_p=True)
            G["save"](); log(f"[{combo}] λ_id={lam} s{seed}: cond(W)={Sl[f's{seed}']['cond_W']} knn_xen={Sl[f's{seed}']['knn_xen']} xfer_main={Sl[f's{seed}']['xfer_main']}")
            del Fmap; gc.collect()

def part_m(combo, W):
    S = G["S"]; Sc = S["M"].setdefault(combo, {})
    if done(Sc): log(f"[{combo}] Phần M: đã xong -> bỏ qua"); return
    cx = ctx(combo); T = cx["ds"]["T"]
    for seed in SEEDS:
        if f"s{seed}" in Sc: continue
        try:
            Fmap, _, _ = get_map(combo, "mlp", seed, lambda seed=seed: usher_run(cx, C.strat_target(cx["lab_def"][T], cx["dx"]["lab"]), seed, hidden=MLP_HIDDEN))
            Sc[f"s{seed}"] = full_eval(cx, f"{combo}/mlp/s{seed}", Fmap, W, with_p=True)
            log(f"[{combo}] MLP s{seed}: lr_map={Sc[f's{seed}']['lr_map']} lr_src={S['lr_src'][combo]} xfer_main={Sc[f's{seed}']['xfer_main']}")
        except OutOfTime: raise
        except Exception as e:
            W.append(f"Phần M {combo} s{seed}: lỗi {e!r}"); Sc[f"s{seed}"] = {"error": repr(e)}
        G["save"](); gc.collect()


# =============================================================================== quyết định
def choose_lam():
    S = G["S"]; info = {}
    for lam in [l for l in LAMS if l != 0.3]:
        ok_all, have = True, True
        for c in L_COMBOS:
            ref, v = S["F"].get(c, {}).get("f200"), S["L"].get(c, {}).get(f"lam{lam:g}")
            if not (done(ref) and done(v)): have = False; break
            kb = beats(*msk(v, "knn_xen_T2"), *msk(ref, "knn_xen_T2"))
            ok = bool(kb and kb["pass"] and not_much_worse(msk(v, "xfer_main_T2")[0], msk(ref, "xfer_main_T2")[0])
                      and not_much_worse(msk(v, "total_v2_T2")[0], msk(ref, "total_v2_T2")[0]))
            ok_all &= ok
        info[lam] = ok_all if have else None
    passed = [l for l, ok in info.items() if ok]
    return (min(passed) if passed else 0.3), info

def choose_c(fstar):
    S = G["S"]; n, ok_all, have = 0, True, 0
    for c in COMBOS:
        Sc = S["C"].get(c, {})
        mf = {f"s{s}": Sc[f"s{s}"]["moe_floor"] for s in SEEDS if "moe_floor" in Sc.get(f"s{s}", {})}
        ref = S["F"].get(c, {}).get(f"f{fstar}")
        if not (done(mf) and done(ref)): continue
        have += 1
        kb = beats(*msk(mf, "rare_purity_T2"), *msk(ref, "rare_purity_T2")); n += bool(kb and kb["pass"])
        ok_all &= bool(not_much_worse(msk(mf, "xfer_main_T2")[0], msk(ref, "xfer_main_T2")[0])
                       and not_much_worse(msk(mf, "total_v2_all_T2")[0], msk(ref, "total_v2_all_T2")[0]))
    return (have == len(COMBOS) and n >= 3 and ok_all), n, have

def choose_m():
    S = G["S"]; lines, n_drop, lin_ok = [], 0, True
    for c in M_COMBOS:
        src = S.get("lr_src", {}).get(c)
        mlp, lin = msk(S["M"].get(c, {}), "lr_map")[0], msk(S["F"].get(c, {}).get("f0", {}), "lr_map")[0]
        if None in (src, mlp, lin): lines.append(f"- M1 {c}: thiếu số liệu"); continue
        drop = mlp < src - 0.01; lok = abs(lin - src) <= 0.01
        n_drop += drop; lin_ok &= lok
        lines.append(f"- M1 {c}: lr_src={src:.4f} | linear lr_map={lin:.4f} (|Δ|≤0.01: {'có' if lok else 'không'}) | MLP lr_map={mlp:.4f} (tụt >0.01: {'có' if drop else 'không'})")
    return bool(n_drop >= 1 and lin_ok), lines


# =============================================================================== main
def main():
    import faulthandler
    faulthandler.dump_traceback_later(int(os.environ.get("HANG_DUMP_S", 1800)), repeat=True, file=sys.stderr)
    os.makedirs(OUT, exist_ok=True)
    R19.import_prior_outputs()
    W = []
    if not SMOKE:
        assert (C.SKETCH, C.N_ITERS, C.N_BOOT, M.N_REF, M.N_TGT, M.N_MIX) == (4300, 10, 200, 20000, 20000, 6000), "lệch cấu hình"
    cfg = (f"CẤU HÌNH vòng 22: USHER-ST mặc định giai đoạn 1 combos={COMBOS} floors={FLOORS} lams={LAMS} (cặp {L_COMBOS}) "
           f"mlp={MLP_HIDDEN} trên {M_COMBOS} seeds={SEEDS} parts={sorted(PARTS)} | chọn trên T2, báo cáo trên H" + (" | SMOKE" if SMOKE else ""))
    log(cfg)
    S = json.load(open(f"{OUT}/summary.json")) if RESUME and os.path.exists(f"{OUT}/summary.json") else {}
    cand = sorted(set(glob.glob("/kaggle/input/**/22_summary*.json", recursive=True)) | set(glob.glob("/kaggle/input/**/summary.json", recursive=True)))
    for fj in cand:
        try:
            Sj = json.load(open(fj)); cj = str(Sj.get("config", ""))
            if "floors=" in cj and (("| SMOKE" in cj) == SMOKE): R19._deep_merge(S, Sj); log(f"Tiếp nối từ {fj}")
        except Exception as e: log(f"bỏ qua {fj}: {e!r}")
    S["config"] = cfg
    for k in ("F", "C", "L", "M"): S.setdefault(k, {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    pats = ["/kaggle/input/**/out22_smoke/*.np[yz]"] if SMOKE else ["/kaggle/input/**/out22/*.np[yz]", "/kaggle/input/**/out21/*.np[yz]"]
    for p in pats:
        for f in glob.glob(p, recursive=True): PRIOR.setdefault(os.path.basename(f), f)
    log(f"Map đã lưu tìm thấy trong Input: {len(PRIOR)} file")

    needed_pairs = sorted({c.split("/")[0] for c in COMBOS})
    D, F, PCTX = R18.setup_pairs(needed_pairs)
    comp_x0 = R17.get_niche_label_comp(needed_pairs[0], S, save)[1]
    kmeans8 = KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(comp_x0)
    comp_x0_16 = R20.fine16_label_comp(needed_pairs[0], S, save)[1]
    R20.KM16 = KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(comp_x0_16)
    G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=kmeans8, pca_cache={})
    try:
        if "F" in PARTS:
            for c in COMBOS: part_f(c, W)
        if "C" in PARTS:
            fstar, _ = choose_floor()
            if fstar is None: W.append("Phần C: Phần F chưa đủ -> chưa chọn được sàn, bỏ C lượt này")
            else:
                for c in COMBOS: part_c(c, fstar, W)
        if "L" in PARTS:
            for c in L_COMBOS: part_l(c, W)
        if "M" in PARTS:
            for c in M_COMBOS: part_m(c, W)
    except OutOfTime as e:
        W.append(f"HẾT NGÂN SÁCH THỜI GIAN ({TIME_BUDGET_S / 3600:.1f} giờ) trước bước: {e}. Add output out22 của lượt này làm Input rồi chạy lại.")
    S["warnings"] = list(dict.fromkeys(W)); save()
    md = summary_md(S, W)
    open(f"{OUT}/summary.md", "w").write(md)
    log("\n" + "=" * 90 + "\n" + md + "\nXONG vòng 22")


# =============================================================================== summary.md
H_COLS = ("xfer_main", "total_v2", "xfer_all", "total_v2_all", "rare_purity", "knn_xen")
T2_COLS = ("rare_purity_T2", "xfer_main_T2", "total_v2_all_T2")

def tbl(rows_src, cols):
    rows = []
    for lab, d in rows_src:
        vals = seeds_of(d)
        if not vals: continue
        r = dict(lab)
        for c in cols:
            m_, sd_ = _ms([v.get(c) for v in vals]); r[c] = f"{m_:.3f}±{sd_:.3f}" if m_ is not None else "—"
        rows.append(r)
    return ["```", pd.DataFrame(rows).to_string(index=False), "```"] if rows else ["(chưa có)"]

def summary_md(S, W):
    L = ["# 22 — chốt cấu hình USHER-ST (F sàn target, C +moe_type, L λ_id scGPT, M map MLP)", S["config"], ""]
    L.append("## Phần F — sàn target (H: báo cáo; cột _T2: dùng để chọn)")
    L += tbl([({"tổ_hợp": c, "sàn": f}, S["F"].get(c, {}).get(f"f{f}")) for c in COMBOS for f in FLOORS], H_COLS + T2_COLS)
    fstar, info = choose_floor() if G else (None, {})
    for f, v in info.items(): L.append(f"- sàn {f}: rare_purity_T2 TB 4 tổ hợp = {v['rare_T2_mean']} | hợp lệ (không mất >0.02 xfer_main_T2/total_v2_all_T2): {'có' if v['valid'] else 'không'}")
    L.append(f"- **F1: floor* = {fstar}**" if fstar is not None else "- F1: Phần F chưa đủ")

    L.append("## Phần C — moe_type trên ȳ của sàn 0 (moe_def) và của floor* (moe_floor)")
    rs = []
    for c in COMBOS:
        Sc = S["C"].get(c, {})
        for k in ("moe_def", "moe_floor"):
            rs.append(({"tổ_hợp": c, "map": k}, {f"s{s}": Sc[f"s{s}"][k] for s in SEEDS if k in Sc.get(f"s{s}", {})}))
    L += tbl(rs, H_COLS + T2_COLS)
    c1 = (False, 0, 0)
    if fstar is not None:
        c1 = choose_c(fstar)
        L.append(f"- **C1:** moe_floor vượt floor* về rare_purity_T2 ở {c1[1]}/{c1[2]} tổ hợp; không mất >0.02 -> {'ĐẠT (gộp moe_type)' if c1[0] else 'không (moe_type chỉ là phụ lục)'}")

    L.append("## Phần L — λ_id cho cặp scGPT (sàn 200; λ=0.3 = map sàn 200 của F)")
    rs = []
    for c in L_COMBOS:
        rs.append(({"tổ_hợp": c, "λ_id": 0.3}, S["F"].get(c, {}).get("f200")))
        for l in LAMS:
            if l != 0.3: rs.append(({"tổ_hợp": c, "λ_id": l}, S["L"].get(c, {}).get(f"lam{l:g}")))
    L += tbl(rs, ("cond_W", "lr_map", "knn_xen", "knn_xen_T2", "xfer_main", "total_v2", "rare_purity", "xfer_main_T2", "total_v2_T2"))
    lam_star, linfo = choose_lam()
    for l, ok in linfo.items(): L.append(f"- λ_id={l}: knn_xen_T2 vượt λ=0.3 ở cả 2 cặp và không mất >0.02 -> {'ĐẠT' if ok else ('không' if ok is False else 'chưa đủ số liệu')}")
    L.append(f"- **L1: λ_id* (cặp scGPT) = {lam_star}**")

    L.append("## Phần M — map MLP (hidden 512, λ_id=0) so với tuyến tính (sàn 0)")
    rs = []
    for c in M_COMBOS:
        rs.append(({"tổ_hợp": c, "map": "linear"}, S["F"].get(c, {}).get("f0")))
        rs.append(({"tổ_hợp": c, "map": "mlp"}, S["M"].get(c, {})))
    L += tbl(rs, ("lr_map", "knn_xen", "cond_W", "r2_W", "xfer_main", "total_v2", "rare_purity"))
    m1, mlines = choose_m(); L += mlines
    L.append(f"- **M1:** {'ĐẠT -> ràng buộc tuyến tính là nguyên nhân bảo toàn (MLP làm mất, tuyến tính không)' if m1 else 'không -> chỉ ghi: map USHER không phá thông tin niche'}")

    if fstar is not None:
        final = f"sàn target = {fstar}" + (" + moe_type (cổng theo loại)" if c1[0] else "") + \
                (f"; λ_id = {lam_star} cho cặp scGPT (đã thử với sàn 200)" if lam_star != 0.3 else "; λ_id = 0.3 mọi cặp")
        L += ["", f"## CẤU HÌNH CUỐI: {final}", "So với default (sàn 0) trên H (`cmp3` theo SD của cấu hình cuối):"]
        for c in COMBOS:
            Sc = S["C"].get(c, {})
            fin = ({f"s{s}": Sc[f"s{s}"]["moe_floor"] for s in SEEDS if "moe_floor" in Sc.get(f"s{s}", {})} if c1[0]
                   else S["F"].get(c, {}).get(f"f{fstar}", {}))
            if c in L_COMBOS and lam_star != 0.3 and not c1[0] and fstar == 200: fin = S["L"].get(c, {}).get(f"lam{lam_star:g}", fin)
            d0 = S["F"].get(c, {}).get("f0", {})
            parts = []
            for k in ("xfer_main", "total_v2", "xfer_all", "total_v2_all", "rare_purity"):
                m_, sd_ = msk(fin, k); b_ = msk(d0, k)[0]
                parts.append(f"{k} {m_:.3f} vs {b_:.3f} ({cmp3(m_, sd_, b_)})" if None not in (m_, b_) else f"{k} —")
            L.append(f"- {c}: " + " | ".join(parts))
        if lam_star != 0.3 and (fstar != 200 or c1[0]):
            L.append(f"- LƯU Ý: λ_id*={lam_star} mới thử với sàn 200 (không moe); tổ hợp với floor*={fstar}" + (" + moe" if c1[0] else "") + " cần một lượt xác nhận.")
    L += ["", "## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(W)] or ["- không có"])
    return "\n".join(L) + "\n"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run23.py
"""23 — Notebook báo cáo: chấm lại MỌI baseline + USHER-ST (mặc định, cuối) trên CÙNG giao thức (ô H), tính UMAP chung
(Xenium + source) cho từng phương pháp, gom số liệu vòng 22 (quét sàn, λ_id, MLP, niche) để vẽ hình ở các cell sau.
Không chạy USHER: map USHER lấy từ out21 (sàn 0 = default) và out22 (sàn 1000). Kết quả: out23/summary.json, out23/umap_*.npz.
"""
import os, sys, glob, json, gc, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run22 as R22
R20, R16, R17, R18, R19 = R22.R20, R22.R16, R22.R17, R22.R18, R22.R19
M, np, pd, log, _unit = R22.M, R22.np, R22.pd, R22.log, R22._unit
SMOKE = R22.SMOKE
OUT = os.environ.get("OUT_DIR_23", "/kaggle/working/out23" + {"maps": "a", "report": "b"}.get(os.environ.get("MODE23", "report"), "") + ("_smoke" if SMOKE else ""))
R17.OUT = R18.OUT = R19.OUT = R20.OUT = R22.OUT = OUT
R22.R21.OUT = OUT
COMBOS = R22.COMBOS
# Nicheformer (quy tắc `data`, emb_niche_data) trên cả 4 cặp, cấu hình cuối — người dùng yêu cầu đủ chỉ số USHER.pdf
WITH_NICHE = os.environ.get("WITH_NICHE", "1") == "1"
NICHE_COMBOS = ([] if not WITH_NICHE else [f"{c.split('/')[0]}/niche" for c in COMBOS])[: (1 if SMOKE else 99)]
NICHE_SEEDS = [0] if SMOKE else [0, 1, 2]
FINAL_FLOOR = int(os.environ.get("FINAL_FLOOR", "200"))   # 24a: cấu hình cuối = sàn 200
MODE = os.environ.get("MODE23", "report")   # "maps" = 23a (GPU, chỉ tạo map Nicheformer); "report" = 23b (CPU, không chạy USHER)
N_UMAP = 1000 if SMOKE else 3000
WITH_SYMPHONY = os.environ.get("WITH_SYMPHONY", "1") == "1"
METHODS = ["raw", "center", "mnn_ref", "harmony", "scanorama"] + (["symphony_h"] if WITH_SYMPHONY else []) + ["usher_default", "usher_final"]
LABEL = {"raw": "raw", "center": "center", "mnn_ref": "MNN-ref", "harmony": "Harmony", "scanorama": "Scanorama",
         "symphony_h": "Symphony", "usher_default": "USHER-ST mặc định", "usher_final": "USHER-ST cuối (sàn 1000)"}


def methods_for(combo):
    if combo.endswith("/niche"): return ["raw", "center", "mnn_ref", "harmony", "scanorama", "usher_final"]
    return METHODS

_orig_load_all = R20.R6.load_all
def _load_all_with_niche(assays, EMB):
    if WITH_NICHE:
        d = R20.M.find_dir("emb_niche_data")
        if d: EMB = {**EMB, "niche": d}
    return _orig_load_all(assays, EMB)
R20.R6.load_all = _load_all_with_niche
_orig_rep = R17.R16_rep_features
def _rep_features(pair, rep, F, PCTX, pca_cache):
    if rep == "niche": return F[(pair, "niche")], F[(R20.TARGET, "niche")]
    return _orig_rep(pair, rep, F, PCTX, pca_cache)
R17.R16_rep_features = _rep_features

def usher_niche_seeds(combo, cx, W):
    """USHER-ST cấu hình cuối (sàn 1000, λ_id 0.3) trên Nicheformer, 3 seed, chấm đủ chỉ số."""
    S, save = R22.G["S"], R22.G["save"]
    Su = S.setdefault("usher_seeds", {}).setdefault(combo, {})
    T = cx["ds"]["T"]; first = None
    for seed in NICHE_SEEDS:
        def make(seed=seed):
            if MODE == "report": return None
            return R22.usher_run(cx, R22.R21.strat_target_floor(cx["lab_def"][T], cx["dx"]["lab"], floor=FINAL_FLOOR), seed)
        Fmap = R22.get_map(combo, f"f{FINAL_FLOOR}", seed, None if MODE == "report" else make)[0]
        if Fmap is None: W.append(f"{combo}: thiếu map f{FINAL_FLOOR}_s{seed} (Add output 23a)"); continue
        if f"s{seed}" not in Su:
            Su[f"s{seed}"] = evaluate(cx, f"{combo}/usher_final/s{seed}", cx["Fs"], Fmap, cx["Fx"], W); save()
        if seed == 0: first = Fmap
    return first

def evaluate(cx, name, Fs_rep, Fmap, Fx_used, W):
    D, PCTX, S, save = R22.G["D"], R22.G["PCTX"], R22.G["S"], R22.G["save"]
    pctx, ds, dx = cx["pctx"], cx["ds"], cx["dx"]
    e, w = R20.eval_all16d(cx["pair"], D, Fs_rep, Fx_used, name, Fmap, pctx); W.extend(w)
    nm = R20.niche_metrics(Fmap, Fx_used, ds, dx, pctx["ref"], cx["comp_s"], cx["comp_x"], R22.G["kmeans8"], cx["Q"])
    et = R20.etrack(cx["pair"], D, PCTX, S, save, Fs_rep, Fx_used, name, Fmap, W)
    rec = R20.recall_table(cx["pair"], D, Fx_used, Fmap, cx["ALL"], pctx["ref"])
    return {**e, "knn_xen": nm["niche_bacc"], **et, "recall_all": rec}

def umap2(Fx_used, Fmap, ix, is_):
    import umap
    from sklearn.decomposition import PCA
    mu = Fx_used.mean(0)
    X = np.vstack([_unit(Fx_used[ix] - mu), _unit(Fmap[is_] - mu)]).astype(np.float32)
    if X.shape[1] > 50: X = PCA(50, random_state=0).fit_transform(X)
    return umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=0).fit_transform(X).astype(np.float32)

def symphony_maps(cx, py, W):
    pctx = cx["pctx"]
    ix, is_, _ = R20.R8.gene_cols(pctx["varx"], pctx["vars_"])
    res = R20.run_baseline_subprocess(py, "symphony_h", pctx["Xx_lite"][:, ix].toarray(), pctx["Xs_lite"][:, is_].toarray(), W)
    return None if res is None else (res[1], res[0])   # (Fs_b, Fx_b)

def process_combo(combo, py, W):
    S, save = R22.G["S"], R22.G["save"]
    Sc = S["methods"].setdefault(combo, {})
    f_um = f"{OUT}/umap_{combo.replace('/', '_')}.npz"
    um = dict(np.load(f_um, allow_pickle=True)) if os.path.exists(f_um) else {}
    MS = methods_for(combo)
    if all(m in Sc for m in MS) and all(m in um for m in MS):
        log(f"[{combo}] đã xong -> bỏ qua"); return
    cx = R22.ctx(combo); ds, dx, Fs, Fx, T = cx["ds"], cx["dx"], cx["Fs"], cx["Fx"], cx["ds"]["T"]
    rng = np.random.RandomState(0)
    ix = np.sort(rng.choice(cx["pctx"]["ref"], min(N_UMAP, len(cx["pctx"]["ref"])), replace=False))
    is_ = np.sort(rng.choice(ds["H"], min(N_UMAP, len(ds["H"])), replace=False))
    um.update(ix=ix, is_=is_, lab_x=dx["lab"][ix].astype(str), lab_s=ds["lab"][is_].astype(str))
    for m in MS:
        if m in Sc and m in um: continue
        R22.step(f"[{combo}] {m}")
        Fs_rep, Fx_used = Fs, Fx
        try:
            if m == "raw": Fmap = Fs
            elif m == "center": Fmap = M.base_center(Fs, Fx, T)
            elif m == "mnn_ref": Fmap = R16.mnn_ref_correction(Fs, Fx, T)[0]
            elif m == "harmony": Fx_used, Fmap = R16.run_harmony(Fx, Fs)
            elif m == "scanorama": Fx_used, Fmap = R16.run_scanorama(Fx, Fs)
            elif m == "symphony_h":
                r = symphony_maps(cx, py, W)
                if r is None: Sc[m] = None; save(); continue
                Fmap, Fx_used = r; Fs_rep = Fmap
            elif combo.endswith("/niche"):
                Fmap = usher_niche_seeds(combo, cx, W)
                if Fmap is None: Sc[m] = None; save(); continue
                Sc[m] = S["usher_seeds"][combo]["s0"]
                um[m] = umap2(Fx_used, Fmap, ix, is_); np.savez(f_um, **um); save(); continue
            else:
                tag, alias = ("f0", "default") if m == "usher_default" else (f"f{FINAL_FLOOR}", {200: "tgt_floor", 0: "default"}.get(FINAL_FLOOR))
                Fmap = R22.get_map(combo, tag, 0, None, alias)[0]
                if Fmap is None: W.append(f"{combo}: thiếu map {tag}_s0 (Add output 21 và 22) -> bỏ {m}"); Sc[m] = None; save(); continue
            Sc[m] = evaluate(cx, f"{combo}/{m}", Fs_rep, Fmap, Fx_used, W)
            um[m] = umap2(Fx_used, Fmap, ix, is_)
            np.savez(f_um, **um); save()
            log(f"[{combo}] {m}: xfer_main={Sc[m]['xfer_main']} total_v2={Sc[m].get('total_v2')} rare_purity={Sc[m].get('rare_purity')}")
        except R22.OutOfTime: raise
        except Exception as e:
            W.append(f"{combo} {m}: lỗi {e!r}"); Sc[m] = None; save()
        gc.collect()


def main():
    os.makedirs(OUT, exist_ok=True)
    R19.import_prior_outputs()
    W = []
    if not os.path.exists(f"{OUT}/summary.json"):   # tiếp nối: chép out23 của lượt trước (nếu đã Add làm Input)
        import shutil
        srcs = [os.path.basename(OUT)] + (["out23"] if MODE == "report" else [])
        for d in srcs:
            fs = [f for f in glob.glob(f"/kaggle/input/**/{d}/*", recursive=True) if f.endswith((".json", ".npz"))]
            for f in fs: shutil.copy(f, OUT); log(f"tiếp nối <- {f}")
            if fs: break
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    S["config"] = f"report23 combos={COMBOS} niche_combos={NICHE_COMBOS} methods={METHODS} final_floor={FINAL_FLOOR} n_umap={N_UMAP}" + (" | SMOKE" if SMOKE else "")
    S.setdefault("methods", {})
    if S.get("final_floor", 1000) != FINAL_FLOOR:   # out23 cũ là sàn 1000 -> bỏ mục USHER-ST cuối, giữ baseline/UMAP baseline
        for c in S["methods"]: S["methods"][c].pop("usher_final", None)
        S.pop("usher_seeds", None)
        for f in glob.glob(f"{OUT}/umap_*.npz"):
            u = dict(np.load(f, allow_pickle=True)); u.pop("usher_final", None); np.savez(f, **u)
        log(f"đổi sàn cuối -> {FINAL_FLOOR}: tính lại USHER-ST cuối")
    S["final_floor"] = FINAL_FLOOR
    # số liệu vòng 22 (quét sàn, C, L, M, lr_src) để vẽ — tìm ở mọi Input
    for fj in sorted(set(glob.glob("/kaggle/input/**/out22/summary.json", recursive=True)) | set(glob.glob("/kaggle/input/**/22_summary*.json", recursive=True))):
        try:
            Sj = json.load(open(fj))
            if "floors=" in str(Sj.get("config", "")) and "| SMOKE" not in str(Sj.get("config", "")):
                S["r22"] = {k: Sj.get(k) for k in ("F", "C", "L", "M", "lr_src", "rare")}; log(f"số liệu vòng 22 <- {fj}"); break
        except Exception as e: log(f"bỏ qua {fj}: {e!r}")
    if "r22" not in S: W.append("Không tìm thấy summary.json vòng 22 -> bảng USHER chỉ có seed 0, thiếu hình quét sàn/λ_id/MLP")
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    for p in ("/kaggle/input/**/out23a/*.np[yz]", "/kaggle/input/**/out22/*.np[yz]", "/kaggle/input/**/out21/*.np[yz]"):
        for f in glob.glob(p, recursive=True): R22.PRIOR.setdefault(os.path.basename(f), f)
    log(f"map USHER đã lưu trong Input: {len(R22.PRIOR)} file")
    needed = sorted({c.split("/")[0] for c in COMBOS})
    D, F, PCTX = R18.setup_pairs(needed)
    comp_x0 = R17.get_niche_label_comp(needed[0], S, save)[1]
    km8 = R20.KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(comp_x0)
    R20.KM16 = R20.KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(R20.fine16_label_comp(needed[0], S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    if MODE == "maps":   # 23a: chỉ chạy USHER cho Nicheformer, lưu map; chấm ở 23b
        try:
            for c in NICHE_COMBOS:
                cx = R22.ctx(c); T = cx["ds"]["T"]
                for seed in NICHE_SEEDS:
                    R22.get_map(c, f"f{FINAL_FLOOR}", seed, lambda seed=seed: R22.usher_run(
                        cx, R22.R21.strat_target_floor(cx["lab_def"][T], cx["dx"]["lab"], floor=FINAL_FLOOR), seed))
        except R22.OutOfTime as e: W.append(f"HẾT NGÂN SÁCH trước {e}; Add output out23a rồi chạy lại.")
        S["warnings"] = list(dict.fromkeys(W)); save(); log("XONG 23a: " + str(sorted(os.listdir(OUT)))); return
    need_sy = WITH_SYMPHONY and any("symphony_h" not in S["methods"].get(c, {}) for c in COMBOS)
    py = R20.setup_venv_b(W) if need_sy else None
    try:
        for c in COMBOS + NICHE_COMBOS: process_combo(c, py, W)
    except R22.OutOfTime as e:
        W.append(f"HẾT NGÂN SÁCH THỜI GIAN trước bước {e}; Add output out23 làm Input rồi chạy lại.")
    S["warnings"] = list(dict.fromkeys(W)); save()
    log("XONG vòng 23. Cảnh báo:\n" + "\n".join(S["warnings"] or ["không có"]))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run23c.py
"""23c — baseline CORAL (CORrelation ALignment, Sun & Saenko 2016): tịnh tiến + khớp hiệp phương sai source về Xenium,
không nhãn, tất định, CPU. Chấm đủ chỉ số (như 23b) + UMAP trên CÙNG mẫu tế bào của 23b, cho 8 tổ hợp (PCA, scGPT, Nicheformer).
Kết quả: out23c/summary.json (methods[combo]["coral"]) + umap_<combo>.npz (khoá "coral"). Gộp vào báo cáo ở 23d."""
import os, sys, json, glob
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import run23 as R23
R22, R20, R17, R18, R19 = R23.R22, R23.R20, R23.R17, R23.R18, R23.R19
np, log = R23.np, R23.log
OUT = os.environ.get("OUT_DIR_23C", "/kaggle/working/out23c" + ("_smoke" if R23.SMOKE else ""))
R17.OUT = R18.OUT = R19.OUT = R20.OUT = R22.OUT = R22.R21.OUT = R23.OUT = OUT
COMBOS = R23.COMBOS + R23.NICHE_COMBOS
RIDGE = float(os.environ.get("CORAL_RIDGE", "0.01"))   # ridge tương đối: C + RIDGE·tr(C)/d·I

def _mat_pow(C, p):
    w, V = np.linalg.eigh(C); w = np.clip(w, 1e-12, None)
    return (V * w ** p) @ V.T

def coral(Fs, Fx, T):
    Fs64, Fx64 = Fs.astype(np.float64), Fx.astype(np.float64)
    mu_s, mu_x = Fs64[T].mean(0), Fx64.mean(0)
    Cs, Cx = np.cov(Fs64[T], rowvar=False), np.cov(Fx64, rowvar=False)
    d = Cs.shape[0]
    Cs += RIDGE * np.trace(Cs) / d * np.eye(d); Cx += RIDGE * np.trace(Cx) / d * np.eye(d)
    A = _mat_pow(Cs, -0.5) @ _mat_pow(Cx, 0.5)
    return ((Fs64 - mu_s) @ A + mu_x).astype(np.float32)

def main():
    os.makedirs(OUT, exist_ok=True); W = []
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    if not S:
        for fj in glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}/summary.json", recursive=True):
            S = json.load(open(fj)); log(f"tiếp nối <- {fj}")
            for f in glob.glob(os.path.join(os.path.dirname(fj), "umap_*.npz")): os.system(f"cp '{f}' {OUT}/")
            break
    S["config"] = f"23c CORAL ridge={RIDGE} combos={COMBOS}"; S.setdefault("methods", {})
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    needed = sorted({c.split("/")[0] for c in COMBOS})
    D, F, PCTX = R18.setup_pairs(needed)
    km8 = R20.KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(R17.get_niche_label_comp(needed[0], S, save)[1])
    R20.KM16 = R20.KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(R20.fine16_label_comp(needed[0], S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    for c in COMBOS:
        f_um = f"{OUT}/umap_{c.replace('/', '_')}.npz"
        if S["methods"].get(c, {}).get("coral") and os.path.exists(f_um): log(f"[{c}] đã xong"); continue
        try:
            R22.step(f"[{c}] CORAL")
            cx = R22.ctx(c); Fs, Fx, ds = cx["Fs"], cx["Fx"], cx["ds"]
            Fmap = coral(Fs, Fx, ds["T"])
            S["methods"].setdefault(c, {})["coral"] = R23.evaluate(cx, f"{c}/coral", Fs, Fmap, Fx, W)
            rng = np.random.RandomState(0)   # cùng cách lấy mẫu với run23 -> cùng tế bào trong UMAP
            ix = np.sort(rng.choice(cx["pctx"]["ref"], min(R23.N_UMAP, len(cx["pctx"]["ref"])), replace=False))
            is_ = np.sort(rng.choice(ds["H"], min(R23.N_UMAP, len(ds["H"])), replace=False))
            np.savez(f_um, ix=ix, is_=is_, coral=R23.umap2(Fx, Fmap, ix, is_)); save()
            log(f"[{c}] CORAL: xfer_main={S['methods'][c]['coral']['xfer_main']} total_v2={S['methods'][c]['coral'].get('total_v2')}")
        except R22.OutOfTime: raise
        except Exception as e: W.append(f"{c}: lỗi {e!r}")
    S["warnings"] = W; save(); log("XONG 23c")

if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run26d.py
"""26d/26e — chạy lại TOÀN BỘ giao thức vòng 23 cho một mô khác (mặc định OV), trên 1 bệnh nhân.

MODE26=maps   -> 26d (GPU): chỉ chạy USHER-ST cấu hình cuối (sàn 200, λ_id 0.3, linear) cho 4 tổ hợp × 3 seed, lưu map vào out26d/.
MODE26=report -> 26e (CPU): chấm điểm mọi phương pháp (raw, center, CORAL, MNN-ref, Harmony, Scanorama, Symphony, USHER-ST)
                 trên cùng giao thức + UMAP chung; KHÔNG chạy USHER (map lấy từ Input out26d). Kết quả out26e/summary.json.

Env: TISSUE (ov|hcc), COMBOS_ALL, FINAL_FLOOR (200), WITH_SYMPHONY (1), WITH_NICHE (0 — Nicheformer hoãn cho mô mới).
CẢNH BÁO: KHÔNG Add bất kỳ output nào của COAD (out17–out23, 03a, 04b của COAD) làm Input — cache và
recoverable_classes.json đặt tên theo assay chứ không theo mô, sẽ lẫn dữ liệu. Notebook kiểm tra điều này và dừng nếu thấy trùng.
"""
import os, sys, glob, json, gc
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))

TISSUE = os.environ.setdefault("TISSUE", "ov").lower()
# Biểu diễn mặc định cho OV = chọn theo accS trên ô H của notebook 26c (BẢNG C2): cosmx->PCA, 3 cái còn lại -> scGPT-spatial.
_DEF = {"ov": "cosmx6k/pca,visiumhd_ff/scgpt,visiumhd_ffpe/scgpt,stereoseq/scgpt"}
os.environ.setdefault("COMBOS_ALL", _DEF.get(TISSUE, "cosmx6k/pca,visiumhd_ff/scgpt,visiumhd_ffpe/pca,stereoseq/scgpt"))
os.environ.setdefault("WITH_NICHE", "0")   # bật lại bằng WITH_NICHE=1 + thêm <pair>/niche vào COMBOS_ALL
os.environ.setdefault("FINAL_FLOOR", "200")
os.environ["MODE23"] = "report"          # không dùng nhánh maps của run23 (nhánh đó chỉ cho Nicheformer)
MODE = os.environ.get("MODE26", "maps")
PREFIX = os.environ.get("RUN_PREFIX", "26")   # 26 = OV, 27 = HCC ... (tên thư mục out<PREFIX>d / out<PREFIX>e)

import run23 as R23
import run23c as R23C
R22, R20, R17, R18, R19, R16 = R23.R22, R23.R20, R23.R17, R23.R18, R23.R19, R23.R16
np, pd, log, M = R23.np, R23.pd, R23.log, R23.M
SMOKE = R23.SMOKE
OUT = os.environ.get("OUT_DIR_26", f"/kaggle/working/out{PREFIX}{'d' if MODE == 'maps' else 'e'}" + ("_smoke" if SMOKE else ""))
R17.OUT = R18.OUT = R19.OUT = R20.OUT = R22.OUT = R22.R21.OUT = R23.OUT = R23C.OUT = OUT
COMBOS = R23.COMBOS
FLOOR = R23.FINAL_FLOOR
SEEDS = [0] if SMOKE else [0, 1, 2]
WITH_SYMPHONY = os.environ.get("WITH_SYMPHONY", "1") == "1"
METHODS = ["raw", "center", "coral", "mnn_ref", "harmony", "scanorama"] + (["symphony_h"] if WITH_SYMPHONY else []) + ["usher_final"]
N_UMAP = R23.N_UMAP


def check_inputs():
    """Dừng sớm nếu Input lẫn dữ liệu của mô khác (lỗi này rất khó phát hiện về sau)."""
    rec = sorted(glob.glob("/kaggle/input/**/recoverable_classes.json", recursive=True))
    emb = sorted(p for p in glob.glob("/kaggle/input/**/emb_scgpt", recursive=True) if os.path.isdir(p))
    lite = sorted(p for p in glob.glob(f"/kaggle/input/**/spatch_{TISSUE}_lite", recursive=True) if os.path.isdir(p))
    log(f"recoverable_classes.json: {rec}\nemb_scgpt: {emb}\nspatch_{TISSUE}_lite: {lite}")
    assert len(rec) == 1, f"phải có ĐÚNG 1 recoverable_classes.json (của 26c), đang thấy {len(rec)}: {rec}"
    assert len(emb) == 1, f"phải có ĐÚNG 1 thư mục emb_scgpt (của 26b), đang thấy {len(emb)}: {emb}"
    assert lite, f"thiếu spatch_{TISSUE}_lite (output 26a)"
    assert not glob.glob("/kaggle/input/**/spatch_coad_lite", recursive=True) and \
           not glob.glob("/kaggle/input/**/spatch-coad-lite", recursive=True), "đang Add dữ liệu COAD -> bỏ ra"
    # embedding scGPT phải đúng mô: id tế bào phải trùng với bản lite
    try:
        import pandas as _pd
        a = next(iter(COMBOS)).split("/")[0]
        e = _pd.read_csv(f"{emb[0]}/{a}_obs.csv", index_col=0, dtype=str).index.astype(str)
        l = _pd.read_parquet(f"{lite[0]}/{a}_obs_full.parquet").index.astype(str)
        ov = len(set(e[:20000]) & set(l)) / max(1, len(set(e[:20000])))
        log(f"kiểm tra emb_scgpt so với lite [{a}]: trùng id {ov:.1%}")
        assert ov > 0.5, f"emb_scgpt KHÔNG khớp mô {TISSUE} (trùng id chỉ {ov:.1%}) -> đang Add nhầm 03a của COAD"
    except AssertionError: raise
    except Exception as e: log(f"(bỏ qua kiểm tra id embedding: {e!r})")


def usher_seeds(combo, cx, W):
    """USHER-ST cấu hình cuối, 3 seed. maps: chạy và lưu map. report: chỉ nạp map + chấm điểm."""
    S, save = R22.G["S"], R22.G["save"]
    Su = S.setdefault("usher_seeds", {}).setdefault(combo, {})
    T = cx["ds"]["T"]; first = None
    for seed in SEEDS:
        make = None if MODE == "report" else (lambda seed=seed: R22.usher_run(
            cx, R22.R21.strat_target_floor(cx["lab_def"][T], cx["dx"]["lab"], floor=FLOOR), seed))
        Fmap = R22.get_map(combo, f"f{FLOOR}", seed, make)[0]
        if Fmap is None: W.append(f"{combo}: thiếu map f{FLOOR}_s{seed} (Add output {PREFIX}d)"); continue
        if MODE == "report" and f"s{seed}" not in Su:
            Su[f"s{seed}"] = R23.evaluate(cx, f"{combo}/usher_final/s{seed}", cx["Fs"], Fmap, cx["Fx"], W); save()
        if first is None: first = Fmap
    return first


def free_combo(combo):
    """Giải phóng context của tổ hợp vừa xong. R22.CTX giữ Fs/Fx/comp_s/comp_x của MỌI tổ hợp đã dựng và không bao giờ
    xoá -> bộ nhớ tăng dần theo số tổ hợp và dễ bị OOM ở tổ hợp thứ 3-4 (27e HCC bị giết giữa chừng ở FFPE)."""
    R22.CTX.pop(combo, None)
    (R22.G.get("pca_cache") or {}).clear()
    gc.collect()


def process_combo(combo, py, W):
    S, save = R22.G["S"], R22.G["save"]
    Sc = S["methods"].setdefault(combo, {})
    f_um = f"{OUT}/umap_{combo.replace('/', '_')}.npz"
    um = dict(np.load(f_um, allow_pickle=True)) if os.path.exists(f_um) else {}
    if all(m in Sc for m in METHODS) and all(m in um for m in METHODS): log(f"[{combo}] đã xong -> bỏ qua"); return
    cx = R22.ctx(combo); ds, dx, Fs, Fx, T = cx["ds"], cx["dx"], cx["Fs"], cx["Fx"], cx["ds"]["T"]
    rng = np.random.RandomState(0)
    ix = np.sort(rng.choice(cx["pctx"]["ref"], min(N_UMAP, len(cx["pctx"]["ref"])), replace=False))
    is_ = np.sort(rng.choice(ds["H"], min(N_UMAP, len(ds["H"])), replace=False))
    um.update(ix=ix, is_=is_, lab_x=dx["lab"][ix].astype(str), lab_s=ds["lab"][is_].astype(str))
    for m in METHODS:
        if m in Sc and m in um: continue
        R22.step(f"[{combo}] {m}")
        Fs_rep, Fx_used = Fs, Fx
        try:
            if m == "raw": Fmap = Fs
            elif m == "center": Fmap = M.base_center(Fs, Fx, T)
            elif m == "coral": Fmap = R23C.coral(Fs, Fx, T)
            elif m == "mnn_ref": Fmap = R16.mnn_ref_correction(Fs, Fx, T)[0]
            elif m == "harmony": Fx_used, Fmap = R16.run_harmony(Fx, Fs)
            elif m == "scanorama": Fx_used, Fmap = R16.run_scanorama(Fx, Fs)
            elif m == "symphony_h":
                r = R23.symphony_maps(cx, py, W)
                if r is None: Sc[m] = None; save(); continue
                Fmap, Fx_used = r; Fs_rep = Fmap
            else:
                Fmap = usher_seeds(combo, cx, W)
                if Fmap is None: Sc[m] = None; save(); continue
                Sc[m] = S["usher_seeds"][combo]["s0"]
                um[m] = R23.umap2(Fx_used, Fmap, ix, is_); np.savez(f_um, **um); save(); continue
            Sc[m] = R23.evaluate(cx, f"{combo}/{m}", Fs_rep, Fmap, Fx_used, W)
            um[m] = R23.umap2(Fx_used, Fmap, ix, is_); np.savez(f_um, **um); save()
            log(f"[{combo}] {m}: xfer_main={Sc[m]['xfer_main']} total_v2={Sc[m].get('total_v2')} rare_purity={Sc[m].get('rare_purity')}")
        except R22.OutOfTime: raise
        except Exception as e: W.append(f"{combo} {m}: lỗi {e!r}"); Sc[m] = None; save()
        gc.collect()
    free_combo(combo)


def main():
    os.makedirs(OUT, exist_ok=True); check_inputs(); W = []
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    if not S:   # tiếp nối từ lượt chạy trước (Add output cùng tên làm Input)
        import shutil
        for fj in sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}/summary.json", recursive=True)):
            for f in glob.glob(os.path.join(os.path.dirname(fj), "*")):
                if f.endswith((".json", ".npz", ".npy")): shutil.copy(f, OUT)
            S = json.load(open(f"{OUT}/summary.json")); log(f"tiếp nối <- {fj}"); break
    S["config"] = (f"{PREFIX}{'d' if MODE == 'maps' else 'e'} mô={TISSUE} combos={COMBOS} methods={METHODS} "
                   f"sàn={FLOOR} seeds={SEEDS}" + (" | SMOKE" if SMOKE else ""))
    S.setdefault("methods", {}); S["final_floor"] = FLOOR; S["tissue"] = TISSUE
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    for pat in (f"/kaggle/input/**/out{PREFIX}d/*.np[yz]", f"{OUT}/*.np[yz]"):
        for f in glob.glob(pat, recursive=True): R22.PRIOR.setdefault(os.path.basename(f), f)
    log(f"map USHER sẵn có: {len(R22.PRIOR)} file")
    needed = sorted({c.split("/")[0] for c in COMBOS})
    D, F, PCTX = R18.setup_pairs(needed)
    km8 = R20.KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(R17.get_niche_label_comp(needed[0], S, save)[1])
    R20.KM16 = R20.KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(R20.fine16_label_comp(needed[0], S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    try:
        py = None if MODE == "maps" else (R20.setup_venv_b(W) if (
            WITH_SYMPHONY and any("symphony_h" not in S["methods"].get(c, {}) for c in COMBOS)) else None)
        for c in COMBOS:   # một tổ hợp hỏng không được giết các tổ hợp còn lại
            try:
                if MODE == "maps": usher_seeds(c, R22.ctx(c), W)
                else: process_combo(c, py, W)
            except R22.OutOfTime: raise
            except Exception as e:
                W.append(f"{c}: BỎ QUA, lỗi {e!r}"); log(f"[{c}] LỖI -> bỏ qua: {e!r}"); S["methods"].setdefault(c, {})
            finally:
                free_combo(c); save()
    except R22.OutOfTime as e:
        W.append(f"HẾT NGÂN SÁCH THỜI GIAN trước {e}; Add output {os.path.basename(OUT)} làm Input rồi chạy lại (tự tiếp nối).")
    S["warnings"] = list(dict.fromkeys(W)); save()
    log(f"XONG {PREFIX}{'d' if MODE == 'maps' else 'e'} ({TISSUE}). File: {sorted(os.listdir(OUT))}\nCảnh báo:\n" + "\n".join(S["warnings"] or ["không có"]))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run26f.py
"""26f/27f — CHẨN ĐOÁN (CPU, không chạy USHER, không tính scib): vì sao USHER-ST trộn lô ít hơn CORAL?
Với mỗi map đã có, phục hồi phép affine thực tế Fmap ≈ Fs·A + b rồi đo:
  mu_gap    = ||TB(Fmap[T]) − TB(Fx)|| / sqrt(tr(Cx))     -> map có kéo trọng tâm về Xenium không (center/CORAL = 0)
  var_ratio = tr(cov(Fmap[T])) / tr(cov(Fx))              -> tổng phương sai có khớp Xenium không (1.0 = khớp)
  shape_err = ||Cx^(-1/2)·C_map·Cx^(-1/2) − I||_F / sqrt(d) -> hiệp phương sai có khớp không (CORAL ≈ 0 theo thiết kế)
  dI        = ||A − I||_F / sqrt(d), cond(A), r2          -> map lệch khỏi phép đồng nhất bao nhiêu
Giả thuyết cần kiểm: USHER-ST sửa trọng tâm nhưng KHÔNG sửa hiệp phương sai (λ_id kéo A về I), nên ở mô mà
lệch giữa hai công nghệ chủ yếu là bậc hai (HCC) thì nó trộn lô kém hẳn CORAL.
Env: TISSUE, COMBOS_ALL, RUN_PREFIX (tìm map ở out<P>d), FINAL_FLOOR.
"""
import os, sys, json
os.environ.setdefault("MODE26", "report")
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
import run26d as R26
np, pd, log, M = R26.np, R26.pd, R26.log, R26.M
R22, R20, R17, R18, R23, R23C = R26.R22, R26.R20, R26.R17, R26.R18, R26.R23, R26.R23C
P = R26.PREFIX
OUT = os.environ.get("OUT_DIR_26F", f"/kaggle/working/out{P}f")
for mod in (R17, R18, R26.R19, R20, R22, R22.R21, R23, R23C): mod.OUT = OUT
COMBOS, FLOOR, N_FIT = R26.COMBOS, R26.FLOOR, 20000


def _mat_pow(C, p):
    w, V = np.linalg.eigh(C); w = np.clip(w, 1e-10 * max(w.max(), 1e-12), None)
    return (V * w ** p) @ V.T

def diag(Fs, Fmap, Fx, T):
    d = Fx.shape[1]
    Cx = np.cov(Fx.astype(np.float64), rowvar=False) + 1e-8 * np.eye(d)
    Cm = np.cov(Fmap[T].astype(np.float64), rowvar=False)
    Wx = _mat_pow(Cx, -0.5)
    idx = np.sort(np.random.RandomState(0).choice(len(Fs), min(N_FIT, len(Fs)), replace=False))
    X = np.hstack([Fs[idx].astype(np.float64), np.ones((len(idx), 1))]); Y = Fmap[idx].astype(np.float64)
    AB, *_ = np.linalg.lstsq(X, Y, rcond=None); A = AB[:-1]
    r2 = 1 - ((Y - X @ AB) ** 2).sum() / max(((Y - Y.mean(0)) ** 2).sum(), 1e-12)
    sv = np.linalg.svd(A, compute_uv=False)
    return {"mu_gap": round(float(np.linalg.norm(Fmap[T].mean(0) - Fx.mean(0)) / np.sqrt(np.trace(Cx))), 4),
            "var_ratio": round(float(np.trace(Cm) / np.trace(Cx)), 4),
            "shape_err": round(float(np.linalg.norm(Wx @ Cm @ Wx - np.eye(d)) / np.sqrt(d)), 4),
            "dI": round(float(np.linalg.norm(A - np.eye(d)) / np.sqrt(d)), 4),
            "cond_A": round(float(sv[0] / max(sv[-1], 1e-12)), 1),
            "sv_med": round(float(np.median(sv)), 4), "r2_affine": round(float(r2), 4)}


def main():
    os.makedirs(OUT, exist_ok=True); R26.check_inputs()
    S = {"config": f"{P}f chẩn đoán affine mô={R26.TISSUE} combos={COMBOS}", "diag": {}}
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    import glob
    for f in glob.glob(f"/kaggle/input/**/out{P}d/*.np[yz]", recursive=True): R22.PRIOR.setdefault(os.path.basename(f), f)
    log(f"map USHER sẵn có: {len(R22.PRIOR)} file")
    D, F, PCTX = R18.setup_pairs(sorted({c.split("/")[0] for c in COMBOS}))
    km8 = R20.KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(R17.get_niche_label_comp(sorted({c.split('/')[0] for c in COMBOS})[0], S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    L = []
    for c in COMBOS:
        try:
            cx = R22.ctx(c); Fs, Fx, T = cx["Fs"], cx["Fx"], cx["ds"]["T"]
            maps = {"raw": Fs, "center": M.base_center(Fs, Fx, T), "coral": R23C.coral(Fs, Fx, T)}
            for sd in (0, 1, 2):
                m_ = R22.get_map(c, f"f{FLOOR}", sd, None)[0]
                if m_ is not None: maps[f"usher_s{sd}"] = m_
            rows = {}
            for nm, Fm in maps.items():
                rows[nm] = diag(Fs, np.asarray(Fm, np.float32), Fx, T)
                log(f"[{c}] {nm}: {rows[nm]}")
            S["diag"][c] = rows; save()
            L += [f"### {c}  (d={Fx.shape[1]}, |T|={len(T)})", pd.DataFrame(rows).T.to_string(), ""]
        except Exception as e:
            L.append(f"### {c}: LỖI {e!r}"); log(f"[{c}] LỖI {e!r}")
        R26.free_combo(c)
    txt = ("CHẨN ĐOÁN AFFINE — mu_gap 0 = khớp trọng tâm Xenium | var_ratio 1 = khớp tổng phương sai |\n"
           "shape_err 0 = khớp hiệp phương sai | dI 0 = map là phép đồng nhất | r2_affine 1 = map đúng là affine\n\n" + "\n".join(L))
    open(f"{OUT}/report.txt", "w").write(txt); log("\n" + txt + f"\nXONG {P}f")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run26g.py
"""26g/27g — thử hai bản vá HẬU KỲ cho map USHER-ST (CPU, KHÔNG chạy lại USHER, dùng map của <P>d):
  usher_c   = USHER + dời trọng tâm về Xenium                      (1 dòng, không đụng hướng/hình dạng)
  usher_cc  = USHER + khớp cả hiệp phương sai (CORAL hậu kỳ trên chính map USHER)
Chẩn đoán 27f cho thấy map USHER còn lệch trọng tâm (mu_gap 0.15-0.29, center/CORAL = 0) và BỊ CO
(var_ratio 0.29-0.63, Xenium = 1). Hai bản vá này kiểm xem phần trộn lô thiếu có phải do đúng hai thứ đó không,
và quan trọng hơn: vá xong thì CHUYỂN NHÃN có giữ được không (nếu mất thì bản vá vô dụng).
Chấm đủ bộ chỉ số như <P>e + UMAP, gộp với out<P>e để so thẳng. Env: TISSUE, COMBOS_ALL, RUN_PREFIX, FINAL_FLOOR.
"""
import os, sys, json, glob, gc
os.environ.setdefault("MODE26", "report")
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
import run26d as R26
import run26f as R26F
np, pd, log, M = R26.np, R26.pd, R26.log, R26.M
R22, R20, R17, R18, R23, R23C = R26.R22, R26.R20, R26.R17, R26.R18, R26.R23, R26.R23C
P = R26.PREFIX
OUT = os.environ.get("OUT_DIR_26G", f"/kaggle/working/out{P}g")
for mod in (R17, R18, R26.R19, R20, R22, R22.R21, R23, R23C, R26F): mod.OUT = OUT
COMBOS, FLOOR = R26.COMBOS, R26.FLOOR
SEEDS = [0] if R26.SMOKE else [int(x) for x in os.environ.get("PATCH_SEEDS", "0,1,2").split(",")]
# Sàn khác để đối chiếu (chỉ seed 0): sàn 200 được chọn ở 24a CHO PIPELINE CHƯA VÁ; sau khi vá mô-men,
# lý do chính của sàn (cứu nhóm hiếm) đã được bản vá lo phần lớn -> phải kiểm lại. COAD có sẵn map sàn 0 ở out21.
ALT_FLOORS = [int(x) for x in os.environ.get("ALT_FLOORS", "").replace(" ", "").split(",") if x]
# Map từ notebook 28 (CORAL-init / sàn 0), chấm theo tag: u_<tag> (thô) và u_<tag>_cc (+ khớp mô-men)
EXTRA_MAPS = [t for t in os.environ.get("EXTRA_MAPS", "").replace(" ", "").split(",") if t]
ALIAS = {200: "tgt_floor", 0: "default"}
NEW = ["usher_c", "usher_cc"]
WITH_PROC = os.environ.get("WITH_PROC", "1") == "1"
# Baseline ĐƯỢC CẤP CÙNG THÔNG TIN NHÃN như M_aux (nhãn giả `lab_def`), rồi khớp mô-men như usher_cc.
# Mục đích: tách "tương ứng do OT tìm" khỏi "tương ứng do nhãn áp đặt" ở CÙNG mô-men bậc hai (logic mục 4e).
# Cả hai TẤT ĐỊNH (rng cố định, nghiệm đóng) -> chấm 1 lần, SD = 0, cùng hạng với center/CORAL.
WITH_CLASSPL = os.environ.get("WITH_CLASSPL", "0") == "1"   # đối chứng Procrustes trực giao (CPU, vài giây mỗi cặp)
# UMAP tham chiếu: tự dựng lại nếu không gộp được từ <P>e, để Hình 2-3 luôn đủ cột (chỉ UMAP, KHÔNG chấm lại)
REF = [m.strip() for m in os.environ.get("UMAP_REF", "center,coral,usher_final").split(",") if m.strip()]


SCANO_RETRY = os.environ.get("SCANO_RETRY", "1") == "1"
_CHK = ["ilisi", "kbet", "bras", "graph_conn", "batch_asw", "nmi", "ari", "asw", "clisi", "xfer_main", "rare_purity"]


def scano_noop(Sc):
    """Scanorama trả về ĐÚNG đầu vào khi find_alignments không tìm được cặp láng giềng tương hỗ nào vượt
    ngưỡng alpha (mặc định 0.10) -> nó KHÔNG hiệu chỉnh gì. Phát hiện bằng cách so trùng khít với raw."""
    a, b = Sc.get("scanorama"), Sc.get("raw")
    return bool(a and b and all(a.get(k) == b.get(k) for k in _CHK))


def scanorama_at(Fx, Fs, alpha=0.0, knn=30):
    """Chạy lại Scanorama với ngưỡng nới lỏng, để baseline được thể hiện ở điều kiện tốt nhất của nó.
    LƯU Ý: Scanorama sửa CẢ embedding tham chiếu (như Harmony) -> vi phạm ràng buộc 'tham chiếu không đổi'."""
    import scanorama
    o = scanorama.assemble([np.asarray(Fx, np.float32), np.asarray(Fs, np.float32)],
                           batch_size=5000, alpha=alpha, knn=knn)
    return np.asarray(o[0], np.float32), np.asarray(o[1], np.float32)


def procrustes_map(Fs, Fx, T, ybar, has):
    """ĐỐI CHỨNG HẬU NGHIỆM, KHÔNG phải phương án thay thế vòng EM.

    Các cặp ghép 1-1 (thể hiện qua đích barycentric ȳ) là SẢN PHẨM của toàn bộ vòng lặp E-M xen kẽ —
    không có M-step thì không có cặp ghép. Vì vậy Procrustes chỉ trả lời đúng một câu hỏi có điều kiện:
    *CHO TRƯỚC các cặp ghép mà pipeline đã tìm ra*, một phép QUAY CỨNG tối ưu có đủ để dựng lại map cuối không,
    hay map affine của M-step cuối còn mang thêm thông tin ngoài phép quay?
    Nó KHÔNG cho phép kết luận "bỏ được M-step".

    Mọi map khớp đúng mô-men với Xenium đều có dạng M(x) = C_x^(1/2)·R·C_s^(-1/2)·(x − μ_s) + μ_x với R trực giao.
    CORAL là R = I. Ở đây R = nghiệm Procrustes từ chính các cặp ghép: R = U·Vᵀ với U,Σ,Vᵀ = SVD(X̃ᵀỸ).
    """
    d = Fx.shape[1]
    mu_s, mu_x = Fs[T].mean(0).astype(np.float64), Fx.mean(0).astype(np.float64)
    Cs = np.cov(Fs[T].astype(np.float64), rowvar=False) + 1e-8 * np.eye(d)
    Cx = np.cov(Fx.astype(np.float64), rowvar=False) + 1e-8 * np.eye(d)
    Ws, Wxi, Wx = R26F._mat_pow(Cs, -0.5), R26F._mat_pow(Cx, -0.5), R26F._mat_pow(Cx, 0.5)
    m = np.asarray(has).astype(bool)
    X = (Fs[T][m].astype(np.float64) - mu_s) @ Ws
    Y = (np.asarray(ybar)[m].astype(np.float64) - mu_x) @ Wxi
    U, _, Vt = np.linalg.svd(X.T @ Y, full_matrices=False)
    R = U @ Vt
    Fm = ((Fs.astype(np.float64) - mu_s) @ Ws @ R @ Wx + mu_x).astype(np.float32)
    return Fm, R, Ws, Wx, int(m.sum())


def rot_of(Fs, Fmap, Ws, Wx, T):
    """Phép quay ẩn trong một map đã khớp mô-men: R = C_s^(1/2)·A·C_x^(-1/2) với A phục hồi bằng bình phương tối thiểu."""
    idx = np.sort(np.random.RandomState(0).choice(len(Fs), min(20000, len(Fs)), replace=False))
    X = np.hstack([Fs[idx].astype(np.float64), np.ones((len(idx), 1))])
    A = np.linalg.lstsq(X, Fmap[idx].astype(np.float64), rcond=None)[0][:-1]
    return np.linalg.pinv(Ws) @ A @ np.linalg.pinv(Wx)


def has_ybar(c, tag, sd):
    b = f"{c.replace('/', '_')}_{tag}_s{sd}_ybar.npz"
    alt = f"{c.replace('/', '_')}_tgt_floor_ybar_s{sd}.npz" if tag == f"f{FLOOR}" else None
    return os.path.exists(f"{OUT}/{b}") or b in R22.PRIOR or (alt is not None and alt in R22.PRIOR)


def has_map(c, tag, sd):
    base = f"{c.replace('/', '_')}_{tag}_s{sd}"
    alt = f"{c.replace('/', '_')}_default_s{sd}.npy" if tag == "f0" else None
    return (os.path.exists(f"{OUT}/{base}.npy") or (base + ".npy") in R22.PRIOR
            or (alt is not None and alt in R22.PRIOR))


def variants(Fs, Fx, T, Fu):
    """Fu = map USHER. usher_c: chỉ dời trọng tâm. usher_cc: CORAL hậu kỳ (dời + khớp hiệp phương sai)."""
    return {"usher_c": (Fu + (Fx.mean(0) - Fu[T].mean(0))).astype(np.float32),
            "usher_cc": R23C.coral(Fu, Fx, T)}


def main():
    os.makedirs(OUT, exist_ok=True); W = []
    if os.environ.get("SKIP_INPUT_CHECK", "0") != "1": R26.check_inputs()
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    if not S:
        for fj in sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}/summary.json", recursive=True)):
            import shutil
            for f in glob.glob(os.path.join(os.path.dirname(fj), "*")):
                if f.endswith((".json", ".npz")): shutil.copy(f, OUT)
            S = json.load(open(f"{OUT}/summary.json")); log(f"tiếp nối <- {fj}"); break
    S.setdefault("methods", {}); S.setdefault("diag", {})
    # gộp kết quả <P>e để bảng so sánh có đủ center / CORAL / USHER gốc
    srcs = [f for pat in (f"out{P}e", "out23d", "out23b", "out23c")
            for f in sorted(glob.glob(f"/kaggle/input/**/{pat}/summary.json", recursive=True))]
    for fj in srcs:
        Se = json.load(open(fj))
        for c, d in Se.get("methods", {}).items():
            for m, v in d.items(): S["methods"].setdefault(c, {}).setdefault(m, v)
        for k_ in ("usher_seeds", "rare", "tissue"):
            if Se.get(k_) is not None: S.setdefault(k_, Se[k_])
        for k_ in ("patch_seeds", "diag"):   # giữ lại điểm đã chấm của bản vá -> không chấm lại 3 seed
            for c_, v_ in (Se.get(k_) or {}).items():
                for m_, d_ in v_.items(): S.setdefault(k_, {}).setdefault(c_, {}).setdefault(m_, d_)
        log(f"gộp số liệu <- {fj} ({sorted({m for d in Se.get('methods', {}).values() for m in d})})")
    for fj in sorted(glob.glob(f"/kaggle/input/**/out{P}i/summary.json", recursive=True)):   # nội suy của vòng 30
        Si = json.load(open(fj))
        for c_, d_ in (Si.get("imp") or {}).items():
            for m_, v_ in d_.items(): S.setdefault("imp", {}).setdefault(c_, {}).setdefault(m_, v_)
        if Si.get("imp_meta"): S.setdefault("imp_meta", {}).update(Si["imp_meta"])
        log(f"gộp nội suy <- {fj}")
    S["config"] = (f"{P}g vá hậu kỳ mô={R26.TISSUE} combos={COMBOS} seeds={SEEDS} sàn={FLOOR} "
                   f"alt_floors={ALT_FLOORS} scano_retry={SCANO_RETRY}"); S["final_floor"] = FLOOR
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    # out<P>j = map C_niche/C_mix của vòng 31. QUÉT /kaggle/working TRƯỚC: khi 26g chạy CHUNG phiên với 31
    # (notebook 32), map seed 1/2 vừa tạo nằm ở working, chưa phải Input -> thiếu dòng này thì chỉ chấm được seed 0.
    R22.scan_maps(P, R22.PRIOR)   # danh sách thư mục map: run22._SUFFIX (sửa gốc 2026-10-07)
    log(f"map USHER sẵn có: {len(R22.PRIOR)} file")
    D, F, PCTX = R18.setup_pairs(sorted({c.split("/")[0] for c in COMBOS}))
    p0 = sorted({c.split("/")[0] for c in COMBOS})[0]
    km8 = R20.KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(R17.get_niche_label_comp(p0, S, save)[1])
    R20.KM16 = R20.KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(R20.fine16_label_comp(p0, S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    for c in COMBOS:
        try:
            Sc = S["methods"].setdefault(c, {})
            bn = f"umap_{c.replace('/', '_')}.npz"; f_um = f"{OUT}/{bn}"
            um = dict(np.load(f_um, allow_pickle=True)) if os.path.exists(f_um) else {}
            # gộp UMAP đã tính của các phương pháp cũ -> Hình 2-3 có đủ mọi cột trên cùng một lưới
            for pat in (f"out{P}e", "out23d", "out23b", "out23c"):
                for fe in sorted(glob.glob(f"/kaggle/input/**/{pat}/{bn}", recursive=True)):
                    e = dict(np.load(fe, allow_pickle=True))
                    if um and not (np.array_equal(um.get("ix"), e.get("ix")) and np.array_equal(um.get("is_"), e.get("is_"))):
                        W.append(f"{c}: mẫu UMAP của {pat} khác -> không gộp"); continue
                    for k_, v_ in e.items(): um.setdefault(k_, v_)
                    log(f"[{c}] gộp UMAP <- {fe} ({sorted(set(e) - {'ix', 'is_', 'lab_x', 'lab_s'})})")
            PS = S.setdefault("patch_seeds", {}).setdefault(c, {})
            need_ref = [m for m in REF if m not in um]
            # Map EXTRA_MAPS thiếu thì điều kiện `done` dưới đây ĐÚNG MỘT CÁCH RỖNG (vì nó có `if has_map`),
            # nên combo bị bỏ qua với dòng "đã xong" mà bảng u_* trống trơn và summary.json không nói gì.
            # Đã xảy ra ở vòng 34 lần 2 (map b05 nằm ở out<P>k, không có trong danh sách quét). Báo to.
            miss_x = [f"{t}@s{sd}" for t in EXTRA_MAPS for sd in SEEDS if not has_map(c, t, sd)]
            if miss_x:
                W.append(f"{c}: KHÔNG tìm thấy map EXTRA_MAPS {miss_x} -> các dòng u_* tương ứng SẼ TRỐNG "
                         f"(kiểm notebook tạo map đã chạy chưa, và thư mục của nó có trong danh sách quét chưa)")
                log(f"[{c}] THIẾU map EXTRA_MAPS: {miss_x}")
            done = all(all(f"s{sd}" in PS.get(m, {}) for sd in SEEDS) for m in NEW) \
                   and all(f"usherf{f}_{t}" in Sc for f in ALT_FLOORS for t in ("c", "cc")) \
                   and all(f"s{sd}" in PS.get(f"u_{t}{x}", {}) for t in EXTRA_MAPS for sd in SEEDS
                           if has_map(c, t, sd) for x in ("", "_cc")) \
                   and (not WITH_PROC or "usher_proc" in Sc or not has_ybar(c, f"f{FLOOR}", SEEDS[0])) \
                   and (not WITH_CLASSPL or "class_ridge_pl_cc" in Sc)
            if done and all(m in um for m in NEW) and not need_ref:
                np.savez(f_um, **um); log(f"[{c}] đã xong"); R26.free_combo(c); continue
            cx = R22.ctx(c); Fs, Fx, ds, T = cx["Fs"], cx["Fx"], cx["ds"], cx["ds"]["T"]
            def load(fl, sd):
                m_ = R22.get_map(c, f"f{fl}", sd, None, ALIAS.get(fl))[0]
                return None if m_ is None else np.asarray(m_, np.float32)
            Fu = load(FLOOR, SEEDS[0])
            if Fu is None: W.append(f"{c}: thiếu map (Add output {P}d)"); R26.free_combo(c); continue
            rng = np.random.RandomState(0)
            ix = np.sort(rng.choice(cx["pctx"]["ref"], min(R23.N_UMAP, len(cx["pctx"]["ref"])), replace=False))
            is_ = np.sort(rng.choice(ds["H"], min(R23.N_UMAP, len(ds["H"])), replace=False))
            if um and not (np.array_equal(um.get("ix"), ix) and np.array_equal(um.get("is_"), is_)):
                W.append(f"{c}: mẫu UMAP cũ khác mẫu mới -> bỏ UMAP cũ"); um = {}
            um.update(ix=ix, is_=is_, lab_x=cx["dx"]["lab"][ix].astype(str), lab_s=ds["lab"][is_].astype(str))
            V = variants(Fs, Fx, T, Fu)
            for nm in need_ref:   # chỉ TÍNH UMAP cho baseline (điểm số lấy từ summary của <P>e)
                Fm = {"center": lambda: M.base_center(Fs, Fx, T), "coral": lambda: R23C.coral(Fs, Fx, T),
                      "raw": lambda: Fs, "usher_final": lambda: Fu}.get(nm, lambda: V.get(nm))()
                if Fm is None: continue
                R22.step(f"[{c}] UMAP {nm}")
                um[nm] = R23.umap2(Fx, np.asarray(Fm, np.float32), ix, is_); np.savez(f_um, **um)
                log(f"[{c}] UMAP {nm}: xong"); gc.collect()
            for sd in SEEDS:   # 3 seed cho chính bản vá (map s1, s2 đã có sẵn, chỉ chấm thêm)
                Fu_s = Fu if sd == SEEDS[0] else load(FLOOR, sd)
                if Fu_s is None: W.append(f"{c}: thiếu map seed {sd}"); continue
                for nm, Fm in variants(Fs, Fx, T, Fu_s).items():
                    if f"s{sd}" in PS.get(nm, {}): continue
                    R22.step(f"[{c}] {nm} s{sd}")
                    ev = R23.evaluate(cx, f"{c}/{nm}/s{sd}", Fs, Fm, Fx, W)
                    PS.setdefault(nm, {})[f"s{sd}"] = ev
                    if sd == SEEDS[0]:
                        Sc[nm] = ev; S["diag"].setdefault(c, {})[nm] = R26F.diag(Fs, Fm, Fx, T)
                        um[nm] = R23.umap2(Fx, Fm, ix, is_); np.savez(f_um, **um)
                    save(); log(f"[{c}] {nm} s{sd}: xfer_main={ev['xfer_main']} total_v2={ev.get('total_v2')} rare={ev.get('rare_purity')}")
                    gc.collect()
            if SCANO_RETRY and scano_noop(Sc) and "scanorama_a0" not in Sc:
                W.append(f"{c}: Scanorama MẶC ĐỊNH không hiệu chỉnh gì (trùng khít raw) -> chạy lại với alpha=0, knn=30")
                try:
                    R22.step(f"[{c}] scanorama_a0")
                    Fx2, Fm = scanorama_at(Fx, Fs, 0.0, 30)
                    Sc["scanorama_a0"] = R23.evaluate(cx, f"{c}/scanorama_a0", Fm, Fm, Fx2, W)
                    Sc["scanorama_a0"]["_noop_default"] = True
                    save(); log(f"[{c}] scanorama_a0: xfer_main={Sc['scanorama_a0']['xfer_main']} "
                                f"total_v2={Sc['scanorama_a0'].get('total_v2')} (mặc định: không hiệu chỉnh)")
                except Exception as e: W.append(f"{c} scanorama_a0: lỗi {e!r}")
                gc.collect()
            if WITH_PROC and "usher_proc" not in Sc:   # đối chứng Procrustes (hậu nghiệm, cho trước cặp ghép)
                _, ybar, hastg = R22.get_map(c, f"f{FLOOR}", SEEDS[0], None, ALIAS.get(FLOOR))
                if ybar is None or hastg is None:
                    W.append(f"{c}: không có ȳ (file *_ybar.npz) -> bỏ đối chứng Procrustes")
                else:
                    try:
                        R22.step(f"[{c}] usher_proc")
                        Fp, Rp, Ws_, Wx_, npair = procrustes_map(Fs, Fx, T, ybar, hastg)
                        Sc["usher_proc"] = R23.evaluate(cx, f"{c}/usher_proc", Fs, Fp, Fx, W)
                        dg = R26F.diag(Fs, Fp, Fx, T)
                        Ru = rot_of(Fs, np.asarray(variants(Fs, Fx, T, Fu)["usher_cc"], np.float32), Ws_, Wx_, T)
                        dg["n_pair"] = npair
                        dg["rot_diff"] = round(float(np.linalg.norm(Rp - Ru) / np.sqrt(2 * Fx.shape[1])), 4)
                        S["diag"].setdefault(c, {})["usher_proc"] = dg
                        save(); log(f"[{c}] usher_proc: xfer_main={Sc['usher_proc']['xfer_main']} "
                                    f"total_v2={Sc['usher_proc'].get('total_v2')} | {dg}")
                    except Exception as e: W.append(f"{c} usher_proc: lỗi {e!r}")
                    gc.collect()
            # map của notebook 28/31. Chấm MỌI seed có map (seed 0 vào bảng chính, mọi seed vào patch_seeds -> có SD).
            for tag in EXTRA_MAPS:
                if not has_map(c, tag, SEEDS[0]):
                    # CẢNH BÁO, không chỉ log: thiếu map ở đây từng làm bảng phán quyết vòng 34 rỗng mà
                    # summary.json không hề nói gì (bước 1 GPU chết, bước 3 im lặng bỏ qua).
                    W.append(f"{c}: KHÔNG có map {tag} (seed {SEEDS[0]}) -> bỏ qua; chạy notebook tạo map trước")
                    log(f"[{c}] chưa có map {tag} -> bỏ qua (chạy notebook 28/31/34 trước)"); continue
                for sd in SEEDS:
                    if not has_map(c, tag, sd):
                        if sd != SEEDS[0]: log(f"[{c}] {tag}: chưa có seed {sd} -> bỏ qua")
                        continue
                    Fe = R22.get_map(c, tag, sd, None, "default" if tag == "f0" else None)[0]
                    if Fe is None: W.append(f"{c}: thiếu map {tag} s{sd}"); continue
                    Fe = np.asarray(Fe, np.float32)
                    for nm, Fm in (("u_" + tag, Fe), ("u_" + tag + "_cc", R23C.coral(Fe, Fx, T))):
                        # UMAP cho map EXTRA (thiếu từ đầu -> u_cmix_cc không có hình UMAP nào, phát hiện 2026-10-08)
                        if sd == SEEDS[0] and nm not in um:
                            um[nm] = R23.umap2(Fx, np.asarray(Fm, np.float32), ix, is_)
                            np.savez(f_um, **um); log(f"[{c}] UMAP {nm}: xong")
                        if f"s{sd}" in PS.get(nm, {}): continue
                        R22.step(f"[{c}] {nm} s{sd}")
                        ev = R23.evaluate(cx, f"{c}/{nm}/s{sd}", Fs, Fm, Fx, W)
                        PS.setdefault(nm, {})[f"s{sd}"] = ev
                        if sd == SEEDS[0]:
                            Sc[nm] = ev; S["diag"].setdefault(c, {})[nm] = R26F.diag(Fs, Fm, Fx, T)
                        save(); log(f"[{c}] {nm} s{sd}: xfer_main={ev['xfer_main']} total_v2={ev.get('total_v2')} "
                                    f"rare={ev.get('rare_purity')}")
                        gc.collect()
            if WITH_CLASSPL:
                lab_pl = cx["lab_def"]   # nhãn giả Y HỆT thứ M_aux dùng -> thông tin khớp tuyệt đối
                # class_shift BỊ BỎ: nó là tịnh tiến thuần (Fs + d), mà CORAL bất biến với tịnh tiến
                # (Cov không đổi, rồi dời về mu_x) -> class_shift_pl_cc ĐỒNG NHẤT với CORAL. Chứng minh được,
                # không cần đo. Giữ lại như một nhận xét giải tích trong bài: khớp mô-men XOÁ mọi tịnh tiến dùng nhãn.
                for nm, mk in (("class_ridge_pl", lambda: M.base_class_ridge(Fs, Fx, T, lab_pl, cx["dx"]["lab"])[0]),):
                    if nm + "_cc" in Sc: continue
                    try:
                        R22.step(f"[{c}] {nm}_cc")
                        Fb = np.asarray(mk(), np.float32)
                        Fm = R23C.coral(Fb, Fx, T)   # khớp mô-men -> cùng mặt phẳng với usher_cc
                        Sc[nm + "_cc"] = R23.evaluate(cx, f"{c}/{nm}_cc", Fs, Fm, Fx, W)
                        S["diag"].setdefault(c, {})[nm + "_cc"] = R26F.diag(Fs, Fm, Fx, T)
                        save(); log(f"[{c}] {nm}_cc: xfer_main={Sc[nm+'_cc']['xfer_main']} "
                                    f"total_v2={Sc[nm+'_cc'].get('total_v2')} rare={Sc[nm+'_cc'].get('rare_purity')}")
                    except Exception as e: W.append(f"{c} {nm}_cc: lỗi {e!r}")
                    gc.collect()
            for fl in ALT_FLOORS:   # đối chiếu sàn khác, chỉ seed 0
                Fa = load(fl, SEEDS[0])
                if Fa is None: W.append(f"{c}: thiếu map sàn {fl}"); continue
                for t, Fm in variants(Fs, Fx, T, Fa).items():
                    nm = f"usherf{fl}_{t.split('_')[1]}"
                    if nm in Sc: continue
                    R22.step(f"[{c}] {nm}")
                    Sc[nm] = R23.evaluate(cx, f"{c}/{nm}", Fs, Fm, Fx, W)
                    S["diag"].setdefault(c, {})[nm] = R26F.diag(Fs, Fm, Fx, T)
                    save(); log(f"[{c}] {nm}: xfer_main={Sc[nm]['xfer_main']} total_v2={Sc[nm].get('total_v2')} rare={Sc[nm].get('rare_purity')}")
                    gc.collect()
        except R22.OutOfTime: W.append(f"HẾT GIỜ trước {c}; Add out{P}g rồi chạy lại"); break
        except Exception as e: W.append(f"{c}: lỗi {e!r}"); log(f"[{c}] LỖI {e!r}")
        R26.free_combo(c); save()
    S["scanorama_noop"] = sorted(c for c, d in S["methods"].items() if d and scano_noop(d))
    if S["scanorama_noop"]: W.append("Scanorama không hiệu chỉnh (trùng khít raw) ở: " + ", ".join(S["scanorama_noop"]))
    S["warnings"] = list(dict.fromkeys(W)); save(); log(f"XONG {P}g. Cảnh báo: " + str(S["warnings"] or "không có"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run26h.py
"""28 (run26h) — Map USHER với KHỞI TẠO CORAL (GPU, chỉ tạo map, không chấm điểm).

Trả lời hai câu còn lại:
  (1) Sửa TRONG thuật toán có hơn vá hậu kỳ không? -> chạy USHER trên source đã CORAL hoá ngay từ đầu.
  (2) Bỏ sàn target được không? -> tạo map sàn 0 cho OV/HCC (COAD đã có sẵn ở out21 dưới tên alias "default").

Cấu hình tạo map (1 seed, đủ để trả lời; 3 seed chỉ chạy nếu kết quả đáng theo):
  ci200 = CORAL-init, sàn 200   |   ci0 = CORAL-init, sàn 0   |   f0 = USHER thường, sàn 0
Chấm điểm ở notebook `g` (CPU) bằng env EXTRA_MAPS. Env: TISSUE, COMBOS_ALL, RUN_PREFIX, CINIT_SEED, CONFIGS.
"""
import os, sys, glob, gc, json
os.environ.setdefault("MODE26", "maps")
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
sys.path.insert(0, os.environ.get("USHER_DIR", "/kaggle/working/usher"))
import run26d as R26
import run23c as R23C
np, log = R26.np, R26.log
R22, R20, R17, R18 = R26.R22, R26.R20, R26.R17, R26.R18
P = R26.PREFIX
OUT = os.environ.get("OUT_DIR_26H", f"/kaggle/working/out{P}h")
for mod in (R17, R18, R26.R19, R20, R22, R22.R21, R26.R23, R23C): mod.OUT = OUT
COMBOS, SEED = R26.COMBOS, int(os.environ.get("CINIT_SEED", "0"))
# (tag, sàn, dùng CORAL-init). Tag quyết định tên file map -> không đụng map cũ.
ALL_CFG = {"ci200": (200, True), "ci0": (0, True), "f0": (0, False)}
CFG = [(t, *ALL_CFG[t]) for t in os.environ.get("CONFIGS", "ci200,ci0,f0").replace(" ", "").split(",") if t in ALL_CFG]


def cinit_ctx(cx):
    """Bản sao context với source đã CORAL hoá; M_aux dựng lại trên source mới (vì nó phụ thuộc Fs)."""
    Fs_c = R23C.coral(cx["Fs"], cx["Fx"], cx["ds"]["T"])
    lab_def, P_def, _ = R22.R21.aux_center(Fs_c, cx["Fx"], cx["ds"], cx["dx"], cx["pctx"]["classes"], R17.K_AUX)
    return {**cx, "Fs": Fs_c, "lab_def": lab_def, "P_def": P_def}


def main():
    os.makedirs(OUT, exist_ok=True)
    if os.environ.get("SKIP_INPUT_CHECK", "0") != "1": R26.check_inputs()
    W = []
    S = {"config": f"{P}h CORAL-init mô={R26.TISSUE} combos={COMBOS} cfg={[c[0] for c in CFG]} seed={SEED}"}
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    for pat in (f"out{P}h", f"out{P}d", "out23a", "out22", "out21"):
        for f in glob.glob(f"/kaggle/input/**/{pat}/*.np[yz]", recursive=True): R22.PRIOR.setdefault(os.path.basename(f), f)
    log(f"map sẵn có: {len(R22.PRIOR)} file | cấu hình: {[c[0] for c in CFG]}")
    D, F, PCTX = R18.setup_pairs(sorted({c.split("/")[0] for c in COMBOS}))
    p0 = sorted({c.split("/")[0] for c in COMBOS})[0]
    km8 = R20.KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(R17.get_niche_label_comp(p0, S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    try:
        for c in COMBOS:
            try:
                cx = R22.ctx(c); cxc = None
                for tag, floor, cinit in CFG:
                    if cinit and cxc is None: R22.step(f"[{c}] CORAL-init context"); cxc = cinit_ctx(cx)
                    cu = cxc if cinit else cx
                    T = cu["ds"]["T"]
                    alias = "default" if (floor == 0 and not cinit) else None   # COAD: map sàn 0 đã có ở out21
                    R22.get_map(c, tag, SEED, lambda cu=cu, T=T, floor=floor: R22.usher_run(
                        cu, R22.R21.strat_target_floor(cu["lab_def"][T], cu["dx"]["lab"], floor=floor), SEED), alias)
                    log(f"[{c}] {tag}: xong")
                    gc.collect()
            except R22.OutOfTime: raise
            except Exception as e: W.append(f"{c}: lỗi {e!r}"); log(f"[{c}] LỖI {e!r}")
            R26.free_combo(c)
    except R22.OutOfTime as e:
        W.append(f"HẾT GIỜ trước {e}; Add out{P}h rồi chạy lại (tự tiếp nối)")
    S["warnings"] = list(dict.fromkeys(W)); save()
    log(f"XONG {P}h. File: {sorted(os.listdir(OUT))}\nCảnh báo: " + str(S["warnings"] or "không có"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/baselines20_env.py
"""baselines20_env.py — chạy TRONG venv riêng (/kaggle/working/venv_b, --system-site-packages), KHÔNG chung
numpy/torch với kernel chính (vòng 20, Phần B: Symphony, scArches). Gọi bằng tiến trình con từ run20.py:
    venv_b/bin/python baselines20_env.py <symphony|scarches> <input.npz> <output.npz>
Trao đổi dữ liệu qua file .npz: input = Xx (Xenium, count thô gene chung), Xs (source, count thô cùng gene, cùng cột);
output = Fx, Fs (embedding hai phía trong CÙNG một không gian reference) + version (phiên bản gói đã dùng).
Mỗi hàm tự bắt lỗi của chính nó; run20.py coi return code != 0 hoặc thiếu output.npz là lỗi -> cảnh báo, bỏ qua,
KHÔNG dừng notebook (baseline lỗi không phải USHER lỗi).
"""
import sys
import numpy as np


def _ver(dist, mod):
    """FIX 20f: symphonypy không có __version__ -> đọc từ metadata gói."""
    v = getattr(mod, "__version__", None)
    if v: return str(v)
    try:
        from importlib.metadata import version
        return version(dist)
    except Exception:
        return "?"


def run_symphony(inp, outp):
    """FIX 20e (Claude chat): theo README symphonypy — reference MỘT batch thì bỏ Harmony (bản trước gọi harmony_integrate
    trên 1 batch -> lỗi ở cả 4 cặp); PCA zero_center=False; reference cắt về HVG; map_embedding ở sy.tl.
    Thử lần lượt: (1) không Harmony, (2) có Harmony; in lỗi từng cách ra stderr."""
    import sys as _sys, traceback
    import scanpy as sc
    import anndata as ad
    import symphonypy as sy
    d = np.load(inp)
    Xx, Xs = d["Xx"].astype(np.float32), d["Xs"].astype(np.float32)
    names = [f"g{i}" for i in range(Xx.shape[1])]
    errs = []
    for use_harmony in (False, True):
        try:
            ref = ad.AnnData(X=Xx.copy()); ref.var_names = names; ref.obs["batch"] = "xenium"
            query = ad.AnnData(X=Xs.copy()); query.var_names = names; query.obs["batch"] = "source"
            sc.pp.normalize_total(ref, target_sum=1e4); sc.pp.log1p(ref)
            sc.pp.highly_variable_genes(ref, n_top_genes=min(2000, ref.shape[1]))
            ref.raw = ref
            ref = ref[:, ref.var["highly_variable"]].copy()
            sc.pp.scale(ref, max_value=10)
            sc.pp.pca(ref, n_comps=min(50, ref.shape[1] - 1), zero_center=False)
            if use_harmony: sy.pp.harmony_integrate(ref, key="batch")
            sc.pp.normalize_total(query, target_sum=1e4); sc.pp.log1p(query)
            map_fn = getattr(sy.tl, "map_embedding", None) or getattr(sy.pp, "map_embedding")
            map_fn(query, ref, key="batch")
            Fx = (ref.obsm["X_pca_harmony"] if "X_pca_harmony" in ref.obsm else ref.obsm["X_pca"]).astype(np.float32)
            Fs_key = next(k for k in ("X_pca_harmony", "X_pca_reference", "X_pca") if k in query.obsm)
            Fs = query.obsm[Fs_key].astype(np.float32)
            print(f"symphony OK (harmony={use_harmony}): query key {Fs_key}, Fx {Fx.shape}, Fs {Fs.shape}")
            np.savez(outp, Fx=Fx, Fs=Fs, version=f"{_ver('symphonypy', sy)}|harmony={use_harmony}")
            return
        except Exception:
            errs.append(f"[harmony={use_harmony}] " + traceback.format_exc()[-1500:])
    print("\n".join(errs), file=_sys.stderr); raise RuntimeError("symphony lỗi ở mọi cách dựng reference")


def run_symphony_h(inp, outp):
    """symphony_h (vòng 21, mục 5): reference Xenium chia 2 batch GIẢ ngẫu nhiên (seed 0) để harmony_integrate có
    việc thật để sửa (bản `symphony` 1 batch thường bỏ qua Harmony — cảnh báo RESEARCH_CONTEXT 5.17), rồi
    map_embedding source vào không gian đã sửa đó."""
    import sys as _sys
    import scanpy as sc
    import anndata as ad
    import symphonypy as sy
    d = np.load(inp)
    Xx, Xs = d["Xx"].astype(np.float32), d["Xs"].astype(np.float32)
    names = [f"g{i}" for i in range(Xx.shape[1])]
    pseudo = np.random.RandomState(0).randint(0, 2, size=Xx.shape[0])
    ref = ad.AnnData(X=Xx.copy()); ref.var_names = names
    ref.obs["pseudo_batch"] = [f"xenium_{b}" for b in pseudo]
    query = ad.AnnData(X=Xs.copy()); query.var_names = names
    query.obs["pseudo_batch"] = "source"
    sc.pp.normalize_total(ref, target_sum=1e4); sc.pp.log1p(ref)
    sc.pp.highly_variable_genes(ref, n_top_genes=min(2000, ref.shape[1]))
    ref.raw = ref
    ref = ref[:, ref.var["highly_variable"]].copy()
    sc.pp.scale(ref, max_value=10)
    sc.pp.pca(ref, n_comps=min(50, ref.shape[1] - 1), zero_center=False)
    sy.pp.harmony_integrate(ref, key="pseudo_batch")
    sc.pp.normalize_total(query, target_sum=1e4); sc.pp.log1p(query)
    map_fn = getattr(sy.tl, "map_embedding", None) or getattr(sy.pp, "map_embedding")
    map_fn(query, ref, key="pseudo_batch")
    Fx = ref.obsm["X_pca_harmony"].astype(np.float32)
    Fs_key = next(k for k in ("X_pca_harmony", "X_pca_reference", "X_pca") if k in query.obsm)
    Fs = query.obsm[Fs_key].astype(np.float32)
    print(f"symphony_h: khoá obsm thực dùng cho query = {Fs_key!r}, Fx {Fx.shape}, Fs {Fs.shape}", file=_sys.stderr)
    np.savez(outp, Fx=Fx, Fs=Fs, version=f"{_ver('symphonypy', sy)}|harmony=2pseudo_batch")


def run_scarches(inp, outp):
    import anndata as ad
    import scvi
    d = np.load(inp)
    Xx, Xs = d["Xx"].astype(np.float32), d["Xs"].astype(np.float32)
    ref = ad.AnnData(X=Xx.copy())
    scvi.model.SCVI.setup_anndata(ref)
    vae = scvi.model.SCVI(ref, n_latent=30)
    vae.train(max_epochs=200)
    Fx = vae.get_latent_representation().astype(np.float32)
    query = ad.AnnData(X=Xs.copy())
    query.obs_names = [f"q{i}" for i in range(query.n_obs)]
    vae_q = scvi.model.SCVI.load_query_data(query, vae)
    vae_q.train(max_epochs=200)
    Fs = vae_q.get_latent_representation().astype(np.float32)
    np.savez(outp, Fx=Fx, Fs=Fs, version=_ver("scvi-tools", scvi))


if __name__ == "__main__":
    method, inp, outp = sys.argv[1], sys.argv[2], sys.argv[3]
    {"symphony": run_symphony, "symphony_h": run_symphony_h, "scarches": run_scarches}[method](inp, outp)

In [ ]:
%%writefile /kaggle/working/run30.py
"""30 — Nội suy biểu hiện gene (CPU, KHÔNG chạy lại USHER, chỉ dùng map đã lưu).

BỐI CẢNH (quan trọng, đọc trước khi diễn giải kết quả):
`nội suy` KHÔNG phải tác vụ mới. Nó đã có từ vòng 06 (`run06.imp_setup` / `imp_map`) và được chấm ở các vòng
08-13, rồi BỊ BỎ khỏi `run11.metrics_pack` từ vòng ~16 trở đi. Kết luận cũ (5.9, 5.13): **USHER giảm nội suy ở
MỌI cặp**, thấp hơn cả `center` (scGPT 0.037 so với center 0.126; Stereo/scGPT tụt còn 0.003).

VÌ SAO CHẤM LẠI BÂY GIỜ — giả thuyết cơ học có thể sai:
Chẩn đoán 27f cho thấy map USHER thô bị CO (var_ratio 0.29-0.63) và lệch trọng tâm (mu_gap 0.15-0.29). Một đám
mây bị co về tâm sinh hiệu ứng "hub": nhiều tế bào source cùng rơi vào một nhúm hàng xóm Xenium -> dự đoán biểu
hiện gần như giống nhau cho mọi tế bào -> tương quan theo gene sụp. Chuyển nhãn thô (8 nhóm) chịu được chuyện
này, nội suy (vài nghìn gene liên tục) thì không. Nếu giả thuyết đúng thì `usher_cc` (đã khớp mô-men) PHẢI phục
hồi phần lớn khoảng cách so với `center`/CORAL. Nếu `usher_cc` vẫn thua, giả thuyết SAI và phải kết luận thẳng:
ánh xạ tuyến tính của M-step làm hỏng cấu trúc lân cận mịn, không chỉ mô-men.

Đây là một dự đoán CÓ THỂ THẤT BẠI. Không được viết lại giả thuyết sau khi thấy số.

Thước đo (nguyên bản vòng 06, không sửa để còn so được với số cũ):
  - Gene: giao hai panel, giữ gene phát hiện ở >= 5% truy vấn VÀ >= 5% tham chiếu (`MIN_DET`).
  - Dự đoán: 15 hàng xóm cosine trong không gian Xenium SAU ánh xạ, lấy trung bình biểu hiện log-norm của chúng.
  - Điểm: trung vị Spearman theo từng GENE qua các tế bào + CI bootstrap gene (1000 lần), kèm bản chỉ gene marker.
  - Hai mốc SÀN bắt buộc (mới, 30): `floor_rand` = hàng xóm ngẫu nhiên; `floor_mean` = trung bình toàn cục của
    tham chiếu. Không có hai mốc này thì không biết 0.12 là tốt hay chỉ là mức nền.

CẢNH BÁO diễn giải: hiệu suất bắt gene khác nhau giữa công nghệ -> chỉ đọc THỨ HẠNG giữa các phương pháp trên
cùng một cặp, không đọc giá trị tuyệt đối, và không so chéo giữa các cặp.

Env: TISSUE, COMBOS_ALL, RUN_PREFIX, FINAL_FLOOR, PATCH_SEEDS, EXTRA_MAPS, IMP_METHODS, OUT_DIR_30.
"""
import os, sys, json, glob, gc
os.environ.setdefault("MODE26", "report")
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
import run26d as R26
import run26g as R26G
np, pd, log, M = R26.np, R26.pd, R26.log, R26.M
R6, R16, R22, R23, R23C = R26.R22.R6, R26.R16, R26.R22, R26.R23, R26.R23C
P = R26.PREFIX
# ĐẢO CHIỀU (IMP_REV30=1): truy vấn = tế bào XENIUM, sự thật = biểu hiện Xenium, tham chiếu = NGUỒN ĐÃ ÁNH XẠ.
# Vì sao: giao thức xuôi (run06.imp_setup: Yq=Ys[q], Yr=Yx[ref]) đặt truy vấn ở phía NGUỒN, nên gene bị rò rỉ
# (embedding nguồn tính từ mọi gene, kể cả gene đang dự đoán — run06.py:512, lệch đo được 0.017-0.036) nằm
# trong chính embedding mà map dịch chuyển. Thông lượng rò rỉ do đó PHỤ THUỘC PHƯƠNG PHÁP, nên không thể so
# usher_cc với cmix/cniche một cách sạch sẽ. Đảo chiều đưa rò rỉ sang embedding Xenium — đóng băng và giống
# hệt ở mọi phương pháp — chỉ còn "tế bào nguồn nào rơi vào lân cận mỗi tế bào Xenium" thay đổi, đúng đại
# lượng cần đo. KHÔNG so giá trị tuyệt đối với bảng xuôi: đây là tác vụ khác, chỉ đọc thứ hạng nội bộ.
IMP_REV = os.environ.get("IMP_REV30", "0") == "1"
OUT = os.environ.get("OUT_DIR_30", f"/kaggle/working/out{P}i" + ("v" if IMP_REV else ""))
for mod in (R26.R17, R26.R18, R26.R19, R26.R20, R22, R22.R21, R23, R23C, R26G): mod.OUT = OUT
COMBOS, FLOOR = R26.COMBOS, R26.FLOOR
SEEDS = [0] if R26.SMOKE else [int(x) for x in os.environ.get("PATCH_SEEDS", "0,1,2").split(",")]
EXTRA_MAPS = [t for t in os.environ.get("EXTRA_MAPS", "").replace(" ", "").split(",") if t]
ALIAS = R26G.ALIAS
# Thứ tự cố định để bảng luôn đọc được. floor_* là MỐC SÀN, không phải phương pháp.
DEFAULT_METHODS = ("floor_mean,floor_rand,raw,center,coral,mnn_ref,harmony,scanorama,"
                   "usher_final,usher_c,usher_cc,usher_proc,class_ridge_pl_cc")
METHODS = [m for m in (os.environ.get("IMP_METHODS") or DEFAULT_METHODS).replace(" ", "").split(",") if m]
SEED_METHODS = ("usher_final", "usher_c", "usher_cc")   # chấm 3 seed; còn lại 1 lần


def imp_of(Fmap, Fx_ref, IMP):
    return R6.imp_map(np.asarray(Fmap, np.float32), Fx_ref, IMP)


_IMP_REV_CACHE = {}


def imp_rev_setup(pair, cx):
    """IMP đảo chiều, cache theo CẶP (shared_lognorm đắt). Trả (IMP, ref_src).

    Truy vấn Xenium lấy từ dx["H"] (giữ lại, không nằm trong tập khớp) đúng cỡ R6.N_Q như chiều xuôi;
    tham chiếu là ds["H"] — tế bào nguồn giữ lại — để cả hai phía đều ngoài tập khớp, đối xứng với chiều xuôi.
    """
    if pair in _IMP_REV_CACHE: return _IMP_REV_CACHE[pair]
    R17 = R26.R17; B, C = R17.B, R17.C
    pctx = cx["pctx"]; ds, dx = pctx["ds"], pctx["dx"]
    Ys, Yx, nG = C.shared_lognorm(pctx["Xs_lite"], pctx["vars_"], pctx["Xx_lite"], pctx["varx"])
    genes = sorted(set(pctx["vars_"]["symbol"].str.upper()) & set(pctx["varx"]["symbol"].str.upper()))
    assert len(genes) == nG == Ys.shape[1], f"{pair}: gene chung lệch"
    q_x, ref_s = B.pick(dx["H"], R6.N_Q), ds["H"]
    # imp_setup(arg1, arg2, genes, q, ref, H): Yq = arg1[q] (sự thật), Yr = arg2[ref] (tham chiếu).
    # Đảo chiều = hoán vị hai đối số đầu.
    IMP = R6.imp_setup(Yx, Ys, genes, q_x, ref_s, dx["H"])
    del Ys, Yx; gc.collect()
    _IMP_REV_CACHE[pair] = (IMP, ref_s)
    return IMP, ref_s


def imp_of_rev(Fmap, Fx, IMP, ref_s):
    """Hàng xóm của tế bào Xenium TRONG đám mây nguồn đã ánh xạ (ngược với run06.imp_map)."""
    nb = R6.knn_idx(Fx[IMP["q"]], Fmap[ref_s])
    return R6.imp_score(R6.avg_rows(nb, IMP["Yr"]), IMP)


def floor_rand(IMP, n_ref, seed=0):
    """Mốc sàn: 15 hàng xóm NGẪU NHIÊN trong tham chiếu. Mọi thông tin căn chỉnh bị xoá, chỉ còn cấu trúc biên
    của ma trận biểu hiện (gene nào hay dương tính). Phương pháp nào không vượt mốc này là vô giá trị."""
    rs = np.random.RandomState(seed)
    nb = rs.randint(0, n_ref, (len(IMP["q"]), R6.K_IMP))
    return R6.imp_score(R6.avg_rows(nb, IMP["Yr"]), IMP)


def floor_mean(IMP, n_ref):
    """Mốc sàn tệ nhất: MỌI tế bào nhận cùng một dự đoán = trung bình tham chiếu. Spearman theo gene của mốc này
    gần 0 theo cấu tạo (vector hằng) -> dùng để xác nhận thước đo không bị thổi phồng bởi hiệu ứng biên."""
    nb = np.tile(np.arange(n_ref)[None, :min(n_ref, 2000)], (len(IMP["q"]), 1))
    return R6.imp_score(R6.avg_rows(nb, IMP["Yr"]), IMP)


def build_maps(c, cx, W):
    """Trả dict {tên: hàm -> Fmap}. Hàm (lười) để không dựng map không cần tới."""
    Fs, Fx, T = cx["Fs"], cx["Fx"], cx["ds"]["T"]

    def usher(sd):
        m_ = R22.get_map(c, f"f{FLOOR}", sd, None, ALIAS.get(FLOOR))[0]
        return None if m_ is None else np.asarray(m_, np.float32)

    def proc():
        r = R22.get_map(c, f"f{FLOOR}", SEEDS[0], None, ALIAS.get(FLOOR))
        if r[0] is None or r[1] is None: return None
        return R26G.procrustes_map(Fs, Fx, T, r[1], r[2])[0]

    G = {"raw": lambda: Fs, "center": lambda: M.base_center(Fs, Fx, T),
         "coral": lambda: R23C.coral(Fs, Fx, T),
         "mnn_ref": lambda: R16.mnn_ref_correction(Fs, Fx, T)[0],
         "harmony": lambda: R16.run_harmony(Fx, Fs)[1],
         "scanorama": lambda: R26G.scanorama_at(Fx, Fs, alpha=0.0, knn=30)[1],
         "usher_proc": proc,
         "class_ridge_pl_cc": lambda: R23C.coral(np.asarray(
             M.base_class_ridge(Fs, Fx, T, cx["lab_def"], cx["dx"]["lab"])[0], np.float32), Fx, T)}
    for sd in SEEDS:
        G[f"usher_final@{sd}"] = lambda sd=sd: usher(sd)
        G[f"usher_c@{sd}"] = lambda sd=sd: (lambda u: None if u is None else R26G.variants(Fs, Fx, T, u)["usher_c"])(usher(sd))
        G[f"usher_cc@{sd}"] = lambda sd=sd: (lambda u: None if u is None else R26G.variants(Fs, Fx, T, u)["usher_cc"])(usher(sd))
    for t in EXTRA_MAPS:   # map của 28/31 (CORAL-init, sàn 0, C_niche, C_mix...) — chấm MỌI seed có map
        for sd in SEEDS:
            def mk(t=t, sd=sd):
                m_ = R22.get_map(c, t, sd, None, "default" if t == "f0" else None)[0]
                return None if m_ is None else np.asarray(m_, np.float32)
            G[f"u_{t}@{sd}"] = mk
            G[f"u_{t}_cc@{sd}"] = lambda t=t, mk=mk: (lambda u: None if u is None else R23C.coral(u, Fx, T))(mk())
    return G


def wanted(c):
    """Tên các mục cần chấm cho combo này, theo METHODS (nở theo seed với SEED_METHODS)."""
    out = []
    for m in METHODS:
        if m in SEED_METHODS: out += [f"{m}@{sd}" for sd in SEEDS]
        else: out.append(m)
    return out + [f"u_{t}{x}@{sd}" for t in EXTRA_MAPS for x in ("", "_cc") for sd in SEEDS]


def main():
    os.makedirs(OUT, exist_ok=True); W = []
    if os.environ.get("SKIP_INPUT_CHECK", "0") != "1": R26.check_inputs()
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    if not S:   # tiếp nối từ Input (KHÔNG chấm lại thứ đã chấm)
        for fj in sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}/summary.json", recursive=True)):
            S = json.load(open(fj)); log(f"tiếp nối <- {fj}"); break
    S.setdefault("imp", {})
    S["config"] = (f"{P}i nội suy gene mô={R26.TISSUE} chiều={'ĐẢO (Xenium<-nguồn)' if IMP_REV else 'xuôi (nguồn<-Xenium)'} "
                   f"combos={COMBOS} sàn={FLOOR} seeds={SEEDS} "
                   f"extra={EXTRA_MAPS} k={R6.K_IMP} min_det={R6.MIN_DET}")
    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
    # /kaggle/working trước: khi 30 chạy CHUNG phiên với 31, map cniche/cmix nằm ở working, chưa phải Input
    R22.scan_maps(P, R22.PRIOR)   # danh sách thư mục map: run22._SUFFIX (sửa gốc 2026-10-07)
    log(f"map sẵn có: {len(R22.PRIOR)} file | sẽ chấm: {wanted(COMBOS[0])}")
    D, F, PCTX = R26.R18.setup_pairs(sorted({c.split("/")[0] for c in COMBOS}))
    p0 = sorted({c.split("/")[0] for c in COMBOS})[0]
    km8 = R26.R20.KMeans(n_clusters=R26.R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(
        R26.R17.get_niche_label_comp(p0, S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    for c in COMBOS:
        try:
            Sc = S["imp"].setdefault(c, {})
            need = [m for m in wanted(c) if m not in Sc]
            if not need: log(f"[{c}] đã xong ({len(Sc)} mục)"); R26.free_combo(c); continue
            cx = R22.ctx(c)
            if IMP_REV:
                IMP, ref = imp_rev_setup(c.split("/")[0], cx); Fx_ref = None   # ref = tế bào NGUỒN
            else:
                IMP, ref = cx["pctx"]["IMP"], cx["pctx"]["ref"]; Fx_ref = cx["Fx"][ref]
            S.setdefault("imp_meta", {})[c] = {"n_gene": int(len(IMP["genes"])), "n_marker": int(IMP["marker"].sum()),
                                               "n_query": int(len(IMP["q"])), "n_ref": int(len(ref))}
            G = build_maps(c, cx, W)
            for nm in need:
                try:
                    if nm == "floor_mean": Sc[nm] = floor_mean(IMP, len(ref))
                    elif nm == "floor_rand": Sc[nm] = floor_rand(IMP, len(ref))
                    else:
                        f_ = G.get(nm)
                        if f_ is None: W.append(f"{c}: không biết mục {nm}"); continue
                        R22.step(f"[{c}] nội suy {nm}")
                        Fm = f_()
                        if Fm is None:
                            (log if nm.endswith(("@1", "@2")) else W.append)(f"{c}: thiếu map cho {nm}"); continue
                        Sc[nm] = (imp_of_rev(Fm, cx["Fx"], IMP, ref) if IMP_REV
                                  else imp_of(Fm, Fx_ref, IMP))
                    log(f"[{c}] {nm}: sp={Sc[nm]['spearman_med']} CI={Sc[nm]['spearman_ci']}")
                    save(); gc.collect()
                except Exception as e:
                    W.append(f"{c}/{nm}: lỗi {e!r}"); log(f"[{c}] {nm} LỖI {e!r}")
        except Exception as e:
            W.append(f"{c}: lỗi {e!r}"); log(f"[{c}] LỖI {e!r}")
        R26.free_combo(c)
    S["warnings"] = list(dict.fromkeys(W)); save()
    log(f"XONG {P}i.\nCảnh báo: " + str(S["warnings"] or "không có"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/run45.py
"""45 — Chỉ số ST không cần chú giải (Moran's I trên nhãn, PAS) + chẩn đoán Moran's I của PHẦN DƯ.

KHÔNG FGW, KHÔNG huấn luyện: chỉ nạp map đã lưu rồi tính. CPU là đủ.

Phần D — chẩn đoán, quyết định FROST 1.0.2 sống hay chết:
    Bản LIÊN TỤC của run20 Phần D. Vòng 20 đo η²(niche) của phần dư (cấu trúc theo LỚP) và được
    0.001-0.005. Nhưng phần dư có thể có tự tương quan KHÔNG GIAN mà không có cấu trúc theo lớp —
    hai thứ khác nhau. Ở đây: fit Θ_glob trên T1 (đúng như run20:237), lấy phần dư trên T2, rồi đo
    Moran's I của từng chiều phần dư theo toạ độ vật lý.
      Moran's I ≈ 0  -> số hạng Laplacian không có gì để cắn; đóng hướng 1.0.2 bằng số liệu.
      Moran's I > 0  -> còn tín hiệu không gian chưa được sửa; số hạng mục tiêu có cơ sở.
    Mốc so: hoán vị toạ độ (giữ nguyên phần dư) -> Moran's I của nhiễu thuần.

Phần M — hai chỉ số ST mới, cho MỌI phương pháp trong bảng chính:
    Nhãn chuyển sang = bỏ phiếu 15-NN nhãn tham chiếu trong không gian đã căn chỉnh (đúng
    run22._knn_bacc), rồi so TỔ CHỨC KHÔNG GIAN của nhãn chuyển với nhãn THẬT:
      - Moran's I trên nhãn: từng loại tế bào một, chỉ báo nhị phân, trọng số từ toạ độ.
      - PAS: tỉ lệ tế bào có nhãn khác đa số láng giềng KHÔNG GIAN của nó.
    CẢ HAI báo cáo dưới dạng |Δ so với nhãn thật|, KHÔNG phải giá trị thô: gán tất cả cùng một nhãn
    cho PAS = 0 và Moran's I suy biến — thấp-hơn-là-tốt-hơn sẽ thưởng cho nhãn vô dụng.

HAI BẪY ĐÃ XỬ LÝ:
  1. Q nằm trên tile giữ lại của bàn cờ 500µm -> kNN không gian nối được hai tile cách nhau qua khe.
     Cắt mọi cạnh dài hơn R_MAX_UM (mặc định 100µm, bằng bán kính niche lớn nhất vòng 19). Log tỉ lệ
     cạnh bị cắt; nếu > 30% thì cảnh báo vì đồ thị đã rời rạc quá.
  2. Moran's I chỉ có nghĩa với nhãn CÓ tổ chức không gian. Lọc theo I của nhãn THẬT >= MORAN_MIN
     (mặc định 0.10), đúng cách bài sketching (NAR 2026) làm. Số nhãn còn lại được ghi vào summary.

ENV: TISSUE, RUN_PREFIX, COMBOS_ALL, OUT_DIR_45, ONLY_PART (D,M,A,B,T,R,K,S,V,W,X,Y), R_MAX_UM, MORAN_MIN, K_SPATIAL,
     ALPHAS, ALPHA_TAG, PART_B_ARMS,
     SPC_TAG, SPC_TS, SPC_FULL, PATCH_N, PATCH_MIN, N_PATCH, PK, GW_N,
     PATCH_SEEDS, EXTRA_MAPS, IMP_METHODS (dùng lại danh sách của run30).
"""
import os, sys, json, glob, time, gc, warnings
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist
from sklearn.metrics import balanced_accuracy_score

import run20 as R20
import run22 as R22
import run30 as R30
R17, R18, R19 = R20.R17, R20.R18, R20.R19
R23, R23C, R26G = R30.R23, R30.R23C, R30.R26G
log, lite_obs, _unit, mode_rows = R20.log, R20.lite_obs, R20._unit, R20.mode_rows
ridge_fit = R20.ridge_fit

TISSUE = os.environ.get("TISSUE", "coad").lower()
P = os.environ.get("RUN_PREFIX", {"coad": "23", "ov": "26", "hcc": "27"}[TISSUE])
OUT = os.environ.get("OUT_DIR_45", f"/kaggle/working/out{P}s")
COMBOS = [c for c in os.environ.get("COMBOS_ALL", "").split(",") if c.strip()]
PARTS = {p.strip().upper() for p in os.environ.get("ONLY_PART", "D,M,A").split(",") if p.strip()}
SEEDS = R30.SEEDS            # dùng CHUNG với run30 để phần D và danh sách map của phần M không lệch seed
R_MAX_UM = float(os.environ.get("R_MAX_UM", 100.0))
MORAN_MIN = float(os.environ.get("MORAN_MIN", 0.10))
K_SPATIAL = int(os.environ.get("K_SPATIAL", 15))
ALPHAS = [float(a) for a in os.environ.get("ALPHAS", "0.05,0.1,0.2,0.3,0.4,0.5").split(",") if a.strip()]
VER = "45.24-tieuchi"   # TEM PHIÊN BẢN: in ra dòng log ĐẦU TIÊN. Hai lần notebook cũ trên Kaggle đã
                    # ngốn 4 giờ CPU mà không ai biết. Thấy tem khác -> notebook trên Kaggle là bản cũ.
ALPHA_TAG = os.environ.get("ALPHA_TAG", "cmix")    # map nền của phần A: cmix = FROST CHƯA CORAL
WN_TOP = [int(x) for x in os.environ.get("WN_TOP", "10,25,50").split(",") if x.strip()]
AGGS = [a.strip() for a in os.environ.get("AGGS", "tb,trungvi").split(",") if a.strip()]
SK_KS = [int(k) for k in os.environ.get("SK_KS", "5,10,15,30").split(",") if k.strip()]
KC = [float(c) for c in os.environ.get("KC", "0.25,0.5,1,2,4").split(",") if c.strip()]  # sigma^2 = c * trung vị s^2
ROT_TAG = os.environ.get("ROT_TAG", "cmix_levns")       # map NỀN của phần R (chưa CORAL)
ROT_TS = [float(t) for t in os.environ.get("ROT_TS", "0,0.25,0.5,0.75,1").split(",") if t.strip()]
ROT_FULL = [float(t) for t in os.environ.get("ROT_FULL", "0,0.5,1").split(",") if t.strip()]
SPC_TAG = os.environ.get("SPC_TAG", "cmix")             # map NỀN của phần X (CHƯA CORAL) -> t=0 = FROST
SPC_TS = [float(t) for t in os.environ.get("SPC_TS", "0,0.25,0.5,0.75,1").split(",") if t.strip()]
SPC_FULL = [float(t) for t in os.environ.get("SPC_FULL", "0,0.5,1").split(",") if t.strip()]
PATCH_N = int(os.environ.get("PATCH_N", 1500))       # số tế bào mục tiêu mỗi bản vá (~cỡ batch USHER)
PATCH_MIN = int(os.environ.get("PATCH_MIN", 300))    # ô dưới ngưỡng này bị bỏ
N_PATCH = int(os.environ.get("N_PATCH", 8))          # số bản vá lấy mỗi bên
PK = int(os.environ.get("PK", 15))                   # k cho tính khiết lân cận
GW_N = int(os.environ.get("GW_N", 800))              # cỡ bản vá khi giải GW thật
N_T_MAX = int(os.environ.get("N_T_MAX", 20000))   # cỡ tập giữ lại của phần T
PART_B_ARMS = [a.strip() for a in os.environ.get(
    "PART_B_ARMS", "pre@0,pre@0.2,pre@0.3,pre@0.4,pre@0.5").split(",") if a.strip()]
FLOOR = R30.FLOOR
T0 = time.time()
TIME_BUDGET_S = float(os.environ.get("TIME_BUDGET_S", 11 * 3600))


class OutOfTime(Exception): pass


def step(msg):
    log(f"    [{time.time() - T0:6.0f}s] {msg}")
    if time.time() - T0 > TIME_BUDGET_S: raise OutOfTime(msg)


# =============================================================================== đồ thị không gian
def spatial_graph(xy, k=None, r_max=None):
    """kNN không gian ĐÃ CẮT cạnh dài. Trả (W hàng-chuẩn-hoá, nn_idx có mặt nạ, tỉ lệ cạnh bị cắt).

    Cắt cạnh là BẮT BUỘC: Q lấy từ tile giữ lại của bàn cờ nên hai tế bào ở hai tile khác nhau có thể
    thành "láng giềng" dù cách nhau qua một khe 500µm. Không cắt thì mọi chỉ số không gian đều sai.
    """
    k = k or K_SPATIAL
    r_max = R_MAX_UM if r_max is None else r_max
    n = len(xy)
    k_eff = min(k, n - 1)
    d, idx = NearestNeighbors(n_neighbors=k_eff + 1, n_jobs=4).fit(xy).kneighbors(xy)
    d, idx = d[:, 1:], idx[:, 1:]           # bỏ chính nó
    ok = d <= r_max
    cut = float(1.0 - ok.mean())
    rows = np.repeat(np.arange(n), k_eff)[ok.ravel()]
    cols = idx.ravel()[ok.ravel()]
    W = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n))
    W = W.maximum(W.T)                       # đối xứng hoá
    deg = np.asarray(W.sum(1)).ravel()
    inv = np.divide(1.0, deg, out=np.zeros_like(deg), where=deg > 0)
    Wn = sp.diags(inv.astype(np.float32)) @ W   # hàng chuẩn hoá; hàng cô lập = toàn 0
    return Wn.tocsr(), idx, ok, cut, deg


def moran_i(Wn, y):
    """Moran's I với W ĐÃ chuẩn hoá hàng. y: (n,) hoặc (n, p). Trả vô hướng hoặc (p,)."""
    y = np.asarray(y, np.float64)
    one_d = y.ndim == 1
    Y = y[:, None] if one_d else y
    Z = Y - Y.mean(0, keepdims=True)
    num = np.einsum("ij,ij->j", Z, Wn @ Z)
    den = np.einsum("ij,ij->j", Z, Z)
    out = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
    return float(out[0]) if one_d else out


def pas(nn_idx, ok, lab):
    """PAS — tỉ lệ tế bào có nhãn KHÁC đa số láng giềng không gian của nó.

    Chỉ tính trên tế bào còn ít nhất một láng giềng sau khi cắt cạnh; tế bào cô lập bị loại và số
    lượng được trả về để ghi log.
    """
    lab = np.asarray(lab)
    codes, _ = pd.factorize(lab)             # nhãn THIẾU -> -1
    nb = np.where(ok, codes[nn_idx], -1)
    # HAI SỬA (phát hiện khi dựng phần S, chạy trên TOÀN BỘ tế bào thay vì chỉ tập high_quality):
    #  1. Tế bào KHÔNG có nhãn phải bị loại khỏi mẫu số, trước đây codes = -1 luôn khác đa số nên
    #     bị tính là "bất thường" -> pas bị thổi lên.
    #  2. Tế bào mà MỌI láng giềng đều không có nhãn làm np.bincount([]) .argmax() ném ValueError.
    #     Phần M không dính vì Q lấy từ ds["H"] (high_quality); phần S chạy trên tất cả thì dính.
    keep = ok.any(1) & (codes >= 0)
    maj = np.full(len(lab), -1)
    n_trong = 0
    for i in np.where(keep)[0]:
        v = nb[i][nb[i] >= 0]
        if len(v) == 0: keep[i] = False; n_trong += 1; continue
        maj[i] = np.bincount(v).argmax()
    n_bo = int(len(lab) - keep.sum())
    if keep.sum() == 0: return None, n_bo
    return float((maj[keep] != codes[keep]).mean()), n_bo


# =============================================================================== Phần M
def transferred_labels(cx, Fmap, q):
    """Nhãn chuyển sang: bỏ phiếu 15-NN nhãn tham chiếu — ĐÚNG run22._knn_bacc, chỉ trả nhãn thay vì bacc."""
    ref, Fx = cx["pctx"]["ref"], cx["Fx"]
    mu = Fx.mean(0)
    nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Fx[ref] - mu)).kneighbors(
        _unit(np.asarray(Fmap, np.float32)[q] - mu), return_distance=False)
    return mode_rows(np.asarray(cx["dx"]["lab"])[ref][nn])


def st_metrics(Wn, nn_idx, ok, lab_pred, lab_true, labels_keep):
    """Moran's I trên nhãn + PAS, báo cáo dưới dạng ĐỘ LỆCH so với nhãn thật."""
    oh_p = np.stack([(lab_pred == c).astype(np.float64) for c in labels_keep], 1)
    oh_t = np.stack([(lab_true == c).astype(np.float64) for c in labels_keep], 1)
    I_p, I_t = moran_i(Wn, oh_p), moran_i(Wn, oh_t)
    pas_p, n_iso = pas(nn_idx, ok, lab_pred)
    pas_t, _ = pas(nn_idx, ok, lab_true)
    return {"moran_abs_dev": round(float(np.abs(I_p - I_t).mean()), 4),
            "moran_pred_mean": round(float(I_p.mean()), 4),
            "moran_true_mean": round(float(I_t.mean()), 4),
            "pas_pred": None if pas_p is None else round(pas_p, 4),
            "pas_true": None if pas_t is None else round(pas_t, 4),
            "pas_abs_dev": None if (pas_p is None or pas_t is None) else round(abs(pas_p - pas_t), 4),
            "n_label": int(len(labels_keep)), "n_isolated": n_iso}


def part_m(combo, cx, S, save, W):
    Sc = S.setdefault("M", {}).setdefault(combo, {})
    pair = cx["pair"]
    q = cx["Q"]
    xy = lite_obs(pair)[["x", "y"]].to_numpy()[q]
    Wn, nn_idx, ok, cut, deg = spatial_graph(xy)
    lab_true = np.asarray(cx["ds"]["lab"])[q]

    # lọc nhãn theo tự tương quan NỀN (trên nhãn thật) — bài sketching NAR 2026 làm đúng bước này
    cnt = pd.Series(lab_true).value_counts()
    cand = [c for c in cnt.index if cnt[c] >= 20]
    I_true_all = moran_i(Wn, np.stack([(lab_true == c).astype(np.float64) for c in cand], 1)) if cand else np.array([])
    labels_keep = [c for c, i_ in zip(cand, I_true_all) if i_ >= MORAN_MIN]
    S.setdefault("M_meta", {})[combo] = {"canh_bi_cat": round(cut, 4), "bac_tb": round(float(deg.mean()), 2),
                                         "n_nhan_xet": len(cand), "n_nhan_giu": len(labels_keep),
                                         "nhan_giu": list(map(str, labels_keep)),
                                         "r_max_um": R_MAX_UM, "moran_min": MORAN_MIN}
    save()
    log(f"[{combo}] đồ thị: cắt {cut:.1%} cạnh, bậc TB {deg.mean():.1f} | nhãn giữ {len(labels_keep)}/{len(cand)}")
    if cut > 0.30:
        W.append(f"{combo}: cắt {cut:.0%} cạnh không gian (> 30%) — đồ thị rời rạc, đọc chỉ số ST thận trọng")
    if not labels_keep:
        W.append(f"{combo}: KHÔNG nhãn nào đạt Moran's I >= {MORAN_MIN} trên nhãn thật -> bỏ phần M")
        return

    G = R30.build_maps(combo, cx, W)
    thieu = []
    for name in R30.wanted(combo):
        if name in Sc: continue
        fn = G.get(name)
        if fn is None: thieu.append(name + "(không có trong build_maps)"); continue
        step(f"[{combo}] M: {name}")
        try:
            Fmap = fn()
        except Exception as e:
            W.append(f"{combo}/{name}: dựng map lỗi {e!r}"); continue
        if Fmap is None:
            thieu.append(name); continue        # KHÔNG im lặng: map không có trên đĩa -> ghi ra
        pred = transferred_labels(cx, Fmap, q)
        Sc[name] = st_metrics(Wn, nn_idx, ok, pred, lab_true, labels_keep)
        save()
        del Fmap; gc.collect()
    if thieu:
        S.setdefault("M_thieu", {})[combo] = thieu
        W.append(f"{combo}: KHÔNG tìm thấy map cho {len(thieu)} mục -> {', '.join(thieu)}")
        log(f"[{combo}] thiếu map: {', '.join(thieu)}")
    save()


# =============================================================================== Phần D
def part_d(combo, cx, S, save, W):
    """Moran's I của phần dư — bản liên tục của run20 Phần D."""
    Sc = S.setdefault("D", {}).setdefault(combo, {})
    pair, Fs = cx["pair"], cx["Fs"]
    ds = cx["ds"]
    T1, T2 = cx["T1"], cx["T2"]
    xy_all = lite_obs(pair)[["x", "y"]].to_numpy()
    for sd in SEEDS:
        key = f"s{sd}"
        if key in Sc: continue
        _, ybar, has = R22.get_map(combo, f"f{FLOOR}", sd, None, R30.ALIAS.get(FLOOR))
        if ybar is None:
            W.append(f"{combo} s{sd}: không có ȳ (ybar npz) -> bỏ phần D"); continue
        step(f"[{combo}] D: seed {sd}")
        Xhas = np.asarray(ds["T"])[has]
        m1h, m2h = np.isin(Xhas, T1), np.isin(Xhas, T2)
        if m1h.sum() < 50 or m2h.sum() < 50:
            W.append(f"{combo} s{sd}: T1/T2 quá nhỏ ({m1h.sum()}/{m2h.sum()}) -> bỏ"); continue
        Xt = np.hstack([Fs[Xhas], np.ones((len(Xhas), 1), np.float32)]).astype(np.float64)
        Yt = np.asarray(ybar)[has].astype(np.float64)
        Theta = ridge_fit(Xt[m1h], Yt[m1h], 1e-3)
        e2 = Yt[m2h] - Xt[m2h] @ Theta                      # phần dư trên nửa GIỮ LẠI
        xy2 = xy_all[Xhas[m2h]]
        Wn, _, _, cut, deg = spatial_graph(xy2)
        I = moran_i(Wn, e2)
        rng = np.random.RandomState(0)
        I_perm = np.array([moran_i(Wn, e2[rng.permutation(len(e2))]) for _ in range(5)])
        Sc[key] = {"moran_tb": round(float(I.mean()), 5), "moran_p90": round(float(np.percentile(I, 90)), 5),
                   "moran_max": round(float(I.max()), 5),
                   "moran_hoanvi_tb": round(float(I_perm.mean()), 5),
                   "moran_hoanvi_sd": round(float(I_perm.std()), 5),
                   "n_T2": int(m2h.sum()), "d": int(e2.shape[1]),
                   "canh_bi_cat": round(cut, 4), "bac_tb": round(float(deg.mean()), 2)}
        save()
        log(f"[{combo}] s{sd} (D): {Sc[key]}")


# =============================================================================== Phần A
def part_a(combo, cx, S, save, W):
    """Sàng α — làm trơn không gian HẬU KỲ trên map FROST. CHẶN TRÊN của 1.0.2, không phải 1.0.2.

    Nghiệm của  min_Z ‖Z−Z₀‖² + α·tr(ZᵀLZ)  là  (I+αL)⁻¹Z₀ ≈ (1−α)Z₀ + α·W_sp·Z₀ — đúng toán tử
    gần kề của số hạng phạt Laplacian định thêm vào M-step. KHÁC: ở đây làm trơn theo TỪNG tế bào,
    còn M-step phải thể hiện qua MỘT ma trận W tuyến tính duy nhất, tự do ít hơn hẳn. Nên:
      đường α hỏng -> 1.0.2 CHẮC CHẮN hỏng, dừng.  đường α đẹp -> 1.0.2 CÓ THỂ được, mới đáng viết.

    Ba nhánh (α=0 của `pre` chính là FROST 1.0; α=0 của `nocoral` chính là u_cmix):
      nocoral smooth(F) · pre coral(smooth(F)) KHỚP Laplacian ở M-step · post smooth(coral(F))

    d_cov = ‖Σ(Z[T])−Σ_x‖_F/‖Σ_x‖_F. CẢNH BÁO ĐỌC: ở nhánh `pre` nó ≈ 0 với MỌI α một cách tầm
    thường, vì CORAL chạy sau cùng nên ép lại khớp mô-men. Đừng đọc đó là "an toàn" — nó chỉ có
    nghĩa cho `nocoral` và `post`. Với `pre`, rủi ro trộn lô phải chấm bằng R23.evaluate (chưa làm
    ở đây): Σ khớp KHÔNG bảo đảm iLISI khớp, vì iLISI phụ thuộc trộn lô CỤC BỘ.
    """
    Sc = S.setdefault("A", {}).setdefault(combo, {})
    Fx, T, q = cx["Fx"], cx["ds"]["T"], cx["Q"]
    sd = SEEDS[0]
    F0 = R22.get_map(combo, ALPHA_TAG, sd, None)[0]
    if F0 is None:                                   # KHÔNG im lặng
        W.append(f"{combo}: không có map {ALPHA_TAG} s{sd} -> bỏ phần A"); return
    F0 = np.asarray(F0, np.float32)

    xy = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    Wn_all, _, _, cut_all, _ = spatial_graph(xy)          # đồ thị TOÀN BỘ tế bào -> để làm trơn
    Wn_q, nn_q, ok_q, cut_q, _ = spatial_graph(xy[q])     # đồ thị trên Q -> để chấm (y hệt part_m)
    iso = np.asarray(Wn_all.sum(1)).ravel() <= 0          # hàng không láng giềng: Wn@Z = 0 -> phải giữ nguyên

    lab_true = np.asarray(cx["ds"]["lab"])[q]
    cnt = pd.Series(lab_true).value_counts()
    cand = [c for c in cnt.index if cnt[c] >= 20]
    I_t = moran_i(Wn_q, np.stack([(lab_true == c).astype(np.float64) for c in cand], 1)) if cand else np.array([])
    keep = [c for c, i_ in zip(cand, I_t) if i_ >= MORAN_MIN]
    if not keep:
        W.append(f"{combo}: không nhãn nào đạt Moran >= {MORAN_MIN} -> bỏ phần A"); return
    S.setdefault("A_meta", {})[combo] = {"n_nhan_giu": len(keep), "nhan_giu": list(map(str, keep)),
                                         "cat_toan_bo": round(cut_all, 4), "cat_Q": round(cut_q, 4),
                                         "n_co_lap": int(iso.sum()), "map_nen": f"{ALPHA_TAG}@s{sd}"}
    save()
    Cx = np.cov(Fx.astype(np.float64), rowvar=False); nCx = float(np.linalg.norm(Cx))

    def smooth(Z, a):
        Zs = (1.0 - a) * Z + a * np.asarray(Wn_all @ Z)
        Zs[iso] = Z[iso]                                  # giữ nguyên tế bào cô lập
        return Zs.astype(np.float32)

    def score(Z):
        pred = transferred_labels(cx, Z, q)
        r = st_metrics(Wn_q, nn_q, ok_q, pred, lab_true, keep)
        r["bacc_xfer"] = round(float(balanced_accuracy_score(lab_true, pred)), 4)
        r["d_cov"] = round(float(np.linalg.norm(np.cov(Z[T].astype(np.float64), rowvar=False) - Cx) / nCx), 4)
        return r

    jobs = [("nocoral", 0.0, lambda: F0), ("pre", 0.0, lambda: R23C.coral(F0, Fx, T))]
    for a in ALPHAS:
        jobs += [("nocoral", a, (lambda a=a: smooth(F0, a))),
                 ("pre",     a, (lambda a=a: R23C.coral(smooth(F0, a), Fx, T))),
                 ("post",    a, (lambda a=a: smooth(R23C.coral(F0, Fx, T), a)))]
    for arm, a, mk in jobs:
        k = f"{arm}@{a:g}"
        if k in Sc: continue
        step(f"[{combo}] A: {k}")
        try:
            Sc[k] = score(np.asarray(mk(), np.float32))
        except Exception as e:
            W.append(f"{combo}/A/{k}: lỗi {e!r}"); log(f"[{combo}] A {k} LỖI {e!r}"); continue
        v = Sc[k]
        log(f"[{combo}] A {k}: pas_dev={v['pas_abs_dev']} moran_dev={v['moran_abs_dev']} "
            f"bacc={v['bacc_xfer']} d_cov={v['d_cov']}")
        save(); gc.collect()
    del Wn_all, Wn_q; gc.collect()


# =============================================================================== Phần B
def part_b(combo, cx, S, save, W):
    """Chấm ĐẦY ĐỦ bằng R23.evaluate ở vài điểm α — cổng CUỐI trước khi viết M-step của 1.0.2.

    Phần A chỉ có `bacc_xfer` (proxy rẻ) và `d_cov`. Thứ CHƯA biết là `batch_v2`/iLISI: `d_cov ≈ 0`
    là điều kiện CẦN chứ không đủ, vì iLISI phụ thuộc trộn lô CỤC BỘ, không chỉ mô-men toàn cục.

    `pre@0` ở đây CHÍNH LÀ FROST 1.0 và là mốc so DUY NHẤT đúng — đừng so với `u_cmix_cc` ở bảng
    của vòng 23e: cùng một map nhưng khác đường tính (seed, Fs/Fx nạp lại), lệch vài phần nghìn.
    """
    if R20.KM16 is None:   # nổ ở giây 0 thay vì sau ~6 phút eval_all16d rồi mới chết trong etrack
        W.append("phần B: R20.KM16 chưa đặt -> etrack sẽ nổ; BỎ phần B")
        log("phần B: THIẾU R20.KM16 -> bỏ"); return
    Sc = S.setdefault("B", {}).setdefault(combo, {})
    Fs, Fx, T, q = cx["Fs"], cx["Fx"], cx["ds"]["T"], cx["Q"]
    xy = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    Wn_all, _, _, _, _ = spatial_graph(xy)
    iso = np.asarray(Wn_all.sum(1)).ravel() <= 0

    # Chỉ số ST tính NGAY TẠI ĐÂY, trên CÙNG map mà R23.evaluate chấm. Trước đây chúng chỉ có ở
    # phần A, đi một đường tính khác, nên không ghép được với total_v2 để kiểm tiêu chí
    # "cải thiện ST mà giữ total". Đó chính là lý do 1.0.2 bị kết luận nhầm là chết.
    Wn_q, nn_q, ok_q, _, _ = spatial_graph(xy[q])
    lab_true = np.asarray(cx["ds"]["lab"])[q]
    cnt = pd.Series(lab_true).value_counts()
    cand = [c for c in cnt.index if cnt[c] >= 20]
    I_t = moran_i(Wn_q, np.stack([(lab_true == c).astype(np.float64) for c in cand], 1)) if cand else np.array([])
    keep = [c for c, i_ in zip(cand, I_t) if i_ >= MORAN_MIN]
    if not keep:
        W.append(f"{combo}: không nhãn nào đạt Moran >= {MORAN_MIN} -> phần B không có chỉ số ST")

    def smooth(Z, a):
        if not a: return Z
        Zs = (1.0 - a) * Z + a * np.asarray(Wn_all @ Z)
        Zs[iso] = Z[iso]                             # giữ nguyên tế bào cô lập
        return Zs.astype(np.float32)

    # Di cư khoá PHẲNG cũ (chỉ 1 seed, trước khi phần B biết tới seed) sang khoá có seed, để
    # không phải chạy lại ~6 phút/nhánh cho những gì đã có trong out<P>s.
    for k0 in [k for k in Sc if "@" in k and "/" not in k]:
        Sc[f"s{SEEDS[0]}/{k0}"] = Sc.pop(k0)

    for sd in SEEDS:
        F0 = R22.get_map(combo, ALPHA_TAG, sd, None)[0]
        if F0 is None:                               # thiếu MỘT seed thì bỏ seed đó, không bỏ cả cặp
            W.append(f"{combo}: không có map {ALPHA_TAG} s{sd} -> bỏ seed này ở phần B"); continue
        F0 = np.asarray(F0, np.float32)
        for k in PART_B_ARMS:
            ten = f"s{sd}/{k}"
            cu = Sc.get(ten, {})
            if "pas_abs_dev" in cu: continue
            if "@" not in k:
                W.append(f"phần B: tên nhánh sai {k!r} (cần dạng pre@0.2)"); continue
            arm, a_ = k.split("@", 1)
            try: a = float(a_)
            except ValueError: W.append(f"phần B: α sai ở {k!r}"); continue
            if arm not in ("nocoral", "pre", "post"):
                W.append(f"phần B: nhánh lạ {arm!r} ở {k!r}"); continue
            step(f"[{combo}] B: {ten}")
            Fm = np.asarray(smooth(F0, a) if arm == "nocoral" else
                            R23C.coral(smooth(F0, a), Fx, T) if arm == "pre" else
                            smooth(R23C.coral(F0, Fx, T), a), np.float32)
            v = dict(cu)
            if "total_v2" not in v:                  # phần đắt; bản di cư đã có thì bỏ qua
                try:
                    v.update(R23.evaluate(cx, f"{combo}/B/{ten}", Fs, Fm, Fx, W))
                except Exception as e:
                    W.append(f"{combo}/B/{ten}: lỗi {e!r}"); log(f"[{combo}] B {ten} LỖI {e!r}")
                    del Fm; gc.collect(); continue
            if keep:
                try:
                    v.update(st_metrics(Wn_q, nn_q, ok_q,
                                        transferred_labels(cx, Fm, q), lab_true, keep))
                except Exception as e:
                    W.append(f"{combo}/B/{ten}: chỉ số ST lỗi {e!r}")
            Sc[ten] = v
            log(f"[{combo}] B {ten}: xfer_main={v.get('xfer_main')} total_v2={v.get('total_v2')} "
                f"batch_v2={v.get('batch_v2')} pas_dev={v.get('pas_abs_dev')} "
                f"moran_dev={v.get('moran_abs_dev')}")
            save(); del Fm; gc.collect()
    del Wn_all, Wn_q; gc.collect()


def import_cache(W):
    """Chép cache niche/labcomp từ BẤT KỲ out<P>* nào trong Input vào OUT.

    `R17.get_niche_label_comp` và `get_pair_niche` chỉ đọc cache từ OUT, không tự quét Input. OUT
    của 45 (`out<P>s`) khởi đầu RỖNG mỗi lần chạy -> dựng lại từ h5ad đầy đủ, ~25 phút trước khi
    tới cặp đầu tiên. Thấy rõ ở mọi log trước (2350s, 1176s...). Chép sang là xong.
    """
    import shutil
    n = 0
    for d in sorted(glob.glob(f"/kaggle/input/**/out{P}*", recursive=True)):
        # labcomp*  bắt CẢ labcomp_<pair>.npy (niche 8 nhóm) LẪN labcomp16_<pair>.npy (niche mịn 16,
        # run20.py:532) — cái sau là thứ KM16 của phần B/T cần, glob "labcomp_*" KHÔNG khớp nó.
        for pat in ("niche_*.npy", "labcomp*.npy"):
            for f in glob.glob(os.path.join(d, pat)):
                dst = os.path.join(OUT, os.path.basename(f))
                if not os.path.exists(dst):
                    shutil.copy(f, dst); n += 1
    log(f"import_cache: chép {n} file cache vào {OUT}")
    if n == 0:
        W.append("import_cache: KHÔNG thấy cache niche/labcomp nào trong Input -> sẽ dựng lại từ đầu (~25 phút)")
    return n


# =============================================================================== Phần R
def coral_xoay(F, Fx, T, L, t):
    """CORAL với bậc tự do QUAY được dùng để làm trơn không gian.

    CORAL chuẩn: Z = (F-mu_s)·Sigma_s^-1/2·Sigma_x^1/2 + mu_x. Nghiệm đó KHÔNG duy nhất: với MỌI
    R trực giao, A_R = Sigma_s^-1/2·R·Sigma_x^1/2 cho Cov(Z) = Sigma_x^1/2·RᵀR·Sigma_x^1/2 = Sigma_x,
    y HỆT nhau. Chọn R = I là tuỳ tiện — đó là một bậc tự do chuẩn đang bị bỏ phí. Nhưng
    Sigma_s^-1/2·R·Sigma_x^1/2 KHÔNG trực giao nên R có đổi hình học kNN, tức đổi nhãn chuyển.

    Dùng nó để tối thiểu độ gồ ghề không gian. Với B = (F-mu_s)Sigma_s^-1/2 và M = BᵀLB:
        tr(ZᵀLZ) = tr(Rᵀ M R Sigma_x)
    Bất đẳng thức vết Ruhe: cực tiểu trên nhóm trực giao đạt khi ghép trị riêng NHỎ nhất của M với
    trị riêng LỚN nhất của Sigma_x, tức R* = U_M[:, ::-1] @ U_Xᵀ (cả hai từ eigh, tăng dần).
    Đổi dấu cột KHÔNG đổi mục tiêu (S·Lambda·S = Lambda với S chéo ±1) nên ép det=+1 được miễn phí.

    t nội suy: R(t) = cực-phân-rã((1-t)I + t R*). t=0 cho ĐÚNG CORAL hiện tại, t=1 cho trơn nhất.
    MỌI t khớp Sigma_x NHƯ NHAU -> phần mô-men bậc hai của batch_v2 bất biến theo cấu tạo. (Sai số
    còn lại ~RIDGE=0.01 là do Ws dựng từ Sigma_s đã ridge; nó GIỐNG nhau ở mọi t nên không ảnh hưởng.)

    Dùng lại RIDGE và _mat_pow của run23c để t=0 tái lập CORAL hiện hành đến từng chữ số.
    """
    F64, Fx64 = np.asarray(F, np.float64), np.asarray(Fx, np.float64)
    mu_s, mu_x = F64[T].mean(0), Fx64.mean(0)
    Cs, Cx = np.cov(F64[T], rowvar=False), np.cov(Fx64, rowvar=False)
    d = Cs.shape[0]
    Cs = Cs + R23C.RIDGE * np.trace(Cs) / d * np.eye(d)
    Cx = Cx + R23C.RIDGE * np.trace(Cx) / d * np.eye(d)
    Ws, Wx = R23C._mat_pow(Cs, -0.5), R23C._mat_pow(Cx, 0.5)
    if t == 0:
        return ((F64 - mu_s) @ (Ws @ Wx) + mu_x).astype(np.float32), None
    B = (F64 - mu_s) @ Ws
    M = B.T @ (L @ B)
    M = 0.5 * (M + M.T)                       # đối xứng hoá chống trôi số
    _, UM = np.linalg.eigh(M); _, UX = np.linalg.eigh(Cx)
    Rs = UM[:, ::-1] @ UX.T
    if np.linalg.det(Rs) < 0:                 # ép về SO(d) cho nội suy mượt; mục tiêu không đổi
        UM2 = UM[:, ::-1].copy(); UM2[:, -1] *= -1; Rs = UM2 @ UX.T
    U_, _, Vt_ = np.linalg.svd((1.0 - t) * np.eye(d) + t * Rs)
    Rt = U_ @ Vt_                             # cực phân rã -> trực giao gần nhất
    return ((F64 - mu_s) @ (Ws @ Rt @ Wx) + mu_x).astype(np.float32), Rs


def part_r(combo, cx, S, save, W):
    """FROST 1.0.3 — CORAL quay theo không gian. Hậu kỳ trên map đã có, CPU, KHÔNG dựng lại map.

    Khuyết tật ST (nhãn chuyển vụn trong không gian, PAS vượt sự thật 4/4) do CHÍNH CORAL gây ra:
    nó thổi các hướng phương sai thấp, lân cận kNN nhiễu hơn, nhãn lật giữa hai tế bào kề nhau.
    Phần A đã thử làm trơn đầu ra -> được PAS nhưng BÓP phương sai. Ở đây Sigma bị ghim cứng, chỉ
    còn quay -> ràng buộc chặt hơn hẳn.
    """
    Sc = S.setdefault("R", {}).setdefault(combo, {})
    Fx, T, q = cx["Fx"], cx["ds"]["T"], cx["Q"]
    sd = SEEDS[0]
    F0 = R22.get_map(combo, ROT_TAG, sd, None)[0]
    if F0 is None:
        W.append(f"{combo}: không có map {ROT_TAG} s{sd} -> bỏ phần R"); return
    F0 = np.asarray(F0, np.float32)

    xy = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    Wn_all, _, _, cut_all, deg_all = spatial_graph(xy)
    # L = D - A = D(I - Wn). Đối xứng vì A đã được đối xứng hoá ở spatial_graph.
    L = sp.diags(deg_all.astype(np.float64)) @ (sp.eye(len(xy), format="csr") - Wn_all.astype(np.float64))
    asym = abs((L - L.T)).max()
    if asym > 1e-6: W.append(f"{combo}: Laplacian lệch đối xứng {asym:.2e} -> nghiệm Ruhe không còn đúng")

    Wn_q, nn_q, ok_q, cut_q, _ = spatial_graph(xy[q])
    lab_true = np.asarray(cx["ds"]["lab"])[q]
    cnt = pd.Series(lab_true).value_counts()
    cand = [c for c in cnt.index if cnt[c] >= 20]
    I_t = moran_i(Wn_q, np.stack([(lab_true == c).astype(np.float64) for c in cand], 1)) if cand else np.array([])
    keep = [c for c, i_ in zip(cand, I_t) if i_ >= MORAN_MIN]
    if not keep:
        W.append(f"{combo}: không nhãn nào đạt Moran >= {MORAN_MIN} -> bỏ phần R"); return
    S.setdefault("R_meta", {})[combo] = {"map_nen": f"{ROT_TAG}@s{sd}", "n_nhan_giu": len(keep),
                                         "cat_toan_bo": round(cut_all, 4), "cat_Q": round(cut_q, 4)}
    Cx0 = np.cov(np.asarray(Fx, np.float64), rowvar=False); nCx = float(np.linalg.norm(Cx0))

    for t in ROT_TS:
        k = f"t{t:g}"
        if k in Sc and ("xfer_main" in Sc[k] or t not in ROT_FULL): continue
        step(f"[{combo}] R: {k}")
        try:
            Z, _ = coral_xoay(F0, Fx, T, L, t)
        except Exception as e:
            W.append(f"{combo}/R/{k}: dựng map lỗi {e!r}"); log(f"[{combo}] R {k} LỖI {e!r}"); continue
        pred = transferred_labels(cx, Z, q)
        r = st_metrics(Wn_q, nn_q, ok_q, pred, lab_true, keep)
        # d_cov ~0.01 (mức RIDGE) và PHẢI không đổi theo t; tăng theo t nghĩa là code sai.
        r["d_cov"] = round(float(np.linalg.norm(
            np.cov(Z[T].astype(np.float64), rowvar=False) - Cx0) / nCx), 4)
        r["rough"] = round(float(np.einsum("ij,ij->", Z.astype(np.float64), L @ Z.astype(np.float64))
                                 / (len(Z) * Z.shape[1])), 6)
        if t in ROT_FULL:
            if R20.KM16 is None:
                W.append(f"{combo}/R/{k}: R20.KM16 chưa đặt -> bỏ chấm đầy đủ")
            else:
                ev = R23.evaluate(cx, f"{combo}/R/{k}", cx["Fs"], Z, Fx, W)
                r.update({kk: ev.get(kk) for kk in
                          ("xfer_main", "xfer_all", "total_v2", "batch_v2", "bio_v2")})
        Sc[k] = r; save()
        log(f"[{combo}] R {k}: pas={r['pas_abs_dev']} moran={r['moran_abs_dev']} d_cov={r['d_cov']} "
            f"rough={r['rough']} total={r.get('total_v2')} xfer={r.get('xfer_main')}")
        del Z; gc.collect()
    del Wn_all, Wn_q, L; gc.collect()


# =============================================================================== Phần W
def part_w(combo, cx, S, save, W):
    """Tự tương quan không gian làm bộ lọc CHIỀU — không phải bộ lọc tế bào.

    CHỖ MỌI THỨ TRƯỚC ĐÂY SAI: tín hiệu không gian đúng 62-80% (pas_true 0.20-0.38). Dùng nó ở cấp
    TỪNG TẾ BÀO thì vô dụng — mỗi lân cận là mẫu cỡ 15 với 1/3 nhiễm bẩn. Nhưng cùng tín hiệu đó ở
    cấp QUẦN THỂ trên 45.000 tế bào có sai số chuẩn ~1/sqrt(n) ~ 0.005. Yếu ở cá thể, rất chắc ở
    tổng thể. Sáu ý tưởng trước (cmix, beta, Laplacian, CORAL quay, co rút, trung vị) đều là MỘT ý
    tưởng duy nhất — gộp láng giềng vào tế bào — dùng ở cấp sai.

    Ở đây không gian KHÔNG nói "tế bào này giống hàng xóm". Nó nói "CHIỀU này mới là sinh học":
    nhiễu kỹ thuật (độ sâu, hiệu ứng lô, tràn tín hiệu giữa bin) KHÔNG có tổ chức không gian; cấu
    trúc mô thì có. Moran's I của từng chiều nhúng, tính trên toàn bộ tế bào, tách được hai thứ.

    BA PHÉP ĐO, và phép thứ hai là cổng:
      1. Phân bố Moran theo chiều, trên nguồn và trên Xenium, kèm sàn hoán vị. Nếu mọi chiều đều
         ~0 thì không có gì để lọc, ý tưởng chết tại chỗ.
      2. TƯƠNG QUAN SPEARMAN giữa Moran_nguồn(d) và Moran_đích(d). Hai bên dùng CÙNG cơ sở (joint
         PCA hoặc cùng không gian scGPT) nên chiều so được với nhau. Tương quan CAO -> "chiều có
         cấu trúc không gian" là tính chất CHUNG, độc lập công nghệ -> đánh trọng số theo nó có
         nghĩa. Tương quan ~0 -> cấu trúc không gian là đặc thù từng công nghệ, trọng số sẽ tuỳ tiện.
      3. Chọn chiều theo Moran có hơn chọn theo PHƯƠNG SAI không: dùng top-N chiều theo mỗi tiêu
         chí rồi chấm bằng chính probe của Phần K/V. Đây là phép thử trực tiếp "không gian có chọn
         ra sinh học không", chứ không phải suy luận gián tiếp.
    """
    Sc = S.setdefault("W", {}).setdefault(combo, {})
    if Sc: return
    pctx, ds = cx["pctx"], cx["ds"]
    Fs, Fx = np.asarray(cx["Fs"], np.float64), np.asarray(cx["Fx"], np.float64)
    g = pctx["groups_ev"]; lab_s = np.asarray(pctx["lab_src_ev"])
    T_, Q_ = np.asarray(ds["T"]), np.asarray(cx["Q"])
    rng = np.random.RandomState(0)

    def moran_cot(Z, xy):
        Wn, _, _, cut, _ = spatial_graph(xy)
        I = moran_i(Wn, Z)
        Ip = np.stack([moran_i(Wn, Z[rng.permutation(len(Z))]) for _ in range(3)])
        return I, float(Ip.std()), cut

    xy_s = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    xy_x = lite_obs(R18.TARGET)[["x", "y"]].to_numpy()
    # Lệch hàng giữa ma trận đặc trưng và bảng toạ độ sẽ cho Moran SAI MÀ KHÔNG NỔ -> chặn ở đây.
    if len(xy_s) != len(Fs) or len(xy_x) != len(Fx):
        W.append(f"{combo}: lệch hàng toạ độ/đặc trưng (nguồn {len(xy_s)} vs {len(Fs)}, "
                 f"đích {len(xy_x)} vs {len(Fx)}) -> bỏ phần W")
        log(f"[{combo}] W: LỆCH HÀNG, bỏ"); return
    if Fs.shape[1] != Fx.shape[1]:
        W.append(f"{combo}: nguồn và đích khác số chiều ({Fs.shape[1]} vs {Fx.shape[1]}) -> "
                 "không so được Moran theo chiều, bỏ phần W")
        return
    step(f"[{combo}] W: Moran theo chiều (nguồn)")
    I_s, sd_s, cut_s = moran_cot(Fs, xy_s)
    step(f"[{combo}] W: Moran theo chiều (Xenium)")
    I_x, sd_x, cut_x = moran_cot(Fx, xy_x)

    from scipy.stats import spearmanr
    var_s = Fs.var(0)
    qs = lambda v: {f"p{p}": round(float(np.percentile(v, p)), 4) for p in (0, 10, 50, 90, 100)}
    Sc["moran_nguon"] = qs(I_s); Sc["moran_dich"] = qs(I_x)
    Sc["sd_hoanvi"] = {"nguon": round(sd_s, 5), "dich": round(sd_x, 5)}
    Sc["cat_canh"] = {"nguon": round(cut_s, 4), "dich": round(cut_x, 4)}
    Sc["d"] = int(Fs.shape[1])
    Sc["tl_chieu_co_tin_hieu"] = {"nguon": round(float((I_s > 3 * sd_s).mean()), 4),
                                  "dich": round(float((I_x > 3 * sd_x).mean()), 4)}
    Sc["spearman_nguon_dich"] = round(float(spearmanr(I_s, I_x).statistic), 4)   # <- CỔNG
    Sc["spearman_moran_var"] = round(float(spearmanr(I_s, var_s).statistic), 4)
    save()

    # 3) chọn chiều theo Moran so với theo phương sai
    def probe(cols):
        Z = np.asarray(Fs[:, cols], np.float32)
        mu = Z[T_].mean(0)
        nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(Z[T_] - mu)).kneighbors(
            _unit(Z[Q_] - mu), return_distance=False)
        return bacc_on(lab_s[Q_], mode_rows(lab_s[T_][nn]), g)

    d = Fs.shape[1]
    Sc["bacc_toanbo"] = probe(np.arange(d))
    for N in WN_TOP:
        if N >= d: continue
        step(f"[{combo}] W: top-{N} chiều")
        Sc[f"bacc_moran_top{N}"] = probe(np.argsort(-I_s)[:N])
        Sc[f"bacc_var_top{N}"] = probe(np.argsort(-var_s)[:N])
        save()
    log(f"[{combo}] W: spearman(nguồn,đích)={Sc['spearman_nguon_dich']} "
        f"spearman(moran,var)={Sc['spearman_moran_var']} "
        f"moran_nguồn p50={Sc['moran_nguon']['p50']} p90={Sc['moran_nguon']['p90']} "
        f"sàn={Sc['sd_hoanvi']['nguon']} | bacc toàn bộ={Sc['bacc_toanbo']}")
    save(); gc.collect()


# =============================================================================== Phần X
def part_x(combo, cx, S, save, W):
    """FROST 1.0.4 — Spatial-CORAL: khớp thêm mô-men bậc hai CÓ TRỌNG SỐ KHÔNG GIAN.

    LÝ DO CÓ TÍNH CHỨNG MINH, không phải quan sát. CORAL tính Cs, Cx từ ma trận tế_bào×chiều. Hoán
    vị thứ tự hàng KHÔNG đổi hiệp phương sai. Vậy CORAL BẤT BIẾN VỚI HOÁN VỊ TẾ BÀO, nên nó mù cấu
    trúc không gian về mặt CẤU TRÚC — không phải vì chưa tinh chỉnh đủ. Phần W đo được chỗ mù đó
    tốn gì: Moran trung vị theo chiều lệch 40-50% giữa nguồn và Xenium (vhd_ff 0.410 vs 0.285;
    stereoseq 0.438 vs 0.285; cosmx6k 0.033 vs 0.131 — ngược chiều). CORAL không chạm tới được.

    Mô-men bậc hai CÓ TRỌNG SỐ KHÔNG GIAN, trong hệ ĐÃ TẨY TRẮNG:
        A = Bᵀ W̃_s B / n_s    với B  = (F - mu_s) Cs^-1/2    (nguồn)
        G = Bxᵀ W̃_x Bx / n_x   với Bx = (X - mu_x) Cx^-1/2    (đích)
    W̃ = (Wn + Wnᵀ)/2: Wn chuẩn hoá theo hàng nên KHÔNG đối xứng, mà nghiệm Ruhe đòi đối xứng.

    Bậc tự do còn lại sau CORAL đúng là nhóm trực giao (lập luận ở coral_xoay):
    Z = (F-mu_s)Cs^-1/2 R Cx^1/2 giữ nguyên Cov(Z) = Cx với MỌI R. Trong hệ tẩy trắng mô-men không
    gian của Z là RᵀAR, nên
        min_R ‖RᵀAR - G‖_F   <=>   max_R tr(RᵀAR G)
    Ruhe: cực đại = Σ λ↑(A)λ↑(G), đạt tại R* = U_A U_Gᵀ (cả hai eigh, TĂNG dần). Khác phần R ở chỗ
    phần R ghép NGƯỢC (cực tiểu vết); ở đây ghép THUẬN.

    HAI CỔNG RẺ, chạy trước mọi phép chấm đắt:
      1. `d_sp_san` — chặn DƯỚI của ‖RᵀAR-G‖/‖G‖ trên TOÀN nhóm trực giao, tính thẳng từ hai phổ.
         Nếu nó ≈ `d_sp_t0` thì phép quay về nguyên tắc không sửa được gì -> chết, khỏi chấm.
      2. nhánh `hoanvi@t1` — R* dựng từ đồ thị nguồn ĐÃ HOÁN VỊ. Bài học phần K: đồ thị thật từng
         THUA đồ thị hoán vị 4/4. Nếu hoanvi ngang t=1 thì thứ mua được không phải thông tin không gian.

    `d_cov` phải ĐỨNG YÊN ở mức RIDGE với mọi t (bất biến theo cấu tạo); tăng theo t nghĩa là code sai.
    Với SPC_TAG="cmix", t=0 tái lập ĐÚNG FROST (u_cmix_cc) nên là mốc so duy nhất đúng của phần này.
    """
    Sc = S.setdefault("X", {}).setdefault(combo, {})
    Fx, T, q = cx["Fx"], cx["ds"]["T"], cx["Q"]
    sd = SEEDS[0]
    F0 = R22.get_map(combo, SPC_TAG, sd, None)[0]
    if F0 is None:
        W.append(f"{combo}: không có map {SPC_TAG} s{sd} -> bỏ phần X"); return
    F0 = np.asarray(F0, np.float64); Fx64 = np.asarray(Fx, np.float64)
    d = F0.shape[1]
    if d != Fx64.shape[1]:
        W.append(f"{combo}: nguồn {d} chiều, đích {Fx64.shape[1]} chiều -> bỏ phần X"); return
    xy_s = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    xy_x = lite_obs(R18.TARGET)[["x", "y"]].to_numpy()
    # Lệch hàng giữa toạ độ và đặc trưng cho mô-men không gian SAI MÀ KHÔNG NỔ -> chặn ở đây (bài học phần W).
    if len(xy_s) != len(F0) or len(xy_x) != len(Fx64):
        W.append(f"{combo}: lệch hàng toạ độ/đặc trưng (nguồn {len(xy_s)}/{len(F0)}, "
                 f"đích {len(xy_x)}/{len(Fx64)}) -> bỏ phần X")
        log(f"[{combo}] X: LỆCH HÀNG, bỏ"); return

    step(f"[{combo}] X: đồ thị + mô-men không gian")
    Wn_s, _, _, cut_s, _ = spatial_graph(xy_s)
    Wn_x, _, _, cut_x, _ = spatial_graph(xy_x)
    Ss = (0.5 * (Wn_s + Wn_s.T)).tocsr()
    Sx = (0.5 * (Wn_x + Wn_x.T)).tocsr()
    del Wn_x; gc.collect()

    mu_s, mu_x = F0[T].mean(0), Fx64.mean(0)
    Cs, Cx0 = np.cov(F0[T], rowvar=False), np.cov(Fx64, rowvar=False)
    nCx = float(np.linalg.norm(Cx0))
    Cs = Cs + R23C.RIDGE * np.trace(Cs) / d * np.eye(d)
    Cx = Cx0 + R23C.RIDGE * np.trace(Cx0) / d * np.eye(d)
    Wsq, Wxq = R23C._mat_pow(Cs, -0.5), R23C._mat_pow(Cx, 0.5)
    sym = lambda Mx: 0.5 * (Mx + Mx.T)
    Bx = (Fx64 - mu_x) @ R23C._mat_pow(Cx, -0.5)
    G = sym(Bx.T @ (Sx @ Bx) / len(Bx))
    del Bx, Sx, Fx64; gc.collect()
    B = (F0 - mu_s) @ Wsq            # giữ B, BỎ F0: Z = B·Rt·Wxq + mu_x, khỏi trừ mu_s lại mỗi vòng
    del F0; gc.collect()
    A = sym(B.T @ (Ss @ B) / len(B))
    perm = np.random.RandomState(0).permutation(len(B))
    Bp = B[perm].astype(np.float32)                       # float32: bản sao này chỉ để dựng nhánh null
    A_h = sym((Bp.T @ (Ss.astype(np.float32) @ Bp)).astype(np.float64) / len(Bp))
    del Bp, Ss; gc.collect()

    nG = float(np.linalg.norm(G))
    lA, lG = np.linalg.eigvalsh(A), np.linalg.eigvalsh(G)
    _, UG = np.linalg.eigh(G)
    san = float(np.sqrt(max(0.0, float(lA @ lA) + float(lG @ lG) - 2.0 * float(lA @ lG)))) / nG
    S.setdefault("X_meta", {})[combo] = {
        "map_nen": f"{SPC_TAG}@s{sd}", "d": int(d),
        "cat_canh": {"nguon": round(cut_s, 4), "dich": round(cut_x, 4)},
        "d_sp_t0": round(float(np.linalg.norm(A - G)) / nG, 4),   # lệch mô-men không gian CỦA FROST
        "d_sp_san": round(san, 4),                                # CỔNG 1: chặn dưới trên MỌI phép quay
        "moran_tb_nguon_trang": round(float(np.trace(A) / d), 4),
        "moran_tb_dich_trang": round(float(np.trace(G) / d), 4)}
    mt = S["X_meta"][combo]
    log(f"[{combo}] X: d_sp t0={mt['d_sp_t0']} sàn={mt['d_sp_san']} | moran(tẩy trắng) "
        f"nguồn={mt['moran_tb_nguon_trang']} đích={mt['moran_tb_dich_trang']}")
    save()

    def R_sao(Am):
        """R* = U_A U_Gᵀ, ghép THUẬN λ↑ với λ↑ -> cực đại tr(RᵀAR G). det<0 thì lật một cột:
        đổi dấu cột không đổi mục tiêu (S·Λ·S = Λ) nên ép về SO(d) cho nội suy cực mượt là miễn phí."""
        _, U = np.linalg.eigh(Am)
        Rs = U @ UG.T
        if np.linalg.det(Rs) < 0:
            U2 = U.copy(); U2[:, -1] *= -1; Rs = U2 @ UG.T
        return Rs

    Rs_that, Rs_hoanvi = R_sao(A), R_sao(A_h)
    del A_h; gc.collect()

    Wn_q, nn_q, ok_q, cut_q, _ = spatial_graph(xy_s[q])
    lab_true = np.asarray(cx["ds"]["lab"])[q]
    cnt = pd.Series(lab_true).value_counts()
    cand = [c for c in cnt.index if cnt[c] >= 20]
    I_t = moran_i(Wn_q, np.stack([(lab_true == c).astype(np.float64) for c in cand], 1)) if cand else np.array([])
    keep = [c for c, i_ in zip(cand, I_t) if i_ >= MORAN_MIN]
    if not keep:
        W.append(f"{combo}: không nhãn nào đạt Moran >= {MORAN_MIN} -> bỏ chấm ST phần X"); return
    mt["n_nhan_giu"] = len(keep); mt["cat_Q"] = round(cut_q, 4)

    for k, t, Rs in [(f"t{t:g}", t, Rs_that) for t in SPC_TS] + [("hoanvi@t1", 1.0, Rs_hoanvi)]:
        if k in Sc and ("xfer_main" in Sc[k] or t not in SPC_FULL): continue
        step(f"[{combo}] X: {k}")
        U_, _, Vt_ = np.linalg.svd((1.0 - t) * np.eye(d) + t * Rs)
        Rt = U_ @ Vt_                                 # cực phân rã -> trực giao gần nhất
        Z = (B @ (Rt @ Wxq) + mu_x).astype(np.float32)
        pred = transferred_labels(cx, Z, q)
        r = st_metrics(Wn_q, nn_q, ok_q, pred, lab_true, keep)
        r["d_sp"] = round(float(np.linalg.norm(Rt.T @ A @ Rt - G)) / nG, 4)
        r["d_cov"] = round(float(np.linalg.norm(
            np.cov(Z[T].astype(np.float64), rowvar=False) - Cx0) / nCx), 4)
        if t in SPC_FULL:
            if R20.KM16 is None:
                W.append(f"{combo}/X/{k}: R20.KM16 chưa đặt -> bỏ chấm đầy đủ")
            else:
                ev = R23.evaluate(cx, f"{combo}/X/{k}", cx["Fs"], Z, Fx, W)
                r.update({kk: ev.get(kk) for kk in
                          ("xfer_main", "xfer_all", "total_v2", "batch_v2", "bio_v2")})
        Sc[k] = r; save()
        log(f"[{combo}] X {k}: d_sp={r['d_sp']} d_cov={r['d_cov']} pas={r['pas_abs_dev']} "
            f"moran={r['moran_abs_dev']} total={r.get('total_v2')} xfer={r.get('xfer_main')} "
            f"batch={r.get('batch_v2')}")
        del Z; gc.collect()
    del B, A, G, Wn_s, Wn_q; gc.collect()


# =============================================================================== Phần Y
def _ban_va(xy, n_muc, n_toi_thieu, n_lay):
    """Chia lát thành ô vuông cạnh L (µm) sao cho ô TRUNG VỊ có ~n_muc tế bào; trả các ô gần n_muc nhất.

    Không dùng diện tích hộp bao: mô có hình dạng bất kỳ nên hộp bao thổi diện tích lên và L bị chọn
    quá nhỏ. Dò L bằng 3 vòng theo trung vị số tế bào thực tế trong ô — bất biến với hình dạng lát.
    """
    rng = xy.max(0) - xy.min(0)
    L = float(np.sqrt(max(rng[0] * rng[1], 1.0) / max(len(xy) / n_muc, 1.0)))
    o = None
    for _ in range(3):
        g = np.floor((xy - xy.min(0)) / L).astype(np.int64)
        key = g[:, 0] * (g[:, 1].max() + 2) + g[:, 1]
        u, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
        ok = cnt >= n_toi_thieu
        if not ok.any(): break
        med = float(np.median(cnt[ok]))
        o = (inv, u, cnt, ok, L)
        if abs(med - n_muc) / n_muc < 0.15: break
        L *= float(np.sqrt(n_muc / max(med, 1.0)))
    if o is None: return [], float("nan")
    inv, u, cnt, ok, L = o
    thu_tu = np.where(ok)[0][np.argsort(np.abs(cnt[ok] - n_muc))][:n_lay]
    return [np.where(inv == j)[0] for j in thu_tu], L


def _tinh_khiet(D, lab, k):
    """Tỉ lệ cặp (tế bào, 1 trong k láng giềng gần nhất theo D) CÙNG nhãn. D là ma trận khoảng cách."""
    k = min(k, len(D) - 1)
    if k < 1: return None
    nb = np.argsort(D, axis=1)[:, 1:k + 1]
    codes, _ = pd.factorize(np.asarray(lab))
    m = codes >= 0
    if not m.any(): return None
    return round(float((codes[nb[m]] == codes[m][:, None]).mean()), 4)


def _hist_cx(D, nbin=20):
    """Lược đồ khoảng cách đôi một SAU chuẩn hoá D/D.max() — đúng phép chuẩn hoá USHER dùng cho C."""
    v = D[np.triu_indices_from(D, 1)]
    mx = v.max()
    if not np.isfinite(mx) or mx <= 0: return np.zeros(nbin)
    h, _ = np.histogram(v / mx, bins=nbin, range=(0, 1))
    return h / max(h.sum(), 1)


def _dong_y_ghep(Pi, lab_a, lab_b):
    """Tỉ lệ KHỐI LƯỢNG mà kế hoạch vận chuyển đặt vào cặp cùng nhãn, kèm mức ngẫu nhiên Σ p_a(c)p_b(c)."""
    la, lb = np.asarray(lab_a).astype(str), np.asarray(lab_b).astype(str)
    uniq = np.unique(np.concatenate([la, lb]))
    tot = float(Pi.sum())
    if tot <= 0: return None, None
    s = 0.0; ch = 0.0
    for c in uniq:
        ia, ib = la == c, lb == c
        if ia.any() and ib.any(): s += float(Pi[np.ix_(ia, ib)].sum())
        ch += (ia.mean()) * (ib.mean())
    return round(s / tot, 4), round(float(ch), 4)


def part_y(combo, cx, S, save, W):
    """CỔNG CPU cho C_sp (toạ độ làm ma trận cấu trúc GW kiểu PASTE) — chạy TRƯỚC khi sửa src_cuda.

    Vì sao phải là cổng chứ không dựng thẳng: đổi cách lấy batch + C_sp là sửa src_cuda rồi chạy lại
    USHER trên GPU, 4 cặp × 3 seed. Tám ý tưởng trước đã chết. Mọi thứ cần biết để đoán ý tưởng này
    sống hay chết đều đo được trên CPU trong vài phút, nên đo trước.

    Vì sao C_sp là khe DUY NHẤT còn hợp lệ cho toạ độ ở bộ dữ liệu này: vòng 17 Phần A đo đăng ký
    affine THẤT BẠI 0/4 (same_label chỉ nhỉnh ngẫu nhiên 2-6 điểm, top1/top2 ≈ 1, stereoseq lệch
    174°/2556µm). Nên mọi cách dùng toạ độ ở chi phí LIÊN-MIỀN (vòng 18 `spatial_b`) là nạp nhiễu.
    Số hạng cấu trúc GW chỉ cần khoảng cách TRONG một lát nên bất biến với xoay/tịnh tiến/lật —
    chính lý do PASTE chọn thiết kế đó, và chính lý do nó miễn nhiễm với việc đăng ký thất bại.

    USHER có sẵn `spatial_utils` cửa sổ không gian nhưng nó dùng CÙNG biên cửa sổ cho hai bên
    ("For affine-aligned data"), tức đòi đúng cái đăng ký bất khả -> không dùng được, phải tự lấy
    bản vá ĐỘC LẬP mỗi bên, cùng cỡ vật lý, vị trí tuỳ ý.

    BA CỔNG:
      1. `tk_kgian` so với `tk_bieuhien` TRÊN CẢ HAI BÊN. Việc duy nhất của ma trận cấu trúc là nói
         "hai tế bào này gần nhau về cấu trúc". Nếu biểu hiện nói chính xác hơn toạ độ thì trộn
         toạ độ vào C chỉ làm bẩn. Đây đúng là logic đã giết Phần W (phương sai thắng Moran).
      2. `chong_lap_sp`: lược đồ khoảng cách đôi một sau chuẩn hoá D/D.max(), nguồn so với đích.
         Thấp -> hai bên ở hai thang mật độ khác nhau, GW sẽ khớp bằng cách CO GIÃN mật độ chứ không
         phải khớp sinh học. Phần S đã đo d1 = 14.9-16.0µm khác nhau giữa các công nghệ.
      3. `gw_sp` so với `gw_kgian_ngau` (toạ độ đều cùng hộp bao) và `gw_ex`: giải GW thật trên một cặp bản vá
         rồi đo tỉ lệ khối lượng ghép vào cặp CÙNG nhãn. Đây là ước lượng TRỰC TIẾP thứ C_sp sẽ mua
         được bên trong USHER, không cần chạy USHER. (Cần POT; thiếu thì bỏ, hai cổng trên vẫn đủ
         để kết luận âm.)
    """
    Sc = S.setdefault("Y", {}).setdefault(combo, {})
    if Sc: return
    pctx = cx["pctx"]
    Fs, Fx = np.asarray(cx["Fs"], np.float32), np.asarray(cx["Fx"], np.float32)
    lab_s, lab_x = np.asarray(pctx["lab_src_ev"]), np.asarray(pctx["lab_tgt_ev"])
    xy_s = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    xy_x = lite_obs(R18.TARGET)[["x", "y"]].to_numpy()
    for ten, a, b, c in (("nguồn", xy_s, Fs, lab_s), ("đích", xy_x, Fx, lab_x)):
        if not (len(a) == len(b) == len(c)):
            W.append(f"{combo}: lệch hàng bên {ten} ({len(a)}/{len(b)}/{len(c)}) -> bỏ phần Y")
            log(f"[{combo}] Y: LỆCH HÀNG {ten}, bỏ"); return

    rng = np.random.RandomState(0)
    ket = {}
    for ten, xy, F, lab in (("nguon", xy_s, Fs, lab_s), ("dich", xy_x, Fx, lab_x)):
        step(f"[{combo}] Y: bản vá bên {ten}")
        bv, L = _ban_va(xy, PATCH_N, PATCH_MIN, N_PATCH)
        if not bv:
            W.append(f"{combo}: bên {ten} không có ô nào đủ {PATCH_MIN} tế bào -> bỏ phần Y"); return
        tk_sp, tk_ex, d1s, hs, hx, ns = [], [], [], [], [], []
        for idx in bv:
            if len(idx) > PATCH_N: idx = idx[rng.choice(len(idx), PATCH_N, replace=False)]
            ns.append(len(idx))
            Dsp = cdist(xy[idx], xy[idx])
            Zn = _unit(F[idx].astype(np.float32))
            Dex = 1.0 - Zn @ Zn.T
            np.fill_diagonal(Dex, 0.0)
            tk_sp.append(_tinh_khiet(Dsp, lab[idx], PK)); tk_ex.append(_tinh_khiet(Dex, lab[idx], PK))
            d = np.partition(Dsp + np.eye(len(idx)) * 1e9, 1, axis=1)[:, 1]
            d1s.append(float(np.median(d)))
            hs.append(_hist_cx(Dsp)); hx.append(_hist_cx(Dex))
        f = lambda v: (round(float(np.mean(v)), 4), round(float(np.std(v)), 4))
        ket[ten] = {"L_um": round(L, 1), "n_ban_va": len(bv), "n_te_bao_tb": int(np.mean(ns)),
                    "d1_um": round(float(np.mean(d1s)), 2),
                    "tk_kgian": f(tk_sp), "tk_bieuhien": f(tk_ex),
                    "_h_sp": np.mean(hs, 0), "_h_ex": np.mean(hx, 0), "_bv": bv}
    for ten in ("nguon", "dich"):
        v = ket[ten]
        Sc[ten] = {k: v[k] for k in v if not k.startswith("_")}
        Sc[ten]["chenh_kgian_tru_bieuhien"] = round(v["tk_kgian"][0] - v["tk_bieuhien"][0], 4)

    tv = lambda p, q: round(float(1.0 - 0.5 * np.abs(p - q).sum()), 4)
    Sc["chong_lap_sp"] = tv(ket["nguon"]["_h_sp"], ket["dich"]["_h_sp"])
    Sc["chong_lap_ex"] = tv(ket["nguon"]["_h_ex"], ket["dich"]["_h_ex"])
    Sc["d1_ti_le_nguon_tren_dich"] = round(ket["nguon"]["d1_um"] / max(ket["dich"]["d1_um"], 1e-9), 3)
    save()
    log(f"[{combo}] Y: tk_kgian nguồn={Sc['nguon']['tk_kgian']} bieuhien={Sc['nguon']['tk_bieuhien']} | "
        f"đích {Sc['dich']['tk_kgian']} vs {Sc['dich']['tk_bieuhien']} | "
        f"chồng lấp sp={Sc['chong_lap_sp']} ex={Sc['chong_lap_ex']} d1 tỉ lệ={Sc['d1_ti_le_nguon_tren_dich']}")

    # ---- cổng 3: giải GW thật trên MỘT cặp bản vá
    try:
        import ot as POT
    except Exception as e:
        W.append(f"{combo}/Y: không có POT -> bỏ cổng GW ({e!r})"); save(); return
    ia, ib = ket["nguon"]["_bv"][0], ket["dich"]["_bv"][0]
    if len(ia) > GW_N: ia = ia[rng.choice(len(ia), GW_N, replace=False)]
    if len(ib) > GW_N: ib = ib[rng.choice(len(ib), GW_N, replace=False)]
    p, q = np.ones(len(ia)) / len(ia), np.ones(len(ib)) / len(ib)
    nrm = lambda D: (D / max(D.max(), 1e-9)).astype(np.float64)
    def Dex_(F, idx):
        Z = _unit(F[idx].astype(np.float32)); D = 1.0 - Z @ Z.T; np.fill_diagonal(D, 0.0); return D
    # NULL phải là HÌNH HỌC NGẪU NHIÊN, không phải hoán vị. GW BẤT BIẾN với phép hoán vị đồng thời
    # hàng-cột (Ca[perm][:,perm] cho đúng cùng một nghiệm), nên "hoán vị toạ độ" là null RỖNG — nó
    # trả đúng con số của nhánh thật. Thay bằng toạ độ đều trong CÙNG hộp bao: phá cấu trúc không
    # gian mà giữ nguyên thang khoảng cách. Mức ngẫu nhiên của `dong_y` đã có sẵn ở `ngau_nhien`.
    bb0, bb1 = xy_s[ia].min(0), xy_s[ia].max(0)
    xy_rnd = bb0 + rng.rand(len(ia), 2) * (bb1 - bb0)
    nhanh = {"sp": (nrm(cdist(xy_s[ia], xy_s[ia])), nrm(cdist(xy_x[ib], xy_x[ib]))),
             "kgian_ngau": (nrm(cdist(xy_rnd, xy_rnd)), nrm(cdist(xy_x[ib], xy_x[ib]))),
             "ex": (nrm(Dex_(Fs, ia)), nrm(Dex_(Fx, ib)))}
    for k, (Ca, Cb) in nhanh.items():
        step(f"[{combo}] Y: GW {k} ({len(ia)}x{len(ib)})")
        try:                         # chữ ký POT đổi giữa các bản -> thử vị trí rồi thử từ khoá
            try:
                Pi, inf = POT.gromov.gromov_wasserstein(Ca, Cb, p, q, "square_loss", log=True, max_iter=200)
            except TypeError:
                Pi, inf = POT.gromov.gromov_wasserstein(C1=Ca, C2=Cb, p=p, q=q, log=True)
        except Exception as e:
            W.append(f"{combo}/Y/GW {k}: lỗi {e!r}"); log(f"[{combo}] Y GW {k} LỖI {e!r}"); continue
        gwd = next((inf[kk] for kk in ("gw_dist", "gw_dist_est", "loss") if kk in inf), np.nan)
        try: gwd = round(float(np.asarray(gwd).ravel()[-1]), 6)
        except Exception: gwd = None
        dy, ch = _dong_y_ghep(np.asarray(Pi), lab_s[ia], lab_x[ib])
        Sc[f"gw_{k}"] = {"gw_dist": gwd,
                         "dong_y": dy, "ngau_nhien": ch,
                         "loi_tren_ngau_nhien": None if dy is None else round(dy - ch, 4)}
        save()
        log(f"[{combo}] Y: GW {k}: gw={gwd} đồng ý={dy} "
            f"(ngẫu nhiên {ch}, lợi {Sc[f'gw_{k}']['loi_tren_ngau_nhien']})")
    gc.collect()


# =============================================================================== Phần V
def _gop(Z, nn_idx, ok, how, chunk=4000):
    """Ước lượng VỊ TRÍ của lân cận: trung bình hoặc TRUNG VỊ theo từng chiều.

    VÌ SAO CÓ HÀM NÀY: pas_true = 0.20-0.38 nghĩa là lân cận là một mẫu bị NHIỄM BẨN 20-38%.
    Điểm sụp đổ của trung bình là 0 — một tế bào khác loại đủ kéo lệch. Của trung vị là 0.5.
    Mức nhiễm bẩn đo được nằm GỌN trong vùng trung vị còn sống mà trung bình đã chết. Mọi thứ dự
    án đã thử (cmix, beta, làm trơn alpha, phạt Laplacian, CORAL quay, Phần K) đều là TRUNG BÌNH.

    Chia khối vì Z[nn_idx] cỡ (n, k, d); với n=45k, k=15, d=512 là 1.4 GB nếu làm một lần.
    """
    n, d = Z.shape
    out = np.empty((n, d), np.float32)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        v = np.asarray(Z[nn_idx[a:b]], np.float32)
        v = np.where(ok[a:b][:, :, None], v, np.nan)
        trong = ~np.isfinite(v).any(1)                 # hàng không còn láng giềng nào
        with warnings.catch_warnings():                # nanmean/nanmedian của lát toàn NaN cảnh báo
            warnings.simplefilter("ignore", RuntimeWarning)   # -> đã xử lý ngay dưới, đừng rác log
            r = np.nanmean(v, 1) if how == "tb" else np.nanmedian(v, 1)
        r[trong] = np.asarray(Z[a:b], np.float32)[trong]   # giữ nguyên tế bào cô lập
        out[a:b] = r
    return out


def part_v(combo, cx, S, save, W):
    """Gộp BỀN VỮNG thay cho trung bình — hệ quả trực tiếp của pas_true = 0.20-0.38.

    Giữ NGUYÊN cơ chế co rút của Phần K (w_i = sigma^2/(sigma^2+s2_i), s2 tính quanh TRUNG BÌNH cho
    MỌI nhánh) để `w` giống hệt nhau giữa các nhánh. Chỉ đổi đúng một thứ: ước lượng vị trí `m_i`.
    Như vậy chênh lệch quy được về ước lượng, không lẫn với việc đổi mức co rút.

    DỰ ĐOÁN KIỂM CHỨNG ĐƯỢC (đặt TRƯỚC khi chạy): nếu nguyên nhân là nhiễm bẩn thì mức hơn của
    trung vị so với trung bình phải TĂNG THEO pas_true. Thứ tự pas_true (đồ thị toàn bộ, k=15):
    cosmx6k .378 > visiumhd_ff .288 > visiumhd_ffpe .258 > stereoseq .197. Mức hơn KHÔNG theo thứ
    tự này thì giả thuyết sai — không được giải thích lại sau khi nhìn số.
    """
    Sc = S.setdefault("V", {}).setdefault(combo, {})
    pctx, ds, Fs = cx["pctx"], cx["ds"], cx["Fs"]
    g = pctx["groups_ev"]; lab_s = np.asarray(pctx["lab_src_ev"])
    T_, Q_ = np.asarray(ds["T"]), np.asarray(cx["Q"])
    xy = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    Wn, nn_idx, ok, cut, _ = spatial_graph(xy)

    def probe(Z):
        mu = np.asarray(Z[T_], np.float32).mean(0)
        nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(np.asarray(Z[T_], np.float32) - mu)
             ).kneighbors(_unit(np.asarray(Z[Q_], np.float32) - mu), return_distance=False)
        return bacc_on(lab_s[Q_], mode_rows(lab_s[T_][nn]), g)

    if "raw" not in Sc:
        step(f"[{combo}] V: raw"); Sc["raw"] = {"bacc": probe(Fs)}; save()
    F64 = np.asarray(Fs, np.float64)
    # s2 và w: tính MỘT LẦN, dùng chung cho mọi nhánh -> cô lập được biến duy nhất là `how`.
    m_tb = np.asarray(Wn @ F64)
    s2 = np.maximum(np.asarray(Wn @ (F64 ** 2).sum(1)) - (m_tb ** 2).sum(1), 0.0) / F64.shape[1]
    iso = np.asarray(Wn.sum(1)).ravel() <= 0
    med_s2 = float(np.median(s2[~iso])) if (~iso).any() else 1.0
    for c in KC:
        for how in AGGS:
            k = f"{how}_c{c:g}"
            if k in Sc: continue
            step(f"[{combo}] V: {k}")
            w = (c * med_s2) / (c * med_s2 + np.maximum(s2, 1e-12)); w[iso] = 0.0
            m = _gop(Fs, nn_idx, ok, how)
            Z = (F64 + w[:, None] * (np.asarray(m, np.float64) - F64)).astype(np.float32)
            Sc[k] = {"bacc": probe(Z), "w_tb": round(float(w.mean()), 4)}
            save(); log(f"[{combo}] V {k}: bacc={Sc[k]['bacc']} (raw={Sc['raw']['bacc']}) w_tb={Sc[k]['w_tb']}")
            del Z, m; gc.collect()
    S.setdefault("V_meta", {})[combo] = {"cat_canh": round(cut, 4),
                                         "nhan": "thô" if pctx.get("use_coarse") else "mịn"}
    save(); del Wn; gc.collect()


# =============================================================================== Phần S
def part_s(combo, cx, S, save, W):
    """THANG ĐO: pas_true thật sự bằng bao nhiêu, ở thang nào, đo bằng đơn vị đo được.

    LÝ DO: con số `pas_true` 0.21-0.40 từng được dùng để kết luận "mô quá dị thể ngay ở thang tế
    bào" là SAI THANG. Nó tính trên `xy[q]` (N_Q = 10000 trên 40-45k tế bào, run45.py:170), tức
    mật độ chỉ ~22% -> khoảng cách 15-NN giãn ra ~sqrt(n/10000) lần. Trong khi Phần K lại dùng đồ
    thị TOÀN BỘ tế bào (run45.py:348). Hai thang khác nhau, không được dùng số này giải thích số kia.

    Ở đây đo TRỰC TIẾP, không ước lượng:
      d1_um      — trung vị khoảng cách tới láng giềng GẦN NHẤT = khoảng cách giữa các tế bào.
                   Đây là thang quy chiếu ĐO ĐƯỢC, thay cho việc giả định đường kính tế bào ~10um.
      dk_um      — trung vị khoảng cách tới láng giềng thứ k.
      dk_tren_d1 — bán kính lân cận tính theo ĐƠN VỊ khoảng cách tế bào. Không cần giả định gì.
      pas_true, moran_true — trên nhãn THẬT, cho từng k, trên CẢ HAI đồ thị.

    PHÁN QUYẾT: pas_true ở thang chặt (đồ thị toàn bộ, k nhỏ) vẫn ~0.2-0.4 -> mô thật sự dị thể ở
    thang tế bào, và kết quả âm của Phần K có lời giải. Nếu nó tụt về ~0.05-0.1 -> lân cận chặt CÓ
    đồng nhất, Phần K hỏng vì lý do khác, và việc đóng hướng ST là vội.
    """
    Sc = S.setdefault("S", {}).setdefault(combo, {})
    xy_full = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    lab_full = np.asarray(cx["ds"]["lab"])
    q = np.asarray(cx["Q"])
    for ten, xy, lab in (("toanbo", xy_full, lab_full), ("Q", xy_full[q], lab_full[q])):
        for kk in SK_KS:
            key = f"{ten}_k{kk}"
            if key in Sc: continue
            step(f"[{combo}] S: {key}")
            nn = NearestNeighbors(n_neighbors=min(kk + 1, len(xy)), n_jobs=4).fit(xy)
            dist, _ = nn.kneighbors(xy)
            Wn, nn_idx, ok, cut, deg = spatial_graph(xy, k=kk, r_max=R_MAX_UM)
            pt, n_iso = pas(nn_idx, ok, lab)
            cnt = pd.Series(lab).value_counts()
            cand = [c for c in cnt.index if cnt[c] >= 20]
            I = moran_i(Wn, np.stack([(lab == c).astype(np.float64) for c in cand], 1)) if cand else np.array([0.0])
            d1, dk = float(np.median(dist[:, 1])), float(np.median(dist[:, -1]))
            Sc[key] = {"n": int(len(xy)), "d1_um": round(d1, 2), "dk_um": round(dk, 2),
                       "dk_tren_d1": round(dk / max(d1, 1e-9), 2), "cat_canh": round(cut, 4),
                       "bac_tb": round(float(deg.mean()), 2), "n_co_lap": int(n_iso),
                       "pas_true": None if pt is None else round(pt, 4),
                       "moran_true_tb": round(float(I.mean()), 4), "n_nhan": len(cand)}
            save(); log(f"[{combo}] S {key}: {Sc[key]}")
            del Wn, dist; gc.collect()


# =============================================================================== Phần K
def _lam_truot(Z, Wn, c):
    """Co rút thích ứng theo ĐỘ DỊ THỂ cục bộ (James-Stein trên đồ thị không gian).

        m_i   = trung bình nhúng của láng giềng không gian
        s2_i  = phương sai TRONG lân cận  -> cao ở ranh giới mô, thấp trong niche đồng nhất
        w_i   = sigma^2 / (sigma^2 + s2_i),   sigma^2 = c * trung vị(s2)
        z'_i  = z_i + w_i * (m_i - z_i)

    Ở ranh giới s2 lớn -> w nhỏ -> KHÔNG làm trơn (giữ tín hiệu thật). Trong vùng đồng nhất -> làm
    trơn mạnh (khử nhiễu). Đây là trung bình hậu nghiệm dưới mô hình Gauss cục bộ; hợp lệ vì Phần D
    đo được phần dư TRẮNG về không gian (Moran 0.007-0.017), tức nhiễu đẳng hướng theo toạ độ.

    sigma^2 cho dưới dạng BỘI của trung vị s2 nên c không thứ nguyên, khỏi phải ước lượng tuyệt đối.
    """
    Z = np.asarray(Z, np.float64)
    m = np.asarray(Wn @ Z)
    s2 = np.maximum(np.asarray(Wn @ (Z ** 2).sum(1)) - (m ** 2).sum(1), 0.0) / Z.shape[1]
    iso = np.asarray(Wn.sum(1)).ravel() <= 0
    med = float(np.median(s2[~iso])) if (~iso).any() else 1.0
    w = (c * med) / (c * med + np.maximum(s2, 1e-12))
    w[iso] = 0.0                                  # tế bào cô lập: giữ nguyên
    return (Z + w[:, None] * (m - Z)).astype(np.float32), w, s2


def part_k(combo, cx, S, save, W):
    """CỔNG: khử nhiễu không gian có nâng được LƯỢNG THÔNG TIN NHÃN trong nhúng nguồn không?

    Phần T đo trần: oracle tuyến tính chỉ hơn FROST 0.006-0.010 ở 3/4 cặp, trong khi Xenium tự
    chuyển cho mình đạt 0.91-0.94. Tức MỌI can thiệp ở tầng ánh xạ đều đã hết dư địa, và đòn bẩy
    duy nhất còn lại là chính BIỂU DIỄN. FM (scGPT-spatial / PCA) nhúng từng tế bào từ vector biểu
    hiện của nó và KHÔNG BAO GIỜ thấy toạ độ -> toạ độ là kênh thông tin nằm ngoài nhúng.

    BA NHÁNH, cùng một mức làm trơn trung bình, nếu không thì so sánh vô nghĩa:
      raw       — mốc, chính là tran_nguon của Phần T
      hoanvi    — làm trơn trên đồ thị dựng từ toạ độ ĐÃ HOÁN VỊ: sàn, phá thông tin không gian
      deu       — làm trơn ĐỀU với alpha = trung bình(w) của nhánh thích ứng
      thichung  — co rút thích ứng

    BẪY ĐÃ BIẾT: làm trơn khiến tế bào gần nhau giống nhau hơn, mà tế bào gần nhau VỐN thường cùng
    nhãn -> tran_nguon tăng một phần do rò rỉ tự tương quan nhãn, không phải do khử nhiễu. Nhánh
    `deu` hứng trọn phần rò rỉ đó. Chỉ phần `thichung` VƯỢT `deu` mới là đóng góp của tính thích ứng.
    """
    Sc = S.setdefault("K", {}).setdefault(combo, {})
    pctx, ds, Fs = cx["pctx"], cx["ds"], cx["Fs"]
    g = pctx["groups_ev"]; lab_s = np.asarray(pctx["lab_src_ev"])
    T_, Q_ = np.asarray(ds["T"]), np.asarray(cx["Q"])
    xy = lite_obs(cx["pair"])[["x", "y"]].to_numpy()
    Wn, _, _, cut, _ = spatial_graph(xy)
    rng = np.random.RandomState(0)
    Wn_p, _, _, _, _ = spatial_graph(xy[rng.permutation(len(xy))])

    def probe(Z):
        mu = np.asarray(Z[T_], np.float32).mean(0)
        nn = NearestNeighbors(n_neighbors=15, n_jobs=4).fit(_unit(np.asarray(Z[T_], np.float32) - mu)
             ).kneighbors(_unit(np.asarray(Z[Q_], np.float32) - mu), return_distance=False)
        return bacc_on(lab_s[Q_], mode_rows(lab_s[T_][nn]), g)

    if "raw" not in Sc:
        step(f"[{combo}] K: raw")
        Sc["raw"] = {"bacc": probe(Fs)}; save()
    for c in KC:
        k = f"c{c:g}"
        if k in Sc: continue
        step(f"[{combo}] K: {k}")
        Za, w, s2 = _lam_truot(Fs, Wn, c)
        a = float(w.mean())
        Zu = ((1 - a) * np.asarray(Fs, np.float64) + a * np.asarray(Wn @ np.asarray(Fs, np.float64))
              ).astype(np.float32)
        Zp, _, _ = _lam_truot(Fs, Wn_p, c)
        Sc[k] = {"thichung": probe(Za), "deu": probe(Zu), "hoanvi": probe(Zp),
                 "w_tb": round(a, 4), "w_p01": round(float(np.percentile(w, 1)), 4),
                 "w_min": round(float(w[w > 0].min()) if (w > 0).any() else 0.0, 4),
                 # ĐUÔI DƯỚI mới là chỗ tính thích ứng hoạt động: đa số tế bào nằm xa ranh giới nên
                 # p10-p90 hẹp là BÌNH THƯỜNG, không có nghĩa là co rút ~ đều (kiểm bằng số: biên có
                 # s2 gấp 3.4 lần, w thấp hơn ~30%, nhưng p10-p90 chỉ rộng 0.025).
                 "tl_bao_ve": round(float((w < 0.5 * a).mean()), 4),
                 "s2_cv": round(float(np.std(s2) / max(np.mean(s2), 1e-12)), 3)}
        save()
        v = Sc[k]
        log(f"[{combo}] K {k}: thichung={v['thichung']} deu={v['deu']} hoanvi={v['hoanvi']} "
            f"raw={Sc['raw']['bacc']} | w_tb={v['w_tb']} w_p01={v['w_p01']} "
            f"bảo_vệ={v['tl_bao_ve']:.1%} s2_cv={v['s2_cv']}")
        del Za, Zu, Zp; gc.collect()
    S.setdefault("K_meta", {})[combo] = {"cat_canh": round(cut, 4), "n_nhom": int(len(g)),
                                         "nhan": "thô" if pctx.get("use_coarse") else "mịn"}
    save(); del Wn, Wn_p; gc.collect()


# =============================================================================== Phần T
def bacc_on(y, p, keep):
    """Chép đúng run05.py:65-67 để số so sánh thẳng được với `xfer_main`."""
    m = np.isin(y, list(keep))
    return round(float(balanced_accuracy_score(y[m], p[m])), 4) if m.sum() else None


def part_t(combo, cx, S, save, W):
    """TRẦN của `xfer_main`: chuyển nhãn 15-NN khi KHÔNG có khoảng cách công nghệ.

    VÌ SAO: ở n=12, `xfer_main` KHÔNG phân biệt được bốn can thiệp khác hẳn nhau về cấu trúc —
    bỏ CORAL (p=0.79), usher_cc (p=0.85), usher_final (p=0.68), class_ridge_pl_cc (p=0.29) — trong
    khi cùng chỉ số đó tách FROST khỏi harmony/scanorama rất rõ. Đó là dấu hiệu bão hoà trong nội
    bộ họ USHER. Phần T đo xem trần nằm ở đâu, để biết `xfer_main` còn dùng làm chỉ số phân biệt
    được không, hay chỉ còn là ràng buộc "không được tụt".

    Oracle theo NHÃN là vô dụng ở đây: đặt mỗi tế bào nguồn cạnh một tế bào Xenium cùng nhãn thật
    thì kNN đương nhiên đúng -> ~1.0 một cách tầm thường. Nên đo hai đại lượng KHÔNG suy biến:
      tran_dich  — Xenium -> Xenium (ref -> phần giữ lại). Cùng chú giải, cùng nhúng, khoảng cách
                   công nghệ BẰNG 0. Đây là trần của chính thước đo.
      tran_nguon — nguồn -> nguồn (T -> Q) trong không gian nguồn. Đo lượng thông tin nhãn mà nhúng
                   nguồn tự mang. M-step là ánh xạ TUYẾN TÍNH nên nó không tạo ra được khả năng
                   phân tách vốn không có; đây là chặn mềm phía nguồn.
      *_ngau     — mốc ngẫu nhiên (hoán vị nhãn), để biết sàn.

    Dùng `lab_tgt_ev`/`lab_src_ev` + `groups_ev` của pctx, tức ĐÚNG nhãn và ĐÚNG nhóm mà
    `xfer_main` chấm (thô hay mịn tuỳ cặp), nếu không thì hai con số không so được với nhau.
    """
    Sc = S.setdefault("T", {}).setdefault(combo, {})
    # KHÔNG dùng `if Sc: return` ở đây: lần đầu viết hàm này chỉ có một khối việc, rồi khối oracle
    # được thêm vào CUỐI -> `if Sc: return` thấy tran_dich đã có là thoát ngay, oracle không bao giờ
    # chạy và KHÔNG ghi cảnh báo nào. Mỗi khối tự gác lấy khoá của nó.
    pctx, ds, Fx, Fs = cx["pctx"], cx["ds"], cx["Fx"], cx["Fs"]
    g = pctx["groups_ev"]
    lab_t, lab_s = np.asarray(pctx["lab_tgt_ev"]), np.asarray(pctx["lab_src_ev"])
    ref = np.asarray(pctx["ref"]); rng = np.random.RandomState(0)

    def knn_vote(Xfit, yfit, Xq, k=15):
        mu = np.asarray(Xfit, np.float32).mean(0)     # cùng giao thức với transferred_labels
        nn = NearestNeighbors(n_neighbors=min(k, len(Xfit)), n_jobs=4).fit(
            _unit(np.asarray(Xfit, np.float32) - mu)).kneighbors(
            _unit(np.asarray(Xq, np.float32) - mu), return_distance=False)
        return mode_rows(np.asarray(yfit)[nn])

    if "tran_dich" not in Sc:
        step(f"[{combo}] T: trần đích (Xenium->Xenium)")
        held = np.setdiff1d(np.arange(len(Fx)), ref)
        if len(held) == 0:
            W.append(f"{combo}: ref phủ toàn bộ Xenium -> không có phần giữ lại, bỏ trần đích")
        else:
            if len(held) > N_T_MAX: held = np.sort(rng.choice(held, N_T_MAX, replace=False))
            pr = knn_vote(Fx[ref], lab_t[ref], Fx[held])
            Sc["tran_dich"] = bacc_on(lab_t[held], pr, g)
            Sc["tran_dich_ngau"] = bacc_on(lab_t[held], rng.permutation(pr), g)
            Sc["n_ref"], Sc["n_giu_lai"] = int(len(ref)), int(len(held))
            save()

    if "tran_nguon" not in Sc:
        step(f"[{combo}] T: trần nguồn (nguồn->nguồn)")
        T_, Q_ = np.asarray(ds["T"]), np.asarray(cx["Q"])
        ps = knn_vote(Fs[T_], lab_s[T_], Fs[Q_])
        Sc["tran_nguon"] = bacc_on(lab_s[Q_], ps, g)
        Sc["tran_nguon_ngau"] = bacc_on(lab_s[Q_], rng.permutation(ps), g)
        Sc["n_nhom"] = int(len(g)); Sc["nhan"] = "thô" if pctx.get("use_coarse") else "mịn"
        save()

    # TRẦN THỨ BA — "ánh xạ tuyến tính tốt nhất nếu biết tương ứng LỚP hoàn hảo".
    # `class_ridge_pl_cc` của bảng chính dùng NHÃN GIẢ (cx["lab_def"], từ M_aux). Đổi sang nhãn THẬT
    # của nguồn là được oracle. KHÔNG suy biến về 1.0: base_class_ridge chỉ ghép theo lớp trên T rồi
    # hồi quy ridge, từng tế bào vẫn phải rơi đúng chỗ; và nó chỉ thấy nhãn của T, đánh giá trên Q.
    # Hiệu số oracle − nhãn giả = CÁI GIÁ CỦA VIỆC ĐOÁN SAI LỚP, tức đúng thứ M_niche của 1.0.3 nhắm vào.
    if "oracle_ridge_cc" not in Sc:
        if R20.KM16 is None:
            W.append("phần T: R20.KM16 chưa đặt -> bỏ oracle_ridge_cc"); log("phần T: THIẾU KM16, bỏ oracle")
        else:
            step(f"[{combo}] T: oracle ridge theo lớp (nhãn THẬT)")
            try:
                Fo = R30.M.base_class_ridge(Fs, cx["Fx"], ds["T"], np.asarray(ds["lab"]),
                                            np.asarray(cx["dx"]["lab"]))[0]
                Fo = R23C.coral(np.asarray(Fo, np.float32), cx["Fx"], ds["T"])   # khớp class_ridge_pl_cc (có CORAL)
                ev = R23.evaluate(cx, f"{combo}/T/oracle_ridge_cc", Fs, np.asarray(Fo, np.float32), cx["Fx"], W)
                Sc["oracle_ridge_cc"] = {k: ev.get(k) for k in
                                         ("xfer_main", "xfer_all", "total_v2", "batch_v2", "bio_v2")}
                save()
            except Exception as e:
                W.append(f"{combo}/T/oracle_ridge_cc: lỗi {e!r}"); log(f"[{combo}] T oracle LỖI {e!r}")
    log(f"[{combo}] T: {Sc}")


# =============================================================================== báo cáo
def report(S):
    L = [f"# 45 — chỉ số ST không chú giải + chẩn đoán phần dư (mô={TISSUE})",
         f"CẤU HÌNH: combos={COMBOS} seeds={SEEDS} k_spatial={K_SPATIAL} r_max={R_MAX_UM}µm "
         f"moran_min={MORAN_MIN} floor={FLOOR} | nhãn chuyển = bỏ phiếu 15-NN (đúng run22._knn_bacc)", ""]
    if S.get("D"):
        L += ["## Phần D — Moran's I của PHẦN DƯ (fit Θ trên T1, dư trên T2)", "```"]
        rows = []
        for c, sc in S["D"].items():
            for k, v in sc.items():
                rows.append({"tổ_hợp": c, "seed": k, **{kk: v[kk] for kk in
                            ("moran_tb", "moran_p90", "moran_hoanvi_tb", "n_T2", "canh_bi_cat")}})
        L += [pd.DataFrame(rows).to_string(index=False), "```"]
        for c, sc in S["D"].items():
            v = [x["moran_tb"] for x in sc.values()]
            h = [x["moran_hoanvi_tb"] for x in sc.values()]
            if not v: continue
            d = float(np.mean(v) - np.mean(h))
            L.append(f"- D1 {c}: Moran's I phần dư = {np.mean(v):.5f} so với hoán vị {np.mean(h):.5f} "
                     f"-> chênh {d:+.5f} -> {'CÓ tín hiệu không gian' if d >= 0.02 else 'không đáng kể'}")
        L.append("- Đọc: chênh < 0.02 nghĩa là phần dư gần như không mang cấu trúc không gian liên tục; "
                 "số hạng Laplacian trong 1.0.2 sẽ không có gì để tác động.")
        L.append("")
    if S.get("M"):
        L += ["## Phần M — Moran's I trên nhãn + PAS (|Δ| so với nhãn thật; THẤP hơn = trung thực hơn)", "```"]
        rows = []
        for c, sc in S["M"].items():
            for m, v in sc.items():
                rows.append({"tổ_hợp": c, "map": m, **{kk: v[kk] for kk in
                            ("moran_abs_dev", "moran_pred_mean", "moran_true_mean", "pas_pred", "pas_true",
                             "pas_abs_dev")}})
        L += [pd.DataFrame(rows).to_string(index=False), "```",
              "- CẢ HAI cột `*_abs_dev` là ĐỘ LỆCH so với nhãn thật, không phải giá trị thô: nhãn suy biến "
              "(gán tất cả cùng một loại) cho PAS = 0 và Moran's I = 0, nên 'thấp hơn là tốt hơn' trên giá trị "
              "thô sẽ thưởng cho nhãn vô dụng.", ""]
    if S.get("A"):
        L += ["## Phần A — sàng α: làm trơn không gian hậu kỳ (CHẶN TRÊN của 1.0.2)", "```"]
        rows = []
        for c, sc in S["A"].items():
            for k, v in sc.items():
                arm, a = k.split("@")
                rows.append({"tổ_hợp": c, "nhánh": arm, "α": float(a),
                             **{kk: v[kk] for kk in ("pas_abs_dev", "moran_abs_dev", "bacc_xfer", "d_cov")}})
        df = pd.DataFrame(rows).sort_values(["tổ_hợp", "nhánh", "α"])
        L += [df.to_string(index=False), "```",
              "- `pre` = coral(smooth(F)) là nhánh KHỚP với Laplacian ở M-step. α=0 của `pre` chính là "
              "FROST 1.0; α=0 của `nocoral` chính là u_cmix.",
              "- `d_cov` ở nhánh `pre` ≈ 0 với MỌI α một cách TẦM THƯỜNG (CORAL chạy sau cùng nên ép lại "
              "khớp mô-men). Đừng đọc đó là an toàn: nó chỉ có nghĩa cho `nocoral` và `post`.", ""]
        for c, sc in S["A"].items():
            b = sc.get("pre@0")
            if not b: continue
            # Tối ưu CÓ RÀNG BUỘC: lọc trước theo bacc rồi mới chọn pas_dev thấp nhất. Lấy min pas_dev
            # trước rồi mới kiểm bacc là SAI — một α hơi kém pas_dev nhưng giữ được bacc sẽ bị che mất.
            cands = [(v["pas_abs_dev"], k) for k, v in sc.items()
                     if k.startswith("pre@") and k != "pre@0" and v.get("pas_abs_dev") is not None]
            if not cands: continue
            ok_b = [t for t in cands if sc[t[1]]["bacc_xfer"] - b["bacc_xfer"] >= -0.02]
            if not ok_b:
                worst = min(cands)
                v_ = (f"MỌI α đều làm bacc tụt > 0.02 (tốt nhất {worst[1]}: "
                      f"{sc[worst[1]]['bacc_xfer'] - b['bacc_xfer']:+.4f}) -> bào vào tín hiệu sinh học: HỎNG")
            else:
                dp, kk = min(ok_b)
                dp -= b["pas_abs_dev"]; db = sc[kk]["bacc_xfer"] - b["bacc_xfer"]
                if dp > -0.01:
                    v_ = ("trong vùng giữ được bacc, pas_dev KHÔNG giảm (<0.01) -> Laplacian không cắn "
                          "qua được CORAL: 1.0.2 CHẾT")
                else:
                    v_ = (f"pas_dev giảm {-dp:.4f} tại {kk}, bacc đổi {db:+.4f} -> ĐƯỜNG SỐNG, "
                          f"chấm đầy đủ R23.evaluate ở α đó")
            L.append(f"- A1 {c}: {v_}")
        L += ["- Đối chiếu chéo: `nocoral` cải thiện mạnh mà `pre` thì không => CORAL xoá công làm trơn "
              "-> phải chuyển sang bản phạt có trọng số A (‖(Wxᵢ−Wxⱼ)A‖²), KHÔNG phải bỏ Laplacian.", ""]
    if S.get("B"):
        MB = ["xfer_main", "xfer_all", "total_v2", "batch_v2", "bio_v2",
              "pas_abs_dev", "moran_abs_dev"]
        gom = {}                                   # (tổ_hợp, nhánh) -> {chỉ số: [giá trị theo seed]}
        for c, sc in S["B"].items():
            for k, v in sc.items():
                nh = k.split("/", 1)[1] if "/" in k else k
                d = gom.setdefault((c, nh), {m: [] for m in MB})
                for m in MB:
                    if isinstance(v.get(m), (int, float)): d[m].append(v[m])
        rows = [{"tổ_hợp": c, "nhánh": nh, "n_seed": max(len(x) for x in d.values()),
                 **{m: (round(float(np.mean(d[m])), 4) if d[m] else None) for m in MB}}
                for (c, nh), d in sorted(gom.items())]
        L += ["## Phần B — chấm ĐẦY ĐỦ ở vài điểm α (TB qua seed)", "```",
              pd.DataFrame(rows).to_string(index=False), "```",
              "- `pre@0` CHÍNH LÀ FROST 1.0 — mốc so duy nhất đúng ở đây, đừng lấy `u_cmix_cc` của bảng 23e.", ""]
        TB = {(r["tổ_hợp"], r["nhánh"]): r for r in rows}
        for c in sorted({r["tổ_hợp"] for r in rows}):
            b = TB.get((c, "pre@0"))
            if not b: L.append(f"- B1 {c}: THIẾU mốc `pre@0` -> không kết luận được"); continue
            for nh in sorted(r["nhánh"] for r in rows if r["tổ_hợp"] == c):
                if nh == "pre@0": continue
                v = TB[(c, nh)]
                g = lambda m: (v.get(m) or 0) - (b.get(m) or 0)
                # TIÊU CHÍ: total_v2 không tụt quá 0.02, VÀ phần ST bù lại TƯƠNG XỨNG.
                # `mat` = phần total_v2 mất (0 nếu tăng); `bu` = trung bình mức giảm của hai chỉ số
                # ST (chúng là ĐỘ LỆCH nên giảm = tốt). Trung bình chứ không lấy max: cả hai chỉ số
                # đều được tính, nên một chỉ số xấu đi sẽ ăn vào phần bù thay vì bị bỏ qua.
                mat = max(0.0, -g("total_v2"))
                bu = (-g("pas_abs_dev") - g("moran_abs_dev")) / 2
                dat = g("total_v2") >= -0.02 and bu > 0 and bu >= mat
                L.append(f"- B1 {c} {nh} (n={v['n_seed']}): total_v2 {g('total_v2'):+.4f}  "
                         f"xfer_main {g('xfer_main'):+.4f}  batch_v2 {g('batch_v2'):+.4f}  | "
                         f"pas_dev {g('pas_abs_dev'):+.4f}  moran_dev {g('moran_abs_dev'):+.4f}  | "
                         f"bù {bu:+.4f} so với mất {mat:.4f}"
                         + ("  -> ĐẠT" if dat else
                            "  -> total tụt > 0.02" if g("total_v2") < -0.02 else
                            "  -> ST không cải thiện" if bu <= 0 else "  -> bù KHÔNG tương xứng"))
        # Wilcoxon ghép cặp trên (tổ_hợp, seed) — 4 cặp × 3 seed = n=12, đúng chuẩn của dự án.
        try:
            from scipy.stats import wilcoxon
            cap = {}
            for c, sc in S["B"].items():
                for k, v in sc.items():
                    if "/" not in k: continue
                    sd, nh = k.split("/", 1)
                    cap.setdefault(nh, {})[(c, sd)] = v
            goc = cap.get("pre@0", {})
            if goc:
                L += ["### Wilcoxon ghép cặp so với `pre@0` (n = số tổ_hợp × seed)", "```"]
                for nh in sorted(k for k in cap if k != "pre@0"):
                    for m in ("total_v2", "xfer_main", "pas_abs_dev", "moran_abs_dev"):
                        a = [(goc[t].get(m), cap[nh][t].get(m)) for t in cap[nh]
                             if t in goc and isinstance(goc[t].get(m), (int, float))
                             and isinstance(cap[nh][t].get(m), (int, float))]
                        if len(a) < 6: continue
                        x = np.array([u for u, _ in a]); y = np.array([v for _, v in a])
                        pv = float("nan") if np.allclose(x, y) else float(wilcoxon(x, y).pvalue)
                        huong = "THẤP hơn tốt" if m.endswith("_dev") else "CAO hơn tốt"
                        L.append(f"{nh:<10} {m:<14} n={len(a):<3} Δ={np.mean(y - x):+.4f}  "
                                 f"p={pv:.4f}  ({huong})")
                L += ["```", "- p >= 0.05 trên `total_v2` = GIỮ VỮNG (hoà). p < 0.05 với Δ ÂM trên "
                      "`pas_abs_dev`/`moran_abs_dev` = chỉ số ST cải thiện có ý nghĩa.", ""]
        except Exception as e:
            L.append(f"- (Wilcoxon bỏ qua: {e!r})")
        L += ["- Tiêu chí: `total_v2` không tụt quá **0.02** so với FROST 1.0, VÀ phần ST bù lại "
              "TƯƠNG XỨNG: `bù = trung bình(−Δpas_dev, −Δmoran_dev)` phải dương VÀ không nhỏ hơn "
              "`mất = max(0, −Δtotal_v2)`. `xfer_main` in ra để theo dõi, KHÔNG phải điều kiện — đọc "
              "nó thay cho `total_v2` chính là sai lầm đã chôn nhầm 1.0.2 một lần.",
              "- TB qua seed chỉ để xem nhanh; kết luận phải chạy Wilcoxon n=12 trên `stat33`.", ""]
    if S.get("T"):
        L += ["## Phần T — TRẦN của `xfer_main` (chuyển nhãn 15-NN, không khoảng cách công nghệ)", "```",
              pd.DataFrame(S["T"]).T.to_string(), "```",
              "- `tran_dich` = Xenium -> Xenium: trần của chính thước đo. So nó với `xfer_main` của FROST "
              "ở `out<P>e`. Sát nhau -> `xfer_main` đã bão hoà, chỉ còn dùng làm ràng buộc KHÔNG ĐƯỢC TỤT, "
              "không dùng để xếp hạng biến thể.",
              "- `tran_nguon` = nguồn -> nguồn: lượng thông tin nhãn nhúng nguồn tự mang. M-step là ánh xạ "
              "TUYẾN TÍNH nên không tạo ra khả năng phân tách vốn không có.",
              "- `*_ngau` là sàn (hoán vị nhãn dự đoán). Nhãn thô hay mịn theo cột `nhan` — đúng cái "
              "`xfer_main` dùng cho cặp đó.",
              "- `tran_nguon` CÓ giám sát (huấn luyện trên nhãn nguồn ở T); FROST không bao giờ thấy nhãn "
              "nguồn. Nên FROST thấp hơn nó không phải là điều đáng trách.", ""]
        orc = {c: (v.get("oracle_ridge_cc") or {}).get("xfer_main") for c, v in S["T"].items()}
        if any(x is not None for x in orc.values()):
            L += ["### Oracle: ridge theo lớp với nhãn THẬT (trần của ÁNH XẠ TUYẾN TÍNH)", "```",
                  pd.DataFrame({c: (v.get("oracle_ridge_cc") or {}) for c, v in S["T"].items()}).T.to_string(),
                  "```",
                  "- So với `class_ridge_pl_cc` (nhãn GIẢ) ở `out<P>e`: hiệu số = cái giá của việc đoán sai lớp.",
                  "- Oracle ≈ `xfer_main` của FROST → căn chỉnh tuyến tính đã sát trần của chính nó; "
                  "muốn hơn phải BỎ ràng buộc tuyến tính, không phải tinh chỉnh chi phí ghép cặp.",
                  "- Oracle cao hơn nhiều → phần lớn khoảng cách đến từ TƯƠNG ỨNG SAI, đúng chỗ ý tưởng A nhắm.", ""]
    if S.get("R"):
        L += ["## Phần R — FROST 1.0.3: CORAL QUAY theo không gian", "```"]
        rows = []
        for c, sc in S["R"].items():
            for k, v in sc.items():
                rows.append({"tổ_hợp": c.split("/")[0], "t": k, **{m: v.get(m) for m in
                            ("pas_abs_dev", "moran_abs_dev", "rough", "d_cov",
                             "xfer_main", "xfer_all", "total_v2", "batch_v2")}})
        L += [pd.DataFrame(rows).to_string(index=False), "```",
              "- `t=0` CHÍNH LÀ CORAL hiện hành (R = I) — mốc so duy nhất đúng. `t=1` là R* tối ưu Ruhe.",
              "- `d_cov` ≈ 0.01 ở MỌI t, KHÔNG phải 0: Ws dựng từ Sigma_s đã cộng RIDGE=0.01 nên "
              "Ws·Sigma_s_thô·Ws ≠ I đúng bằng lượng ridge. Thứ phải kiểm là nó **không đổi theo t** "
              "(bất biến quay). d_cov tăng theo t mới là lỗi code.",
              "- `rough` = tr(ZᵀLZ)/(n·d), hàm mục tiêu được tối thiểu. Chỉ đảm bảo `rough(t=1) < "
              "rough(t=0)`; ở t trung gian nó CÓ THỂ nhô lên vì nội suy cực phân rã không phải đường "
              "trắc địa trên SO(d) (đã thấy +0.1% tại t=0.25 khi thử bằng số). `rough(1) >= rough(0)` "
              "mới là dấu hiệu R* tính sai.", ""]
        for c, sc in S["R"].items():
            b = sc.get("t0")
            if not b: continue
            for k, v in sc.items():
                if k == "t0" or v.get("total_v2") is None or b.get("total_v2") is None: continue
                dp = (v.get("pas_abs_dev") or 0) - (b.get("pas_abs_dev") or 0)
                dt = v["total_v2"] - b["total_v2"]; dx = (v.get("xfer_main") or 0) - (b.get("xfer_main") or 0)
                ok = dp < -0.005 and dt >= -0.01
                L.append(f"- R1 {c} {k}: pas {dp:+.4f}  total_v2 {dt:+.4f}  xfer_main {dx:+.4f}"
                         + ("  -> ĐẠT (ST tốt lên, total giữ)" if ok else "  -> không đạt"))
        L += ["- Tiêu chí 1.0.3: `pas_abs_dev` giảm > 0.005 và `total_v2` không tụt quá 0.01 so với t=0. "
              "`xfer_main` in ra để biết giá phải trả, không dùng làm điều kiện.", ""]
    if S.get("K"):
        L += ["## Phần K — CỔNG: khử nhiễu không gian có nâng thông tin nhãn trong nhúng nguồn không?", "```"]
        rows = []
        for c, sc in S["K"].items():
            r0 = sc.get("raw", {}).get("bacc")
            for k, v in sc.items():
                if k == "raw": continue
                rows.append({"tổ_hợp": c.split("/")[0], "c": k, "raw": r0, "hoanvi": v["hoanvi"],
                             "deu": v["deu"], "thichung": v["thichung"],
                             "thichung-deu": round((v["thichung"] or 0) - (v["deu"] or 0), 4),
                             "w_tb": v["w_tb"], "w_p01": v["w_p01"],
                             "tl_bao_ve": v["tl_bao_ve"], "s2_cv": v["s2_cv"]})
        L += [pd.DataFrame(rows).to_string(index=False), "```",
              "- `raw` = chính `tran_nguon` của Phần T. `hoanvi` = làm trơn trên toạ độ đã hoán vị (sàn).",
              "- **Cột quyết định là `thichung-deu`.** `deu` có CÙNG mức làm trơn trung bình nên nó hứng "
              "trọn phần tăng do rò rỉ tự tương quan nhãn theo không gian. Chỉ phần vượt mới là đóng "
              "góp của TÍNH THÍCH ỨNG — tức của ý tưởng, chứ không phải của việc làm trơn nói chung.",
              "- Tính thích ứng hoạt động ở ĐUÔI DƯỚI của `w`, không ở phần giữa: đa số tế bào nằm xa "
              "ranh giới mô nên p10-p90 hẹp là BÌNH THƯỜNG. Hai cột cần nhìn là `w_p01` (mức bảo vệ "
              "ở tế bào dị thể nhất) và `tl_bao_ve` = tỉ lệ tế bào có w < 0.5·w_tb. `tl_bao_ve` ~ 0 "
              "mới nghĩa là co rút thực sự ĐỀU và ý tưởng rỗng.",
              "- Kiểm bằng số trên dữ liệu tổng hợp có ranh giới rõ: s2 ở biên gấp 3.4 lần, w ở biên "
              "thấp hơn ~30%, MSE so với trạng thái thật: thô 1.004 / đều 0.144 / thích ứng 0.110.",
              "- CẢNH BÁO ĐỌC: cổng này chỉ nói về thông tin trong ĐẶC TRƯNG NGUỒN. Việc khử nhiễu có "
              "giúp CĂN CHỈNH hay không thì phải dựng lại map (GPU) mới biết. Nó dùng để GIẾT ý tưởng "
              "rẻ, không phải để xác nhận.", ""]
    if S.get("S"):
        L += ["## Phần S — THANG ĐO: pas_true theo bán kính lân cận, đo bằng đơn vị đo được", "```"]
        rows = []
        for c, sc in S["S"].items():
            for k, v in sc.items():
                g_, kk_ = k.rsplit("_k", 1)
                rows.append({"tổ_hợp": c.split("/")[0], "đồ_thị": g_, "k": int(kk_), **v})
        df = pd.DataFrame(rows).sort_values(["tổ_hợp", "đồ_thị", "k"])
        L += [df.to_string(index=False), "```",
              "- `d1_um` = trung vị khoảng cách tới láng giềng GẦN NHẤT = khoảng cách giữa các tế bào, "
              "ĐO ĐƯỢC. `dk_tren_d1` = bán kính lân cận tính theo đơn vị đó — không cần giả định đường "
              "kính tế bào.",
              "- Hàng `đồ_thị=toanbo` là thang mà **Phần K** dùng; hàng `Q` là thang mà **Phần M** "
              "(pas_true 0.21-0.40 đã công bố) dùng. Trước đây tôi lấy số của hàng này giải thích hàng "
              "kia — SAI THANG, Phần S có để sửa đúng chỗ đó.",
              "- PHÁN QUYẾT: `pas_true` ở `toanbo` với k nhỏ vẫn ~0.2-0.4 -> mô dị thể thật ở thang tế "
              "bào, kết quả âm của Phần K có lời giải, hướng ST đóng. Tụt về ~0.05-0.1 -> lân cận chặt "
              "CÓ đồng nhất, Phần K hỏng vì lý do khác, và việc đóng hướng ST là vội.", ""]
    if S.get("V"):
        L += ["## Phần V — gộp BỀN VỮNG thay cho trung bình", "```"]
        PAS = {"cosmx6k": .378, "visiumhd_ff": .288, "visiumhd_ffpe": .258, "stereoseq": .197}
        rows, hieu = [], {}
        for c, sc in S["V"].items():
            nm = c.split("/")[0]; r0 = sc.get("raw", {}).get("bacc")
            for cc in sorted({k.split("_c")[1] for k in sc if k != "raw"}, key=float):
                d = {"tổ_hợp": nm, "c": cc, "raw": r0, "pas_true": PAS.get(nm)}
                for how in ("tb", "trungvi"):
                    d[how] = sc.get(f"{how}_c{cc}", {}).get("bacc")
                if d.get("tb") is not None and d.get("trungvi") is not None:
                    d["trungvi-tb"] = round(d["trungvi"] - d["tb"], 4)
                    hieu.setdefault(nm, []).append(d["trungvi-tb"])
                rows.append(d)
        L += [pd.DataFrame(rows).to_string(index=False), "```",
              "- `w` giống HỆT nhau giữa các nhánh (s2 tính quanh trung bình, một lần). Biến duy nhất "
              "là ước lượng vị trí -> chênh lệch quy thẳng về nó.", ""]
        if hieu:
            xh = sorted(hieu.items(), key=lambda t: -PAS.get(t[0], 0))
            L.append("- DỰ ĐOÁN đặt TRƯỚC: mức hơn của trung vị phải TĂNG THEO `pas_true`. Thứ tự "
                     "pas_true giảm dần: " + " > ".join(k for k, _ in xh))
            L.append("  Mức hơn tốt nhất theo thứ tự đó: "
                     + " | ".join(f"{k} {max(v):+.4f}" for k, v in xh))
            best = [max(v) for _, v in xh]
            L.append("  -> " + ("ĐƠN ĐIỆU GIẢM, khớp dự đoán: nhiễm bẩn là nguyên nhân"
                                if all(best[i] >= best[i + 1] for i in range(len(best) - 1))
                                else "KHÔNG đơn điệu -> giả thuyết nhiễm bẩn KHÔNG được số liệu đỡ; "
                                     "không được diễn giải lại sau khi nhìn kết quả"))
        L += ["- Mốc phải vượt là `raw`, không phải `tb`: trung vị hơn trung bình mà cả hai vẫn dưới "
              "`raw` thì vẫn là làm trơn có hại, chỉ là hại ít hơn.", ""]
    if S.get("W"):
        L += ["## Phần W — tự tương quan không gian làm bộ lọc CHIỀU (không phải bộ lọc tế bào)", "```"]
        rows = []
        for c, v in S["W"].items():
            r = {"tổ_hợp": c.split("/")[0], "d": v.get("d"),
                 "moran_ng_p50": (v.get("moran_nguon") or {}).get("p50"),
                 "moran_ng_p90": (v.get("moran_nguon") or {}).get("p90"),
                 "moran_ng_max": (v.get("moran_nguon") or {}).get("p100"),
                 "sàn_hoánvị": (v.get("sd_hoanvi") or {}).get("nguon"),
                 "tl_có_tín_hiệu": (v.get("tl_chieu_co_tin_hieu") or {}).get("nguon"),
                 "SPEARMAN_ng_đích": v.get("spearman_nguon_dich"),
                 "spearman_moran_var": v.get("spearman_moran_var"),
                 "bacc_toànbộ": v.get("bacc_toanbo")}
            for k in sorted(v):
                if k.startswith("bacc_moran_top") or k.startswith("bacc_var_top"): r[k[5:]] = v[k]
            rows.append(r)
        L += [pd.DataFrame(rows).to_string(index=False), "```",
              "- **Cổng là `SPEARMAN_ng_đích`**: tương quan giữa Moran theo chiều của nguồn và của "
              "Xenium. Hai bên cùng cơ sở nên chiều so được. CAO -> 'chiều có cấu trúc không gian' là "
              "tính chất chung, độc lập công nghệ, đánh trọng số theo nó có nghĩa. ~0 -> cấu trúc "
              "không gian là đặc thù công nghệ, trọng số sẽ tuỳ tiện và ý tưởng chết.",
              "- `tl_có_tín_hiệu` = tỉ lệ chiều có Moran > 3·sàn hoán vị. ~0 -> không có gì để lọc.",
              "- `spearman_moran_var` cao -> chiều Moran cao cũng là chiều phương sai cao, nên đánh "
              "trọng số gần như không đổi gì. Thấp hoặc âm -> không gian surface được chiều mà phương "
              "sai bỏ sót; ĐÓ mới là chỗ có lợi.",
              "- `moran_topN` so với `var_topN`: phép thử TRỰC TIẾP 'chọn chiều theo không gian có hơn "
              "chọn theo phương sai không'. Đây là bằng chứng, không phải suy luận gián tiếp.", ""]
    if S.get("X"):
        L += ["## Phần X — FROST 1.0.4 Spatial-CORAL (khớp mô-men bậc hai CÓ TRỌNG SỐ KHÔNG GIAN)", "```"]
        rows = []
        for c, sc in S["X"].items():
            mt = S.get("X_meta", {}).get(c, {})
            for k, v in sc.items():
                rows.append({"tổ_hợp": c, "nhánh": k, "sàn_d_sp": mt.get("d_sp_san"),
                             **{kk: v.get(kk) for kk in ("d_sp", "d_cov", "pas_abs_dev", "moran_abs_dev",
                                "xfer_main", "total_v2", "batch_v2", "bio_v2")}})
        L += [pd.DataFrame(rows).to_string(index=False), "```"]
        if S.get("X_meta"):
            L += ["", "```", pd.DataFrame(S["X_meta"]).T.to_string(), "```"]
        L += ["", "CÁCH ĐỌC — ba cổng, theo thứ tự; trượt cổng nào thì dừng tại đó:",
              "- CỔNG 1 `d_sp_san` so với `d_sp_t0`: sàn là chặn dưới trên TOÀN nhóm trực giao. "
              "Sàn ≈ t0 -> phép quay về nguyên tắc không sửa được lệch mô-men không gian, 1.0.4 chết "
              "bất kể các số còn lại. Sàn << t0 -> có chỗ để sửa.",
              "- CỔNG 2 `d_cov` phải đứng yên ở mức RIDGE≈0.01 với MỌI t. Tăng theo t = code sai, "
              "KHÔNG phải phát hiện khoa học (bài học phần R: RIDGE phá bất biến chuẩn ở d=512).",
              "- CỔNG 3 `hoanvi@t1` so với `t1`: R* dựng từ đồ thị ĐÃ HOÁN VỊ. Ngang nhau -> thứ mua "
              "được là bậc tự do quay nói chung, KHÔNG phải thông tin không gian; hết tính mới.",
              "- Chỉ khi qua cả ba mới đọc tới `total_v2`/`xfer_main`. Mốc là `t0` (= FROST khi "
              "SPC_TAG=cmix), KHÔNG phải bảng u_cmix_cc của vòng 23e.", ""]
    if S.get("Y"):
        L += ["## Phần Y — CỔNG CPU cho C_sp (toạ độ làm ma trận cấu trúc GW kiểu PASTE)", "```"]
        rows = []
        for c, sc in S["Y"].items():
            for ben in ("nguon", "dich"):
                v = sc.get(ben, {})
                rows.append({"tổ_hợp": c, "bên": ben, "L_µm": v.get("L_um"), "n_vá": v.get("n_ban_va"),
                             "n_tb": v.get("n_te_bao_tb"), "d1_µm": v.get("d1_um"),
                             "tk_kgian": v.get("tk_kgian"), "tk_biểu_hiện": v.get("tk_bieuhien"),
                             "chênh": v.get("chenh_kgian_tru_bieuhien")})
        L += [pd.DataFrame(rows).to_string(index=False), "```", "", "```"]
        rows = []
        for c, sc in S["Y"].items():
            r = {"tổ_hợp": c, "chồng_lấp_sp": sc.get("chong_lap_sp"), "chồng_lấp_ex": sc.get("chong_lap_ex"),
                 "d1_tỉ_lệ": sc.get("d1_ti_le_nguon_tren_dich")}
            for k in ("sp", "kgian_ngau", "ex"):
                g = sc.get(f"gw_{k}") or {}
                r[f"gw_{k}"] = g.get("gw_dist"); r[f"đồng_ý_{k}"] = g.get("dong_y")
                r[f"lợi_{k}"] = g.get("loi_tren_ngau_nhien")
            rows.append(r)
        L += [pd.DataFrame(rows).to_string(index=False), "```"]
        L += ["", "CÁCH ĐỌC — ba cổng; đây là cổng CPU đặt TRƯỚC việc sửa src_cuda và chạy lại USHER trên GPU:",
              "- CỔNG 1 `chênh = tk_kgian − tk_biểu_hiện`, TRÊN CẢ HAI BÊN. Việc duy nhất của ma trận "
              "cấu trúc là nói 'hai tế bào này gần nhau về cấu trúc'. Chênh âm rõ ở cả hai bên -> "
              "biểu hiện đã nói chính xác hơn toạ độ, trộn toạ độ vào C chỉ làm bẩn. Đúng logic đã "
              "giết Phần W (phương sai thắng Moran).",
              "- CỔNG 2 `chồng_lấp_sp` (lược đồ khoảng cách sau chuẩn hoá D/D.max(), nguồn vs đích). "
              "Thấp, hoặc `d1_tỉ_lệ` lệch xa 1 -> hai bên ở hai thang mật độ khác nhau, GW sẽ khớp "
              "bằng cách CO GIÃN mật độ chứ không khớp sinh học.",
              "- CỔNG 3 `lợi_sp` so với `lợi_kgian_ngau` và `lợi_ex`: đồng ý nhãn của kế hoạch GW "
              "trừ mức ngẫu nhiên. `lợi_sp` ≈ `lợi_kgian_ngau` -> cấu trúc không gian không mang gì. "
              "`lợi_sp` << `lợi_ex` -> C_sp thua C biểu hiện, chỉ đáng trộn với λ nhỏ (PASTE thấy "
              "α=0.1 tốt nhất, α=1 khớp đúng 0 cặp).",
              "- Null là HÌNH HỌC NGẪU NHIÊN, không phải hoán vị: GW bất biến với hoán vị đồng thời "
              "hàng-cột nên 'hoán vị toạ độ' là null rỗng.", ""]
    if S.get("M_meta"):
        L += ["## Đồ thị không gian", "```", pd.DataFrame(S["M_meta"]).T.to_string(), "```", ""]
    if S.get("A_meta"):
        L += ["## Đồ thị không gian — phần A", "```", pd.DataFrame(S["A_meta"]).T.to_string(), "```", ""]
    L += ["## Cảnh báo"] + ([f"- {w}" for w in dict.fromkeys(S.get("warnings", []))] or ["- không có"])
    open(f"{OUT}/summary.md", "w").write("\n".join(L) + "\n")
    log("\n".join(L))


def main():
    assert COMBOS, "COMBOS_ALL rỗng"
    os.makedirs(OUT, exist_ok=True)
    # PHẢI là việc ĐẦU TIÊN: run30.py:47 gán OUT của mọi module về out<P>i ngay lúc import, nên mọi
    # hàm có cache (get_niche_label_comp, setup_pairs, get_pair_niche...) sẽ ghi vào thư mục đó nếu
    # gọi trước khi khôi phục -> FileNotFoundError: out<P>i/labcomp_*.npy.
    for m in (R17, R18, R19, R20, R22, R22.R21, R23, R23C, R26G):
        if hasattr(m, "OUT"): m.OUT = OUT
    log(f"45 [{VER}] | mô={TISSUE} | combos={COMBOS} | phần={sorted(PARTS)} | OUT={OUT}")
    W0 = []
    import_cache(W0)     # PHẢI trước get_niche_label_comp ở dưới, nếu không nó dựng lại từ đầu
    S = json.load(open(f"{OUT}/summary.json")) if os.path.exists(f"{OUT}/summary.json") else {}
    if not S:
        for fj in sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}/summary.json", recursive=True)):
            S = json.load(open(fj)); log(f"tiếp nối <- {fj}"); break

    def save(): json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)

    needed = sorted({c.split("/")[0] for c in COMBOS})
    D, F, PCTX = R18.setup_pairs(needed)
    from sklearn.cluster import KMeans
    km8 = KMeans(n_clusters=R20.K_NICHE_TYPES, random_state=0, n_init=10).fit(
        R17.get_niche_label_comp(needed[0], S, save)[1])
    R22.G.update(D=D, F=F, PCTX=PCTX, S=S, save=save, kmeans8=km8, pca_cache={})
    if PARTS & {"B", "T", "R", "X"}:
        # BẮT BUỘC cho phần B VÀ T: R23.evaluate -> R20.etrack -> eval_e_track(..., KM16, ...) -> KM16.predict.
        # run20.py:101 khai KM16 = None và CHỈ run20.main() đặt; sáu module khác đều phải tự đặt lại
        # (run21:308, run22:346, run23:168, run23c:40, run26d:147, run26g:150 — run21:306 ghi rõ đây là
        # lỗi từng mắc). Chỉ đặt khi chạy B vì fine16_label_comp có thể phải dựng cache, phần D/M/A không cần.
        step("dựng KMeans niche mịn 16 (cho phần B/T/R/X)")
        R20.KM16 = R20.KMeans(n_clusters=R20.K_FINE16, random_state=0, n_init=10).fit(
            R20.fine16_label_comp(needed[0], S, save)[1])
    R22.PRIOR.clear(); R22.scan_maps(P, R22.PRIOR, log)   # bài học 42: PRIOR phải quét trước get_map
    W = S.setdefault("warnings", []); W.extend(W0)
    S["config"] = (f"45 mô={TISSUE} combos={COMBOS} r_max={R_MAX_UM} moran_min={MORAN_MIN} k={K_SPATIAL} "
                   f"phần={sorted(PARTS)} alphas={ALPHAS} alpha_tag={ALPHA_TAG} B={PART_B_ARMS}")
    try:
        # Mỗi phần bọc riêng: trước đây main CHỈ bắt OutOfTime, nên một lỗi bất kỳ trong một phần
        # giết cả lần chạy và KHÔNG ghi lại gì — đã mất nhiều giờ vì chuyện này. Giờ lỗi của một
        # phần/một cặp được ghi thành cảnh báo kèm traceback rút gọn, các phần còn lại vẫn chạy.
        import traceback
        for combo in COMBOS:
            cx = R22.ctx(combo)
            for ten, fn in (("D", part_d), ("M", part_m), ("A", part_a), ("B", part_b),
                            ("T", part_t), ("R", part_r), ("K", part_k), ("S", part_s),
                            ("V", part_v), ("W", part_w), ("X", part_x), ("Y", part_y)):
                if ten not in PARTS: continue
                try:
                    fn(combo, cx, S, save, W)
                except OutOfTime:
                    raise
                except Exception as e:
                    tb = traceback.format_exc().strip().splitlines()
                    W.append(f"{combo}/phần {ten}: LỖI {e!r} | {tb[-2].strip() if len(tb) > 1 else ''}")
                    log(f"[{combo}] phần {ten} LỖI:\n" + "\n".join(tb[-6:]))
                    save()
    except OutOfTime as e:
        W.append(f"HẾT GIỜ tại {e}"); log(f"HẾT GIỜ: {e}")
    S["warnings"] = W; save()
    report(S)


if __name__ == "__main__":
    main()

---
## Chạy (CPU)

In [ ]:
!cd /kaggle/working && OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 PYTHONUNBUFFERED=1 RESUME=1 \
    SMOKE=$SMOKE TISSUE=$TISSUE OUT_DIR_45=$OUT_DIR_45 ONLY_PART=$ONLY_PART R_MAX_UM=$R_MAX_UM MORAN_MIN=$MORAN_MIN K_SPATIAL=$K_SPATIAL ALPHAS=$ALPHAS ALPHA_TAG=$ALPHA_TAG PART_B_ARMS=$PART_B_ARMS N_T_MAX=$N_T_MAX ROT_TAG=$ROT_TAG ROT_TS=$ROT_TS ROT_FULL=$ROT_FULL KC=$KC SK_KS=$SK_KS AGGS=$AGGS WN_TOP=$WN_TOP SPC_TAG=$SPC_TAG SPC_TS=$SPC_TS SPC_FULL=$SPC_FULL PATCH_N=$PATCH_N PATCH_MIN=$PATCH_MIN N_PATCH=$N_PATCH PK=$PK GW_N=$GW_N SKIP_INPUT_CHECK=1 RUN_PREFIX=$RUN_PREFIX COMBOS_ALL=$COMBOS_ALL FINAL_FLOOR=$FINAL_FLOOR WITH_SYMPHONY=$WITH_SYMPHONY WITH_NICHE=$WITH_NICHE PATCH_SEEDS=$PATCH_SEEDS ALT_FLOORS=$ALT_FLOORS SCANO_RETRY=$SCANO_RETRY EXTRA_MAPS=$EXTRA_MAPS \
    python -u run45.py 2>&1 | tee -a run45.log | grep --line-buffered -v -i -E "warn|^\s*$|it/s\]|s/it\]|%\||geosketch|PCA with"

Tải `/kaggle/working/out23s/summary.md` và `summary.json`.